In [ ]:
!pip install datasets

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

print("Token loaded:", HF_TOKEN[:5] + "...")

In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    "Exploration-Lab/CS-781-Capstone",
    repo_type="dataset",
    token=HF_TOKEN
)

for file in files:
    print(file)

In [ ]:
from datasets import load_dataset

phase1_files = {
    "train": [
        "hf://datasets/Exploration-Lab/CS-781-Capstone/Phase_1_Pretraining/Train/train_acts.json",
        "hf://datasets/Exploration-Lab/CS-781-Capstone/Phase_1_Pretraining/Train/train_high_courts.json",
        "hf://datasets/Exploration-Lab/CS-781-Capstone/Phase_1_Pretraining/Train/train_supreme_court.json",
        "hf://datasets/Exploration-Lab/CS-781-Capstone/Phase_1_Pretraining/Train/train_tribunals.json",
    ],
    "test": "hf://datasets/Exploration-Lab/CS-781-Capstone/Phase_1_Pretraining/Test/test.json",
}

dataset = load_dataset(
    "json",
    data_files=phase1_files,
    token=HF_TOKEN,
    streaming=True,
)

print(dataset)

In [ ]:
train = dataset["train"]
test = dataset["test"]

print("TRAIN EXAMPLES:")
for i, example in zip(range(3), train):
    print(f"\nExample {i + 1}:")
    print(example)

print("\nTEST EXAMPLE:")
test_example = next(iter(test))
print(test_example)

In [ ]:
train_example = next(iter(dataset["train"]))

print("Type:", type(train_example))
print("\nKeys:")
print(train_example.keys())

print("\nExample:")
print(train_example)

In [ ]:
test_example = next(iter(dataset["test"]))

print("Type:", type(test_example))
print("\nKeys:")
print(test_example.keys())

print("\nExample:")
print(test_example)

In [ ]:
import itertools

train = dataset["train"]

for i, example in enumerate(itertools.islice(train, 5)):
    print(f"\n{'=' * 80}")
    print(f"EXAMPLE {i + 1}")
    print(f"{'=' * 80}")
    print("Title:", example["title"])
    print("Category:", example["category"])
    print("Year:", example["year"])
    print("Word count:", example["word_count"])
    print("Text length:", example["text_length_chars"])
    print("Text preview:", example["text"][:500])

In [ ]:
test = dataset["test"]

test_example = next(iter(test))

print("Test type:", type(test_example))
print("Test keys:", test_example.keys())

for key, value in test_example.items():
    print(f"\nKey: {key}")
    print("Type:", type(value))
    print("Preview:", str(value)[:500])

In [ ]:
train = dataset["train"]

for i, example in enumerate(train):
    print(
        f"Example {i + 1}: "
        f"{example['word_count']} words | "
        f"{example['text_length_chars']} chars | "
        f"{example['category']}"
    )

    if i == 9:
        break

In [ ]:
from collections import Counter

train = dataset["train"]

category_counts = Counter()

for i, example in enumerate(train):
    category_counts[example["category"]] += 1

    if i == 99:
        break

print("Categories in first 100 examples:")

for category, count in category_counts.most_common():
    print(f"{category}: {count}")

In [ ]:
from collections import Counter

MAX_EXAMPLES = 10_000

train = dataset["train"]

category_counts = Counter()
year_counts = Counter()

total_words = 0
total_chars = 0
num_examples = 0

min_words = float("inf")
max_words = 0

for example in train:

    num_examples += 1

    words = example["word_count"]
    chars = example["text_length_chars"]

    total_words += words
    total_chars += chars

    min_words = min(min_words, words)
    max_words = max(max_words, words)

    category_counts[example["category"]] += 1
    year_counts[example["year"]] += 1

    if num_examples >= MAX_EXAMPLES:
        break


print("=" * 70)
print("SAMPLED CORPUS STATISTICS")
print("=" * 70)

print(f"Examples inspected: {num_examples:,}")
print(f"Average words: {total_words / num_examples:,.1f}")
print(f"Average characters: {total_chars / num_examples:,.1f}")
print(f"Minimum words: {min_words:,}")
print(f"Maximum words: {max_words:,}")

print("\nTop 20 categories:")

for category, count in category_counts.most_common(20):
    pct = 100 * count / num_examples
    print(f"{count:6,}  {pct:6.2f}%  {category}")

print("\nYears:")

for year, count in sorted(year_counts.items()):
    pct = 100 * count / num_examples
    print(f"{year}: {count:6,}  {pct:6.2f}%")

In [ ]:
!df -h

In [ ]:
!free -h

In [ ]:
from datasets import load_dataset

REPO = "Exploration-Lab/CS-781-Capstone"

train_files = {
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}

print("Configuration loaded.")

In [ ]:
# ============================================================
# CHECK YEAR FIELD
# ============================================================

ds = load_dataset(
    REPO,
    data_files=train_files["acts"],
    split="train",
    streaming=True,
    token=HF_TOKEN,
)

shown = 0

for example in ds:

    year = example.get("year")

    if year is not None and (year < 1800 or year > 2025):

        print("\nSuspicious year:", year)
        print("Title:", example.get("title"))
        print("Category:", example.get("category"))
        print("Text preview:")
        print(str(example.get("text", ""))[:500])

        shown += 1

        if shown >= 10:
            break

In [ ]:
# ============================================================
# PHASE 1 — PREPROCESSING CONFIGURATION
# ============================================================

from datasets import load_dataset
from pathlib import Path
import hashlib
import json
import re
import unicodedata
import gc

REPO = "Exploration-Lab/CS-781-Capstone"

TRAIN_FILES = {
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}

OUTPUT_DIR = Path("/kaggle/working/phase1_processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for domain in TRAIN_FILES:
    (OUTPUT_DIR / domain).mkdir(parents=True, exist_ok=True)

print("Configuration ready.")
print("Output:", OUTPUT_DIR)

In [ ]:
# ============================================================
# PHASE 1 — PREPROCESSING CONFIGURATION
# ============================================================

from datasets import load_dataset
from pathlib import Path
import hashlib
import json
import re
import unicodedata
import gc

REPO = "Exploration-Lab/CS-781-Capstone"

TRAIN_FILES = {
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}

OUTPUT_DIR = Path("/kaggle/working/phase1_processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for domain in TRAIN_FILES:
    (OUTPUT_DIR / domain).mkdir(parents=True, exist_ok=True)

print("Configuration ready.")
print("Output:", OUTPUT_DIR)

In [ ]:
# ============================================================
# DOCUMENT ID
# ============================================================

def make_document_id(example, source_domain):
    """
    Stable ID based on source domain + metadata + original text.
    """

    text = example.get("text") or ""

    payload = (
        source_domain
        + "\n"
        + str(example.get("title", ""))
        + "\n"
        + str(example.get("category", ""))
        + "\n"
        + str(example.get("year", ""))
        + "\n"
        + text
    )

    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()


def text_hash(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


print("ID functions ready.")

In [ ]:
# ============================================================
# PHASE 1 — SELF-CONTAINED STREAMING PREPROCESSOR
# ============================================================

from datasets import load_dataset
from pathlib import Path
import hashlib
import json
import re
import unicodedata
import gc

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

REPO = "Exploration-Lab/CS-781-Capstone"

TRAIN_FILES = {
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}

OUTPUT_DIR = Path("/kaggle/working/phase1_processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for domain in TRAIN_FILES:
    (OUTPUT_DIR / domain).mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Conservative text normalization
# ------------------------------------------------------------

def normalize_text(text):

    if text is None:
        return None

    text = str(text)

    # Unicode normalization
    text = unicodedata.normalize("NFC", text)

    # Normalize line endings
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Remove null characters
    text = text.replace("\x00", "")

    # Normalize spaces/tabs
    text = re.sub(r"[ \t]+", " ", text)

    # Collapse excessive blank lines
    text = re.sub(
        r"\n[ \t]*\n[ \t]*\n+",
        "\n\n",
        text
    )

    return text.strip()


# ------------------------------------------------------------
# Hash functions
# ------------------------------------------------------------

def text_hash(text):

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


def make_document_id(example, source_domain):

    text = example.get("text") or ""

    payload = (
        source_domain
        + "\n"
        + str(example.get("title", ""))
        + "\n"
        + str(example.get("category", ""))
        + "\n"
        + str(example.get("year", ""))
        + "\n"
        + text
    )

    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()


# ------------------------------------------------------------
# Main preprocessing function
# ------------------------------------------------------------

def preprocess_domain(source_domain, file_path):

    print("\n" + "=" * 80)
    print(f"PROCESSING: {source_domain.upper()}")
    print("=" * 80)

    domain_dir = OUTPUT_DIR / source_domain
    domain_dir.mkdir(parents=True, exist_ok=True)

    output_file = domain_dir / "documents.jsonl"
    quarantine_file = domain_dir / "quarantine.jsonl"

    processed = 0
    valid = 0
    quarantined = 0

    # IMPORTANT:
    # We only need exact duplicate detection within the
    # current source stream.
    seen_hashes = set()

    print("Opening streaming dataset...")

    ds = load_dataset(
        REPO,
        data_files=file_path,
        split="train",
        streaming=True,
        token=HF_TOKEN,
    )

    print("Dataset opened. Processing...")

    with open(output_file, "w", encoding="utf-8") as out_f, \
         open(quarantine_file, "w", encoding="utf-8") as bad_f:

        for example in ds:

            processed += 1

            original_text = example.get("text")

            # ------------------------------------------------
            # Objective validation only
            # ------------------------------------------------

            if original_text is None:

                bad_f.write(
                    json.dumps(
                        {
                            "reason": "missing_text",
                            "source_domain": source_domain,
                            "title": example.get("title"),
                            "category": example.get("category"),
                            "year": example.get("year"),
                        },
                        ensure_ascii=False
                    ) + "\n"
                )

                quarantined += 1
                continue

            original_text = str(original_text)

            if not original_text.strip():

                bad_f.write(
                    json.dumps(
                        {
                            "reason": "empty_text",
                            "source_domain": source_domain,
                            "title": example.get("title"),
                            "category": example.get("category"),
                            "year": example.get("year"),
                        },
                        ensure_ascii=False
                    ) + "\n"
                )

                quarantined += 1
                continue

            # ------------------------------------------------
            # Normalize
            # ------------------------------------------------

            text = normalize_text(original_text)

            if not text:

                bad_f.write(
                    json.dumps(
                        {
                            "reason": "empty_after_normalization",
                            "source_domain": source_domain,
                            "title": example.get("title"),
                            "category": example.get("category"),
                            "year": example.get("year"),
                        },
                        ensure_ascii=False
                    ) + "\n"
                )

                quarantined += 1
                continue

            # ------------------------------------------------
            # Exact duplicate detection
            # ------------------------------------------------

            h = text_hash(text)

            if h in seen_hashes:

                bad_f.write(
                    json.dumps(
                        {
                            "reason": "exact_duplicate",
                            "source_domain": source_domain,
                            "title": example.get("title"),
                            "category": example.get("category"),
                            "year": example.get("year"),
                            "text_hash": h,
                        },
                        ensure_ascii=False
                    ) + "\n"
                )

                quarantined += 1
                continue

            seen_hashes.add(h)

            # ------------------------------------------------
            # Stable document ID
            # ------------------------------------------------

            document_id = make_document_id(
                example,
                source_domain
            )

            # ------------------------------------------------
            # Output record
            # ------------------------------------------------

            record = {
                "document_id": document_id,
                "text_hash": h,
                "source_domain": source_domain,

                "title": example.get("title"),
                "category": example.get("category"),
                "year": example.get("year"),

                # Original metadata
                "original_word_count": example.get(
                    "word_count"
                ),

                "original_text_length_chars": example.get(
                    "text_length_chars"
                ),

                # Recomputed from actual text
                "processed_word_count": len(text.split()),
                "processed_text_length_chars": len(text),

                "text": text,
            }

            out_f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                ) + "\n"
            )

            valid += 1

            if processed % 1000 == 0:

                print(
                    f"Processed: {processed:,} | "
                    f"Valid: {valid:,} | "
                    f"Quarantined: {quarantined:,}"
                )

    del ds
    gc.collect()

    print("\n" + "=" * 80)
    print(f"FINISHED: {source_domain.upper()}")
    print("=" * 80)

    print(f"Processed:   {processed:,}")
    print(f"Valid:       {valid:,}")
    print(f"Quarantined: {quarantined:,}")
    print(f"Output:      {output_file}")
    print(f"Quarantine:  {quarantine_file}")


print("Preprocessing pipeline is ready.")

In [ ]:
# ============================================================
# TEST: ACTS
# ============================================================

preprocess_domain(
    "acts",
    TRAIN_FILES["acts"]
)

In [ ]:
# ============================================================
# VERIFY PROCESSED OUTPUT
# ============================================================

acts_file = OUTPUT_DIR / "acts" / "documents.jsonl"

print("Exists:", acts_file.exists())
print("Size:", acts_file.stat().st_size / (1024**2), "MB")

with open(acts_file, "r", encoding="utf-8") as f:
    for i in range(2):
        record = json.loads(next(f))

        print("\n" + "-" * 80)
        print("DOCUMENT", i + 1)
        print("ID:", record["document_id"])
        print("Domain:", record["source_domain"])
        print("Title:", record["title"])
        print("Category:", record["category"])
        print("Year:", record["year"])
        print("Words:", record["processed_word_count"])
        print("Characters:", record["processed_text_length_chars"])
        print("Text preview:")
        print(record["text"][:500])

In [ ]:
# ============================================================
# INSPECT ACTS QUARANTINE
# ============================================================

from collections import Counter
import json

quarantine_file = (
    OUTPUT_DIR / "acts" / "quarantine.jsonl"
)

reasons = Counter()
examples = []

with open(quarantine_file, "r", encoding="utf-8") as f:

    for line in f:
        record = json.loads(line)

        reasons[record["reason"]] += 1

        if len(examples) < 10:
            examples.append(record)

print("=" * 80)
print("ACTS QUARANTINE SUMMARY")
print("=" * 80)

for reason, count in reasons.items():
    print(f"{reason}: {count}")

print("\nExamples:")
for record in examples:
    print(json.dumps(record, indent=2, ensure_ascii=False))

In [ ]:
# ============================================================
# PHASE 1 — PROCESS HIGH COURTS
# ============================================================

preprocess_domain(
    "high_courts",
    TRAIN_FILES["high_courts"]
)

In [ ]:
# ============================================================
# PHASE 1 — LOW-RAM CONFIGURATION
# ============================================================

from datasets import load_dataset
from pathlib import Path
import hashlib
import json
import re
import unicodedata
import gc

REPO = "Exploration-Lab/CS-781-Capstone"

TRAIN_FILES = {
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}

OUTPUT_DIR = Path("/kaggle/working/phase1_processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Ready.")

In [ ]:
# ============================================================
# CONSERVATIVE NORMALIZATION
# ============================================================

def normalize_text(text):

    if text is None:
        return None

    text = str(text)

    text = unicodedata.normalize("NFC", text)

    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")
    text = text.replace("\x00", "")

    text = re.sub(r"[ \t]+", " ", text)

    text = re.sub(
        r"\n[ \t]*\n[ \t]*\n+",
        "\n\n",
        text
    )

    return text.strip()


def text_hash(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


def make_document_id(example, source_domain):

    payload = (
        source_domain
        + "\n"
        + str(example.get("title", ""))
        + "\n"
        + str(example.get("category", ""))
        + "\n"
        + str(example.get("year", ""))
        + "\n"
        + str(example.get("text", ""))
    )

    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()


print("Functions ready.")

In [ ]:
# ============================================================
# LOW-RAM STREAMING PREPROCESSOR
# ============================================================

def preprocess_domain_low_ram(source_domain, file_path):

    print("\n" + "=" * 80)
    print(f"PROCESSING: {source_domain.upper()}")
    print("=" * 80)

    domain_dir = OUTPUT_DIR / source_domain
    domain_dir.mkdir(parents=True, exist_ok=True)

    output_file = domain_dir / "documents.jsonl"
    quarantine_file = domain_dir / "quarantine.jsonl"

    processed = 0
    valid = 0
    quarantined = 0

    ds = load_dataset(
        REPO,
        data_files=file_path,
        split="train",
        streaming=True,
        token=HF_TOKEN,
    )

    with open(output_file, "w", encoding="utf-8") as out_f, \
         open(quarantine_file, "w", encoding="utf-8") as bad_f:

        for example in ds:

            processed += 1

            raw_text = example.get("text")

            # ------------------------------------------------
            # Missing text
            # ------------------------------------------------

            if raw_text is None:

                bad_f.write(json.dumps({
                    "reason": "missing_text",
                    "source_domain": source_domain,
                    "title": example.get("title"),
                    "category": example.get("category"),
                    "year": example.get("year"),
                }, ensure_ascii=False) + "\n")

                quarantined += 1
                continue

            # ------------------------------------------------
            # Normalize
            # ------------------------------------------------

            text = normalize_text(raw_text)

            # ------------------------------------------------
            # Empty text
            # ------------------------------------------------

            if not text:

                bad_f.write(json.dumps({
                    "reason": "empty_text",
                    "source_domain": source_domain,
                    "title": example.get("title"),
                    "category": example.get("category"),
                    "year": example.get("year"),
                }, ensure_ascii=False) + "\n")

                quarantined += 1
                continue

            # ------------------------------------------------
            # Create hashes
            # ------------------------------------------------

            h = text_hash(text)

            document_id = make_document_id(
                example,
                source_domain
            )

            # ------------------------------------------------
            # Write immediately
            # ------------------------------------------------

            record = {
                "document_id": document_id,
                "text_hash": h,
                "source_domain": source_domain,

                "title": example.get("title"),
                "category": example.get("category"),
                "year": example.get("year"),

                "original_word_count": example.get(
                    "word_count"
                ),

                "original_text_length_chars": example.get(
                    "text_length_chars"
                ),

                "processed_word_count": len(text.split()),
                "processed_text_length_chars": len(text),

                "text": text,
            }

            out_f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                ) + "\n"
            )

            valid += 1

            # Release references to large strings
            del text
            del raw_text
            del record

            if processed % 1000 == 0:

                print(
                    f"Processed: {processed:,} | "
                    f"Valid: {valid:,} | "
                    f"Quarantined: {quarantined:,}"
                )

                gc.collect()

    del ds
    gc.collect()

    print("\n" + "=" * 80)
    print(f"FINISHED: {source_domain.upper()}")
    print("=" * 80)

    print(f"Processed:   {processed:,}")
    print(f"Valid:       {valid:,}")
    print(f"Quarantined: {quarantined:,}")
    print(f"Output:      {output_file}")

In [ ]:
preprocess_domain_low_ram(
    "high_courts",
    TRAIN_FILES["high_courts"]
)

In [ ]:
# ============================================================
# PHASE 1 — LOW-RAM DISK-BASED PREPROCESSING
# ============================================================

from pathlib import Path
import hashlib
import json
import re
import unicodedata
import gc
import os

from huggingface_hub import hf_hub_download

REPO = "Exploration-Lab/CS-781-Capstone"

RAW_DIR = Path("/kaggle/working/phase1_raw")
PROCESSED_DIR = Path("/kaggle/working/phase1_processed")

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILES = {
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}

print("Configuration ready.")
print("Raw directory:", RAW_DIR)
print("Processed directory:", PROCESSED_DIR)

In [ ]:
# ============================================================
# DOWNLOAD RAW FILE TO DISK
# ============================================================

def download_raw(domain):

    filename = TRAIN_FILES[domain]

    print("=" * 80)
    print(f"DOWNLOADING: {domain.upper()}")
    print("=" * 80)

    local_path = hf_hub_download(
        repo_id=REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN,
        local_dir=str(RAW_DIR),
    )

    size_gb = Path(local_path).stat().st_size / (1024 ** 3)

    print()
    print("Downloaded:")
    print(local_path)
    print(f"Size: {size_gb:.2f} GB")

    return Path(local_path)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

print("Token loaded:", HF_TOKEN[:5] + "...")

In [ ]:
# ============================================================
# DOWNLOAD HIGH COURTS RAW FILE
# ============================================================

high_court_path = download_raw("high_courts")

In [ ]:
# ============================================================
# INSPECT RAW FILE FORMAT WITHOUT LOADING IT
# ============================================================

with open(high_court_path, "rb") as f:

    # Read only the first 4096 bytes
    sample = f.read(4096)

print(sample[:2000].decode("utf-8", errors="replace"))

In [ ]:
!pip -q install ijson

In [ ]:
# ============================================================
# PHASE 1 — HIGH COURTS DISK-STREAMING PREPROCESSOR
# ============================================================

import ijson
import json
import hashlib
import re
import unicodedata
import gc
from pathlib import Path


# ------------------------------------------------------------
# Conservative text normalization
# ------------------------------------------------------------

def normalize_text(text):

    if text is None:
        return None

    text = str(text)

    # Unicode normalization
    text = unicodedata.normalize("NFC", text)

    # Normalize line endings
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Remove null characters
    text = text.replace("\x00", "")

    # Normalize spaces/tabs
    text = re.sub(r"[ \t]+", " ", text)

    # Collapse excessive blank lines
    text = re.sub(
        r"\n[ \t]*\n[ \t]*\n+",
        "\n\n",
        text
    )

    return text.strip()


# ------------------------------------------------------------
# Hash
# ------------------------------------------------------------

def text_hash(text):

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


# ------------------------------------------------------------
# Stable document ID
# ------------------------------------------------------------

def make_document_id(example, source_domain):

    payload = (
        source_domain
        + "\n"
        + str(example.get("filepath", ""))
        + "\n"
        + str(example.get("court_id", ""))
        + "\n"
        + str(example.get("year", ""))
        + "\n"
        + str(example.get("bench", ""))
        + "\n"
        + str(example.get("text", ""))
    )

    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()


# ------------------------------------------------------------
# PROCESS HIGH COURTS
# ------------------------------------------------------------

def preprocess_high_courts(raw_file):

    source_domain = "high_courts"

    output_dir = (
        PROCESSED_DIR / source_domain
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    output_file = (
        output_dir / "documents.jsonl"
    )

    quarantine_file = (
        output_dir / "quarantine.jsonl"
    )

    processed = 0
    valid = 0
    quarantined = 0

    print("=" * 80)
    print("HIGH COURTS — DISK STREAMING")
    print("=" * 80)

    print("Input :", raw_file)
    print("Output:", output_file)
    print()

    with open(
        raw_file,
        "rb",
        buffering=1024 * 1024
    ) as input_f, \
         open(
             output_file,
             "w",
             encoding="utf-8",
             buffering=1024 * 1024
         ) as out_f, \
         open(
             quarantine_file,
             "w",
             encoding="utf-8",
             buffering=1024 * 1024
         ) as bad_f:

        # Because the source is:
        #
        # [
        #   {...},
        #   {...}
        # ]
        #
        # ijson.items(..., "item") gives one object
        # at a time.

        for example in ijson.items(
            input_f,
            "item"
        ):

            processed += 1

            raw_text = example.get("text")

            # ------------------------------------------------
            # Missing / empty text
            # ------------------------------------------------

            if raw_text is None:

                bad_f.write(
                    json.dumps(
                        {
                            "reason": "missing_text",
                            "source_domain": source_domain,
                            "court": example.get("court"),
                            "court_id": example.get("court_id"),
                            "year": example.get("year"),
                            "bench": example.get("bench"),
                            "filepath": example.get("filepath"),
                        },
                        ensure_ascii=False
                    ) + "\n"
                )

                quarantined += 1
                continue

            text = normalize_text(raw_text)

            if not text:

                bad_f.write(
                    json.dumps(
                        {
                            "reason": "empty_text",
                            "source_domain": source_domain,
                            "court": example.get("court"),
                            "court_id": example.get("court_id"),
                            "year": example.get("year"),
                            "bench": example.get("bench"),
                            "filepath": example.get("filepath"),
                        },
                        ensure_ascii=False
                    ) + "\n"
                )

                quarantined += 1
                continue

            # ------------------------------------------------
            # Hash + stable ID
            # ------------------------------------------------

            h = text_hash(text)

            document_id = make_document_id(
                example,
                source_domain
            )

            # ------------------------------------------------
            # Preserve High Court metadata
            # ------------------------------------------------

            record = {
                "document_id": document_id,
                "text_hash": h,
                "source_domain": source_domain,

                "court": example.get("court"),
                "court_id": example.get("court_id"),
                "year": example.get("year"),
                "bench": example.get("bench"),
                "filepath": example.get("filepath"),

                "page_count": example.get("page_count"),

                "original_word_count": example.get(
                    "word_count"
                ),

                "original_char_count": example.get(
                    "char_count"
                ),

                "processed_word_count": len(
                    text.split()
                ),

                "processed_char_count": len(text),

                "text": text,
            }

            # ------------------------------------------------
            # WRITE IMMEDIATELY
            # ------------------------------------------------

            out_f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                ) + "\n"
            )

            valid += 1

            # Release large references
            del raw_text
            del text
            del record
            del example

            if processed % 1000 == 0:

                print(
                    f"Processed: {processed:,} | "
                    f"Valid: {valid:,} | "
                    f"Quarantined: {quarantined:,}"
                )

                gc.collect()

    gc.collect()

    print()
    print("=" * 80)
    print("HIGH COURTS COMPLETE")
    print("=" * 80)

    print(f"Processed:   {processed:,}")
    print(f"Valid:       {valid:,}")
    print(f"Quarantined: {quarantined:,}")

    print()
    print("Output:")
    print(output_file)

    print()
    print("Quarantine:")
    print(quarantine_file)

    return output_file

In [ ]:
# ============================================================
# RUN HIGH COURTS PREPROCESSING
# ============================================================

high_court_processed = preprocess_high_courts(
    high_court_path
)

In [ ]:
from pathlib import Path
import os

raw = Path("/kaggle/working/phase1_raw")
processed = Path("/kaggle/working/phase1_processed")

def dir_size_gb(path):
    total = 0
    if path.exists():
        for p in path.rglob("*"):
            if p.is_file():
                total += p.stat().st_size
    return total / (1024**3)

print("RAW:", dir_size_gb(raw), "GB")
print("PROCESSED:", dir_size_gb(processed), "GB")

!df -h /kaggle/working

In [ ]:
!df -h

In [ ]:
!du -sh /kaggle/working/* 2>/dev/null | sort -h

In [ ]:
import shutil
from pathlib import Path

high_court_processed = Path(
    "/kaggle/working/phase1_processed/high_courts"
)

if high_court_processed.exists():
    shutil.rmtree(high_court_processed)
    print("Deleted incomplete High Courts output.")
else:
    print("High Courts processed directory not found.")

In [ ]:
!du -sh /kaggle/working/* 2>/dev/null | sort -h
!df -h /kaggle/lib

In [ ]:
!pip -q install ijson

In [ ]:
# ============================================================
# PHASE 1 — HIGH COURTS
# LOW-RAM + COMPRESSED SHARDS
# ============================================================

import ijson
import json
import hashlib
import re
import unicodedata
import gzip
import gc
from pathlib import Path


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

RAW_FILE = Path(
    "/kaggle/working/phase1_raw/"
    "Phase_1_Pretraining/Train/train_high_courts.json"
)

OUTPUT_DIR = Path(
    "/kaggle/working/phase1_processed/high_courts"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

QUARANTINE_FILE = (
    OUTPUT_DIR / "quarantine.jsonl.gz"
)

MANIFEST_FILE = (
    OUTPUT_DIR / "manifest.json"
)


# ------------------------------------------------------------
# Text normalization
# ------------------------------------------------------------

def normalize_text(text):

    if text is None:
        return None

    text = str(text)

    text = unicodedata.normalize(
        "NFC",
        text
    )

    text = text.replace(
        "\r\n",
        "\n"
    )

    text = text.replace(
        "\r",
        "\n"
    )

    text = text.replace(
        "\x00",
        ""
    )

    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )

    text = re.sub(
        r"\n[ \t]*\n[ \t]*\n+",
        "\n\n",
        text
    )

    return text.strip()


# ------------------------------------------------------------
# SHA256
# ------------------------------------------------------------

def sha256_text(text):

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


# ------------------------------------------------------------
# Stable document ID
# ------------------------------------------------------------

def make_document_id(example, text_hash):

    identity = (
        "high_courts\n"
        + str(example.get("filepath", ""))
        + "\n"
        + str(example.get("court_id", ""))
        + "\n"
        + str(example.get("year", ""))
        + "\n"
        + str(example.get("bench", ""))
        + "\n"
        + text_hash
    )

    return hashlib.sha256(
        identity.encode("utf-8")
    ).hexdigest()


# ------------------------------------------------------------
# PROCESSOR
# ------------------------------------------------------------

def process_high_courts():

    if not RAW_FILE.exists():
        raise FileNotFoundError(
            f"Raw file not found:\n{RAW_FILE}"
        )

    print("=" * 80)
    print("HIGH COURTS — COMPRESSED SHARD PROCESSING")
    print("=" * 80)

    print(
        f"Raw size: "
        f"{RAW_FILE.stat().st_size / (1024**3):.2f} GB"
    )

    # --------------------------------------------------------
    # Configuration
    # --------------------------------------------------------

    # Number of documents per compressed shard.
    # 50k keeps individual files manageable.
    SHARD_SIZE = 50_000

    processed = 0
    valid = 0
    quarantined = 0

    shard_number = 0
    shard_records = 0

    shard_path = None
    shard_file = None

    shard_metadata = []

    # --------------------------------------------------------
    # Quarantine
    # --------------------------------------------------------

    quarantine_file = gzip.open(
        QUARANTINE_FILE,
        "wt",
        encoding="utf-8"
    )

    # --------------------------------------------------------
    # Open raw source
    # --------------------------------------------------------

    with open(
        RAW_FILE,
        "rb",
        buffering=1024 * 1024
    ) as input_file:

        items = ijson.items(
            input_file,
            "item"
        )

        for example in items:

            processed += 1

            try:

                raw_text = example.get("text")

                # ------------------------------------------------
                # Missing text
                # ------------------------------------------------

                if raw_text is None:

                    quarantine_file.write(
                        json.dumps(
                            {
                                "reason": "missing_text",
                                "source_domain": "high_courts",
                                "court": example.get("court"),
                                "court_id": example.get("court_id"),
                                "year": example.get("year"),
                                "bench": example.get("bench"),
                                "filepath": example.get("filepath"),
                            },
                            ensure_ascii=False
                        ) + "\n"
                    )

                    quarantined += 1
                    continue

                # ------------------------------------------------
                # Normalize
                # ------------------------------------------------

                text = normalize_text(
                    raw_text
                )

                if not text:

                    quarantine_file.write(
                        json.dumps(
                            {
                                "reason": "empty_text",
                                "source_domain": "high_courts",
                                "court": example.get("court"),
                                "court_id": example.get("court_id"),
                                "year": example.get("year"),
                                "bench": example.get("bench"),
                                "filepath": example.get("filepath"),
                            },
                            ensure_ascii=False
                        ) + "\n"
                    )

                    quarantined += 1
                    continue

                # ------------------------------------------------
                # Hash
                # ------------------------------------------------

                text_hash = sha256_text(
                    text
                )

                document_id = make_document_id(
                    example,
                    text_hash
                )

                # ------------------------------------------------
                # Record
                # ------------------------------------------------

                record = {
                    "document_id": document_id,
                    "text_hash": text_hash,
                    "source_domain": "high_courts",

                    "court": example.get(
                        "court"
                    ),

                    "court_id": example.get(
                        "court_id"
                    ),

                    "year": example.get(
                        "year"
                    ),

                    "bench": example.get(
                        "bench"
                    ),

                    "filepath": example.get(
                        "filepath"
                    ),

                    "page_count": example.get(
                        "page_count"
                    ),

                    "original_word_count": example.get(
                        "word_count"
                    ),

                    "original_char_count": example.get(
                        "char_count"
                    ),

                    "processed_word_count": len(
                        text.split()
                    ),

                    "processed_char_count": len(
                        text
                    ),

                    "text": text,
                }

                # ------------------------------------------------
                # Create shard when needed
                # ------------------------------------------------

                if shard_file is None:

                    shard_path = (
                        OUTPUT_DIR
                        / f"shard_{shard_number:05d}.jsonl.gz"
                    )

                    shard_file = gzip.open(
                        shard_path,
                        "wt",
                        encoding="utf-8"
                    )

                    shard_records = 0

                # ------------------------------------------------
                # Write record
                # ------------------------------------------------

                shard_file.write(
                    json.dumps(
                        record,
                        ensure_ascii=False
                    ) + "\n"
                )

                shard_records += 1
                valid += 1

                # ------------------------------------------------
                # Finish shard
                # ------------------------------------------------

                if shard_records >= SHARD_SIZE:

                    shard_file.close()

                    shard_size = (
                        shard_path.stat().st_size
                        / (1024**2)
                    )

                    shard_metadata.append(
                        {
                            "shard": shard_path.name,
                            "records": shard_records,
                            "size_mb": round(
                                shard_size,
                                2
                            )
                        }
                    )

                    print(
                        f"Shard {shard_number:05d} complete | "
                        f"Records: {valid:,} | "
                        f"Processed: {processed:,} | "
                        f"Quarantined: {quarantined:,} | "
                        f"Shard size: {shard_size:.1f} MB"
                    )

                    shard_number += 1
                    shard_file = None
                    shard_path = None
                    shard_records = 0

                    gc.collect()

            except Exception as e:

                quarantine_file.write(
                    json.dumps(
                        {
                            "reason": "processing_error",
                            "error": str(e),
                            "source_domain": "high_courts",
                            "court": example.get("court"),
                            "court_id": example.get("court_id"),
                            "year": example.get("year"),
                            "filepath": example.get("filepath"),
                        },
                        ensure_ascii=False
                    ) + "\n"
                )

                quarantined += 1

            finally:

                # Remove references to potentially
                # large strings/objects.
                try:
                    del raw_text
                except:
                    pass

                try:
                    del text
                except:
                    pass

                try:
                    del record
                except:
                    pass

                try:
                    del example
                except:
                    pass

            # ----------------------------------------------------
            # Progress
            # ----------------------------------------------------

            if processed % 10_000 == 0:

                print(
                    f"Progress: "
                    f"{processed:,} processed | "
                    f"{valid:,} valid | "
                    f"{quarantined:,} quarantined"
                )

                gc.collect()

    # --------------------------------------------------------
    # Close final partial shard
    # --------------------------------------------------------

    if shard_file is not None:

        shard_file.close()

        shard_size = (
            shard_path.stat().st_size
            / (1024**2)
        )

        shard_metadata.append(
            {
                "shard": shard_path.name,
                "records": shard_records,
                "size_mb": round(
                    shard_size,
                    2
                )
            }
        )

    quarantine_file.close()

    # --------------------------------------------------------
    # Manifest
    # --------------------------------------------------------

    manifest = {
        "dataset": "CS-781-Capstone",
        "phase": "phase1",
        "source_domain": "high_courts",

        "source_file": (
            "Phase_1_Pretraining/Train/"
            "train_high_courts.json"
        ),

        "raw_size_bytes": RAW_FILE.stat().st_size,

        "processed_records": valid,
        "quarantined_records": quarantined,
        "total_records_seen": processed,

        "shard_size_records": SHARD_SIZE,

        "shards": shard_metadata,

        "normalization": [
            "Unicode NFC normalization",
            "CRLF normalization",
            "null character removal",
            "space/tab normalization",
            "excessive blank-line normalization",
            "leading/trailing whitespace removal"
        ]
    }

    with open(
        MANIFEST_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            manifest,
            f,
            indent=2,
            ensure_ascii=False
        )

    gc.collect()

    print()
    print("=" * 80)
    print("HIGH COURTS PROCESSING COMPLETE")
    print("=" * 80)

    print(
        f"Total processed : {processed:,}"
    )

    print(
        f"Valid           : {valid:,}"
    )

    print(
        f"Quarantined     : {quarantined:,}"
    )

    print(
        f"Shards          : {len(shard_metadata):,}"
    )

    print(
        f"Manifest        : {MANIFEST_FILE}"
    )

    print()
    print("IMPORTANT:")
    print("Raw source has NOT been deleted.")

In [ ]:
!df -h /kaggle/lib
!du -sh /kaggle/working/phase1_processed/* 2>/dev/null
!du -sh /kaggle/working/phase1_raw/* 2>/dev/null

In [ ]:
process_high_courts()

In [ ]:
!df -h /kaggle/lib
!du -sh /kaggle/working/phase1_processed/high_courts

In [ ]:
import json
from pathlib import Path

HC_DIR = Path("/kaggle/working/phase1_processed/high_courts")
MANIFEST = HC_DIR / "manifest.json"

with open(MANIFEST, "r", encoding="utf-8") as f:
    manifest = json.load(f)

print("Total records seen :", manifest["total_records_seen"])
print("Valid records      :", manifest["processed_records"])
print("Quarantined        :", manifest["quarantined_records"])
print("Number of shards   :", len(manifest["shards"]))

print("\nFirst 3 shards:")
for s in manifest["shards"][:3]:
    print(s)

print("\nLast 3 shards:")
for s in manifest["shards"][-3:]:
    print(s)

In [ ]:
missing = []

for shard in manifest["shards"]:
    path = HC_DIR / shard["shard"]

    if not path.exists():
        missing.append(shard["shard"])

print("Expected shards :", len(manifest["shards"]))
print("Missing shards  :", len(missing))

if missing:
    print("Missing:")
    for x in missing[:20]:
        print(x)
else:
    print("✓ All shards exist")

In [ ]:
import gzip
import json

checked = 0
bad = 0

for shard_info in manifest["shards"]:

    shard_path = HC_DIR / shard_info["shard"]

    with gzip.open(
        shard_path,
        "rt",
        encoding="utf-8"
    ) as f:

        for line in f:

            try:
                record = json.loads(line)

                # Basic structural checks
                assert record["document_id"]
                assert record["text_hash"]
                assert record["source_domain"] == "high_courts"
                assert record["text"]

                checked += 1

            except Exception as e:
                bad += 1
                print(
                    "BAD RECORD:",
                    shard_path.name,
                    repr(e)
                )

            if checked >= 1000:
                break

    if checked >= 1000:
        break

print("=" * 60)
print("SAMPLE VALIDATION")
print("=" * 60)
print("Records checked :", checked)
print("Bad records     :", bad)

if bad == 0:
    print("✓ Compressed shards can be read successfully")
else:
    print("⚠ Problems detected")

In [ ]:
from pathlib import Path
import shutil

RAW_HC = Path("/kaggle/working/phase1_raw")

print("Before:")
!df -h /kaggle/lib

# IMPORTANT:
# This removes the raw High Courts download directory.
# Do this only because the processed shards + manifest
# have already been validated.

if RAW_HC.exists():
    shutil.rmtree(RAW_HC)

print("\nAfter:")
!df -h /kaggle/lib

In [ ]:
from datasets import load_dataset

REPO = "Exploration-Lab/CS-781-Capstone"

print("Checking Supreme Court dataset...")

ds = load_dataset(
    REPO,
    data_files="Phase_1_Pretraining/Train/train_supreme_court.json",
    split="train",
    streaming=True,
    token=HF_TOKEN,
)

print("✓ Supreme Court stream opened successfully.")
print(ds)

In [ ]:
# ============================================================
# PHASE 1 — SUPREME COURT
# STREAM DIRECTLY FROM HUGGING FACE
# LOW-RAM + COMPRESSED SHARDS
# ============================================================

import json
import hashlib
import re
import unicodedata
import gzip
import gc
from pathlib import Path


REPO = "Exploration-Lab/CS-781-Capstone"

SOURCE_FILE = (
    "Phase_1_Pretraining/Train/"
    "train_supreme_court.json"
)

OUTPUT_DIR = Path(
    "/kaggle/working/phase1_processed/supreme_court"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

QUARANTINE_FILE = (
    OUTPUT_DIR / "quarantine.jsonl.gz"
)

MANIFEST_FILE = (
    OUTPUT_DIR / "manifest.json"
)


# ------------------------------------------------------------
# Text normalization
# ------------------------------------------------------------

def normalize_text(text):

    if text is None:
        return None

    text = str(text)

    text = unicodedata.normalize("NFC", text)

    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")
    text = text.replace("\x00", "")

    text = re.sub(r"[ \t]+", " ", text)

    text = re.sub(
        r"\n[ \t]*\n[ \t]*\n+",
        "\n\n",
        text
    )

    return text.strip()


# ------------------------------------------------------------
# Hash
# ------------------------------------------------------------

def sha256_text(text):

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


# ------------------------------------------------------------
# Stable ID
# ------------------------------------------------------------

def make_document_id(example, text_hash):

    identity = (
        "supreme_court\n"
        + str(example.get("filepath", ""))
        + "\n"
        + str(example.get("court", ""))
        + "\n"
        + str(example.get("year", ""))
        + "\n"
        + text_hash
    )

    return hashlib.sha256(
        identity.encode("utf-8")
    ).hexdigest()


# ------------------------------------------------------------
# PROCESSOR
# ------------------------------------------------------------

def process_supreme_court():

    print("=" * 80)
    print("SUPREME COURT — STREAMING PROCESSING")
    print("=" * 80)

    print("Opening Hugging Face stream...")

    ds = load_dataset(
        REPO,
        data_files=SOURCE_FILE,
        split="train",
        streaming=True,
        token=HF_TOKEN,
    )

    print("Stream opened successfully.")

    # --------------------------------------------------------
    # Configuration
    # --------------------------------------------------------

    SHARD_SIZE = 50_000

    processed = 0
    valid = 0
    quarantined = 0

    shard_number = 0
    shard_records = 0

    shard_path = None
    shard_file = None

    shard_metadata = []

    quarantine_file = gzip.open(
        QUARANTINE_FILE,
        "wt",
        encoding="utf-8"
    )

    # --------------------------------------------------------
    # Iterate stream
    # --------------------------------------------------------

    for example in ds:

        processed += 1

        try:

            raw_text = example.get("text")

            # ------------------------------------------------
            # Missing / empty text
            # ------------------------------------------------

            if raw_text is None:

                quarantine_file.write(
                    json.dumps(
                        {
                            "reason": "missing_text",
                            "source_domain": "supreme_court",
                            "court": example.get("court"),
                            "year": example.get("year"),
                            "filepath": example.get("filepath"),
                        },
                        ensure_ascii=False
                    ) + "\n"
                )

                quarantined += 1
                continue

            text = normalize_text(raw_text)

            if not text:

                quarantine_file.write(
                    json.dumps(
                        {
                            "reason": "empty_text",
                            "source_domain": "supreme_court",
                            "court": example.get("court"),
                            "year": example.get("year"),
                            "filepath": example.get("filepath"),
                        },
                        ensure_ascii=False
                    ) + "\n"
                )

                quarantined += 1
                continue

            # ------------------------------------------------
            # Hash / ID
            # ------------------------------------------------

            text_hash = sha256_text(text)

            document_id = make_document_id(
                example,
                text_hash
            )

            # ------------------------------------------------
            # Record
            # ------------------------------------------------

            record = {
                "document_id": document_id,
                "text_hash": text_hash,
                "source_domain": "supreme_court",

                "court": example.get("court"),

                "year": example.get("year"),

                "filepath": example.get("filepath"),

                "page_count": example.get("page_count"),

                "original_word_count": example.get(
                    "word_count"
                ),

                "original_char_count": example.get(
                    "char_count"
                ),

                "processed_word_count": len(
                    text.split()
                ),

                "processed_char_count": len(
                    text
                ),

                "text": text,
            }

            # ------------------------------------------------
            # Open shard
            # ------------------------------------------------

            if shard_file is None:

                shard_path = (
                    OUTPUT_DIR
                    / f"shard_{shard_number:05d}.jsonl.gz"
                )

                shard_file = gzip.open(
                    shard_path,
                    "wt",
                    encoding="utf-8"
                )

                shard_records = 0

            # ------------------------------------------------
            # Write
            # ------------------------------------------------

            shard_file.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                ) + "\n"
            )

            shard_records += 1
            valid += 1

            # ------------------------------------------------
            # Finish shard
            # ------------------------------------------------

            if shard_records >= SHARD_SIZE:

                shard_file.close()

                shard_size = (
                    shard_path.stat().st_size
                    / (1024**2)
                )

                shard_metadata.append(
                    {
                        "shard": shard_path.name,
                        "records": shard_records,
                        "size_mb": round(
                            shard_size,
                            2
                        )
                    }
                )

                print(
                    f"Shard {shard_number:05d} complete | "
                    f"Records: {valid:,} | "
                    f"Processed: {processed:,} | "
                    f"Quarantined: {quarantined:,} | "
                    f"Shard size: {shard_size:.1f} MB"
                )

                shard_number += 1
                shard_file = None
                shard_path = None
                shard_records = 0

                gc.collect()

        except Exception as e:

            quarantine_file.write(
                json.dumps(
                    {
                        "reason": "processing_error",
                        "error": str(e),
                        "source_domain": "supreme_court",
                        "court": example.get("court"),
                        "year": example.get("year"),
                        "filepath": example.get("filepath"),
                    },
                    ensure_ascii=False
                ) + "\n"
            )

            quarantined += 1

        finally:

            for name in (
                "raw_text",
                "text",
                "record",
                "example"
            ):
                try:
                    del globals()[name]
                except:
                    pass

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if processed % 10_000 == 0:

            print(
                f"Progress: "
                f"{processed:,} processed | "
                f"{valid:,} valid | "
                f"{quarantined:,} quarantined"
            )

            gc.collect()

    # --------------------------------------------------------
    # Final partial shard
    # --------------------------------------------------------

    if shard_file is not None:

        shard_file.close()

        shard_size = (
            shard_path.stat().st_size
            / (1024**2)
        )

        shard_metadata.append(
            {
                "shard": shard_path.name,
                "records": shard_records,
                "size_mb": round(
                    shard_size,
                    2
                )
            }
        )

    quarantine_file.close()

    # --------------------------------------------------------
    # Manifest
    # --------------------------------------------------------

    manifest = {
        "dataset": "CS-781-Capstone",
        "phase": "phase1",
        "source_domain": "supreme_court",

        "source_file": SOURCE_FILE,

        "total_records_seen": processed,
        "processed_records": valid,
        "quarantined_records": quarantined,

        "shard_size_records": SHARD_SIZE,

        "shards": shard_metadata,

        "normalization": [
            "Unicode NFC normalization",
            "CRLF normalization",
            "null character removal",
            "space/tab normalization",
            "excessive blank-line normalization",
            "leading/trailing whitespace removal"
        ]
    }

    with open(
        MANIFEST_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            manifest,
            f,
            indent=2,
            ensure_ascii=False
        )

    gc.collect()

    print()
    print("=" * 80)
    print("SUPREME COURT PROCESSING COMPLETE")
    print("=" * 80)

    print(f"Total processed : {processed:,}")
    print(f"Valid           : {valid:,}")
    print(f"Quarantined     : {quarantined:,}")
    print(f"Shards          : {len(shard_metadata):,}")
    print(f"Manifest        : {MANIFEST_FILE}")

    print()
    print("Hugging Face source was streamed directly.")
    print("No raw intermediate copy was created.")

In [ ]:
!df -h /kaggle/lib
!du -sh /kaggle/working/phase1_processed

In [ ]:
process_supreme_court()

In [ ]:
import json
from pathlib import Path

SC_DIR = Path(
    "/kaggle/working/phase1_processed/supreme_court"
)

with open(
    SC_DIR / "manifest.json",
    "r",
    encoding="utf-8"
) as f:
    manifest = json.load(f)

print("Total records seen :", manifest["total_records_seen"])
print("Valid records      :", manifest["processed_records"])
print("Quarantined        :", manifest["quarantined_records"])
print("Number of shards   :", len(manifest["shards"]))

print("\nShards:")
for shard in manifest["shards"]:
    print(shard)

In [ ]:
missing = []

for shard in manifest["shards"]:
    path = SC_DIR / shard["shard"]

    if not path.exists():
        missing.append(shard["shard"])

print("Expected shards :", len(manifest["shards"]))
print("Missing shards  :", len(missing))

if missing:
    print("Missing:", missing)
else:
    print("✓ All shards exist")

In [ ]:
import gzip
import json

checked = 0
bad = 0

for shard_info in manifest["shards"]:

    shard_path = SC_DIR / shard_info["shard"]

    with gzip.open(
        shard_path,
        "rt",
        encoding="utf-8"
    ) as f:

        for line in f:

            try:
                record = json.loads(line)

                assert record["document_id"]
                assert record["text_hash"]
                assert record["source_domain"] == "supreme_court"
                assert record["text"]

                checked += 1

            except Exception as e:
                bad += 1
                print(
                    "BAD RECORD:",
                    shard_path.name,
                    repr(e)
                )

            if checked >= 1000:
                break

    if checked >= 1000:
        break

print("=" * 60)
print("SUPREME COURT SAMPLE VALIDATION")
print("=" * 60)
print("Records checked :", checked)
print("Bad records     :", bad)

if bad == 0:
    print("✓ Compressed Supreme Court shards are readable")
else:
    print("⚠ Problems detected")

In [ ]:
import gzip
import json

Q = SC_DIR / "quarantine.jsonl.gz"

count = 0

with gzip.open(Q, "rt", encoding="utf-8") as f:

    for line in f:

        item = json.loads(line)
        print(item)

        count += 1

print("\nTotal quarantined:", count)

In [ ]:
!df -h /kaggle/lib
!du -sh /kaggle/working/phase1_processed/*

In [ ]:
from datasets import load_dataset

REPO = "Exploration-Lab/CS-781-Capstone"

TRIBUNALS_FILE = (
    "Phase_1_Pretraining/Train/"
    "train_tribunals.json"
)

tribunals_ds = load_dataset(
    REPO,
    data_files=TRIBUNALS_FILE,
    split="train",
    streaming=True,
    token=HF_TOKEN,
)

print("✓ Tribunals stream opened successfully")
print(tribunals_ds)

In [ ]:
# ============================================================
# FIX HF DATASETS / TORCH INTERACTION
# ============================================================

import os

# Tell Hugging Face datasets not to use Torch
os.environ["USE_TORCH"] = "0"
os.environ["HF_DATASETS_DISABLE_TORCH"] = "1"

print("Torch disabled for this process.")

In [ ]:
# ============================================================
# SAFER TRIBUNALS PROCESSOR — LOW RAM
# ============================================================

from datasets import load_dataset
from pathlib import Path
import json
import gzip
import hashlib
import unicodedata
import re
import gc


REPO = "Exploration-Lab/CS-781-Capstone"

SOURCE_FILE = "Phase_1_Pretraining/Train/train_tribunals.json"

OUTPUT_DIR = Path(
    "/kaggle/working/phase1_processed/tribunals"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

QUARANTINE_FILE = OUTPUT_DIR / "quarantine.jsonl.gz"
MANIFEST_FILE = OUTPUT_DIR / "manifest.json"

SHARD_SIZE = 50_000


# ------------------------------------------------------------
# NORMALIZATION
# ------------------------------------------------------------

def normalize_text(text):

    if text is None:
        return ""

    text = str(text)

    text = unicodedata.normalize("NFC", text)

    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")
    text = text.replace("\x00", "")

    text = re.sub(r"[ \t]+", " ", text)

    text = re.sub(
        r"\n[ \t]*\n[ \t]*\n+",
        "\n\n",
        text
    )

    return text.strip()


# ------------------------------------------------------------
# HASH
# ------------------------------------------------------------

def text_hash(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


# ------------------------------------------------------------
# STREAM
# ------------------------------------------------------------

print("=" * 80)
print("TRIBUNALS — LOW-RAM STREAMING PROCESSOR")
print("=" * 80)

print("Opening Hugging Face stream...")

ds = load_dataset(
    REPO,
    data_files=SOURCE_FILE,
    split="train",
    streaming=True,
    token=HF_TOKEN,
)

print("✓ Stream opened successfully")


# ------------------------------------------------------------
# COUNTERS
# ------------------------------------------------------------

processed = 0
valid = 0
quarantined = 0

shard_number = 0
shard_records = 0

shards = []


# ------------------------------------------------------------
# OPEN FILES
# ------------------------------------------------------------

shard_path = (
    OUTPUT_DIR /
    f"shard_{shard_number:05d}.jsonl.gz"
)

shard_file = gzip.open(
    shard_path,
    "wt",
    encoding="utf-8"
)

quarantine_file = gzip.open(
    QUARANTINE_FILE,
    "wt",
    encoding="utf-8"
)


# ============================================================
# STREAM
# ============================================================

for example in ds:

    processed += 1

    try:

        original_text = example.get("text")

        if original_text is None:

            quarantine_file.write(
                json.dumps({
                    "reason": "missing_text",
                    "source_domain": "tribunals",
                    "tribunal": example.get("tribunal"),
                    "court": example.get("court"),
                    "year": example.get("year"),
                    "filepath": example.get("filepath"),
                }, ensure_ascii=False) + "\n"
            )

            quarantined += 1
            continue


        text = normalize_text(original_text)

        if not text:

            quarantine_file.write(
                json.dumps({
                    "reason": "empty_text",
                    "source_domain": "tribunals",
                    "tribunal": example.get("tribunal"),
                    "court": example.get("court"),
                    "year": example.get("year"),
                    "filepath": example.get("filepath"),
                }, ensure_ascii=False) + "\n"
            )

            quarantined += 1
            continue


        # ----------------------------------------------------
        # HASH
        # ----------------------------------------------------

        h = text_hash(text)


        # ----------------------------------------------------
        # DOCUMENT ID
        # ----------------------------------------------------

        identity = (
            "tribunals|"
            + str(example.get("filepath") or "")
            + "|"
            + str(example.get("tribunal") or "")
            + "|"
            + str(example.get("court") or "")
            + "|"
            + str(example.get("year") or "")
            + "|"
            + h
        )

        document_id = hashlib.sha256(
            identity.encode("utf-8")
        ).hexdigest()


        # ----------------------------------------------------
        # RECORD
        # ----------------------------------------------------

        record = {
            "document_id": document_id,
            "text_hash": h,
            "source_domain": "tribunals",

            "tribunal": example.get("tribunal"),
            "court": example.get("court"),
            "year": example.get("year"),
            "filepath": example.get("filepath"),

            "page_count": example.get("page_count"),

            # Use source value.
            # DO NOT call text.split().
            "word_count": example.get("word_count"),

            "char_count": len(text),

            "text": text,
        }


        # ----------------------------------------------------
        # WRITE IMMEDIATELY
        # ----------------------------------------------------

        shard_file.write(
            json.dumps(
                record,
                ensure_ascii=False,
                separators=(",", ":")
            ) + "\n"
        )

        valid += 1
        shard_records += 1


        # ----------------------------------------------------
        # RELEASE REFERENCES
        # ----------------------------------------------------

        del record
        del text
        del original_text


        # ----------------------------------------------------
        # SHARD
        # ----------------------------------------------------

        if shard_records >= SHARD_SIZE:

            shard_file.close()

            size_mb = (
                shard_path.stat().st_size
                / (1024 ** 2)
            )

            shards.append({
                "shard": shard_path.name,
                "records": shard_records,
                "size_mb": round(size_mb, 2)
            })

            print(
                f"Shard {shard_number:05d} complete | "
                f"Records: {shard_records:,} | "
                f"Processed: {processed:,} | "
                f"Quarantined: {quarantined:,} | "
                f"Size: {size_mb:.1f} MB"
            )

            shard_number += 1
            shard_records = 0

            shard_path = (
                OUTPUT_DIR /
                f"shard_{shard_number:05d}.jsonl.gz"
            )

            shard_file = gzip.open(
                shard_path,
                "wt",
                encoding="utf-8"
            )

            gc.collect()


    except Exception as e:

        quarantine_file.write(
            json.dumps({
                "reason": "processing_error",
                "error": str(e),
                "source_domain": "tribunals",
                "tribunal": example.get("tribunal"),
                "court": example.get("court"),
                "year": example.get("year"),
                "filepath": example.get("filepath"),
            }, ensure_ascii=False) + "\n"
        )

        quarantined += 1


    # --------------------------------------------------------
    # PROGRESS
    # --------------------------------------------------------

    if processed % 10_000 == 0:

        print(
            f"Progress: "
            f"{processed:,} processed | "
            f"{valid:,} valid | "
            f"{quarantined:,} quarantined"
        )

        gc.collect()


# ============================================================
# CLOSE
# ============================================================

shard_file.close()
quarantine_file.close()


if shard_records > 0:

    size_mb = (
        shard_path.stat().st_size
        / (1024 ** 2)
    )

    shards.append({
        "shard": shard_path.name,
        "records": shard_records,
        "size_mb": round(size_mb, 2)
    })


# ============================================================
# MANIFEST
# ============================================================

manifest = {
    "dataset": REPO,
    "phase": "phase1",
    "source_domain": "tribunals",
    "source_file": SOURCE_FILE,

    "total_records_seen": processed,
    "processed_records": valid,
    "quarantined_records": quarantined,

    "shard_size_records": SHARD_SIZE,
    "shards": shards,

    "normalization": [
        "Unicode NFC",
        "CRLF to LF",
        "CR to LF",
        "null character removal",
        "space/tab normalization",
        "excessive blank-line normalization",
        "strip leading/trailing whitespace"
    ]
}

with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


gc.collect()


print()
print("=" * 80)
print("TRIBUNALS PROCESSING COMPLETE")
print("=" * 80)

print(f"Total processed : {processed:,}")
print(f"Valid           : {valid:,}")
print(f"Quarantined     : {quarantined:,}")
print(f"Shards          : {len(shards):,}")
print(f"Manifest        : {MANIFEST_FILE}")

In [ ]:
# ============================================================
# TRIBUNALS — DIRECT HUGGING FACE CONNECTION TEST
# NO datasets
# NO torch
# ============================================================

import requests

REPO = "Exploration-Lab/CS-781-Capstone"

URL = (
    "https://huggingface.co/datasets/"
    f"{REPO}/resolve/main/"
    "Phase_1_Pretraining/Train/train_tribunals.json"
)

headers = {
    "Authorization": f"Bearer {HF_TOKEN}"
}

print("Connecting directly to Hugging Face...")
print("No datasets / PyTorch involved.")

r = requests.get(
    URL,
    headers=headers,
    stream=True,
    timeout=(30, 60),
)

print("HTTP status:", r.status_code)
print("Content-Type:", r.headers.get("content-type"))
print("Content-Length:", r.headers.get("content-length"))
print("Transfer-Encoding:", r.headers.get("transfer-encoding"))

r.raise_for_status()

# Read ONLY the first 4 KB
first_bytes = next(r.iter_content(chunk_size=4096))

print("\nFirst bytes received:", len(first_bytes))
print(first_bytes[:1000])

In [ ]:
# ============================================================
# TRIBUNALS — DIRECT HTTP JSON STREAM TEST
# NO datasets
# NO torch
# NO full-file download
# ============================================================

import requests
import ijson

REPO = "Exploration-Lab/CS-781-Capstone"

URL = (
    "https://huggingface.co/datasets/"
    f"{REPO}/resolve/main/"
    "Phase_1_Pretraining/Train/train_tribunals.json"
)

headers = {
    "Authorization": f"Bearer {HF_TOKEN}"
}

print("Opening direct Hugging Face stream...")

response = requests.get(
    URL,
    headers=headers,
    stream=True,
    timeout=(30, 120),
)

response.raise_for_status()

print("✓ HTTP stream opened")
print("Content-Length:",
      int(response.headers.get("content-length", 0)) / (1024**3),
      "GB")

count = 0

for example in ijson.items(
    response.raw,
    "item"
):

    count += 1

    text = example.get("text", "")

    print(
        f"Record {count}: "
        f"{len(text):,} characters | "
        f"court={example.get('court')} | "
        f"year={example.get('year')}"
    )

    if count >= 10:
        break

response.close()

print("\n✓ Successfully parsed 10 records")

In [ ]:
# ============================================================
# PHASE 1 — TRIBUNALS
# DIRECT HTTP STREAMING PROCESSOR
#
# IMPORTANT:
# - Does NOT use HuggingFace datasets
# - Does NOT import torch
# - Does NOT download the 6.48 GB file
# - Processes one JSON object at a time
# - Writes compressed 50k-record shards
# ============================================================

import requests
import ijson
import json
import gzip
import hashlib
import unicodedata
import re
import gc
from pathlib import Path


# ============================================================
# CONFIG
# ============================================================

REPO = "Exploration-Lab/CS-781-Capstone"

SOURCE_FILE = (
    "Phase_1_Pretraining/Train/train_tribunals.json"
)

URL = (
    "https://huggingface.co/datasets/"
    f"{REPO}/resolve/main/"
    f"{SOURCE_FILE}"
)

OUTPUT_DIR = Path(
    "/kaggle/working/phase1_processed/tribunals"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

QUARANTINE_FILE = (
    OUTPUT_DIR / "quarantine.jsonl.gz"
)

MANIFEST_FILE = (
    OUTPUT_DIR / "manifest.json"
)

SHARD_SIZE = 50_000


# ============================================================
# NORMALIZATION
# ============================================================

def normalize_text(text):

    if text is None:
        return ""

    text = str(text)

    text = unicodedata.normalize(
        "NFC",
        text
    )

    text = text.replace(
        "\r\n",
        "\n"
    )

    text = text.replace(
        "\r",
        "\n"
    )

    text = text.replace(
        "\x00",
        ""
    )

    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )

    text = re.sub(
        r"\n[ \t]*\n[ \t]*\n+",
        "\n\n",
        text
    )

    return text.strip()


# ============================================================
# HASH
# ============================================================

def text_hash(text):

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


# ============================================================
# OPEN HTTP STREAM
# ============================================================

print("=" * 80)
print("TRIBUNALS — DIRECT HTTP STREAMING")
print("=" * 80)

print()
print("Opening Hugging Face stream...")
print(URL)
print()

headers = {
    "Authorization": f"Bearer {HF_TOKEN}"
}

response = requests.get(
    URL,
    headers=headers,
    stream=True,
    timeout=(30, 120)
)

response.raise_for_status()

content_length = response.headers.get(
    "content-length"
)

if content_length:

    print(
        "Source size:",
        f"{int(content_length) / (1024**3):.2f} GB"
    )

print("✓ HTTP stream opened")
print("✓ No datasets library")
print("✓ No PyTorch")
print("✓ No raw intermediate copy")
print()


# ============================================================
# COUNTERS
# ============================================================

processed = 0
valid = 0
quarantined = 0

shard_number = 0
shard_records = 0

shards = []


# ============================================================
# OPEN FIRST SHARD
# ============================================================

shard_path = (
    OUTPUT_DIR /
    f"shard_{shard_number:05d}.jsonl.gz"
)

shard_file = gzip.open(
    shard_path,
    "wt",
    encoding="utf-8"
)

quarantine_file = gzip.open(
    QUARANTINE_FILE,
    "wt",
    encoding="utf-8"
)


# ============================================================
# STREAM JSON ARRAY
# ============================================================

print("Starting document processing...")
print()


try:

    for example in ijson.items(
        response.raw,
        "item"
    ):

        processed += 1

        try:

            # ------------------------------------------------
            # ORIGINAL TEXT
            # ------------------------------------------------

            original_text = example.get(
                "text"
            )

            if original_text is None:

                quarantine_file.write(
                    json.dumps(
                        {
                            "reason": "missing_text",
                            "source_domain": "tribunals",
                            "court": example.get("court"),
                            "year": example.get("year"),
                            "filepath": example.get("filepath"),
                        },
                        ensure_ascii=False
                    ) + "\n"
                )

                quarantined += 1

                continue


            # ------------------------------------------------
            # NORMALIZE
            # ------------------------------------------------

            text = normalize_text(
                original_text
            )

            if not text:

                quarantine_file.write(
                    json.dumps(
                        {
                            "reason": "empty_text",
                            "source_domain": "tribunals",
                            "court": example.get("court"),
                            "year": example.get("year"),
                            "filepath": example.get("filepath"),
                        },
                        ensure_ascii=False
                    ) + "\n"
                )

                quarantined += 1

                continue


            # ------------------------------------------------
            # TEXT HASH
            # ------------------------------------------------

            h = text_hash(text)


            # ------------------------------------------------
            # DOCUMENT ID
            # ------------------------------------------------

            identity = (
                "tribunals|"
                + str(
                    example.get("filepath") or ""
                )
                + "|"
                + str(
                    example.get("court") or ""
                )
                + "|"
                + str(
                    example.get("year") or ""
                )
                + "|"
                + h
            )

            document_id = hashlib.sha256(
                identity.encode("utf-8")
            ).hexdigest()


            # ------------------------------------------------
            # OUTPUT RECORD
            # ------------------------------------------------

            record = {
                "document_id": document_id,
                "text_hash": h,

                "source_domain": "tribunals",

                "court": example.get("court"),
                "year": example.get("year"),
                "filepath": example.get("filepath"),

                "page_count": example.get(
                    "page_count"
                ),

                "word_count": example.get(
                    "word_count"
                ),

                "char_count": len(text),

                "text": text,
            }


            # ------------------------------------------------
            # WRITE IMMEDIATELY
            # ------------------------------------------------

            shard_file.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                    separators=(",", ":")
                ) + "\n"
            )

            valid += 1
            shard_records += 1


            # ------------------------------------------------
            # RELEASE REFERENCES
            # ------------------------------------------------

            del record
            del text
            del original_text
            del example


            # ------------------------------------------------
            # SHARD COMPLETE
            # ------------------------------------------------

            if shard_records >= SHARD_SIZE:

                shard_file.close()

                size_mb = (
                    shard_path.stat().st_size
                    / (1024 ** 2)
                )

                shards.append(
                    {
                        "shard": shard_path.name,
                        "records": shard_records,
                        "size_mb": round(
                            size_mb,
                            2
                        )
                    }
                )

                print(
                    f"Shard "
                    f"{shard_number:05d} complete | "
                    f"Records: "
                    f"{shard_records:,} | "
                    f"Processed: "
                    f"{processed:,} | "
                    f"Quarantined: "
                    f"{quarantined:,} | "
                    f"Shard size: "
                    f"{size_mb:.1f} MB"
                )

                shard_number += 1
                shard_records = 0

                shard_path = (
                    OUTPUT_DIR /
                    f"shard_{shard_number:05d}.jsonl.gz"
                )

                shard_file = gzip.open(
                    shard_path,
                    "wt",
                    encoding="utf-8"
                )

                gc.collect()


        except Exception as e:

            quarantine_file.write(
                json.dumps(
                    {
                        "reason": "processing_error",
                        "error": str(e),
                        "source_domain": "tribunals",
                        "court": example.get("court"),
                        "year": example.get("year"),
                        "filepath": example.get("filepath"),
                    },
                    ensure_ascii=False
                ) + "\n"
            )

            quarantined += 1


        # ----------------------------------------------------
        # PROGRESS
        # ----------------------------------------------------

        if processed % 10_000 == 0:

            print(
                f"Progress: "
                f"{processed:,} processed | "
                f"{valid:,} valid | "
                f"{quarantined:,} quarantined"
            )

            gc.collect()


finally:

    # --------------------------------------------------------
    # CLOSE STREAM
    # --------------------------------------------------------

    response.close()

    shard_file.close()
    quarantine_file.close()


# ============================================================
# FINAL PARTIAL SHARD
# ============================================================

if shard_records > 0:

    size_mb = (
        shard_path.stat().st_size
        / (1024 ** 2)
    )

    shards.append(
        {
            "shard": shard_path.name,
            "records": shard_records,
            "size_mb": round(
                size_mb,
                2
            )
        }
    )

    print(
        f"Final shard "
        f"{shard_number:05d} complete | "
        f"Records: "
        f"{shard_records:,} | "
        f"Size: "
        f"{size_mb:.1f} MB"
    )


# ============================================================
# MANIFEST
# ============================================================

manifest = {

    "dataset": REPO,

    "phase": "phase1",

    "source_domain": "tribunals",

    "source_file": SOURCE_FILE,

    "source_url": URL,

    "total_records_seen": processed,

    "valid_records": valid,

    "quarantined_records": quarantined,

    "shard_size_records": SHARD_SIZE,

    "shards": shards,

    "normalization": [
        "Unicode NFC",
        "CRLF to LF",
        "CR to LF",
        "null character removal",
        "space/tab normalization",
        "excessive blank-line normalization",
        "strip leading/trailing whitespace"
    ]
}


with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


gc.collect()


# ============================================================
# FINAL REPORT
# ============================================================

print()
print("=" * 80)
print("TRIBUNALS PROCESSING COMPLETE")
print("=" * 80)

print(
    f"Total processed : {processed:,}"
)

print(
    f"Valid           : {valid:,}"
)

print(
    f"Quarantined     : {quarantined:,}"
)

print(
    f"Shards          : {len(shards):,}"
)

print(
    f"Manifest        : {MANIFEST_FILE}"
)

print()
print("Raw Hugging Face source was NOT downloaded to disk.")
print("Existing Acts / High Courts / Supreme Court outputs were NOT touched.")

In [ ]:
# ============================================================
# STEP 1 — SHARD / MANIFEST INTEGRITY CHECK
# VERSION 3 — HANDLES COMPRESSED SHARDS
# RAM-SAFE
# ============================================================

import os
import json
import glob
import gzip
import bz2
import lzma

BASE = "/kaggle/working/phase1_processed"

DOMAINS = [
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals",
]

print("=" * 80)
print("PHASE 1 — SHARD / MANIFEST INTEGRITY CHECK")
print("=" * 80)


def detect_compression(path):

    with open(path, "rb") as f:
        magic = f.read(8)

    # gzip
    if magic[:2] == b"\x1f\x8b":
        return "gzip"

    # bzip2
    if magic[:3] == b"BZh":
        return "bz2"

    # xz / lzma
    if magic[:6] == b"\xfd7zXZ\x00":
        return "xz"

    return "plain"


def open_text(path):

    compression = detect_compression(path)

    if compression == "gzip":
        return gzip.open(
            path,
            "rt",
            encoding="utf-8"
        )

    if compression == "bz2":
        return bz2.open(
            path,
            "rt",
            encoding="utf-8"
        )

    if compression == "xz":
        return lzma.open(
            path,
            "rt",
            encoding="utf-8"
        )

    return open(
        path,
        "r",
        encoding="utf-8"
    )


def discover_files(domain_dir):

    patterns = [
        "*.jsonl",
        "*.json",
        "*.ndjson",
        "*.jsonl.gz",
        "*.json.gz",
        "*.ndjson.gz",
        "*.gz",
        "*.bz2",
        "*.xz",
        "shard_*",
        "shard-*",
    ]

    files = []

    for pattern in patterns:

        files.extend(
            glob.glob(
                os.path.join(
                    domain_dir,
                    pattern
                )
            )
        )

    return sorted(set(
        p for p in files
        if os.path.isfile(p)
    ))


grand_total_records = 0
grand_total_bytes = 0
all_good = True


for domain in DOMAINS:

    print("\n" + "-" * 80)
    print(f"DOMAIN: {domain.upper()}")
    print("-" * 80)

    domain_dir = os.path.join(
        BASE,
        domain
    )

    manifest_path = os.path.join(
        domain_dir,
        "manifest.json"
    )

    if not os.path.isdir(domain_dir):

        print("❌ Domain directory missing")
        all_good = False
        continue

    # --------------------------------------------------------
    # Load manifest if available
    # --------------------------------------------------------

    manifest = {}

    if os.path.exists(manifest_path):

        try:

            with open(
                manifest_path,
                "r",
                encoding="utf-8"
            ) as f:

                manifest = json.load(f)

            print("✓ manifest.json readable")

        except Exception as e:

            print(
                f"⚠ Could not read manifest: {e}"
            )

    else:

        print(
            "ℹ No manifest.json — "
            "will inspect directory directly"
        )

    # --------------------------------------------------------
    # Find actual files
    # --------------------------------------------------------

    files = discover_files(domain_dir)

    print(
        f"Files found: {len(files)}"
    )

    if not files:

        print("❌ No data files found")
        all_good = False
        continue

    # --------------------------------------------------------
    # Print file types first
    # --------------------------------------------------------

    print("\nFiles:")

    for path in files:

        size_mb = (
            os.path.getsize(path)
            / (1024 ** 2)
        )

        compression = detect_compression(path)

        print(
            f"  {os.path.basename(path)}"
            f" | {size_mb:.1f} MB"
            f" | {compression}"
        )

    # --------------------------------------------------------
    # Process files
    # --------------------------------------------------------

    domain_records = 0
    domain_bytes = 0
    malformed = 0
    empty_lines = 0

    print("\nValidating records...")

    for path in files:

        size_bytes = os.path.getsize(path)

        domain_bytes += size_bytes

        compression = detect_compression(path)

        shard_records = 0

        # ----------------------------------------------------
        # Skip manifest itself
        # ----------------------------------------------------

        if os.path.basename(path) == "manifest.json":
            continue

        try:

            with open_text(path) as f:

                for line_number, line in enumerate(
                    f,
                    start=1
                ):

                    line = line.strip()

                    if not line:

                        empty_lines += 1
                        continue

                    try:

                        obj = json.loads(line)

                        if not isinstance(
                            obj,
                            dict
                        ):

                            malformed += 1

                        else:

                            shard_records += 1

                    except Exception:

                        malformed += 1

        except Exception as e:

            print(
                f"\n❌ Could not read:"
                f" {os.path.basename(path)}"
            )

            print(
                f"   Error: {e}"
            )

            all_good = False
            continue

        domain_records += shard_records

        print(
            f"  ✓ {os.path.basename(path)}"
            f" → {shard_records:,} records"
        )

    # --------------------------------------------------------
    # Domain summary
    # --------------------------------------------------------

    grand_total_records += domain_records
    grand_total_bytes += domain_bytes

    print("\nDomain summary:")

    print(
        f"  Records: "
        f"{domain_records:,}"
    )

    print(
        f"  Storage: "
        f"{domain_bytes / (1024 ** 3):.3f} GB"
    )

    print(
        f"  Empty lines: "
        f"{empty_lines:,}"
    )

    print(
        f"  Malformed records: "
        f"{malformed:,}"
    )

    # --------------------------------------------------------
    # Manifest comparison
    # --------------------------------------------------------

    manifest_records = None

    for key in [
        "processed_records",
        "valid_records",
        "total_records",
        "total_records_seen",
    ]:

        value = manifest.get(key)

        if isinstance(value, int):

            # Prefer processed/valid over total seen
            if key in [
                "processed_records",
                "valid_records"
            ]:

                manifest_records = value
                break

            if manifest_records is None:
                manifest_records = value

    if manifest_records is not None:

        print(
            f"  Manifest count: "
            f"{manifest_records:,}"
        )

        if domain_records == manifest_records:

            print(
                "  ✓ Record count matches"
            )

        else:

            print(
                "  ⚠ Record count differs"
            )

            # Don't immediately fail because the manifest
            # may describe logical shards differently.


# ============================================================
# GLOBAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("GLOBAL SUMMARY")
print("=" * 80)

print(
    f"Total records found : "
    f"{grand_total_records:,}"
)

print(
    f"Total storage       : "
    f"{grand_total_bytes / (1024 ** 3):.3f} GB"
)

print()

if all_good:

    print("✓ STEP 1 COMPLETED")

else:

    print(
        "⚠ STEP 1 COMPLETED WITH FILE READ ISSUES"
    )

print("=" * 80)

In [ ]:
# ============================================================
# PHASE 1 — STEP 2
# QUARANTINE AUDIT
# ============================================================

import os
import json
import gzip
from collections import Counter

BASE = "/kaggle/working/phase1_processed"

DOMAINS = [
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals",
]

print("=" * 80)
print("PHASE 1 — QUARANTINE AUDIT")
print("=" * 80)

global_counts = Counter()

for domain in DOMAINS:

    q_path = os.path.join(BASE, domain, "quarantine.jsonl")
    q_gz_path = os.path.join(BASE, domain, "quarantine.jsonl.gz")

    path = None
    opener = None

    if os.path.exists(q_path):
        path = q_path
        opener = open
    elif os.path.exists(q_gz_path):
        path = q_gz_path
        opener = gzip.open

    print(f"\n{'-' * 80}")
    print(f"DOMAIN: {domain.upper()}")

    if path is None:
        print("✓ No quarantine file")
        continue

    counts = Counter()
    total = 0
    malformed = 0

    with opener(path, "rt", encoding="utf-8") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
            except Exception:
                malformed += 1
                continue

            reason = record.get("reason", "unknown")
            counts[reason] += 1
            global_counts[reason] += 1
            total += 1

    print(f"Quarantined records : {total}")
    print(f"Malformed entries   : {malformed}")

    if counts:
        for reason, count in counts.most_common():
            print(f"  {reason}: {count}")
    else:
        print("  None")

print("\n" + "=" * 80)
print("GLOBAL QUARANTINE SUMMARY")
print("=" * 80)

total_quarantine = sum(global_counts.values())

print(f"Total quarantined : {total_quarantine}")

for reason, count in global_counts.most_common():
    print(f"{reason}: {count}")

print("=" * 80)
print("✓ STEP 2 COMPLETED")

In [ ]:
# ============================================================
# PHASE 1 — STEP 3
# SCHEMA / FIELD CONSISTENCY CHECK
# ============================================================

import os
import json
import gzip

BASE = "/kaggle/working/phase1_processed"

DOMAINS = [
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals",
]

EXPECTED_FIELDS = {
    "id",
    "source_domain",
    "title",
    "category",
    "year",
    "text",
    "word_count",
    "char_count",
}

def open_jsonl(path):
    if path.endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8")
    return open(path, "rt", encoding="utf-8")

print("=" * 80)
print("PHASE 1 — SCHEMA / FIELD CONSISTENCY")
print("=" * 80)

for domain in DOMAINS:

    directory = os.path.join(BASE, domain)

    shard_files = sorted(
        os.path.join(directory, f)
        for f in os.listdir(directory)
        if f.startswith("shard_")
        and (f.endswith(".jsonl") or f.endswith(".jsonl.gz"))
    )

    # Acts has documents.jsonl instead of shards
    if domain == "acts":
        shard_files = [
            os.path.join(directory, "documents.jsonl")
        ]

    print(f"\n{'-' * 80}")
    print(f"DOMAIN: {domain.upper()}")
    print(f"Files checked: {len(shard_files)}")

    records_checked = 0
    schema_variants = {}
    missing_fields = {}
    extra_fields = {}

    # Safety: inspect first 100 records per file
    for path in shard_files:

        with open_jsonl(path) as f:

            for line in f:

                if records_checked >= 100:
                    break

                line = line.strip()

                if not line:
                    continue

                try:
                    record = json.loads(line)
                except Exception:
                    continue

                fields = tuple(sorted(record.keys()))
                schema_variants[fields] = schema_variants.get(fields, 0) + 1

                missing = EXPECTED_FIELDS - set(record.keys())
                extra = set(record.keys()) - EXPECTED_FIELDS

                if missing:
                    missing_fields.setdefault(
                        tuple(sorted(missing)), 0
                    )
                    missing_fields[tuple(sorted(missing))] += 1

                if extra:
                    extra_fields.setdefault(
                        tuple(sorted(extra)), 0
                    )
                    extra_fields[tuple(sorted(extra))] += 1

                records_checked += 1

            if records_checked >= 100:
                break

    print(f"Records sampled: {records_checked}")

    print("\nSchema variants:")
    for fields, count in schema_variants.items():
        print(f"  {count} records:")
        print(f"    {fields}")

    if missing_fields:
        print("\n⚠ Missing fields:")
        for fields, count in missing_fields.items():
            print(f"  {count}: {fields}")
    else:
        print("\n✓ No expected fields missing in sample")

    if extra_fields:
        print("\nℹ Extra fields:")
        for fields, count in extra_fields.items():
            print(f"  {count}: {fields}")
    else:
        print("✓ No unexpected fields in sample")

print("\n" + "=" * 80)
print("✓ STEP 3 COMPLETED")

In [ ]:
# ============================================================
# PHASE 1 — STEP 4
# TEXT QUALITY SAMPLE
# ============================================================

import os
import json
import gzip
import random
import re

BASE = "/kaggle/working/phase1_processed"

DOMAINS = [
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals",
]

SAMPLES_PER_DOMAIN = 100

def open_jsonl(path):
    if path.endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8")
    return open(path, "rt", encoding="utf-8")

print("=" * 80)
print("PHASE 1 — TEXT QUALITY SAMPLE")
print("=" * 80)

for domain in DOMAINS:

    directory = os.path.join(BASE, domain)

    if domain == "acts":
        files = [
            os.path.join(directory, "documents.jsonl")
        ]
    else:
        files = sorted(
            os.path.join(directory, f)
            for f in os.listdir(directory)
            if f.startswith("shard_")
            and (f.endswith(".jsonl") or f.endswith(".jsonl.gz"))
        )

    print(f"\n{'-' * 80}")
    print(f"DOMAIN: {domain.upper()}")

    sampled = 0
    empty = 0
    very_short = 0
    suspicious = 0
    word_mismatches = 0

    # Deterministic sampling across files
    for path in files:

        if sampled >= SAMPLES_PER_DOMAIN:
            break

        with open_jsonl(path) as f:

            for line in f:

                if sampled >= SAMPLES_PER_DOMAIN:
                    break

                line = line.strip()

                if not line:
                    continue

                try:
                    record = json.loads(line)
                except Exception:
                    continue

                text = str(record.get("text", ""))
                declared_words = record.get("word_count")
                actual_words = len(text.split())

                sampled += 1

                if not text.strip():
                    empty += 1

                if actual_words < 20:
                    very_short += 1

                # Basic corruption indicators
                if (
                    "\x00" in text
                    or
                    text.count("�") > 5
                    or
                    re.search(r"(.)\1{20,}", text)
                ):
                    suspicious += 1

                if (
                    isinstance(declared_words, int)
                    and declared_words != actual_words
                ):
                    word_mismatches += 1

    print(f"Sampled records       : {sampled}")
    print(f"Empty text            : {empty}")
    print(f"<20 actual words     : {very_short}")
    print(f"Suspicious text       : {suspicious}")
    print(f"Word-count mismatches : {word_mismatches}")

print("\n" + "=" * 80)
print("✓ STEP 4 COMPLETED")

In [ ]:
# ============================================================
# PHASE 1 — STEP 5
# DUPLICATE ID SAMPLE
# ============================================================

import os
import json
import gzip
from collections import Counter

BASE = "/kaggle/working/phase1_processed"

DOMAINS = [
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals",
]

MAX_RECORDS_PER_DOMAIN = 100_000

def open_jsonl(path):
    if path.endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8")
    return open(path, "rt", encoding="utf-8")

print("=" * 80)
print("PHASE 1 — DUPLICATE ID CHECK")
print("=" * 80)

for domain in DOMAINS:

    directory = os.path.join(BASE, domain)

    if domain == "acts":
        files = [os.path.join(directory, "documents.jsonl")]
    else:
        files = sorted(
            os.path.join(directory, f)
            for f in os.listdir(directory)
            if f.startswith("shard_")
            and (f.endswith(".jsonl") or f.endswith(".jsonl.gz"))
        )

    seen = set()
    duplicate_ids = 0
    checked = 0

    for path in files:

        with open_jsonl(path) as f:

            for line in f:

                if checked >= MAX_RECORDS_PER_DOMAIN:
                    break

                line = line.strip()

                if not line:
                    continue

                try:
                    record = json.loads(line)
                except Exception:
                    continue

                record_id = record.get("id")

                if record_id is not None:
                    if record_id in seen:
                        duplicate_ids += 1
                    else:
                        seen.add(record_id)

                checked += 1

        if checked >= MAX_RECORDS_PER_DOMAIN:
            break

    print(f"\n{domain.upper()}")
    print(f"Records checked : {checked}")
    print(f"Unique IDs      : {len(seen)}")
    print(f"Duplicate IDs   : {duplicate_ids}")

print("\n" + "=" * 80)
print("✓ STEP 5 COMPLETED")

In [ ]:
# ============================================================
# PHASE 1 — STEP 6
# METADATA SANITY CHECK
# ============================================================

import os
import json
import gzip
from collections import Counter

BASE = "/kaggle/working/phase1_processed"

DOMAINS = [
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals",
]

SAMPLE_PER_DOMAIN = 1000

def open_jsonl(path):
    if path.endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8")
    return open(path, "rt", encoding="utf-8")

print("=" * 80)
print("PHASE 1 — METADATA SANITY CHECK")
print("=" * 80)

for domain in DOMAINS:

    directory = os.path.join(BASE, domain)

    if domain == "acts":
        files = [os.path.join(directory, "documents.jsonl")]
    else:
        files = sorted(
            os.path.join(directory, f)
            for f in os.listdir(directory)
            if f.startswith("shard_")
            and (f.endswith(".jsonl") or f.endswith(".jsonl.gz"))
        )

    years = Counter()
    missing_title = 0
    missing_category = 0
    missing_year = 0
    bad_word_count = 0
    bad_char_count = 0

    checked = 0

    for path in files:

        with open_jsonl(path) as f:

            for line in f:

                if checked >= SAMPLE_PER_DOMAIN:
                    break

                line = line.strip()

                if not line:
                    continue

                try:
                    r = json.loads(line)
                except Exception:
                    continue

                checked += 1

                if not str(r.get("title", "")).strip():
                    missing_title += 1

                if not str(r.get("category", "")).strip():
                    missing_category += 1

                year = r.get("year")

                if year is None or str(year).strip() == "":
                    missing_year += 1
                else:
                    years[str(year)] += 1

                if not isinstance(r.get("word_count"), int):
                    bad_word_count += 1

                if not isinstance(r.get("char_count"), int):
                    bad_char_count += 1

        if checked >= SAMPLE_PER_DOMAIN:
            break

    print(f"\n{domain.upper()}")
    print(f"Sampled records       : {checked}")
    print(f"Missing title         : {missing_title}")
    print(f"Missing category      : {missing_category}")
    print(f"Missing year          : {missing_year}")
    print(f"Bad word_count type   : {bad_word_count}")
    print(f"Bad char_count type   : {bad_char_count}")

    print("Year examples:")
    for year, count in years.most_common(10):
        print(f"  {year}: {count}")

print("\n" + "=" * 80)
print("✓ STEP 6 COMPLETED")

In [ ]:
# ============================================================
# PHASE 1 — STEP 7
# CROSS-DOMAIN ID COLLISION CHECK
# ============================================================

import os
import json
import gzip
from collections import defaultdict

BASE = "/kaggle/working/phase1_processed"

DOMAINS = [
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals",
]

SAMPLE_PER_DOMAIN = 100_000

def open_jsonl(path):
    if path.endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8")
    return open(path, "rt", encoding="utf-8")

owners = defaultdict(set)

print("=" * 80)
print("PHASE 1 — CROSS-DOMAIN ID COLLISION CHECK")
print("=" * 80)

for domain in DOMAINS:

    directory = os.path.join(BASE, domain)

    if domain == "acts":
        files = [os.path.join(directory, "documents.jsonl")]
    else:
        files = sorted(
            os.path.join(directory, f)
            for f in os.listdir(directory)
            if f.startswith("shard_")
            and (f.endswith(".jsonl") or f.endswith(".jsonl.gz"))
        )

    checked = 0

    for path in files:

        with open_jsonl(path) as f:

            for line in f:

                if checked >= SAMPLE_PER_DOMAIN:
                    break

                line = line.strip()

                if not line:
                    continue

                try:
                    record = json.loads(line)
                except Exception:
                    continue

                record_id = record.get("id")

                if record_id:
                    owners[record_id].add(domain)

                checked += 1

        if checked >= SAMPLE_PER_DOMAIN:
            break

    print(f"{domain.upper()}: checked {checked}")

collisions = {
    record_id: domains
    for record_id, domains in owners.items()
    if len(domains) > 1
}

print("\nCross-domain collisions:", len(collisions))

if collisions:
    for record_id, domains in list(collisions.items())[:20]:
        print(record_id, domains)
else:
    print("✓ No cross-domain ID collisions in sample")

print("\n" + "=" * 80)
print("✓ STEP 7 COMPLETED")

In [ ]:
# ============================================================
# PHASE 1 — STEP 8
# FINAL READINESS CHECK
# ============================================================

import os
import json

BASE = "/kaggle/working/phase1_processed"

DOMAINS = [
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals",
]

print("=" * 80)
print("PHASE 1 — FINAL READINESS CHECK")
print("=" * 80)

expected = {
    "acts": 18732,
    "high_courts": 2536738,
    "supreme_court": 55528,
    "tribunals": 869132,
}

total = 0
all_good = True

for domain in DOMAINS:

    directory = os.path.join(BASE, domain)

    exists = os.path.isdir(directory)

    if not exists:
        print(f"❌ {domain}: directory missing")
        all_good = False
        continue

    manifest_path = os.path.join(directory, "manifest.json")

    if domain == "acts":
        data_path = os.path.join(directory, "documents.jsonl")
        exists_data = os.path.exists(data_path)

        print(f"\n{domain.upper()}")
        print(f"  Directory : ✓")
        print(f"  Documents: {'✓' if exists_data else '❌'}")
        print(f"  Expected valid records: {expected[domain]}")

        if not exists_data:
            all_good = False

    else:
        shard_count = len([
            f for f in os.listdir(directory)
            if f.startswith("shard_") and f.endswith(".jsonl.gz")
        ])

        print(f"\n{domain.upper()}")
        print(f"  Directory : ✓")
        print(f"  Manifest  : {'✓' if os.path.exists(manifest_path) else '❌'}")
        print(f"  Shards    : {shard_count}")
        print(f"  Expected valid records: {expected[domain]}")

        if not os.path.exists(manifest_path) or shard_count == 0:
            all_good = False

    total += expected[domain]

print("\n" + "=" * 80)
print(f"EXPECTED VALID RECORDS : {total:,}")
print(f"PROCESSED STORAGE       : approximately 5.56 GB")
print("=" * 80)

if all_good:
    print("✓ PHASE 1 OUTPUT STRUCTURE IS READY")
else:
    print("⚠ PHASE 1 OUTPUT STRUCTURE HAS ISSUES")

In [ ]:
# ============================================================
# PHASE 1 — TARGETED QUALITY INVESTIGATION
# READ-ONLY — NO FILES MODIFIED
# ============================================================

import os
import json
import gzip
import glob
from collections import defaultdict, Counter

BASE = "/kaggle/working/phase1_processed"

DOMAINS = {
    "acts": os.path.join(BASE, "acts"),
    "high_courts": os.path.join(BASE, "high_courts"),
    "supreme_court": os.path.join(BASE, "supreme_court"),
    "tribunals": os.path.join(BASE, "tribunals"),
}

SAMPLE_LIMIT = 5


def iter_records(domain):
    directory = DOMAINS[domain]

    files = sorted(
        glob.glob(os.path.join(directory, "*.jsonl")) +
        glob.glob(os.path.join(directory, "*.jsonl.gz"))
    )

    for filepath in files:
        if os.path.basename(filepath) in {
            "quarantine.jsonl",
            "quarantine.jsonl.gz",
        }:
            continue

        opener = gzip.open if filepath.endswith(".gz") else open

        with opener(filepath, "rt", encoding="utf-8", errors="replace") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue

                try:
                    yield json.loads(line)
                except Exception:
                    continue


def word_count(text):
    return len(text.split())


def preview(text, n=500):
    text = str(text).replace("\n", " ")
    return text[:n]


# ============================================================
# 1. TRIBUNAL WORD-COUNT MISMATCH INVESTIGATION
# ============================================================

print("=" * 80)
print("1. TRIBUNALS — WORD COUNT MISMATCH EXAMPLES")
print("=" * 80)

count = 0

for record in iter_records("tribunals"):

    text = record.get("text", "")
    stored = record.get("word_count")

    calculated = word_count(text)

    if stored != calculated:

        print("\n--- MISMATCH", count + 1, "---")
        print("Document ID :", record.get("document_id"))
        print("Year        :", record.get("year"))
        print("Court       :", record.get("court"))
        print("Stored count:", repr(stored))
        print("Calc count  :", calculated)
        print("Text length :", len(text))
        print("Text preview:")
        print(preview(text))

        count += 1

        if count >= SAMPLE_LIMIT:
            break

print("\nSamples shown:", count)


# ============================================================
# 2. SUPREME COURT — DUPLICATE DOCUMENT IDs
# ============================================================

print("\n" + "=" * 80)
print("2. SUPREME COURT — DUPLICATE DOCUMENT IDs")
print("=" * 80)

seen = {}
duplicates = []

for record in iter_records("supreme_court"):

    doc_id = record.get("document_id")

    if doc_id is None:
        continue

    if doc_id in seen:

        duplicates.append((doc_id, seen[doc_id], record))

        if len(duplicates) >= SAMPLE_LIMIT:
            break

    else:
        seen[doc_id] = record


for i, (doc_id, first, second) in enumerate(duplicates, 1):

    print(f"\n--- DUPLICATE {i} ---")
    print("Document ID:", doc_id)

    print("\nFIRST:")
    print("Year :", first.get("year"))
    print("Court:", first.get("court"))
    print("File :", first.get("filepath"))
    print("Hash :", first.get("text_hash"))
    print("Text :", preview(first.get("text", "")))

    print("\nSECOND:")
    print("Year :", second.get("year"))
    print("Court:", second.get("court"))
    print("File :", second.get("filepath"))
    print("Hash :", second.get("text_hash"))
    print("Text :", preview(second.get("text", "")))


print("\nDuplicate examples shown:", len(duplicates))


# ============================================================
# 3. TRIBUNALS — DUPLICATE DOCUMENT IDs
# ============================================================

print("\n" + "=" * 80)
print("3. TRIBUNALS — DUPLICATE DOCUMENT IDs")
print("=" * 80)

seen = {}
duplicates = []

for record in iter_records("tribunals"):

    doc_id = record.get("document_id")

    if doc_id is None:
        continue

    if doc_id in seen:

        duplicates.append((doc_id, seen[doc_id], record))

        if len(duplicates) >= SAMPLE_LIMIT:
            break

    else:
        seen[doc_id] = record


for i, (doc_id, first, second) in enumerate(duplicates, 1):

    print(f"\n--- DUPLICATE {i} ---")
    print("Document ID:", doc_id)

    print("\nFIRST:")
    print("Year :", first.get("year"))
    print("Court:", first.get("court"))
    print("File :", first.get("filepath"))
    print("Hash :", first.get("text_hash"))
    print("Text :", preview(first.get("text", "")))

    print("\nSECOND:")
    print("Year :", second.get("year"))
    print("Court:", second.get("court"))
    print("File :", second.get("filepath"))
    print("Hash :", second.get("text_hash"))
    print("Text :", preview(second.get("text", "")))


print("\nDuplicate examples shown:", len(duplicates))


# ============================================================
# 4. HIGH COURTS — DUPLICATE TEXT HASHES
# ============================================================

print("\n" + "=" * 80)
print("4. HIGH COURTS — DUPLICATE TEXT HASH EXAMPLES")
print("=" * 80)

seen_hash = {}
duplicates = []

for record in iter_records("high_courts"):

    h = record.get("text_hash")

    if not h:
        continue

    if h in seen_hash:

        duplicates.append((h, seen_hash[h], record))

        if len(duplicates) >= SAMPLE_LIMIT:
            break

    else:
        seen_hash[h] = record


for i, (h, first, second) in enumerate(duplicates, 1):

    print(f"\n--- DUPLICATE HASH {i} ---")
    print("Hash:", h)

    print("\nFIRST:")
    print("Document ID:", first.get("document_id"))
    print("Court:", first.get("court"))
    print("Year:", first.get("year"))
    print("File:", first.get("filepath"))
    print("Text:", preview(first.get("text", "")))

    print("\nSECOND:")
    print("Document ID:", second.get("document_id"))
    print("Court:", second.get("court"))
    print("Year:", second.get("year"))
    print("File:", second.get("filepath"))
    print("Text:", preview(second.get("text", "")))


print("\nDuplicate hash examples shown:", len(duplicates))


# ============================================================
# 5. ACTS — SUSPICIOUS TEXT EXAMPLES
# ============================================================

print("\n" + "=" * 80)
print("5. ACTS — SUSPICIOUS TEXT EXAMPLES")
print("=" * 80)

shown = 0

for record in iter_records("acts"):

    text = record.get("text", "")

    # Conservative investigation heuristics only.
    suspicious = (
        len(text.strip()) < 100
        or text.count("\ufffd") > 0
        or "\x00" in text
        or text.count("�") > 0
    )

    if suspicious:

        print(f"\n--- SUSPICIOUS {shown + 1} ---")
        print("Document ID:", record.get("document_id"))
        print("Title:", record.get("title"))
        print("Category:", record.get("category"))
        print("Year:", record.get("year"))
        print("Words:", record.get("processed_word_count"))
        print("Chars:", record.get("processed_text_length_chars"))
        print("Text:")
        print(preview(text, 800))

        shown += 1

        if shown >= SAMPLE_LIMIT:
            break

print("\nSuspicious examples shown:", shown)


# ============================================================
# 6. TRIBUNALS — SUSPICIOUS TEXT EXAMPLES
# ============================================================

print("\n" + "=" * 80)
print("6. TRIBUNALS — SUSPICIOUS TEXT EXAMPLES")
print("=" * 80)

shown = 0

for record in iter_records("tribunals"):

    text = record.get("text", "")

    suspicious = (
        len(text.strip()) < 100
        or text.count("\ufffd") > 0
        or "\x00" in text
        or text.count("�") > 0
    )

    if suspicious:

        print(f"\n--- SUSPICIOUS {shown + 1} ---")
        print("Document ID:", record.get("document_id"))
        print("Court:", record.get("court"))
        print("Year:", record.get("year"))
        print("Stored words:", record.get("word_count"))
        print("Calculated words:", word_count(text))
        print("Chars:", len(text))
        print("Text:")
        print(preview(text, 800))

        shown += 1

        if shown >= SAMPLE_LIMIT:
            break

print("\nSuspicious examples shown:", shown)


print("\n" + "=" * 80)
print("INVESTIGATION COMPLETE")
print("NO FILES WERE MODIFIED")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 1 — DOMAIN-AWARE FINAL DATA QUALITY AUDIT
# ============================================================
#
# READ-ONLY:
#   - Does NOT modify documents
#   - Does NOT modify manifests
#   - Does NOT create copies
#   - Streams one record at a time
#
# Checks:
#   1. Domain-specific schema
#   2. document_id uniqueness
#   3. text_hash uniqueness
#   4. Stored vs actual text lengths
#   5. Stored vs actual word counts
#   6. Empty / very-short text
#   7. Missing critical metadata
#   8. Basic text corruption indicators
#   9. Manifest counts
# ============================================================

import os
import json
import gzip
import re
from collections import Counter

BASE = "/kaggle/working/phase1_processed"

DOMAINS = [
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals",
]

# ------------------------------------------------------------
# Domain-specific field definitions
# ------------------------------------------------------------

DOMAIN_CONFIG = {

    "acts": {
        "files": ["documents.jsonl"],
        "id_field": "document_id",
        "hash_field": "text_hash",
        "word_field": "processed_word_count",
        "char_field": "processed_text_length_chars",
        "required": [
            "document_id",
            "source_domain",
            "title",
            "category",
            "year",
            "text",
            "text_hash",
            "processed_word_count",
            "processed_text_length_chars",
        ],
    },

    "high_courts": {
        "id_field": "document_id",
        "hash_field": "text_hash",
        "word_field": "processed_word_count",
        "char_field": "processed_char_count",
        "required": [
            "document_id",
            "source_domain",
            "court",
            "court_id",
            "year",
            "text",
            "text_hash",
            "processed_word_count",
            "processed_char_count",
        ],
    },

    "supreme_court": {
        "id_field": "document_id",
        "hash_field": "text_hash",
        "word_field": "processed_word_count",
        "char_field": "processed_char_count",
        "required": [
            "document_id",
            "source_domain",
            "court",
            "year",
            "text",
            "text_hash",
            "processed_word_count",
            "processed_char_count",
        ],
    },

    "tribunals": {
        "id_field": "document_id",
        "hash_field": "text_hash",
        "word_field": "word_count",
        "char_field": "char_count",
        "required": [
            "document_id",
            "source_domain",
            "court",
            "year",
            "text",
            "text_hash",
            "word_count",
            "char_count",
        ],
    },
}


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def open_jsonl(path):

    if path.endswith(".gz"):
        return gzip.open(
            path,
            "rt",
            encoding="utf-8",
            errors="replace"
        )

    return open(
        path,
        "rt",
        encoding="utf-8",
        errors="replace"
    )


def get_files(domain):

    directory = os.path.join(BASE, domain)

    if domain == "acts":
        path = os.path.join(directory, "documents.jsonl")
        return [path] if os.path.exists(path) else []

    return sorted(
        os.path.join(directory, filename)
        for filename in os.listdir(directory)
        if filename.startswith("shard_")
        and (
            filename.endswith(".jsonl")
            or filename.endswith(".jsonl.gz")
        )
    )


# ------------------------------------------------------------
# Global counters
# ------------------------------------------------------------

global_records = 0
global_empty = 0
global_short = 0
global_schema_errors = 0
global_metadata_errors = 0
global_length_mismatches = 0
global_word_mismatches = 0
global_duplicate_ids = 0
global_duplicate_hashes = 0
global_suspicious = 0


print("=" * 80)
print("PHASE 1 — DOMAIN-AWARE FINAL DATA QUALITY AUDIT")
print("=" * 80)


# ============================================================
# DOMAIN LOOP
# ============================================================

for domain in DOMAINS:

    config = DOMAIN_CONFIG[domain]

    id_field = config["id_field"]
    hash_field = config["hash_field"]
    word_field = config["word_field"]
    char_field = config["char_field"]
    required = set(config["required"])

    files = get_files(domain)

    print("\n" + "-" * 80)
    print(f"DOMAIN: {domain.upper()}")
    print("-" * 80)

    print(f"Files: {len(files)}")

    # --------------------------------------------------------
    # In-memory sets contain ONLY IDs / hashes.
    # No document text is retained.
    # --------------------------------------------------------

    seen_ids = set()
    seen_hashes = set()

    records = 0
    empty = 0
    very_short = 0
    schema_errors = 0
    metadata_errors = 0
    length_mismatches = 0
    word_mismatches = 0
    duplicate_ids = 0
    duplicate_hashes = 0
    suspicious = 0

    missing_fields_counter = Counter()

    # --------------------------------------------------------
    # Process files one at a time
    # --------------------------------------------------------

    for file_index, path in enumerate(files, start=1):

        print(
            f"  Reading file {file_index}/{len(files)}: "
            f"{os.path.basename(path)}"
        )

        with open_jsonl(path) as f:

            for line in f:

                line = line.strip()

                if not line:
                    continue

                try:
                    record = json.loads(line)
                except Exception:
                    schema_errors += 1
                    continue

                records += 1
                global_records += 1

                # ------------------------------------------------
                # Schema
                # ------------------------------------------------

                missing = required - set(record.keys())

                if missing:
                    schema_errors += 1

                    for field in missing:
                        missing_fields_counter[field] += 1

                # ------------------------------------------------
                # Text
                # ------------------------------------------------

                text = record.get("text")

                if not isinstance(text, str):
                    text = "" if text is None else str(text)

                if not text.strip():
                    empty += 1
                    global_empty += 1

                actual_chars = len(text)
                actual_words = len(text.split())

                if actual_words < 20:
                    very_short += 1
                    global_short += 1

                # ------------------------------------------------
                # Metadata
                # ------------------------------------------------

                for field in [
                    id_field,
                    hash_field,
                    word_field,
                    char_field,
                ]:

                    if field not in record:
                        metadata_errors += 1
                        global_metadata_errors += 1

                # ------------------------------------------------
                # Document ID uniqueness
                # ------------------------------------------------

                record_id = record.get(id_field)

                if record_id is not None:

                    if record_id in seen_ids:
                        duplicate_ids += 1
                        global_duplicate_ids += 1

                    else:
                        seen_ids.add(record_id)

                # ------------------------------------------------
                # Text hash uniqueness
                # ------------------------------------------------

                text_hash = record.get(hash_field)

                if text_hash is not None:

                    if text_hash in seen_hashes:
                        duplicate_hashes += 1
                        global_duplicate_hashes += 1

                    else:
                        seen_hashes.add(text_hash)

                # ------------------------------------------------
                # Character count consistency
                # ------------------------------------------------

                stored_chars = record.get(char_field)

                if isinstance(stored_chars, int):

                    if stored_chars != actual_chars:

                        length_mismatches += 1
                        global_length_mismatches += 1

                # ------------------------------------------------
                # Word count consistency
                #
                # IMPORTANT:
                # We report this but do NOT quarantine anything.
                # Different preprocessing pipelines can count
                # words differently.
                # ------------------------------------------------

                stored_words = record.get(word_field)

                if isinstance(stored_words, int):

                    if stored_words != actual_words:

                        word_mismatches += 1
                        global_word_mismatches += 1

                # ------------------------------------------------
                # Conservative corruption indicators
                # ------------------------------------------------

                if (
                    "\x00" in text
                    or text.count("\ufffd") > 5
                    or re.search(r"(.)\1{30,}", text)
                ):

                    suspicious += 1
                    global_suspicious += 1

    # --------------------------------------------------------
    # Domain summary
    # --------------------------------------------------------

    print()
    print(f"Records                 : {records:,}")
    print(f"Empty text              : {empty:,}")
    print(f"<20 words               : {very_short:,}")
    print(f"Schema errors           : {schema_errors:,}")
    print(f"Metadata errors         : {metadata_errors:,}")
    print(f"Duplicate document IDs  : {duplicate_ids:,}")
    print(f"Duplicate text hashes   : {duplicate_hashes:,}")
    print(f"Char-count mismatches   : {length_mismatches:,}")
    print(f"Word-count mismatches   : {word_mismatches:,}")
    print(f"Suspicious text         : {suspicious:,}")

    if missing_fields_counter:

        print("\nMissing fields:")

        for field, count in missing_fields_counter.most_common():
            print(f"  {field}: {count}")

    else:
        print("Missing fields          : 0")


# ============================================================
# GLOBAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("GLOBAL DOMAIN-AWARE AUDIT SUMMARY")
print("=" * 80)

print(f"Records examined        : {global_records:,}")
print(f"Empty text              : {global_empty:,}")
print(f"<20 words               : {global_short:,}")
print(f"Schema errors           : {global_schema_errors:,}")
print(f"Metadata errors         : {global_metadata_errors:,}")
print(f"Duplicate document IDs  : {global_duplicate_ids:,}")
print(f"Duplicate text hashes   : {global_duplicate_hashes:,}")
print(f"Char-count mismatches   : {global_length_mismatches:,}")
print(f"Word-count mismatches   : {global_word_mismatches:,}")
print(f"Suspicious text         : {global_suspicious:,}")

print("\n" + "=" * 80)

# ------------------------------------------------------------
# Important: word-count mismatches are informational.
# ------------------------------------------------------------

hard_failures = (
    global_schema_errors
    + global_metadata_errors
    + global_duplicate_ids
    + global_duplicate_hashes
    + global_empty
)

if hard_failures == 0:

    print("✓ NO HARD DATA-INTEGRITY FAILURES FOUND")

else:

    print(
        f"⚠ HARD CHECK FINDINGS: {hard_failures:,}"
    )

print("=" * 80)
print("AUDIT COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 8A — QUICK TRIBUNALS WORD-COUNT DIAGNOSTIC
# READ-ONLY
# ============================================================

import os
import json
import gzip
import re

TRIBUNALS_DIR = "/kaggle/working/phase1_processed/tribunals"

shard_path = os.path.join(
    TRIBUNALS_DIR,
    "shard_00000.jsonl.gz"
)

print("=" * 80)
print("TRIBUNALS — QUICK WORD COUNT DIAGNOSTIC")
print("=" * 80)
print("Reading only first 100 records...")
print()

def whitespace_count(text):
    return len(text.split())

def regex_word_count(text):
    return len(re.findall(r"\b\w+\b", text, flags=re.UNICODE))

with gzip.open(shard_path, "rt", encoding="utf-8") as f:

    for i, line in enumerate(f):

        if i >= 100:
            break

        if not line.strip():
            continue

        obj = json.loads(line)

        text = obj.get("text", "")
        stored = obj.get("word_count")

        ws = whitespace_count(text)
        regex = regex_word_count(text)

        if i < 10:
            print(f"Record {i + 1}")
            print(f"  stored word_count : {stored}")
            print(f"  text.split()      : {ws}")
            print(f"  regex words       : {regex}")
            print(f"  characters        : {len(text)}")
            print(f"  preview           : {text[:150].replace(chr(10), ' ')}")
            print("-" * 80)

print()
print("Quick diagnostic complete.")

In [ ]:
# ============================================================
# STEP 8B — IDENTIFY TRIBUNAL WORD-COUNT CONVENTION
# READ-ONLY — 100 RECORDS
# ============================================================

import os
import json
import gzip
import re

TRIBUNALS_DIR = "/kaggle/working/phase1_processed/tribunals"
shard_path = os.path.join(TRIBUNALS_DIR, "shard_00000.jsonl.gz")

def count_whitespace(text):
    return len(text.split())

def count_regex_words(text):
    return len(re.findall(r"\b\w+\b", text, flags=re.UNICODE))

def count_non_whitespace_tokens(text):
    return len(re.findall(r"\S+", text))

methods = {
    "split": count_whitespace,
    "regex_words": count_regex_words,
    "non_whitespace": count_non_whitespace_tokens,
}

results = {
    name: 0
    for name in methods
}

records = 0

with gzip.open(shard_path, "rt", encoding="utf-8") as f:
    for line in f:

        if records >= 100:
            break

        if not line.strip():
            continue

        obj = json.loads(line)

        text = obj.get("text", "")
        stored = obj.get("word_count")

        records += 1

        for name, fn in methods.items():
            if fn(text) == stored:
                results[name] += 1

print("=" * 80)
print("TRIBUNALS — WORD COUNT CONVENTION TEST")
print("=" * 80)

print(f"Records tested: {records}")
print()

for name, matches in results.items():
    print(f"{name:20s}: {matches:3d}/{records} exact matches")

print()
print("Interpretation:")
print("- If one method is close to 100/100, that is probably the convention.")
print("- If none matches, the stored field likely came from the raw dataset.")
print("- We will NOT rewrite the records based on this test.")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 8C — TRIBUNALS WORD-COUNT RELATIONSHIP
# READ-ONLY — 1,000 RECORDS
# ============================================================

import os
import json
import gzip
import statistics

TRIBUNALS_DIR = "/kaggle/working/phase1_processed/tribunals"
shard_path = os.path.join(TRIBUNALS_DIR, "shard_00000.jsonl.gz")

differences = []
ratios = []
examples = []

with gzip.open(shard_path, "rt", encoding="utf-8") as f:

    checked = 0

    for line in f:

        if checked >= 1000:
            break

        if not line.strip():
            continue

        obj = json.loads(line)

        text = obj.get("text", "")
        stored = obj.get("word_count")

        actual = len(text.split())

        if isinstance(stored, int) and actual > 0:
            difference = stored - actual
            ratio = stored / actual

            differences.append(difference)
            ratios.append(ratio)

            if len(examples) < 15:
                examples.append({
                    "stored": stored,
                    "actual": actual,
                    "difference": difference,
                    "ratio": round(ratio, 4),
                    "chars": len(text),
                })

        checked += 1

print("=" * 80)
print("TRIBUNALS — STORED vs ACTUAL WORD COUNT")
print("=" * 80)

print(f"Records checked : {checked:,}")
print()

print("Difference statistics (stored - text.split()):")
print(f"  Minimum : {min(differences):,}")
print(f"  Maximum : {max(differences):,}")
print(f"  Mean    : {statistics.mean(differences):,.2f}")
print(f"  Median  : {statistics.median(differences):,.2f}")
print()

print("Ratio statistics (stored / actual):")
print(f"  Minimum : {min(ratios):.4f}")
print(f"  Maximum : {max(ratios):.4f}")
print(f"  Mean    : {statistics.mean(ratios):.4f}")
print(f"  Median  : {statistics.median(ratios):.4f}")
print()

print("Examples:")
print("-" * 80)

for i, x in enumerate(examples, 1):
    print(
        f"{i:2d}. "
        f"stored={x['stored']:,} | "
        f"actual={x['actual']:,} | "
        f"diff={x['difference']:+,} | "
        f"ratio={x['ratio']:.4f} | "
        f"chars={x['chars']:,}"
    )

print("=" * 80)

In [ ]:
# ============================================================
# STEP 9A — SUPREME COURT DUPLICATE INVESTIGATION
# READ-ONLY
# ============================================================

import os
import json
import gzip
from collections import defaultdict

SC_DIR = "/kaggle/working/phase1_processed/supreme_court"

files = sorted(
    os.path.join(SC_DIR, f)
    for f in os.listdir(SC_DIR)
    if f.startswith("shard_") and f.endswith(".jsonl.gz")
)

by_id = defaultdict(list)
by_hash = defaultdict(list)

for path in files:
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            obj = json.loads(line)

            doc_id = obj.get("document_id")
            text_hash = obj.get("text_hash")

            by_id[doc_id].append(obj)
            by_hash[text_hash].append(obj)

duplicate_ids = {
    k: v for k, v in by_id.items()
    if k is not None and len(v) > 1
}

duplicate_hashes = {
    k: v for k, v in by_hash.items()
    if k is not None and len(v) > 1
}

print("=" * 80)
print("SUPREME COURT — DUPLICATE INVESTIGATION")
print("=" * 80)

print(f"Unique document IDs : {len(by_id):,}")
print(f"Duplicate IDs       : {len(duplicate_ids):,}")
print(f"Unique text hashes  : {len(by_hash):,}")
print(f"Duplicate hashes    : {len(duplicate_hashes):,}")
print()

print("SAMPLE DUPLICATE IDs")
print("-" * 80)

shown = 0

for doc_id, records in duplicate_ids.items():

    print(f"\nID: {doc_id}")
    print(f"Occurrences: {len(records)}")

    for i, obj in enumerate(records[:5], 1):
        print(
            f"  {i}. "
            f"year={obj.get('year')} | "
            f"court={obj.get('court')} | "
            f"filepath={obj.get('filepath')}"
        )
        print(
            f"     chars={obj.get('processed_char_count')} | "
            f"words={obj.get('processed_word_count')} | "
            f"hash={obj.get('text_hash')}"
        )

    shown += 1

    if shown >= 10:
        break

print()
print("=" * 80)

In [ ]:
# =============================================================================
# SUPREME COURT — DUPLICATE LOCATION DIAGNOSTIC
# =============================================================================

import os
import json
import gzip
from collections import defaultdict

BASE = "/kaggle/working/phase1_processed/supreme_court"

shards = sorted(
    f for f in os.listdir(BASE)
    if f.startswith("shard_") and f.endswith(".jsonl.gz")
)

print("=" * 80)
print("SUPREME COURT — DUPLICATE LOCATION DIAGNOSTIC")
print("=" * 80)

# -------------------------------------------------------------------------
# Pass 1: map document_id -> (shard, line number)
# -------------------------------------------------------------------------

id_locations = defaultdict(list)

total_records = 0

for shard in shards:
    path = os.path.join(BASE, shard)

    print(f"Reading {shard}...")

    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            if not line.strip():
                continue

            obj = json.loads(line)
            doc_id = obj.get("document_id")

            if doc_id:
                id_locations[doc_id].append(
                    (shard, line_number)
                )

            total_records += 1


# -------------------------------------------------------------------------
# Identify duplicates
# -------------------------------------------------------------------------

duplicates = {
    doc_id: locations
    for doc_id, locations in id_locations.items()
    if len(locations) > 1
}

duplicate_records = sum(
    len(locations) - 1
    for locations in duplicates.values()
)

print()
print("-" * 80)
print(f"Total records scanned       : {total_records:,}")
print(f"Unique document IDs         : {len(id_locations):,}")
print(f"Duplicate document IDs      : {len(duplicates):,}")
print(f"Duplicate records to remove : {duplicate_records:,}")
print("-" * 80)


# -------------------------------------------------------------------------
# Distribution
# -------------------------------------------------------------------------

same_shard = 0
cross_shard = 0

for doc_id, locations in duplicates.items():
    shard_set = {x[0] for x in locations}

    if len(shard_set) == 1:
        same_shard += 1
    else:
        cross_shard += 1

print()
print("DUPLICATE LOCATION")
print("-" * 80)
print(f"Duplicate IDs within same shard : {same_shard:,}")
print(f"Duplicate IDs across shards     : {cross_shard:,}")


# -------------------------------------------------------------------------
# Show examples
# -------------------------------------------------------------------------

print()
print("SAMPLE DUPLICATES")
print("-" * 80)

for i, (doc_id, locations) in enumerate(
    list(duplicates.items())[:20],
    start=1
):
    print(f"{i:02d}. {doc_id}")
    for shard, line_number in locations:
        print(f"      {shard} @ line {line_number}")


# -------------------------------------------------------------------------
# Check whether duplicates are adjacent
# -------------------------------------------------------------------------

adjacent = 0
non_adjacent = 0

for doc_id, locations in duplicates.items():

    if len(locations) != 2:
        continue

    (shard1, line1), (shard2, line2) = locations

    if shard1 == shard2 and abs(line1 - line2) == 1:
        adjacent += 1
    else:
        non_adjacent += 1

print()
print("ADJACENCY CHECK")
print("-" * 80)
print(f"Adjacent duplicate pairs     : {adjacent:,}")
print(f"Non-adjacent duplicate pairs  : {non_adjacent:,}")

print()
print("=" * 80)
print("DIAGNOSTIC COMPLETE")
print("=" * 80)

In [ ]:
# =============================================================================
# SUPREME COURT — STREAMING DOCUMENT-ID DEDUPLICATION
# =============================================================================
#
# IMPORTANT:
# - Original Supreme Court directory is NOT modified.
# - No datasets library.
# - No PyTorch.
# - No full dataset loaded into RAM.
# - Keeps FIRST occurrence of each document_id.
# - Writes to a temporary directory.
# =============================================================================

import os
import json
import gzip
import shutil
from pathlib import Path

BASE = Path("/kaggle/working/phase1_processed/supreme_court")
TEMP = Path("/kaggle/working/phase1_processed/supreme_court_dedup_temp")

# -------------------------------------------------------------------------
# Safety check
# -------------------------------------------------------------------------

if TEMP.exists():
    raise RuntimeError(
        f"Temporary directory already exists:\n{TEMP}\n"
        "Delete/inspect it manually before rerunning."
    )

TEMP.mkdir(parents=True)

shards = sorted(
    BASE.glob("shard_*.jsonl.gz")
)

print("=" * 80)
print("SUPREME COURT — STREAMING DOCUMENT-ID DEDUPLICATION")
print("=" * 80)

print(f"Source directory : {BASE}")
print(f"Output directory : {TEMP}")
print(f"Source shards    : {len(shards)}")
print()

# -------------------------------------------------------------------------
# Counters
# -------------------------------------------------------------------------

seen_ids = set()

total_read = 0
kept = 0
duplicates_removed = 0
missing_id = 0

# We preserve the existing shard structure.
# Each output shard contains the same records as the source shard,
# except duplicate document IDs are omitted.
#
# This means shard record counts may decrease.
# -------------------------------------------------------------------------

for shard_index, source_path in enumerate(shards):

    output_path = TEMP / source_path.name

    shard_read = 0
    shard_kept = 0
    shard_duplicates = 0

    print(
        f"[{shard_index + 1}/{len(shards)}] "
        f"Processing {source_path.name}"
    )

    with gzip.open(source_path, "rt", encoding="utf-8") as src, \
         gzip.open(output_path, "wt", encoding="utf-8") as dst:

        for line_number, line in enumerate(src, start=1):

            if not line.strip():
                continue

            obj = json.loads(line)

            total_read += 1
            shard_read += 1

            document_id = obj.get("document_id")

            if not document_id:
                # Do NOT silently discard records with no ID.
                # Preserve them and count them separately.
                missing_id += 1

                dst.write(
                    json.dumps(
                        obj,
                        ensure_ascii=False,
                        separators=(",", ":")
                    )
                    + "\n"
                )

                kept += 1
                shard_kept += 1
                continue

            if document_id in seen_ids:

                duplicates_removed += 1
                shard_duplicates += 1

                continue

            seen_ids.add(document_id)

            dst.write(
                json.dumps(
                    obj,
                    ensure_ascii=False,
                    separators=(",", ":")
                )
                + "\n"
            )

            kept += 1
            shard_kept += 1

    print(
        f"    Read       : {shard_read:,}"
    )
    print(
        f"    Kept       : {shard_kept:,}"
    )
    print(
        f"    Duplicates : {shard_duplicates:,}"
    )

# -------------------------------------------------------------------------
# Copy quarantine unchanged
# -------------------------------------------------------------------------

source_quarantine = BASE / "quarantine.jsonl.gz"

if source_quarantine.exists():
    shutil.copy2(
        source_quarantine,
        TEMP / "quarantine.jsonl.gz"
    )

# -------------------------------------------------------------------------
# Copy manifest as backup/reference only.
#
# We intentionally DO NOT claim it is still valid because the shard
# record counts changed. A new manifest will be generated after verification.
# -------------------------------------------------------------------------

source_manifest = BASE / "manifest.json"

if source_manifest.exists():
    shutil.copy2(
        source_manifest,
        TEMP / "manifest_before_dedup.json"
    )

# -------------------------------------------------------------------------
# Summary
# -------------------------------------------------------------------------

print()
print("=" * 80)
print("DEDUPLICATION COMPLETE")
print("=" * 80)

print(f"Records read             : {total_read:,}")
print(f"Records kept             : {kept:,}")
print(f"Duplicates removed      : {duplicates_removed:,}")
print(f"Records without ID       : {missing_id:,}")
print(f"Unique IDs tracked       : {len(seen_ids):,}")

print()
print(f"Expected original        : 55,528")
print(f"Expected after dedup     : 55,363")
print(f"Expected removed         : 165")

print()
print(f"Temporary output:")
print(f"  {TEMP}")

print()
print("IMPORTANT:")
print("Original Supreme Court data has NOT been modified.")
print("=" * 80)

In [ ]:
# =============================================================================
# SUPREME COURT — POST-DEDUP VERIFICATION
# =============================================================================

import os
import json
import gzip
from collections import Counter

TEMP = "/kaggle/working/phase1_processed/supreme_court_dedup_temp"

print("=" * 80)
print("SUPREME COURT — POST-DEDUP VERIFICATION")
print("=" * 80)

shards = sorted(
    f for f in os.listdir(TEMP)
    if f.startswith("shard_") and f.endswith(".jsonl.gz")
)

seen_ids = set()

total_records = 0
duplicate_ids = 0
malformed = 0
empty_lines = 0

for shard in shards:

    path = os.path.join(TEMP, shard)

    shard_records = 0
    shard_duplicates = 0

    print(f"Reading {shard}...")

    with gzip.open(path, "rt", encoding="utf-8") as f:

        for line_number, line in enumerate(f, start=1):

            if not line.strip():
                empty_lines += 1
                continue

            try:
                obj = json.loads(line)
            except Exception:
                malformed += 1
                continue

            total_records += 1
            shard_records += 1

            document_id = obj.get("document_id")

            if document_id in seen_ids:
                duplicate_ids += 1
                shard_duplicates += 1
            else:
                seen_ids.add(document_id)

    print(
        f"  Records    : {shard_records:,}"
        f" | duplicates found: {shard_duplicates:,}"
    )

print()
print("-" * 80)
print(f"Total records       : {total_records:,}")
print(f"Unique document IDs : {len(seen_ids):,}")
print(f"Duplicate IDs       : {duplicate_ids:,}")
print(f"Empty lines         : {empty_lines:,}")
print(f"Malformed records   : {malformed:,}")
print("-" * 80)

# -------------------------------------------------------------------------
# Expected values
# -------------------------------------------------------------------------

assert total_records == 55_363, (
    f"Expected 55,363 records, found {total_records}"
)

assert len(seen_ids) == 55_363, (
    f"Expected 55,363 unique IDs, found {len(seen_ids)}"
)

assert duplicate_ids == 0, (
    f"Found {duplicate_ids} duplicate IDs after deduplication"
)

assert malformed == 0, (
    f"Found {malformed} malformed records"
)

print()
print("✓ Record count correct")
print("✓ All document IDs unique")
print("✓ No duplicate IDs remain")
print("✓ No malformed records")
print()
print("=" * 80)
print("POST-DEDUP VERIFICATION PASSED")
print("=" * 80)

In [ ]:
# =============================================================================
# SUPREME COURT — PROMOTE VERIFIED DEDUPLICATED OUTPUT
# =============================================================================

import os
import shutil
from pathlib import Path
from datetime import datetime

BASE = Path("/kaggle/working/phase1_processed/supreme_court")
TEMP = Path("/kaggle/working/phase1_processed/supreme_court_dedup_temp")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
BACKUP = Path(
    f"/kaggle/working/phase1_processed/supreme_court_backup_{timestamp}"
)

print("=" * 80)
print("SUPREME COURT — PROMOTING VERIFIED DEDUPLICATED DATA")
print("=" * 80)

# -------------------------------------------------------------------------
# Safety checks
# -------------------------------------------------------------------------

if not BASE.exists():
    raise RuntimeError(f"Original directory missing: {BASE}")

if not TEMP.exists():
    raise RuntimeError(f"Deduplicated directory missing: {TEMP}")

required_shards = [
    TEMP / "shard_00000.jsonl.gz",
    TEMP / "shard_00001.jsonl.gz",
]

for path in required_shards:
    if not path.exists():
        raise RuntimeError(f"Missing verified shard: {path}")

print("✓ Original directory exists")
print("✓ Deduplicated directory exists")
print("✓ Required shards present")

# -------------------------------------------------------------------------
# Backup original
# -------------------------------------------------------------------------

print()
print(f"Creating backup:")
print(f"  {BASE}")
print(f"→ {BACKUP}")

shutil.copytree(BASE, BACKUP)

print("✓ Original backed up")

# -------------------------------------------------------------------------
# Remove only the original directory
# -------------------------------------------------------------------------

print()
print("Replacing original Supreme Court directory...")

shutil.rmtree(BASE)

# -------------------------------------------------------------------------
# Promote verified directory
# -------------------------------------------------------------------------

TEMP.rename(BASE)

print("✓ Deduplicated directory promoted")

# -------------------------------------------------------------------------
# Final paths
# -------------------------------------------------------------------------

print()
print("=" * 80)
print("PROMOTION COMPLETE")
print("=" * 80)

print(f"Active dataset:")
print(f"  {BASE}")

print()
print(f"Backup of pre-dedup dataset:")
print(f"  {BACKUP}")

print()
print("IMPORTANT:")
print("The original data is preserved in the backup.")
print("The active Supreme Court dataset now contains 55,363 records.")
print("=" * 80)

In [ ]:
from pathlib import Path
import gzip
import json

BASE = Path("/kaggle/working/phase1_processed")

ACTIVE = BASE / "supreme_court"
BACKUP = BASE / "supreme_court_backup_20260928_102432"

def count_records(directory):
    total = 0

    for path in sorted(directory.glob("shard_*.jsonl.gz")):
        with gzip.open(path, "rt", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    total += 1

    return total


print("=" * 80)
print("SUPREME COURT — ACTIVE vs ORIGINAL")
print("=" * 80)

print("\nACTIVE DATASET")
print("-" * 80)
print(ACTIVE)
print(f"Exists : {ACTIVE.exists()}")

if ACTIVE.exists():
    print(f"Records: {count_records(ACTIVE):,}")

print("\nORIGINAL BACKUP")
print("-" * 80)
print(BACKUP)
print(f"Exists : {BACKUP.exists()}")

if BACKUP.exists():
    print(f"Records: {count_records(BACKUP):,}")

print("\n" + "=" * 80)

if ACTIVE.exists() and BACKUP.exists():
    active_count = count_records(ACTIVE)
    backup_count = count_records(BACKUP)

    print(f"Original records : {backup_count:,}")
    print(f"Active records   : {active_count:,}")
    print(f"Difference       : {backup_count - active_count:,}")

    assert backup_count == 55_528
    assert active_count == 55_363

    print("\n✓ Original backup = 55,528")
    print("✓ Active deduplicated = 55,363")
    print("✓ 165 records removed from active copy")

In [ ]:
from pathlib import Path
import gzip
import json
from collections import defaultdict

BASE = Path("/kaggle/working/phase1_processed")
TRIB = BASE / "tribunals"

print("=" * 80)
print("TRIBUNALS — DUPLICATE DOCUMENT-ID LOCATION DIAGNOSTIC")
print("=" * 80)

shards = sorted(TRIB.glob("shard_*.jsonl.gz"))

if not shards:
    raise RuntimeError(f"No Tribunal shards found: {TRIB}")

print(f"Source shards: {len(shards)}")

locations = defaultdict(list)

total_records = 0

for shard_index, shard in enumerate(shards, start=1):

    print(
        f"Reading shard {shard_index}/{len(shards)}: "
        f"{shard.name}"
    )

    with gzip.open(shard, "rt", encoding="utf-8") as f:

        for line_number, line in enumerate(f, start=1):

            if not line.strip():
                continue

            record = json.loads(line)

            total_records += 1

            document_id = record.get("document_id")

            if document_id:
                locations[document_id].append(
                    (shard.name, line_number)
                )

duplicate_ids = {
    doc_id: locs
    for doc_id, locs in locations.items()
    if len(locs) > 1
}

duplicate_records = sum(
    len(locs) - 1
    for locs in duplicate_ids.values()
)

same_shard = 0
cross_shard = 0

for locs in duplicate_ids.values():

    shard_names = {x[0] for x in locs}

    if len(shard_names) == 1:
        same_shard += 1
    else:
        cross_shard += 1

print("\n" + "=" * 80)
print("RESULT")
print("=" * 80)

print(f"Total records scanned       : {total_records:,}")
print(f"Unique document IDs         : {len(locations):,}")
print(f"Duplicate document IDs      : {len(duplicate_ids):,}")
print(f"Duplicate records to remove : {duplicate_records:,}")

print("\n" + "-" * 80)
print("DUPLICATE LOCATION")
print("-" * 80)

print(f"Duplicate IDs within same shard : {same_shard:,}")
print(f"Duplicate IDs across shards     : {cross_shard:,}")

print("\n" + "-" * 80)
print("SAMPLE DUPLICATES")
print("-" * 80)

for i, (doc_id, locs) in enumerate(
    list(duplicate_ids.items())[:20],
    start=1
):

    print(f"{i:02d}. {doc_id}")

    for shard_name, line_number in locs:
        print(
            f"      {shard_name} @ line {line_number}"
        )

print("\n" + "=" * 80)
print("DIAGNOSTIC COMPLETE")
print("=" * 80)

In [ ]:
from pathlib import Path
import gzip
import json
from collections import defaultdict

BASE = Path("/kaggle/working/phase1_processed")
TRIB = BASE / "tribunals"

print("=" * 80)
print("TRIBUNALS — DUPLICATE RECORD CONTENT ANALYSIS")
print("=" * 80)

shards = sorted(TRIB.glob("shard_*.jsonl.gz"))

# -------------------------------------------------------------------------
# PASS 1: identify duplicated IDs
# -------------------------------------------------------------------------

id_counts = defaultdict(int)

for i, shard in enumerate(shards, start=1):

    print(f"Pass 1 — shard {i}/{len(shards)}: {shard.name}")

    with gzip.open(shard, "rt", encoding="utf-8") as f:

        for line in f:

            if not line.strip():
                continue

            record = json.loads(line)
            document_id = record.get("document_id")

            if document_id:
                id_counts[document_id] += 1


duplicate_ids = {
    doc_id
    for doc_id, count in id_counts.items()
    if count > 1
}

print("\n" + "-" * 80)
print(f"Duplicate IDs identified : {len(duplicate_ids):,}")
print("-" * 80)

# -------------------------------------------------------------------------
# PASS 2: inspect content of duplicate IDs
# -------------------------------------------------------------------------

duplicate_records = defaultdict(list)

for i, shard in enumerate(shards, start=1):

    print(f"Pass 2 — shard {i}/{len(shards)}: {shard.name}")

    with gzip.open(shard, "rt", encoding="utf-8") as f:

        for line_number, line in enumerate(f, start=1):

            if not line.strip():
                continue

            record = json.loads(line)

            document_id = record.get("document_id")

            if document_id in duplicate_ids:

                duplicate_records[document_id].append({
                    "shard": shard.name,
                    "line": line_number,
                    "text_hash": record.get("text_hash"),
                    "text": record.get("text", ""),
                    "year": record.get("year"),
                    "court": record.get("court"),
                    "filepath": record.get("filepath"),
                    "word_count": record.get("word_count"),
                    "char_count": record.get("char_count"),
                })

# -------------------------------------------------------------------------
# Compare duplicate occurrences
# -------------------------------------------------------------------------

exact_same_text = 0
different_text = 0
same_hash = 0
different_hash = 0
same_metadata = 0
different_metadata = 0

for doc_id, records in duplicate_records.items():

    hashes = {
        r["text_hash"]
        for r in records
    }

    texts = {
        r["text"]
        for r in records
    }

    metadata = {
        (
            r["year"],
            r["court"],
            r["filepath"],
            r["word_count"],
            r["char_count"],
        )
        for r in records
    }

    if len(texts) == 1:
        exact_same_text += 1
    else:
        different_text += 1

    if len(hashes) == 1:
        same_hash += 1
    else:
        different_hash += 1

    if len(metadata) == 1:
        same_metadata += 1
    else:
        different_metadata += 1

# -------------------------------------------------------------------------
# Summary
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("DUPLICATE CONTENT SUMMARY")
print("=" * 80)

print(f"Duplicate document IDs       : {len(duplicate_ids):,}")
print(f"IDs with identical text      : {exact_same_text:,}")
print(f"IDs with different text      : {different_text:,}")
print(f"IDs with identical hash      : {same_hash:,}")
print(f"IDs with different hash      : {different_hash:,}")
print(f"IDs with identical metadata  : {same_metadata:,}")
print(f"IDs with different metadata  : {different_metadata:,}")

# -------------------------------------------------------------------------
# Show examples where duplicates differ
# -------------------------------------------------------------------------

different_examples = []

for doc_id, records in duplicate_records.items():

    hashes = {
        r["text_hash"]
        for r in records
    }

    texts = {
        r["text"]
        for r in records
    }

    if len(texts) > 1 or len(hashes) > 1:
        different_examples.append(
            (doc_id, records)
        )

print("\n" + "=" * 80)
print("DIFFERENT-CONTENT DUPLICATE EXAMPLES")
print("=" * 80)

if not different_examples:

    print("✓ No duplicate document IDs have different text/hash.")

else:

    print(
        f"Found {len(different_examples):,} duplicate IDs "
        "with differing content."
    )

    for idx, (doc_id, records) in enumerate(
        different_examples[:20],
        start=1
    ):

        print("\n" + "-" * 80)
        print(f"{idx:02d}. document_id = {doc_id}")
        print(f"Occurrences = {len(records)}")

        for j, r in enumerate(records, start=1):

            print(
                f"\n  Occurrence {j}:"
                f"\n    shard      = {r['shard']}"
                f"\n    line       = {r['line']}"
                f"\n    text_hash  = {r['text_hash']}"
                f"\n    year       = {r['year']}"
                f"\n    court      = {r['court']}"
                f"\n    filepath   = {r['filepath']}"
                f"\n    word_count = {r['word_count']}"
                f"\n    char_count = {r['char_count']}"
                f"\n    text chars = {len(r['text'])}"
            )

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# TRIBUNALS — STREAMING DOCUMENT-ID DEDUPLICATION
# Creates a temporary verified copy.
# DOES NOT modify the original dataset.
# ============================================================

from pathlib import Path
import gzip
import json
import shutil

SOURCE = Path("/kaggle/working/phase1_processed/tribunals")
TEMP = Path("/kaggle/working/phase1_processed/tribunals_dedup_temp")

print("=" * 80)
print("TRIBUNALS — STREAMING DOCUMENT-ID DEDUPLICATION")
print("=" * 80)

if not SOURCE.exists():
    raise RuntimeError(f"Source directory missing: {SOURCE}")

shards = sorted(SOURCE.glob("shard_*.jsonl.gz"))

if not shards:
    raise RuntimeError(f"No Tribunal shards found in: {SOURCE}")

print(f"Source directory : {SOURCE}")
print(f"Output directory : {TEMP}")
print(f"Source shards    : {len(shards)}")

# Safety: remove an incomplete previous temporary directory.
if TEMP.exists():
    print("\nRemoving previous temporary directory...")
    shutil.rmtree(TEMP)

TEMP.mkdir(parents=True, exist_ok=True)

seen_ids = set()

total_read = 0
total_kept = 0
total_duplicates = 0
records_without_id = 0

for shard_index, shard in enumerate(shards, start=1):

    print(
        f"\n[{shard_index}/{len(shards)}] "
        f"Processing {shard.name}"
    )

    output_shard = TEMP / shard.name

    shard_read = 0
    shard_kept = 0
    shard_duplicates = 0

    with gzip.open(shard, "rt", encoding="utf-8") as fin, \
         gzip.open(output_shard, "wt", encoding="utf-8") as fout:

        for line in fin:

            if not line.strip():
                continue

            record = json.loads(line)

            shard_read += 1
            total_read += 1

            document_id = record.get("document_id")

            # Records without document_id are retained.
            if document_id is None:
                records_without_id += 1
                fout.write(
                    json.dumps(
                        record,
                        ensure_ascii=False,
                        separators=(",", ":")
                    ) + "\n"
                )
                shard_kept += 1
                total_kept += 1
                continue

            # Keep first occurrence only.
            if document_id in seen_ids:
                shard_duplicates += 1
                total_duplicates += 1
                continue

            seen_ids.add(document_id)

            fout.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                    separators=(",", ":")
                ) + "\n"
            )

            shard_kept += 1
            total_kept += 1

    print(f"    Read       : {shard_read:,}")
    print(f"    Kept       : {shard_kept:,}")
    print(f"    Duplicates : {shard_duplicates:,}")


print("\n" + "=" * 80)
print("DEDUPLICATION COMPLETE")
print("=" * 80)

print(f"Records read             : {total_read:,}")
print(f"Records kept             : {total_kept:,}")
print(f"Duplicates removed       : {total_duplicates:,}")
print(f"Records without ID       : {records_without_id:,}")
print(f"Unique IDs tracked       : {len(seen_ids):,}")

print("\nExpected original        : 869,132")
print("Expected after dedup     : 864,198")
print("Expected removed         : 4,934")

if total_read != 869132:
    raise RuntimeError(
        f"Unexpected source record count: {total_read:,}"
    )

if total_kept != 864198:
    raise RuntimeError(
        f"Unexpected deduplicated count: {total_kept:,}"
    )

if total_duplicates != 4934:
    raise RuntimeError(
        f"Unexpected duplicate count: {total_duplicates:,}"
    )

if len(seen_ids) != 864198:
    raise RuntimeError(
        f"Unexpected unique ID count: {len(seen_ids):,}"
    )

print("\nTemporary output:")
print(f"  {TEMP}")

print("\nIMPORTANT:")
print("Original Tribunals data has NOT been modified.")

print("=" * 80)

In [ ]:
# ============================================================
# TRIBUNALS — POST-DEDUP VERIFICATION
# ============================================================

from pathlib import Path
import gzip
import json

TEMP = Path(
    "/kaggle/working/phase1_processed/tribunals_dedup_temp"
)

print("=" * 80)
print("TRIBUNALS — POST-DEDUP VERIFICATION")
print("=" * 80)

if not TEMP.exists():
    raise RuntimeError(
        f"Temporary deduplicated directory missing: {TEMP}"
    )

shards = sorted(TEMP.glob("shard_*.jsonl.gz"))

if not shards:
    raise RuntimeError(
        f"No deduplicated shards found in: {TEMP}"
    )

seen_ids = set()

total_records = 0
duplicate_ids = 0
empty_lines = 0
malformed_records = 0

for i, shard in enumerate(shards, start=1):

    print(f"Reading {shard.name}...")

    shard_records = 0
    shard_duplicates = 0

    with gzip.open(shard, "rt", encoding="utf-8") as f:

        for line_number, line in enumerate(f, start=1):

            if not line.strip():
                empty_lines += 1
                continue

            try:
                record = json.loads(line)
            except Exception:
                malformed_records += 1
                continue

            total_records += 1
            shard_records += 1

            document_id = record.get("document_id")

            if document_id is None:
                raise RuntimeError(
                    f"Missing document_id in "
                    f"{shard.name} line {line_number}"
                )

            if document_id in seen_ids:
                duplicate_ids += 1
                shard_duplicates += 1
            else:
                seen_ids.add(document_id)

    print(
        f"  Records    : {shard_records:,} | "
        f"duplicates found: {shard_duplicates:,}"
    )

print("\n" + "-" * 80)

print(f"Total records       : {total_records:,}")
print(f"Unique document IDs : {len(seen_ids):,}")
print(f"Duplicate IDs       : {duplicate_ids:,}")
print(f"Empty lines         : {empty_lines:,}")
print(f"Malformed records   : {malformed_records:,}")

print("-" * 80)

# ------------------------------------------------------------
# HARD CHECKS
# ------------------------------------------------------------

if total_records != 864198:
    raise RuntimeError(
        f"Expected 864,198 records, found {total_records:,}"
    )

if len(seen_ids) != 864198:
    raise RuntimeError(
        f"Expected 864,198 unique IDs, found {len(seen_ids):,}"
    )

if duplicate_ids != 0:
    raise RuntimeError(
        f"Duplicate IDs remain: {duplicate_ids:,}"
    )

if empty_lines != 0:
    raise RuntimeError(
        f"Empty lines found: {empty_lines:,}"
    )

if malformed_records != 0:
    raise RuntimeError(
        f"Malformed records found: {malformed_records:,}"
    )

print("\n✓ Record count correct")
print("✓ All document IDs unique")
print("✓ No duplicate IDs remain")
print("✓ No malformed records")

print("\n" + "=" * 80)
print("POST-DEDUP VERIFICATION PASSED")
print("=" * 80)

In [ ]:
# ============================================================
# TRIBUNALS — PROMOTE VERIFIED DEDUPLICATED DATA
# ============================================================

from pathlib import Path
from datetime import datetime
import shutil

ACTIVE = Path(
    "/kaggle/working/phase1_processed/tribunals"
)

TEMP = Path(
    "/kaggle/working/phase1_processed/tribunals_dedup_temp"
)

print("=" * 80)
print("TRIBUNALS — PROMOTING VERIFIED DEDUPLICATED DATA")
print("=" * 80)

# ------------------------------------------------------------
# SAFETY CHECKS
# ------------------------------------------------------------

if not ACTIVE.exists():
    raise RuntimeError(
        f"Active Tribunals directory missing: {ACTIVE}"
    )

if not TEMP.exists():
    raise RuntimeError(
        f"Verified temporary directory missing: {TEMP}"
    )

shards = sorted(TEMP.glob("shard_*.jsonl.gz"))

if len(shards) != 18:
    raise RuntimeError(
        f"Expected 18 deduplicated shards, found {len(shards)}"
    )

print("✓ Original directory exists")
print("✓ Deduplicated directory exists")
print("✓ Required shards present")

# ------------------------------------------------------------
# CREATE BACKUP
# ------------------------------------------------------------

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

BACKUP = Path(
    f"/kaggle/working/phase1_processed/"
    f"tribunals_backup_{timestamp}"
)

print("\nCreating backup:")
print(f"  {ACTIVE}")
print(f"→ {BACKUP}")

if BACKUP.exists():
    raise RuntimeError(
        f"Backup directory already exists: {BACKUP}"
    )

shutil.copytree(ACTIVE, BACKUP)

print("✓ Original backed up")

# ------------------------------------------------------------
# REPLACE ACTIVE DIRECTORY
# ------------------------------------------------------------

print("\nReplacing original Tribunals directory...")

shutil.rmtree(ACTIVE)
shutil.move(str(TEMP), str(ACTIVE))

print("✓ Deduplicated directory promoted")

# ------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PROMOTION COMPLETE")
print("=" * 80)

print("Active dataset:")
print(f"  {ACTIVE}")

print("\nBackup of pre-dedup dataset:")
print(f"  {BACKUP}")

print("\nIMPORTANT:")
print("The original data is preserved in the backup.")
print("The active Tribunals dataset should now contain 864,198 records.")

print("=" * 80)

In [ ]:
# ============================================================
# TRIBUNALS — ACTIVE vs ORIGINAL BACKUP
# ============================================================

from pathlib import Path
import gzip
import json

ACTIVE = Path(
    "/kaggle/working/phase1_processed/tribunals"
)

BACKUP = Path(
    "/kaggle/working/phase1_processed/tribunals_backup_20260928_104758"
)


def count_records(directory):
    total = 0

    shards = sorted(directory.glob("shard_*.jsonl.gz"))

    for shard in shards:
        with gzip.open(shard, "rt", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    total += 1

    return total


print("=" * 80)
print("TRIBUNALS — ACTIVE vs ORIGINAL")
print("=" * 80)

if not ACTIVE.exists():
    raise RuntimeError(f"Active directory missing: {ACTIVE}")

if not BACKUP.exists():
    raise RuntimeError(f"Backup directory missing: {BACKUP}")

active_count = count_records(ACTIVE)
backup_count = count_records(BACKUP)

print("\nACTIVE DATASET")
print("-" * 80)
print(ACTIVE)
print(f"Exists : {ACTIVE.exists()}")
print(f"Records: {active_count:,}")

print("\nORIGINAL BACKUP")
print("-" * 80)
print(BACKUP)
print(f"Exists : {BACKUP.exists()}")
print(f"Records: {backup_count:,}")

print("\n" + "=" * 80)

difference = backup_count - active_count

print(f"Original records : {backup_count:,}")
print(f"Active records   : {active_count:,}")
print(f"Difference       : {difference:,}")

if backup_count != 869132:
    raise RuntimeError(
        f"Backup record count incorrect: {backup_count:,}"
    )

if active_count != 864198:
    raise RuntimeError(
        f"Active record count incorrect: {active_count:,}"
    )

if difference != 4934:
    raise RuntimeError(
        f"Expected 4,934 removed, found {difference:,}"
    )

print("\n✓ Original backup = 869,132")
print("✓ Active deduplicated = 864,198")
print("✓ 4,934 records removed from active copy")

print("=" * 80)

In [ ]:
# ============================================================
# HIGH COURTS — DUPLICATE TEXT HASH LOCATION DIAGNOSTIC
# ============================================================

from pathlib import Path
import gzip
import json
from collections import defaultdict

SOURCE = Path(
    "/kaggle/working/phase1_processed/high_courts"
)

print("=" * 80)
print("HIGH COURTS — DUPLICATE TEXT HASH LOCATION DIAGNOSTIC")
print("=" * 80)

shards = sorted(SOURCE.glob("shard_*.jsonl.gz"))

if not shards:
    raise RuntimeError(f"No High Court shards found: {SOURCE}")

print(f"Source shards: {len(shards)}")

# hash -> list of (shard, line, document_id)
hash_locations = defaultdict(list)

total_records = 0

for shard_index, shard in enumerate(shards, start=1):

    print(
        f"Reading shard {shard_index}/{len(shards)}: "
        f"{shard.name}"
    )

    with gzip.open(shard, "rt", encoding="utf-8") as f:

        for line_number, line in enumerate(f, start=1):

            if not line.strip():
                continue

            record = json.loads(line)

            total_records += 1

            text_hash = record.get("text_hash")
            document_id = record.get("document_id")

            if text_hash:
                hash_locations[text_hash].append(
                    (
                        shard.name,
                        line_number,
                        document_id
                    )
                )

# ------------------------------------------------------------
# ANALYSIS
# ------------------------------------------------------------

duplicate_hashes = {
    h: locations
    for h, locations in hash_locations.items()
    if len(locations) > 1
}

duplicate_records = sum(
    len(locations) - 1
    for locations in duplicate_hashes.values()
)

unique_duplicate_records = sum(
    len(locations)
    for locations in duplicate_hashes.values()
)

within_shard = 0
across_shards = 0

for locations in duplicate_hashes.values():

    shards_for_hash = {
        location[0]
        for location in locations
    }

    if len(shards_for_hash) == 1:
        within_shard += 1
    else:
        across_shards += 1

# ------------------------------------------------------------
# RESULT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("RESULT")
print("=" * 80)

print(f"Total records scanned       : {total_records:,}")
print(f"Unique text hashes          : {len(hash_locations):,}")
print(f"Duplicate text hashes       : {len(duplicate_hashes):,}")
print(f"Duplicate records to remove : {duplicate_records:,}")
print(f"Records belonging to groups : {unique_duplicate_records:,}")

print("\n" + "-" * 80)
print("DUPLICATE HASH LOCATION")
print("-" * 80)

print(
    f"Duplicate hashes within same shard : {within_shard:,}"
)

print(
    f"Duplicate hashes across shards     : {across_shards:,}"
)

# ------------------------------------------------------------
# SAMPLE DUPLICATE HASHES
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("SAMPLE DUPLICATES")
print("-" * 80)

for index, (text_hash, locations) in enumerate(
    list(duplicate_hashes.items())[:20],
    start=1
):

    print(f"{index:02d}. HASH: {text_hash}")

    for shard_name, line_number, document_id in locations[:10]:

        print(
            f"      {shard_name} @ line {line_number}"
            f" | document_id={document_id}"
        )

    if len(locations) > 10:
        print(
            f"      ... {len(locations) - 10} more occurrences"
        )

print("\n" + "=" * 80)
print("DIAGNOSTIC COMPLETE")
print("=" * 80)

print("\nIMPORTANT:")
print("No High Court records were modified.")

In [ ]:
from pathlib import Path
import gzip
import json

BASE = Path("/kaggle/working/phase1_processed")

DOMAINS = {
    "acts": BASE / "acts",
    "high_courts": BASE / "high_courts",
    "supreme_court": BASE / "supreme_court",
    "tribunals": BASE / "tribunals",
}

EXPECTED = {
    "acts": 18_732,
    "high_courts": 2_536_738,
    "supreme_court": 55_363,
    "tribunals": 864_198,
}


def count_records(directory):
    total = 0

    for path in sorted(directory.glob("*.jsonl")):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    total += 1

    for path in sorted(directory.glob("*.jsonl.gz")):
        with gzip.open(path, "rt", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    total += 1

    return total


print("=" * 80)
print("PHASE 1 — ACTIVE DATASET VERIFICATION")
print("=" * 80)

all_ok = True

for domain, path in DOMAINS.items():
    print(f"\n{domain.upper()}")
    print("-" * 80)

    if not path.exists():
        print("✗ Directory missing")
        all_ok = False
        continue

    actual = count_records(path)
    expected = EXPECTED[domain]

    print(f"Directory : {path}")
    print(f"Expected  : {expected:,}")
    print(f"Actual    : {actual:,}")

    if actual == expected:
        print("✓ Record count correct")
    else:
        print(f"✗ Record count mismatch: {actual - expected:+,}")
        all_ok = False

print("\n" + "=" * 80)

if all_ok:
    print("✓ ALL ACTIVE DATASETS VERIFIED")
else:
    print("✗ VERIFICATION FAILED — DO NOT CONTINUE")

print("=" * 80)

In [ ]:
import json
import gzip
from pathlib import Path

BASE = Path("/kaggle/working/phase1_processed")

DOMAINS = {
    "acts": BASE / "acts",
    "high_courts": BASE / "high_courts",
    "supreme_court": BASE / "supreme_court",
    "tribunals": BASE / "tribunals",
}

REQUIRED = {
    "source_domain",
    "document_id",
    "text",
    "text_hash",
}

print("=" * 80)
print("PHASE 1 — ACTIVE DATA REQUIRED-FIELD CHECK")
print("=" * 80)

global_errors = 0

for domain, directory in DOMAINS.items():

    print(f"\n{domain.upper()}")
    print("-" * 80)

    files = sorted(directory.glob("*.jsonl")) + sorted(directory.glob("*.jsonl.gz"))

    checked = 0
    errors = 0

    for path in files:
        opener = gzip.open if path.suffix == ".gz" else open
        mode = "rt"

        with opener(path, mode, encoding="utf-8") as f:
            for line in f:

                if not line.strip():
                    continue

                try:
                    record = json.loads(line)
                except Exception:
                    errors += 1
                    continue

                missing = REQUIRED - set(record.keys())

                if missing:
                    errors += 1
                    if errors <= 3:
                        print(
                            f"Missing fields in {path.name}: "
                            f"{sorted(missing)}"
                        )

                checked += 1

                # Sampling is enough here because schema was already
                # exhaustively validated in Phase 1.
                if checked >= 1000:
                    break

    global_errors += errors

    print(f"Records sampled : {checked:,}")
    print(f"Schema errors   : {errors:,}")

    if errors == 0:
        print("✓ Required fields present")
    else:
        print("✗ Schema problems found")

print("\n" + "=" * 80)

if global_errors == 0:
    print("✓ ACTIVE DATASETS PASSED REQUIRED-FIELD CHECK")
else:
    print(f"✗ TOTAL SCHEMA ERRORS: {global_errors}")

print("=" * 80)

In [ ]:
FINAL_COUNTS = {
    "acts": 18_732,
    "high_courts": 2_536_738,
    "supreme_court": 55_363,
    "tribunals": 864_198,
}

total = sum(FINAL_COUNTS.values())

print("=" * 80)
print("PHASE 1 — FINAL ACTIVE DATASET INVENTORY")
print("=" * 80)

for domain, count in FINAL_COUNTS.items():
    print(f"{domain.upper():20s}: {count:>12,}")

print("-" * 80)
print(f"{'TOTAL':20s}: {total:>12,}")

print("=" * 80)
print("✓ SUPREME COURT: 55,363 active records")
print("✓ TRIBUNALS    : 864,198 active records")
print("✓ ACTS         : 18,732 active records")
print("✓ HIGH COURTS  : 2,536,738 active records")
print("=" * 80)

In [ ]:
from pathlib import Path

BASE = Path("/kaggle/working/phase1_processed")

DOMAINS = {
    "acts": BASE / "acts",
    "high_courts": BASE / "high_courts",
    "supreme_court": BASE / "supreme_court",
    "tribunals": BASE / "tribunals",
}

print("=" * 80)
print("PHASE 1 — ACTIVE DIRECTORY FILE INVENTORY")
print("=" * 80)

for domain, directory in DOMAINS.items():

    print(f"\n{domain.upper()}")
    print("-" * 80)
    print(f"Directory: {directory}")

    files = sorted(
        p for p in directory.iterdir()
        if p.is_file()
    )

    for p in files:
        print(f"{p.name:45s} {p.stat().st_size / (1024**2):10.2f} MB")

    print(f"\nFiles found: {len(files)}")

print("\n" + "=" * 80)
print("INVENTORY COMPLETE")
print("=" * 80)

In [ ]:
from pathlib import Path
import gzip

BASE = Path("/kaggle/working/phase1_processed")

DOMAINS = {
    "acts": BASE / "acts",
    "high_courts": BASE / "high_courts",
    "supreme_court": BASE / "supreme_court",
    "tribunals": BASE / "tribunals",
}

EXPECTED = {
    "acts": 18_732,
    "high_courts": 2_536_738,
    "supreme_court": 55_363,
    "tribunals": 864_198,
}


def active_files(directory):
    """
    Return only actual active dataset files.
    Quarantine files are deliberately excluded.
    """

    files = []

    for path in sorted(directory.iterdir()):

        if not path.is_file():
            continue

        name = path.name.lower()

        # Explicitly exclude quarantine artifacts
        if "quarantine" in name:
            continue

        # Active JSONL / compressed JSONL files
        if name.endswith(".jsonl") or name.endswith(".jsonl.gz"):
            files.append(path)

    return files


def count_records(files):

    total = 0

    for path in files:

        if path.name.endswith(".gz"):
            opener = gzip.open
        else:
            opener = open

        with opener(path, "rt", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    total += 1

    return total


print("=" * 80)
print("PHASE 1 — CORRECTED ACTIVE DATASET VERIFICATION")
print("=" * 80)

all_ok = True

for domain, directory in DOMAINS.items():

    print(f"\n{domain.upper()}")
    print("-" * 80)

    files = active_files(directory)

    print("Active files:")

    for path in files:
        print(f"  {path.name}")

    actual = count_records(files)
    expected = EXPECTED[domain]

    print(f"\nExpected records : {expected:,}")
    print(f"Actual records   : {actual:,}")

    if actual == expected:
        print("✓ Record count correct")
    else:
        print(f"✗ MISMATCH: {actual - expected:+,}")
        all_ok = False

print("\n" + "=" * 80)

if all_ok:
    print("✓ ALL ACTIVE DATASETS VERIFIED")
else:
    print("✗ VERIFICATION FAILED")

print("=" * 80)

In [ ]:
import json
import gzip
from pathlib import Path

BASE = Path("/kaggle/working/phase1_processed")

DOMAINS = {
    "acts": BASE / "acts",
    "high_courts": BASE / "high_courts",
    "supreme_court": BASE / "supreme_court",
    "tribunals": BASE / "tribunals",
}

REQUIRED = {
    "source_domain",
    "document_id",
    "text",
    "text_hash",
}


def active_files(directory):

    files = []

    for path in sorted(directory.iterdir()):

        if not path.is_file():
            continue

        name = path.name.lower()

        if "quarantine" in name:
            continue

        if name.endswith(".jsonl") or name.endswith(".jsonl.gz"):
            files.append(path)

    return files


print("=" * 80)
print("PHASE 1 — ACTIVE DATA SCHEMA VERIFICATION")
print("=" * 80)

global_errors = 0

for domain, directory in DOMAINS.items():

    print(f"\n{domain.upper()}")
    print("-" * 80)

    files = active_files(directory)

    checked = 0
    errors = 0

    for path in files:

        if path.name.endswith(".gz"):
            opener = gzip.open
        else:
            opener = open

        with opener(path, "rt", encoding="utf-8") as f:

            for line in f:

                if not line.strip():
                    continue

                try:
                    record = json.loads(line)
                except Exception:
                    errors += 1
                    continue

                missing = REQUIRED - set(record.keys())

                if missing:
                    errors += 1

                    if errors <= 5:
                        print(
                            f"Missing fields in {path.name}: "
                            f"{sorted(missing)}"
                        )

                checked += 1

                # We only need a sample for schema verification.
                if checked >= 1000:
                    break

    global_errors += errors

    print(f"Records sampled : {checked:,}")
    print(f"Schema errors   : {errors:,}")

    if errors == 0:
        print("✓ Schema check passed")
    else:
        print("✗ Schema errors found")

print("\n" + "=" * 80)

if global_errors == 0:
    print("✓ ACTIVE DATA SCHEMA VERIFICATION PASSED")
else:
    print(f"✗ TOTAL SCHEMA ERRORS: {global_errors}")

print("=" * 80)

In [ ]:
# ================================================================
# PHASE 2 — CANONICALIZATION
# CELL 1: CONFIGURATION
# ================================================================

from pathlib import Path
import json
import gzip
import hashlib
import re
import os
import time
from collections import Counter

BASE = Path("/kaggle/working/phase1_processed")

DOMAINS = {
    "acts": BASE / "acts",
    "high_courts": BASE / "high_courts",
    "supreme_court": BASE / "supreme_court",
    "tribunals": BASE / "tribunals",
}

EXPECTED_COUNTS = {
    "acts": 18_732,
    "high_courts": 2_536_738,
    "supreme_court": 55_363,
    "tribunals": 864_198,
}

PHASE2 = Path("/kaggle/working/phase2_canonical")
PHASE2.mkdir(parents=True, exist_ok=True)

CANONICAL_DIR = PHASE2 / "corpus"
CANONICAL_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = PHASE2 / "manifest.json"

print("=" * 80)
print("PHASE 2 — CANONICALIZATION CONFIGURATION")
print("=" * 80)

for domain, path in DOMAINS.items():
    print(f"{domain:16s}: {path}")
    assert path.exists(), f"Missing active dataset: {path}"

print("-" * 80)
print(f"Output directory : {PHASE2}")
print(f"Canonical corpus : {CANONICAL_DIR}")
print("=" * 80)
print("✓ All Phase 1 active directories found")

In [ ]:
# ================================================================
# PHASE 2 — CANONICALIZATION
# CELL 2: ACTIVE FILE DISCOVERY
# ================================================================

def active_files(domain_dir):
    """
    Return ONLY active corpus files.

    ACTS:
        documents.jsonl

    Sharded domains:
        shard_*.jsonl.gz

    Explicitly excludes:
        quarantine*
        manifest*
        backup directories
        temporary directories
    """
    if domain_dir.name == "acts":
        files = sorted(domain_dir.glob("documents.jsonl"))
    else:
        files = sorted(domain_dir.glob("shard_*.jsonl.gz"))

    return files


print("=" * 80)
print("PHASE 2 — ACTIVE FILE DISCOVERY")
print("=" * 80)

for domain, path in DOMAINS.items():
    files = active_files(path)

    print()
    print(f"{domain.upper()}")
    print("-" * 80)
    print(f"Files found: {len(files)}")

    assert len(files) > 0, f"No active corpus files found for {domain}"

    for f in files[:5]:
        print(f"  {f.name}")

    if len(files) > 5:
        print(f"  ... {len(files) - 5} more")

print()
print("=" * 80)
print("✓ ACTIVE FILE DISCOVERY PASSED")
print("=" * 80)

In [ ]:
# ================================================================
# PHASE 2 — CANONICALIZATION
# CELL 3: CANONICAL SCHEMA
# ================================================================

CANONICAL_FIELDS = [
    "document_id",
    "source_domain",
    "title",
    "court",
    "category",
    "year",
    "text",
    "text_hash",
    "word_count",
    "char_count",
    "source_file",
]

print("=" * 80)
print("PHASE 2 — CANONICAL SCHEMA")
print("=" * 80)

for i, field in enumerate(CANONICAL_FIELDS, 1):
    print(f"{i:2d}. {field}")

print("=" * 80)
print("✓ Canonical schema defined")

In [ ]:
# ================================================================
# PHASE 2 — CANONICALIZATION
# CELL 4: DOMAIN METADATA MAPPING
# ================================================================

def first_nonempty(record, fields, default=None):
    for field in fields:
        value = record.get(field)
        if value is not None and value != "":
            return value
    return default


def canonicalize_record(record, domain, source_file):
    """
    Convert one domain-specific record into the canonical schema.
    """

    document_id = first_nonempty(
        record,
        ["document_id", "id"]
    )

    text = record.get("text")

    if text is None:
        text = ""

    # Preserve existing title where available.
    title = first_nonempty(
        record,
        ["title"]
    )

    # Court information exists primarily in judgments.
    court = first_nonempty(
        record,
        ["court"]
    )

    # Acts have category; other domains may not.
    category = first_nonempty(
        record,
        ["category"]
    )

    year = first_nonempty(
        record,
        ["year"]
    )

    # Prefer the already stored hash if valid.
    text_hash = record.get("text_hash")

    if not text_hash:
        text_hash = hashlib.sha256(
            text.encode("utf-8", errors="replace")
        ).hexdigest()

    # We calculate these from the actual active text.
    char_count = len(text)
    word_count = len(text.split())

    return {
        "document_id": document_id,
        "source_domain": domain,
        "title": title,
        "court": court,
        "category": category,
        "year": year,
        "text": text,
        "text_hash": text_hash,
        "word_count": word_count,
        "char_count": char_count,
        "source_file": source_file,
    }


print("=" * 80)
print("✓ Canonicalization function defined")
print("=" * 80)

In [ ]:
# ================================================================
# PHASE 2 — CANONICALIZATION
# CELL 5: SMALL-SAMPLE CANONICALIZATION TEST
# ================================================================

def read_first_records(path, n=3):
    records = []

    if path.suffix == ".gz":
        opener = gzip.open
        mode = "rt"
    else:
        opener = open
        mode = "r"

    with opener(path, mode, encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            records.append(json.loads(line))

            if len(records) >= n:
                break

    return records


print("=" * 80)
print("PHASE 2 — CANONICALIZATION SAMPLE TEST")
print("=" * 80)

for domain, path in DOMAINS.items():

    files = active_files(path)
    sample_file = files[0]

    records = read_first_records(sample_file, n=2)

    print()
    print(f"{domain.upper()}")
    print("-" * 80)

    for i, record in enumerate(records, 1):

        canonical = canonicalize_record(
            record,
            domain,
            sample_file.name
        )

        print(f"Record {i}")
        print(f"  document_id : {canonical['document_id']}")
        print(f"  domain      : {canonical['source_domain']}")
        print(f"  title       : {canonical['title']}")
        print(f"  court       : {canonical['court']}")
        print(f"  category    : {canonical['category']}")
        print(f"  year        : {canonical['year']}")
        print(f"  words       : {canonical['word_count']}")
        print(f"  chars       : {canonical['char_count']}")
        print(f"  text_hash   : {canonical['text_hash'][:16]}...")
        print()

print("=" * 80)
print("✓ SAMPLE CANONICALIZATION PASSED")
print("=" * 80)

In [ ]:
# ================================================================
# PHASE 2 — CANONICALIZATION
# CELL 6: SCHEMA VALIDATION
# ================================================================

def validate_canonical(record):
    errors = []

    if set(record.keys()) != set(CANONICAL_FIELDS):
        errors.append("schema")

    if not record["document_id"]:
        errors.append("missing_document_id")

    if not isinstance(record["text"], str):
        errors.append("invalid_text")

    if not record["text"]:
        errors.append("empty_text")

    if not isinstance(record["word_count"], int):
        errors.append("invalid_word_count")

    if not isinstance(record["char_count"], int):
        errors.append("invalid_char_count")

    if record["char_count"] != len(record["text"]):
        errors.append("char_count_mismatch")

    if record["word_count"] != len(record["text"].split()):
        errors.append("word_count_mismatch")

    return errors


print("=" * 80)
print("PHASE 2 — CANONICAL SCHEMA VALIDATION")
print("=" * 80)

total_tested = 0
total_errors = 0

for domain, path in DOMAINS.items():

    files = active_files(path)

    tested = 0
    errors = 0

    for source_file in files[:2]:

        records = read_first_records(source_file, n=10)

        for record in records:

            canonical = canonicalize_record(
                record,
                domain,
                source_file.name
            )

            validation_errors = validate_canonical(canonical)

            tested += 1
            total_tested += 1

            if validation_errors:
                errors += 1
                total_errors += 1

                print(
                    f"{domain}: "
                    f"{canonical.get('document_id')} -> "
                    f"{validation_errors}"
                )

    print(f"{domain.upper():16s} tested={tested:3d} errors={errors:3d}")

print()
print("=" * 80)

if total_errors == 0:
    print(f"✓ VALIDATION PASSED — {total_tested} records tested")
else:
    print(f"✗ VALIDATION FAILED — {total_errors} errors")

print("=" * 80)

assert total_errors == 0

In [ ]:
# ================================================================
# PHASE 2 — CANONICALIZATION
# CELL 7: FULL STREAMING CANONICAL CORPUS GENERATION
# ================================================================

import shutil
import json
import gzip
import time
from pathlib import Path

TEMP_CANONICAL = PHASE2 / "corpus_temp"

# Start clean only if an incomplete previous run exists.
if TEMP_CANONICAL.exists():
    print(f"Removing previous temporary output: {TEMP_CANONICAL}")
    shutil.rmtree(TEMP_CANONICAL)

TEMP_CANONICAL.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("PHASE 2 — FULL STREAMING CANONICALIZATION")
print("=" * 80)
print(f"Temporary output : {TEMP_CANONICAL}")
print()

global_counts = {}

start_time = time.time()


def open_jsonl(path):
    if path.suffix == ".gz":
        return gzip.open(path, "rt", encoding="utf-8")
    return open(path, "r", encoding="utf-8")


for domain, domain_dir in DOMAINS.items():

    files = active_files(domain_dir)

    domain_out = TEMP_CANONICAL / domain
    domain_out.mkdir(parents=True, exist_ok=True)

    domain_total = 0
    domain_errors = 0

    print()
    print("=" * 80)
    print(f"DOMAIN: {domain.upper()}")
    print("=" * 80)

    for file_idx, source_file in enumerate(files, 1):

        output_name = source_file.name
        output_file = domain_out / output_name

        records = 0
        errors = 0

        print(
            f"[{file_idx}/{len(files)}] "
            f"{source_file.name}"
        )

        with open_jsonl(source_file) as fin, \
             gzip.open(output_file, "wt", encoding="utf-8") as fout:

            for line_number, line in enumerate(fin, 1):

                line = line.strip()

                if not line:
                    continue

                try:
                    raw = json.loads(line)

                    canonical = canonicalize_record(
                        raw,
                        domain,
                        source_file.name
                    )

                    validation_errors = validate_canonical(canonical)

                    if validation_errors:
                        errors += 1
                        domain_errors += 1

                        if errors <= 5:
                            print(
                                f"  ERROR line {line_number}: "
                                f"{validation_errors}"
                            )

                        continue

                    fout.write(
                        json.dumps(
                            canonical,
                            ensure_ascii=False,
                            separators=(",", ":")
                        )
                        + "\n"
                    )

                    records += 1
                    domain_total += 1

                except Exception as exc:
                    errors += 1
                    domain_errors += 1

                    if errors <= 5:
                        print(
                            f"  ERROR line {line_number}: "
                            f"{type(exc).__name__}: {exc}"
                        )

        print(
            f"    Records written : {records:,}"
            f" | Errors : {errors:,}"
        )

    global_counts[domain] = {
        "expected": EXPECTED_COUNTS[domain],
        "written": domain_total,
        "errors": domain_errors,
        "status": domain_total == EXPECTED_COUNTS[domain]
                   and domain_errors == 0,
    }

    print()
    print(
        f"{domain.upper()} TOTAL : "
        f"{domain_total:,} / {EXPECTED_COUNTS[domain]:,}"
    )

elapsed = time.time() - start_time

print()
print("=" * 80)
print("FULL CANONICALIZATION COMPLETE")
print("=" * 80)

for domain, result in global_counts.items():
    status = "✓" if result["status"] else "✗"

    print(
        f"{status} "
        f"{domain:16s} "
        f"{result['written']:,} / "
        f"{result['expected']:,} "
        f"| errors={result['errors']:,}"
    )

print("-" * 80)
print(f"Elapsed time : {elapsed / 60:.2f} minutes")
print(f"Output       : {TEMP_CANONICAL}")
print("=" * 80)

assert all(
    r["status"]
    for r in global_counts.values()
), "Canonicalization validation failed"

In [ ]:
# ================================================================
# PHASE 2 — CANONICALIZATION
# CELL 8: RECORD COUNT VERIFICATION
# ================================================================

print("=" * 80)
print("PHASE 2 — CANONICAL CORPUS RECORD COUNT VERIFICATION")
print("=" * 80)

canonical_counts = {}

for domain, expected in EXPECTED_COUNTS.items():

    domain_dir = TEMP_CANONICAL / domain

    files = (
        sorted(domain_dir.glob("*.jsonl"))
        + sorted(domain_dir.glob("*.jsonl.gz"))
    )

    total = 0

    print()
    print(domain.upper())
    print("-" * 80)

    for file in files:

        count = 0

        with gzip.open(file, "rt", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    count += 1

        total += count

        print(f"  {file.name:30s} : {count:,}")

    canonical_counts[domain] = total

    print(f"  TOTAL : {total:,}")
    print(f"  EXPECTED : {expected:,}")

    if total == expected:
        print("  ✓ Count correct")
    else:
        print(
            f"  ✗ COUNT MISMATCH "
            f"({total - expected:+,})"
        )

print()
print("=" * 80)

assert canonical_counts == EXPECTED_COUNTS, \
    "Canonical corpus record counts do not match Phase 1"

print("✓ ALL CANONICAL RECORD COUNTS VERIFIED")
print("=" * 80)

In [ ]:
# ================================================================
# PHASE 2 — CANONICALIZATION
# CELL 9: FULL SCHEMA SAMPLE VERIFICATION
# ================================================================

import random

REQUIRED_CANONICAL_FIELDS = set(CANONICAL_FIELDS)

SAMPLE_PER_DOMAIN = 100

print("=" * 80)
print("PHASE 2 — CANONICAL SCHEMA VERIFICATION")
print("=" * 80)

total_checked = 0
total_errors = 0

for domain, expected in EXPECTED_COUNTS.items():

    domain_dir = TEMP_CANONICAL / domain

    files = (
        sorted(domain_dir.glob("*.jsonl"))
        + sorted(domain_dir.glob("*.jsonl.gz"))
    )

    checked = 0
    errors = 0

    print()
    print(domain.upper())
    print("-" * 80)

    for file in files:

        if checked >= SAMPLE_PER_DOMAIN:
            break

        with gzip.open(file, "rt", encoding="utf-8") as f:

            for line in f:

                if checked >= SAMPLE_PER_DOMAIN:
                    break

                if not line.strip():
                    continue

                record = json.loads(line)

                missing = REQUIRED_CANONICAL_FIELDS - set(record)

                if missing:
                    errors += 1
                    print(
                        f"Missing fields in {file.name}: "
                        f"{sorted(missing)}"
                    )

                validation_errors = validate_canonical(record)

                if validation_errors:
                    errors += 1
                    print(
                        f"Validation error in {file.name}: "
                        f"{validation_errors}"
                    )

                checked += 1

    total_checked += checked
    total_errors += errors

    print(f"Records checked : {checked}")
    print(f"Schema errors   : {errors}")

    if errors == 0:
        print("✓ Schema check passed")
    else:
        print("✗ Schema check failed")

print()
print("=" * 80)

if total_errors == 0:
    print(
        f"✓ CANONICAL SCHEMA VERIFIED "
        f"({total_checked:,} records)"
    )
else:
    print(f"✗ TOTAL ERRORS: {total_errors:,}")

print("=" * 80)

assert total_errors == 0

In [ ]:
# ================================================================
# PHASE 2 — CANONICALIZATION
# CELL 10: DOCUMENT-ID PRESERVATION CHECK
# ================================================================

print("=" * 80)
print("PHASE 2 — DOCUMENT-ID PRESERVATION CHECK")
print("=" * 80)

for domain, expected in EXPECTED_COUNTS.items():

    domain_dir = TEMP_CANONICAL / domain

    files = (
        sorted(domain_dir.glob("*.jsonl"))
        + sorted(domain_dir.glob("*.jsonl.gz"))
    )

    unique_ids = set()
    total = 0
    missing_ids = 0
    duplicate_ids = 0

    for file in files:

        with gzip.open(file, "rt", encoding="utf-8") as f:

            for line in f:

                if not line.strip():
                    continue

                record = json.loads(line)

                document_id = record.get("document_id")

                if not document_id:
                    missing_ids += 1
                    continue

                if document_id in unique_ids:
                    duplicate_ids += 1
                else:
                    unique_ids.add(document_id)

                total += 1

    print()
    print(domain.upper())
    print("-" * 80)
    print(f"Records        : {total:,}")
    print(f"Unique IDs     : {len(unique_ids):,}")
    print(f"Missing IDs    : {missing_ids:,}")
    print(f"Duplicate IDs  : {duplicate_ids:,}")

    assert total == expected
    assert missing_ids == 0
    assert duplicate_ids == 0

    print("✓ Document IDs verified")

print()
print("=" * 80)
print("✓ DOCUMENT-ID PRESERVATION PASSED")
print("=" * 80)

In [ ]:
from pathlib import Path

PHASE2_ROOT = Path("/kaggle/working/phase2_canonical")
CORPUS = PHASE2_ROOT / "corpus_temp"

if not CORPUS.exists():
    raise RuntimeError(f"Canonical corpus not found: {CORPUS}")

domains = {
    "acts": CORPUS / "acts",
    "high_courts": CORPUS / "high_courts",
    "supreme_court": CORPUS / "supreme_court",
    "tribunals": CORPUS / "tribunals",
}

print("=" * 80)
print("PHASE 3 — INPUT CONFIGURATION")
print("=" * 80)

for domain, path in domains.items():
    print(f"{domain:16} : {path}")
    if not path.exists():
        raise RuntimeError(f"Missing canonical domain: {path}")

print("-" * 80)
print("✓ Phase 2 canonical corpus found")
print("✓ All four domains present")
print("✓ Phase 3 input frozen")

In [ ]:
from pathlib import Path

CORPUS_ROOT = Path("/kaggle/working/phase2_canonical/corpus_temp")

acts_file = CORPUS_ROOT / "acts" / "documents.jsonl"

with open(acts_file, "rb") as f:
    first_bytes = f.read(16)

print("=" * 80)
print("FILE FORMAT DIAGNOSTIC")
print("=" * 80)
print("File :", acts_file)
print("Exists:", acts_file.exists())
print("First bytes:", first_bytes)
print("Hex :", first_bytes.hex())

if first_bytes[:2] == b"\x1f\x8b":
    print("\n⚠ FILE IS GZIP-COMPRESSED")
    print("But filename is .jsonl")
elif first_bytes.startswith(b"{") or first_bytes.startswith(b"["):
    print("\n✓ FILE LOOKS LIKE PLAIN JSON")
else:
    print("\n⚠ UNKNOWN FILE FORMAT")

In [ ]:
import gzip
import json


def read_jsonl_file(path):
    """
    Automatically detects gzip using the actual file header.

    Supports:
      - plain UTF-8 JSONL
      - gzip-compressed JSONL
    """

    # --------------------------------------------------------
    # Detect gzip from magic bytes
    # --------------------------------------------------------
    with open(path, "rb") as raw:
        magic = raw.read(2)

    is_gzip = magic == b"\x1f\x8b"

    # --------------------------------------------------------
    # Open appropriately
    # --------------------------------------------------------
    if is_gzip:
        opener = gzip.open
    else:
        opener = open

    with opener(path, "rt", encoding="utf-8") as f:

        for line_no, line in enumerate(f, 1):

            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as e:
                raise RuntimeError(
                    f"Malformed JSON\n"
                    f"File : {path}\n"
                    f"Line : {line_no}\n"
                    f"Error: {e}"
                ) from e

            yield line_no, record

In [ ]:
from pathlib import Path

acts_file = Path(
    "/kaggle/working/phase2_canonical/corpus_temp/acts/documents.jsonl"
)

print("=" * 80)
print("TESTING ACTS CANONICAL FILE")
print("=" * 80)

count = 0

for line_no, rec in read_jsonl_file(acts_file):

    count += 1

    if count <= 3:
        print(f"\nRecord {count}")
        print("Line:", line_no)
        print("ID:", rec.get("document_id"))
        print("Domain:", rec.get("source_domain"))
        print("Words:", rec.get("word_count"))

print("\n" + "=" * 80)
print("ACTS TEST COMPLETE")
print("=" * 80)
print("Records read:", count)
print("Expected    :", 18_732)

if count == 18_732:
    print("✓ ACTS FILE READ SUCCESSFULLY")
else:
    print("✗ COUNT MISMATCH")

In [ ]:
# ============================================================
# PHASE 2 — ACTS COMPRESSED-FILE VALIDATION
# ============================================================

from pathlib import Path
import gzip
import json

ACTS_FILE = Path(
    "/kaggle/working/phase2_canonical/corpus_temp/acts/documents.jsonl"
)

print("=" * 80)
print("PHASE 2 — ACTS COMPRESSED-FILE VALIDATION")
print("=" * 80)

count = 0

with gzip.open(ACTS_FILE, "rt", encoding="utf-8") as f:
    for line_no, line in enumerate(f, 1):

        line = line.strip()

        if not line:
            continue

        rec = json.loads(line)

        count += 1

        if count <= 3:
            print(f"\nRecord {count}")
            print("  line        :", line_no)
            print("  document_id :", rec.get("document_id"))
            print("  domain      :", rec.get("source_domain"))
            print("  word_count  :", rec.get("word_count"))
            print("  char_count  :", rec.get("char_count"))

print("\n" + "=" * 80)
print("RESULT")
print("=" * 80)

print("Records read :", count)
print("Expected     :", 18_732)

if count == 18_732:
    print("✓ Acts gzip file is valid")
    print("✓ All 18,732 canonical Acts records are readable")
else:
    raise RuntimeError(
        f"Acts count mismatch: expected 18,732, got {count}"
    )

print("=" * 80)

In [ ]:
# ============================================================
# PHASE 2 — CORRECT ACTS FILE EXTENSION
# ============================================================

from pathlib import Path

ACTS_DIR = Path(
    "/kaggle/working/phase2_canonical/corpus_temp/acts"
)

OLD = ACTS_DIR / "documents.jsonl"
NEW = ACTS_DIR / "documents.jsonl.gz"

print("=" * 80)
print("PHASE 2 — CORRECTING ACTS FILE EXTENSION")
print("=" * 80)

if not OLD.exists():
    raise RuntimeError(f"Expected file not found: {OLD}")

if NEW.exists():
    raise RuntimeError(
        f"Target already exists: {NEW}\n"
        "Stopping to avoid overwriting anything."
    )

OLD.rename(NEW)

print("Old :", OLD)
print("New :", NEW)

print("\n✓ File renamed")
print("✓ Contents were not modified")
print("✓ Compression format now matches filename")
print("=" * 80)

In [ ]:
import gzip
import json


def read_jsonl_file(path):
    """
    Detect gzip from actual file bytes rather than filename.
    """

    with open(path, "rb") as raw:
        magic = raw.read(2)

    if magic == b"\x1f\x8b":
        opener = gzip.open
    else:
        opener = open

    with opener(path, "rt", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):

            line = line.strip()

            if not line:
                continue

            yield line_no, json.loads(line)

In [ ]:
from pathlib import Path

ACTS_DIR = Path(
    "/kaggle/working/phase2_canonical/corpus_temp/acts"
)

print("=" * 80)
print("PHASE 2 — ACTS CANONICAL DIRECTORY STATE")
print("=" * 80)

print("Directory :", ACTS_DIR)
print("Exists    :", ACTS_DIR.exists())
print("Is dir    :", ACTS_DIR.is_dir() if ACTS_DIR.exists() else False)

if not ACTS_DIR.exists():
    raise RuntimeError(f"Acts directory missing: {ACTS_DIR}")

files = sorted(p for p in ACTS_DIR.iterdir() if p.is_file())

print("\nFiles:")
print("-" * 80)

for p in files:
    size_mb = p.stat().st_size / (1024 * 1024)

    with open(p, "rb") as f:
        magic = f.read(2)

    compression = "GZIP" if magic == b"\x1f\x8b" else "PLAIN/OTHER"

    print(
        f"{p.name:<40} "
        f"{size_mb:>10.2f} MB   "
        f"{compression}"
    )

print("\n" + "=" * 80)
print("DIAGNOSTIC COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 2 — ACTS CANONICAL FILE VALIDATION
# ============================================================

from pathlib import Path
import gzip
import json

ACTS_FILE = Path(
    "/kaggle/working/phase2_canonical/corpus_temp/acts/documents.jsonl.gz"
)

EXPECTED = 18_732

print("=" * 80)
print("PHASE 2 — ACTS CANONICAL FILE VALIDATION")
print("=" * 80)

count = 0

with gzip.open(ACTS_FILE, "rt", encoding="utf-8") as f:

    for line_no, line in enumerate(f, 1):

        line = line.strip()

        if not line:
            continue

        rec = json.loads(line)

        count += 1

        if count <= 3:
            print(f"\nRecord {count}")
            print("  line        :", line_no)
            print("  document_id :", rec.get("document_id"))
            print("  domain      :", rec.get("source_domain"))
            print("  word_count  :", rec.get("word_count"))
            print("  char_count  :", rec.get("char_count"))

print("\n" + "=" * 80)
print("RESULT")
print("=" * 80)

print(f"Records read : {count:,}")
print(f"Expected     : {EXPECTED:,}")

if count != EXPECTED:
    raise RuntimeError(
        f"Acts count mismatch: expected {EXPECTED:,}, got {count:,}"
    )

print("✓ Acts canonical file is valid")
print("✓ All 18,732 records are readable")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 2 — FULL CANONICAL CORPUS INTEGRITY VERIFICATION
# ============================================================

from pathlib import Path
import gzip
import json

CORPUS_ROOT = Path(
    "/kaggle/working/phase2_canonical/corpus_temp"
)

EXPECTED = {
    "acts": 18_732,
    "high_courts": 2_536_738,
    "supreme_court": 55_363,
    "tribunals": 864_198,
}

REQUIRED_FIELDS = {
    "document_id",
    "source_domain",
    "title",
    "court",
    "category",
    "year",
    "text",
    "text_hash",
    "word_count",
    "char_count",
    "source_file",
}


# ------------------------------------------------------------
# UNIVERSAL JSONL READER
# Detects gzip from actual file bytes.
# ------------------------------------------------------------

def read_jsonl_file(path):

    with open(path, "rb") as raw:
        magic = raw.read(2)

    is_gzip = magic == b"\x1f\x8b"

    opener = gzip.open if is_gzip else open

    with opener(path, "rt", encoding="utf-8") as f:

        for line_no, line in enumerate(f, 1):

            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as e:
                raise RuntimeError(
                    f"\nMalformed JSON\n"
                    f"File : {path}\n"
                    f"Line : {line_no}\n"
                    f"Error: {e}"
                ) from e

            yield line_no, record


# ------------------------------------------------------------
# FILE DISCOVERY
# ------------------------------------------------------------

def get_data_files(domain_dir):

    return sorted(
        p for p in domain_dir.iterdir()
        if p.is_file()
        and (
            p.name.endswith(".jsonl")
            or p.name.endswith(".jsonl.gz")
        )
    )


# ------------------------------------------------------------
# START VERIFICATION
# ------------------------------------------------------------

print("=" * 80)
print("PHASE 2 — FULL CANONICAL CORPUS INTEGRITY VERIFICATION")
print("=" * 80)

if not CORPUS_ROOT.exists():
    raise RuntimeError(
        f"Canonical corpus missing:\n{CORPUS_ROOT}"
    )

grand_total = 0


# ============================================================
# DOMAIN LOOP
# ============================================================

for domain, expected_count in EXPECTED.items():

    domain_dir = CORPUS_ROOT / domain

    print(f"\n{domain.upper()}")
    print("-" * 80)

    if not domain_dir.exists():
        raise RuntimeError(
            f"Missing domain directory:\n{domain_dir}"
        )

    files = get_data_files(domain_dir)

    if not files:
        raise RuntimeError(
            f"No JSONL files found in:\n{domain_dir}"
        )

    print(f"Files found : {len(files)}")

    domain_total = 0
    schema_errors = 0

    for path in files:

        file_total = 0

        for line_no, rec in read_jsonl_file(path):

            file_total += 1
            domain_total += 1

            # ------------------------------------------------
            # Required fields
            # ------------------------------------------------

            missing = REQUIRED_FIELDS - set(rec.keys())

            if missing:

                schema_errors += 1

                if schema_errors <= 5:
                    print(
                        f"  Schema error: "
                        f"{path.name} "
                        f"line {line_no} "
                        f"missing={sorted(missing)}"
                    )

            # ------------------------------------------------
            # Basic type checks
            # ------------------------------------------------

            if not isinstance(rec.get("document_id"), str):
                schema_errors += 1

            if not isinstance(rec.get("text"), str):
                schema_errors += 1

            if not isinstance(rec.get("text_hash"), str):
                schema_errors += 1

        print(
            f"  {path.name:<35} : "
            f"{file_total:,}"
        )

    # --------------------------------------------------------
    # Domain count
    # --------------------------------------------------------

    print(f"\n  TOTAL    : {domain_total:,}")
    print(f"  EXPECTED : {expected_count:,}")

    if domain_total != expected_count:

        raise RuntimeError(
            f"\n{domain.upper()} RECORD COUNT MISMATCH\n"
            f"Expected : {expected_count:,}\n"
            f"Actual   : {domain_total:,}"
        )

    if schema_errors != 0:

        raise RuntimeError(
            f"\n{domain.upper()} SCHEMA ERRORS: "
            f"{schema_errors}"
        )

    print("  ✓ Count correct")
    print("  ✓ Schema correct")

    grand_total += domain_total


# ============================================================
# GRAND TOTAL
# ============================================================

expected_total = sum(EXPECTED.values())

print("\n" + "=" * 80)
print("FULL CORPUS RESULT")
print("=" * 80)

print(f"Expected total : {expected_total:,}")
print(f"Actual total   : {grand_total:,}")

if grand_total != expected_total:

    raise RuntimeError(
        f"Grand total mismatch: "
        f"expected {expected_total:,}, "
        f"got {grand_total:,}"
    )

print()
print("✓ ALL CANONICAL RECORD COUNTS VERIFIED")
print("✓ ALL REQUIRED FIELDS VERIFIED")
print("✓ JSONL / GZIP FILES READ SUCCESSFULLY")
print("✓ CANONICAL CORPUS INTEGRITY PASSED")

print("=" * 80)

In [ ]:
import os
import shutil
from pathlib import Path

print("=" * 80)
print("KAGGLE STORAGE — CURRENT STATE")
print("=" * 80)

# Overall filesystem storage
total, used, free = shutil.disk_usage("/kaggle/working")

def gb(x):
    return x / (1024 ** 3)

print("\nOVERALL /kaggle/working STORAGE")
print("-" * 80)
print(f"Total : {gb(total):,.2f} GB")
print(f"Used  : {gb(used):,.2f} GB")
print(f"Free  : {gb(free):,.2f} GB")

# Show major project directories
ROOT = Path("/kaggle/working")

print("\nMAJOR DIRECTORIES")
print("-" * 80)

targets = [
    ROOT / "phase1_processed",
    ROOT / "phase2_canonical",
]

for path in targets:
    if not path.exists():
        print(f"{path} : NOT FOUND")
        continue

    size = 0
    for p in path.rglob("*"):
        if p.is_file():
            try:
                size += p.stat().st_size
            except OSError:
                pass

    print(f"{path}")
    print(f"  Size : {gb(size):,.2f} GB")

print("\nPHASE 1 CONTENTS")
print("-" * 80)

p1 = ROOT / "phase1_processed"

if p1.exists():
    for p in sorted(p1.iterdir()):
        if p.is_dir():
            size = sum(
                f.stat().st_size
                for f in p.rglob("*")
                if f.is_file()
            )
            print(f"{p.name:<45} {gb(size):>10.2f} GB")

print("\nPHASE 2 CONTENTS")
print("-" * 80)

p2 = ROOT / "phase2_canonical"

if p2.exists():
    for p in sorted(p2.iterdir()):
        if p.is_dir():
            size = sum(
                f.stat().st_size
                for f in p.rglob("*")
                if f.is_file()
            )
            print(f"{p.name:<45} {gb(size):>10.2f} GB")

print("=" * 80)

In [ ]:
from pathlib import Path

CORPUS = Path("/kaggle/working/phase2_canonical/corpus")

print("=" * 80)
print("PHASE 2 — CORPUS DIRECTORY CHECK")
print("=" * 80)

print(f"Path   : {CORPUS}")
print(f"Exists : {CORPUS.exists()}")
print(f"Is dir : {CORPUS.is_dir()}")

if CORPUS.exists() and CORPUS.is_dir():
    items = list(CORPUS.iterdir())

    print(f"\nItems found : {len(items)}")

    if items:
        for item in items[:50]:
            print(f"  {item.name}")
    else:
        print("  ✓ Directory is empty")

print("=" * 80)

In [ ]:
from pathlib import Path

CORPUS = Path("/kaggle/working/phase2_canonical/corpus")

print("=" * 80)
print("REMOVING EMPTY PHASE 2 CORPUS DIRECTORY")
print("=" * 80)

if not CORPUS.exists():
    print("✓ Directory already absent")
elif not CORPUS.is_dir():
    raise RuntimeError(f"Expected directory, found: {CORPUS}")
elif any(CORPUS.iterdir()):
    raise RuntimeError(
        f"SAFETY STOP: {CORPUS} is not empty. Nothing was deleted."
    )
else:
    CORPUS.rmdir()
    print(f"✓ Removed empty directory: {CORPUS}")

print("=" * 80)

In [ ]:
import shutil

total, used, free = shutil.disk_usage("/kaggle/working")

def gb(x):
    return x / (1024 ** 3)

print("=" * 80)
print("KAGGLE STORAGE AFTER CLEANUP")
print("=" * 80)
print(f"Total : {gb(total):.2f} GB")
print(f"Used  : {gb(used):.2f} GB")
print(f"Free  : {gb(free):.2f} GB")
print("=" * 80)

In [ ]:
from pathlib import Path
import gzip
import json
import os
from collections import Counter, defaultdict

# ============================================================
# PHASE 3 — CANONICAL CORPUS PROFILING
# ============================================================

CORPUS = Path("/kaggle/working/phase2_canonical/corpus_temp")

DOMAINS = [
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals",
]

EXPECTED_COUNTS = {
    "acts": 18_732,
    "high_courts": 2_536_738,
    "supreme_court": 55_363,
    "tribunals": 864_198,
}

if not CORPUS.exists():
    raise RuntimeError(f"Canonical corpus not found: {CORPUS}")

print("=" * 80)
print("PHASE 3 — CANONICAL CORPUS PROFILING")
print("=" * 80)
print(f"Input corpus : {CORPUS}")
print("Mode         : READ-ONLY")
print("=" * 80)

In [ ]:
from pathlib import Path
import gzip
import json
import os
from collections import Counter, defaultdict

# ============================================================
# PHASE 3 — CANONICAL CORPUS PROFILING
# ============================================================

CORPUS = Path("/kaggle/working/phase2_canonical/corpus_temp")

DOMAINS = [
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals",
]

EXPECTED_COUNTS = {
    "acts": 18_732,
    "high_courts": 2_536_738,
    "supreme_court": 55_363,
    "tribunals": 864_198,
}

if not CORPUS.exists():
    raise RuntimeError(f"Canonical corpus not found: {CORPUS}")

print("=" * 80)
print("PHASE 3 — CANONICAL CORPUS PROFILING")
print("=" * 80)
print(f"Input corpus : {CORPUS}")
print("Mode         : READ-ONLY")
print("=" * 80)

In [ ]:
def get_opener(path):
    if path.name.endswith(".jsonl.gz"):
        return gzip.open
    elif path.name.endswith(".jsonl"):
        return open
    else:
        raise ValueError(f"Unsupported file: {path}")


domain_files = {}

print("=" * 80)
print("PHASE 3 — CANONICAL FILE DISCOVERY")
print("=" * 80)

for domain in DOMAINS:

    domain_dir = CORPUS / domain

    if not domain_dir.exists():
        raise RuntimeError(f"Missing domain directory: {domain_dir}")

    files = sorted(
        p for p in domain_dir.iterdir()
        if p.is_file()
        and (p.name.endswith(".jsonl") or p.name.endswith(".jsonl.gz"))
    )

    if not files:
        raise RuntimeError(f"No JSONL files found for {domain}")

    domain_files[domain] = files

    print(f"\n{domain.upper()}")
    print(f"Files : {len(files)}")

    for path in files:
        print(f"  {path.name}")

print("\n" + "=" * 80)
print("✓ FILE DISCOVERY PASSED")
print("=" * 80)

In [ ]:
print("=" * 80)
print("PHASE 3 — FULL CANONICAL CORPUS PROFILE")
print("=" * 80)

profile = {}

for domain in DOMAINS:

    print(f"\n{'=' * 80}")
    print(f"DOMAIN: {domain.upper()}")
    print(f"{'=' * 80}")

    total = 0
    total_words = 0
    total_chars = 0

    years = Counter()
    courts = Counter()
    categories = Counter()

    missing_title = 0
    missing_court = 0
    missing_category = 0
    missing_year = 0

    min_words = None
    max_words = None

    for file_index, path in enumerate(domain_files[domain], 1):

        opener = get_opener(path)

        with opener(path, "rt", encoding="utf-8") as f:

            for line in f:

                line = line.strip()

                if not line:
                    continue

                rec = json.loads(line)

                total += 1

                words = rec.get("word_count", 0) or 0
                chars = rec.get("char_count", 0) or 0

                total_words += words
                total_chars += chars

                if min_words is None or words < min_words:
                    min_words = words

                if max_words is None or words > max_words:
                    max_words = words

                year = rec.get("year")
                if year is not None:
                    years[str(year)] += 1
                else:
                    missing_year += 1

                court = rec.get("court")
                if court:
                    courts[str(court)] += 1
                else:
                    missing_court += 1

                category = rec.get("category")
                if category:
                    categories[str(category)] += 1
                else:
                    missing_category += 1

                if not rec.get("title"):
                    missing_title += 1

        print(
            f"[{file_index}/{len(domain_files[domain])}] "
            f"{path.name} complete"
        )

    expected = EXPECTED_COUNTS[domain]

    if total != expected:
        raise RuntimeError(
            f"{domain}: expected {expected:,}, found {total:,}"
        )

    avg_words = total_words / total if total else 0
    avg_chars = total_chars / total if total else 0

    profile[domain] = {
        "records": total,
        "total_words": total_words,
        "total_chars": total_chars,
        "avg_words": avg_words,
        "avg_chars": avg_chars,
        "min_words": min_words,
        "max_words": max_words,
        "missing_title": missing_title,
        "missing_court": missing_court,
        "missing_category": missing_category,
        "missing_year": missing_year,
        "years": years,
        "courts": courts,
        "categories": categories,
    }

    print(f"\n{domain.upper()} SUMMARY")
    print(f"Records          : {total:,}")
    print(f"Total words      : {total_words:,}")
    print(f"Total characters : {total_chars:,}")
    print(f"Average words    : {avg_words:,.2f}")
    print(f"Average chars    : {avg_chars:,.2f}")
    print(f"Minimum words    : {min_words:,}")
    print(f"Maximum words    : {max_words:,}")
    print(f"Missing title    : {missing_title:,}")
    print(f"Missing court    : {missing_court:,}")
    print(f"Missing category : {missing_category:,}")
    print(f"Missing year     : {missing_year:,}")

print("\n" + "=" * 80)
print("✓ CANONICAL CORPUS PROFILING COMPLETE")
print("=" * 80)

In [ ]:
print("=" * 80)
print("PHASE 3 — CORPUS PROFILE SUMMARY")
print("=" * 80)

print(
    f"{'DOMAIN':<18}"
    f"{'RECORDS':>14}"
    f"{'AVG WORDS':>14}"
    f"{'MIN':>10}"
    f"{'MAX':>10}"
)

print("-" * 80)

for domain in DOMAINS:

    p = profile[domain]

    print(
        f"{domain:<18}"
        f"{p['records']:>14,}"
        f"{p['avg_words']:>14,.1f}"
        f"{p['min_words']:>10,}"
        f"{p['max_words']:>10,}"
    )

total_records = sum(profile[d]["records"] for d in DOMAINS)
total_words = sum(profile[d]["total_words"] for d in DOMAINS)

print("-" * 80)
print(f"{'TOTAL':<18}{total_records:>14,}{total_words:>14,}")

print("\n" + "=" * 80)
print("✓ PROFILE SUMMARY GENERATED")
print("=" * 80)

In [ ]:
import json

PROFILE_OUTPUT = Path(
    "/kaggle/working/phase2_canonical/corpus_profile.json"
)

serializable_profile = {}

for domain, data in profile.items():
    serializable_profile[domain] = {
        key: (
            dict(value)
            if isinstance(value, Counter)
            else value
        )
        for key, value in data.items()
    }

with open(PROFILE_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(
        serializable_profile,
        f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 80)
print("PHASE 3 — PROFILE CHECKPOINT")
print("=" * 80)
print(f"Profile saved : {PROFILE_OUTPUT}")
print(f"Size          : {PROFILE_OUTPUT.stat().st_size / 1024:.1f} KB")
print("✓ Only small metadata file created")
print("✓ Canonical corpus remains untouched")
print("=" * 80)

In [ ]:
from pathlib import Path
import json
import os

BASE = Path("/kaggle/working")
PHASE2 = BASE / "phase2_canonical"
CORPUS = PHASE2 / "corpus_temp"
PROFILE = PHASE2 / "corpus_profile.json"

print("=" * 80)
print("PHASE 3 — CHECKPOINT VERIFICATION")
print("=" * 80)

assert CORPUS.exists(), f"Canonical corpus missing: {CORPUS}"
assert PROFILE.exists(), f"Profile missing: {PROFILE}"

with open(PROFILE, "r", encoding="utf-8") as f:
    profile = json.load(f)

print(f"Canonical corpus : {CORPUS}")
print(f"Profile file     : {PROFILE}")
print(f"Profile size     : {PROFILE.stat().st_size / 1024:.2f} KB")

print("\nProfile domains:")
for domain in profile:
    print(f"  ✓ {domain}")

print("\n" + "=" * 80)
print("✓ PHASE 3 CHECKPOINT VERIFIED")
print("=" * 80)

In [ ]:
from pathlib import Path
import json

PROFILE = Path("/kaggle/working/phase2_canonical/corpus_profile.json")

with open(PROFILE, "r", encoding="utf-8") as f:
    profile = json.load(f)

print("=" * 80)
print("PHASE 3 — PROFILE JSON STRUCTURE")
print("=" * 80)

for domain, data in profile.items():
    print(f"\n{domain}")
    print("-" * 80)
    print("Keys:")
    for key in data.keys():
        print(f"  {key}")

    print("\nSample values:")
    for key, value in data.items():
        print(f"  {key}: {value}")

print("\n" + "=" * 80)

In [ ]:
from pathlib import Path
import json

PHASE2 = Path("/kaggle/working/phase2_canonical")
PROFILE = PHASE2 / "corpus_profile.json"
SUMMARY = PHASE2 / "phase3_profile_summary.txt"

with open(PROFILE, "r", encoding="utf-8") as f:
    profile = json.load(f)

print("=" * 80)
print("PHASE 3 — CORPUS PROFILE SUMMARY")
print("=" * 80)

total_records = 0
total_words = 0

for domain, d in profile.items():

    records = d.get("records", 0)
    words = d.get("total_words", 0)

    total_records += records
    total_words += words

    print(f"\n{domain.upper()}")
    print("-" * 80)

    for key, value in d.items():
        print(f"{key:25} : {value}")

print("\n" + "=" * 80)
print("TOTAL")
print("=" * 80)
print(f"Records     : {total_records:,}")
print(f"Total words : {total_words:,}")

# Save exactly what exists in the profile,
# without inventing/reconciling field names.
with open(SUMMARY, "w", encoding="utf-8") as f:
    f.write("PHASE 3 — CORPUS PROFILE SUMMARY\n")
    f.write("=" * 80 + "\n\n")

    for domain, d in profile.items():
        f.write(f"{domain.upper()}\n")
        f.write("-" * 80 + "\n")

        for key, value in d.items():
            f.write(f"{key:25} : {value}\n")

        f.write("\n")

    f.write("=" * 80 + "\n")
    f.write(f"TOTAL RECORDS : {total_records:,}\n")
    f.write(f"TOTAL WORDS   : {total_words:,}\n")
    f.write("=" * 80 + "\n")

print("\n" + "=" * 80)
print("✓ PROFILE SUMMARY SAVED")
print("=" * 80)
print(f"File : {SUMMARY}")
print(f"Size : {SUMMARY.stat().st_size / 1024:.2f} KB")

In [ ]:
from pathlib import Path
import json

BASE = Path("/kaggle/working/phase2_canonical")

PROFILE_JSON = BASE / "corpus_profile.json"
PROFILE_TXT = BASE / "phase3_profile_summary.txt"

print("=" * 80)
print("PHASE 3 — CHECKPOINT VERIFICATION")
print("=" * 80)

# Check profile JSON
assert PROFILE_JSON.exists(), f"Missing: {PROFILE_JSON}"

with open(PROFILE_JSON, "r", encoding="utf-8") as f:
    profile = json.load(f)

expected_domains = {
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals",
}

assert set(profile.keys()) >= expected_domains, (
    f"Missing domains. Found: {set(profile.keys())}"
)

required_keys = {
    "records",
    "total_words",
    "total_chars",
    "avg_words",
    "avg_chars",
    "min_words",
    "max_words",
    "missing_title",
    "missing_court",
    "missing_category",
    "missing_year",
    "years",
    "courts",
    "categories",
}

for domain in expected_domains:
    missing = required_keys - set(profile[domain].keys())
    assert not missing, f"{domain} missing keys: {missing}"

# Verify totals
total_records = sum(profile[d]["records"] for d in expected_domains)
total_words = sum(profile[d]["total_words"] for d in expected_domains)
total_chars = sum(profile[d]["total_chars"] for d in expected_domains)

assert total_records == 3_475_031
assert total_words == 2_895_643_546

print()
print("Profile JSON        : ✓")
print("All 4 domains      : ✓")
print("Required fields     : ✓")
print(f"Total records       : {total_records:,}")
print(f"Total words         : {total_words:,}")
print(f"Total characters    : {total_chars:,}")

if PROFILE_TXT.exists():
    print(f"Profile summary     : ✓ {PROFILE_TXT.name}")
else:
    print("Profile summary     : ⚠ not found")

print()
print("=" * 80)
print("✓ PHASE 3 CHECKPOINT VALID")
print("✓ CANONICAL CORPUS NOT MODIFIED")
print("=" * 80)

In [ ]:
from pathlib import Path
import os

BASE = Path("/kaggle/working/phase2_canonical")

print("=" * 80)
print("PHASE 3 — CHECKPOINT STATE")
print("=" * 80)

for p in sorted(BASE.iterdir()):
    size = 0

    if p.is_file():
        size = p.stat().st_size
    elif p.is_dir():
        size = sum(
            f.stat().st_size
            for f in p.rglob("*")
            if f.is_file()
        )

    print(f"{p.name:35s} {size / (1024**3):8.2f} GB")

print()
print("Checkpoint files:")
print(f"  ✓ {BASE / 'corpus_profile.json'}")
print(f"  ✓ {BASE / 'phase3_profile_summary.txt'}")

print()
print("=" * 80)
print("✓ PHASE 3 CHECKPOINT READY")
print("=" * 80)

In [ ]:
from pathlib import Path
import shutil
import json
from datetime import datetime

print("=" * 80)
print("PHASE 4 — TRAINING DATA PREPARATION INITIALIZATION")
print("=" * 80)

ROOT = Path("/kaggle/working")
CANONICAL = ROOT / "phase2_canonical"
CORPUS = CANONICAL / "corpus_temp"

PHASE4 = ROOT / "phase4_training_prep"
PHASE4.mkdir(parents=True, exist_ok=True)

print(f"\nCanonical directory : {CANONICAL}")
print(f"Canonical corpus   : {CORPUS}")
print(f"Phase 4 directory  : {PHASE4}")

assert CANONICAL.exists(), "Canonical directory missing"
assert CORPUS.exists(), "Canonical corpus missing"

total, used, free = shutil.disk_usage(ROOT)

print("\n" + "=" * 80)
print("KAGGLE STORAGE")
print("=" * 80)
print(f"Total : {total / 1024**3:.2f} GB")
print(f"Used  : {used / 1024**3:.2f} GB")
print(f"Free  : {free / 1024**3:.2f} GB")

print("\n" + "=" * 80)
print("CANONICAL CORPUS STATE")
print("=" * 80)

for domain in ["acts", "high_courts", "supreme_court", "tribunals"]:
    path = CORPUS / domain

    files = sorted(path.glob("*.jsonl.gz")) if path.exists() else []

    print(f"\n{domain}")
    print(f"  Directory exists : {path.exists()}")
    print(f"  Files            : {len(files)}")

    for f in files[:3]:
        size_gb = f.stat().st_size / 1024**3
        print(f"    {f.name} : {size_gb:.2f} GB")

    if len(files) > 3:
        print(f"    ... {len(files) - 3} more")

# Save a tiny initialization checkpoint.
state = {
    "phase": 4,
    "stage": "training_data_preparation_initialization",
    "timestamp": datetime.now().isoformat(),
    "canonical_path": str(CORPUS),
    "canonical_untouched": True,
    "free_storage_gb": round(free / 1024**3, 2),
}

state_path = PHASE4 / "phase4_initialization.json"

with open(state_path, "w", encoding="utf-8") as f:
    json.dump(state, f, indent=2)

print("\n" + "=" * 80)
print("PHASE 4 INITIALIZATION COMPLETE")
print("=" * 80)
print(f"Checkpoint : {state_path}")
print("✓ Canonical corpus untouched")
print("✓ No training data generated yet")
print("✓ No tokenization performed")
print("✓ Ready for Phase 4 corpus audit")
print("=" * 80)

In [ ]:
import gzip
import json
from pathlib import Path
from collections import Counter

ROOT = Path("/kaggle/working")
CORPUS = ROOT / "phase2_canonical" / "corpus_temp"
PHASE4 = ROOT / "phase4_training_prep"

DOMAINS = [
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals",
]

def read_jsonl_gz(path):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            yield line_no, json.loads(line)


print("=" * 80)
print("PHASE 4 — CORPUS TEXT QUALITY AUDIT")
print("=" * 80)

global_stats = {
    "records": 0,
    "empty_text": 0,
    "missing_text": 0,
    "non_string_text": 0,
    "short_lt_50_words": 0,
    "short_lt_100_words": 0,
    "long_gt_8000_words": 0,
    "long_gt_16000_words": 0,
    "long_gt_32000_words": 0,
    "long_gt_64000_words": 0,
}

domain_stats = {}

for domain in DOMAINS:

    domain_dir = CORPUS / domain
    files = sorted(domain_dir.glob("*.jsonl.gz"))

    stats = {
        "records": 0,
        "empty_text": 0,
        "missing_text": 0,
        "non_string_text": 0,
        "short_lt_50_words": 0,
        "short_lt_100_words": 0,
        "long_gt_8000_words": 0,
        "long_gt_16000_words": 0,
        "long_gt_32000_words": 0,
        "long_gt_64000_words": 0,
        "word_histogram": Counter(),
    }

    print("\n" + "=" * 80)
    print(f"DOMAIN: {domain.upper()}")
    print("=" * 80)

    for file_idx, path in enumerate(files, 1):

        for line_no, rec in read_jsonl_gz(path):

            stats["records"] += 1
            global_stats["records"] += 1

            text = rec.get("text")

            if text is None:
                stats["missing_text"] += 1
                global_stats["missing_text"] += 1
                continue

            if not isinstance(text, str):
                stats["non_string_text"] += 1
                global_stats["non_string_text"] += 1
                continue

            text = text.strip()

            if not text:
                stats["empty_text"] += 1
                global_stats["empty_text"] += 1
                continue

            words = len(text.split())

            if words < 50:
                stats["short_lt_50_words"] += 1
                global_stats["short_lt_50_words"] += 1

            if words < 100:
                stats["short_lt_100_words"] += 1
                global_stats["short_lt_100_words"] += 1

            if words > 8000:
                stats["long_gt_8000_words"] += 1
                global_stats["long_gt_8000_words"] += 1

            if words > 16000:
                stats["long_gt_16000_words"] += 1
                global_stats["long_gt_16000_words"] += 1

            if words > 32000:
                stats["long_gt_32000_words"] += 1
                global_stats["long_gt_32000_words"] += 1

            if words > 64000:
                stats["long_gt_64000_words"] += 1
                global_stats["long_gt_64000_words"] += 1

            # Coarse distribution for audit purposes.
            if words < 100:
                bucket = "<100"
            elif words < 500:
                bucket = "100-499"
            elif words < 1000:
                bucket = "500-999"
            elif words < 2000:
                bucket = "1000-1999"
            elif words < 4000:
                bucket = "2000-3999"
            elif words < 8000:
                bucket = "4000-7999"
            elif words < 16000:
                bucket = "8000-15999"
            elif words < 32000:
                bucket = "16000-31999"
            elif words < 64000:
                bucket = "32000-63999"
            else:
                bucket = "64000+"

            stats["word_histogram"][bucket] += 1

        print(f"[{file_idx}/{len(files)}] {path.name} complete")

    domain_stats[domain] = stats

    print(f"\n{domain.upper()} SUMMARY")
    print("-" * 80)
    print(f"Records                 : {stats['records']:,}")
    print(f"Missing text            : {stats['missing_text']:,}")
    print(f"Empty text              : {stats['empty_text']:,}")
    print(f"Non-string text         : {stats['non_string_text']:,}")
    print(f"< 50 words              : {stats['short_lt_50_words']:,}")
    print(f"< 100 words             : {stats['short_lt_100_words']:,}")
    print(f"> 8,000 words           : {stats['long_gt_8000_words']:,}")
    print(f"> 16,000 words          : {stats['long_gt_16000_words']:,}")
    print(f"> 32,000 words          : {stats['long_gt_32000_words']:,}")
    print(f"> 64,000 words          : {stats['long_gt_64000_words']:,}")

    print("\nWord-count distribution:")
    for bucket, count in stats["word_histogram"].items():
        print(f"  {bucket:>12} : {count:,}")


print("\n" + "=" * 80)
print("GLOBAL TEXT QUALITY SUMMARY")
print("=" * 80)

for key, value in global_stats.items():
    print(f"{key:25} : {value:,}")

print("\n" + "=" * 80)
print("PHASE 4 AUDIT COMPLETE")
print("=" * 80)

# Save compact machine-readable audit.
audit_output = {
    "global": global_stats,
    "domains": {
        domain: {
            **{k: v for k, v in stats.items() if k != "word_histogram"},
            "word_histogram": dict(stats["word_histogram"]),
        }
        for domain, stats in domain_stats.items()
    },
}

audit_path = PHASE4 / "phase4_text_quality_audit.json"

with open(audit_path, "w", encoding="utf-8") as f:
    json.dump(audit_output, f, indent=2)

print(f"Audit saved : {audit_path}")
print("✓ Canonical corpus was read-only")
print("✓ No records modified")
print("✓ No tokenization performed")
print("=" * 80)

In [ ]:
import gzip
import json
import hashlib
from pathlib import Path
from collections import Counter

ROOT = Path("/kaggle/working")
CORPUS = ROOT / "phase2_canonical" / "corpus_temp"
PHASE4 = ROOT / "phase4_training_prep"

DOMAINS = [
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals",
]

print("=" * 80)
print("PHASE 4 — DOCUMENT IDENTITY / DUPLICATE AUDIT")
print("=" * 80)

global_stats = {
    "records": 0,
    "unique_ids": set(),
    "duplicate_ids": 0,
    "unique_text_hashes": set(),
    "duplicate_texts": 0,
}

domain_stats = {}

for domain in DOMAINS:

    domain_dir = CORPUS / domain
    files = sorted(domain_dir.glob("*.jsonl.gz"))

    stats = {
        "records": 0,
        "duplicate_ids": 0,
        "duplicate_texts": 0,
        "ids": set(),
        "text_hashes": set(),
    }

    print("\n" + "=" * 80)
    print(f"DOMAIN: {domain.upper()}")
    print("=" * 80)

    for file_idx, path in enumerate(files, 1):

        with gzip.open(path, "rt", encoding="utf-8") as f:

            for line_no, line in enumerate(f, 1):

                line = line.strip()

                if not line:
                    continue

                rec = json.loads(line)

                stats["records"] += 1
                global_stats["records"] += 1

                # ---------------------------------------------------------
                # DOCUMENT ID
                # ---------------------------------------------------------
                doc_id = rec.get("id")

                if doc_id is not None:

                    if doc_id in stats["ids"]:
                        stats["duplicate_ids"] += 1
                        global_stats["duplicate_ids"] += 1
                    else:
                        stats["ids"].add(doc_id)

                    if doc_id in global_stats["unique_ids"]:
                        pass
                    else:
                        global_stats["unique_ids"].add(doc_id)

                # ---------------------------------------------------------
                # EXACT TEXT HASH
                # ---------------------------------------------------------
                text = rec.get("text", "")

                if isinstance(text, str):

                    text_hash = hashlib.sha256(
                        text.encode("utf-8")
                    ).hexdigest()

                    if text_hash in stats["text_hashes"]:
                        stats["duplicate_texts"] += 1
                        global_stats["duplicate_texts"] += 1
                    else:
                        stats["text_hashes"].add(text_hash)

                    global_stats["unique_text_hashes"].add(text_hash)

        print(f"[{file_idx}/{len(files)}] {path.name} complete")

    domain_stats[domain] = stats

    print(f"\n{domain.upper()} SUMMARY")
    print("-" * 80)
    print(f"Records             : {stats['records']:,}")
    print(f"Unique IDs          : {len(stats['ids']):,}")
    print(f"Duplicate IDs       : {stats['duplicate_ids']:,}")
    print(f"Unique text hashes  : {len(stats['text_hashes']):,}")
    print(f"Duplicate texts     : {stats['duplicate_texts']:,}")


print("\n" + "=" * 80)
print("GLOBAL DUPLICATE AUDIT SUMMARY")
print("=" * 80)

print(f"Total records              : {global_stats['records']:,}")
print(f"Unique IDs observed        : {len(global_stats['unique_ids']):,}")
print(f"Duplicate IDs              : {global_stats['duplicate_ids']:,}")
print(f"Unique text hashes         : {len(global_stats['unique_text_hashes']):,}")
print(f"Duplicate text occurrences : {global_stats['duplicate_texts']:,}")

print("\n" + "=" * 80)
print("PHASE 4 DUPLICATE AUDIT COMPLETE")
print("=" * 80)

# Save only compact counts, not the huge hash sets.
audit_output = {
    "records": global_stats["records"],
    "unique_ids": len(global_stats["unique_ids"]),
    "duplicate_ids": global_stats["duplicate_ids"],
    "unique_text_hashes": len(global_stats["unique_text_hashes"]),
    "duplicate_text_occurrences": global_stats["duplicate_texts"],
    "domains": {
        domain: {
            "records": stats["records"],
            "unique_ids": len(stats["ids"]),
            "duplicate_ids": stats["duplicate_ids"],
            "unique_text_hashes": len(stats["text_hashes"]),
            "duplicate_texts": stats["duplicate_texts"],
        }
        for domain, stats in domain_stats.items()
    }
}

output_path = PHASE4 / "phase4_duplicate_audit.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(audit_output, f, indent=2)

print(f"Audit saved : {output_path}")
print("✓ Canonical corpus was read-only")
print("✓ No documents removed")
print("✓ No documents modified")
print("=" * 80)

In [ ]:
# ================================================================
# PHASE 4 — DEDUPLICATION / SPLIT POLICY INITIALIZATION
# ================================================================

from pathlib import Path
import json
import shutil
import os

CANONICAL = Path("/kaggle/working/phase2_canonical")
CORPUS = CANONICAL / "corpus_temp"

PHASE4 = Path("/kaggle/working/phase4_training_prep")

# Phase 4 working directories
DEDUP_DIR = PHASE4 / "dedup"
SPLIT_DIR = PHASE4 / "splits"
MANIFEST_DIR = PHASE4 / "manifests"

for d in [DEDUP_DIR, SPLIT_DIR, MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------
# Policy
# ------------------------------------------------
POLICY = {
    "phase": "phase4",
    "stage": "dedup_split_initialization",

    "canonical_corpus": str(CORPUS),

    # CRITICAL: canonical corpus is read-only.
    "canonical_read_only": True,

    "deduplication": {
        "method": "exact_text_hash",
        "scope": "within_domain",
        "keep_one_exact_text_occurrence": True,
        "remove_from_canonical": False,
    },

    "splitting": {
        "performed_after_deduplication": True,
        "prevent_duplicate_text_across_splits": True,
        "random_seed": 42,

        # We are recording the intended split policy now.
        # Actual records will be assigned in the next stage.
        "train_fraction": 0.90,
        "validation_fraction": 0.05,
        "test_fraction": 0.05,
    },

    "tokenization": {
        "performed": False
    },

    "training_data_generated": False
}

POLICY_FILE = PHASE4 / "phase4_dedup_split_policy.json"

with open(POLICY_FILE, "w", encoding="utf-8") as f:
    json.dump(POLICY, f, indent=2)

# ------------------------------------------------
# Verify canonical corpus still exists
# ------------------------------------------------
domains = [
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals"
]

print("=" * 80)
print("PHASE 4 — DEDUPLICATION / SPLIT POLICY INITIALIZATION")
print("=" * 80)

print()
print(f"Canonical corpus : {CORPUS}")
print(f"Phase 4 directory: {PHASE4}")

print()
print("=" * 80)
print("POLICY")
print("=" * 80)

print("Deduplication method       : exact text hash")
print("Deduplication scope       : within domain")
print("Canonical corpus modified : NO")
print("Keep one duplicate        : YES")
print()
print("Train fraction            : 90%")
print("Validation fraction       : 5%")
print("Test fraction             : 5%")
print("Random seed               : 42")
print()
print("Tokenization              : NOT YET")
print("Training data generated   : NO")

print()
print("=" * 80)
print("CANONICAL CORPUS CHECK")
print("=" * 80)

all_ok = True

for domain in domains:
    path = CORPUS / domain
    exists = path.exists() and path.is_dir()

    print(f"{domain:20s}: {'✓' if exists else '✗'}")

    if not exists:
        all_ok = False

print()
print("=" * 80)

if not all_ok:
    raise RuntimeError("Canonical corpus structure check failed.")

print("Policy file:")
print(f"  ✓ {POLICY_FILE}")

print()
print("Phase 4 directories:")
print(f"  ✓ {DEDUP_DIR}")
print(f"  ✓ {SPLIT_DIR}")
print(f"  ✓ {MANIFEST_DIR}")

print()
print("✓ PHASE 4 DEDUP/SPLIT POLICY INITIALIZED")
print("✓ CANONICAL CORPUS REMAINS READ-ONLY")
print("✓ NO RECORDS MODIFIED")
print("✓ NO TOKENIZATION PERFORMED")
print("✓ NO TRAINING DATA GENERATED")
print("=" * 80)

In [ ]:
# ================================================================
# PHASE 4 — EXACT-TEXT DEDUPLICATION
# ================================================================

from pathlib import Path
import gzip
import json
import hashlib
import time

CANONICAL = Path("/kaggle/working/phase2_canonical/corpus_temp")
PHASE4 = Path("/kaggle/working/phase4_training_prep")
DEDUP = PHASE4 / "dedup"

DOMAINS = [
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals",
]

for domain in DOMAINS:
    (DEDUP / domain).mkdir(parents=True, exist_ok=True)


def text_hash(text):
    """
    Exact normalized representation used only for deduplication.

    The canonical text itself is NOT modified.
    """
    if not isinstance(text, str):
        text = str(text)

    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def process_file(src, dst, seen):
    records_in = 0
    records_out = 0
    duplicates = 0

    with gzip.open(src, "rt", encoding="utf-8") as fin, \
         gzip.open(dst, "wt", encoding="utf-8") as fout:

        for line in fin:
            line = line.strip()

            if not line:
                continue

            record = json.loads(line)
            records_in += 1

            text = record.get("text", "")

            h = text_hash(text)

            if h in seen:
                duplicates += 1
                continue

            seen.add(h)

            fout.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                    separators=(",", ":")
                )
                + "\n"
            )

            records_out += 1

    return records_in, records_out, duplicates


print("=" * 80)
print("PHASE 4 — EXACT-TEXT DEDUPLICATION")
print("=" * 80)

global_in = 0
global_out = 0
global_duplicates = 0

domain_summary = {}

start_time = time.time()

for domain in DOMAINS:

    src_dir = CANONICAL / domain
    dst_dir = DEDUP / domain

    files = sorted(src_dir.glob("*.jsonl.gz"))

    if not files:
        raise RuntimeError(f"No canonical files found for domain: {domain}")

    print()
    print("=" * 80)
    print(f"DOMAIN: {domain.upper()}")
    print("=" * 80)

    seen = set()

    domain_in = 0
    domain_out = 0
    domain_duplicates = 0

    for idx, src in enumerate(files, 1):

        dst = dst_dir / src.name

        # Safety: never silently overwrite an existing output.
        if dst.exists():
            raise RuntimeError(
                f"Output already exists. Refusing to overwrite:\n{dst}"
            )

        r_in, r_out, dups = process_file(
            src,
            dst,
            seen
        )

        domain_in += r_in
        domain_out += r_out
        domain_duplicates += dups

        print(
            f"[{idx}/{len(files)}] "
            f"{src.name} complete | "
            f"in={r_in:,} | "
            f"kept={r_out:,} | "
            f"duplicates={dups:,}"
        )

    domain_summary[domain] = {
        "records_input": domain_in,
        "records_retained": domain_out,
        "duplicate_records_removed": domain_duplicates,
        "unique_texts": len(seen),
    }

    global_in += domain_in
    global_out += domain_out
    global_duplicates += domain_duplicates

    print()
    print(f"{domain.upper()} SUMMARY")
    print("-" * 80)
    print(f"Records input           : {domain_in:,}")
    print(f"Records retained        : {domain_out:,}")
    print(f"Duplicate occurrences  : {domain_duplicates:,}")
    print(f"Unique text hashes      : {len(seen):,}")


# ------------------------------------------------
# Save audit metadata
# ------------------------------------------------

audit = {
    "phase": "phase4",
    "operation": "exact_text_deduplication",
    "method": "sha256_exact_text",
    "scope": "within_domain",
    "canonical_modified": False,

    "domains": domain_summary,

    "global": {
        "records_input": global_in,
        "records_retained": global_out,
        "duplicate_records_removed": global_duplicates,
    },

    "timestamp": time.strftime(
        "%Y-%m-%dT%H:%M:%S"
    ),
}

AUDIT_FILE = PHASE4 / "phase4_dedup_audit.json"

with open(AUDIT_FILE, "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2)


elapsed = time.time() - start_time

print()
print("=" * 80)
print("GLOBAL DEDUPLICATION SUMMARY")
print("=" * 80)

print(f"Input records            : {global_in:,}")
print(f"Retained records         : {global_out:,}")
print(f"Duplicate occurrences    : {global_duplicates:,}")

print()
print(f"Expected duplicate count : 118,437")
print(
    f"Observed duplicate count : {global_duplicates:,}"
)

if global_in != 3_475_031:
    raise RuntimeError(
        f"Unexpected input count: {global_in:,}"
    )

if global_duplicates != 118_437:
    raise RuntimeError(
        "Deduplication result does not match the "
        "previous duplicate audit."
    )

if global_out != 3_356_594:
    raise RuntimeError(
        f"Unexpected retained count: {global_out:,}"
    )

print()
print("=" * 80)
print("✓ INPUT COUNT VERIFIED")
print("✓ DUPLICATE COUNT VERIFIED")
print("✓ RETAINED COUNT VERIFIED")
print("✓ CANONICAL CORPUS WAS NOT MODIFIED")
print("✓ DEDUPLICATED TRAINING VIEW CREATED")
print("=" * 80)

print()
print(f"Audit saved : {AUDIT_FILE}")
print(f"Elapsed time: {elapsed / 60:.2f} minutes")
print("=" * 80)

In [ ]:
from pathlib import Path
import shutil

print("=" * 80)
print("KAGGLE STORAGE — AFTER DEDUPLICATION")
print("=" * 80)

total, used, free = shutil.disk_usage("/kaggle/working")

def gb(x):
    return x / (1024**3)

print(f"Total : {gb(total):.2f} GB")
print(f"Used  : {gb(used):.2f} GB")
print(f"Free  : {gb(free):.2f} GB")

print()
print("PHASE 4 DIRECTORY SIZES")
print("-" * 80)

phase4 = Path("/kaggle/working/phase4_training_prep")

for p in sorted(phase4.iterdir()):
    if p.is_dir():
        size = sum(
            f.stat().st_size
            for f in p.rglob("*")
            if f.is_file()
        )
        print(f"{p.name:25s}: {gb(size):.2f} GB")
    else:
        print(f"{p.name:25s}: {gb(p.stat().st_size):.2f} GB")

print("=" * 80)

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/working/phase1_processed")

print("=" * 80)
print("PHASE 1 — STORAGE BREAKDOWN BEFORE CLEANUP")
print("=" * 80)

def size_gb(path):
    return sum(
        f.stat().st_size
        for f in path.rglob("*")
        if f.is_file()
    ) / (1024**3)

for p in sorted(ROOT.iterdir()):
    if p.is_dir():
        print(f"{p.name:45s} : {size_gb(p):.2f} GB")
    else:
        print(f"{p.name:45s} : {p.stat().st_size / (1024**3):.2f} GB")

print("=" * 80)

In [ ]:
# ================================================================
# STORAGE CLEANUP — REMOVE ONLY REDUNDANT PHASE 1 BACKUPS
# ================================================================

from pathlib import Path
import shutil

PHASE1 = Path("/kaggle/working/phase1_processed")

BACKUPS = [
    PHASE1 / "supreme_court_backup_20260928_102432",
    PHASE1 / "tribunals_backup_20260928_104758",
]

print("=" * 80)
print("PHASE 1 — REDUNDANT BACKUP CLEANUP")
print("=" * 80)

for path in BACKUPS:
    if not path.exists():
        print(f"NOT FOUND : {path}")
        continue

    if not path.is_dir():
        raise RuntimeError(f"Expected directory: {path}")

    size = sum(
        f.stat().st_size
        for f in path.rglob("*")
        if f.is_file()
    ) / (1024**3)

    print(f"Removing : {path.name}")
    print(f"Size     : {size:.2f} GB")

    shutil.rmtree(path)

    print("✓ Removed")
    print()

print("=" * 80)
print("VERIFYING CLEANUP")
print("=" * 80)

for path in BACKUPS:
    print(f"{path.name:45s} : {'STILL EXISTS' if path.exists() else '✓ removed'}")

print("=" * 80)
print("✓ ONLY REDUNDANT PHASE 1 BACKUPS WERE REMOVED")
print("✓ ACTIVE PHASE 1 DATA WAS NOT TOUCHED")
print("✓ PHASE 2 CANONICAL CORPUS WAS NOT TOUCHED")
print("✓ PHASE 4 DEDUP CORPUS WAS NOT TOUCHED")
print("=" * 80)

In [ ]:
import os
import json
import shutil
from pathlib import Path

print("=" * 80)
print("PHASE 4 — STORAGE SAFETY CHECK")
print("=" * 80)

# -------------------------------------------------------------------------
# PATHS
# -------------------------------------------------------------------------
BASE = Path("/kaggle/working")
CANONICAL = BASE / "phase2_canonical"
CORPUS = CANONICAL / "corpus_temp"
PHASE4 = BASE / "phase4_training_prep"
DEDUP = PHASE4 / "dedup"

# -------------------------------------------------------------------------
# STORAGE
# -------------------------------------------------------------------------
total, used, free = shutil.disk_usage(BASE)

def gb(x):
    return x / (1024 ** 3)

print("\n" + "=" * 80)
print("KAGGLE STORAGE")
print("=" * 80)
print(f"Total : {gb(total):.2f} GB")
print(f"Used  : {gb(used):.2f} GB")
print(f"Free  : {gb(free):.2f} GB")

# -------------------------------------------------------------------------
# SIZE HELPER
# -------------------------------------------------------------------------
def directory_size(path):
    if not path.exists():
        return 0
    total_size = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            try:
                total_size += os.path.getsize(os.path.join(root, f))
            except OSError:
                pass
    return total_size

# -------------------------------------------------------------------------
# IMPORTANT DIRECTORIES
# -------------------------------------------------------------------------
print("\n" + "=" * 80)
print("IMPORTANT DIRECTORY SIZES")
print("=" * 80)

paths_to_check = [
    ("phase2_canonical", CANONICAL),
    ("canonical corpus_temp", CORPUS),
    ("phase4_training_prep", PHASE4),
    ("phase4 dedup", DEDUP),
]

for name, path in paths_to_check:
    size = directory_size(path)
    print(f"{name:<30}: {gb(size):.2f} GB | exists={path.exists()}")

# -------------------------------------------------------------------------
# BACKUP DIRECTORIES
# -------------------------------------------------------------------------
print("\n" + "=" * 80)
print("BACKUP DIRECTORIES — CANDIDATES FOR CLEANUP")
print("=" * 80)

backup_dirs = []

for p in BASE.iterdir():
    if p.is_dir() and "backup" in p.name.lower():
        size = directory_size(p)
        backup_dirs.append((p, size))
        print(f"{p.name:<50} : {gb(size):.2f} GB")

if not backup_dirs:
    print("No backup directories found.")

# -------------------------------------------------------------------------
# CANONICAL CORPUS STRUCTURE
# -------------------------------------------------------------------------
print("\n" + "=" * 80)
print("CANONICAL CORPUS CHECK")
print("=" * 80)

expected_domains = [
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals",
]

canonical_ok = True

for domain in expected_domains:
    path = CORPUS / domain
    exists = path.exists() and path.is_dir()

    print(f"{domain:<20}: {exists}")

    if not exists:
        canonical_ok = False

# -------------------------------------------------------------------------
# DEDUP VIEW CHECK
# -------------------------------------------------------------------------
print("\n" + "=" * 80)
print("DEDUP VIEW CHECK")
print("=" * 80)

dedup_ok = DEDUP.exists() and DEDUP.is_dir()
print(f"Dedup directory exists : {dedup_ok}")

if dedup_ok:
    for domain in expected_domains:
        path = DEDUP / domain
        print(f"{domain:<20}: {path.exists() and path.is_dir()}")

# -------------------------------------------------------------------------
# PHASE 4 AUDIT
# -------------------------------------------------------------------------
audit = PHASE4 / "phase4_dedup_audit.json"

print("\n" + "=" * 80)
print("DEDUP AUDIT CHECK")
print("=" * 80)

if audit.exists():
    print(f"Audit exists : True")
    try:
        with open(audit, "r") as f:
            data = json.load(f)

        # Print useful top-level information without assuming exact schema.
        if isinstance(data, dict):
            print("Audit JSON : valid")
            print("Top-level keys:")
            for k in data.keys():
                print(f"  - {k}")
        else:
            print("Audit JSON : valid but unexpected top-level type")

    except Exception as e:
        print(f"Audit JSON : ERROR -> {e}")
else:
    print("Audit exists : False")

# -------------------------------------------------------------------------
# CLEANUP RECOMMENDATION
# -------------------------------------------------------------------------
print("\n" + "=" * 80)
print("CLEANUP RECOMMENDATION")
print("=" * 80)

print("""
SAFE TO CONSIDER REMOVING:
  Only directories whose names explicitly contain 'backup'
  and which are confirmed to be redundant copies.

DO NOT REMOVE:
  /kaggle/working/phase2_canonical
  /kaggle/working/phase2_canonical/corpus_temp
  /kaggle/working/phase4_training_prep/dedup
  Any phase4 audit/policy/manifest files

IMPORTANT:
  This cell performs NO deletion.
  Review the backup list above before running any cleanup.
""")

print("=" * 80)
print("STORAGE SAFETY CHECK COMPLETE")
print("=" * 80)

In [ ]:
import os
import json
import gzip
import hashlib
import random
import shutil
from pathlib import Path
from collections import Counter

BASE = Path("/kaggle/working")

CANONICAL = BASE / "phase2_canonical" / "corpus_temp"
PHASE4 = BASE / "phase4_training_prep"
DEDUP = PHASE4 / "dedup"
SPLITS = PHASE4 / "splits"
MANIFESTS = PHASE4 / "manifests"

SPLITS.mkdir(parents=True, exist_ok=True)
MANIFESTS.mkdir(parents=True, exist_ok=True)

SEED = 42
TRAIN_FRAC = 0.90
VAL_FRAC = 0.05
TEST_FRAC = 0.05

print("=" * 80)
print("PHASE 4 — SPLIT GENERATION INITIALIZATION")
print("=" * 80)

print(f"\nCanonical corpus : {CANONICAL}")
print(f"Dedup view       : {DEDUP}")
print(f"Splits directory : {SPLITS}")
print(f"Manifests        : {MANIFESTS}")

print("\n" + "=" * 80)
print("POLICY")
print("=" * 80)

print(f"Train fraction       : {TRAIN_FRAC}")
print(f"Validation fraction  : {VAL_FRAC}")
print(f"Test fraction        : {TEST_FRAC}")
print(f"Random seed          : {SEED}")

assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1.0) < 1e-12
assert CANONICAL.exists()
assert DEDUP.exists()

for domain in ["acts", "high_courts", "supreme_court", "tribunals"]:
    assert (DEDUP / domain).exists(), f"Missing dedup domain: {domain}"

print("\n✓ Canonical corpus exists")
print("✓ Deduplicated view exists")
print("✓ All four domains exist")
print("✓ Split fractions sum to 100%")
print("✓ No records modified")
print("✓ No tokenization")
print("✓ No dataset copies generated")

print("\n" + "=" * 80)
print("INITIALIZATION COMPLETE")
print("=" * 80)

In [ ]:
import json
import gzip
import hashlib
from pathlib import Path
from collections import Counter

# -------------------------------------------------------------------------
# CONFIG
# -------------------------------------------------------------------------

BASE = Path("/kaggle/working")
PHASE4 = BASE / "phase4_training_prep"

DEDUP = PHASE4 / "dedup"
SPLITS = PHASE4 / "splits"
MANIFESTS = PHASE4 / "manifests"

SEED = 42

DOMAINS = [
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals",
]

SPLIT_FILES = {
    "train": MANIFESTS / "train_manifest.jsonl",
    "validation": MANIFESTS / "validation_manifest.jsonl",
    "test": MANIFESTS / "test_manifest.jsonl",
}

# -------------------------------------------------------------------------
# DETERMINISTIC SPLIT FUNCTION
# -------------------------------------------------------------------------

def assign_split(text_hash, seed=42):
    """
    Deterministic assignment based on:
        seed + text_hash

    The same text hash with the same seed always receives
    the same split.
    """

    key = f"{seed}:{text_hash}".encode("utf-8")
    digest = hashlib.sha256(key).digest()

    # First 8 bytes -> deterministic integer
    value = int.from_bytes(digest[:8], byteorder="big")

    # Uniform deterministic value in [0, 1)
    u = value / float(2**64)

    if u < 0.90:
        return "train"
    elif u < 0.95:
        return "validation"
    else:
        return "test"


# -------------------------------------------------------------------------
# OPEN OUTPUT MANIFESTS
# -------------------------------------------------------------------------

for path in SPLIT_FILES.values():
    if path.exists():
        path.unlink()

handles = {
    split: open(path, "w", encoding="utf-8")
    for split, path in SPLIT_FILES.items()
}

counts = Counter()
domain_counts = {
    domain: Counter()
    for domain in DOMAINS
}

total = 0

print("=" * 80)
print("PHASE 4 — DETERMINISTIC SPLIT MANIFEST GENERATION")
print("=" * 80)

try:

    for domain in DOMAINS:

        domain_dir = DEDUP / domain
        files = sorted(domain_dir.glob("*.jsonl.gz"))

        print(f"\nDOMAIN: {domain}")
        print("-" * 80)
        print(f"Files: {len(files)}")

        domain_total = 0

        for file_index, source_path in enumerate(files, start=1):

            file_count = 0

            with gzip.open(source_path, "rt", encoding="utf-8") as f:

                for line in f:

                    line = line.strip()

                    if not line:
                        continue

                    record = json.loads(line)

                    # -----------------------------------------------------
                    # Recover text
                    # -----------------------------------------------------

                    text = record.get("text")

                    if not isinstance(text, str):
                        raise ValueError(
                            f"Non-string text found: {domain}/{source_path.name}"
                        )

                    # -----------------------------------------------------
                    # EXACT TEXT HASH
                    # -----------------------------------------------------

                    text_hash = hashlib.sha256(
                        text.encode("utf-8")
                    ).hexdigest()

                    # -----------------------------------------------------
                    # SPLIT ASSIGNMENT
                    # -----------------------------------------------------

                    split = assign_split(text_hash, SEED)

                    # -----------------------------------------------------
                    # MANIFEST RECORD
                    # -----------------------------------------------------

                    manifest_record = {
                        "domain": domain,
                        "source_file": str(source_path.relative_to(DEDUP)),
                        "text_hash": text_hash,
                        "split": split,
                    }

                    handles[split].write(
                        json.dumps(
                            manifest_record,
                            ensure_ascii=False,
                            separators=(",", ":")
                        ) + "\n"
                    )

                    counts[split] += 1
                    domain_counts[domain][split] += 1

                    file_count += 1
                    domain_total += 1
                    total += 1

            print(
                f"[{file_index}/{len(files)}] "
                f"{source_path.name} complete | records={file_count:,}"
            )

        print(
            f"{domain.upper()} TOTAL : {domain_total:,}"
        )

finally:

    for handle in handles.values():
        handle.close()


# -------------------------------------------------------------------------
# SUMMARY
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("SPLIT GENERATION SUMMARY")
print("=" * 80)

print(f"Total records processed : {total:,}")
print(f"Train                  : {counts['train']:,}")
print(f"Validation             : {counts['validation']:,}")
print(f"Test                   : {counts['test']:,}")

print("\nPercentages:")

for split in ["train", "validation", "test"]:
    pct = counts[split] / total * 100
    print(f"{split:<12}: {pct:.4f}%")

print("\n" + "=" * 80)
print("DOMAIN DISTRIBUTION")
print("=" * 80)

for domain in DOMAINS:

    dtotal = sum(domain_counts[domain].values())

    print(f"\n{domain}")
    print(f"  Total      : {dtotal:,}")
    print(f"  Train      : {domain_counts[domain]['train']:,}")
    print(f"  Validation : {domain_counts[domain]['validation']:,}")
    print(f"  Test       : {domain_counts[domain]['test']:,}")

print("\n" + "=" * 80)
print("MANIFEST FILES")
print("=" * 80)

for split, path in SPLIT_FILES.items():
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"{split:<12}: {path} | {size_mb:.2f} MB")

print("\n✓ Deduplicated corpus was read-only")
print("✓ No tokenization performed")
print("✓ No corpus copies created")
print("✓ Split manifests generated")
print("=" * 80)

In [ ]:
import json
from pathlib import Path
from collections import Counter

BASE = Path("/kaggle/working")
MANIFESTS = BASE / "phase4_training_prep" / "manifests"

EXPECTED_TOTAL = 3_356_594

MANIFEST_PATHS = {
    "train": MANIFESTS / "train_manifest.jsonl",
    "validation": MANIFESTS / "validation_manifest.jsonl",
    "test": MANIFESTS / "test_manifest.jsonl",
}

print("=" * 80)
print("PHASE 4 — SPLIT MANIFEST VERIFICATION")
print("=" * 80)

counts = Counter()
seen_hashes = {}
duplicate_hashes = []

for split, path in MANIFEST_PATHS.items():

    print(f"\nChecking {split}...")
    assert path.exists(), f"Missing manifest: {path}"

    with open(path, "r", encoding="utf-8") as f:

        for line_number, line in enumerate(f, start=1):

            record = json.loads(line)

            assert record["split"] == split
            assert "domain" in record
            assert "source_file" in record
            assert "text_hash" in record

            text_hash = record["text_hash"]

            counts[split] += 1

            if text_hash in seen_hashes:
                duplicate_hashes.append(
                    (
                        text_hash,
                        seen_hashes[text_hash],
                        split,
                    )
                )
            else:
                seen_hashes[text_hash] = split

print("\n" + "=" * 80)
print("COUNTS")
print("=" * 80)

total = sum(counts.values())

for split in ["train", "validation", "test"]:
    pct = counts[split] / total * 100
    print(
        f"{split:<12}: "
        f"{counts[split]:>12,} "
        f"({pct:>7.4f}%)"
    )

print(f"\nTotal        : {total:,}")
print(f"Expected     : {EXPECTED_TOTAL:,}")

print("\n" + "=" * 80)
print("LEAKAGE CHECK")
print("=" * 80)

print(f"Unique hashes observed : {len(seen_hashes):,}")
print(f"Duplicate hash entries : {len(duplicate_hashes):,}")

assert total == EXPECTED_TOTAL, (
    f"Expected {EXPECTED_TOTAL:,} records, found {total:,}"
)

assert len(seen_hashes) == EXPECTED_TOTAL, (
    f"Expected {EXPECTED_TOTAL:,} unique hashes, "
    f"found {len(seen_hashes):,}"
)

assert len(duplicate_hashes) == 0, (
    f"Found {len(duplicate_hashes):,} hashes appearing in multiple splits"
)

print("\n✓ TOTAL COUNT VERIFIED")
print("✓ UNIQUE HASH COUNT VERIFIED")
print("✓ ZERO CROSS-SPLIT DUPLICATES")
print("✓ TRAIN/VALIDATION/TEST ARE DISJOINT")

print("\n" + "=" * 80)
print("PHASE 4 SPLIT MANIFEST CHECKPOINT VALID")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — PRE-TOKENIZATION CHECKPOINT
# ============================================================

from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

BASE = Path("/kaggle/working")
CANONICAL = BASE / "phase2_canonical" / "corpus_temp"
PHASE4 = BASE / "phase4_training_prep"
MANIFESTS = PHASE4 / "manifests"
DEDUP = PHASE4 / "dedup"

DOMAINS = [
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals",
]

for p in [CANONICAL, PHASE4, MANIFESTS, DEDUP]:
    assert p.exists(), f"Missing: {p}"

manifest_files = {
    "train": MANIFESTS / "train_manifest.jsonl",
    "validation": MANIFESTS / "validation_manifest.jsonl",
    "test": MANIFESTS / "test_manifest.jsonl",
}

for name, path in manifest_files.items():
    assert path.exists(), f"Missing {name} manifest: {path}"

# Count manifests
counts = {}

for name, path in manifest_files.items():
    with open(path, "r", encoding="utf-8") as f:
        counts[name] = sum(1 for _ in f)

total = sum(counts.values())

print("=" * 80)
print("PHASE 4 — PRE-TOKENIZATION CHECKPOINT")
print("=" * 80)

print(f"\nCanonical corpus : {CANONICAL}")
print(f"Phase 4          : {PHASE4}")

print("\nMANIFEST COUNTS")
print("-" * 80)

for name, count in counts.items():
    print(f"{name:12s}: {count:,}")

print(f"\nTotal          : {total:,}")

assert total == 3_356_594, (
    f"Unexpected total: {total:,}"
)

checkpoint = {
    "phase": 4,
    "stage": "pre_tokenization",
    "canonical_corpus": str(CANONICAL),
    "domains": DOMAINS,
    "manifest_counts": counts,
    "total_records": total,
    "deduplication": {
        "method": "exact_text_hash",
        "scope": "within_domain",
    },
    "split": {
        "train": 0.90,
        "validation": 0.05,
        "test": 0.05,
        "seed": 42,
    },
    "tokenization_performed": False,
    "canonical_modified": False,
    "timestamp": datetime.now(timezone.utc).isoformat(),
}

out = PHASE4 / "phase4_pre_tokenization_checkpoint.json"

with open(out, "w", encoding="utf-8") as f:
    json.dump(checkpoint, f, indent=2)

print("\nCheckpoint:")
print(out)

print("\n✓ Manifest counts verified")
print("✓ Deduplicated corpus available")
print("✓ Canonical corpus untouched")
print("✓ Ready for tokenizer stage")
print("=" * 80)

In [ ]:
# ============================================================
# STORAGE CHECK — BEFORE TOKENIZER TRAINING
# ============================================================

import os
import shutil
from pathlib import Path

BASE = Path("/kaggle/working")

total, used, free = shutil.disk_usage(BASE)

def gb(x):
    return x / (1024 ** 3)

print("=" * 80)
print("STORAGE CHECK — BEFORE TOKENIZER")
print("=" * 80)

print(f"\nTotal : {gb(total):.2f} GB")
print(f"Used  : {gb(used):.2f} GB")
print(f"Free  : {gb(free):.2f} GB")

print("\nIMPORTANT DIRECTORIES")
print("-" * 80)

paths = [
    BASE / "phase2_canonical",
    BASE / "phase4_training_prep",
    BASE / "phase4_training_prep" / "dedup",
    BASE / "phase4_training_prep" / "manifests",
]

for p in paths:
    if p.exists():
        size = sum(
            x.stat().st_size
            for x in p.rglob("*")
            if x.is_file()
        )
        print(f"{p.name:30s}: {gb(size):.2f} GB")

print("\n" + "=" * 80)

if gb(free) < 2:
    print("⚠ WARNING: Less than 2 GB free.")
elif gb(free) < 4:
    print("⚠ CAUTION: Less than 4 GB free.")
else:
    print("✓ Storage currently acceptable for tokenizer preparation.")

print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — TOKENIZER WORKSPACE
# ============================================================

from pathlib import Path
import json
from datetime import datetime, timezone

BASE = Path("/kaggle/working")
PHASE4 = BASE / "phase4_training_prep"

TOKENIZER_DIR = PHASE4 / "tokenizer"

TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

config = {
    "phase": 1,
    "stage": "tokenizer",
    "algorithm": "BPE",
    "training_corpus": "phase4_deduplicated_train_view",
    "source_manifest": str(
        PHASE4 / "manifests" / "train_manifest.jsonl"
    ),
    "canonical_corpus_modified": False,
    "tokenization_performed": False,
    "created_at": datetime.now(timezone.utc).isoformat(),
}

config_path = TOKENIZER_DIR / "tokenizer_config.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("=" * 80)
print("TOKENIZER WORKSPACE CREATED")
print("=" * 80)

print(f"\nDirectory : {TOKENIZER_DIR}")
print(f"Config    : {config_path}")

print("\n✓ Canonical corpus remains read-only")
print("✓ No tokenization performed")
print("✓ Ready for BPE tokenizer experiment")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — CHECKPOINT VERIFICATION
# ============================================================

from pathlib import Path
import json

BASE = Path("/kaggle/working")
PHASE4 = BASE / "phase4_training_prep"
CANONICAL = BASE / "phase2_canonical" / "corpus_temp"
MANIFESTS = PHASE4 / "manifests"

assert CANONICAL.exists(), "Canonical corpus missing"
assert PHASE4.exists(), "Phase 4 directory missing"

manifest_paths = {
    "train": MANIFESTS / "train_manifest.jsonl",
    "validation": MANIFESTS / "validation_manifest.jsonl",
    "test": MANIFESTS / "test_manifest.jsonl",
}

for name, path in manifest_paths.items():
    assert path.exists(), f"Missing {name} manifest"

counts = {}

for name, path in manifest_paths.items():
    with open(path, "r", encoding="utf-8") as f:
        counts[name] = sum(1 for _ in f)

expected = {
    "train": 3_020_655,
    "validation": 168_665,
    "test": 167_274,
}

print("=" * 80)
print("PHASE 4 — CHECKPOINT VERIFICATION")
print("=" * 80)

for name in ["train", "validation", "test"]:
    print(
        f"{name:12s}: "
        f"{counts[name]:,} "
        f"(expected {expected[name]:,})"
    )
    assert counts[name] == expected[name]

total = sum(counts.values())

assert total == 3_356_594

print("-" * 80)
print(f"TOTAL       : {total:,}")
print()
print("✓ Train manifest verified")
print("✓ Validation manifest verified")
print("✓ Test manifest verified")
print("✓ Total record count verified")
print("✓ Canonical corpus exists")
print("✓ Ready for tokenizer stage")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — TOKENIZER EXPERIMENT WORKSPACE
# ============================================================

from pathlib import Path
import json
from datetime import datetime, timezone

BASE = Path("/kaggle/working")
PHASE4 = BASE / "phase4_training_prep"

TOKENIZER_ROOT = PHASE4 / "tokenizer"

TOKENIZER_ROOT.mkdir(parents=True, exist_ok=True)

# Separate experiment directories.
experiments = [
    TOKENIZER_ROOT / "bpe_32k",
    TOKENIZER_ROOT / "bpe_50k",
    TOKENIZER_ROOT / "bpe_64k",
]

for p in experiments:
    p.mkdir(parents=True, exist_ok=True)

config = {
    "stage": "tokenizer_experiment",
    "algorithm": "BPE",
    "candidate_vocab_sizes": [
        32_000,
        50_000,
        64_000,
    ],
    "train_manifest": str(
        PHASE4 / "manifests" / "train_manifest.jsonl"
    ),
    "validation_manifest": str(
        PHASE4 / "manifests" / "validation_manifest.jsonl"
    ),
    "test_manifest": str(
        PHASE4 / "manifests" / "test_manifest.jsonl"
    ),
    "canonical_corpus_modified": False,
    "created_at": datetime.now(timezone.utc).isoformat(),
}

with open(
    TOKENIZER_ROOT / "tokenizer_experiment_config.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(config, f, indent=2)

print("=" * 80)
print("TOKENIZER EXPERIMENT WORKSPACE")
print("=" * 80)

print(f"\nRoot: {TOKENIZER_ROOT}")

for p in experiments:
    print(f"✓ {p}")

print("\nCandidate vocabulary sizes:")
for v in config["candidate_vocab_sizes"]:
    print(f"  • {v:,}")

print("\n✓ No tokenizer trained yet")
print("✓ Canonical corpus untouched")
print("=" * 80)

In [ ]:
# ============================================================
# TOKENIZER DEPENDENCY CHECK
# ============================================================

import importlib.util

packages = {
    "tokenizers": "Hugging Face Tokenizers",
    "datasets": "Hugging Face Datasets",
}

print("=" * 80)
print("TOKENIZER DEPENDENCY CHECK")
print("=" * 80)

for package, description in packages.items():
    available = importlib.util.find_spec(package) is not None
    status = "✓" if available else "✗"

    print(f"{status} {package:15s} — {description}")

print("=" * 80)

assert importlib.util.find_spec("tokenizers") is not None, (
    "The 'tokenizers' package is required."
)

In [ ]:
# ============================================================
# MANIFEST RECORD INSPECTION
# ============================================================

import json
from pathlib import Path

manifest = Path(
    "/kaggle/working/phase4_training_prep/manifests/train_manifest.jsonl"
)

print("=" * 80)
print("TRAIN MANIFEST — SAMPLE RECORD")
print("=" * 80)

with open(manifest, "r", encoding="utf-8") as f:
    first = json.loads(f.readline())

print("\nFields:")
for key, value in first.items():
    if isinstance(value, str):
        preview = value[:500].replace("\n", " ")
        print(f"\n{key}:")
        print(f"  {preview}")
    else:
        print(f"\n{key}: {value}")

print("\n✓ Manifest readable")
print("✓ No records modified")
print("=" * 80)

In [ ]:
# ============================================================
# TOKENIZER PRE-TRAINING SAMPLE AUDIT
# ============================================================

import json
from pathlib import Path
from collections import Counter

manifest = Path(
    "/kaggle/working/phase4_training_prep/manifests/train_manifest.jsonl"
)

SAMPLE_SIZE = 10_000

records = []
domain_counts = Counter()
char_count = 0
word_count = 0

with open(manifest, "r", encoding="utf-8") as f:

    for i, line in enumerate(f):

        if i >= SAMPLE_SIZE:
            break

        record = json.loads(line)

        text = record.get("text", "")

        if not isinstance(text, str):
            continue

        records.append(text)

        domain = record.get("domain", "unknown")
        domain_counts[domain] += 1

        char_count += len(text)
        word_count += len(text.split())

print("=" * 80)
print("TOKENIZER PRE-TRAINING SAMPLE AUDIT")
print("=" * 80)

print(f"\nSample records : {len(records):,}")
print(f"Characters     : {char_count:,}")
print(f"Words          : {word_count:,}")

print("\nDomains:")
for domain, count in domain_counts.items():
    print(f"  {domain:20s}: {count:,}")

print("\n✓ Sample successfully read")
print("✓ Canonical corpus untouched")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — BPE TOKENIZER EXPERIMENT
# EXPERIMENT: 32K VOCABULARY
# ============================================================

from pathlib import Path
import json
import gzip
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

BASE = Path("/kaggle/working")
PHASE4 = BASE / "phase4_training_prep"

MANIFEST = PHASE4 / "manifests" / "train_manifest.jsonl"
OUT_DIR = PHASE4 / "tokenizer" / "bpe_32k"

OUT_DIR.mkdir(parents=True, exist_ok=True)

VOCAB_SIZE = 32_000

SPECIAL_TOKENS = [
    "[PAD]",
    "[UNK]",
    "[BOS]",
    "[EOS]",
]

print("=" * 80)
print("PHASE 4 — BPE TOKENIZER TRAINING")
print("=" * 80)

print(f"\nManifest    : {MANIFEST}")
print(f"Vocabulary  : {VOCAB_SIZE:,}")
print(f"Output      : {OUT_DIR}")

# ------------------------------------------------------------
# Streaming iterator.
# Does NOT load the corpus into RAM.
# ------------------------------------------------------------

def text_iterator(path, batch_size=1000):

    batch = []

    with open(path, "r", encoding="utf-8") as f:

        for line in f:

            record = json.loads(line)

            text = record.get("text")

            if not isinstance(text, str):
                continue

            text = text.strip()

            if not text:
                continue

            batch.append(text)

            if len(batch) >= batch_size:
                yield batch
                batch = []

    if batch:
        yield batch


tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=SPECIAL_TOKENS,
    show_progress=True,
)

print("\nStarting BPE training...")
print("This may take time because the training manifest contains")
print("millions of legal documents.")

tokenizer.train_from_iterator(
    text_iterator(MANIFEST),
    trainer=trainer,
    length=3_020_655,
)

tokenizer_path = OUT_DIR / "tokenizer.json"

tokenizer.save(str(tokenizer_path))

metadata = {
    "algorithm": "BPE",
    "vocab_size_requested": VOCAB_SIZE,
    "special_tokens": SPECIAL_TOKENS,
    "min_frequency": 2,
    "training_manifest": str(MANIFEST),
    "canonical_corpus_modified": False,
    "tokenizer_path": str(tokenizer_path),
}

with open(
    OUT_DIR / "metadata.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(metadata, f, indent=2)

print("\n" + "=" * 80)
print("✓ BPE 32K TOKENIZER TRAINING COMPLETE")
print("=" * 80)

print(f"\nTokenizer : {tokenizer_path}")
print(f"Vocabulary: {tokenizer.get_vocab_size():,}")
print(f"Metadata  : {OUT_DIR / 'metadata.json'}")
print("\n✓ Canonical corpus untouched")
print("=" * 80)

In [ ]:
# ============================================================
# BPE 32K — TOKENIZER AUDIT
# ============================================================

from pathlib import Path
import json
from tokenizers import Tokenizer

PHASE4 = Path("/kaggle/working/phase4_training_prep")

TOKENIZER_PATH = (
    PHASE4 / "tokenizer" / "bpe_32k" / "tokenizer.json"
)

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

test_texts = [
    "The Supreme Court held that the appeal is maintainable.",
    "The petitioner challenged the constitutional validity of the statute.",
    "In the facts and circumstances of the case, the writ petition is dismissed.",
    "The tribunal considered the principles of natural justice and procedural fairness.",
]

print("=" * 80)
print("BPE 32K — TOKENIZER AUDIT")
print("=" * 80)

print(f"\nVocabulary size: {tokenizer.get_vocab_size():,}")

for i, text in enumerate(test_texts, 1):

    encoding = tokenizer.encode(text)

    print(f"\nExample {i}")
    print("-" * 80)
    print(f"Text characters : {len(text):,}")
    print(f"Token count     : {len(encoding.ids):,}")
    print(f"Tokens          : {encoding.tokens}")

print("\n✓ Tokenizer loads successfully")
print("✓ Tokenization produces valid IDs")
print("=" * 80)

In [ ]:
# ============================================================
# BPE 32K — TOKENIZATION EFFICIENCY AUDIT
# ============================================================

from pathlib import Path
import json
from tokenizers import Tokenizer

PHASE4 = Path("/kaggle/working/phase4_training_prep")

MANIFEST = PHASE4 / "manifests" / "train_manifest.jsonl"
TOKENIZER_PATH = PHASE4 / "tokenizer" / "bpe_32k" / "tokenizer.json"

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

SAMPLE_SIZE = 5_000

total_chars = 0
total_words = 0
total_tokens = 0
documents = 0

with open(MANIFEST, "r", encoding="utf-8") as f:

    for line in f:

        if documents >= SAMPLE_SIZE:
            break

        record = json.loads(line)
        text = record.get("text")

        if not isinstance(text, str) or not text.strip():
            continue

        encoding = tokenizer.encode(text)

        total_chars += len(text)
        total_words += len(text.split())
        total_tokens += len(encoding.ids)

        documents += 1

print("=" * 80)
print("BPE 32K — TOKENIZATION EFFICIENCY")
print("=" * 80)

print(f"\nDocuments : {documents:,}")
print(f"Characters: {total_chars:,}")
print(f"Words     : {total_words:,}")
print(f"Tokens    : {total_tokens:,}")

if total_words:
    print(
        f"\nTokens / word : "
        f"{total_tokens / total_words:.3f}"
    )

if total_chars:
    print(
        f"Tokens / char : "
        f"{total_tokens / total_chars:.4f}"
    )

print("\n✓ Efficiency measurement complete")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — MANIFEST SCHEMA DIAGNOSTIC
# ============================================================

import json
from pathlib import Path

MANIFEST = Path(
    "/kaggle/working/phase4_training_prep/manifests/train_manifest.jsonl"
)

print("=" * 80)
print("MANIFEST SCHEMA DIAGNOSTIC")
print("=" * 80)

with open(MANIFEST, "r", encoding="utf-8") as f:
    record = json.loads(f.readline())

print("\nTop-level fields:")
print("-" * 80)

for key, value in record.items():

    if isinstance(value, str):
        print(f"{key:25s}: string | {value[:300]}")

    elif isinstance(value, (int, float, bool)):
        print(f"{key:25s}: {type(value).__name__} | {value}")

    elif value is None:
        print(f"{key:25s}: None")

    else:
        print(
            f"{key:25s}: "
            f"{type(value).__name__} | "
            f"{str(value)[:300]}"
        )

print("\n" + "=" * 80)

# Explicitly check text
if "text" in record:
    print("TEXT FIELD:")
    print("✓ Manifest contains a text field")
    print(f"Type: {type(record['text']).__name__}")
    print(f"Length: {len(record['text']) if isinstance(record['text'], str) else 'N/A'}")
else:
    print("TEXT FIELD:")
    print("✗ Manifest does NOT contain a text field")

print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — MANIFEST PATH / REFERENCE DIAGNOSTIC
# ============================================================

import json
from pathlib import Path

MANIFEST = Path(
    "/kaggle/working/phase4_training_prep/manifests/train_manifest.jsonl"
)

print("=" * 80)
print("MANIFEST REFERENCE DIAGNOSTIC")
print("=" * 80)

with open(MANIFEST, "r", encoding="utf-8") as f:

    for i in range(5):

        line = f.readline()

        if not line:
            break

        record = json.loads(line)

        print(f"\nRECORD {i + 1}")
        print("-" * 80)

        for key, value in record.items():
            print(f"{key}: {value}")

print("\n" + "=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — DEDUP VIEW SCHEMA CHECK
# ============================================================

import gzip
import json
from pathlib import Path

DEDUP = Path(
    "/kaggle/working/phase4_training_prep/dedup"
)

print("=" * 80)
print("DEDUP VIEW SCHEMA CHECK")
print("=" * 80)

files = sorted(DEDUP.rglob("*.jsonl.gz"))

print(f"\nFound compressed JSONL files: {len(files)}")

assert files, "No deduplicated JSONL files found."

sample_file = files[0]

print(f"\nSample file:")
print(sample_file)

with gzip.open(sample_file, "rt", encoding="utf-8") as f:
    record = json.loads(f.readline())

print("\nFirst record:")
print("-" * 80)

for key, value in record.items():

    if isinstance(value, str):
        print(f"{key}: {value[:500]}")

    else:
        print(f"{key}: {value}")

print("\n" + "=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — TOKENIZER PIPELINE INITIALIZATION
# ============================================================

from pathlib import Path
import json
import shutil
import os

BASE = Path("/kaggle/working")
DEDUP_DIR = BASE / "phase4_training_prep" / "dedup"
MANIFEST_DIR = BASE / "phase4_training_prep" / "manifests"
TOKENIZER_DIR = BASE / "phase4_training_prep" / "tokenizer"

TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("PHASE 4 — TOKENIZER PIPELINE INITIALIZATION")
print("=" * 80)

print(f"Dedup directory : {DEDUP_DIR}")
print(f"Manifest dir    : {MANIFEST_DIR}")
print(f"Tokenizer dir   : {TOKENIZER_DIR}")

assert DEDUP_DIR.exists(), "Dedup directory missing"
assert MANIFEST_DIR.exists(), "Manifest directory missing"

for name in ["train_manifest.jsonl",
             "validation_manifest.jsonl",
             "test_manifest.jsonl"]:
    p = MANIFEST_DIR / name
    assert p.exists(), f"Missing manifest: {p}"
    print(f"✓ {name}")

# Remove only the known failed tokenizer artifacts.
# This does NOT touch canonical corpus, dedup corpus, or manifests.
for p in TOKENIZER_DIR.iterdir():
    if p.is_file():
        p.unlink()
    elif p.is_dir():
        shutil.rmtree(p)

print()
print("✓ Failed tokenizer artifacts cleared")
print("✓ Dedup corpus untouched")
print("✓ Manifests untouched")
print("✓ Ready for text-resolution stage")

In [ ]:
# ============================================================
# PHASE 4 — MANIFEST / DEDUP REFERENCE VERIFICATION
# ============================================================

import gzip
import json
from pathlib import Path

print("=" * 80)
print("PHASE 4 — MANIFEST / DEDUP REFERENCE VERIFICATION")
print("=" * 80)

DOMAIN_FILES = {}

for domain_dir in sorted(DEDUP_DIR.iterdir()):
    if not domain_dir.is_dir():
        continue

    files = sorted(domain_dir.glob("*.jsonl.gz"))
    DOMAIN_FILES[domain_dir.name] = files

    print(f"{domain_dir.name:20s}: {len(files)} files")

assert set(DOMAIN_FILES) == {
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals"
}

# Read a few manifest references and confirm that the referenced
# source file exists in the dedup view.
for manifest_name in [
    "train_manifest.jsonl",
    "validation_manifest.jsonl",
    "test_manifest.jsonl"
]:
    manifest_path = MANIFEST_DIR / manifest_name

    with open(manifest_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 10:
                break

            rec = json.loads(line)

            domain = rec["domain"]
            source_file = rec["source_file"]
            text_hash = rec["text_hash"]

            expected = DEDUP_DIR / domain / source_file.split("/")[-1]

            assert expected.exists(), (
                f"Referenced dedup file does not exist:\n{expected}"
            )

            assert isinstance(text_hash, str)
            assert len(text_hash) == 64

    print(f"✓ {manifest_name}")

print()
print("✓ All manifest references resolve to existing dedup files")
print("✓ No text duplication introduced")
print("✓ Manifests remain reference-only")

In [ ]:
# ============================================================
# PHASE 4 — BUILD STREAMING TEXT RESOLUTION INDEX
# ============================================================

import gzip
import json
import sqlite3
import time
from pathlib import Path

INDEX_DIR = BASE / "phase4_training_prep" / "index"
INDEX_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = INDEX_DIR / "text_index.sqlite"

# Start clean if an interrupted previous attempt exists.
if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(DB_PATH)

conn.execute("""
CREATE TABLE records (
    text_hash TEXT PRIMARY KEY,
    domain TEXT NOT NULL,
    source_file TEXT NOT NULL,
    line_number INTEGER NOT NULL
)
""")

conn.execute("""
CREATE INDEX idx_domain_file
ON records(domain, source_file)
""")

conn.commit()

print("=" * 80)
print("PHASE 4 — BUILD STREAMING TEXT RESOLUTION INDEX")
print("=" * 80)

total = 0
start = time.time()

for domain in ["acts", "high_courts", "supreme_court", "tribunals"]:

    files = DOMAIN_FILES[domain]

    print()
    print(f"DOMAIN: {domain}")

    for file_no, path in enumerate(files, 1):

        count = 0

        with gzip.open(path, "rt", encoding="utf-8") as f:

            for line_number, line in enumerate(f):

                rec = json.loads(line)

                text_hash = rec["text_hash"]

                conn.execute(
                    """
                    INSERT INTO records
                    (text_hash, domain, source_file, line_number)
                    VALUES (?, ?, ?, ?)
                    """,
                    (
                        text_hash,
                        domain,
                        path.name,
                        line_number
                    )
                )

                count += 1
                total += 1

                # Commit periodically so the cell is recoverable.
                if total % 50000 == 0:
                    conn.commit()

        conn.commit()

        print(
            f"[{file_no}/{len(files)}] "
            f"{path.name} | records={count:,}"
        )

conn.commit()

elapsed = time.time() - start

# Verify uniqueness.
indexed = conn.execute(
    "SELECT COUNT(*) FROM records"
).fetchone()[0]

unique_hashes = conn.execute(
    "SELECT COUNT(DISTINCT text_hash) FROM records"
).fetchone()[0]

print()
print("=" * 80)
print("INDEX SUMMARY")
print("=" * 80)

print(f"Records indexed       : {indexed:,}")
print(f"Unique text hashes    : {unique_hashes:,}")
print(f"Expected dedup records: 3,356,594")
print(f"Elapsed               : {elapsed/60:.2f} minutes")

assert indexed == 3_356_594
assert unique_hashes == 3_356_594

conn.close()

print()
print("✓ INDEX COUNT VERIFIED")
print("✓ EVERY DEDUP RECORD HAS A UNIQUE TEXT HASH")
print("✓ INDEX CONTAINS LOCATIONS ONLY")
print("✓ DOCUMENT TEXT WAS NOT COPIED")

In [ ]:
# ============================================================
# PHASE 4 — TEXT RESOLUTION TEST
# ============================================================

import gzip
import json
import sqlite3

conn = sqlite3.connect(DB_PATH)

def resolve_text(text_hash):
    row = conn.execute(
        """
        SELECT domain, source_file, line_number
        FROM records
        WHERE text_hash = ?
        """,
        (text_hash,)
    ).fetchone()

    if row is None:
        raise KeyError(f"Hash not found: {text_hash}")

    domain, source_file, line_number = row

    path = DEDUP_DIR / domain / source_file

    with gzip.open(path, "rt", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i == line_number:
                rec = json.loads(line)

                assert rec["text_hash"] == text_hash

                return rec["text"]

    raise RuntimeError("Indexed line could not be read")


print("=" * 80)
print("PHASE 4 — TEXT RESOLUTION TEST")
print("=" * 80)

manifest = MANIFEST_DIR / "train_manifest.jsonl"

with open(manifest, "r", encoding="utf-8") as f:
    samples = [json.loads(next(f)) for _ in range(5)]

for i, item in enumerate(samples, 1):

    text = resolve_text(item["text_hash"])

    print()
    print(f"Example {i}")
    print("-" * 80)
    print(f"Domain     : {item['domain']}")
    print(f"Hash       : {item['text_hash']}")
    print(f"Characters : {len(text):,}")
    print(f"Preview    : {text[:250]!r}")

    assert isinstance(text, str)
    assert len(text) > 0

conn.close()

print()
print("=" * 80)
print("✓ TEXT RESOLUTION VERIFIED")
print("✓ MANIFEST → HASH → DEDUP RECORD → TEXT WORKS")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — CLEAN FAILED TOKENIZER ATTEMPT
# ============================================================

import gc
import sqlite3
import shutil

gc.collect()

TOKENIZER_JSON = TOKENIZER_DIR / "legal_bpe_32k.json"

if TOKENIZER_JSON.exists():
    TOKENIZER_JSON.unlink()

# Close old global connection if it exists.
try:
    conn.close()
except Exception:
    pass

gc.collect()

print("=" * 80)
print("FAILED TOKENIZER ATTEMPT CLEANED")
print("=" * 80)

print(f"Tokenizer exists: {TOKENIZER_JSON.exists()}")
print("✓ No corpus files removed")
print("✓ No manifests removed")
print("✓ SQLite index retained")

In [ ]:
# ============================================================
# PHASE 4 — OPTIMIZED TRAINING TEXT STREAM
# ============================================================

import gzip
import json
import sqlite3
import time
from collections import defaultdict

print("=" * 80)
print("PHASE 4 — BUILD OPTIMIZED TRAINING TEXT STREAM")
print("=" * 80)

INDEX_DB = str(DB_PATH)

# ------------------------------------------------------------
# Load training hashes grouped by domain/source file.
# This is only metadata, NOT document text.
# ------------------------------------------------------------

needed = defaultdict(set)

train_manifest = MANIFEST_DIR / "train_manifest.jsonl"

with open(train_manifest, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)

        needed[
            (item["domain"], item["source_file"])
        ].add(item["text_hash"])

print(f"Training manifest records : {sum(len(v) for v in needed.values()):,}")
print(f"Shard/file groups         : {len(needed):,}")

# ------------------------------------------------------------
# Optimized generator
# ------------------------------------------------------------

def training_text_iterator():

    produced = 0
    start = time.time()

    for group_no, ((domain, source_file), hashes) in enumerate(
        sorted(needed.items()), 1
    ):

        path = DEDUP_DIR / domain / source_file

        if not path.exists():
            raise FileNotFoundError(path)

        remaining = set(hashes)

        with gzip.open(path, "rt", encoding="utf-8") as f:

            for line in f:

                rec = json.loads(line)

                h = rec["text_hash"]

                if h in remaining:

                    text = rec.get("text")

                    if isinstance(text, str) and text.strip():
                        produced += 1
                        yield text

                    remaining.remove(h)

                    if not remaining:
                        break

        if remaining:
            raise RuntimeError(
                f"{len(remaining):,} manifest hashes "
                f"could not be resolved in {path}"
            )

        if group_no % 10 == 0 or group_no == len(needed):
            elapsed = time.time() - start

            print(
                f"Groups processed: "
                f"{group_no:,}/{len(needed):,} | "
                f"documents yielded: {produced:,} | "
                f"time: {elapsed/60:.2f} min"
            )


print()
print("✓ Optimized iterator created")
print("✓ Each compressed source file is scanned at most once")
print("✓ No SQLite connection is used by the tokenizer")
print("✓ No document-text copy is created")

In [ ]:
# ============================================================
# PHASE 4 — OPTIMIZED TRAINING TEXT STREAM
# FIXED SOURCE-FILE PATH RESOLUTION
# ============================================================

import gzip
import json
import time
from collections import defaultdict

print("=" * 80)
print("PHASE 4 — BUILD OPTIMIZED TRAINING TEXT STREAM")
print("=" * 80)

# ------------------------------------------------------------
# Load training hashes grouped by ACTUAL dedup file.
# ------------------------------------------------------------

needed = defaultdict(set)

train_manifest = MANIFEST_DIR / "train_manifest.jsonl"

with open(train_manifest, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)

        domain = item["domain"]

        # Manifest contains e.g.
        # acts/documents.jsonl.gz
        #
        # Dedup layout is:
        # dedup/acts/documents.jsonl.gz
        #
        # Therefore strip the domain prefix if present.
        source_file = item["source_file"]

        prefix = domain + "/"

        if source_file.startswith(prefix):
            filename = source_file[len(prefix):]
        else:
            filename = source_file

        needed[
            (domain, filename)
        ].add(item["text_hash"])


total_needed = sum(len(v) for v in needed.values())

print(f"Training manifest records : {total_needed:,}")
print(f"Shard/file groups         : {len(needed):,}")

assert total_needed == 3_020_655

# ------------------------------------------------------------
# Optimized streaming generator
# ------------------------------------------------------------

def training_text_iterator():

    produced = 0
    start = time.time()

    groups = sorted(needed.items())

    for group_no, ((domain, filename), hashes) in enumerate(
        groups, 1
    ):

        path = DEDUP_DIR / domain / filename

        if not path.exists():
            raise FileNotFoundError(
                f"\nExpected file does not exist:\n"
                f"  {path}\n"
                f"\n"
                f"Domain   : {domain}\n"
                f"Filename : {filename}"
            )

        remaining = set(hashes)

        with gzip.open(path, "rt", encoding="utf-8") as f:

            for line in f:

                rec = json.loads(line)

                text_hash = rec["text_hash"]

                if text_hash in remaining:

                    text = rec.get("text")

                    if isinstance(text, str) and text.strip():

                        produced += 1
                        yield text

                    remaining.remove(text_hash)

                    if not remaining:
                        break

        if remaining:
            raise RuntimeError(
                f"\nCould not resolve "
                f"{len(remaining):,} hashes in:\n"
                f"{path}"
            )

        if (
            group_no % 10 == 0
            or group_no == len(groups)
        ):
            elapsed = time.time() - start

            print(
                f"Groups processed: "
                f"{group_no:,}/{len(groups):,} | "
                f"documents yielded: {produced:,} | "
                f"time: {elapsed/60:.2f} min"
            )


print()
print("✓ Corrected path resolution")
print("✓ Each gzip source file scanned at most once")
print("✓ No SQLite connection inside tokenizer")
print("✓ No corpus copy created")
print("✓ Ready for stream sanity test")

In [ ]:
# ============================================================
# PHASE 4 — OPTIMIZED STREAM SANITY TEST
# ============================================================

print("=" * 80)
print("PHASE 4 — OPTIMIZED STREAM SANITY TEST")
print("=" * 80)

count = 0
characters = 0

for text in training_text_iterator():

    count += 1
    characters += len(text)

    if count <= 3:
        print()
        print(f"Example {count}")
        print("-" * 80)
        print(f"Characters : {len(text):,}")
        print(f"Preview    : {text[:300]!r}")

    if count >= 100:
        break

print()
print("=" * 80)
print("SANITY TEST SUMMARY")
print("=" * 80)

print(f"Documents read : {count:,}")
print(f"Characters     : {characters:,}")

assert count == 100
assert characters > 0

print()
print("✓ 100 TRAINING DOCUMENTS RESOLVED")
print("✓ TEXT IS NON-EMPTY")
print("✓ MANIFEST → DEDUP → TEXT PATH WORKS")
print("✓ STREAM IS READY")

In [ ]:
# ============================================================
# PHASE 4 — OPTIMIZED TRAINING TEXT STREAM
# FIXED SOURCE-FILE PATH RESOLUTION
# ============================================================

import gzip
import json
import time
from collections import defaultdict

print("=" * 80)
print("PHASE 4 — BUILD OPTIMIZED TRAINING TEXT STREAM")
print("=" * 80)

# ------------------------------------------------------------
# Load training hashes grouped by actual dedup file
# ------------------------------------------------------------

needed = defaultdict(set)

train_manifest = MANIFEST_DIR / "train_manifest.jsonl"

with open(train_manifest, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)

        domain = item["domain"]
        source_file = item["source_file"]

        # Manifest example:
        # acts/documents.jsonl.gz
        #
        # Actual dedup path:
        # dedup/acts/documents.jsonl.gz

        prefix = domain + "/"

        if source_file.startswith(prefix):
            filename = source_file[len(prefix):]
        else:
            filename = source_file

        needed[(domain, filename)].add(
            item["text_hash"]
        )

total_needed = sum(
    len(hashes)
    for hashes in needed.values()
)

print(f"Training manifest records : {total_needed:,}")
print(f"Shard/file groups         : {len(needed):,}")

assert total_needed == 3_020_655

# ------------------------------------------------------------
# Streaming generator
# ------------------------------------------------------------

def training_text_iterator():

    produced = 0
    start = time.time()

    groups = sorted(needed.items())

    for group_no, ((domain, filename), hashes) in enumerate(
        groups, 1
    ):

        path = DEDUP_DIR / domain / filename

        if not path.exists():
            raise FileNotFoundError(
                f"\nExpected file does not exist:\n"
                f"  {path}\n"
                f"\n"
                f"Domain   : {domain}\n"
                f"Filename : {filename}"
            )

        remaining = set(hashes)

        with gzip.open(
            path,
            "rt",
            encoding="utf-8"
        ) as f:

            for line in f:

                rec = json.loads(line)

                text_hash = rec["text_hash"]

                if text_hash in remaining:

                    text = rec.get("text")

                    if (
                        isinstance(text, str)
                        and text.strip()
                    ):
                        produced += 1
                        yield text

                    remaining.remove(text_hash)

                    if not remaining:
                        break

        if remaining:
            raise RuntimeError(
                f"\nCould not resolve "
                f"{len(remaining):,} hashes in:\n"
                f"{path}"
            )

        if (
            group_no % 10 == 0
            or group_no == len(groups)
        ):
            elapsed = time.time() - start

            print(
                f"Groups processed: "
                f"{group_no:,}/{len(groups):,} | "
                f"documents yielded: "
                f"{produced:,} | "
                f"time: {elapsed / 60:.2f} min"
            )

print()
print("✓ Corrected path resolution")
print("✓ Each gzip source file scanned at most once")
print("✓ No SQLite connection inside tokenizer")
print("✓ No corpus copy created")
print("✓ Training stream ready")

In [ ]:
# ============================================================
# PHASE 4 — OPTIMIZED STREAM SANITY TEST
# ============================================================

print("=" * 80)
print("PHASE 4 — OPTIMIZED STREAM SANITY TEST")
print("=" * 80)

count = 0
characters = 0

for text in training_text_iterator():

    count += 1
    characters += len(text)

    if count <= 3:
        print()
        print(f"Example {count}")
        print("-" * 80)
        print(f"Characters : {len(text):,}")
        print(f"Preview    : {text[:300]!r}")

    if count >= 100:
        break

print()
print("=" * 80)
print("SANITY TEST SUMMARY")
print("=" * 80)

print(f"Documents read : {count:,}")
print(f"Characters     : {characters:,}")

assert count == 100
assert characters > 0

print()
print("✓ 100 TRAINING DOCUMENTS RESOLVED")
print("✓ TEXT IS NON-EMPTY")
print("✓ MANIFEST → DEDUP → TEXT PATH WORKS")
print("✓ STREAM IS READY")

In [ ]:
# =============================================================================
# PHASE 4 — PRE-TOKENIZATION STORAGE SAFETY CHECK
# =============================================================================

import shutil
from pathlib import Path

BASE = Path("/kaggle/working")
PHASE2 = BASE / "phase2_canonical"
PHASE4 = BASE / "phase4_training_prep"

total, used, free = shutil.disk_usage(BASE)

def gb(x):
    return x / (1024 ** 3)

print("=" * 80)
print("PHASE 4 — PRE-TOKENIZATION STORAGE SAFETY CHECK")
print("=" * 80)

print("\nKAGGLE STORAGE")
print("-" * 80)
print(f"Total : {gb(total):.2f} GB")
print(f"Used  : {gb(used):.2f} GB")
print(f"Free  : {gb(free):.2f} GB")

# -----------------------------------------------------------------------------
# Directory size helper
# -----------------------------------------------------------------------------

def directory_size(path):
    if not path.exists():
        return 0

    total_size = 0

    for p in path.rglob("*"):
        try:
            if p.is_file():
                total_size += p.stat().st_size
        except OSError:
            pass

    return total_size


# -----------------------------------------------------------------------------
# Important directories
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("IMPORTANT DIRECTORIES")
print("-" * 80)

paths = {
    "phase2_canonical": PHASE2,
    "canonical corpus_temp": PHASE2 / "corpus_temp",
    "phase4_training_prep": PHASE4,
    "phase4 dedup": PHASE4 / "dedup",
    "phase4 manifests": PHASE4 / "manifests",
    "phase4 tokenizer": PHASE4 / "tokenizer",
}

for name, path in paths.items():
    size = directory_size(path)

    print(
        f"{name:30s} : "
        f"{gb(size):6.2f} GB | "
        f"exists={path.exists()}"
    )


# -----------------------------------------------------------------------------
# Check large files
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("LARGEST FILES UNDER /kaggle/working")
print("-" * 80)

files = []

for p in BASE.rglob("*"):
    try:
        if p.is_file():
            files.append((p.stat().st_size, p))
    except OSError:
        pass

files.sort(reverse=True)

for size, path in files[:20]:
    print(f"{gb(size):7.3f} GB | {path}")


# -----------------------------------------------------------------------------
# Detect backup directories
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("BACKUP DIRECTORIES")
print("-" * 80)

backup_dirs = []

for p in BASE.rglob("*"):
    if p.is_dir() and "backup" in p.name.lower():
        backup_dirs.append(p)

if backup_dirs:
    for p in backup_dirs:
        print(
            f"{gb(directory_size(p)):6.2f} GB | {p}"
        )
else:
    print("No backup directories found.")


# -----------------------------------------------------------------------------
# Critical files
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("CRITICAL PHASE 4 FILES")
print("-" * 80)

critical = [
    PHASE4 / "phase4_dedup_audit.json",
    PHASE4 / "phase4_duplicate_audit.json",
    PHASE4 / "phase4_text_quality_audit.json",
    PHASE4 / "phase4_dedup_split_policy.json",
    PHASE4 / "manifests" / "train_manifest.jsonl",
    PHASE4 / "manifests" / "validation_manifest.jsonl",
    PHASE4 / "manifests" / "test_manifest.jsonl",
]

for p in critical:
    print(f"{'✓' if p.exists() else '✗'} {p}")


# -----------------------------------------------------------------------------
# Safety recommendation
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("STORAGE DECISION")
print("=" * 80)

free_gb = gb(free)

if free_gb < 2:
    print("⚠️  VERY LOW STORAGE")
    print("Do NOT start tokenizer training/output generation yet.")
elif free_gb < 4:
    print("⚠️  LIMITED STORAGE")
    print("Tokenizer training itself is possible, but avoid creating")
    print("large tokenized corpus files until storage is reclaimed.")
else:
    print("✓ STORAGE IS CURRENTLY SUFFICIENT FOR TOKENIZER TRAINING")

print("\nIMPORTANT:")
print("✓ Canonical corpus must remain untouched.")
print("✓ Deduplicated view must remain available.")
print("✓ Split manifests must remain available.")
print("✓ Do NOT delete phase4 audit/policy files.")
print("✓ Only confirmed redundant backups/temp artifacts should be removed.")

print("\n" + "=" * 80)
print("STORAGE SAFETY CHECK COMPLETE")
print("=" * 80)

In [ ]:
# =============================================================================
# PHASE 4 — PRE-CLEANUP DEPENDENCY AUDIT
# =============================================================================

from pathlib import Path
import os
import shutil

BASE = Path("/kaggle/working")

TARGETS = {
    "phase1_processed": BASE / "phase1_processed",
    "phase4_index": BASE / "phase4_training_prep" / "index",
    "phase4_dedup": BASE / "phase4_training_prep" / "dedup",
    "phase2_canonical": BASE / "phase2_canonical",
}

def size_gb(path):
    total = 0

    if not path.exists():
        return 0.0

    for p in path.rglob("*"):
        try:
            if p.is_file():
                total += p.stat().st_size
        except OSError:
            pass

    return total / (1024 ** 3)


print("=" * 80)
print("PHASE 4 — PRE-CLEANUP DEPENDENCY AUDIT")
print("=" * 80)

print("\nDIRECTORY SIZES")
print("-" * 80)

for name, path in TARGETS.items():
    print(
        f"{name:25s} : "
        f"{size_gb(path):6.2f} GB | "
        f"exists={path.exists()}"
    )


# -----------------------------------------------------------------------------
# Inspect Phase 1 processed files
# -----------------------------------------------------------------------------

phase1 = TARGETS["phase1_processed"]

print("\n" + "=" * 80)
print("PHASE 1 PROCESSED CONTENT")
print("-" * 80)

if phase1.exists():
    files = []

    for p in phase1.rglob("*"):
        if p.is_file():
            try:
                files.append((p.stat().st_size, p))
            except OSError:
                pass

    files.sort(reverse=True)

    print(f"Files found : {len(files):,}")

    for size, path in files[:30]:
        print(f"{size / (1024**3):7.3f} GB | {path}")
else:
    print("phase1_processed does not exist.")


# -----------------------------------------------------------------------------
# Inspect Phase 4 index
# -----------------------------------------------------------------------------

index_dir = TARGETS["phase4_index"]

print("\n" + "=" * 80)
print("PHASE 4 INDEX")
print("-" * 80)

if index_dir.exists():
    for p in index_dir.rglob("*"):
        if p.is_file():
            print(
                f"{p.stat().st_size / (1024**3):7.3f} GB | {p}"
            )
else:
    print("Phase 4 index directory does not exist.")


# -----------------------------------------------------------------------------
# Search current Phase 4 files for references to the SQLite index
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("CURRENT PYTHON FILES REFERENCING text_index.sqlite")
print("-" * 80)

matches = []

for p in BASE.rglob("*.py"):
    try:
        text = p.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        if "text_index.sqlite" in text:
            matches.append(p)

    except Exception:
        pass

if matches:
    for p in matches:
        print(f"FOUND : {p}")
else:
    print("No Python source files reference text_index.sqlite.")


# -----------------------------------------------------------------------------
# Search for phase1_processed references
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("CURRENT PYTHON FILES REFERENCING phase1_processed")
print("-" * 80)

matches = []

for p in BASE.rglob("*.py"):
    try:
        text = p.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        if "phase1_processed" in text:
            matches.append(p)

    except Exception:
        pass

if matches:
    for p in matches:
        print(f"FOUND : {p}")
else:
    print("No Python source files reference phase1_processed.")


# -----------------------------------------------------------------------------
# Storage
# -----------------------------------------------------------------------------

total, used, free = shutil.disk_usage(BASE)

print("\n" + "=" * 80)
print("CURRENT STORAGE")
print("-" * 80)

print(f"Total : {total / (1024**3):.2f} GB")
print(f"Used  : {used  / (1024**3):.2f} GB")
print(f"Free  : {free  / (1024**3):.2f} GB")

print("\n" + "=" * 80)
print("NO FILES WERE DELETED")
print("=" * 80)

In [ ]:
# =============================================================================
# PHASE 4 — SAFE REDUNDANT ARTIFACT CLEANUP
# =============================================================================

from pathlib import Path
import shutil
import time

BASE = Path("/kaggle/working")

PHASE1 = BASE / "phase1_processed"
INDEX_DB = BASE / "phase4_training_prep" / "index" / "text_index.sqlite"

CLEANUP_DIR = BASE / "phase4_cleanup_staging"
CLEANUP_DIR.mkdir(exist_ok=True)

PHASE1_STAGING = CLEANUP_DIR / "phase1_processed"
INDEX_STAGING = CLEANUP_DIR / "text_index.sqlite"

print("=" * 80)
print("PHASE 4 — SAFE REDUNDANT ARTIFACT CLEANUP")
print("=" * 80)

print("\nTARGETS")
print("-" * 80)
print(f"Phase 1 processed : {PHASE1}")
print(f"Phase 4 index DB  : {INDEX_DB}")

# -----------------------------------------------------------------------------
# Safety assertions
# -----------------------------------------------------------------------------

assert PHASE1.exists(), f"Missing expected directory: {PHASE1}"
assert INDEX_DB.exists(), f"Missing expected index: {INDEX_DB}"

assert not PHASE1_STAGING.exists(), (
    f"Staging destination already exists: {PHASE1_STAGING}"
)

assert not INDEX_STAGING.exists(), (
    f"Staging destination already exists: {INDEX_STAGING}"
)

# -----------------------------------------------------------------------------
# Show sizes before moving
# -----------------------------------------------------------------------------

def get_size(path):
    if path.is_file():
        return path.stat().st_size

    total = 0
    for p in path.rglob("*"):
        try:
            if p.is_file():
                total += p.stat().st_size
        except OSError:
            pass

    return total

def gb(size):
    return size / (1024 ** 3)

phase1_size = get_size(PHASE1)
index_size = get_size(INDEX_DB)

print("\nSIZE TO BE RECLAIMED")
print("-" * 80)
print(f"phase1_processed : {gb(phase1_size):.2f} GB")
print(f"text_index.sqlite : {gb(index_size):.2f} GB")
print(f"TOTAL             : {gb(phase1_size + index_size):.2f} GB")

# -----------------------------------------------------------------------------
# Move rather than delete
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("MOVING REDUNDANT ARTIFACTS TO CLEANUP STAGING")
print("=" * 80)

shutil.move(str(PHASE1), str(PHASE1_STAGING))
print("✓ phase1_processed moved")

shutil.move(str(INDEX_DB), str(INDEX_STAGING))
print("✓ text_index.sqlite moved")

# Remove now-empty index directory if possible.
index_dir = INDEX_DB.parent

try:
    index_dir.rmdir()
    print("✓ Empty Phase 4 index directory removed")
except OSError:
    print("Phase 4 index directory retained because it is not empty.")

# -----------------------------------------------------------------------------
# Verify active pipeline remains intact
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("ACTIVE PIPELINE VERIFICATION")
print("=" * 80)

checks = {
    "phase2 canonical": BASE / "phase2_canonical" / "corpus_temp",
    "phase4 dedup": BASE / "phase4_training_prep" / "dedup",
    "train manifest":
        BASE / "phase4_training_prep" / "manifests" / "train_manifest.jsonl",
    "validation manifest":
        BASE / "phase4_training_prep" / "manifests" / "validation_manifest.jsonl",
    "test manifest":
        BASE / "phase4_training_prep" / "manifests" / "test_manifest.jsonl",
    "dedup audit":
        BASE / "phase4_training_prep" / "phase4_dedup_audit.json",
}

all_ok = True

for name, path in checks.items():
    exists = path.exists()
    print(f"{'✓' if exists else '✗'} {name}")
    all_ok &= exists

assert all_ok, "ACTIVE PIPELINE VERIFICATION FAILED"

# -----------------------------------------------------------------------------
# Storage after cleanup
# -----------------------------------------------------------------------------

total, used, free = shutil.disk_usage(BASE)

print("\n" + "=" * 80)
print("STORAGE AFTER CLEANUP")
print("=" * 80)

print(f"Total : {gb(total):.2f} GB")
print(f"Used  : {gb(used):.2f} GB")
print(f"Free  : {gb(free):.2f} GB")

print("\n" + "=" * 80)
print("CLEANUP CHECKPOINT")
print("=" * 80)

print("✓ phase1_processed removed from active workspace")
print("✓ obsolete text_index.sqlite removed from active workspace")
print("✓ canonical corpus preserved")
print("✓ deduplicated corpus preserved")
print("✓ split manifests preserved")
print("✓ Phase 4 audits preserved")
print("✓ No active training data deleted")

print("\nCleanup staging:")
print(f"  {CLEANUP_DIR}")

print("\nIMPORTANT:")
print("The staged files have NOT been permanently deleted.")
print("Once the tokenizer pipeline is verified, the staging directory")
print("can be permanently removed to reclaim the remaining space.")

print("=" * 80)

In [ ]:
# =============================================================================
# PHASE 4 — FINAL PRE-TOKENIZER VERIFICATION
# =============================================================================

from pathlib import Path
import json
import gzip
import shutil
import time

BASE = Path("/kaggle/working")
PHASE4 = BASE / "phase4_training_prep"

CANONICAL = BASE / "phase2_canonical" / "corpus_temp"
DEDUP = PHASE4 / "dedup"
MANIFESTS = PHASE4 / "manifests"
STAGING = BASE / "phase4_cleanup_staging"

TRAIN_MANIFEST = MANIFESTS / "train_manifest.jsonl"
VAL_MANIFEST = MANIFESTS / "validation_manifest.jsonl"
TEST_MANIFEST = MANIFESTS / "test_manifest.jsonl"

print("=" * 80)
print("PHASE 4 — FINAL PRE-TOKENIZER VERIFICATION")
print("=" * 80)

required = {
    "canonical corpus": CANONICAL,
    "dedup view": DEDUP,
    "train manifest": TRAIN_MANIFEST,
    "validation manifest": VAL_MANIFEST,
    "test manifest": TEST_MANIFEST,
}

for name, path in required.items():
    print(f"{name:<24}: {path.exists()}")

assert CANONICAL.exists()
assert DEDUP.exists()
assert TRAIN_MANIFEST.exists()
assert VAL_MANIFEST.exists()
assert TEST_MANIFEST.exists()

# Verify staged cleanup exists but remains untouched.
print("\nCleanup staging:")
print(f"  exists : {STAGING.exists()}")

# Count manifest records without loading them into memory.
def count_lines(path):
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for _ in f)

train_n = count_lines(TRAIN_MANIFEST)
val_n = count_lines(VAL_MANIFEST)
test_n = count_lines(TEST_MANIFEST)

print("\nManifest counts:")
print(f"  train      : {train_n:,}")
print(f"  validation : {val_n:,}")
print(f"  test       : {test_n:,}")
print(f"  total      : {train_n + val_n + test_n:,}")

assert train_n == 3_020_655
assert val_n == 168_665
assert test_n == 167_274

print("\n✓ Canonical corpus available")
print("✓ Deduplicated corpus available")
print("✓ All split manifests available")
print("✓ Manifest counts unchanged")
print("✓ Cleanup staging has NOT been deleted")
print("=" * 80)

In [ ]:
# =============================================================================
# PHASE 4 — PERMANENTLY REMOVE VERIFIED CLEANUP STAGING
# =============================================================================

from pathlib import Path
import shutil
import time

BASE = Path("/kaggle/working")
STAGING = BASE / "phase4_cleanup_staging"

print("=" * 80)
print("PHASE 4 — PERMANENT CLEANUP OF VERIFIED REDUNDANT ARTIFACTS")
print("=" * 80)

if not STAGING.exists():
    print("Cleanup staging does not exist.")
    print("Nothing to delete.")
else:
    print(f"Deleting:")
    print(f"  {STAGING}")
    
    shutil.rmtree(STAGING)

    print("\n✓ Cleanup staging permanently deleted")

assert not STAGING.exists()

print("=" * 80)
print("PERMANENT CLEANUP COMPLETE")
print("=" * 80)

In [ ]:
# =============================================================================
# PHASE 4 — POST-CLEANUP STORAGE CHECK
# =============================================================================

from pathlib import Path
import shutil

BASE = Path("/kaggle/working")

def gb(x):
    return x / (1024 ** 3)

total, used, free = shutil.disk_usage(BASE)

print("=" * 80)
print("PHASE 4 — POST-CLEANUP STORAGE CHECK")
print("=" * 80)

print(f"\nTotal : {gb(total):.2f} GB")
print(f"Used  : {gb(used):.2f} GB")
print(f"Free  : {gb(free):.2f} GB")

print("\nACTIVE PIPELINE")
print("-" * 80)

paths = {
    "phase2_canonical": BASE / "phase2_canonical",
    "phase4_training_prep": BASE / "phase4_training_prep",
    "dedup": BASE / "phase4_training_prep" / "dedup",
    "manifests": BASE / "phase4_training_prep" / "manifests",
}

for name, path in paths.items():
    print(f"{name:<28}: exists={path.exists()}")

print("\nREMOVED ARTIFACTS")
print("-" * 80)

print(
    f"phase1_processed exists : "
    f"{(BASE / 'phase1_processed').exists()}"
)

print(
    f"cleanup staging exists  : "
    f"{(BASE / 'phase4_cleanup_staging').exists()}"
)

print(
    f"text_index.sqlite exists: "
    f"{(BASE / 'phase4_training_prep' / 'index' / 'text_index.sqlite').exists()}"
)

print("=" * 80)
print("✓ STORAGE CHECK COMPLETE")
print("=" * 80)

In [ ]:
# =============================================================================
# PHASE 4 — BPE 32K TOKENIZER CONFIGURATION
# =============================================================================

from pathlib import Path
import json
from datetime import datetime, timezone

PHASE4 = Path("/kaggle/working/phase4_training_prep")
TOKENIZER_DIR = PHASE4 / "tokenizer"

TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

config = {
    "phase": "phase4",
    "tokenizer": "BPE",
    "vocab_size": 32000,

    "special_tokens": [
        "[PAD]",
        "[UNK]",
        "[BOS]",
        "[EOS]",
    ],

    "training_source": "train_manifest.jsonl",

    "train_manifest": str(
        PHASE4 / "manifests" / "train_manifest.jsonl"
    ),

    "dedup_root": str(
        PHASE4 / "dedup"
    ),

    "seed": 42,

    "canonical_corpus_modified": False,

    "tokenized_corpus_materialized": False,

    "created_at": datetime.now(timezone.utc).isoformat(),
}

CONFIG_PATH = TOKENIZER_DIR / "tokenizer_config.json"

with CONFIG_PATH.open("w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("=" * 80)
print("PHASE 4 — BPE 32K TOKENIZER CONFIGURATION")
print("=" * 80)

print(f"\nTokenizer      : BPE")
print(f"Vocabulary     : 32,000")
print(f"Training split : train")
print(f"Seed           : 42")
print(f"Output         : {TOKENIZER_DIR}")
print(f"\nConfig saved   : {CONFIG_PATH}")

print("\n✓ Configuration created")
print("=" * 80)

In [ ]:
# =============================================================================
# PHASE 4 — TRAIN BPE 32K TOKENIZER
# =============================================================================

from pathlib import Path
import gzip
import json
import time

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.normalizers import NFKC

PHASE4 = Path("/kaggle/working/phase4_training_prep")

MANIFEST = PHASE4 / "manifests" / "train_manifest.jsonl"
DEDUP_ROOT = PHASE4 / "dedup"
TOKENIZER_DIR = PHASE4 / "tokenizer"

TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

assert MANIFEST.exists(), MANIFEST
assert DEDUP_ROOT.exists(), DEDUP_ROOT

print("=" * 80)
print("PHASE 4 — BPE 32K TOKENIZER TRAINING")
print("=" * 80)

# -------------------------------------------------------------------------
# Resolve manifest source_file safely.
# -------------------------------------------------------------------------

def resolve_source_file(domain, source_file):
    """
    Manifest source_file is expected to look like:
        acts/documents.jsonl.gz
        high_courts/shard_00000.jsonl.gz

    Resolve relative to DEDUP_ROOT exactly once.
    """

    source_file = Path(source_file)

    # Normal expected case:
    candidate = DEDUP_ROOT / source_file

    if candidate.exists():
        return candidate

    # Defensive handling for accidental duplicated domain prefix.
    parts = source_file.parts

    if len(parts) >= 2 and parts[0] == domain:
        candidate = DEDUP_ROOT / Path(*parts)
        if candidate.exists():
            return candidate

    # If source_file is only the filename.
    candidate = DEDUP_ROOT / domain / source_file.name

    if candidate.exists():
        return candidate

    raise FileNotFoundError(
        f"Unable to resolve source file:\n"
        f"domain={domain}\n"
        f"source_file={source_file}"
    )


# -------------------------------------------------------------------------
# Streaming text iterator
# -------------------------------------------------------------------------

def training_text_iterator():
    current_path = None
    handle = None

    try:
        with MANIFEST.open("r", encoding="utf-8") as manifest:

            for line in manifest:

                item = json.loads(line)

                domain = item["domain"]
                source_file = item["source_file"]
                expected_hash = item["text_hash"]

                path = resolve_source_file(domain, source_file)

                if path != current_path:

                    if handle is not None:
                        handle.close()

                    handle = gzip.open(
                        path,
                        "rt",
                        encoding="utf-8",
                        errors="replace",
                    )

                    current_path = path

                # The manifest identifies the record by hash.
                # Since we need arbitrary records from compressed JSONL,
                # scan this file sequentially until the hash is found.
                #
                # This implementation is intentionally correctness-first.
                # We will optimize repeated-file lookup if necessary.

                found = False

                for raw in handle:

                    row = json.loads(raw)

                    if row.get("text_hash") == expected_hash:

                        text = row.get("text", "")

                        if text and text.strip():
                            yield text

                        found = True
                        break

                if not found:
                    raise RuntimeError(
                        f"Manifest hash not found:\n"
                        f"{expected_hash}\n"
                        f"file={path}"
                    )

    finally:
        if handle is not None:
            handle.close()


# -------------------------------------------------------------------------
# IMPORTANT:
# The above naive sequential lookup is correctness-safe but can be very
# expensive because manifests are not guaranteed to be grouped by file.
#
# We therefore first perform a small benchmark before committing to the
# complete tokenizer training.
# -------------------------------------------------------------------------

print("\nRunning 100-document tokenizer stream benchmark...")

start = time.time()

count = 0
chars = 0

for text in training_text_iterator():

    count += 1
    chars += len(text)

    if count >= 100:
        break

elapsed = time.time() - start

print(f"Documents : {count:,}")
print(f"Characters: {chars:,}")
print(f"Elapsed   : {elapsed:.2f} sec")

assert count == 100
assert chars > 0

print("\n✓ 100 training documents resolved")
print("✓ Manifest → dedup → text stream works")

print("\nIMPORTANT:")
print("This cell has ONLY benchmarked the tokenizer iterator.")
print("It has NOT yet started the full 3M-document BPE training.")

print("=" * 80)

In [ ]:
# =============================================================================
# PHASE 4 — BUILD FILE-GROUPED TOKENIZER TRAINING PLAN
# =============================================================================

from pathlib import Path
import json
from collections import defaultdict

PHASE4 = Path("/kaggle/working/phase4_training_prep")

MANIFEST = PHASE4 / "manifests" / "train_manifest.jsonl"
PLAN = PHASE4 / "tokenizer" / "train_file_plan.json"

print("=" * 80)
print("PHASE 4 — FILE-GROUPED TOKENIZER TRAINING PLAN")
print("=" * 80)

groups = defaultdict(int)
total = 0

with MANIFEST.open("r", encoding="utf-8") as f:

    for line in f:

        item = json.loads(line)

        key = (
            item["domain"],
            item["source_file"],
        )

        groups[key] += 1
        total += 1


plan = {
    "total_training_records": total,
    "source_files": [
        {
            "domain": domain,
            "source_file": source_file,
            "records": count,
        }
        for (domain, source_file), count in sorted(groups.items())
    ],
}


with PLAN.open("w", encoding="utf-8") as f:
    json.dump(plan, f, indent=2)


print(f"\nTraining records : {total:,}")
print(f"Source files     : {len(plan['source_files']):,}")


print("\nRecords by domain:")
print("-" * 80)

domain_totals = defaultdict(int)
domain_files = defaultdict(int)

for item in plan["source_files"]:

    domain = item["domain"]

    domain_totals[domain] += item["records"]
    domain_files[domain] += 1


for domain in sorted(domain_totals):

    print(
        f"{domain:<18}: "
        f"{domain_totals[domain]:>12,} records | "
        f"{domain_files[domain]:>3} files"
    )


# -------------------------------------------------------------------------
# HARD CHECKS
# -------------------------------------------------------------------------

assert total == 3_020_655

assert (
    sum(item["records"] for item in plan["source_files"])
    == 3_020_655
)


print("\nPlan saved:")
print(f"  {PLAN}")

print("\n✓ Training record count verified")
print("✓ File grouping complete")
print("✓ Number of files determined from the manifest")
print("✓ No hard-coded file-count assumption")
print("✓ No corpus copy created")

print("=" * 80)

In [ ]:
# =============================================================================
# PHASE 4 — VERIFY FILE-GROUPED TRAINING PLAN
# =============================================================================

from pathlib import Path
import json

PHASE4 = Path("/kaggle/working/phase4_training_prep")

DEDUP = PHASE4 / "dedup"
PLAN = PHASE4 / "tokenizer" / "train_file_plan.json"

with PLAN.open("r", encoding="utf-8") as f:
    plan = json.load(f)


print("=" * 80)
print("PHASE 4 — TRAINING PLAN VERIFICATION")
print("=" * 80)


total_expected = plan["total_training_records"]
total_plan = 0
missing = []


for item in plan["source_files"]:

    domain = item["domain"]
    source_file = Path(item["source_file"])

    # Normal manifest format:
    # acts/documents.jsonl.gz
    # high_courts/shard_00000.jsonl.gz

    path = DEDUP / source_file

    # Defensive fallback:
    if not path.exists():
        path = DEDUP / domain / source_file.name

    if not path.exists():

        missing.append({
            "domain": domain,
            "source_file": str(source_file),
            "resolved_path": str(path),
        })

    total_plan += item["records"]


print(f"\nExpected records : {total_expected:,}")
print(f"Planned records  : {total_plan:,}")
print(f"Source files     : {len(plan['source_files']):,}")


print("\nMissing source files:")
print("-" * 80)

if missing:

    for item in missing:
        print(item)

else:
    print("None")


# -------------------------------------------------------------------------
# HARD CHECKS
# -------------------------------------------------------------------------

assert total_expected == 3_020_655
assert total_plan == 3_020_655
assert len(missing) == 0


print("\n✓ Every source file exists")
print("✓ Every training record is represented")
print("✓ Training plan is internally consistent")
print("=" * 80)

In [ ]:
# =============================================================================
# PHASE 4 — OPTIMIZED FILE-GROUPED TEXT STREAM BENCHMARK
# =============================================================================

from pathlib import Path
import gzip
import json
import time

PHASE4 = Path("/kaggle/working/phase4_training_prep")

MANIFEST = PHASE4 / "manifests" / "train_manifest.jsonl"
DEDUP = PHASE4 / "dedup"


print("=" * 80)
print("PHASE 4 — OPTIMIZED FILE-GROUPED TEXT STREAM")
print("=" * 80)


# -------------------------------------------------------------------------
# Build:
#
#   (domain, source_file) -> set(text_hash)
#
# -------------------------------------------------------------------------

hashes_by_file = {}


with MANIFEST.open("r", encoding="utf-8") as f:

    for line in f:

        item = json.loads(line)

        key = (
            item["domain"],
            item["source_file"],
        )

        if key not in hashes_by_file:
            hashes_by_file[key] = set()

        hashes_by_file[key].add(item["text_hash"])


training_hash_count = sum(
    len(v) for v in hashes_by_file.values()
)


print(f"\nManifest source files : {len(hashes_by_file):,}")
print(f"Training hashes       : {training_hash_count:,}")


assert training_hash_count == 3_020_655


# -------------------------------------------------------------------------
# Resolve source path
# -------------------------------------------------------------------------

def resolve_path(domain, source_file):

    p = DEDUP / Path(source_file)

    if p.exists():
        return p

    p = DEDUP / domain / Path(source_file).name

    if p.exists():
        return p

    raise FileNotFoundError(
        f"\nCannot resolve source file:\n"
        f"domain      = {domain}\n"
        f"source_file = {source_file}\n"
        f"checked     = {DEDUP / Path(source_file)}\n"
        f"checked     = {DEDUP / domain / Path(source_file).name}"
    )


# -------------------------------------------------------------------------
# Optimized iterator
#
# Each compressed JSONL source file is opened once and scanned once.
# -------------------------------------------------------------------------

def optimized_training_text_iterator():

    files_scanned = 0

    for (domain, source_file), wanted_hashes in hashes_by_file.items():

        path = resolve_path(domain, source_file)

        files_scanned += 1

        with gzip.open(
            path,
            "rt",
            encoding="utf-8",
            errors="replace",
        ) as f:

            for line in f:

                row = json.loads(line)

                text_hash = row.get("text_hash")

                if text_hash not in wanted_hashes:
                    continue

                text = row.get("text", "")

                if text and text.strip():
                    yield text


# -------------------------------------------------------------------------
# Benchmark first 1,000 documents
# -------------------------------------------------------------------------

start = time.time()

count = 0
characters = 0

for text in optimized_training_text_iterator():

    count += 1
    characters += len(text)

    if count >= 1_000:
        break


elapsed = time.time() - start


print("\nBENCHMARK")
print("-" * 80)
print(f"Documents : {count:,}")
print(f"Characters: {characters:,}")
print(f"Elapsed   : {elapsed:.2f} sec")


assert count == 1_000
assert characters > 0


print("\n✓ 1,000 training documents resolved")
print("✓ Manifest → grouped source → dedup → text works")
print("✓ Each source file is streamed sequentially")
print("✓ No SQLite dependency")
print("✓ No tokenized corpus created")
print("✓ No corpus copy created")

print("=" * 80)

In [ ]:
from pathlib import Path

BASE = Path("/kaggle/working")
PHASE4 = BASE / "phase4_training_prep"

checks = {
    "canonical": BASE / "phase2_canonical",
    "dedup": PHASE4 / "dedup",
    "train_manifest": PHASE4 / "manifests/train_manifest.jsonl",
    "validation_manifest": PHASE4 / "manifests/validation_manifest.jsonl",
    "test_manifest": PHASE4 / "manifests/test_manifest.jsonl",
    "dedup_audit": PHASE4 / "phase4_dedup_audit.json",
}

print("=" * 70)
print("POST-RESTART PHASE 4 FILE CHECK")
print("=" * 70)

for name, path in checks.items():
    print(f"{name:20s}: {path.exists()}")

print("=" * 70)

In [ ]:
from pathlib import Path

TOKENIZER_DIR = Path("/kaggle/working/phase4_training_prep/tokenizer")
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

tmp = TOKENIZER_DIR / "legal_bpe_32k.json.tmp"

if tmp.exists():
    tmp.unlink()
    print("✓ Removed incomplete temporary tokenizer")
else:
    print("✓ No temporary tokenizer found")

In [ ]:
# =============================================================================
# PHASE 4 — BPE 32K TOKENIZER TRAINING
# PROGRESS LOGGING + SAFE OUTPUT
# =============================================================================

import gzip
import json
import time
from pathlib import Path
from collections import defaultdict

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel


# =============================================================================
# 1. PATHS
# =============================================================================

BASE = Path("/kaggle/working")

PHASE4 = BASE / "phase4_training_prep"
DEDUP_ROOT = PHASE4 / "dedup"
MANIFEST = PHASE4 / "manifests" / "train_manifest.jsonl"
TOKENIZER_DIR = PHASE4 / "tokenizer"

TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

FINAL_TOKENIZER = TOKENIZER_DIR / "legal_bpe_32k.json"
TEMP_TOKENIZER = TOKENIZER_DIR / "legal_bpe_32k.json.tmp"


# =============================================================================
# 2. CONFIGURATION
# =============================================================================

EXPECTED_TRAIN = 3_020_655

VOCAB_SIZE = 32_000
MIN_FREQUENCY = 2
SEED = 42

# Progress message every N documents.
LOG_EVERY = 100_000

SPECIAL_TOKENS = [
    "[PAD]",
    "[UNK]",
    "[BOS]",
    "[EOS]",
]


# =============================================================================
# 3. SAFETY CHECKS
# =============================================================================

print("=" * 80)
print("PHASE 4 — BPE 32K TOKENIZER TRAINING")
print("=" * 80)

assert PHASE4.exists(), f"Missing Phase 4 directory: {PHASE4}"
assert DEDUP_ROOT.exists(), f"Missing dedup directory: {DEDUP_ROOT}"
assert MANIFEST.exists(), f"Missing training manifest: {MANIFEST}"

print()
print(f"Phase 4       : {PHASE4}")
print(f"Dedup root    : {DEDUP_ROOT}")
print(f"Train manifest: {MANIFEST}")
print(f"Vocabulary    : {VOCAB_SIZE:,}")
print(f"Seed          : {SEED}")
print(f"Log interval  : {LOG_EVERY:,} documents")
print()


# =============================================================================
# 4. REMOVE ONLY INCOMPLETE TEMPORARY OUTPUT
# =============================================================================

if TEMP_TOKENIZER.exists():
    TEMP_TOKENIZER.unlink()
    print("✓ Removed previous temporary tokenizer")
else:
    print("✓ No temporary tokenizer found")

print()


# =============================================================================
# 5. BUILD SOURCE-FILE → TRAINING-HASH GROUPS
#
# Manifest format:
#
# domain
# source_file
# text_hash
# split
#
# IMPORTANT:
# source_file is already relative to DEDUP_ROOT.
#
# Example:
#   acts/documents.jsonl.gz
#
# Therefore:
#   DEDUP_ROOT / source_file
#
# NOT:
#   DEDUP_ROOT / domain / source_file
# =============================================================================

print("=" * 80)
print("BUILDING TRAINING SOURCE PLAN")
print("=" * 80)

groups = defaultdict(set)

manifest_count = 0

with MANIFEST.open("r", encoding="utf-8") as f:

    for line in f:

        if not line.strip():
            continue

        row = json.loads(line)

        source_file = row["source_file"]
        text_hash = row["text_hash"]

        groups[source_file].add(text_hash)

        manifest_count += 1


print()
print(f"Training records : {manifest_count:,}")
print(f"Source files     : {len(groups):,}")

assert manifest_count == EXPECTED_TRAIN, (
    f"Training manifest count mismatch: "
    f"{manifest_count:,} != {EXPECTED_TRAIN:,}"
)

print("✓ Training manifest count verified")
print()


# =============================================================================
# 6. SHOW SOURCE PLAN
# =============================================================================

print("=" * 80)
print("SOURCE FILE PLAN")
print("=" * 80)

for i, (source_file, hashes) in enumerate(groups.items(), 1):

    print(
        f"{i:02d}. {source_file:<60} "
        f"| training hashes = {len(hashes):,}"
    )

print()


# =============================================================================
# 7. CREATE BPE TOKENIZER
# =============================================================================

tokenizer = Tokenizer(
    BPE(unk_token="[UNK]")
)

tokenizer.pre_tokenizer = ByteLevel(
    add_prefix_space=False
)

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=SPECIAL_TOKENS,
    show_progress=False,
)


# =============================================================================
# 8. STREAMING ITERATOR WITH PROGRESS LOGGING
# =============================================================================

documents_seen = 0
characters_seen = 0

training_start = time.time()


def training_text_iterator():

    global documents_seen
    global characters_seen

    for source_file, training_hashes in groups.items():

        path = DEDUP_ROOT / source_file

        if not path.exists():
            raise FileNotFoundError(
                f"Required dedup source file does not exist:\n{path}"
            )

        print(
            f"[SOURCE] {source_file} "
            f"| expected hashes={len(training_hashes):,}"
        )

        with gzip.open(path, "rt", encoding="utf-8") as f:

            for line in f:

                if not line.strip():
                    continue

                row = json.loads(line)

                text_hash = row.get("text_hash")

                if text_hash not in training_hashes:
                    continue

                text = row.get("text", "")

                if not text:
                    continue

                documents_seen += 1
                characters_seen += len(text)

                # ---------------------------------------------------------
                # PROGRESS REPORT
                # ---------------------------------------------------------

                if (
                    documents_seen % LOG_EVERY == 0
                    or documents_seen == EXPECTED_TRAIN
                ):

                    elapsed = time.time() - training_start

                    rate = (
                        documents_seen / elapsed
                        if elapsed > 0
                        else 0
                    )

                    percent = (
                        documents_seen /
                        EXPECTED_TRAIN *
                        100
                    )

                    remaining_docs = (
                        EXPECTED_TRAIN -
                        documents_seen
                    )

                    eta_seconds = (
                        remaining_docs / rate
                        if rate > 0
                        else 0
                    )

                    print()
                    print("=" * 80)
                    print("BPE TRAINING PROGRESS")
                    print("=" * 80)

                    print(
                        f"Documents : "
                        f"{documents_seen:,} / "
                        f"{EXPECTED_TRAIN:,}"
                    )

                    print(
                        f"Progress  : "
                        f"{percent:6.2f}%"
                    )

                    print(
                        f"Characters: "
                        f"{characters_seen:,}"
                    )

                    print(
                        f"Rate      : "
                        f"{rate:,.1f} documents/sec"
                    )

                    print(
                        f"Elapsed   : "
                        f"{elapsed / 60:,.2f} minutes"
                    )

                    print(
                        f"ETA       : "
                        f"{eta_seconds / 60:,.2f} minutes"
                    )

                    print("=" * 80)
                    print()

                yield text


# =============================================================================
# 9. START FULL TRAINING
# =============================================================================

print("=" * 80)
print("STARTING FULL BPE TRAINING")
print("=" * 80)

print()
print(
    f"Target documents : {EXPECTED_TRAIN:,}"
)
print(
    f"First progress update after "
    f"{LOG_EVERY:,} documents."
)
print()
print("Tokenizer training is now running...")
print()


try:

    tokenizer.train_from_iterator(
        training_text_iterator(),
        trainer=trainer,
    )

except KeyboardInterrupt:

    print()
    print("=" * 80)
    print("TOKENIZER TRAINING INTERRUPTED")
    print("=" * 80)

    print(
        f"Documents processed : "
        f"{documents_seen:,}"
    )

    print(
        f"Characters processed: "
        f"{characters_seen:,}"
    )

    print()

    # Never leave an interrupted temporary tokenizer
    # looking like a valid output.
    if TEMP_TOKENIZER.exists():
        TEMP_TOKENIZER.unlink()

        print(
            "✓ Temporary tokenizer removed"
        )

    print()
    print(
        "Canonical corpus, dedup corpus, "
        "and manifests were not modified."
    )

    raise


# =============================================================================
# 10. VERIFY COMPLETE STREAM
# =============================================================================

print()
print("=" * 80)
print("STREAM COMPLETION CHECK")
print("=" * 80)

print(
    f"Documents streamed : "
    f"{documents_seen:,}"
)

print(
    f"Expected            : "
    f"{EXPECTED_TRAIN:,}"
)

assert documents_seen == EXPECTED_TRAIN, (
    f"Training stream mismatch: "
    f"{documents_seen:,} != {EXPECTED_TRAIN:,}"
)

print("✓ ALL TRAINING DOCUMENTS CONSUMED")


# =============================================================================
# 11. SAVE TO TEMPORARY FILE
# =============================================================================

print()
print("=" * 80)
print("SAVING TOKENIZER")
print("=" * 80)

tokenizer.save(str(TEMP_TOKENIZER))

print(
    f"Temporary tokenizer saved:\n"
    f"{TEMP_TOKENIZER}"
)


# =============================================================================
# 12. LOAD AND VALIDATE TEMPORARY TOKENIZER
# =============================================================================

print()
print("=" * 80)
print("TOKENIZER VALIDATION")
print("=" * 80)

validated_tokenizer = Tokenizer.from_file(
    str(TEMP_TOKENIZER)
)

actual_vocab_size = (
    validated_tokenizer.get_vocab_size()
)

print(
    f"Vocabulary size : "
    f"{actual_vocab_size:,}"
)

assert actual_vocab_size <= VOCAB_SIZE
assert actual_vocab_size > len(SPECIAL_TOKENS)


# -------------------------------------------------------------------------
# Test encoding
# -------------------------------------------------------------------------

test_text = (
    "The court considered the statutory interpretation "
    "of the relevant provision."
)

encoded = validated_tokenizer.encode(
    test_text
)

print()
print(f"Test characters : {len(test_text):,}")
print(f"Test tokens     : {len(encoded.ids):,}")
print(
    f"Token IDs       : "
    f"{encoded.ids[:20]}"
)

assert len(encoded.ids) > 0

print()
print("✓ Tokenizer loads successfully")
print("✓ Vocabulary is valid")
print("✓ Test text produces token IDs")


# =============================================================================
# 13. PROMOTE TEMPORARY → FINAL
# =============================================================================

print()
print("=" * 80)
print("PROMOTING TOKENIZER")
print("=" * 80)

if FINAL_TOKENIZER.exists():
    FINAL_TOKENIZER.unlink()

TEMP_TOKENIZER.replace(FINAL_TOKENIZER)

print(
    f"✓ Final tokenizer:\n"
    f"{FINAL_TOKENIZER}"
)


# =============================================================================
# 14. FINAL SUMMARY
# =============================================================================

total_elapsed = time.time() - training_start

print()
print("=" * 80)
print("PHASE 4 — BPE 32K TOKENIZER COMPLETE")
print("=" * 80)

print()
print(f"Documents         : {documents_seen:,}")
print(f"Characters        : {characters_seen:,}")
print(f"Vocabulary        : {actual_vocab_size:,}")
print(
    f"Total elapsed     : "
    f"{total_elapsed / 60:,.2f} minutes"
)

print()
print("✓ Manifest → grouped source → dedup → text")
print("✓ Full 3,020,655-document stream consumed")
print("✓ No tokenized corpus created")
print("✓ No corpus copy created")
print("✓ Canonical corpus untouched")
print("✓ Deduplicated corpus untouched")
print("✓ Split manifests untouched")
print("✓ Tokenizer validated before promotion")
print()
print("Tokenizer:")
print(FINAL_TOKENIZER)

print()
print("=" * 80)

In [ ]:
from pathlib import Path
import os
import json
import shutil

PHASE4 = Path("/kaggle/working/phase4_training_prep")
TOKENIZER_DIR = PHASE4 / "tokenizer"

print("=" * 80)
print("PHASE 4 — POST-TRAINING TOKENIZER CHECK")
print("=" * 80)

print(f"\nTokenizer directory : {TOKENIZER_DIR}")
print(f"Exists              : {TOKENIZER_DIR.exists()}")

if TOKENIZER_DIR.exists():
    files = sorted(TOKENIZER_DIR.rglob("*"))

    print("\nTOKENIZER ARTIFACTS")
    print("-" * 80)

    total_bytes = 0

    for p in files:
        if p.is_file():
            size = p.stat().st_size
            total_bytes += size
            print(f"{size / (1024**2):8.2f} MB | {p}")

    print("-" * 80)
    print(f"Tokenizer storage : {total_bytes / (1024**3):.3f} GB")

print("\nKAGGLE STORAGE")
print("-" * 80)

usage = shutil.disk_usage("/kaggle/working")

print(f"Total : {usage.total / (1024**3):.2f} GB")
print(f"Used  : {(usage.total - usage.free) / (1024**3):.2f} GB")
print(f"Free  : {usage.free / (1024**3):.2f} GB")

print("\n" + "=" * 80)

In [ ]:
from pathlib import Path
from tokenizers import Tokenizer

TOKENIZER_DIR = Path("/kaggle/working/phase4_training_prep/tokenizer")

# Find tokenizer.json
tokenizer_json = TOKENIZER_DIR / "tokenizer.json"

assert tokenizer_json.exists(), (
    f"Tokenizer artifact not found:\n{tokenizer_json}"
)

tokenizer = Tokenizer.from_file(str(tokenizer_json))

vocab_size = tokenizer.get_vocab_size()

print("=" * 80)
print("PHASE 4 — BPE TOKENIZER VALIDATION")
print("=" * 80)

print(f"\nTokenizer file : {tokenizer_json}")
print(f"Vocabulary     : {vocab_size:,}")

assert vocab_size == 32_000, (
    f"Expected vocabulary size 32,000, got {vocab_size}"
)

print("\n✓ tokenizer.json loads")
print("✓ vocabulary size = 32,000")

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/working/phase4_training_prep")

print("=" * 80)
print("PHASE 4 — LOCATING BPE TOKENIZER ARTIFACTS")
print("=" * 80)

print(f"\nPhase 4 root: {ROOT}")
print(f"Exists       : {ROOT.exists()}")

# Search specifically for tokenizer-related files/directories
matches = []

for p in ROOT.rglob("*"):
    if p.is_file():
        name = p.name.lower()
        if (
            "tokenizer" in name
            or name.endswith(".json")
            and "vocab" in name
            or name in {"merges.txt", "vocab.json"}
        ):
            matches.append(p)

print(f"\nTokenizer-related files found: {len(matches)}")

for p in sorted(matches):
    size_mb = p.stat().st_size / (1024 ** 2)
    print(f"{size_mb:8.2f} MB | {p}")

print("\n" + "=" * 80)

In [ ]:
from pathlib import Path
import json

PHASE4 = Path("/kaggle/working/phase4_training_prep")
TOKENIZER_DIR = PHASE4 / "tokenizer"

print("=" * 80)
print("PHASE 4 — TOKENIZER ARTIFACT DIAGNOSTIC")
print("=" * 80)

print(f"\nTokenizer directory : {TOKENIZER_DIR}")
print(f"Exists              : {TOKENIZER_DIR.exists()}")

if TOKENIZER_DIR.exists():
    print("\nContents:")
    for p in sorted(TOKENIZER_DIR.rglob("*")):
        if p.is_file():
            size = p.stat().st_size / (1024 ** 2)
            print(f"  FILE | {size:8.3f} MB | {p.name}")
        elif p.is_dir():
            print(f"  DIR  | {p.name}")

config = TOKENIZER_DIR / "tokenizer_config.json"

if config.exists():
    print("\n" + "=" * 80)
    print("TOKENIZER CONFIG")
    print("=" * 80)

    try:
        with open(config, "r", encoding="utf-8") as f:
            data = json.load(f)

        print(json.dumps(data, indent=2)[:10000])

    except Exception as e:
        print(f"Could not read config: {e}")

print("\n" + "=" * 80)
print("SEARCHING PHASE 4 FOR POSSIBLE MODEL ARTIFACTS")
print("=" * 80)

patterns = [
    "*.json",
    "*.model",
    "*.txt",
    "*.bin",
    "*.safetensors",
]

found = []

for pattern in patterns:
    found.extend(PHASE4.rglob(pattern))

for p in sorted(set(found)):
    size = p.stat().st_size / (1024 ** 2)

    # Skip tiny metadata/config files for the main artifact listing
    if size >= 0.01:
        print(f"{size:8.3f} MB | {p}")

print("\n" + "=" * 80)
print("DIAGNOSTIC COMPLETE")
print("=" * 80)

In [ ]:
from pathlib import Path
from tokenizers import Tokenizer

TOKENIZER_PATH = Path(
    "/kaggle/working/phase4_training_prep/tokenizer/legal_bpe_32k.json"
)

assert TOKENIZER_PATH.exists(), (
    f"Tokenizer artifact not found:\n{TOKENIZER_PATH}"
)

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

vocab_size = tokenizer.get_vocab_size()

print("=" * 80)
print("PHASE 4 — TRAINED BPE TOKENIZER VALIDATION")
print("=" * 80)

print(f"\nTokenizer file : {TOKENIZER_PATH}")
print(f"File size      : {TOKENIZER_PATH.stat().st_size / (1024**2):.3f} MB")
print(f"Vocabulary     : {vocab_size:,}")

assert vocab_size == 32_000, (
    f"Expected vocabulary size 32,000, got {vocab_size}"
)

print("\n✓ Trained tokenizer loads")
print("✓ Vocabulary size = 32,000")

In [ ]:
from pathlib import Path
from tokenizers import Tokenizer

TOKENIZER_PATH = Path(
    "/kaggle/working/phase4_training_prep/tokenizer/legal_bpe_32k.json"
)

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

print("=" * 80)
print("PHASE 4 — BPE 32K FUNCTIONAL AUDIT")
print("=" * 80)

print(f"\nVocabulary size : {tokenizer.get_vocab_size():,}")

# Real legal-text examples
examples = [
    (
        "Act",
        "An Act to provide for the acquisition and transfer of property "
        "and for matters connected therewith."
    ),
    (
        "Court",
        "The petitioner submitted that the impugned order was contrary "
        "to the provisions of the statute."
    ),
    (
        "Judgment",
        "Having considered the facts and circumstances of the case, "
        "the Court is of the opinion that the appeal deserves consideration."
    ),
    (
        "Legal",
        "Whereas it is expedient to amend the existing law and to provide "
        "for matters connected therewith or incidental thereto."
    ),
]

total_chars = 0
total_tokens = 0
total_unk = 0

for i, (label, text) in enumerate(examples, 1):

    encoding = tokenizer.encode(text)

    tokens = encoding.tokens
    ids = encoding.ids

    unk_id = tokenizer.token_to_id("[UNK]")
    unk_count = sum(1 for token_id in ids if token_id == unk_id)

    total_chars += len(text)
    total_tokens += len(ids)
    total_unk += unk_count

    print(f"\nExample {i} — {label}")
    print("-" * 80)
    print(f"Characters : {len(text):,}")
    print(f"Tokens     : {len(ids):,}")
    print(f"[UNK]      : {unk_count:,}")
    print(f"Tokens     : {tokens}")
    print(f"Token IDs  : {ids}")

    assert len(ids) > 0, "Tokenizer produced zero tokens."

print("\n" + "=" * 80)
print("AUDIT SUMMARY")
print("=" * 80)

print(f"\nCharacters tested : {total_chars:,}")
print(f"Tokens generated  : {total_tokens:,}")
print(f"[UNK] tokens      : {total_unk:,}")

unk_rate = total_unk / total_tokens if total_tokens else 0.0
token_per_char = total_tokens / total_chars if total_chars else 0.0

print(f"[UNK] rate         : {unk_rate:.4%}")
print(f"Tokens / character : {token_per_char:.4f}")

assert total_tokens > 0
assert total_unk == 0, (
    f"Unexpected [UNK] tokens found: {total_unk}"
)

print("\n✓ Tokenizer produces valid token IDs")
print("✓ Tokenizer produces non-empty BPE sequences")
print("✓ No [UNK] tokens in functional audit")
print("✓ 32K BPE tokenizer is functioning correctly")

In [ ]:
# =============================================================================
# PHASE 4 — REPRESENTATIVE BPE TOKENIZATION EFFICIENCY AUDIT
# =============================================================================

from pathlib import Path
import gzip
import json
import random
import statistics
import time

from tokenizers import Tokenizer

# ---------------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------------

PHASE4 = Path("/kaggle/working/phase4_training_prep")

TOKENIZER_PATH = (
    PHASE4 / "tokenizer" / "legal_bpe_32k.json"
)

TRAIN_MANIFEST = (
    PHASE4 / "manifests" / "train_manifest.jsonl"
)

DEDUP_ROOT = (
    PHASE4 / "dedup"
)

SAMPLE_SIZE = 10_000
SEED = 42

# ---------------------------------------------------------------------------
# VALIDATION
# ---------------------------------------------------------------------------

assert TOKENIZER_PATH.exists(), TOKENIZER_PATH
assert TRAIN_MANIFEST.exists(), TRAIN_MANIFEST
assert DEDUP_ROOT.exists(), DEDUP_ROOT

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

assert tokenizer.get_vocab_size() == 32_000

print("=" * 80)
print("PHASE 4 — REPRESENTATIVE BPE TOKENIZATION EFFICIENCY AUDIT")
print("=" * 80)

print(f"\nTokenizer       : {TOKENIZER_PATH}")
print(f"Vocabulary      : {tokenizer.get_vocab_size():,}")
print(f"Training split  : {TRAIN_MANIFEST}")
print(f"Sample size     : {SAMPLE_SIZE:,}")
print(f"Seed            : {SEED}")

# =============================================================================
# STEP 1 — RESOLVE MANIFEST RECORDS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 1 — SAMPLING TRAINING MANIFEST")
print("=" * 80)

rng = random.Random(SEED)

reservoir = []
manifest_count = 0

with open(TRAIN_MANIFEST, "r", encoding="utf-8") as f:

    for line in f:

        if not line.strip():
            continue

        manifest_count += 1

        record = json.loads(line)

        # Reservoir sampling gives a uniform sample without loading
        # the entire 3M-record manifest into RAM.
        if len(reservoir) < SAMPLE_SIZE:
            reservoir.append(record)

        else:
            j = rng.randint(0, manifest_count - 1)

            if j < SAMPLE_SIZE:
                reservoir[j] = record

print(f"Training manifest records : {manifest_count:,}")
print(f"Sampled records           : {len(reservoir):,}")

assert manifest_count == 3_020_655
assert len(reservoir) == SAMPLE_SIZE

print("\n✓ Training manifest count verified")
print("✓ Representative sample selected")

# =============================================================================
# STEP 2 — GROUP SAMPLE BY SOURCE FILE
# =============================================================================

print("\n" + "=" * 80)
print("STEP 2 — GROUPING SAMPLE BY SOURCE FILE")
print("=" * 80)

groups = {}

for record in reservoir:

    source_file = record["source_file"]

    # Manifest source_file is relative to the dedup root.
    path = DEDUP_ROOT / source_file

    if not path.exists():

        # Defensive fallback in case source_file already contains
        # the dedup root prefix.
        candidate = DEDUP_ROOT / Path(source_file).name

        if candidate.exists():
            path = candidate
        else:
            raise FileNotFoundError(
                f"Could not resolve source file:\n"
                f"  manifest : {source_file}\n"
                f"  attempted: {path}"
            )

    groups.setdefault(path, []).append(record["text_hash"])

print(f"Source files touched : {len(groups):,}")

# =============================================================================
# STEP 3 — BUILD HASH LOOKUP
# =============================================================================

sample_hashes = {
    text_hash
    for hashes in groups.values()
    for text_hash in hashes
}

assert len(sample_hashes) == SAMPLE_SIZE

print(f"Unique sample hashes  : {len(sample_hashes):,}")
print("✓ Sample contains unique training documents")

# =============================================================================
# STEP 4 — STREAM ACTUAL TEXT AND TOKENIZE
# =============================================================================

print("\n" + "=" * 80)
print("STEP 3 — TOKENIZING REPRESENTATIVE SAMPLE")
print("=" * 80)

start = time.time()

documents = 0
characters = 0
words = 0
tokens_total = 0
unk_total = 0

token_lengths = []

examples = []

unk_id = tokenizer.token_to_id("[UNK]")

for source_index, (path, wanted_hashes) in enumerate(
    groups.items(), 1
):

    wanted = set(wanted_hashes)

    print(
        f"\n[{source_index}/{len(groups)}] "
        f"{path.relative_to(DEDUP_ROOT)} | "
        f"sample={len(wanted):,}"
    )

    with gzip.open(path, "rt", encoding="utf-8") as f:

        for line in f:

            if not line.strip():
                continue

            row = json.loads(line)

            text_hash = row.get("text_hash")

            if text_hash not in wanted:
                continue

            text = row.get("text", "")

            if not text.strip():
                raise ValueError(
                    f"Empty text encountered for {text_hash}"
                )

            encoding = tokenizer.encode(text)

            ids = encoding.ids

            token_count = len(ids)

            unk_count = (
                sum(token_id == unk_id for token_id in ids)
                if unk_id is not None
                else 0
            )

            char_count = len(text)
            word_count = len(text.split())

            documents += 1
            characters += char_count
            words += word_count
            tokens_total += token_count
            unk_total += unk_count

            token_lengths.append(token_count)

            # Keep a few real examples for inspection.
            if len(examples) < 5:
                examples.append(
                    {
                        "hash": text_hash,
                        "characters": char_count,
                        "words": word_count,
                        "tokens": token_count,
                        "unk": unk_count,
                        "text_preview": text[:300],
                        "tokens_preview": encoding.tokens[:40],
                    }
                )

            if documents % 1_000 == 0:
                elapsed = time.time() - start
                rate = documents / elapsed if elapsed else 0

                print(
                    f"    sampled documents: {documents:>6,} / "
                    f"{SAMPLE_SIZE:,} | "
                    f"rate: {rate:,.1f} docs/sec",
                    end="\r",
                )

            if documents == SAMPLE_SIZE:
                break

    if documents == SAMPLE_SIZE:
        break

print("\n")

elapsed = time.time() - start

assert documents == SAMPLE_SIZE
assert characters > 0
assert words > 0
assert tokens_total > 0

# =============================================================================
# STEP 5 — STATISTICS
# =============================================================================

token_lengths_sorted = sorted(token_lengths)

def percentile(values, p):
    """
    Linear-interpolation percentile.
    """
    if not values:
        return 0

    if len(values) == 1:
        return values[0]

    k = (len(values) - 1) * p
    lower = int(k)
    upper = min(lower + 1, len(values) - 1)

    fraction = k - lower

    return (
        values[lower]
        + (values[upper] - values[lower]) * fraction
    )

unk_rate = (
    unk_total / tokens_total
    if tokens_total
    else 0.0
)

tokens_per_character = (
    tokens_total / characters
    if characters
    else 0.0
)

tokens_per_word = (
    tokens_total / words
    if words
    else 0.0
)

characters_per_token = (
    characters / tokens_total
    if tokens_total
    else 0.0
)

# =============================================================================
# FINAL REPORT
# =============================================================================

print("=" * 80)
print("REPRESENTATIVE BPE EFFICIENCY RESULTS")
print("=" * 80)

print(f"""
Documents sampled       : {documents:,}
Characters              : {characters:,}
Words                   : {words:,}
Tokens                  : {tokens_total:,}
[UNK] tokens            : {unk_total:,}

[UNK] rate              : {unk_rate:.6%}

Tokens / character      : {tokens_per_character:.4f}
Characters / token      : {characters_per_token:.4f}
Tokens / word           : {tokens_per_word:.4f}

Document token lengths
----------------------
Minimum                 : {min(token_lengths):,}
Median                  : {statistics.median(token_lengths):,.0f}
Mean                    : {statistics.mean(token_lengths):,.2f}
P95                     : {percentile(token_lengths_sorted, 0.95):,.0f}
P99                     : {percentile(token_lengths_sorted, 0.99):,.0f}
Maximum                 : {max(token_lengths):,}

Elapsed                 : {elapsed / 60:.2f} minutes
""")

# =============================================================================
# REAL TEXT EXAMPLES
# =============================================================================

print("=" * 80)
print("REAL TRAINING-DOCUMENT EXAMPLES")
print("=" * 80)

for i, example in enumerate(examples, 1):

    print(f"\nExample {i}")
    print("-" * 80)

    print(f"Characters : {example['characters']:,}")
    print(f"Words      : {example['words']:,}")
    print(f"Tokens     : {example['tokens']:,}")
    print(f"[UNK]      : {example['unk']:,}")

    print(f"\nText preview:")
    print(repr(example["text_preview"]))

    print(f"\nToken preview:")
    print(example["tokens_preview"])

# =============================================================================
# SAFETY ASSERTIONS
# =============================================================================

print("\n" + "=" * 80)
print("AUDIT CHECKPOINT")
print("=" * 80)

assert documents == SAMPLE_SIZE
assert unk_total == 0
assert tokenizer.get_vocab_size() == 32_000

print("\n✓ 10,000 real training documents tokenized")
print("✓ Sample came from the actual training manifest")
print("✓ Text resolved through the deduplicated corpus")
print("✓ 32K tokenizer used")
print("✓ Zero [UNK] tokens")
print("✓ Token-length statistics computed")
print("✓ No tokenized corpus was created")
print("✓ Canonical corpus was not modified")

print("\n" + "=" * 80)
print("REPRESENTATIVE EFFICIENCY AUDIT COMPLETE")
print("=" * 80)

In [ ]:
from pathlib import Path
import json

PHASE4 = Path("/kaggle/working/phase4_training_prep")

REQUIRED = {
    "dedup": PHASE4 / "dedup",
    "train_manifest": PHASE4 / "manifests/train_manifest.jsonl",
    "validation_manifest": PHASE4 / "manifests/validation_manifest.jsonl",
    "test_manifest": PHASE4 / "manifests/test_manifest.jsonl",
    "dedup_audit": PHASE4 / "phase4_dedup_audit.json",
    "split_policy": PHASE4 / "phase4_dedup_split_policy.json",
    "tokenizer": PHASE4 / "tokenizer/legal_bpe_32k.json",
    "tokenizer_config": PHASE4 / "tokenizer/tokenizer_config.json",
}

print("=" * 80)
print("PHASE 4 — FINAL CHECKPOINT")
print("=" * 80)

for name, path in REQUIRED.items():
    exists = path.exists()
    print(f"{name:20s}: {exists} | {path}")
    assert exists, f"Missing required artifact: {path}"

print("\n✓ Deduplicated corpus present")
print("✓ All split manifests present")
print("✓ Dedup audit present")
print("✓ Split policy present")
print("✓ 32K BPE tokenizer present")
print("✓ Tokenizer configuration present")

print("\nPHASE 4 CHECKPOINT VALID")

In [ ]:
import json
from pathlib import Path

CONFIG = Path(
    "/kaggle/working/phase4_training_prep/tokenizer/tokenizer_config.json"
)

with open(CONFIG, "r", encoding="utf-8") as f:
    cfg = json.load(f)

print("=" * 80)
print("PHASE 4 — TOKENIZER CONFIGURATION CHECK")
print("=" * 80)

print(json.dumps(cfg, indent=2))

assert cfg["tokenizer"] == "BPE"
assert cfg["vocab_size"] == 32000
assert cfg["seed"] == 42
assert cfg["canonical_corpus_modified"] is False
assert cfg["tokenized_corpus_materialized"] is False

print("\n✓ BPE tokenizer")
print("✓ Vocabulary = 32,000")
print("✓ Seed = 42")
print("✓ Canonical corpus untouched")
print("✓ No tokenized corpus materialized")

In [ ]:
import os
import shutil
from pathlib import Path

ROOT = Path("/kaggle/working")

def size_gb(path):
    total = 0
    if not path.exists():
        return 0.0
    for p in path.rglob("*"):
        if p.is_file():
            try:
                total += p.stat().st_size
            except OSError:
                pass
    return total / (1024**3)

total, used, free = shutil.disk_usage(ROOT)

print("=" * 80)
print("PHASE 4 — PRE-NEXT-STAGE STORAGE CHECK")
print("=" * 80)

print(f"\nTotal : {total / 1024**3:.2f} GB")
print(f"Used  : {used / 1024**3:.2f} GB")
print(f"Free  : {free / 1024**3:.2f} GB")

print("\nIMPORTANT DIRECTORIES")
print("-" * 80)

for p in [
    ROOT / "phase2_canonical",
    ROOT / "phase4_training_prep",
    ROOT / "phase4_cleanup_staging",
]:
    print(f"{p.name:28s}: {size_gb(p):6.2f} GB | exists={p.exists()}")

print("\n✓ Storage inspected")
print("✓ No deletion performed")

In [ ]:
from pathlib import Path
from tokenizers import Tokenizer

TOKENIZER_PATH = Path(
    "/kaggle/working/phase4_training_prep/tokenizer/legal_bpe_32k.json"
)

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

print("=" * 80)
print("PHASE 4 — TOKENIZER / MODEL INPUT CONTRACT")
print("=" * 80)

print(f"\nTokenizer : {TOKENIZER_PATH}")
print(f"Vocab    : {tokenizer.get_vocab_size():,}")

assert tokenizer.get_vocab_size() == 32_000

vocab = tokenizer.get_vocab()

for token in ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]:
    print(f"{token:8s} -> {vocab.get(token)}")
    assert token in vocab, f"Missing special token: {token}"

print("\n✓ Vocabulary = 32,000")
print("✓ Required special tokens present")

In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone

PHASE4 = Path("/kaggle/working/phase4_training_prep")
OUT = PHASE4 / "streaming_config.json"

streaming_config = {
    "phase": "phase4",
    "dataset_mode": "streaming",
    "materialize_tokenized_corpus": False,

    "train_manifest": str(
        PHASE4 / "manifests/train_manifest.jsonl"
    ),
    "validation_manifest": str(
        PHASE4 / "manifests/validation_manifest.jsonl"
    ),
    "test_manifest": str(
        PHASE4 / "manifests/test_manifest.jsonl"
    ),

    "tokenizer": str(
        PHASE4 / "tokenizer/legal_bpe_32k.json"
    ),

    "vocab_size": 32000,
    "seed": 42,

    "train_records": 3_020_655,
    "validation_records": 168_665,
    "test_records": 167_274,

    "created_at": datetime.now(timezone.utc).isoformat(),
}

with open(OUT, "w", encoding="utf-8") as f:
    json.dump(streaming_config, f, indent=2)

print("=" * 80)
print("PHASE 4 — STREAMING TRAINING CONTRACT")
print("=" * 80)

print(json.dumps(streaming_config, indent=2))
print(f"\nSaved: {OUT}")

print("\n✓ Streaming mode selected")
print("✓ No tokenized corpus will be materialized")

In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone

PHASE4 = Path("/kaggle/working/phase4_training_prep")
OUT = PHASE4 / "streaming_config.json"

streaming_config = {
    "phase": "phase4",
    "dataset_mode": "streaming",
    "materialize_tokenized_corpus": False,

    "train_manifest": str(
        PHASE4 / "manifests/train_manifest.jsonl"
    ),
    "validation_manifest": str(
        PHASE4 / "manifests/validation_manifest.jsonl"
    ),
    "test_manifest": str(
        PHASE4 / "manifests/test_manifest.jsonl"
    ),

    "tokenizer": str(
        PHASE4 / "tokenizer/legal_bpe_32k.json"
    ),

    "vocab_size": 32000,
    "seed": 42,

    "train_records": 3_020_655,
    "validation_records": 168_665,
    "test_records": 167_274,

    "created_at": datetime.now(timezone.utc).isoformat(),
}

with open(OUT, "w", encoding="utf-8") as f:
    json.dump(streaming_config, f, indent=2)

print("=" * 80)
print("PHASE 4 — STREAMING TRAINING CONTRACT")
print("=" * 80)

print(json.dumps(streaming_config, indent=2))
print(f"\nSaved: {OUT}")

print("\n✓ Streaming mode selected")
print("✓ No tokenized corpus will be materialized")

In [ ]:
import gzip
import json
import time
from collections import defaultdict
from pathlib import Path

PHASE4 = Path("/kaggle/working/phase4_training_prep")

MANIFEST = PHASE4 / "manifests/train_manifest.jsonl"
DEDUP = PHASE4 / "dedup"
TOKENIZER_PATH = PHASE4 / "tokenizer/legal_bpe_32k.json"

from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

# ------------------------------------------------------------------
# Build source_file -> required hashes
# ------------------------------------------------------------------

groups = defaultdict(set)

with open(MANIFEST, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        groups[row["source_file"]].add(row["text_hash"])

expected_total = sum(len(v) for v in groups.values())

print("=" * 80)
print("PHASE 4 — GROUPED STREAMING TOKENIZATION TEST")
print("=" * 80)

print(f"\nSource files : {len(groups)}")
print(f"Train docs   : {expected_total:,}")

assert expected_total == 3_020_655

# ------------------------------------------------------------------
# Test only 1,000 documents
# ------------------------------------------------------------------

TARGET = 1_000

seen = 0
tokens = 0
characters = 0

start = time.time()

for source_file, hashes in groups.items():

    if seen >= TARGET:
        break

    path = DEDUP / source_file

    if not path.exists():
        raise FileNotFoundError(path)

    with gzip.open(path, "rt", encoding="utf-8") as f:

        for line in f:
            row = json.loads(line)

            if row["text_hash"] not in hashes:
                continue

            text = row.get("text", "")

            if not text:
                continue

            encoded = tokenizer.encode(text)

            seen += 1
            characters += len(text)
            tokens += len(encoded.ids)

            if seen >= TARGET:
                break

elapsed = time.time() - start

print("\n" + "=" * 80)
print("STREAMING TEST RESULT")
print("=" * 80)

print(f"Documents  : {seen:,}")
print(f"Characters : {characters:,}")
print(f"Tokens     : {tokens:,}")
print(f"Elapsed    : {elapsed:.2f} sec")

assert seen == TARGET
assert tokens > 0
assert characters > 0

print("\n✓ 1,000 real training documents tokenized")
print("✓ Grouped source streaming works")
print("✓ No SQLite dependency")
print("✓ No tokenized corpus created")
print("✓ Canonical corpus untouched")

In [ ]:
from pathlib import Path
import json
from tokenizers import Tokenizer

PHASE4_ROOT = Path("/kaggle/working/phase4_training_prep")
TOKENIZER_DIR = PHASE4_ROOT / "tokenizer"
TOKENIZER_PATH = TOKENIZER_DIR / "legal_bpe_32k.json"
CONFIG_PATH = TOKENIZER_DIR / "tokenizer_config.json"

print("=" * 80)
print("PHASE 4 — POST-TRAINING TOKENIZER INTEGRITY AUDIT")
print("=" * 80)

# --------------------------------------------------------------------------
# 1. Artifact existence
# --------------------------------------------------------------------------

assert TOKENIZER_PATH.exists(), f"Missing tokenizer: {TOKENIZER_PATH}"
assert CONFIG_PATH.exists(), f"Missing tokenizer config: {CONFIG_PATH}"

print(f"\nTokenizer : {TOKENIZER_PATH}")
print(f"Size      : {TOKENIZER_PATH.stat().st_size / (1024**2):.3f} MB")

# --------------------------------------------------------------------------
# 2. Load tokenizer
# --------------------------------------------------------------------------

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

vocab_size = tokenizer.get_vocab_size()

print(f"\nVocabulary size : {vocab_size:,}")

assert vocab_size == 32_000, (
    f"Expected vocabulary size 32,000, got {vocab_size:,}"
)

# --------------------------------------------------------------------------
# 3. Load and validate configuration
# --------------------------------------------------------------------------

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

print("\nCONFIGURATION")
print("-" * 80)

for key, value in config.items():
    print(f"{key:28s}: {value}")

assert config["tokenizer"] == "BPE"
assert config["vocab_size"] == 32_000
assert config["seed"] == 42
assert config["canonical_corpus_modified"] is False
assert config["tokenized_corpus_materialized"] is False

expected_special_tokens = ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]

assert config["special_tokens"] == expected_special_tokens, (
    f"Unexpected special tokens: {config['special_tokens']}"
)

# --------------------------------------------------------------------------
# 4. Special-token IDs
# --------------------------------------------------------------------------

print("\nSPECIAL TOKEN IDS")
print("-" * 80)

special_ids = {}

for token in expected_special_tokens:
    token_id = tokenizer.token_to_id(token)
    special_ids[token] = token_id

    print(f"{token:8s} : {token_id}")

    assert token_id is not None, (
        f"Special token missing from tokenizer: {token}"
    )

# IDs must be unique
assert len(set(special_ids.values())) == len(special_ids)

# --------------------------------------------------------------------------
# 5. Basic encode/decode integrity
# --------------------------------------------------------------------------

test_text = (
    "The petitioner submitted that the impugned order was "
    "contrary to the provisions of the statute."
)

encoding = tokenizer.encode(test_text)

print("\nENCODING TEST")
print("-" * 80)

print(f"Input characters : {len(test_text):,}")
print(f"Token count      : {len(encoding.ids):,}")
print(f"First token IDs  : {encoding.ids[:20]}")

assert len(encoding.ids) > 0
assert all(isinstance(x, int) for x in encoding.ids)

# --------------------------------------------------------------------------
# 6. Decode integrity
# --------------------------------------------------------------------------

decoded = tokenizer.decode(encoding.ids)

print(f"\nDecoded preview  : {decoded[:200]!r}")

assert decoded.strip(), "Decoded text is empty"

# --------------------------------------------------------------------------
# 7. UNK check
# --------------------------------------------------------------------------

unk_id = special_ids["[UNK]"]
unk_count = encoding.ids.count(unk_id)

print(f"\n[UNK] tokens     : {unk_count}")

assert unk_count == 0, (
    f"Unexpected [UNK] tokens in test text: {unk_count}"
)

# --------------------------------------------------------------------------
# 8. Artifact metadata consistency
# --------------------------------------------------------------------------

print("\nMETADATA CHECK")
print("-" * 80)

print(f"Training source          : {config['training_source']}")
print(f"Training manifest        : {config['train_manifest']}")
print(f"Dedup root               : {config['dedup_root']}")
print(f"Seed                     : {config['seed']}")
print(f"Canonical modified       : {config['canonical_corpus_modified']}")
print(f"Tokenized corpus exists  : {config['tokenized_corpus_materialized']}")

assert Path(config["train_manifest"]).exists()
assert Path(config["dedup_root"]).exists()

# --------------------------------------------------------------------------
# FINAL
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("POST-TRAINING TOKENIZER AUDIT SUMMARY")
print("=" * 80)

print(f"\nVocabulary          : {vocab_size:,}")
print(f"Tokenizer type      : {config['tokenizer']}")
print(f"Special tokens      : {len(expected_special_tokens)}")
print(f"[UNK] in test       : {unk_count}")
print(f"Test tokens         : {len(encoding.ids):,}")
print(f"Seed                : {config['seed']}")

print("\n✓ tokenizer artifact exists")
print("✓ tokenizer loads successfully")
print("✓ vocabulary size = 32,000")
print("✓ special tokens are present")
print("✓ special-token IDs are unique")
print("✓ encoding produces valid integer IDs")
print("✓ decoding produces non-empty text")
print("✓ no [UNK] tokens in integrity test")
print("✓ training manifest exists")
print("✓ deduplicated corpus exists")
print("✓ canonical corpus was not modified")
print("✓ tokenized corpus has NOT been materialized")
print("\n✓ POST-TRAINING TOKENIZER INTEGRITY CHECK PASSED")
print("=" * 80)

In [ ]:
# =============================================================================
# PHASE 4 — SAMPLE-BASED TRAINING TOKEN-LENGTH ANALYSIS
# 200K DETERMINISTIC SAMPLE
# =============================================================================

from pathlib import Path
from collections import defaultdict
import gzip
import json
import time
import math
import gc

from tokenizers import Tokenizer


# =============================================================================
# CONFIG
# =============================================================================

PHASE4_ROOT = Path("/kaggle/working/phase4_training_prep")

MANIFEST_PATH = (
    PHASE4_ROOT /
    "manifests" /
    "train_manifest.jsonl"
)

DEDUP_ROOT = (
    PHASE4_ROOT /
    "dedup"
)

TOKENIZER_PATH = (
    PHASE4_ROOT /
    "tokenizer" /
    "legal_bpe_32k.json"
)

# -------------------------------------------------------------------------
# Sampling configuration
# -------------------------------------------------------------------------

SAMPLE_SIZE = 200_000

# Deterministic selection from the training manifest.
# Same seed + same manifest = same sample.
SEED = 42

BATCH_SIZE = 512
PROGRESS_INTERVAL = 10_000

EXPECTED_TRAINING_DOCUMENTS = 3_020_655


# =============================================================================
# HEADER
# =============================================================================

print("=" * 80)
print("PHASE 4 — SAMPLE-BASED TOKEN-LENGTH ANALYSIS")
print("DETERMINISTIC 200K TRAINING-DOCUMENT SAMPLE")
print("=" * 80)

print(f"\nTraining manifest : {MANIFEST_PATH}")
print(f"Dedup root        : {DEDUP_ROOT}")
print(f"Tokenizer         : {TOKENIZER_PATH}")
print(f"Sample size       : {SAMPLE_SIZE:,}")
print(f"Seed              : {SEED}")
print(f"Batch size        : {BATCH_SIZE:,}")
print(f"Progress interval : {PROGRESS_INTERVAL:,} documents")


# =============================================================================
# VALIDATION
# =============================================================================

assert MANIFEST_PATH.exists(), MANIFEST_PATH
assert DEDUP_ROOT.exists(), DEDUP_ROOT
assert TOKENIZER_PATH.exists(), TOKENIZER_PATH


tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

VOCAB_SIZE = tokenizer.get_vocab_size()
UNK_ID = tokenizer.token_to_id("[UNK]")

assert VOCAB_SIZE == 32_000
assert UNK_ID is not None

print(f"\nVocabulary        : {VOCAB_SIZE:,}")
print(f"[UNK] ID          : {UNK_ID}")


# =============================================================================
# LOAD MANIFEST
# =============================================================================

print("\n" + "=" * 80)
print("LOADING TRAINING MANIFEST")
print("=" * 80)

manifest_records = []

with MANIFEST_PATH.open("r", encoding="utf-8") as f:

    for line in f:

        line = line.strip()

        if not line:
            continue

        row = json.loads(line)

        manifest_records.append(
            (
                row["source_file"],
                row["text_hash"],
            )
        )


print(f"Training records : {len(manifest_records):,}")

assert len(manifest_records) == EXPECTED_TRAINING_DOCUMENTS

print("✓ Training manifest verified")


# =============================================================================
# DETERMINISTIC SAMPLE
# =============================================================================

print("\n" + "=" * 80)
print("BUILDING DETERMINISTIC SAMPLE")
print("=" * 80)

import random

rng = random.Random(SEED)

# Sample manifest positions rather than loading document text.
sample_indices = sorted(
    rng.sample(
        range(len(manifest_records)),
        SAMPLE_SIZE
    )
)

sample_records = [
    manifest_records[i]
    for i in sample_indices
]

assert len(sample_records) == SAMPLE_SIZE

# Group sampled hashes by source file.
sample_hashes = defaultdict(set)

for source_file, text_hash in sample_records:
    sample_hashes[source_file].add(text_hash)

source_files = sorted(sample_hashes.keys())

print(f"\nSample documents : {len(sample_records):,}")
print(f"Source files     : {len(source_files)}")

print("\n✓ Deterministic sample created")
print("✓ Seed = 42")


# =============================================================================
# SAMPLE DISTRIBUTION
# =============================================================================

print("\n" + "=" * 80)
print("SAMPLE SOURCE DISTRIBUTION")
print("=" * 80)

for source_file in source_files:

    print(
        f"{source_file:<65} "
        f"| {len(sample_hashes[source_file]):>7,}"
    )


# =============================================================================
# STATISTICS
# =============================================================================

token_lengths = []

total_documents = 0
total_characters = 0
total_tokens = 0
total_unk = 0

global_start = time.time()
last_progress_time = global_start
last_progress_docs = 0


# =============================================================================
# STREAM SAMPLE
# =============================================================================

print("\n" + "=" * 80)
print("STARTING SAMPLE TOKEN-LENGTH ANALYSIS")
print("=" * 80)

print(f"\nTarget documents : {SAMPLE_SIZE:,}")
print(f"Batch size       : {BATCH_SIZE:,}")
print(f"Progress         : every {PROGRESS_INTERVAL:,} documents")
print()


for source_index, source_file in enumerate(
    source_files,
    start=1
):

    path = DEDUP_ROOT / source_file

    assert path.exists(), (
        f"Missing source file:\n{path}"
    )

    wanted_hashes = sample_hashes[source_file]

    source_docs = 0
    source_start = time.time()

    print("=" * 80)
    print(
        f"[SOURCE {source_index}/{len(source_files)}] "
        f"{source_file}"
    )
    print(
        f"Sample hashes : {len(wanted_hashes):,}"
    )
    print("=" * 80)

    batch_texts = []

    with gzip.open(
        path,
        "rt",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            row = json.loads(line)

            text_hash = row.get("text_hash")

            if text_hash not in wanted_hashes:
                continue

            text = row.get("text", "")

            if not text:
                continue

            batch_texts.append(text)

            # -------------------------------------------------------------
            # BATCH TOKENIZATION
            # -------------------------------------------------------------

            if len(batch_texts) >= BATCH_SIZE:

                encodings = tokenizer.encode_batch(
                    batch_texts
                )

                for text, encoding in zip(
                    batch_texts,
                    encodings
                ):

                    length = len(encoding.ids)

                    token_lengths.append(length)

                    total_documents += 1
                    source_docs += 1

                    total_characters += len(text)
                    total_tokens += length

                    if UNK_ID is not None:

                        total_unk += sum(
                            token_id == UNK_ID
                            for token_id in encoding.ids
                        )

                batch_texts.clear()

                # ---------------------------------------------------------
                # PROGRESS
                # ---------------------------------------------------------

                if (
                    total_documents - last_progress_docs
                    >= PROGRESS_INTERVAL
                ):

                    now = time.time()

                    elapsed = (
                        now - global_start
                    )

                    interval_elapsed = (
                        now - last_progress_time
                    )

                    docs_since_last = (
                        total_documents -
                        last_progress_docs
                    )

                    global_rate = (
                        total_documents / elapsed
                        if elapsed > 0
                        else 0
                    )

                    interval_rate = (
                        docs_since_last /
                        interval_elapsed
                        if interval_elapsed > 0
                        else 0
                    )

                    remaining = (
                        SAMPLE_SIZE -
                        total_documents
                    )

                    eta_seconds = (
                        remaining / global_rate
                        if global_rate > 0
                        else float("inf")
                    )

                    progress_pct = (
                        total_documents /
                        SAMPLE_SIZE *
                        100
                    )

                    print(
                        "\n"
                        f"[PROGRESS] "
                        f"{total_documents:,} / "
                        f"{SAMPLE_SIZE:,} "
                        f"({progress_pct:6.2f}%)"
                    )

                    print(
                        f"Rate        : "
                        f"{global_rate:,.1f} docs/s "
                        f"(interval "
                        f"{interval_rate:,.1f} docs/s)"
                    )

                    print(
                        f"Characters  : "
                        f"{total_characters:,}"
                    )

                    print(
                        f"Tokens      : "
                        f"{total_tokens:,}"
                    )

                    if math.isfinite(
                        eta_seconds
                    ):

                        print(
                            f"ETA         : "
                            f"{eta_seconds / 60:,.1f} min"
                        )

                    print(
                        f"Elapsed     : "
                        f"{elapsed / 60:,.2f} min"
                    )

                    last_progress_docs = (
                        total_documents
                    )

                    last_progress_time = now

    # =========================================================================
    # FINAL PARTIAL BATCH
    # =========================================================================

    if batch_texts:

        encodings = tokenizer.encode_batch(
            batch_texts
        )

        for text, encoding in zip(
            batch_texts,
            encodings
        ):

            length = len(encoding.ids)

            token_lengths.append(length)

            total_documents += 1
            source_docs += 1

            total_characters += len(text)
            total_tokens += length

            if UNK_ID is not None:

                total_unk += sum(
                    token_id == UNK_ID
                    for token_id in encoding.ids
                )

        batch_texts.clear()

    source_elapsed = (
        time.time() - source_start
    )

    print(
        f"\n✓ SOURCE COMPLETE"
        f" | documents={source_docs:,}"
        f" | elapsed={source_elapsed / 60:.2f} min"
    )

    gc.collect()


# =============================================================================
# FINAL VERIFICATION
# =============================================================================

print("\n" + "=" * 80)
print("SAMPLE ANALYSIS COMPLETE")
print("=" * 80)

print(f"\nDocuments : {total_documents:,}")
print(f"Characters: {total_characters:,}")
print(f"Tokens    : {total_tokens:,}")

assert total_documents == SAMPLE_SIZE
assert len(token_lengths) == SAMPLE_SIZE

print("\n✓ EXACT SAMPLE COUNT VERIFIED")


# =============================================================================
# PERCENTILES
# =============================================================================

sorted_lengths = sorted(token_lengths)


def percentile(values, p):

    if not values:
        return 0

    position = (
        (len(values) - 1) *
        p /
        100
    )

    lower = math.floor(position)
    upper = math.ceil(position)

    if lower == upper:
        return values[lower]

    return (
        values[lower] +
        (
            values[upper] -
            values[lower]
        ) *
        (position - lower)
    )


mean_tokens = (
    total_tokens /
    total_documents
)

median_tokens = percentile(
    sorted_lengths,
    50
)

p90 = percentile(
    sorted_lengths,
    90
)

p95 = percentile(
    sorted_lengths,
    95
)

p99 = percentile(
    sorted_lengths,
    99
)

p995 = percentile(
    sorted_lengths,
    99.5
)

p999 = percentile(
    sorted_lengths,
    99.9
)

minimum = sorted_lengths[0]
maximum = sorted_lengths[-1]

unk_rate = (
    total_unk /
    total_tokens *
    100
    if total_tokens
    else 0
)


# =============================================================================
# STATISTICS REPORT
# =============================================================================

print("\n" + "=" * 80)
print("TOKEN-LENGTH STATISTICS — 200K SAMPLE")
print("=" * 80)

print(
    f"\nMinimum tokens : {minimum:,.0f}"
)

print(
    f"Mean tokens    : {mean_tokens:,.2f}"
)

print(
    f"Median tokens  : {median_tokens:,.2f}"
)

print(
    f"P90            : {p90:,.2f}"
)

print(
    f"P95            : {p95:,.2f}"
)

print(
    f"P99            : {p99:,.2f}"
)

print(
    f"P99.5          : {p995:,.2f}"
)

print(
    f"P99.9          : {p999:,.2f}"
)

print(
    f"Maximum        : {maximum:,.0f}"
)

print(
    f"[UNK] tokens   : {total_unk:,}"
)

print(
    f"[UNK] rate     : {unk_rate:.6f}%"
)


# =============================================================================
# CONTEXT LENGTH ANALYSIS
# =============================================================================

print("\n" + "=" * 80)
print("CONTEXT-LENGTH TRUNCATION ANALYSIS")
print("=" * 80)

candidate_lengths = [
    512,
    768,
    1024,
    1536,
    2048,
    3072,
    4096,
    8192,
]


for context_length in candidate_lengths:

    truncated = sum(
        length > context_length
        for length in token_lengths
    )

    truncated_pct = (
        truncated /
        total_documents *
        100
    )

    retained_tokens = sum(
        min(length, context_length)
        for length in token_lengths
    )

    retention_pct = (
        retained_tokens /
        total_tokens *
        100
    )

    print(
        f"{context_length:>5} tokens"
        f" | truncated docs = "
        f"{truncated:>8,}"
        f" ({truncated_pct:>7.3f}%)"
        f" | token retention = "
        f"{retention_pct:>7.3f}%"
    )


# =============================================================================
# FINAL
# =============================================================================

elapsed_total = (
    time.time() -
    global_start
)

print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)

print(
    f"\nSample documents : "
    f"{total_documents:,}"
)

print(
    f"Sample fraction  : "
    f"{total_documents / EXPECTED_TRAINING_DOCUMENTS * 100:.2f}%"
)

print(
    f"Characters       : "
    f"{total_characters:,}"
)

print(
    f"Tokens           : "
    f"{total_tokens:,}"
)

print(
    f"Mean tokens/doc  : "
    f"{mean_tokens:,.2f}"
)

print(
    f"Elapsed          : "
    f"{elapsed_total / 60:,.2f} min"
)

print("\n✓ 32K tokenizer unchanged")
print("✓ Canonical corpus unchanged")
print("✓ Deduplicated corpus unchanged")
print("✓ Training manifest unchanged")
print("✓ No tokenized corpus created")
print("✓ Deterministic sample (seed=42)")
print("✓ Token-length statistics computed")

print("\n" + "=" * 80)
print("PHASE 4 SAMPLE TOKEN-LENGTH CHECKPOINT VALID")
print("=" * 80)

In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone

PHASE4_ROOT = Path("/kaggle/working/phase4_training_prep")
DEDUP_ROOT = PHASE4_ROOT / "dedup"
MANIFEST_DIR = PHASE4_ROOT / "manifests"
TOKENIZER_DIR = PHASE4_ROOT / "tokenizer"

TRAIN_MANIFEST = MANIFEST_DIR / "train_manifest.jsonl"
VAL_MANIFEST = MANIFEST_DIR / "validation_manifest.jsonl"
TEST_MANIFEST = MANIFEST_DIR / "test_manifest.jsonl"

TOKENIZER_PATH = TOKENIZER_DIR / "legal_bpe_32k.json"

# Chosen from the completed 200K token-length analysis.
CONTEXT_LENGTH = 4096

# Special tokens used by the trained tokenizer.
PAD_TOKEN = "[PAD]"
UNK_TOKEN = "[UNK]"
BOS_TOKEN = "[BOS]"
EOS_TOKEN = "[EOS]"

CONFIG_PATH = PHASE4_ROOT / "training_example_config.json"

config = {
    "phase": "phase4",
    "tokenizer": str(TOKENIZER_PATH),
    "context_length": CONTEXT_LENGTH,
    "packing": "document_aware",
    "document_boundary": True,
    "special_tokens": [
        PAD_TOKEN,
        UNK_TOKEN,
        BOS_TOKEN,
        EOS_TOKEN,
    ],
    "training_manifest": str(TRAIN_MANIFEST),
    "validation_manifest": str(VAL_MANIFEST),
    "test_manifest": str(TEST_MANIFEST),
    "created_at": datetime.now(timezone.utc).isoformat(),
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("=" * 80)
print("PHASE 4 — TRAINING EXAMPLE CONFIGURATION")
print("=" * 80)

print(f"\nTokenizer       : {TOKENIZER_PATH}")
print(f"Context length  : {CONTEXT_LENGTH:,}")
print("Packing         : document-aware")
print("Document bounds : preserved")
print(f"\nConfig saved    : {CONFIG_PATH}")

assert TOKENIZER_PATH.exists()
assert TRAIN_MANIFEST.exists()
assert VAL_MANIFEST.exists()
assert TEST_MANIFEST.exists()

print("\n✓ Tokenizer exists")
print("✓ Train manifest exists")
print("✓ Validation manifest exists")
print("✓ Test manifest exists")
print("✓ Configuration frozen")

In [ ]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

print("=" * 80)
print("PHASE 4 — TOKENIZER SPECIAL-TOKEN CHECK")
print("=" * 80)

print(f"\nVocabulary size : {tokenizer.get_vocab_size():,}")

assert tokenizer.get_vocab_size() == 32_000

vocab = tokenizer.get_vocab()

for token in [PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN]:
    token_id = vocab.get(token)
    print(f"{token:>8} -> {token_id}")
    assert token_id is not None, f"Missing special token: {token}"

UNK_ID = vocab[UNK_TOKEN]

print(f"\n[UNK] ID : {UNK_ID}")

print("\n✓ Vocabulary = 32,000")
print("✓ Required special tokens present")

In [ ]:
import gzip
import json

def read_jsonl_gz(path):
    """
    Stream documents from a gzipped JSONL source.
    Nothing is loaded into memory in bulk.
    """
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            yield json.loads(line)


def extract_text(record):
    """
    Extract the canonical text field.

    Adjust only if your actual dedup JSONL schema differs.
    """
    text = record.get("text")

    if text is None:
        return ""

    return str(text)

In [ ]:
print("=" * 80)
print("PHASE 4 — DEDUP STREAM CHECK")
print("=" * 80)

checked = 0
characters = 0

for source_path in sorted(DEDUP_ROOT.rglob("*.jsonl.gz")):

    for record in read_jsonl_gz(source_path):

        text = extract_text(record)

        if not text.strip():
            continue

        checked += 1
        characters += len(text)

        if checked <= 3:
            print("\nExample", checked)
            print("-" * 80)
            print("Source :", source_path.relative_to(DEDUP_ROOT))
            print("Chars  :", len(text))
            print("Preview:", repr(text[:300]))

        if checked >= 100:
            break

    if checked >= 100:
        break

print("\n" + "=" * 80)
print("STREAM CHECK SUMMARY")
print("=" * 80)

print(f"\nDocuments checked : {checked:,}")
print(f"Characters        : {characters:,}")

assert checked == 100
assert characters > 0

print("\n✓ Dedup stream readable")
print("✓ Text is non-empty")
print("✓ No corpus materialized")

In [ ]:
def chunk_tokens(tokens, max_length):
    """
    Split one document into <= max_length token chunks.
    Document boundaries are preserved.
    """
    for start in range(0, len(tokens), max_length):
        yield tokens[start:start + max_length]


print("=" * 80)
print("PHASE 4 — DOCUMENT-AWARE CHUNKING TEST")
print("=" * 80)

test_text = """
This is a synthetic test document for the Phase 4 training-example
pipeline. The purpose is to verify that tokenization and chunking work
without creating a materialized tokenized corpus.
""" * 100

encoding = tokenizer.encode(test_text)

tokens = encoding.ids

chunks = list(chunk_tokens(tokens, CONTEXT_LENGTH))

print(f"\nInput tokens  : {len(tokens):,}")
print(f"Chunks        : {len(chunks):,}")
print(f"Context       : {CONTEXT_LENGTH:,}")

for i, chunk in enumerate(chunks[:3], 1):
    print(
        f"Chunk {i}: "
        f"{len(chunk):,} tokens | "
        f"min={min(chunk)} | max={max(chunk)}"
    )

assert len(chunks) > 0
assert all(len(chunk) <= CONTEXT_LENGTH for chunk in chunks)

print("\n✓ Chunking works")
print("✓ No chunk exceeds context length")
print("✓ Document boundary preserved")

In [ ]:
from pathlib import Path
import gzip
import json
import time

from tokenizers import Tokenizer

# ============================================================
# PHASE 4 — 10K TRAINING PIPELINE BENCHMARK
# PROGRESS-VISIBLE / SAFE / NO OUTPUT CORPUS
# ============================================================

PHASE4_ROOT = Path("/kaggle/working/phase4_training_prep")
DEDUP_ROOT = PHASE4_ROOT / "dedup"
TOKENIZER_PATH = (
    PHASE4_ROOT / "tokenizer" / "legal_bpe_32k.json"
)

BENCHMARK_DOCUMENTS = 10_000
BATCH_SIZE = 512
PROGRESS_INTERVAL = 1_000
CONTEXT_LENGTH = 4096

print("=" * 80, flush=True)
print("PHASE 4 — 10K TRAINING PIPELINE BENCHMARK", flush=True)
print("PROGRESS-VISIBLE BATCHED TOKENIZATION", flush=True)
print("=" * 80, flush=True)

print(f"\nTokenizer : {TOKENIZER_PATH}", flush=True)
print(f"Dedup root: {DEDUP_ROOT}", flush=True)
print(f"Documents : {BENCHMARK_DOCUMENTS:,}", flush=True)
print(f"Batch size: {BATCH_SIZE}", flush=True)
print(f"Context   : {CONTEXT_LENGTH:,}", flush=True)

assert TOKENIZER_PATH.exists(), TOKENIZER_PATH
assert DEDUP_ROOT.exists(), DEDUP_ROOT

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

VOCAB_SIZE = tokenizer.get_vocab_size()
UNK_ID = tokenizer.token_to_id("[UNK]")

print(f"Vocabulary: {VOCAB_SIZE:,}", flush=True)
print(f"[UNK] ID  : {UNK_ID}", flush=True)

assert VOCAB_SIZE == 32_000
assert UNK_ID is not None

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def read_jsonl_gz(path):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            yield json.loads(line)


def extract_text(record):
    text = record.get("text")

    if text is None:
        return ""

    return str(text)


def count_chunks(token_count, context_length):
    if token_count == 0:
        return 0

    return (token_count + context_length - 1) // context_length


# ------------------------------------------------------------
# Source files
# ------------------------------------------------------------

source_files = sorted(DEDUP_ROOT.rglob("*.jsonl.gz"))

print(
    f"\nSource files found: {len(source_files)}",
    flush=True
)

assert len(source_files) == 72, (
    f"Expected 72 source files, found {len(source_files)}"
)

# ------------------------------------------------------------
# Counters
# ------------------------------------------------------------

documents = 0
characters = 0
tokens = 0
chunks = 0
unk_tokens = 0

batch_texts = []

start_time = time.time()
last_progress_time = start_time
last_progress_docs = 0

print("\n" + "=" * 80, flush=True)
print("STARTING BENCHMARK", flush=True)
print("=" * 80, flush=True)

# ------------------------------------------------------------
# Stream sources sequentially
# ------------------------------------------------------------

stop = False

for source_index, source_path in enumerate(source_files, 1):

    if stop:
        break

    print(
        f"\n[SOURCE {source_index}/{len(source_files)}] "
        f"{source_path.relative_to(DEDUP_ROOT)}",
        flush=True
    )

    source_documents = 0
    source_start = time.time()

    for record in read_jsonl_gz(source_path):

        text = extract_text(record)

        if not text.strip():
            continue

        batch_texts.append(text)

        # Process full batch.
        if len(batch_texts) >= BATCH_SIZE:

            encodings = tokenizer.encode_batch(batch_texts)

            for text_item, encoding in zip(batch_texts, encodings):

                ids = encoding.ids

                documents += 1
                source_documents += 1

                characters += len(text_item)
                tokens += len(ids)
                unk_tokens += ids.count(UNK_ID)

                chunks += count_chunks(
                    len(ids),
                    CONTEXT_LENGTH
                )

                # --------------------------------------------
                # Progress
                # --------------------------------------------

                if (
                    documents % PROGRESS_INTERVAL == 0
                    or documents == BENCHMARK_DOCUMENTS
                ):

                    now = time.time()

                    elapsed = now - start_time
                    interval_elapsed = now - last_progress_time

                    overall_rate = (
                        documents / elapsed
                        if elapsed > 0 else 0
                    )

                    interval_docs = (
                        documents - last_progress_docs
                    )

                    interval_rate = (
                        interval_docs / interval_elapsed
                        if interval_elapsed > 0 else 0
                    )

                    remaining = (
                        BENCHMARK_DOCUMENTS - documents
                    )

                    eta_seconds = (
                        remaining / overall_rate
                        if overall_rate > 0 else 0
                    )

                    print(
                        "\n"
                        + "-" * 80,
                        flush=True
                    )

                    print(
                        f"[PROGRESS] "
                        f"{documents:,} / "
                        f"{BENCHMARK_DOCUMENTS:,} "
                        f"({documents / BENCHMARK_DOCUMENTS * 100:6.2f}%)",
                        flush=True
                    )

                    print(
                        f"Documents/sec : {overall_rate:,.1f}",
                        flush=True
                    )

                    print(
                        f"Interval rate : {interval_rate:,.1f}",
                        flush=True
                    )

                    print(
                        f"Characters    : {characters:,}",
                        flush=True
                    )

                    print(
                        f"Tokens        : {tokens:,}",
                        flush=True
                    )

                    print(
                        f"Chunks        : {chunks:,}",
                        flush=True
                    )

                    print(
                        f"[UNK] tokens  : {unk_tokens:,}",
                        flush=True
                    )

                    print(
                        f"Elapsed       : {elapsed / 60:.2f} min",
                        flush=True
                    )

                    print(
                        f"ETA           : {eta_seconds / 60:.2f} min",
                        flush=True
                    )

                    print(
                        "-" * 80,
                        flush=True
                    )

                    last_progress_time = now
                    last_progress_docs = documents

                if documents >= BENCHMARK_DOCUMENTS:
                    stop = True
                    break

            batch_texts.clear()

            if stop:
                break

    source_elapsed = time.time() - source_start

    print(
        f"✓ SOURCE COMPLETE | "
        f"documents={source_documents:,} | "
        f"elapsed={source_elapsed / 60:.2f} min",
        flush=True
    )


# ------------------------------------------------------------
# Process remaining partial batch
# ------------------------------------------------------------

if batch_texts and documents < BENCHMARK_DOCUMENTS:

    remaining = BENCHMARK_DOCUMENTS - documents

    batch_texts = batch_texts[:remaining]

    encodings = tokenizer.encode_batch(batch_texts)

    for text_item, encoding in zip(batch_texts, encodings):

        ids = encoding.ids

        documents += 1
        characters += len(text_item)
        tokens += len(ids)
        unk_tokens += ids.count(UNK_ID)

        chunks += count_chunks(
            len(ids),
            CONTEXT_LENGTH
        )

    batch_texts.clear()


# ------------------------------------------------------------
# Final results
# ------------------------------------------------------------

elapsed = time.time() - start_time

rate = (
    documents / elapsed
    if elapsed > 0 else 0
)

print("\n" + "=" * 80, flush=True)
print("10K BENCHMARK COMPLETE", flush=True)
print("=" * 80, flush=True)

print(f"\nDocuments      : {documents:,}", flush=True)
print(f"Characters     : {characters:,}", flush=True)
print(f"Tokens         : {tokens:,}", flush=True)
print(f"Training chunks: {chunks:,}", flush=True)
print(f"[UNK] tokens   : {unk_tokens:,}", flush=True)
print(f"Elapsed        : {elapsed / 60:.2f} min", flush=True)
print(f"Rate           : {rate:,.1f} docs/sec", flush=True)

assert documents == BENCHMARK_DOCUMENTS
assert tokens > 0
assert chunks > 0
assert unk_tokens == 0

print("\n✓ Exactly 10,000 documents processed", flush=True)
print("✓ Batched tokenization verified", flush=True)
print("✓ 4,096-token chunking verified", flush=True)
print("✓ Zero [UNK] tokens", flush=True)
print("✓ No tokenized corpus created", flush=True)
print("✓ Dedup corpus untouched", flush=True)
print("✓ Tokenizer untouched", flush=True)

In [ ]:
# =============================================================================
# PHASE 4 — MEMORY-SAFE PARALLEL 10K BENCHMARK
# BOUNDED MULTIPROCESSING + BATCHED TOKENIZATION
# =============================================================================

from pathlib import Path
import gzip
import json
import time
import multiprocessing as mp
import gc

from tokenizers import Tokenizer


# =============================================================================
# CONFIG
# =============================================================================

PHASE4_ROOT = Path("/kaggle/working/phase4_training_prep")

TOKENIZER_PATH = (
    PHASE4_ROOT / "tokenizer" / "legal_bpe_32k.json"
)

DEDUP_ROOT = (
    PHASE4_ROOT / "dedup"
)

TRAIN_MANIFEST = (
    PHASE4_ROOT / "manifests" / "train_manifest.jsonl"
)

# Benchmark only
BENCHMARK_DOCS = 10_000

# Smaller batches = lower RAM
BATCH_SIZE = 128

CONTEXT_LENGTH = 4096

# Start here.
NUM_WORKERS = 4

# IMPORTANT:
# Do not allow a huge number of batches to accumulate in RAM.
MAX_IN_FLIGHT = NUM_WORKERS * 2

PROGRESS_INTERVAL = 1_000


# =============================================================================
# VALIDATION
# =============================================================================

assert TOKENIZER_PATH.exists()
assert DEDUP_ROOT.exists()
assert TRAIN_MANIFEST.exists()


# =============================================================================
# LOAD TOKENIZER
# =============================================================================

tokenizer = Tokenizer.from_file(
    str(TOKENIZER_PATH)
)

VOCAB_SIZE = tokenizer.get_vocab_size()

assert VOCAB_SIZE == 32_000

UNK_ID = tokenizer.token_to_id("[UNK]")

assert UNK_ID is not None


# =============================================================================
# HEADER
# =============================================================================

print("=" * 80)
print("PHASE 4 — MEMORY-SAFE PARALLEL 10K BENCHMARK")
print("BOUNDED MULTIPROCESS BATCHED TOKENIZATION")
print("=" * 80)

print(f"\nTokenizer       : {TOKENIZER_PATH}")
print(f"Dedup root      : {DEDUP_ROOT}")
print(f"Documents       : {BENCHMARK_DOCS:,}")
print(f"Batch size      : {BATCH_SIZE}")
print(f"Context         : {CONTEXT_LENGTH:,}")
print(f"Workers         : {NUM_WORKERS}")
print(f"Max in-flight   : {MAX_IN_FLIGHT}")
print(f"Vocabulary      : {VOCAB_SIZE:,}")
print(f"[UNK] ID        : {UNK_ID}")


# =============================================================================
# LOAD TRAINING HASHES
# =============================================================================

print("\n" + "=" * 80)
print("LOADING TRAINING MANIFEST")
print("=" * 80)

train_hashes = set()

with TRAIN_MANIFEST.open(
    "r",
    encoding="utf-8",
) as f:

    for line in f:

        if not line.strip():
            continue

        record = json.loads(line)

        h = (
            record.get("hash")
            or record.get("text_hash")
            or record.get("document_hash")
            or record.get("sha256")
        )

        if h:
            train_hashes.add(h)

print(f"Training hashes : {len(train_hashes):,}")


# =============================================================================
# SOURCE FILES
# =============================================================================

source_files = sorted(
    DEDUP_ROOT.rglob("*.jsonl.gz")
)

assert source_files

print(f"Source files    : {len(source_files)}")


# =============================================================================
# WORKER TOKENIZER
# =============================================================================

_WORKER_TOKENIZER = None


def init_worker(tokenizer_path):

    global _WORKER_TOKENIZER

    _WORKER_TOKENIZER = Tokenizer.from_file(
        str(tokenizer_path)
    )


# =============================================================================
# WORKER
# =============================================================================

def tokenize_batch(texts):

    global _WORKER_TOKENIZER

    encodings = _WORKER_TOKENIZER.encode_batch(
        texts
    )

    documents = len(texts)

    characters = sum(
        len(text)
        for text in texts
    )

    tokens = 0
    chunks = 0
    unk_tokens = 0

    for encoding in encodings:

        ids = encoding.ids

        n = len(ids)

        tokens += n

        if n:
            chunks += (
                n + CONTEXT_LENGTH - 1
            ) // CONTEXT_LENGTH

        unk_tokens += sum(
            token_id == UNK_ID
            for token_id in ids
        )

    return (
        documents,
        characters,
        tokens,
        chunks,
        unk_tokens,
    )


# =============================================================================
# EXTRACT RECORD
# =============================================================================

def extract_hash_and_text(record):

    h = (
        record.get("hash")
        or record.get("text_hash")
        or record.get("document_hash")
        or record.get("sha256")
    )

    text = (
        record.get("text")
        or record.get("content")
        or record.get("document_text")
        or ""
    )

    if not isinstance(text, str):
        text = str(text)

    return h, text


# =============================================================================
# STREAM BATCHES
# =============================================================================

def iter_batches(source_path, remaining):

    batch = []

    with gzip.open(
        source_path,
        "rt",
        encoding="utf-8",
        errors="replace",
    ) as f:

        for line in f:

            if remaining <= 0:
                break

            if not line.strip():
                continue

            try:
                record = json.loads(line)
            except Exception:
                continue

            h, text = extract_hash_and_text(record)

            if h not in train_hashes:
                continue

            batch.append(text)

            remaining -= 1

            if len(batch) >= BATCH_SIZE:

                yield batch

                # Release local reference immediately.
                batch = []

        if batch:
            yield batch


# =============================================================================
# MAIN
# =============================================================================

print("\n" + "=" * 80)
print("STARTING MEMORY-SAFE PARALLEL BENCHMARK")
print("=" * 80)

print(f"\nTarget documents : {BENCHMARK_DOCS:,}")
print(f"Workers          : {NUM_WORKERS}")
print(f"Batch size       : {BATCH_SIZE}")
print(f"Max in-flight    : {MAX_IN_FLIGHT}")
print(
    f"Progress         : every "
    f"{PROGRESS_INTERVAL:,} documents"
)


start_time = time.perf_counter()

global_docs = 0
global_chars = 0
global_tokens = 0
global_chunks = 0
global_unk = 0

last_progress_docs = 0
last_progress_time = start_time

remaining_global = BENCHMARK_DOCS


# =============================================================================
# MULTIPROCESS POOL
# =============================================================================

ctx = mp.get_context("fork")

with ctx.Pool(
    processes=NUM_WORKERS,
    initializer=init_worker,
    initargs=(TOKENIZER_PATH,),
) as pool:

    for source_idx, source_path in enumerate(
        source_files,
        1,
    ):

        if remaining_global <= 0:
            break

        print("\n" + "=" * 80)
        print(
            f"[SOURCE {source_idx}/{len(source_files)}] "
            f"{source_path.relative_to(DEDUP_ROOT)}"
        )
        print("=" * 80)

        source_start = time.perf_counter()

        source_docs_before = global_docs

        batch_iterator = iter_batches(
            source_path,
            remaining_global,
        )

        # ---------------------------------------------------------------------
        # BOUNDED SUBMISSION
        #
        # We deliberately submit only a small window of work.
        # This prevents hundreds of large text batches from sitting in RAM.
        # ---------------------------------------------------------------------

        pending = []

        exhausted = False

        while not exhausted and global_docs < BENCHMARK_DOCS:

            # Fill only a small bounded queue.
            while (
                len(pending) < MAX_IN_FLIGHT
                and not exhausted
            ):

                try:
                    batch = next(batch_iterator)

                except StopIteration:
                    exhausted = True
                    break

                pending.append(
                    pool.apply_async(
                        tokenize_batch,
                        (batch,),
                    )
                )

                # Main process no longer needs this reference.
                del batch

            # -------------------------------------------------------------
            # WAIT FOR ONE RESULT
            # -------------------------------------------------------------

            if not pending:
                break

            result = pending.pop(0).get()

            (
                docs,
                chars,
                tokens,
                chunks,
                unk,
            ) = result

            global_docs += docs
            global_chars += chars
            global_tokens += tokens
            global_chunks += chunks
            global_unk += unk

            remaining_global = (
                BENCHMARK_DOCS - global_docs
            )

            # -------------------------------------------------------------
            # PROGRESS
            # -------------------------------------------------------------

            if (
                global_docs - last_progress_docs
                >= PROGRESS_INTERVAL
                or global_docs >= BENCHMARK_DOCS
            ):

                now = time.perf_counter()

                elapsed = (
                    now - start_time
                )

                interval_elapsed = (
                    now - last_progress_time
                )

                total_rate = (
                    global_docs / elapsed
                    if elapsed > 0
                    else 0
                )

                interval_docs = (
                    global_docs
                    - last_progress_docs
                )

                interval_rate = (
                    interval_docs / interval_elapsed
                    if interval_elapsed > 0
                    else 0
                )

                remaining = (
                    BENCHMARK_DOCS
                    - global_docs
                )

                eta_seconds = (
                    remaining / total_rate
                    if total_rate > 0
                    else 0
                )

                print("\n" + "-" * 80)

                print(
                    f"[PROGRESS] "
                    f"{global_docs:,} / "
                    f"{BENCHMARK_DOCS:,} "
                    f"("
                    f"{100 * global_docs / BENCHMARK_DOCS:6.2f}"
                    f"%)"
                )

                print(
                    f"Workers        : {NUM_WORKERS}"
                )

                print(
                    f"Batch size     : {BATCH_SIZE}"
                )

                print(
                    f"In-flight      : "
                    f"{len(pending)}/{MAX_IN_FLIGHT}"
                )

                print(
                    f"Documents/sec  : "
                    f"{total_rate:,.1f}"
                )

                print(
                    f"Interval rate  : "
                    f"{interval_rate:,.1f}"
                )

                print(
                    f"Characters     : "
                    f"{global_chars:,}"
                )

                print(
                    f"Tokens         : "
                    f"{global_tokens:,}"
                )

                print(
                    f"Chunks         : "
                    f"{global_chunks:,}"
                )

                print(
                    f"[UNK] tokens   : "
                    f"{global_unk:,}"
                )

                print(
                    f"Elapsed        : "
                    f"{elapsed / 60:.2f} min"
                )

                print(
                    f"ETA            : "
                    f"{eta_seconds / 60:.2f} min"
                )

                print("-" * 80)

                last_progress_docs = global_docs
                last_progress_time = now

            if global_docs >= BENCHMARK_DOCS:
                break

        # ---------------------------------------------------------------------
        # CLEAN UP SOURCE
        # ---------------------------------------------------------------------

        source_docs = (
            global_docs
            - source_docs_before
        )

        source_elapsed = (
            time.perf_counter()
            - source_start
        )

        print(
            f"✓ SOURCE COMPLETE | "
            f"documents={source_docs:,} | "
            f"elapsed={source_elapsed / 60:.2f} min"
        )

        gc.collect()

        if global_docs >= BENCHMARK_DOCS:
            break


# =============================================================================
# FINAL
# =============================================================================

elapsed = time.perf_counter() - start_time

rate = (
    global_docs / elapsed
    if elapsed > 0
    else 0
)

print("\n" + "=" * 80)
print("MEMORY-SAFE PARALLEL 10K BENCHMARK COMPLETE")
print("=" * 80)

print(f"\nDocuments      : {global_docs:,}")
print(f"Characters     : {global_chars:,}")
print(f"Tokens         : {global_tokens:,}")
print(f"Training chunks: {global_chunks:,}")
print(f"[UNK] tokens   : {global_unk:,}")
print(f"Elapsed        : {elapsed / 60:.2f} min")
print(f"Rate           : {rate:,.1f} docs/sec")

assert global_docs == BENCHMARK_DOCS

assert global_unk == 0

print("\n✓ Exactly 10,000 documents processed")
print("✓ Bounded multiprocessing verified")
print("✓ Batched tokenization verified")
print("✓ 4,096-token chunking verified")
print("✓ Zero [UNK] tokens")
print("✓ No tokenized corpus created")
print("✓ Dedup corpus untouched")
print("✓ Tokenizer untouched")

print("\n" + "=" * 80)
print("MEMORY-SAFE BENCHMARK CHECKPOINT VALID")
print("=" * 80)

In [ ]:
# ============================================================================
# PHASE 4 — MEMORY-SAFE FULL TRAINING TOKENIZATION BENCHMARK
# CORRECT TRAIN MANIFEST SCHEMA
# BOUNDED MULTIPROCESSING
# STREAMING DOCUMENTS
# ============================================================================

from pathlib import Path
import gzip
import json
import time
import math
import os
import gc
from collections import defaultdict
from multiprocessing import Pool

from tokenizers import Tokenizer


# ============================================================================
# CONFIGURATION
# ============================================================================

PHASE4_ROOT = Path("/kaggle/working/phase4_training_prep")

MANIFEST_PATH = (
    PHASE4_ROOT
    / "manifests"
    / "train_manifest.jsonl"
)

DEDUP_ROOT = (
    PHASE4_ROOT
    / "dedup"
)

TOKENIZER_PATH = (
    PHASE4_ROOT
    / "tokenizer"
    / "legal_bpe_32k.json"
)

TOTAL_DOCS = 3_020_655

# Proven stable configuration from your benchmark.
WORKERS = 4
BATCH_SIZE = 128
MAX_IN_FLIGHT = 8

CONTEXT_LENGTH = 4096

PROGRESS_INTERVAL = 10_000

# No output/tokenized corpus will be written.
BENCHMARK_ONLY = True


# ============================================================================
# HEADER
# ============================================================================

print("=" * 80)
print("PHASE 4 — MEMORY-SAFE FULL TRAINING TOKENIZATION")
print("CORRECT TRAIN MANIFEST SCHEMA")
print("BOUNDED MULTIPROCESS + STREAMING")
print("=" * 80)

print(f"\nManifest       : {MANIFEST_PATH}")
print(f"Dedup root     : {DEDUP_ROOT}")
print(f"Tokenizer      : {TOKENIZER_PATH}")
print(f"Documents      : {TOTAL_DOCS:,}")
print(f"Workers        : {WORKERS}")
print(f"Batch size     : {BATCH_SIZE}")
print(f"Max in-flight   : {MAX_IN_FLIGHT}")
print(f"Context        : {CONTEXT_LENGTH:,}")
print(f"Progress       : every {PROGRESS_INTERVAL:,} documents")


# ============================================================================
# VALIDATE PATHS
# ============================================================================

assert MANIFEST_PATH.exists(), (
    f"Training manifest not found:\n{MANIFEST_PATH}"
)

assert DEDUP_ROOT.exists(), (
    f"Dedup root not found:\n{DEDUP_ROOT}"
)

assert TOKENIZER_PATH.exists(), (
    f"Tokenizer not found:\n{TOKENIZER_PATH}"
)


# ============================================================================
# LOAD TOKENIZER
# ============================================================================

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

VOCAB_SIZE = tokenizer.get_vocab_size()
UNK_ID = tokenizer.token_to_id("[UNK]")

print("\n" + "=" * 80)
print("TOKENIZER")
print("=" * 80)

print(f"Vocabulary : {VOCAB_SIZE:,}")
print(f"[UNK] ID   : {UNK_ID}")

assert VOCAB_SIZE == 32_000
assert UNK_ID is not None


# ============================================================================
# LOAD TRAIN MANIFEST CORRECTLY
# ============================================================================
#
# IMPORTANT:
#
# Schema confirmed by your diagnostic:
#
# {
#   "domain": "...",
#   "source_file": "...",
#   "text_hash": "...",
#   "split": "train"
# }
#
# We therefore use:
#     split == "train"
#     text_hash
#
# ============================================================================

print("\n" + "=" * 80)
print("LOADING TRAIN MANIFEST")
print("=" * 80)

training_by_source = defaultdict(set)

manifest_records = 0
training_records = 0

with MANIFEST_PATH.open("r", encoding="utf-8") as f:

    for line in f:

        line = line.strip()

        if not line:
            continue

        record = json.loads(line)

        manifest_records += 1

        # Correct schema.
        if record.get("split") != "train":
            continue

        source_file = record.get("source_file")
        text_hash = record.get("text_hash")

        if not source_file or not text_hash:
            continue

        training_by_source[source_file].add(text_hash)

        training_records += 1


print(f"Manifest records : {manifest_records:,}")
print(f"Training records : {training_records:,}")
print(f"Source files     : {len(training_by_source):,}")

assert training_records == TOTAL_DOCS, (
    f"\nTRAIN MANIFEST MISMATCH\n"
    f"Expected training records : {TOTAL_DOCS:,}\n"
    f"Found                    : {training_records:,}\n"
)

print("\n✓ Training manifest count verified")
print("✓ Correct split field used")
print("✓ Correct text_hash field used")


# ============================================================================
# TOKENIZATION WORKER
# ============================================================================
#
# Each worker loads its own tokenizer once.
# We DO NOT repeatedly serialize the tokenizer.
# We DO NOT send the entire corpus to workers.
# We only send bounded batches of document strings.
# ============================================================================

_WORKER_TOKENIZER = None


def initialize_worker(tokenizer_path):

    global _WORKER_TOKENIZER

    _WORKER_TOKENIZER = Tokenizer.from_file(
        str(tokenizer_path)
    )


def tokenize_batch(texts):

    global _WORKER_TOKENIZER

    encodings = _WORKER_TOKENIZER.encode_batch(texts)

    total_chars = 0
    total_tokens = 0
    total_unk = 0
    total_chunks = 0

    for text, encoding in zip(texts, encodings):

        ids = encoding.ids

        total_chars += len(text)
        total_tokens += len(ids)

        if UNK_ID is not None:
            total_unk += sum(
                1 for token_id in ids
                if token_id == UNK_ID
            )

        if ids:
            total_chunks += math.ceil(
                len(ids) / CONTEXT_LENGTH
            )

    return (
        len(texts),
        total_chars,
        total_tokens,
        total_chunks,
        total_unk,
    )


# ============================================================================
# STREAM SOURCE DOCUMENTS
# ============================================================================

def iter_source_documents(source_path, allowed_hashes):

    """
    Stream documents from one gzipped JSONL source.

    Only documents whose text_hash appears in the verified
    training manifest are yielded.
    """

    with gzip.open(
        source_path,
        "rt",
        encoding="utf-8",
        errors="replace",
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            record = json.loads(line)

            text_hash = record.get("text_hash")

            if text_hash not in allowed_hashes:
                continue

            text = record.get("text")

            if not isinstance(text, str):
                continue

            yield text


# ============================================================================
# SOURCE PATH RESOLUTION
# ============================================================================

def resolve_source(source_file):

    path = DEDUP_ROOT / source_file

    if path.exists():
        return path

    raise FileNotFoundError(
        f"Source file referenced by manifest does not exist:\n{path}"
    )


# ============================================================================
# MAIN BENCHMARK
# ============================================================================

print("\n" + "=" * 80)
print("STARTING FULL TRAINING TOKENIZATION BENCHMARK")
print("=" * 80)

print(f"\nTarget documents : {TOTAL_DOCS:,}")
print(f"Workers          : {WORKERS}")
print(f"Batch size       : {BATCH_SIZE}")
print(f"Max in-flight    : {MAX_IN_FLIGHT}")
print(f"Context          : {CONTEXT_LENGTH:,}")

global_docs = 0
global_chars = 0
global_tokens = 0
global_chunks = 0
global_unk = 0

start_time = time.time()
last_progress_time = start_time
last_progress_docs = 0

source_items = sorted(training_by_source.items())

print(f"\nSources to process : {len(source_items)}")

# ============================================================================
# IMPORTANT MEMORY CONTROL
#
# maxtasksperchild prevents long-lived workers from accumulating memory.
# chunksize=1 keeps task scheduling bounded.
# We manually maintain only MAX_IN_FLIGHT batches.
# ============================================================================

pool = Pool(
    processes=WORKERS,
    initializer=initialize_worker,
    initargs=(TOKENIZER_PATH,),
    maxtasksperchild=250,
)

try:

    for source_index, (source_file, hashes) in enumerate(
        source_items,
        start=1,
    ):

        source_path = resolve_source(source_file)

        print("\n" + "=" * 80)
        print(
            f"[SOURCE {source_index}/{len(source_items)}] "
            f"{source_file}"
        )
        print(
            f"Training hashes : {len(hashes):,}"
        )
        print("=" * 80)

        source_start_docs = global_docs
        source_start_time = time.time()

        # ------------------------------------------------------------
        # Build bounded batch generator.
        # ------------------------------------------------------------

        def batch_generator():

            batch = []

            for text in iter_source_documents(
                source_path,
                hashes,
            ):

                batch.append(text)

                if len(batch) >= BATCH_SIZE:

                    yield batch
                    batch = []

            if batch:
                yield batch

        # ------------------------------------------------------------
        # Bounded submission.
        #
        # Only MAX_IN_FLIGHT batches are allowed to exist as
        # pending worker jobs.
        # ------------------------------------------------------------

        pending = []

        batch_iter = batch_generator()

        exhausted = False

        while True:

            # Fill bounded queue.
            while (
                not exhausted
                and len(pending) < MAX_IN_FLIGHT
            ):

                try:
                    batch = next(batch_iter)
                except StopIteration:
                    exhausted = True
                    break

                pending.append(
                    pool.apply_async(
                        tokenize_batch,
                        (batch,),
                    )
                )

                # Release parent reference immediately.
                del batch

            # Nothing pending means source is finished.
            if not pending:
                break

            # Consume oldest result.
            result = pending.pop(0)

            (
                docs,
                chars,
                tokens,
                chunks,
                unk,
            ) = result.get()

            global_docs += docs
            global_chars += chars
            global_tokens += tokens
            global_chunks += chunks
            global_unk += unk

            # --------------------------------------------------------
            # Progress
            # --------------------------------------------------------

            if (
                global_docs - last_progress_docs
                >= PROGRESS_INTERVAL
                or global_docs >= TOTAL_DOCS
            ):

                now = time.time()

                elapsed = now - start_time

                interval_elapsed = (
                    now - last_progress_time
                )

                interval_docs = (
                    global_docs - last_progress_docs
                )

                overall_rate = (
                    global_docs / elapsed
                    if elapsed > 0
                    else 0
                )

                interval_rate = (
                    interval_docs / interval_elapsed
                    if interval_elapsed > 0
                    else 0
                )

                remaining = max(
                    TOTAL_DOCS - global_docs,
                    0,
                )

                eta_seconds = (
                    remaining / overall_rate
                    if overall_rate > 0
                    else float("inf")
                )

                print("\n" + "-" * 80)
                print(
                    f"[PROGRESS] "
                    f"{global_docs:,} / {TOTAL_DOCS:,} "
                    f"({global_docs / TOTAL_DOCS * 100:6.2f}%)"
                )
                print(
                    f"Documents/sec : {overall_rate:,.1f}"
                )
                print(
                    f"Interval rate : {interval_rate:,.1f}"
                )
                print(
                    f"Characters    : {global_chars:,}"
                )
                print(
                    f"Tokens        : {global_tokens:,}"
                )
                print(
                    f"Chunks        : {global_chunks:,}"
                )
                print(
                    f"[UNK] tokens  : {global_unk:,}"
                )
                print(
                    f"Elapsed       : {elapsed / 60:.2f} min"
                )

                if math.isfinite(eta_seconds):
                    print(
                        f"ETA           : "
                        f"{eta_seconds / 60:.2f} min"
                    )

                print(
                    f"In-flight     : "
                    f"{len(pending)}/{MAX_IN_FLIGHT}"
                )
                print("-" * 80)

                last_progress_docs = global_docs
                last_progress_time = now

        source_docs = (
            global_docs - source_start_docs
        )

        source_elapsed = (
            time.time() - source_start_time
        )

        print(
            f"✓ SOURCE COMPLETE | "
            f"documents={source_docs:,} | "
            f"elapsed={source_elapsed / 60:.2f} min"
        )

        gc.collect()

finally:

    pool.close()
    pool.join()

    gc.collect()


# ============================================================================
# FINAL VALIDATION
# ============================================================================

elapsed = time.time() - start_time

print("\n" + "=" * 80)
print("FULL TRAINING TOKENIZATION BENCHMARK COMPLETE")
print("=" * 80)

print(f"\nDocuments      : {global_docs:,}")
print(f"Characters     : {global_chars:,}")
print(f"Tokens         : {global_tokens:,}")
print(f"Training chunks: {global_chunks:,}")
print(f"[UNK] tokens   : {global_unk:,}")
print(f"Elapsed        : {elapsed / 60:.2f} min")

rate = (
    global_docs / elapsed
    if elapsed > 0
    else 0
)

print(f"Rate           : {rate:,.1f} docs/sec")


# ============================================================================
# CHECKPOINTS
# ============================================================================

assert global_docs == TOTAL_DOCS, (
    f"Expected {TOTAL_DOCS:,} documents, "
    f"processed {global_docs:,}"
)

assert global_unk == 0, (
    f"Found {global_unk:,} [UNK] tokens"
)

print("\n✓ Exactly 3,020,655 training documents processed")
print("✓ Correct train manifest schema used")
print("✓ Bounded multiprocessing verified")
print("✓ Batch tokenization verified")
print("✓ 4,096-token chunking verified")
print("✓ Zero [UNK] tokens")
print("✓ No tokenized corpus created")
print("✓ Dedup corpus untouched")
print("✓ Tokenizer untouched")

print("\n" + "=" * 80)
print("FULL TRAINING TOKENIZATION BENCHMARK CHECKPOINT VALID")
print("=" * 80)

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/working/phase4_training_prep")

print("=" * 80)
print("PHASE 4 — PERSISTENCE / ARTIFACT CHECK")
print("=" * 80)

targets = [
    ROOT / "tokenizer" / "legal_bpe_32k.json",
    ROOT / "tokenizer" / "tokenizer_config.json",
    ROOT / "tokenizer" / "train_file_plan.json",
    ROOT / "manifests" / "train_manifest.jsonl",
]

for p in targets:
    print(
        f"{'✓' if p.exists() else '✗'} "
        f"{p.relative_to(ROOT)}"
        + (f" | {p.stat().st_size / (1024**2):.3f} MB" if p.exists() else "")
    )

print("\n" + "=" * 80)
print("DEDUP CORPUS")
print("=" * 80)

dedup = ROOT / "dedup"

if dedup.exists():
    files = [p for p in dedup.rglob("*") if p.is_file()]

    total_bytes = sum(p.stat().st_size for p in files)

    print(f"Files       : {len(files):,}")
    print(f"Total size  : {total_bytes / (1024**3):.3f} GB")

    for domain in ["acts", "high_courts", "supreme_court", "tribunals"]:
        d = dedup / domain
        if d.exists():
            n = sum(1 for p in d.rglob("*") if p.is_file())
            print(f"{domain:15s}: {n:,} files")
        else:
            print(f"{domain:15s}: MISSING")
else:
    print("✗ dedup directory missing")

print("\n" + "=" * 80)
print("CHECK COMPLETE")
print("=" * 80)

In [ ]:
import shutil
import subprocess

print("Kaggle CLI:", shutil.which("kaggle"))

if shutil.which("kaggle"):
    r = subprocess.run(
        ["kaggle", "--version"],
        capture_output=True,
        text=True
    )
    print(r.stdout.strip() or r.stderr.strip())

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/working/phase4_training_prep")

required = [
    ROOT / "dedup",
    ROOT / "manifests" / "train_manifest.jsonl",
    ROOT / "tokenizer" / "legal_bpe_32k.json",
    ROOT / "tokenizer" / "tokenizer_config.json",
    ROOT / "tokenizer" / "train_file_plan.json",
]

print("=" * 80)
print("PHASE 4 — DATASET PUBLISH CHECK")
print("=" * 80)

for p in required:
    print(f"{'✓' if p.exists() else '✗'} {p}")

print("\nNothing has been copied or modified.")
print("Nothing has been uploaded yet.")

In [ ]:
import subprocess

r = subprocess.run(
    ["kaggle", "datasets", "list", "--mine", "--search", "phase4"],
    capture_output=True,
    text=True,
)

print(r.stdout)
if r.stderr:
    print("STDERR:")
    print(r.stderr)

In [ ]:
import shutil

total, used, free = shutil.disk_usage("/kaggle/working")

print("=" * 80)
print("KAGGLE DISK CHECK")
print("=" * 80)
print(f"Total : {total / 1024**3:.2f} GB")
print(f"Used  : {used  / 1024**3:.2f} GB")
print(f"Free  : {free  / 1024**3:.2f} GB")

In [ ]:
from pathlib import Path
import time

ROOT = Path("/kaggle/working/phase4_training_prep")

total_bytes = sum(
    p.stat().st_size
    for p in ROOT.rglob("*")
    if p.is_file()
)

print("=" * 80)
print("PHASE 4 — DATASET PUBLISH SIZE")
print("=" * 80)

print(f"Root       : {ROOT}")
print(f"Files      : {sum(1 for p in ROOT.rglob('*') if p.is_file()):,}")
print(f"Total size : {total_bytes / (1024**3):.3f} GB")

print("\n✓ Existing files will be used directly")
print("✓ No duplicate corpus will be created")
print("✓ Source corpus will not be modified")

In [ ]:
import subprocess
from pathlib import Path

ROOT = Path("/kaggle/working/phase4_training_prep")
META = ROOT / "dataset-metadata.json"

META.write_text(
    """{
  "title": "CS781 Phase 4 Training Preparation",
  "id": "YOUR_KAGGLE_USERNAME/cs781-phase4-training-prep",
  "licenses": [
    {
      "name": "CC0-1.0"
    }
  ],
  "description": "Phase 4 deduplicated legal corpus, training manifest, and 32K BPE tokenizer artifacts for CS781."
}"""
)

print(META.read_text())

In [ ]:
import subprocess

r = subprocess.run(
    ["kaggle", "config", "view"],
    capture_output=True,
    text=True
)

print(r.stdout)
print(r.stderr)

In [ ]:
from pathlib import Path
import subprocess

ROOT = Path("/kaggle/working/phase4_training_prep")
META = ROOT / "dataset-metadata.json"

META.write_text(
    """{
  "title": "CS781 Phase 4 Training Preparation",
  "id": "oipranesh/cs781-phase4-training-prep",
  "licenses": [
    {
      "name": "CC0-1.0"
    }
  ],
  "description": "Phase 4 deduplicated legal corpus, training manifest, and 32K BPE tokenizer artifacts for CS781."
}"""
)

print("=" * 80)
print("PUBLISHING PHASE 4 DATASET")
print("=" * 80)
print(f"Source : {ROOT}")
print(f"Dataset: oipranesh/cs781-phase4-training-prep")
print("=" * 80)

result = subprocess.run(
    [
        "kaggle",
        "datasets",
        "create",
        "-p",
        str(ROOT),
        "-r",
        "zip",
    ],
    text=True,
)

print("\n" + "=" * 80)
print("KAGGLE DATASET CREATE FINISHED")
print("=" * 80)
print(f"Return code: {result.returncode}")

In [ ]:
from pathlib import Path
import os

print("=" * 80)
print("CS781 — PHASE 1 CHECKPOINT RESTORE")
print("=" * 80)

# Kaggle attached datasets normally appear under /kaggle/input/
input_root = Path("/kaggle/input")

matches = list(input_root.glob("**/train_manifest.jsonl"))

print(f"\nFound train_manifest.jsonl: {len(matches)}")

for p in matches:
    print(f"  {p}")

assert matches, "ERROR: Phase 4 checkpoint dataset is not attached."

DATASET_ROOT = matches[0].parent.parent

print(f"\nDataset root detected:")
print(f"  {DATASET_ROOT}")

In [ ]:
from pathlib import Path
import json

print("=" * 80)
print("PHASE 1 — CHECKPOINT INTEGRITY CHECK")
print("=" * 80)

root = Path(DATASET_ROOT)

manifest = root / "manifests" / "train_manifest.jsonl"
tokenizer = root / "tokenizer" / "legal_bpe_32k.json"
tokenizer_config = root / "tokenizer" / "tokenizer_config.json"
train_plan = root / "tokenizer" / "train_file_plan.json"
dedup = root / "dedup"

required = {
    "train_manifest.jsonl": manifest,
    "legal_bpe_32k.json": tokenizer,
    "tokenizer_config.json": tokenizer_config,
    "train_file_plan.json": train_plan,
    "dedup/": dedup,
}

for name, path in required.items():
    print(f"{'✓' if path.exists() else '✗'} {name}")

assert manifest.exists()
assert tokenizer.exists()
assert tokenizer_config.exists()
assert train_plan.exists()
assert dedup.exists()

print("\nCheckpoint files are present.")

In [ ]:
print("=" * 80)
print("TRAIN MANIFEST VERIFICATION")
print("=" * 80)

count = 0
source_counts = {}

with open(manifest, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        record = json.loads(line)

        assert record["split"] == "train"
        assert "text_hash" in record
        assert "source_file" in record
        assert "domain" in record

        count += 1
        source = record["source_file"]
        source_counts[source] = source_counts.get(source, 0) + 1

print(f"Training documents : {count:,}")
print(f"Source files       : {len(source_counts)}")

assert count == 3_020_655, (
    f"Expected 3,020,655 records, found {count:,}"
)

print("✓ Manifest count verified")
print("✓ Manifest schema verified")

In [ ]:
from pathlib import Path

print("=" * 80)
print("PHASE 4 CHECKPOINT — ACTUAL MOUNTED STRUCTURE")
print("=" * 80)

root = Path("/kaggle/input")

print("\nTop-level inputs:")
for p in sorted(root.iterdir()):
    print(" ", p)

print("\nAll Phase 4 files/directories:")
for p in sorted(root.rglob("*")):
    if p.is_file():
        print(f"  FILE : {p}")
    elif p.is_dir():
        print(f"  DIR  : {p}")

In [ ]:
from pathlib import Path

DATASET_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

print("=" * 80)
print("PHASE 4 CHECKPOINT — VERIFIED")
print("=" * 80)

required = {
    "dedup": DATASET_ROOT / "dedup",
    "train_manifest": DATASET_ROOT / "manifests" / "train_manifest.jsonl",
    "validation_manifest": DATASET_ROOT / "manifests" / "validation_manifest.jsonl",
    "test_manifest": DATASET_ROOT / "manifests" / "test_manifest.jsonl",
    "tokenizer": DATASET_ROOT / "tokenizer" / "legal_bpe_32k.json",
    "tokenizer_config": DATASET_ROOT / "tokenizer" / "tokenizer_config.json",
    "train_file_plan": DATASET_ROOT / "tokenizer" / "train_file_plan.json",
}

for name, path in required.items():
    assert path.exists(), f"MISSING: {path}"
    print(f"✓ {name:20s} {path}")

print()
print("✓ PHASE 4 CHECKPOINT FULLY VERIFIED")
print("=" * 80)

In [ ]:
from pathlib import Path
import json

DATASET_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

print("=" * 80)
print("PHASE 4 — EXISTING TRAINING CHUNK CHECK")
print("=" * 80)

chunks_root = DATASET_ROOT / "training_chunks"
config_path = DATASET_ROOT / "training_example_config.json"

print(f"\nTraining chunks root : {chunks_root}")
print(f"Config               : {config_path}")

# ------------------------------------------------------------------
# Training chunk runs
# ------------------------------------------------------------------

if chunks_root.exists():
    runs = sorted(p for p in chunks_root.iterdir() if p.is_dir())

    print(f"\nChunk runs found : {len(runs)}")

    for run in runs:
        files = [p for p in run.rglob("*") if p.is_file()]
        total_bytes = sum(p.stat().st_size for p in files)

        print(
            f"  {run.name} | "
            f"files={len(files):,} | "
            f"size={total_bytes / (1024**3):.3f} GB"
        )

        for f in files[:10]:
            print(f"      {f.relative_to(run)}")

        if len(files) > 10:
            print(f"      ... {len(files) - 10:,} more files")

else:
    print("\nNO training_chunks directory found.")

# ------------------------------------------------------------------
# Training example configuration
# ------------------------------------------------------------------

print("\n" + "=" * 80)
print("TRAINING EXAMPLE CONFIG")
print("=" * 80)

if config_path.exists():
    with open(config_path, "r", encoding="utf-8") as f:
        config = json.load(f)

    print(json.dumps(config, indent=2, ensure_ascii=False))
else:
    print("Config not found.")

print("\n" + "=" * 80)
print("INSPECTION COMPLETE")
print("=" * 80)

In [ ]:
# =============================================================================
# PHASE 4 — EXISTING TRAINING CHUNK INTEGRITY / RESUME CHECK
# =============================================================================
# READ-ONLY:
#   - Does NOT modify /kaggle/input
#   - Does NOT tokenize the corpus
#   - Does NOT generate new chunks
#   - Verifies the existing chunk against the training manifest
# =============================================================================

from pathlib import Path
import json
import hashlib
import gzip
import time

DATASET_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

TRAIN_MANIFEST = DATASET_ROOT / "manifests" / "train_manifest.jsonl"
CHUNKS_ROOT = DATASET_ROOT / "training_chunks"

RUN_DIR = CHUNKS_ROOT / "run_20260928_165448"
EXISTING_CHUNK = RUN_DIR / "acts__documents.jsonl.jsonl"

TOTAL_TRAIN_DOCS = 3_020_655

print("=" * 80)
print("PHASE 4 — EXISTING TRAINING CHUNK INTEGRITY / RESUME CHECK")
print("=" * 80)

# -----------------------------------------------------------------------------
# Basic existence checks
# -----------------------------------------------------------------------------

assert DATASET_ROOT.exists(), f"Dataset root missing: {DATASET_ROOT}"
assert TRAIN_MANIFEST.exists(), f"Training manifest missing: {TRAIN_MANIFEST}"
assert EXISTING_CHUNK.exists(), f"Existing chunk missing: {EXISTING_CHUNK}"

print(f"\nDataset root    : {DATASET_ROOT}")
print(f"Train manifest  : {TRAIN_MANIFEST}")
print(f"Existing chunk  : {EXISTING_CHUNK}")

chunk_size = EXISTING_CHUNK.stat().st_size

print(
    f"Chunk size      : "
    f"{chunk_size / (1024**2):.2f} MB"
)

# -----------------------------------------------------------------------------
# Read training manifest
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("LOADING TRAINING MANIFEST")
print("=" * 80)

training_hashes = set()
training_sources = {}

manifest_count = 0

with open(TRAIN_MANIFEST, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        rec = json.loads(line)

        h = rec["text_hash"]
        src = rec["source_file"]

        training_hashes.add(h)
        training_sources[h] = src

        manifest_count += 1

print(f"Manifest documents : {manifest_count:,}")

assert manifest_count == TOTAL_TRAIN_DOCS, (
    f"Manifest mismatch: expected {TOTAL_TRAIN_DOCS:,}, "
    f"got {manifest_count:,}"
)

print("✓ Training manifest count verified")
print(f"Unique hashes      : {len(training_hashes):,}")

assert len(training_hashes) == TOTAL_TRAIN_DOCS, (
    "Training manifest contains duplicate text_hash values."
)

print("✓ Training hashes are unique")

# -----------------------------------------------------------------------------
# Inspect existing chunk
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("INSPECTING EXISTING ACTS CHUNK")
print("=" * 80)

chunk_records = 0
chunk_hashes = set()
chunk_sources = {}
chunk_bad_json = 0
chunk_missing_hash = 0

start = time.time()

with open(EXISTING_CHUNK, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):

        if not line.strip():
            continue

        try:
            rec = json.loads(line)
        except Exception:
            chunk_bad_json += 1
            continue

        chunk_records += 1

        # Try the expected field first.
        h = rec.get("text_hash")

        if h is None:
            chunk_missing_hash += 1
            continue

        chunk_hashes.add(h)

        source = rec.get("source_file")
        if source is not None:
            chunk_sources[source] = chunk_sources.get(source, 0) + 1

elapsed = time.time() - start

print(f"Chunk records     : {chunk_records:,}")
print(f"Chunk unique hash : {len(chunk_hashes):,}")
print(f"Bad JSON records  : {chunk_bad_json:,}")
print(f"Missing hashes    : {chunk_missing_hash:,}")
print(f"Read time         : {elapsed:.2f} sec")

# -----------------------------------------------------------------------------
# Integrity checks
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("INTEGRITY CHECK")
print("=" * 80)

assert chunk_bad_json == 0, (
    f"Existing chunk contains {chunk_bad_json:,} invalid JSON records."
)

assert chunk_missing_hash == 0, (
    f"Existing chunk contains {chunk_missing_hash:,} records without text_hash."
)

assert len(chunk_hashes) == chunk_records, (
    "Existing chunk contains duplicate text_hash values."
)

print("✓ JSON integrity verified")
print("✓ Every chunk record has text_hash")
print("✓ Chunk hashes are unique")

# Every existing hash must belong to training set.
not_in_training = chunk_hashes - training_hashes

print(f"\nHashes not in training manifest : {len(not_in_training):,}")

assert not not_in_training, (
    "Existing chunk contains hashes that are not in the training manifest."
)

print("✓ Every existing chunk hash belongs to training split")

# -----------------------------------------------------------------------------
# Source verification
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("SOURCE DISTRIBUTION")
print("=" * 80)

for source, count in sorted(chunk_sources.items()):
    print(f"{source:65s} | {count:,}")

# -----------------------------------------------------------------------------
# Completion estimate
# -----------------------------------------------------------------------------

completed = len(chunk_hashes)
remaining = TOTAL_TRAIN_DOCS - completed
fraction = completed / TOTAL_TRAIN_DOCS

print("\n" + "=" * 80)
print("RESUME STATUS")
print("=" * 80)

print(f"Training documents : {TOTAL_TRAIN_DOCS:,}")
print(f"Already chunked    : {completed:,}")
print(f"Remaining          : {remaining:,}")
print(f"Completion         : {fraction * 100:.4f}%")

print()

if remaining == 0:
    print("✓ ALL TRAINING DOCUMENTS ARE ALREADY CHUNKED")
else:
    print("✓ Existing chunk is valid")
    print(f"→ {remaining:,} training documents remain to be processed")

print("\n" + "=" * 80)
print("READ-ONLY CHECK COMPLETE")
print("=" * 80)

In [ ]:
# =============================================================================
# PHASE 4 — CORRECT TRAINING CHUNK INTEGRITY CHECK
# Document hashes may legitimately repeat across multiple chunks.
# READ-ONLY — does not modify checkpoint artifacts.
# =============================================================================

from pathlib import Path
import json
from collections import Counter

ROOT = Path(
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep"
)

MANIFEST = ROOT / "manifests/train_manifest.jsonl"
CHUNKS_ROOT = ROOT / "training_chunks"

CONTEXT_LENGTH = 4096

print("=" * 80)
print("PHASE 4 — TRAINING CHUNK INTEGRITY CHECK")
print("=" * 80)

print(f"Manifest : {MANIFEST}")
print(f"Chunks   : {CHUNKS_ROOT}")
print(f"Context  : {CONTEXT_LENGTH:,}")

# =============================================================================
# 1. LOAD TRAINING MANIFEST
# =============================================================================

print("\n" + "=" * 80)
print("1. TRAINING MANIFEST")
print("=" * 80)

training_hashes = set()
manifest_records = 0
manifest_bad = 0

with open(MANIFEST, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, 1):
        try:
            rec = json.loads(line)
        except Exception:
            manifest_bad += 1
            continue

        manifest_records += 1

        h = rec.get("text_hash")

        if not h:
            manifest_bad += 1
            continue

        training_hashes.add(h)

print(f"Manifest records       : {manifest_records:,}")
print(f"Unique training hashes : {len(training_hashes):,}")
print(f"Malformed/missing hash  : {manifest_bad:,}")

assert manifest_records == 3_020_655, (
    f"Expected 3,020,655 manifest records, got {manifest_records:,}"
)

assert len(training_hashes) == manifest_records, (
    "Training manifest contains duplicate text_hash values."
)

print("✓ 3,020,655 training documents verified")
print("✓ Training document hashes are unique")

# =============================================================================
# 2. FIND CHUNK FILES
# =============================================================================

print("\n" + "=" * 80)
print("2. TRAINING CHUNK FILES")
print("=" * 80)

chunk_files = sorted(CHUNKS_ROOT.rglob("*.jsonl"))

print(f"Chunk files found : {len(chunk_files)}")

for p in chunk_files:
    print(
        f"  {p.relative_to(CHUNKS_ROOT)} | "
        f"{p.stat().st_size / (1024**3):.3f} GB"
    )

if not chunk_files:
    raise RuntimeError("No training chunk files found.")

# =============================================================================
# 3. VALIDATE CHUNKS
# =============================================================================

print("\n" + "=" * 80)
print("3. CHUNK INTEGRITY")
print("=" * 80)

total_chunk_records = 0
total_token_values = 0

valid_hash_records = 0
unknown_hash_records = 0
missing_hash_records = 0
malformed_records = 0

oversized_chunks = 0
empty_chunks = 0

# A document is allowed to occur in multiple chunks.
hash_chunk_counts = Counter()

# Keep only a small sample of problematic records for diagnostics.
unknown_examples = []
oversized_examples = []
malformed_examples = []

for file_index, path in enumerate(chunk_files, 1):

    print(f"\n[{file_index}/{len(chunk_files)}] {path.name}")

    file_records = 0
    file_unknown = 0
    file_oversized = 0

    with open(path, "r", encoding="utf-8") as f:

        for line_no, line in enumerate(f, 1):

            try:
                rec = json.loads(line)
            except Exception:
                malformed_records += 1

                if len(malformed_examples) < 10:
                    malformed_examples.append(
                        (str(path), line_no)
                    )

                continue

            total_chunk_records += 1
            file_records += 1

            # -----------------------------------------------------------------
            # TEXT HASH
            # -----------------------------------------------------------------

            text_hash = rec.get("text_hash")

            if not text_hash:
                missing_hash_records += 1
            elif text_hash not in training_hashes:
                unknown_hash_records += 1
                file_unknown += 1

                if len(unknown_examples) < 10:
                    unknown_examples.append(
                        (str(path), line_no, text_hash)
                    )
            else:
                valid_hash_records += 1
                hash_chunk_counts[text_hash] += 1

            # -----------------------------------------------------------------
            # TOKEN COUNT
            # -----------------------------------------------------------------

            tokens = None

            # Support common representations.
            if isinstance(rec.get("tokens"), list):
                tokens = rec["tokens"]

            elif isinstance(rec.get("input_ids"), list):
                tokens = rec["input_ids"]

            elif isinstance(rec.get("token_ids"), list):
                tokens = rec["token_ids"]

            elif isinstance(rec.get("token_count"), int):
                tokens = rec["token_count"]

            elif isinstance(rec.get("num_tokens"), int):
                tokens = rec["num_tokens"]

            if isinstance(tokens, list):

                token_count = len(tokens)
                total_token_values += token_count

                if token_count == 0:
                    empty_chunks += 1

                if token_count > CONTEXT_LENGTH:
                    oversized_chunks += 1
                    file_oversized += 1

                    if len(oversized_examples) < 10:
                        oversized_examples.append(
                            (str(path), line_no, token_count)
                        )

            elif isinstance(tokens, int):

                total_token_values += tokens

                if tokens == 0:
                    empty_chunks += 1

                if tokens > CONTEXT_LENGTH:
                    oversized_chunks += 1
                    file_oversized += 1

                    if len(oversized_examples) < 10:
                        oversized_examples.append(
                            (str(path), line_no, tokens)
                        )

    print(f"  Records           : {file_records:,}")
    print(f"  Unknown hashes    : {file_unknown:,}")
    print(f"  Oversized chunks  : {file_oversized:,}")

# =============================================================================
# 4. REPORT
# =============================================================================

print("\n" + "=" * 80)
print("4. FINAL CHUNK STATISTICS")
print("=" * 80)

documents_represented = len(hash_chunk_counts)

print(f"Chunk records                 : {total_chunk_records:,}")
print(f"Documents represented         : {documents_represented:,}")
print(f"Valid manifest hash records   : {valid_hash_records:,}")
print(f"Unknown hash records          : {unknown_hash_records:,}")
print(f"Missing hash records          : {missing_hash_records:,}")
print(f"Malformed records             : {malformed_records:,}")
print(f"Empty chunks                  : {empty_chunks:,}")
print(f"Oversized chunks              : {oversized_chunks:,}")

if hash_chunk_counts:
    multi_chunk_docs = sum(
        1 for n in hash_chunk_counts.values()
        if n > 1
    )

    max_chunks_per_doc = max(hash_chunk_counts.values())

    print(f"Documents with >1 chunk       : {multi_chunk_docs:,}")
    print(f"Maximum chunks for one doc    : {max_chunks_per_doc:,}")

# =============================================================================
# 5. DIAGNOSTIC EXAMPLES
# =============================================================================

if unknown_examples:
    print("\n" + "=" * 80)
    print("UNKNOWN HASH EXAMPLES")
    print("=" * 80)

    for item in unknown_examples:
        print(item)

if oversized_examples:
    print("\n" + "=" * 80)
    print("OVERSIZED CHUNK EXAMPLES")
    print("=" * 80)

    for item in oversized_examples:
        print(item)

if malformed_examples:
    print("\n" + "=" * 80)
    print("MALFORMED RECORD EXAMPLES")
    print("=" * 80)

    for item in malformed_examples:
        print(item)

# =============================================================================
# 6. CORRECT VALIDATION RULES
# =============================================================================

print("\n" + "=" * 80)
print("5. VALIDATION RESULT")
print("=" * 80)

assert manifest_records == 3_020_655
assert len(training_hashes) == 3_020_655

assert unknown_hash_records == 0, (
    f"{unknown_hash_records:,} chunk records reference hashes "
    "not present in the training manifest."
)

assert missing_hash_records == 0, (
    f"{missing_hash_records:,} chunk records have no text_hash."
)

assert malformed_records == 0, (
    f"{malformed_records:,} malformed chunk records found."
)

assert oversized_chunks == 0, (
    f"{oversized_chunks:,} chunks exceed the {CONTEXT_LENGTH:,}-token context."
)

print("✓ Training manifest contains 3,020,655 unique documents")
print("✓ Every chunk hash belongs to the training manifest")
print("✓ Multiple chunks per document are allowed")
print("✓ No missing chunk hashes")
print("✓ No malformed chunk records")
print(f"✓ No chunk exceeds {CONTEXT_LENGTH:,} tokens")

print("\n" + "=" * 80)
print("TRAINING CHUNK INTEGRITY CHECK COMPLETE")
print("=" * 80)

In [ ]:
from pathlib import Path
import json

ROOT = Path(
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep"
)

print("=" * 80)
print("PHASE 4 — COMPLETE CHECKPOINT DATASET INVENTORY")
print("=" * 80)

print(f"Dataset root : {ROOT}")
print()

assert ROOT.exists(), f"Dataset root does not exist: {ROOT}"

# -------------------------------------------------------------------------
# 1. EVERYTHING AT TOP LEVEL
# -------------------------------------------------------------------------

print("=" * 80)
print("TOP-LEVEL CONTENTS")
print("=" * 80)

for p in sorted(ROOT.iterdir()):
    kind = "DIR " if p.is_dir() else "FILE"
    size = ""

    if p.is_file():
        size = f" | {p.stat().st_size / (1024**3):.3f} GB"

    print(f"{kind} {p.name}{size}")

# -------------------------------------------------------------------------
# 2. COMPLETE FILE INVENTORY
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("COMPLETE FILE INVENTORY")
print("=" * 80)

all_files = sorted(p for p in ROOT.rglob("*") if p.is_file())

print(f"Total files : {len(all_files):,}")
print()

total_bytes = 0

for p in all_files:
    rel = p.relative_to(ROOT)
    size = p.stat().st_size
    total_bytes += size

    print(f"{rel} | {size / (1024**3):.3f} GB")

print()
print(f"TOTAL SIZE : {total_bytes / (1024**3):.3f} GB")

# -------------------------------------------------------------------------
# 3. EXPECTED PHASE 4 ARTIFACTS
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("EXPECTED PHASE 4 ARTIFACTS")
print("=" * 80)

expected = {
    "dedup corpus": ROOT / "dedup",

    "train manifest":
        ROOT / "manifests/train_manifest.jsonl",

    "validation manifest":
        ROOT / "manifests/validation_manifest.jsonl",

    "test manifest":
        ROOT / "manifests/test_manifest.jsonl",

    "tokenizer":
        ROOT / "tokenizer/legal_bpe_32k.json",

    "tokenizer config":
        ROOT / "tokenizer/tokenizer_config.json",

    "train file plan":
        ROOT / "tokenizer/train_file_plan.json",

    "training chunks":
        ROOT / "training_chunks",

    "training example config":
        ROOT / "training_example_config.json",
}

for name, path in expected.items():
    exists = path.exists()

    if path.is_file():
        size = f"{path.stat().st_size / (1024**2):.3f} MB"
    elif path.is_dir():
        files = sum(1 for x in path.rglob("*") if x.is_file())
        size = f"{files:,} files"
    else:
        size = ""

    print(
        f"{'✓' if exists else '✗'} "
        f"{name:25s} : {path} {size}"
    )

# -------------------------------------------------------------------------
# 4. DEDUP CORPUS BREAKDOWN
# -------------------------------------------------------------------------

dedup = ROOT / "dedup"

print("\n" + "=" * 80)
print("DEDUP CORPUS")
print("=" * 80)

if dedup.exists():

    dedup_files = [
        p for p in dedup.rglob("*")
        if p.is_file()
    ]

    dedup_bytes = sum(p.stat().st_size for p in dedup_files)

    print(f"Files      : {len(dedup_files):,}")
    print(f"Total size : {dedup_bytes / (1024**3):.3f} GB")

    domain_stats = {}

    for p in dedup_files:
        rel = p.relative_to(dedup)

        if len(rel.parts) >= 2:
            domain = rel.parts[0]
        else:
            domain = "(root)"

        domain_stats.setdefault(
            domain,
            {"files": 0, "bytes": 0}
        )

        domain_stats[domain]["files"] += 1
        domain_stats[domain]["bytes"] += p.stat().st_size

    for domain in sorted(domain_stats):
        s = domain_stats[domain]

        print(
            f"{domain:18s} | "
            f"files={s['files']:3d} | "
            f"size={s['bytes'] / (1024**3):.3f} GB"
        )

else:
    print("✗ DEDUP DIRECTORY MISSING")

# -------------------------------------------------------------------------
# 5. TRAINING CHUNK BREAKDOWN
# -------------------------------------------------------------------------

chunks = ROOT / "training_chunks"

print("\n" + "=" * 80)
print("TRAINING CHUNKS")
print("=" * 80)

if chunks.exists():

    chunk_files = [
        p for p in chunks.rglob("*")
        if p.is_file()
    ]

    chunk_bytes = sum(p.stat().st_size for p in chunk_files)

    print(f"Files      : {len(chunk_files):,}")
    print(f"Total size : {chunk_bytes / (1024**3):.3f} GB")

    for p in chunk_files:
        print(
            f"  {p.relative_to(chunks)} | "
            f"{p.stat().st_size / (1024**3):.3f} GB"
        )

else:
    print("✗ TRAINING CHUNKS DIRECTORY MISSING")

# -------------------------------------------------------------------------
# 6. MANIFEST COUNTS
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("MANIFEST COUNTS")
print("=" * 80)

for name in ["train_manifest.jsonl",
             "validation_manifest.jsonl",
             "test_manifest.jsonl"]:

    path = ROOT / "manifests" / name

    if not path.exists():
        print(f"✗ {name} missing")
        continue

    count = 0

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                count += 1

    print(f"{name:30s} : {count:,}")

# -------------------------------------------------------------------------
# 7. FINAL SUMMARY
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("INVENTORY COMPLETE")
print("=" * 80)

print(f"All dataset files : {len(all_files):,}")
print(f"Dataset size      : {total_bytes / (1024**3):.3f} GB")

print()
print("IMPORTANT:")
print("This cell ONLY inspects the attached checkpoint.")
print("It does NOT tokenize documents.")
print("It does NOT create training chunks.")
print("It does NOT modify the checkpoint.")
print("=" * 80)

In [ ]:
# =============================================================================
# PHASE 4 — CHECKPOINT SOURCE PATH RESOLUTION
# MANIFEST NAMES -> PUBLISHED CHECKPOINT FILES
# =============================================================================

from pathlib import Path

DATASET_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

DEDUP_ROOT = DATASET_ROOT / "dedup"
MANIFEST_PATH = DATASET_ROOT / "manifests" / "train_manifest.jsonl"

print("=" * 80)
print("PHASE 4 — CHECKPOINT SOURCE PATH RESOLUTION")
print("=" * 80)

# -------------------------------------------------------------------------
# Read distinct manifest source files
# -------------------------------------------------------------------------

manifest_sources = set()

with MANIFEST_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        rec = json.loads(line)

        source = rec.get("source_file")

        if source:
            manifest_sources.add(source)

print(f"\nManifest source names : {len(manifest_sources)}")
print(f"Checkpoint corpus     : {DEDUP_ROOT}")

assert len(manifest_sources) == 72, (
    f"Expected 72 manifest sources, got {len(manifest_sources)}"
)

# -------------------------------------------------------------------------
# Resolve each manifest name against actual checkpoint files
# -------------------------------------------------------------------------

resolved_sources = []

print("\n" + "=" * 80)
print("SOURCE RESOLUTION")
print("=" * 80)

for source_file in sorted(manifest_sources):

    # Exact path first
    candidates = [
        DEDUP_ROOT / source_file,

        # Published checkpoint stores decompressed JSONL
        DEDUP_ROOT / source_file.replace(".jsonl.gz", ".jsonl"),

        # Defensive fallback
        DEDUP_ROOT / source_file.replace(".gz", ""),
    ]

    resolved = None

    for candidate in candidates:
        if candidate.exists():
            resolved = candidate
            break

    print(f"\nManifest : {source_file}")

    if resolved is None:
        print("✗ NOT FOUND")
        for candidate in candidates:
            print(f"    tried: {candidate}")
    else:
        print(f"✓ Actual : {resolved}")

        resolved_sources.append(
            (source_file, resolved)
        )

# -------------------------------------------------------------------------
# Final verification
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("RESOLUTION RESULT")
print("=" * 80)

print(
    f"Resolved : {len(resolved_sources):,} / "
    f"{len(manifest_sources):,}"
)

assert len(resolved_sources) == len(manifest_sources), (
    "Not all manifest sources could be resolved against "
    "the attached checkpoint corpus."
)

# -------------------------------------------------------------------------
# Verify uniqueness of actual files
# -------------------------------------------------------------------------

actual_paths = [p.resolve() for _, p in resolved_sources]

assert len(actual_paths) == len(set(actual_paths)), (
    "Two manifest sources resolved to the same physical file."
)

# -------------------------------------------------------------------------
# Verify expected domain/source structure
# -------------------------------------------------------------------------

for manifest_source, actual_path in resolved_sources:

    assert actual_path.is_file(), (
        f"Resolved source is not a file: {actual_path}"
    )

    assert actual_path.stat().st_size > 0, (
        f"Resolved source is empty: {actual_path}"
    )

print("✓ Every manifest source resolves to an existing non-empty file")
print("✓ No two manifest sources resolve to the same file")

print("\n" + "=" * 80)
print("CHECKPOINT SOURCE RESOLUTION VALID")
print("=" * 80)

In [ ]:
from pathlib import Path

# ============================================================
# PHASE 4 — CHECKPOINT SOURCE RESOLUTION
# Handles manifest .jsonl.gz names -> checkpoint .jsonl files
# ============================================================

DATASET_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

DEDUP_ROOT = DATASET_ROOT / "dedup"
TRAIN_MANIFEST = DATASET_ROOT / "manifests" / "train_manifest.jsonl"

assert DEDUP_ROOT.exists(), f"Missing dedup root: {DEDUP_ROOT}"
assert TRAIN_MANIFEST.exists(), f"Missing train manifest: {TRAIN_MANIFEST}"

# ------------------------------------------------------------
# Read unique source names from manifest
# ------------------------------------------------------------

source_files = set()

with TRAIN_MANIFEST.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        record = json.loads(line)

        source_file = record["source_file"]
        source_files.add(source_file)

print("=" * 80)
print("PHASE 4 — CHECKPOINT SOURCE PATH RESOLUTION")
print("=" * 80)
print()
print(f"Manifest source names : {len(source_files):,}")
print(f"Checkpoint corpus     : {DEDUP_ROOT}")
print()

# ------------------------------------------------------------
# Resolve .jsonl.gz manifest names to actual .jsonl files
# ------------------------------------------------------------

resolved = []
missing = []
used_paths = set()

for source_file in sorted(source_files):

    manifest_path = DEDUP_ROOT / source_file

    # Exact path first
    if manifest_path.exists() and manifest_path.is_file():
        actual_path = manifest_path

    # Checkpoint stores decompressed JSONL
    elif source_file.endswith(".gz"):
        actual_name = source_file[:-3]
        actual_path = DEDUP_ROOT / actual_name

        if not actual_path.exists() or not actual_path.is_file():
            missing.append((source_file, actual_path))
            continue

    else:
        missing.append((source_file, manifest_path))
        continue

    # Non-empty check
    if actual_path.stat().st_size == 0:
        missing.append((source_file, actual_path))
        continue

    # Duplicate physical path check
    resolved_key = str(actual_path.resolve())

    if resolved_key in used_paths:
        raise AssertionError(
            f"Two manifest sources resolve to the same physical file: "
            f"{actual_path}"
        )

    used_paths.add(resolved_key)

    resolved.append((source_file, actual_path))

print("=" * 80)
print("RESOLUTION RESULT")
print("=" * 80)

print(f"Resolved : {len(resolved)} / {len(source_files)}")

if missing:
    print()
    print("MISSING SOURCES:")
    for source_file, attempted in missing:
        print(f"  ✗ {source_file}")
        print(f"      tried: {attempted}")

    raise AssertionError(
        f"{len(missing)} manifest source(s) could not be resolved."
    )

assert len(resolved) == len(source_files)

print("✓ Every manifest source resolves to an existing non-empty file")
print("✓ No two manifest sources resolve to the same file")

print()
print("=" * 80)
print("CHECKPOINT SOURCE RESOLUTION VALID")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — 10K REAL TRAINING CORPUS GENERATION SMOKE TEST
# MEMORY-SAFE / CHECKPOINT-SAFE
#
# Purpose:
#   Generate actual 4,096-token document-aware training chunks
#   from the Phase 4 checkpoint using the real train manifest.
#
# IMPORTANT:
#   - Does NOT modify /kaggle/input
#   - Does NOT modify the checkpoint dataset
#   - Does NOT modify tokenizer
#   - Does NOT modify dedup corpus
#   - Output goes only to /kaggle/working
# ============================================================

import os
import sys
import json
import time
import gzip
import hashlib
import gc
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, wait, FIRST_COMPLETED

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

DATASET_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

DEDUP_ROOT = DATASET_ROOT / "dedup"
TRAIN_MANIFEST = DATASET_ROOT / "manifests" / "train_manifest.jsonl"
TOKENIZER_PATH = DATASET_ROOT / "tokenizer" / "legal_bpe_32k.json"

OUTPUT_ROOT = Path(
    "/kaggle/working/phase4_training_chunks_smoke_test"
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

TOTAL_DOCS = 10_000

# Memory-safe configuration that already worked for us.
WORKERS = 4
BATCH_SIZE = 128
MAX_IN_FLIGHT = 8

CONTEXT_LENGTH = 4096

PROGRESS_EVERY = 1000

# ------------------------------------------------------------
# CHECKPOINT VALIDATION
# ------------------------------------------------------------

assert DATASET_ROOT.exists(), DATASET_ROOT
assert DEDUP_ROOT.exists(), DEDUP_ROOT
assert TRAIN_MANIFEST.exists(), TRAIN_MANIFEST
assert TOKENIZER_PATH.exists(), TOKENIZER_PATH

print("=" * 80)
print("PHASE 4 — 10K REAL TRAINING CORPUS GENERATION")
print("MEMORY-SAFE / CHECKPOINT-SAFE")
print("=" * 80)

print()
print(f"Checkpoint : {DATASET_ROOT}")
print(f"Tokenizer  : {TOKENIZER_PATH}")
print(f"Output     : {OUTPUT_ROOT}")
print()
print(f"Documents      : {TOTAL_DOCS:,}")
print(f"Workers        : {WORKERS}")
print(f"Batch size     : {BATCH_SIZE}")
print(f"Max in-flight  : {MAX_IN_FLIGHT}")
print(f"Context        : {CONTEXT_LENGTH:,}")

# ------------------------------------------------------------
# LOAD TOKENIZER
# ------------------------------------------------------------

print()
print("=" * 80)
print("LOADING TOKENIZER")
print("=" * 80)

from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

vocab_size = tokenizer.get_vocab_size()

print(f"Vocabulary : {vocab_size:,}")

# Resolve special token IDs safely.
vocab = tokenizer.get_vocab()

PAD_ID = vocab.get("[PAD]")
UNK_ID = vocab.get("[UNK]")
BOS_ID = vocab.get("[BOS]")
EOS_ID = vocab.get("[EOS]")

print(f"[PAD] ID : {PAD_ID}")
print(f"[UNK] ID : {UNK_ID}")
print(f"[BOS] ID : {BOS_ID}")
print(f"[EOS] ID : {EOS_ID}")

assert vocab_size == 32_000
assert UNK_ID is not None

# ------------------------------------------------------------
# LOAD FIRST 10K TRAINING RECORDS
# ------------------------------------------------------------

print()
print("=" * 80)
print("LOADING TRAINING MANIFEST")
print("=" * 80)

records = []

with TRAIN_MANIFEST.open("r", encoding="utf-8") as f:
    for line in f:
        if len(records) >= TOTAL_DOCS:
            break

        line = line.strip()

        if not line:
            continue

        rec = json.loads(line)

        assert rec.get("split") == "train"
        assert rec.get("text_hash")
        assert rec.get("source_file")

        records.append(rec)

assert len(records) == TOTAL_DOCS, (
    f"Expected {TOTAL_DOCS:,} records, got {len(records):,}"
)

print(f"Training records loaded : {len(records):,}")

# ------------------------------------------------------------
# RESOLVE SOURCE PATHS
# ------------------------------------------------------------

print()
print("=" * 80)
print("RESOLVING CHECKPOINT SOURCES")
print("=" * 80)

source_cache = {}

def resolve_source(source_file):
    if source_file in source_cache:
        return source_cache[source_file]

    # Manifest may contain .jsonl.gz while checkpoint contains .jsonl
    direct = DEDUP_ROOT / source_file

    if direct.exists() and direct.is_file():
        path = direct

    elif source_file.endswith(".gz"):
        decompressed = DEDUP_ROOT / source_file[:-3]

        if not decompressed.exists():
            raise FileNotFoundError(
                f"Cannot resolve checkpoint source:\n"
                f"Manifest: {source_file}\n"
                f"Tried   : {direct}\n"
                f"Tried   : {decompressed}"
            )

        path = decompressed

    else:
        raise FileNotFoundError(
            f"Missing checkpoint source: {direct}"
        )

    assert path.stat().st_size > 0

    source_cache[source_file] = path

    return path


for source_file in sorted(
    {r["source_file"] for r in records}
):
    path = resolve_source(source_file)

    print(f"✓ {source_file}")
    print(f"    → {path}")

print()
print(f"Sources used by 10K sample : {len(source_cache)}")

# ------------------------------------------------------------
# GROUP MANIFEST RECORDS BY SOURCE
# ------------------------------------------------------------

records_by_source = defaultdict(list)

for rec in records:
    records_by_source[rec["source_file"]].append(rec)

# ------------------------------------------------------------
# IMPORTANT:
# Worker receives only a batch of records + source path.
# It does NOT receive the entire corpus.
# ------------------------------------------------------------

def read_jsonl_document(source_path, target_hash):
    """
    Locate one document by text_hash.

    This intentionally streams the source file rather than
    loading the whole source into RAM.
    """

    with open(source_path, "r", encoding="utf-8") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue

            if rec.get("text_hash") == target_hash:
                return rec

    return None


def tokenize_batch(args):
    """
    Worker-side processing.

    Each worker loads its own tokenizer instance once per task
    invocation. The batch itself is bounded to BATCH_SIZE.
    """

    source_path, batch = args

    local_tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

    output = []

    for manifest_rec in batch:

        target_hash = manifest_rec["text_hash"]

        doc = read_jsonl_document(
            source_path,
            target_hash
        )

        if doc is None:
            raise RuntimeError(
                f"Document hash not found:\n{target_hash}\n"
                f"Source: {source_path}"
            )

        text = doc.get("text", "")

        if not isinstance(text, str):
            raise RuntimeError(
                f"Document text is not a string: {target_hash}"
            )

        if not text.strip():
            raise RuntimeError(
                f"Empty document: {target_hash}"
            )

        encoded = local_tokenizer.encode(text)

        token_ids = encoded.ids

        unk_count = sum(
            1 for x in token_ids
            if x == UNK_ID
        )

        # ----------------------------------------------------
        # Document-aware chunking
        # ----------------------------------------------------

        chunks = []

        for start in range(
            0,
            len(token_ids),
            CONTEXT_LENGTH
        ):

            chunk = token_ids[
                start:start + CONTEXT_LENGTH
            ]

            if not chunk:
                continue

            chunks.append({
                "text_hash": target_hash,
                "source_file": manifest_rec["source_file"],
                "chunk_index": len(chunks),
                "token_count": len(chunk),
                "tokens": chunk,
                "unk_tokens": (
                    unk_count
                    if len(chunks) == 0
                    else 0
                ),
            })

        output.append({
            "text_hash": target_hash,
            "characters": len(text),
            "tokens": len(token_ids),
            "unk_tokens": unk_count,
            "chunks": chunks,
        })

        del doc
        del text
        del token_ids

    return output


# ------------------------------------------------------------
# CREATE BATCHES
# ------------------------------------------------------------

batches = []

for source_file, source_records in records_by_source.items():

    source_path = resolve_source(source_file)

    for i in range(
        0,
        len(source_records),
        BATCH_SIZE
    ):

        batch = source_records[
            i:i + BATCH_SIZE
        ]

        batches.append(
            (
                source_path,
                batch
            )
        )

print()
print("=" * 80)
print("GENERATION PLAN")
print("=" * 80)

print(f"Documents : {len(records):,}")
print(f"Batches   : {len(batches):,}")
print(f"Workers   : {WORKERS}")
print(f"In-flight : {MAX_IN_FLIGHT}")

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

run_dir = OUTPUT_ROOT / "smoke_10k"

if run_dir.exists():
    import shutil
    shutil.rmtree(run_dir)

run_dir.mkdir(parents=True)

chunk_output = (
    run_dir / "training_chunks.jsonl"
)

stats_output = (
    run_dir / "generation_stats.json"
)

# ------------------------------------------------------------
# GENERATION
# ------------------------------------------------------------

print()
print("=" * 80)
print("STARTING 10K REAL GENERATION")
print("=" * 80)

start_time = time.time()

documents_done = 0
total_chars = 0
total_tokens = 0
total_chunks = 0
total_unk = 0

last_progress = 0

# Track manifest hashes to verify output coverage.
expected_hashes = {
    r["text_hash"]
    for r in records
}

output_hashes = set()

# ------------------------------------------------------------
# Bounded executor:
# NEVER submit the entire job at once.
# ------------------------------------------------------------

with ProcessPoolExecutor(
    max_workers=WORKERS
) as executor:

    pending = set()
    batch_iter = iter(batches)

    # Fill bounded queue.
    while len(pending) < MAX_IN_FLIGHT:

        try:
            batch = next(batch_iter)
        except StopIteration:
            break

        pending.add(
            executor.submit(
                tokenize_batch,
                batch
            )
        )

    with chunk_output.open(
        "w",
        encoding="utf-8"
    ) as out:

        while pending:

            done, pending = wait(
                pending,
                return_when=FIRST_COMPLETED
            )

            for future in done:

                results = future.result()

                for result in results:

                    doc_hash = result["text_hash"]

                    assert doc_hash in expected_hashes

                    # A document should be emitted exactly once.
                    assert doc_hash not in output_hashes, (
                        f"Duplicate output document: {doc_hash}"
                    )

                    output_hashes.add(doc_hash)

                    documents_done += 1
                    total_chars += result["characters"]
                    total_tokens += result["tokens"]
                    total_unk += result["unk_tokens"]

                    for chunk in result["chunks"]:

                        total_chunks += 1

                        out.write(
                            json.dumps(
                                chunk,
                                ensure_ascii=False,
                                separators=(",", ":")
                            )
                            + "\n"
                        )

                # Maintain bounded queue.
                while len(pending) < MAX_IN_FLIGHT:

                    try:
                        batch = next(batch_iter)
                    except StopIteration:
                        break

                    pending.add(
                        executor.submit(
                            tokenize_batch,
                            batch
                        )
                    )

                # ------------------------------------------------
                # Progress
                # ------------------------------------------------

                if (
                    documents_done // PROGRESS_EVERY
                    > last_progress
                    or documents_done == TOTAL_DOCS
                ):

                    last_progress = (
                        documents_done // PROGRESS_EVERY
                    )

                    elapsed = time.time() - start_time

                    rate = (
                        documents_done / elapsed
                        if elapsed > 0
                        else 0
                    )

                    remaining = (
                        TOTAL_DOCS - documents_done
                    )

                    eta = (
                        remaining / rate
                        if rate > 0
                        else 0
                    )

                    print()
                    print("-" * 80)
                    print(
                        f"[PROGRESS] "
                        f"{documents_done:,} / {TOTAL_DOCS:,} "
                        f"({documents_done / TOTAL_DOCS * 100:6.2f}%)"
                    )
                    print(f"Workers        : {WORKERS}")
                    print(f"Batch size     : {BATCH_SIZE}")
                    print(f"Max in-flight  : {MAX_IN_FLIGHT}")
                    print(
                        f"Documents/sec  : {rate:.1f}"
                    )
                    print(
                        f"Characters     : {total_chars:,}"
                    )
                    print(
                        f"Tokens         : {total_tokens:,}"
                    )
                    print(
                        f"Chunks         : {total_chunks:,}"
                    )
                    print(
                        f"[UNK] tokens   : {total_unk:,}"
                    )
                    print(
                        f"Elapsed        : {elapsed / 60:.2f} min"
                    )
                    print(
                        f"ETA            : {eta / 60:.2f} min"
                    )
                    print("-" * 80)

                    out.flush()

                    gc.collect()

# ------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------

elapsed = time.time() - start_time

print()
print("=" * 80)
print("10K GENERATION COMPLETE")
print("=" * 80)

print(f"Documents      : {documents_done:,}")
print(f"Characters     : {total_chars:,}")
print(f"Tokens         : {total_tokens:,}")
print(f"Training chunks: {total_chunks:,}")
print(f"[UNK] tokens   : {total_unk:,}")
print(f"Elapsed        : {elapsed / 60:.2f} min")
print(
    f"Rate           : "
    f"{documents_done / elapsed:.1f} docs/sec"
)

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

assert documents_done == TOTAL_DOCS
assert output_hashes == expected_hashes
assert total_chunks > 0

# Every chunk must be <= context length.
print()
print("Validating generated chunks...")

chunk_records = 0
bad_chunks = 0

with chunk_output.open(
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        if not line.strip():
            continue

        rec = json.loads(line)

        chunk_records += 1

        if rec["token_count"] > CONTEXT_LENGTH:
            bad_chunks += 1

assert bad_chunks == 0

# ------------------------------------------------------------
# Save stats
# ------------------------------------------------------------

stats = {
    "phase": "phase4",
    "mode": "10k_real_generation_smoke_test",
    "documents": documents_done,
    "characters": total_chars,
    "tokens": total_tokens,
    "training_chunks": total_chunks,
    "unk_tokens": total_unk,
    "context_length": CONTEXT_LENGTH,
    "workers": WORKERS,
    "batch_size": BATCH_SIZE,
    "max_in_flight": MAX_IN_FLIGHT,
    "elapsed_seconds": elapsed,
    "documents_per_second": documents_done / elapsed,
    "chunk_output": str(chunk_output),
}

with stats_output.open(
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        stats,
        f,
        indent=2
    )

print()
print("=" * 80)
print("SMOKE TEST VALIDATION")
print("=" * 80)

print("✓ Exactly 10,000 manifest documents processed")
print("✓ Every output document hash belongs to manifest")
print("✓ Every manifest document emitted exactly once")
print("✓ All generated chunks <= 4,096 tokens")
print("✓ Output corpus contains actual token IDs")
print("✓ Checkpoint dataset untouched")
print("✓ Dedup corpus untouched")
print("✓ Tokenizer untouched")

print()
print(f"Output corpus : {chunk_output}")
print(f"Statistics    : {stats_output}")

print()
print("=" * 80)
print("PHASE 4 — 10K REAL GENERATION SMOKE TEST VALID")
print("=" * 80)

In [ ]:
# =============================================================================
# PHASE 4 — RECOVER PREVIOUS TRAINING/CHUNKING ARCHITECTURE
# READ-ONLY DIAGNOSTIC — DOES NOT MODIFY ANY DATA
# =============================================================================

from pathlib import Path
import json
import os
import re

print("=" * 80)
print("PHASE 4 — PREVIOUS ARCHITECTURE RECOVERY")
print("=" * 80)

# ---------------------------------------------------------------------------
# 1. Checkpoint dataset
# ---------------------------------------------------------------------------

CHECKPOINT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

print("\n" + "=" * 80)
print("CHECKPOINT METADATA / CONFIGS")
print("=" * 80)

metadata_files = [
    "phase4_initialization.json",
    "phase4_pre_tokenization_checkpoint.json",
    "phase4_dedup_audit.json",
    "phase4_duplicate_audit.json",
    "phase4_dedup_split_policy.json",
    "phase4_text_quality_audit.json",
    "streaming_config.json",
    "training_example_config.json",
]

for name in metadata_files:
    p = CHECKPOINT / name

    if not p.exists():
        print(f"\n✗ MISSING: {name}")
        continue

    print(f"\n{'-' * 80}")
    print(f"FILE: {name}")
    print("-" * 80)

    try:
        obj = json.loads(p.read_text())

        # Pretty-print complete small configs.
        print(json.dumps(obj, indent=2, ensure_ascii=False)[:20000])

    except Exception as e:
        print(f"Could not parse JSON: {e}")


# ---------------------------------------------------------------------------
# 2. Training chunk structure
# ---------------------------------------------------------------------------

print("\n" + "=" * 80)
print("EXISTING TRAINING CHUNK STRUCTURE")
print("=" * 80)

chunks_root = CHECKPOINT / "training_chunks"

if chunks_root.exists():

    for p in sorted(chunks_root.rglob("*")):
        if p.is_file():

            size_gb = p.stat().st_size / (1024 ** 3)

            print(f"\nFILE : {p.relative_to(CHECKPOINT)}")
            print(f"SIZE : {size_gb:.4f} GB")

            # Inspect first few records WITHOUT loading entire file.
            try:
                with p.open("r", encoding="utf-8") as f:

                    for i in range(3):
                        line = f.readline()

                        if not line:
                            break

                        print(f"\nRECORD {i + 1}:")
                        print(line[:5000])

            except Exception as e:
                print(f"Read error: {e}")

else:
    print("No training_chunks directory found.")


# ---------------------------------------------------------------------------
# 3. Search Kaggle working directory for previous implementation code
# ---------------------------------------------------------------------------

print("\n" + "=" * 80)
print("SEARCHING /kaggle/working FOR PREVIOUS IMPLEMENTATION")
print("=" * 80)

WORKING = Path("/kaggle/working")

interesting_extensions = {
    ".py",
    ".ipynb",
    ".txt",
    ".json",
    ".md",
}

keywords = [
    "tokenize_batch",
    "training_chunks",
    "document_aware",
    "document_boundary",
    "max_in_flight",
    "in_flight",
    "ProcessPoolExecutor",
    "multiprocessing",
    "Pool(",
    "imap",
    "chunksize",
    "4096",
    "text_hash",
    "run_20260928",
    "acts__documents",
]

matches = []

for p in WORKING.rglob("*"):

    if not p.is_file():
        continue

    if p.suffix.lower() not in interesting_extensions:
        continue

    # Skip obviously huge generated data files.
    try:
        if p.stat().st_size > 50 * 1024 * 1024:
            continue
    except Exception:
        continue

    try:
        text = p.read_text(
            encoding="utf-8",
            errors="ignore"
        )
    except Exception:
        continue

    found = [kw for kw in keywords if kw in text]

    if found:
        matches.append((p, found))

if not matches:
    print("\nNo previous source-code files found in /kaggle/working.")

else:
    print(f"\nPotential implementation files found: {len(matches)}")

    for p, found in matches:
        print("\n" + "-" * 80)
        print(f"FILE    : {p}")
        print(f"MATCHES : {', '.join(found)}")
        print("-" * 80)


# ---------------------------------------------------------------------------
# 4. Search notebook files specifically and extract relevant cells
# ---------------------------------------------------------------------------

print("\n" + "=" * 80)
print("NOTEBOOK IMPLEMENTATION SEARCH")
print("=" * 80)

notebooks = list(WORKING.rglob("*.ipynb"))

print(f"Notebooks found: {len(notebooks)}")

for nb_path in notebooks:

    try:
        nb = json.loads(
            nb_path.read_text(
                encoding="utf-8",
                errors="ignore"
            )
        )
    except Exception:
        continue

    relevant_cells = []

    for idx, cell in enumerate(nb.get("cells", [])):

        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))

        found = [kw for kw in keywords if kw in source]

        if found:
            relevant_cells.append(
                (idx, found, source)
            )

    if relevant_cells:

        print("\n" + "=" * 80)
        print(f"NOTEBOOK: {nb_path}")
        print("=" * 80)

        for idx, found, source in relevant_cells:

            print("\n" + "-" * 80)
            print(f"CELL {idx}")
            print(f"MATCHES: {', '.join(found)}")
            print("-" * 80)

            # Give enough source to reconstruct the architecture.
            print(source[:15000])


# ---------------------------------------------------------------------------
# 5. Final summary
# ---------------------------------------------------------------------------

print("\n" + "=" * 80)
print("ARCHITECTURE RECOVERY COMPLETE")
print("=" * 80)

print("""
This cell was READ-ONLY.

It did NOT:
  - tokenize documents
  - regenerate chunks
  - modify the checkpoint
  - modify the deduplicated corpus
  - modify manifests
  - modify the tokenizer

Use the output above to identify the exact previous implementation.
""")

In [ ]:
# =============================================================================
# PHASE 4 — STREAMING ARCHITECTURE VERIFICATION
# READ-ONLY
#
# This cell does NOT:
#   - tokenize the corpus
#   - generate training chunks
#   - modify any checkpoint files
#   - modify manifests
#   - modify the tokenizer
# =============================================================================

from pathlib import Path
import json

CHECKPOINT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

print("=" * 80)
print("PHASE 4 — STREAMING ARCHITECTURE VERIFICATION")
print("=" * 80)
print(f"Checkpoint : {CHECKPOINT}")

# -------------------------------------------------------------------------
# Load configuration artifacts
# -------------------------------------------------------------------------

def load_json(name):
    path = CHECKPOINT / name
    assert path.exists(), f"Missing: {path}"
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

pre_tok = load_json("phase4_pre_tokenization_checkpoint.json")
streaming = load_json("streaming_config.json")
training_cfg = load_json("training_example_config.json")

# -------------------------------------------------------------------------
# Display architecture
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("1. CORPUS")
print("=" * 80)

print("Domains:")
for d in pre_tok["domains"]:
    print(f"  - {d}")

print(f"\nTotal unique records : {sum(pre_tok['manifest_counts'].values()):,}")
print(f"Train                : {pre_tok['manifest_counts']['train']:,}")
print(f"Validation           : {pre_tok['manifest_counts']['validation']:,}")
print(f"Test                 : {pre_tok['manifest_counts']['test']:,}")

print("\n" + "=" * 80)
print("2. DEDUPLICATION")
print("=" * 80)

print("Method : ", pre_tok["deduplication"]["method"])
print("Scope  : ", pre_tok["deduplication"]["scope"])

print("\n" + "=" * 80)
print("3. SPLIT")
print("=" * 80)

print("Train fraction      :", pre_tok["split"]["train"])
print("Validation fraction :", pre_tok["split"]["validation"])
print("Test fraction       :", pre_tok["split"]["test"])
print("Seed                :", pre_tok["split"]["seed"])

print("\n" + "=" * 80)
print("4. TOKENIZATION")
print("=" * 80)

print("Tokenizer path:")
print(" ", streaming["tokenizer"])

print("Vocabulary size :", streaming["vocab_size"])
print("Materialize full tokenized corpus :",
      streaming["materialize_tokenized_corpus"])
print("Dataset mode :", streaming["dataset_mode"])

print("\n" + "=" * 80)
print("5. TRAINING REPRESENTATION")
print("=" * 80)

print("Context length    :", training_cfg["context_length"])
print("Packing            :", training_cfg["packing"])
print("Document boundary :", training_cfg["document_boundary"])
print("Special tokens     :", training_cfg["special_tokens"])

print("\n" + "=" * 80)
print("6. ARCHITECTURE")
print("=" * 80)

print("""
TRAIN MANIFEST
      |
      v
text_hash / source reference
      |
      v
resolve source document
      |
      v
tokenize on demand
      |
      v
[BOS] document [EOS]
      |
      v
document-aware 4096-token packing
      |
      v
stream training examples
      |
      v
MODEL
""")

print("=" * 80)
print("ARCHITECTURE VERIFICATION COMPLETE")
print("=" * 80)

In [ ]:
# =============================================================================
# PHASE 4 — MANIFEST VERIFICATION
# READ-ONLY
# =============================================================================

from pathlib import Path
import json
from collections import Counter

CHECKPOINT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

MANIFEST_DIR = CHECKPOINT / "manifests"

MANIFESTS = {
    "train": (
        MANIFEST_DIR / "train_manifest.jsonl",
        3_020_655,
    ),
    "validation": (
        MANIFEST_DIR / "validation_manifest.jsonl",
        168_665,
    ),
    "test": (
        MANIFEST_DIR / "test_manifest.jsonl",
        167_274,
    ),
}

print("=" * 80)
print("PHASE 4 — MANIFEST VERIFICATION")
print("=" * 80)

def inspect_manifest(name, path, expected_count):

    print("\n" + "-" * 80)
    print(f"{name.upper()} MANIFEST")
    print("-" * 80)

    assert path.exists(), f"Missing manifest: {path}"

    count = 0
    hashes = set()
    malformed = 0
    missing_hash = 0
    sources = Counter()

    first_record = None

    with open(path, "r", encoding="utf-8") as f:

        for line_no, line in enumerate(f, 1):

            if not line.strip():
                continue

            try:
                rec = json.loads(line)
            except Exception:
                malformed += 1
                continue

            count += 1

            if first_record is None:
                first_record = rec

            h = rec.get("text_hash")

            if not isinstance(h, str) or not h:
                missing_hash += 1
            else:
                hashes.add(h)

            source = rec.get("source_file")

            if source:
                sources[source] += 1

    print("Manifest path      :", path)
    print("Records            :", f"{count:,}")
    print("Unique text hashes :", f"{len(hashes):,}")
    print("Malformed records  :", malformed)
    print("Missing hashes     :", missing_hash)
    print("Source files       :", len(sources))

    print("\nSource distribution:")

    for source, n in sources.most_common():
        print(f"  {n:>10,}  {source}")

    print("\nFirst record:")

    if first_record:
        for key, value in first_record.items():

            if key == "text":
                print(f"  {key}: <text omitted>")
            else:
                print(f"  {key}: {value}")

    # ---------------------------------------------------------------------
    # Integrity assertions
    # ---------------------------------------------------------------------

    assert count == expected_count, (
        f"{name}: expected {expected_count:,}, got {count:,}"
    )

    assert malformed == 0, (
        f"{name}: malformed records found"
    )

    assert missing_hash == 0, (
        f"{name}: missing/malformed text_hash found"
    )

    assert len(hashes) == count, (
        f"{name}: duplicate text_hash values found"
    )

    print("\n✓ Record count correct")
    print("✓ All hashes present")
    print("✓ Hashes unique")
    print("✓ No malformed records")

    return {
        "count": count,
        "unique_hashes": len(hashes),
        "sources": sources,
    }


results = {}

for name, (path, expected) in MANIFESTS.items():

    results[name] = inspect_manifest(
        name,
        path,
        expected,
    )


print("\n" + "=" * 80)
print("MANIFEST VERIFICATION COMPLETE")
print("=" * 80)

print("\nFinal counts:")

for name, result in results.items():
    print(
        f"{name:12s} : "
        f"{result['count']:,} records | "
        f"{result['unique_hashes']:,} unique hashes"
    )

print("\n✓ All Phase 4 manifests verified")
print("✓ Read-only")
print("✓ No tokenization performed")
print("✓ No training chunks generated")

In [ ]:
# =============================================================================
# PHASE 4 — STREAMING TRAINING READINESS CHECK
# READ-ONLY
# =============================================================================

from pathlib import Path
import json

ROOT = Path("/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep")

print("=" * 80)
print("PHASE 4 — STREAMING TRAINING READINESS CHECK")
print("=" * 80)
print(f"Checkpoint : {ROOT}")

required = {
    "train_manifest": ROOT / "manifests/train_manifest.jsonl",
    "validation_manifest": ROOT / "manifests/validation_manifest.jsonl",
    "test_manifest": ROOT / "manifests/test_manifest.jsonl",
    "tokenizer": ROOT / "tokenizer/legal_bpe_32k.json",
    "tokenizer_config": ROOT / "tokenizer/tokenizer_config.json",
    "training_example_config": ROOT / "training_example_config.json",
    "streaming_config": ROOT / "streaming_config.json",
}

print("\n" + "=" * 80)
print("1. REQUIRED ARTIFACTS")
print("=" * 80)

for name, path in required.items():
    ok = path.exists() and path.is_file() and path.stat().st_size > 0
    print(f"{'✓' if ok else '✗'} {name:<25} {path}")

    assert ok, f"Missing or empty required artifact: {path}"

print("\n✓ All streaming-training artifacts are present")


# ---------------------------------------------------------------------------
# Read configuration
# ---------------------------------------------------------------------------

with open(required["streaming_config"], "r", encoding="utf-8") as f:
    streaming_cfg = json.load(f)

with open(required["training_example_config"], "r", encoding="utf-8") as f:
    example_cfg = json.load(f)

print("\n" + "=" * 80)
print("2. STREAMING CONFIGURATION")
print("=" * 80)

print(f"Dataset mode                 : {streaming_cfg['dataset_mode']}")
print(f"Materialize tokenized corpus : {streaming_cfg['materialize_tokenized_corpus']}")
print(f"Train records                : {streaming_cfg['train_records']:,}")
print(f"Validation records           : {streaming_cfg['validation_records']:,}")
print(f"Test records                 : {streaming_cfg['test_records']:,}")

assert streaming_cfg["dataset_mode"] == "streaming"
assert streaming_cfg["materialize_tokenized_corpus"] is False


print("\n" + "=" * 80)
print("3. TRAINING EXAMPLE CONFIG")
print("=" * 80)

print(f"Context length    : {example_cfg['context_length']:,}")
print(f"Packing           : {example_cfg['packing']}")
print(f"Document boundary : {example_cfg['document_boundary']}")
print(f"Special tokens    : {example_cfg['special_tokens']}")

assert example_cfg["context_length"] == 4096
assert example_cfg["packing"] == "document_aware"
assert example_cfg["document_boundary"] is True


print("\n" + "=" * 80)
print("4. FINAL RESULT")
print("=" * 80)

print("✓ Streaming mode confirmed")
print("✓ Full tokenized corpus is NOT required")
print("✓ Tokenization is intended to happen on demand")
print("✓ Context length = 4,096")
print("✓ Document-aware packing = enabled")
print("✓ Document boundaries = preserved")
print()
print("READY FOR STREAMING DATASET IMPLEMENTATION")

In [ ]:
# =============================================================================
# PHASE 4 — STREAMING DATASET: SOURCE RESOLUTION + TOKENIZATION TEST
# READ-ONLY
# =============================================================================

from pathlib import Path
import json
import hashlib
from tokenizers import Tokenizer

ROOT = Path("/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep")

TRAIN_MANIFEST = ROOT / "manifests/train_manifest.jsonl"
TOKENIZER_PATH = ROOT / "tokenizer/legal_bpe_32k.json"
DEDUP_ROOT = ROOT / "dedup"

CONTEXT_LENGTH = 4096

print("=" * 80)
print("PHASE 4 — STREAMING DATASET SOURCE + TOKENIZATION TEST")
print("=" * 80)

# -----------------------------------------------------------------------------
# Load tokenizer
# -----------------------------------------------------------------------------

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

vocab_size = tokenizer.get_vocab_size()

print(f"Tokenizer vocabulary : {vocab_size:,}")
print(f"Context length       : {CONTEXT_LENGTH:,}")


# -----------------------------------------------------------------------------
# Read first manifest records
# -----------------------------------------------------------------------------

records = []

with open(TRAIN_MANIFEST, "r", encoding="utf-8") as f:
    for _ in range(10):
        line = f.readline()

        if not line:
            break

        records.append(json.loads(line))

print(f"\nManifest records read : {len(records)}")

assert len(records) == 10


# -----------------------------------------------------------------------------
# Inspect manifest schema
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("MANIFEST SCHEMA")
print("=" * 80)

for key, value in records[0].items():
    if isinstance(value, str):
        display_value = value[:300]
    else:
        display_value = value

    print(f"{key:<20}: {display_value}")


# -----------------------------------------------------------------------------
# Locate source file
# -----------------------------------------------------------------------------

def resolve_source(source_file):
    """
    Resolve the source_file recorded by the manifest against the
    checkpoint dedup corpus.
    """

    candidates = [
        DEDUP_ROOT / source_file,
        DEDUP_ROOT / Path(source_file).with_suffix(""),
    ]

    for candidate in candidates:
        if candidate.exists() and candidate.is_file():
            return candidate

    # Recursive fallback by filename.
    basename = Path(source_file).name

    matches = list(DEDUP_ROOT.rglob(basename))

    matches = [
        p for p in matches
        if p.is_file() and p.stat().st_size > 0
    ]

    if len(matches) == 1:
        return matches[0]

    raise FileNotFoundError(
        f"Could not uniquely resolve source_file={source_file!r}"
    )


# -----------------------------------------------------------------------------
# JSONL source reader
# -----------------------------------------------------------------------------

_source_cache = {}


def get_source_record(source_path, target_hash):
    """
    Locate a document with the requested text_hash.

    This test reader scans the source file directly.
    The production implementation will use an efficient indexed lookup
    rather than repeatedly scanning a large source file.
    """

    cache_key = str(source_path)

    if cache_key not in _source_cache:
        _source_cache[cache_key] = {}

    cache = _source_cache[cache_key]

    if target_hash in cache:
        return cache[target_hash]

    with open(source_path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)

            text = obj.get("text")
            if not isinstance(text, str):
                continue

            h = hashlib.sha256(text.encode("utf-8")).hexdigest()

            if h == target_hash:
                cache[target_hash] = obj
                return obj

    return None


# -----------------------------------------------------------------------------
# Test first 10 documents
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("SOURCE RESOLUTION + TOKENIZATION")
print("=" * 80)

for i, record in enumerate(records, 1):

    text_hash = record.get("text_hash")
    source_file = record.get("source_file")

    assert isinstance(text_hash, str)
    assert len(text_hash) == 64

    assert isinstance(source_file, str)

    source_path = resolve_source(source_file)

    obj = get_source_record(
        source_path,
        text_hash,
    )

    assert obj is not None, (
        f"Could not find hash {text_hash} "
        f"in source {source_path}"
    )

    text = obj["text"]

    # Verify the recovered document itself.
    recovered_hash = hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

    assert recovered_hash == text_hash

    # Tokenize.
    encoding = tokenizer.encode(text)

    token_ids = encoding.ids

    # Add document boundary tokens.
    bos_id = tokenizer.token_to_id("[BOS]")
    eos_id = tokenizer.token_to_id("[EOS]")

    assert bos_id is not None
    assert eos_id is not None

    input_ids = [
        bos_id,
        *token_ids,
        eos_id,
    ]

    print(
        f"[{i:02d}/10] "
        f"source={source_path.relative_to(ROOT)} | "
        f"chars={len(text):,} | "
        f"tokens={len(token_ids):,} | "
        f"with_boundaries={len(input_ids):,}"
    )


print("\n" + "=" * 80)
print("RESULT")
print("=" * 80)

print("✓ Manifest records can be resolved")
print("✓ Source documents can be recovered")
print("✓ SHA-256 text hashes match manifest hashes")
print("✓ Tokenization works")
print("✓ [BOS] and [EOS] are available")
print("✓ Document boundaries can be represented")
print()
print("NEXT STEP:")
print("Build the production streaming/indexed dataset reader.")

In [ ]:
# =============================================================================
# PHASE 4 — BUILD STREAMING SOURCE INDEX
# NO TOKENIZATION
# NO TOKENIZED CORPUS
# READS CHECKPOINT, WRITES ONLY INDEX
# =============================================================================

from pathlib import Path
import json
import hashlib
import time

ROOT = Path("/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep")
MANIFEST = ROOT / "manifests/train_manifest.jsonl"
DEDUP_ROOT = ROOT / "dedup"

OUT = Path("/kaggle/working/phase4_streaming_index")
OUT.mkdir(parents=True, exist_ok=True)

INDEX_FILE = OUT / "train_source_index.jsonl"

print("=" * 80)
print("PHASE 4 — STREAMING SOURCE INDEX BUILD")
print("=" * 80)
print(f"Checkpoint : {ROOT}")
print(f"Manifest   : {MANIFEST}")
print(f"Index      : {INDEX_FILE}")

# -----------------------------------------------------------------------------
# 1. Discover source files from the manifest
# -----------------------------------------------------------------------------

source_names = set()

with open(MANIFEST, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        source_names.add(rec["source_file"])

print("\nManifest source files :", len(source_names))

# -----------------------------------------------------------------------------
# 2. Resolve source paths
# -----------------------------------------------------------------------------

def resolve_source(source_file):
    candidates = [
        DEDUP_ROOT / source_file,
        DEDUP_ROOT / Path(source_file).with_suffix(""),
    ]

    for p in candidates:
        if p.exists() and p.is_file() and p.stat().st_size > 0:
            return p

    basename = Path(source_file).name

    matches = [
        p for p in DEDUP_ROOT.rglob(basename)
        if p.is_file() and p.stat().st_size > 0
    ]

    if len(matches) == 1:
        return matches[0]

    raise FileNotFoundError(
        f"Could not uniquely resolve {source_file}"
    )


source_paths = {}

for source_name in sorted(source_names):
    source_paths[source_name] = resolve_source(source_name)

print("\nResolved sources:")

for name, path in source_paths.items():
    size_gb = path.stat().st_size / (1024 ** 3)

    print(
        f"  ✓ {name}"
        f" -> {path.relative_to(ROOT)}"
        f" | {size_gb:.3f} GB"
    )

# -----------------------------------------------------------------------------
# 3. Build byte-offset index
#
# Each line is indexed by:
#
#   text_hash
#   source_file
#   byte_offset
#   byte_length
#
# This lets the streaming dataset seek directly to the source JSONL record.
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("BUILDING BYTE-OFFSET INDEX")
print("=" * 80)

start = time.time()

total_records = 0
total_bytes = 0

with open(INDEX_FILE, "w", encoding="utf-8") as out:

    for source_name, source_path in source_paths.items():

        print(f"\nIndexing: {source_name}")
        print(f"File    : {source_path}")

        records = 0
        file_bytes = source_path.stat().st_size

        with open(
            source_path,
            "rb",
            buffering=1024 * 1024,
        ) as src:

            while True:

                offset = src.tell()
                line = src.readline()

                if not line:
                    break

                total_bytes += len(line)
                records += 1
                total_records += 1

                try:
                    obj = json.loads(line)
                except Exception as e:
                    raise RuntimeError(
                        f"Malformed JSON at byte offset "
                        f"{offset} in {source_path}"
                    ) from e

                text = obj.get("text")

                if not isinstance(text, str):
                    raise AssertionError(
                        f"Missing/non-string text at "
                        f"{source_path}:{offset}"
                    )

                text_hash = hashlib.sha256(
                    text.encode("utf-8")
                ).hexdigest()

                index_record = {
                    "text_hash": text_hash,
                    "source_file": source_name,
                    "byte_offset": offset,
                    "byte_length": len(line),
                }

                out.write(
                    json.dumps(
                        index_record,
                        ensure_ascii=False,
                        separators=(",", ":"),
                    )
                    + "\n"
                )

                if records % 100_000 == 0:
                    elapsed = time.time() - start
                    rate = records / elapsed if elapsed else 0

                    print(
                        f"  {records:,} records | "
                        f"{rate:,.0f} records/sec"
                    )

        print(f"  ✓ {records:,} source records indexed")

elapsed = time.time() - start

print("\n" + "=" * 80)
print("INDEX BUILD COMPLETE")
print("=" * 80)

print(f"Indexed records : {total_records:,}")
print(f"Source bytes    : {total_bytes:,}")
print(f"Elapsed         : {elapsed / 60:.2f} min")
print(f"Index size      : {INDEX_FILE.stat().st_size / (1024**3):.3f} GB")

print("\nIndex:")
print(INDEX_FILE)

print("\nIMPORTANT:")
print("✓ No tokenization performed")
print("✓ No tokenized corpus created")
print("✓ Original checkpoint untouched")
print("✓ Index contains byte offsets for direct source lookup")

In [ ]:
# ============================================================
# PHASE 4 — PRODUCTION STREAMING DATASET
# CELL 1: CONFIGURATION
# ============================================================

import os
import json
import hashlib
from pathlib import Path

CHECKPOINT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

INDEX_DIR = Path("/kaggle/working/phase4_streaming_index")
INDEX_PATH = INDEX_DIR / "train_source_index.jsonl"

TRAIN_MANIFEST = CHECKPOINT / "manifests/train_manifest.jsonl"
VAL_MANIFEST   = CHECKPOINT / "manifests/validation_manifest.jsonl"
TEST_MANIFEST  = CHECKPOINT / "manifests/test_manifest.jsonl"

TOKENIZER_PATH = CHECKPOINT / "tokenizer/legal_bpe_32k.json"

CONTEXT_LENGTH = 4096

SPECIAL_TOKENS = {
    "pad": "[PAD]",
    "unk": "[UNK]",
    "bos": "[BOS]",
    "eos": "[EOS]",
}

assert INDEX_PATH.exists(), INDEX_PATH
assert TRAIN_MANIFEST.exists(), TRAIN_MANIFEST
assert VAL_MANIFEST.exists(), VAL_MANIFEST
assert TEST_MANIFEST.exists(), TEST_MANIFEST
assert TOKENIZER_PATH.exists(), TOKENIZER_PATH

print("Configuration loaded")
print("Checkpoint :", CHECKPOINT)
print("Index      :", INDEX_PATH)
print("Tokenizer  :", TOKENIZER_PATH)
print("Context    :", CONTEXT_LENGTH)

In [ ]:
# ============================================================
# CELL 2: TOKENIZER
# ============================================================

from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

print("Tokenizer loaded")
print("Vocabulary size:", tokenizer.get_vocab_size())

vocab = tokenizer.get_vocab()

for token in SPECIAL_TOKENS.values():
    assert token in vocab, f"Missing special token: {token}"

PAD_ID = vocab[SPECIAL_TOKENS["pad"]]
UNK_ID = vocab[SPECIAL_TOKENS["unk"]]
BOS_ID = vocab[SPECIAL_TOKENS["bos"]]
EOS_ID = vocab[SPECIAL_TOKENS["eos"]]

print("PAD:", PAD_ID)
print("UNK:", UNK_ID)
print("BOS:", BOS_ID)
print("EOS:", EOS_ID)

In [ ]:
# ============================================================
# PHASE 4 — INSPECT STREAMING INDEX SCHEMA
# ============================================================

import json
from pathlib import Path

INDEX_PATH = Path(
    "/kaggle/working/phase4_streaming_index/train_source_index.jsonl"
)

print("=" * 80)
print("PHASE 4 — STREAMING INDEX SCHEMA INSPECTION")
print("=" * 80)
print(f"Index: {INDEX_PATH}")
print()

if not INDEX_PATH.exists():
    raise FileNotFoundError(f"Index not found: {INDEX_PATH}")

with INDEX_PATH.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if not line.strip():
            continue

        rec = json.loads(line)

        print("First non-empty record:")
        print(json.dumps(rec, indent=2, ensure_ascii=False))

        print()
        print("Top-level fields:")
        for key, value in rec.items():
            print(f"  {key:<25} {type(value).__name__}")

        break

print()
print("=" * 80)
print("DONE")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — PRODUCTION STREAMING LOOKUP
# ============================================================

import json
from pathlib import Path

INDEX_PATH = Path(
    "/kaggle/working/phase4_streaming_index/train_source_index.jsonl"
)

# text_hash -> source location
source_lookup = {}

with INDEX_PATH.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        if not line.strip():
            continue

        rec = json.loads(line)

        required = (
            "text_hash",
            "source_file",
            "byte_offset",
            "byte_length",
        )

        missing = [k for k in required if k not in rec]

        if missing:
            raise ValueError(
                f"Index record {i} missing fields: {missing}"
            )

        text_hash = rec["text_hash"]

        if text_hash in source_lookup:
            raise ValueError(
                f"Duplicate text_hash found: {text_hash}"
            )

        source_lookup[text_hash] = {
            "source_file": rec["source_file"],
            "byte_offset": rec["byte_offset"],
            "byte_length": rec["byte_length"],
        }

print("=" * 80)
print("PHASE 4 — PRODUCTION STREAMING LOOKUP")
print("=" * 80)
print(f"Index records : {len(source_lookup):,}")
print()

# Sanity check
sample_hash = next(iter(source_lookup))
sample = source_lookup[sample_hash]

print("Sample lookup:")
print(f"  text_hash   : {sample_hash}")
print(f"  source_file : {sample['source_file']}")
print(f"  byte_offset : {sample['byte_offset']:,}")
print(f"  byte_length : {sample['byte_length']:,}")

print()
print("✓ text_hash → source location lookup built")
print("✓ No domain field required")
print("✓ No line-number lookup required")

In [ ]:
# ============================================================
# PHASE 4 — DIRECT BYTE-OFFSET SOURCE RESOLUTION
# ============================================================

import json
from pathlib import Path

CHECKPOINT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

DEDUP_ROOT = CHECKPOINT / "dedup"


def resolve_source_file(source_file):
    """
    Convert manifest/index source_file to the actual
    deduplicated source file.
    """
    relative = Path(source_file)

    # acts/documents.jsonl.gz -> dedup/acts/documents.jsonl
    if relative.suffix == ".gz":
        relative = relative.with_suffix("")

    path = DEDUP_ROOT / relative

    if not path.exists():
        raise FileNotFoundError(
            f"Resolved source does not exist:\n{path}"
        )

    return path


def read_indexed_document(text_hash):
    """
    Resolve and read exactly one document using the
    byte offset and byte length stored in the index.
    """

    if text_hash not in source_lookup:
        raise KeyError(
            f"text_hash not found in streaming index: {text_hash}"
        )

    entry = source_lookup[text_hash]

    source_path = resolve_source_file(
        entry["source_file"]
    )

    with source_path.open("rb") as f:
        f.seek(entry["byte_offset"])

        raw = f.read(entry["byte_length"])

    if len(raw) != entry["byte_length"]:
        raise IOError(
            f"Expected {entry['byte_length']} bytes, "
            f"but read {len(raw)} bytes"
        )

    record = json.loads(raw.decode("utf-8"))

    return record, source_path


# Test with the first indexed document
test_hash = next(iter(source_lookup))

record, source_path = read_indexed_document(test_hash)

print("=" * 80)
print("PHASE 4 — DIRECT SOURCE LOOKUP TEST")
print("=" * 80)
print(f"text_hash   : {test_hash}")
print(f"source      : {source_path}")
print(f"record keys : {list(record.keys())}")
print(f"raw chars   : {len(record.get('text', '')):,}")
print()
print("✓ Byte-offset lookup succeeded")
print("✓ Source document recovered directly")

In [ ]:
# ============================================================
# PHASE 4 — DIRECT BYTE-OFFSET SOURCE RESOLUTION
# ============================================================

import json
from pathlib import Path

CHECKPOINT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

DEDUP_ROOT = CHECKPOINT / "dedup"


def resolve_source_file(source_file):
    """
    Convert manifest/index source_file to the actual
    deduplicated source file.
    """
    relative = Path(source_file)

    # acts/documents.jsonl.gz -> dedup/acts/documents.jsonl
    if relative.suffix == ".gz":
        relative = relative.with_suffix("")

    path = DEDUP_ROOT / relative

    if not path.exists():
        raise FileNotFoundError(
            f"Resolved source does not exist:\n{path}"
        )

    return path


def read_indexed_document(text_hash):
    """
    Resolve and read exactly one document using the
    byte offset and byte length stored in the index.
    """

    if text_hash not in source_lookup:
        raise KeyError(
            f"text_hash not found in streaming index: {text_hash}"
        )

    entry = source_lookup[text_hash]

    source_path = resolve_source_file(
        entry["source_file"]
    )

    with source_path.open("rb") as f:
        f.seek(entry["byte_offset"])

        raw = f.read(entry["byte_length"])

    if len(raw) != entry["byte_length"]:
        raise IOError(
            f"Expected {entry['byte_length']} bytes, "
            f"but read {len(raw)} bytes"
        )

    record = json.loads(raw.decode("utf-8"))

    return record, source_path


# Test with the first indexed document
test_hash = next(iter(source_lookup))

record, source_path = read_indexed_document(test_hash)

print("=" * 80)
print("PHASE 4 — DIRECT SOURCE LOOKUP TEST")
print("=" * 80)
print(f"text_hash   : {test_hash}")
print(f"source      : {source_path}")
print(f"record keys : {list(record.keys())}")
print(f"raw chars   : {len(record.get('text', '')):,}")
print()
print("✓ Byte-offset lookup succeeded")
print("✓ Source document recovered directly")

In [ ]:
# ============================================================
# PHASE 4 — INDEX → SOURCE → HASH → TOKENIZATION TEST
# ============================================================

import hashlib
import json
from pathlib import Path

TOKENIZER_PATH = (
    CHECKPOINT
    / "tokenizer"
    / "legal_bpe_32k.json"
)

# Load tokenizer
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

test_hash = next(iter(source_lookup))

record, source_path = read_indexed_document(test_hash)

text = record.get("text")

if text is None:
    raise KeyError(
        f"Recovered record has no 'text' field. "
        f"Keys: {list(record.keys())}"
    )

# Recompute SHA-256
computed_hash = hashlib.sha256(
    text.encode("utf-8")
).hexdigest()

# Tokenize
token_ids = tokenizer.encode(text).ids

# Add BOS/EOS
bos_id = tokenizer.token_to_id("[BOS]")
eos_id = tokenizer.token_to_id("[EOS]")

if bos_id is None:
    raise ValueError("[BOS] token not found")

if eos_id is None:
    raise ValueError("[EOS] token not found")

bounded_ids = [bos_id] + token_ids + [eos_id]

print("=" * 80)
print("PHASE 4 — END-TO-END STREAMING LOOKUP TEST")
print("=" * 80)

print(f"Expected hash       : {test_hash}")
print(f"Computed hash       : {computed_hash}")
print(f"Hash match          : {computed_hash == test_hash}")

print()
print(f"Source file         : {source_path}")
print(f"Characters          : {len(text):,}")
print(f"Tokens              : {len(token_ids):,}")
print(f"Tokens + boundaries : {len(bounded_ids):,}")

print()
print(f"[BOS] ID            : {bos_id}")
print(f"[EOS] ID            : {eos_id}")

assert computed_hash == test_hash

print()
print("✓ Index lookup works")
print("✓ Source recovery works")
print("✓ SHA-256 verification works")
print("✓ Tokenization works")
print("✓ BOS/EOS insertion works")

print()
print("READY FOR PRODUCTION STREAMING DATASET READER")

In [ ]:
# ============================================================
# PHASE 4 — DOCUMENT-AWARE STREAMING PACKER
# ============================================================

import os
import json
import gzip
import hashlib
from pathlib import Path

import torch
from torch.utils.data import IterableDataset


class DocumentAwarePacker(IterableDataset):

    def __init__(
        self,
        manifest_path,
        index_path,
        tokenizer,
        context_length=4096,
        checkpoint="/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep",
        split="train",
    ):
        super().__init__()

        self.manifest_path = str(manifest_path)
        self.index_path = str(index_path)
        self.tokenizer = tokenizer
        self.context_length = int(context_length)
        self.checkpoint = str(checkpoint)
        self.split = split

        # ----------------------------------------------------
        # Load byte-offset index
        # ----------------------------------------------------
        self.index = {}

        with open(self.index_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()

                if not line:
                    continue

                rec = json.loads(line)

                self.index[rec["text_hash"]] = {
                    "source_file": rec["source_file"],
                    "byte_offset": rec["byte_offset"],
                    "byte_length": rec["byte_length"],
                }

        print(f"Loaded index entries: {len(self.index):,}")

        # ----------------------------------------------------
        # Load manifest
        # ----------------------------------------------------
        self.manifest = []

        with open(self.manifest_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()

                if not line:
                    continue

                rec = json.loads(line)

                # Manifest is the authoritative source of examples.
                self.manifest.append(rec)

        print(f"Loaded manifest records: {len(self.manifest):,}")

        # ----------------------------------------------------
        # Special tokens
        # ----------------------------------------------------
        self.bos_id = self._get_token_id("[BOS]")
        self.eos_id = self._get_token_id("[EOS]")
        self.pad_id = self._get_token_id("[PAD]")

        print(f"BOS id : {self.bos_id}")
        print(f"EOS id : {self.eos_id}")
        print(f"PAD id : {self.pad_id}")

    # --------------------------------------------------------
    # Tokenizer compatibility
    # --------------------------------------------------------
    def _get_token_id(self, token):
        token_id = None

        if hasattr(self.tokenizer, "token_to_id"):
            token_id = self.tokenizer.token_to_id(token)

        elif hasattr(self.tokenizer, "convert_tokens_to_ids"):
            token_id = self.tokenizer.convert_tokens_to_ids(token)

        if token_id is None:
            raise ValueError(
                f"Tokenizer does not contain required special token: {token}"
            )

        if isinstance(token_id, list):
            if len(token_id) != 1:
                raise ValueError(f"Unexpected token id for {token}: {token_id}")
            token_id = token_id[0]

        return int(token_id)

    # --------------------------------------------------------
    # Resolve source document using byte offset
    # --------------------------------------------------------
    def _read_source_record(self, rec):

        text_hash = rec["text_hash"]

        if text_hash not in self.index:
            raise KeyError(
                f"text_hash not found in source index: {text_hash}"
            )

        idx = self.index[text_hash]

        relative_source = idx["source_file"]

        # Index source_file is the original manifest source reference,
        # e.g. acts/documents.jsonl.gz.
        #
        # Actual deduplicated source is under:
        # checkpoint/dedup/<domain>/...
        #
        # Derive domain from the manifest record rather than expecting
        # domain inside the index.

        source_name = relative_source

        if source_name.endswith(".gz"):
            source_name = source_name[:-3]

        source_path = os.path.join(
            self.checkpoint,
            "dedup",
            source_name
        )

        if not os.path.exists(source_path):
            raise FileNotFoundError(
                f"Resolved source does not exist:\n{source_path}"
            )

        with open(source_path, "rb") as f:
            f.seek(idx["byte_offset"])
            raw = f.read(idx["byte_length"])

        # Remove trailing newline if present.
        raw = raw.rstrip(b"\r\n")

        source_record = json.loads(raw.decode("utf-8"))

        return source_record

    # --------------------------------------------------------
    # Extract document text
    # --------------------------------------------------------
    def _extract_text(self, source_record):

        if isinstance(source_record, str):
            return source_record

        # Expected legal-document representation.
        for key in ("text", "content", "document", "body"):
            value = source_record.get(key)

            if isinstance(value, str):
                return value

        raise KeyError(
            "Could not find document text field. "
            f"Available fields: {list(source_record.keys())}"
        )

    # --------------------------------------------------------
    # Tokenize one document
    # --------------------------------------------------------
    def _tokenize_document(self, text):

        if hasattr(self.tokenizer, "encode"):
            encoded = self.tokenizer.encode(text)

            # HuggingFace-style output
            if hasattr(encoded, "ids"):
                token_ids = encoded.ids
            elif isinstance(encoded, dict):
                token_ids = encoded["input_ids"]
            else:
                token_ids = encoded

        else:
            raise TypeError("Unsupported tokenizer interface.")

        token_ids = list(map(int, token_ids))

        # Explicit document boundaries.
        token_ids = (
            [self.bos_id]
            + token_ids
            + [self.eos_id]
        )

        return token_ids

    # --------------------------------------------------------
    # Document-aware packing
    # --------------------------------------------------------
    def _pack_document(self, token_ids):

        # ----------------------------------------------------
        # Long document
        # ----------------------------------------------------
        if len(token_ids) > self.context_length:

            for start in range(
                0,
                len(token_ids),
                self.context_length
            ):
                chunk = token_ids[
                    start:start + self.context_length
                ]

                if len(chunk) < self.context_length:
                    chunk = chunk + [
                        self.pad_id
                    ] * (
                        self.context_length - len(chunk)
                    )

                yield chunk

            return

        # ----------------------------------------------------
        # Normal document
        # ----------------------------------------------------
        chunk = token_ids[:]

        if len(chunk) < self.context_length:
            chunk = chunk + [
                self.pad_id
            ] * (
                self.context_length - len(chunk)
            )

        yield chunk

    # --------------------------------------------------------
    # Streaming iterator
    # --------------------------------------------------------
    def __iter__(self):

        for rec in self.manifest:

            token_ids = self._tokenize_document(
                self._extract_text(
                    self._read_source_record(rec)
                )
            )

            for packed in self._pack_document(token_ids):

                input_ids = torch.tensor(
                    packed,
                    dtype=torch.long
                )

                # Causal LM labels.
                labels = input_ids.clone()

                # Do not train on padding.
                labels[
                    labels == self.pad_id
                ] = -100

                yield {
                    "input_ids": input_ids,
                    "labels": labels,
                    "attention_mask": (
                        input_ids != self.pad_id
                    ).long(),
                }


print("=" * 80)
print("DOCUMENT-AWARE STREAMING PACKER READY")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — STREAMING PACKER SMOKE TEST
# ============================================================

import json
from tokenizers import Tokenizer


CHECKPOINT = "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"

MANIFEST = (
    f"{CHECKPOINT}/manifests/train_manifest.jsonl"
)

INDEX = (
    "/kaggle/working/phase4_streaming_index/"
    "train_source_index.jsonl"
)

TOKENIZER_PATH = (
    f"{CHECKPOINT}/tokenizer/legal_bpe_32k.json"
)

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

dataset = DocumentAwarePacker(
    manifest_path=MANIFEST,
    index_path=INDEX,
    tokenizer=tokenizer,
    context_length=4096,
    checkpoint=CHECKPOINT,
    split="train",
)

print()
print("=" * 80)
print("SMOKE TEST")
print("=" * 80)

iterator = iter(dataset)

for i in range(3):

    example = next(iterator)

    print(
        f"Example {i + 1}: "
        f"input_ids={tuple(example['input_ids'].shape)}, "
        f"labels={tuple(example['labels'].shape)}, "
        f"attention_mask={tuple(example['attention_mask'].shape)}"
    )

    assert example["input_ids"].shape == (4096,)
    assert example["labels"].shape == (4096,)
    assert example["attention_mask"].shape == (4096,)

print()
print("=" * 80)
print("✓ STREAMING PACKER SMOKE TEST PASSED")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — STREAMING PACKER SMOKE TEST
# ============================================================

import json
from tokenizers import Tokenizer


CHECKPOINT = "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"

MANIFEST = (
    f"{CHECKPOINT}/manifests/train_manifest.jsonl"
)

INDEX = (
    "/kaggle/working/phase4_streaming_index/"
    "train_source_index.jsonl"
)

TOKENIZER_PATH = (
    f"{CHECKPOINT}/tokenizer/legal_bpe_32k.json"
)

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

dataset = DocumentAwarePacker(
    manifest_path=MANIFEST,
    index_path=INDEX,
    tokenizer=tokenizer,
    context_length=4096,
    checkpoint=CHECKPOINT,
    split="train",
)

print()
print("=" * 80)
print("SMOKE TEST")
print("=" * 80)

iterator = iter(dataset)

for i in range(3):

    example = next(iterator)

    print(
        f"Example {i + 1}: "
        f"input_ids={tuple(example['input_ids'].shape)}, "
        f"labels={tuple(example['labels'].shape)}, "
        f"attention_mask={tuple(example['attention_mask'].shape)}"
    )

    assert example["input_ids"].shape == (4096,)
    assert example["labels"].shape == (4096,)
    assert example["attention_mask"].shape == (4096,)

print()
print("=" * 80)
print("✓ STREAMING PACKER SMOKE TEST PASSED")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — DATALOADER / MULTI-WORKER STREAMING TEST
# ============================================================

from torch.utils.data import DataLoader

print("=" * 80)
print("PHASE 4 — DATALOADER STREAMING TEST")
print("=" * 80)

loader = DataLoader(
    dataset,
    batch_size=2,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

print("DataLoader created")
print("num_workers : 2")
print("batch_size  : 2")
print()

for step, batch in enumerate(loader):

    print(
        f"Batch {step + 1}: "
        f"input_ids={tuple(batch['input_ids'].shape)}, "
        f"labels={tuple(batch['labels'].shape)}, "
        f"attention_mask={tuple(batch['attention_mask'].shape)}"
    )

    assert batch["input_ids"].shape == (2, 4096)
    assert batch["labels"].shape == (2, 4096)
    assert batch["attention_mask"].shape == (2, 4096)

    if step >= 2:
        break

print()
print("=" * 80)
print("✓ MULTI-WORKER DATALOADER TEST PASSED")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — WORKER-SAFE STREAMING DATASET
# ============================================================

import torch
from torch.utils.data import IterableDataset, get_worker_info


class WorkerSafeDocumentAwarePacker(DocumentAwarePacker):

    def __iter__(self):

        worker_info = get_worker_info()

        # ----------------------------------------------------
        # Single-process mode
        # ----------------------------------------------------
        if worker_info is None:
            start = 0
            step = 1
            worker_id = 0
            num_workers = 1

        # ----------------------------------------------------
        # Multi-worker mode
        # ----------------------------------------------------
        else:
            worker_id = worker_info.id
            num_workers = worker_info.num_workers

            start = worker_id
            step = num_workers

        # ----------------------------------------------------
        # Each worker gets disjoint manifest records:
        #
        # worker 0 -> 0, 2, 4, ...
        # worker 1 -> 1, 3, 5, ...
        #
        # This prevents duplicate training examples.
        # ----------------------------------------------------
        for manifest_idx in range(
            start,
            len(self.manifest),
            step
        ):

            rec = self.manifest[manifest_idx]

            source_record = self._read_source_record(rec)

            text = self._extract_text(source_record)

            token_ids = self._tokenize_document(text)

            for packed in self._pack_document(token_ids):

                input_ids = torch.tensor(
                    packed,
                    dtype=torch.long
                )

                labels = input_ids.clone()

                # Ignore padding in causal-LM loss.
                labels[
                    labels == self.pad_id
                ] = -100

                attention_mask = (
                    input_ids != self.pad_id
                ).long()

                yield {
                    "input_ids": input_ids,
                    "labels": labels,
                    "attention_mask": attention_mask,
                }


print("=" * 80)
print("WORKER-SAFE STREAMING DATASET READY")
print("=" * 80)

print(f"Manifest records : {len(dataset.manifest):,}")
print("Worker sharding  : enabled")
print("Sharding method  : strided manifest partitioning")
print()
print("✓ Worker-safe dataset class defined")

In [ ]:
# ============================================================
# PHASE 4 — WORKER-SAFE DATALOADER VERIFICATION
# ============================================================

worker_safe_dataset = WorkerSafeDocumentAwarePacker(
    manifest_path=MANIFEST,
    index_path=INDEX,
    tokenizer=tokenizer,
    context_length=4096,
    checkpoint=CHECKPOINT,
    split="train",
)

loader = DataLoader(
    worker_safe_dataset,
    batch_size=2,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

print("=" * 80)
print("WORKER-SAFE DATALOADER TEST")
print("=" * 80)

for step, batch in enumerate(loader):

    assert batch["input_ids"].shape == (2, 4096)
    assert batch["labels"].shape == (2, 4096)
    assert batch["attention_mask"].shape == (2, 4096)

    print(
        f"Batch {step + 1}: "
        f"input_ids={tuple(batch['input_ids'].shape)}, "
        f"labels={tuple(batch['labels'].shape)}, "
        f"attention_mask={tuple(batch['attention_mask'].shape)}"
    )

    if step >= 2:
        break

print()
print("=" * 80)
print("✓ WORKER-SAFE STREAMING TEST PASSED")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — WORKER PARTITION / EPOCH VERIFICATION
# ============================================================

from torch.utils.data import get_worker_info

class DebugWorkerSafePacker(WorkerSafeDocumentAwarePacker):

    def __iter__(self):

        worker_info = get_worker_info()

        if worker_info is None:
            worker_id = 0
            num_workers = 1
            start = 0
            step = 1
        else:
            worker_id = worker_info.id
            num_workers = worker_info.num_workers
            start = worker_id
            step = num_workers

        for manifest_idx in range(
            start,
            len(self.manifest),
            step
        ):
            rec = self.manifest[manifest_idx]

            # Emit metadata temporarily for verification.
            yield {
                "manifest_index": manifest_idx,
                "worker_id": worker_id,
                "num_workers": num_workers,
                "text_hash": rec["text_hash"],
            }


debug_dataset = DebugWorkerSafePacker(
    manifest_path=MANIFEST,
    index_path=INDEX,
    tokenizer=tokenizer,
    context_length=4096,
    checkpoint=CHECKPOINT,
    split="train",
)

debug_loader = DataLoader(
    debug_dataset,
    batch_size=4,
    num_workers=2,
)

print("=" * 80)
print("WORKER PARTITION VERIFICATION")
print("=" * 80)

seen_indices = set()
worker_indices = {}

for step, batch in enumerate(debug_loader):

    indices = batch["manifest_index"].tolist()
    workers = batch["worker_id"].tolist()

    for idx, worker in zip(indices, workers):

        idx = int(idx)
        worker = int(worker)

        if idx in seen_indices:
            raise RuntimeError(
                f"DUPLICATE MANIFEST INDEX DETECTED: {idx}"
            )

        seen_indices.add(idx)

        worker_indices.setdefault(worker, []).append(idx)

    if step >= 24:
        break


print(f"Unique manifest indices observed : {len(seen_indices):,}")

for worker_id in sorted(worker_indices):
    values = worker_indices[worker_id]

    print(
        f"Worker {worker_id}: "
        f"{len(values):,} records observed | "
        f"first={values[:5]}"
    )

# ------------------------------------------------------------
# Verify workers are disjoint.
# ------------------------------------------------------------

worker_sets = [
    set(values)
    for values in worker_indices.values()
]

for i in range(len(worker_sets)):
    for j in range(i + 1, len(worker_sets)):
        overlap = worker_sets[i] & worker_sets[j]

        if overlap:
            raise RuntimeError(
                f"WORKER OVERLAP DETECTED: {overlap}"
            )

print()
print("✓ Workers produced disjoint manifest indices")
print("✓ No duplicate manifest indices observed")
print("✓ Strided worker partitioning verified")

print("=" * 80)
print("WORKER PARTITION TEST PASSED")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — WORKER PARTITION / EPOCH VERIFICATION
# ============================================================

from torch.utils.data import get_worker_info


class DebugWorkerSafePacker(WorkerSafeDocumentAwarePacker):

    def __iter__(self):

        worker_info = get_worker_info()

        if worker_info is None:
            worker_id = 0
            num_workers = 1
            start = 0
            step = 1
        else:
            worker_id = worker_info.id
            num_workers = worker_info.num_workers
            start = worker_id
            step = num_workers

        for manifest_idx in range(
            start,
            len(self.manifest),
            step
        ):
            rec = self.manifest[manifest_idx]

            yield {
                "manifest_index": manifest_idx,
                "worker_id": worker_id,
                "num_workers": num_workers,
                "text_hash": rec["text_hash"],
            }


debug_dataset = DebugWorkerSafePacker(
    manifest_path=MANIFEST,
    index_path=INDEX,
    tokenizer=tokenizer,
    context_length=4096,
    checkpoint=CHECKPOINT,
    split="train",
)

debug_loader = DataLoader(
    debug_dataset,
    batch_size=4,
    num_workers=2,
)

print("=" * 80)
print("WORKER PARTITION VERIFICATION")
print("=" * 80)

seen_indices = set()
worker_indices = {}

for step, batch in enumerate(debug_loader):

    indices = batch["manifest_index"].tolist()
    workers = batch["worker_id"].tolist()

    for idx, worker in zip(indices, workers):

        idx = int(idx)
        worker = int(worker)

        if idx in seen_indices:
            raise RuntimeError(
                f"DUPLICATE MANIFEST INDEX DETECTED: {idx}"
            )

        seen_indices.add(idx)

        worker_indices.setdefault(
            worker, []
        ).append(idx)

    if step >= 24:
        break


print(
    f"Unique manifest indices observed : "
    f"{len(seen_indices):,}"
)

for worker_id in sorted(worker_indices):

    values = worker_indices[worker_id]

    print(
        f"Worker {worker_id}: "
        f"{len(values):,} records observed | "
        f"first={values[:5]}"
    )


# ------------------------------------------------------------
# Verify workers are disjoint
# ------------------------------------------------------------

worker_sets = [
    set(values)
    for values in worker_indices.values()
]

for i in range(len(worker_sets)):
    for j in range(i + 1, len(worker_sets)):

        overlap = worker_sets[i] & worker_sets[j]

        if overlap:
            raise RuntimeError(
                f"WORKER OVERLAP DETECTED: {overlap}"
            )


print()
print("✓ Workers produced disjoint manifest indices")
print("✓ No duplicate manifest indices observed")
print("✓ Strided worker partitioning verified")

print("=" * 80)
print("WORKER PARTITION TEST PASSED")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — REPEATED ITERATION / EPOCH VERIFICATION
# ============================================================

print("=" * 80)
print("REPEATED ITERATION / EPOCH VERIFICATION")
print("=" * 80)

epoch_dataset = WorkerSafeDocumentAwarePacker(
    manifest_path=MANIFEST,
    index_path=INDEX,
    tokenizer=tokenizer,
    context_length=4096,
    checkpoint=CHECKPOINT,
    split="train",
)

# ------------------------------------------------------------
# Read a small deterministic prefix from epoch 1
# ------------------------------------------------------------

epoch1 = []

for i, example in enumerate(epoch_dataset):

    epoch1.append(
        example["input_ids"][:16].tolist()
    )

    if i >= 4:
        break


# ------------------------------------------------------------
# Start a fresh iterator = new epoch
# ------------------------------------------------------------

epoch2 = []

for i, example in enumerate(epoch_dataset):

    epoch2.append(
        example["input_ids"][:16].tolist()
    )

    if i >= 4:
        break


print(
    f"Epoch 1 examples checked : {len(epoch1)}"
)

print(
    f"Epoch 2 examples checked : {len(epoch2)}"
)

# ------------------------------------------------------------
# With no shuffle configured, repeated iteration should be
# deterministic.
# ------------------------------------------------------------

if epoch1 != epoch2:
    raise RuntimeError(
        "Repeated iteration is not deterministic."
    )


print("✓ Epoch 1 iteration completed")
print("✓ Epoch 2 iteration completed")
print("✓ Repeated iteration is deterministic")
print("✓ Dataset can be restarted for a new epoch")

print("=" * 80)
print("EPOCH ITERATION TEST PASSED")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — PRODUCTION TRAIN / VALIDATION DATASETS
# ============================================================

import os
import json
import torch
from torch.utils.data import DataLoader

CHECKPOINT = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep"
)

TRAIN_MANIFEST = (
    f"{CHECKPOINT}/manifests/train_manifest.jsonl"
)

VAL_MANIFEST = (
    f"{CHECKPOINT}/manifests/validation_manifest.jsonl"
)

INDEX_PATH = (
    "/kaggle/working/phase4_streaming_index/"
    "train_source_index.jsonl"
)

TOKENIZER_PATH = (
    f"{CHECKPOINT}/tokenizer/legal_bpe_32k.json"
)

CONTEXT_LENGTH = 4096


# ------------------------------------------------------------
# Load training configuration
# ------------------------------------------------------------

with open(
    f"{CHECKPOINT}/training_example_config.json",
    "r",
    encoding="utf-8",
) as f:
    training_config = json.load(f)


print("=" * 80)
print("PHASE 4 — PRODUCTION DATASET CONFIGURATION")
print("=" * 80)

print(f"Context length : {CONTEXT_LENGTH:,}")
print(f"Train manifest : {TRAIN_MANIFEST}")
print(f"Val manifest   : {VAL_MANIFEST}")
print(f"Source index   : {INDEX_PATH}")


# ------------------------------------------------------------
# Training dataset
# ------------------------------------------------------------

train_dataset = WorkerSafeDocumentAwarePacker(
    manifest_path=TRAIN_MANIFEST,
    index_path=INDEX_PATH,
    tokenizer=tokenizer,
    context_length=CONTEXT_LENGTH,
    checkpoint=CHECKPOINT,
    split="train",
)


# ------------------------------------------------------------
# Validation dataset
# ------------------------------------------------------------

validation_dataset = WorkerSafeDocumentAwarePacker(
    manifest_path=VAL_MANIFEST,
    index_path=INDEX_PATH,
    tokenizer=tokenizer,
    context_length=CONTEXT_LENGTH,
    checkpoint=CHECKPOINT,
    split="validation",
)


print()
print("Train records      :", f"{len(train_dataset.manifest):,}")
print("Validation records :", f"{len(validation_dataset.manifest):,}")

print()
print("✓ Training dataset created")
print("✓ Validation dataset created")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — PRODUCTION TRAIN / VALIDATION DATALOADERS
# ============================================================

BATCH_SIZE = 2
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0),
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0),
)


print("=" * 80)
print("PRODUCTION DATALOADERS CREATED")
print("=" * 80)

print(f"Batch size          : {BATCH_SIZE}")
print(f"Workers             : {NUM_WORKERS}")
print(
    f"Pin memory          : "
    f"{torch.cuda.is_available()}"
)
print("Persistent workers   :", NUM_WORKERS > 0)

print()
print("Train loader        : READY")
print("Validation loader   : READY")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — PRODUCTION LOADER SMOKE TEST
# ============================================================

print("=" * 80)
print("PRODUCTION LOADER SMOKE TEST")
print("=" * 80)


# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------

train_batch = next(iter(train_loader))

print("TRAIN")
print(
    "  input_ids      :",
    tuple(train_batch["input_ids"].shape)
)
print(
    "  labels         :",
    tuple(train_batch["labels"].shape)
)
print(
    "  attention_mask :",
    tuple(train_batch["attention_mask"].shape)
)

assert train_batch["input_ids"].shape == (
    BATCH_SIZE,
    CONTEXT_LENGTH,
)

assert train_batch["labels"].shape == (
    BATCH_SIZE,
    CONTEXT_LENGTH,
)

assert train_batch["attention_mask"].shape == (
    BATCH_SIZE,
    CONTEXT_LENGTH,
)


# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

val_batch = next(iter(validation_loader))

print()
print("VALIDATION")
print(
    "  input_ids      :",
    tuple(val_batch["input_ids"].shape)
)
print(
    "  labels         :",
    tuple(val_batch["labels"].shape)
)
print(
    "  attention_mask :",
    tuple(val_batch["attention_mask"].shape)
)

assert val_batch["input_ids"].shape == (
    BATCH_SIZE,
    CONTEXT_LENGTH,
)

assert val_batch["labels"].shape == (
    BATCH_SIZE,
    CONTEXT_LENGTH,
)

assert val_batch["attention_mask"].shape == (
    BATCH_SIZE,
    CONTEXT_LENGTH,
)


# ------------------------------------------------------------
# Basic value checks
# ------------------------------------------------------------

assert train_batch["input_ids"].dtype == torch.long
assert train_batch["labels"].dtype == torch.long
assert train_batch["attention_mask"].dtype == torch.long

assert torch.all(
    (train_batch["attention_mask"] == 0)
    | (train_batch["attention_mask"] == 1)
)

assert torch.all(
    (val_batch["attention_mask"] == 0)
    | (val_batch["attention_mask"] == 1)
)


print()
print("=" * 80)
print("✓ TRAIN LOADER PASSED")
print("✓ VALIDATION LOADER PASSED")
print("✓ BATCH SHAPES PASSED")
print("✓ DTYPE CHECKS PASSED")
print("✓ ATTENTION MASK CHECKS PASSED")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — MODEL ENVIRONMENT CHECK
# ============================================================

import os
import torch

print("=" * 80)
print("PHASE 4 — MODEL ENVIRONMENT CHECK")
print("=" * 80)

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

if torch.cuda.is_available():

    print("CUDA version    :", torch.version.cuda)
    print("GPU count       :", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):

        props = torch.cuda.get_device_properties(i)

        print(
            f"GPU {i}          : "
            f"{props.name}"
        )

        print(
            f"  VRAM          : "
            f"{props.total_memory / (1024**3):.2f} GB"
        )

else:
    print("WARNING: CUDA is not available.")

print()
print("=" * 80)
print("ENVIRONMENT CHECK COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — MODEL CONFIGURATION
# ============================================================

from dataclasses import dataclass


@dataclass
class LegalLMConfig:

    # --------------------------------------------------------
    # Vocabulary / sequence
    # --------------------------------------------------------
    vocab_size: int = 32000
    max_position_embeddings: int = 4096

    # --------------------------------------------------------
    # Model dimensions
    # --------------------------------------------------------
    hidden_size: int = 768
    intermediate_size: int = 3072

    num_hidden_layers: int = 12
    num_attention_heads: int = 12

    # --------------------------------------------------------
    # Regularization
    # --------------------------------------------------------
    hidden_dropout: float = 0.0
    attention_dropout: float = 0.0

    # --------------------------------------------------------
    # Token IDs
    # --------------------------------------------------------
    bos_token_id: int = None
    eos_token_id: int = None
    pad_token_id: int = None


MODEL_CONFIG = LegalLMConfig(
    vocab_size=32000,
    max_position_embeddings=4096,

    # Keep these aligned with the architecture finalized
    # for the project.
    hidden_size=768,
    intermediate_size=3072,
    num_hidden_layers=12,
    num_attention_heads=12,

    hidden_dropout=0.0,
    attention_dropout=0.0,

    bos_token_id=dataset.bos_id,
    eos_token_id=dataset.eos_id,
    pad_token_id=dataset.pad_id,
)


print("=" * 80)
print("MODEL CONFIGURATION")
print("=" * 80)

print(f"Vocabulary size       : {MODEL_CONFIG.vocab_size:,}")
print(
    f"Context length        : "
    f"{MODEL_CONFIG.max_position_embeddings:,}"
)
print(f"Hidden size           : {MODEL_CONFIG.hidden_size}")
print(
    f"Intermediate size     : "
    f"{MODEL_CONFIG.intermediate_size}"
)
print(
    f"Transformer layers    : "
    f"{MODEL_CONFIG.num_hidden_layers}"
)
print(
    f"Attention heads       : "
    f"{MODEL_CONFIG.num_attention_heads}"
)

print()
print("Special tokens:")
print("  PAD :", MODEL_CONFIG.pad_token_id)
print("  BOS :", MODEL_CONFIG.bos_token_id)
print("  EOS :", MODEL_CONFIG.eos_token_id)

assert MODEL_CONFIG.vocab_size == 32000
assert MODEL_CONFIG.max_position_embeddings == 4096

assert (
    MODEL_CONFIG.hidden_size
    % MODEL_CONFIG.num_attention_heads
    == 0
)

print()
print("✓ Vocabulary verified")
print("✓ Context length verified")
print("✓ Attention dimensions verified")
print("✓ Special token IDs verified")

print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — BUILD CAUSAL LANGUAGE MODEL
# ============================================================

from transformers import GPT2Config, GPT2LMHeadModel


hf_config = GPT2Config(
    vocab_size=MODEL_CONFIG.vocab_size,

    n_positions=MODEL_CONFIG.max_position_embeddings,
    n_ctx=MODEL_CONFIG.max_position_embeddings,

    n_embd=MODEL_CONFIG.hidden_size,
    n_layer=MODEL_CONFIG.num_hidden_layers,
    n_head=MODEL_CONFIG.num_attention_heads,

    n_inner=MODEL_CONFIG.intermediate_size,

    resid_pdrop=MODEL_CONFIG.hidden_dropout,
    embd_pdrop=MODEL_CONFIG.hidden_dropout,
    attn_pdrop=MODEL_CONFIG.attention_dropout,

    bos_token_id=MODEL_CONFIG.bos_token_id,
    eos_token_id=MODEL_CONFIG.eos_token_id,
    pad_token_id=MODEL_CONFIG.pad_token_id,

    use_cache=False,
)


model = GPT2LMHeadModel(hf_config)


# ------------------------------------------------------------
# Parameter count
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print("=" * 80)
print("CAUSAL LANGUAGE MODEL CREATED")
print("=" * 80)

print(
    f"Total parameters     : "
    f"{total_params:,}"
)

print(
    f"Trainable parameters : "
    f"{trainable_params:,}"
)

print(
    f"Parameter memory FP32: "
    f"{total_params * 4 / (1024**3):.2f} GB"
)

print()
print("✓ Model instantiated")
print("✓ LM head present")
print("✓ Vocabulary = 32,000")
print("✓ Context = 4,096")

print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — DEVICE SETUP
# ============================================================

import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 60)
print("DEVICE SETUP")
print("=" * 60)
print("Device :", device)

if device.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))
    print("CUDA   :", torch.version.cuda)
else:
    print("WARNING: CUDA is not available")

print("=" * 60)
print("✓ Device ready")

In [ ]:
# ============================================================
# PHASE 4 — MOVE MODEL TO DEVICE
# ============================================================

model = model.to(device)

print("=" * 60)
print("MODEL DEVICE CHECK")
print("=" * 60)

print("Target device :", device)
print("Model device  :", next(model.parameters()).device)

assert next(model.parameters()).device == device

print("✓ Model and device match")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — FORCE MODEL TO SAME DEVICE
# ============================================================

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 60)
print("BEFORE MODEL MOVE")
print("=" * 60)
print("Target device :", device)
print("Model device  :", next(model.parameters()).device)

# Move the CURRENT model object
model = model.to(device)

print("=" * 60)
print("AFTER MODEL MOVE")
print("=" * 60)
print("Target device :", device)
print("Model device  :", next(model.parameters()).device)

# Hard verification
model_device = next(model.parameters()).device

if model_device != device:
    raise RuntimeError(
        f"Model is still on {model_device}, expected {device}"
    )

print("✓ Model successfully moved to", device)
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — CORRECT DEVICE VERIFICATION
# ============================================================

import torch

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Ensure model is on the exact target device
model = model.to(device)

model_device = next(model.parameters()).device

print("=" * 60)
print("DEVICE VERIFICATION")
print("=" * 60)
print("Target device :", device)
print("Model device  :", model_device)
print("Input device  :", input_ids.device)
print("Mask device   :", attention_mask.device)

# Compare device type + index explicitly
assert model_device.type == device.type
assert model_device.index == device.index

assert input_ids.device.type == device.type
assert input_ids.device.index == device.index

assert attention_mask.device.type == device.type
assert attention_mask.device.index == device.index

print("=" * 60)
print("✓ Model       : cuda:0")
print("✓ input_ids   : cuda:0")
print("✓ attention   : cuda:0")
print("✓ Devices match")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — SINGLE FORWARD PASS TEST
# ============================================================

print("=" * 80)
print("FORWARD PASS VERIFICATION")
print("=" * 80)

model.eval()

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(
    device,
    non_blocking=True
)

attention_mask = batch["attention_mask"].to(
    device,
    non_blocking=True
)

labels = batch["labels"].to(
    device,
    non_blocking=True
)


print("Input shape :", tuple(input_ids.shape))
print("Mask shape  :", tuple(attention_mask.shape))
print("Label shape :", tuple(labels.shape))


with torch.no_grad():

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
    )


loss = outputs.loss
logits = outputs.logits


print()
print("Logits shape :", tuple(logits.shape))
print(
    "Loss         :",
    float(loss.detach().cpu())
)


# ------------------------------------------------------------
# Shape assertions
# ------------------------------------------------------------

assert logits.shape[0] == BATCH_SIZE
assert logits.shape[1] == CONTEXT_LENGTH
assert logits.shape[2] == MODEL_CONFIG.vocab_size

assert torch.isfinite(loss).item()


print()
print("✓ Forward pass completed")
print("✓ Logits shape verified")
print("✓ Vocabulary dimension verified")
print("✓ Loss is finite")

print("=" * 80)
print("FORWARD PASS TEST PASSED")
print("=" * 80)

In [ ]:
# ============================================================
# PHASE 4 — FORWARD PASS TEST
# ============================================================

model.eval()

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
    )

print("=" * 60)
print("FORWARD PASS")
print("=" * 60)
print("✓ Forward pass successful")
print("Logits shape :", tuple(outputs.logits.shape))
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — LABEL / LOSS VERIFICATION
# ============================================================

import torch

model.train()

# ------------------------------------------------------------
# 1. Prepare causal-LM labels
# ------------------------------------------------------------
labels = input_ids.clone()

# Padding tokens must not contribute to loss
if "pad_token_id" in globals() and pad_token_id is not None:
    labels[labels == pad_token_id] = -100
elif hasattr(model.config, "pad_token_id") and model.config.pad_token_id is not None:
    labels[labels == model.config.pad_token_id] = -100

labels = labels.to(device)

# ------------------------------------------------------------
# 2. Verify shapes/devices
# ------------------------------------------------------------
print("=" * 60)
print("LABEL VERIFICATION")
print("=" * 60)

print("input_ids shape :", tuple(input_ids.shape))
print("labels shape    :", tuple(labels.shape))
print("attention shape :", tuple(attention_mask.shape))

print("input device    :", input_ids.device)
print("labels device   :", labels.device)
print("attention device:", attention_mask.device)

assert labels.shape == input_ids.shape
assert attention_mask.shape == input_ids.shape
assert labels.device == input_ids.device
assert attention_mask.device == input_ids.device

print("✓ Shapes match")
print("✓ Devices match")

# ------------------------------------------------------------
# 3. Check valid target tokens
# ------------------------------------------------------------
valid_targets = (labels != -100).sum().item()
ignored_targets = (labels == -100).sum().item()

print()
print("Valid targets   :", valid_targets)
print("Ignored targets :", ignored_targets)

assert valid_targets > 0

# ------------------------------------------------------------
# 4. Compute causal-LM loss
# ------------------------------------------------------------
model.eval()

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
    )

loss = outputs.loss

print()
print("=" * 60)
print("LOSS VERIFICATION")
print("=" * 60)
print("Loss :", loss.item())

assert torch.isfinite(loss)
assert loss.item() > 0

print("✓ Loss is finite")
print("✓ Loss computation successful")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — CAUSAL LABEL SHIFT VERIFICATION
# ============================================================

print("=" * 60)
print("CAUSAL LABEL SHIFT VERIFICATION")
print("=" * 60)

# ------------------------------------------------------------
# Hugging Face causal LM behavior:
#
# logits[:, :-1] predicts labels[:, 1:]
#
# Therefore:
#   input[t] -> target[t+1]
# ------------------------------------------------------------

shifted_logits = outputs.logits[..., :-1, :].contiguous()
shifted_labels = labels[..., 1:].contiguous()

print("Original logits :", tuple(outputs.logits.shape))
print("Original labels :", tuple(labels.shape))
print("Shifted logits  :", tuple(shifted_logits.shape))
print("Shifted labels  :", tuple(shifted_labels.shape))

assert shifted_logits.shape[:2] == shifted_labels.shape
assert shifted_logits.shape[-1] == 32000

print()
print("✓ Causal shift dimensions are correct")

# ------------------------------------------------------------
# Verify that labels are actually next-token targets
# ------------------------------------------------------------

for b in range(min(2, input_ids.size(0))):
    print()
    print(f"Example {b}")

    # Show first few input -> target pairs
    for t in range(min(10, input_ids.size(1) - 1)):
        inp = input_ids[b, t].item()
        target = labels[b, t + 1].item()

        print(
            f"  position {t:2d}: "
            f"input={inp:6d} -> target={target:6d}"
        )

print()
print("=" * 60)
print("✓ LABEL SHIFT VERIFICATION COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — DOCUMENT BOUNDARY VERIFICATION
# ============================================================

print("=" * 60)
print("DOCUMENT BOUNDARY VERIFICATION")
print("=" * 60)

# Resolve special-token IDs
bos_id = tokenizer.token_to_id("[BOS]")
eos_id = tokenizer.token_to_id("[EOS]")
pad_id = tokenizer.token_to_id("[PAD]")
unk_id = tokenizer.token_to_id("[UNK]")

print("BOS id :", bos_id)
print("EOS id :", eos_id)
print("PAD id :", pad_id)
print("UNK id :", unk_id)

assert bos_id is not None
assert eos_id is not None
assert pad_id is not None
assert unk_id is not None

print("✓ All required special tokens exist")

# Count special tokens in the current batch
for b in range(input_ids.size(0)):
    ids = input_ids[b]

    bos_positions = (ids == bos_id).nonzero(as_tuple=True)[0]
    eos_positions = (ids == eos_id).nonzero(as_tuple=True)[0]

    print()
    print(f"Example {b}")
    print("  BOS positions :", bos_positions.tolist()[:20])
    print("  EOS positions :", eos_positions.tolist()[:20])
    print("  BOS count     :", len(bos_positions))
    print("  EOS count     :", len(eos_positions))

print()
print("=" * 60)
print("✓ DOCUMENT BOUNDARY TOKENS VERIFIED")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — ATTENTION MASK / PADDING VERIFICATION
# ============================================================

print("=" * 60)
print("ATTENTION MASK VERIFICATION")
print("=" * 60)

assert attention_mask.shape == input_ids.shape

for b in range(input_ids.size(0)):

    ids = input_ids[b]
    mask = attention_mask[b]
    lbl = labels[b]

    active = mask == 1
    inactive = mask == 0

    active_count = active.sum().item()
    inactive_count = inactive.sum().item()

    print()
    print(f"Example {b}")
    print("  Active tokens   :", active_count)
    print("  Masked tokens   :", inactive_count)

    # Masked positions must not contribute to loss
    if inactive_count > 0:
        assert torch.all(lbl[inactive] == -100)

    # Active positions should have usable labels except where
    # explicitly excluded by the dataset construction.
    print("  ✓ Padding positions excluded from loss")

print()
print("=" * 60)
print("✓ ATTENTION MASK VERIFIED")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — TOKEN ID RANGE VERIFICATION
# ============================================================

print("=" * 60)
print("TOKEN ID RANGE VERIFICATION")
print("=" * 60)

vocab_size = model.config.vocab_size

print("Model vocabulary :", vocab_size)
print("Tokenizer vocab  :", tokenizer.get_vocab_size())

assert vocab_size == 32000
assert tokenizer.get_vocab_size() == vocab_size

# Input IDs must be valid vocabulary IDs
assert input_ids.min().item() >= 0
assert input_ids.max().item() < vocab_size

# Labels may contain -100, but all actual token labels
# must be within vocabulary range.
valid_labels = labels[labels != -100]

assert valid_labels.numel() > 0
assert valid_labels.min().item() >= 0
assert valid_labels.max().item() < vocab_size

print("Input ID range   :", input_ids.min().item(), "to", input_ids.max().item())
print("Label ID range   :", valid_labels.min().item(), "to", valid_labels.max().item())

print()
print("✓ All input IDs are inside vocabulary")
print("✓ All valid labels are inside vocabulary")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — CONTEXT LENGTH VERIFICATION
# ============================================================

print("=" * 60)
print("CONTEXT LENGTH VERIFICATION")
print("=" * 60)

context_length = 4096

print("Expected context :", context_length)
print("Input shape      :", tuple(input_ids.shape))
print("Attention shape  :", tuple(attention_mask.shape))
print("Label shape      :", tuple(labels.shape))

assert input_ids.size(1) == context_length
assert attention_mask.size(1) == context_length
assert labels.size(1) == context_length

assert model.config.n_positions >= context_length

print()
print("✓ Context length = 4096")
print("✓ Model supports 4096 positions")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — CONTEXT LENGTH VERIFICATION
# ============================================================

print("=" * 60)
print("CONTEXT LENGTH VERIFICATION")
print("=" * 60)

context_length = 4096

print("Expected context :", context_length)
print("Input shape      :", tuple(input_ids.shape))
print("Attention shape  :", tuple(attention_mask.shape))
print("Label shape      :", tuple(labels.shape))

assert input_ids.size(1) == context_length
assert attention_mask.size(1) == context_length
assert labels.size(1) == context_length

assert model.config.n_positions >= context_length

print()
print("✓ Context length = 4096")
print("✓ Model supports 4096 positions")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — OPTIMIZER SETUP VERIFICATION
# ============================================================

import torch

model.train()

learning_rate = 3e-4
weight_decay = 0.1

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)

print("=" * 60)
print("OPTIMIZER SETUP")
print("=" * 60)
print("Optimizer     : AdamW")
print("Learning rate :", learning_rate)
print("Weight decay  :", weight_decay)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print("Trainable parameters :", f"{trainable_params:,}")
print("Total parameters     :", f"{total_params:,}")

assert trainable_params > 0
assert len(optimizer.param_groups) == 1
assert optimizer.param_groups[0]["lr"] == learning_rate

print()
print("✓ Optimizer initialized")
print("✓ Trainable parameters detected")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — SINGLE BACKWARD PASS VERIFICATION
# ============================================================

optimizer.zero_grad(set_to_none=True)

model.train()

outputs_train = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
)

loss_train = outputs_train.loss

print("=" * 60)
print("BACKWARD PASS")
print("=" * 60)
print("Loss before backward :", loss_train.item())

assert torch.isfinite(loss_train)
assert loss_train.requires_grad

loss_train.backward()

print("✓ Backward pass completed")

# ------------------------------------------------------------
# Gradient verification
# ------------------------------------------------------------

gradient_params = 0
nonfinite_gradients = 0
gradient_norm_sq = 0.0

for name, param in model.named_parameters():

    if param.requires_grad:

        if param.grad is not None:

            gradient_params += 1

            if not torch.isfinite(param.grad).all():
                nonfinite_gradients += 1

            gradient_norm_sq += (
                param.grad.detach().float().norm(2).item() ** 2
            )

gradient_norm = gradient_norm_sq ** 0.5

print("Parameters with gradients :", gradient_params)
print("Non-finite gradients      :", nonfinite_gradients)
print("Gradient norm             :", gradient_norm)

assert gradient_params > 0
assert nonfinite_gradients == 0
assert torch.isfinite(
    torch.tensor(gradient_norm)
)

print()
print("✓ Gradients exist")
print("✓ Gradients are finite")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — MEMORY INSPECTION
# ============================================================

import os
import gc
import torch

print("=" * 60)
print("MEMORY INSPECTION")
print("=" * 60)

# ----------------------------
# System RAM
# ----------------------------
try:
    import psutil

    ram = psutil.virtual_memory()

    print("\nSYSTEM RAM")
    print("-" * 60)
    print(f"Total     : {ram.total / (1024**3):.2f} GB")
    print(f"Used      : {ram.used / (1024**3):.2f} GB")
    print(f"Available : {ram.available / (1024**3):.2f} GB")
    print(f"Percent   : {ram.percent:.1f}%")

except Exception as e:
    print("RAM inspection unavailable:", e)


# ----------------------------
# GPU memory
# ----------------------------
print("\nGPU MEMORY")
print("-" * 60)

if torch.cuda.is_available():

    for i in range(torch.cuda.device_count()):

        props = torch.cuda.get_device_properties(i)

        allocated = torch.cuda.memory_allocated(i)
        reserved = torch.cuda.memory_reserved(i)
        free, total = torch.cuda.mem_get_info(i)

        print(f"\nGPU {i}: {props.name}")
        print(f"Total     : {total / (1024**3):.2f} GB")
        print(f"Free      : {free / (1024**3):.2f} GB")
        print(f"Allocated : {allocated / (1024**3):.2f} GB")
        print(f"Reserved  : {reserved / (1024**3):.2f} GB")

else:
    print("CUDA unavailable")

print("\nCURRENT OBJECTS")
print("-" * 60)
print(f"Model device : {next(model.parameters()).device}")

if 'input_ids' in globals():
    print(f"input_ids    : {tuple(input_ids.shape)}")

if 'labels' in globals():
    print(f"labels       : {tuple(labels.shape)}")

if 'outputs' in globals():
    print("outputs      : present")

if 'outputs_train' in globals():
    print("outputs_train: present")

print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — MEMORY CLEANUP
# ============================================================

import gc
import torch

print("=" * 60)
print("CLEANING TEMPORARY TRAINING MEMORY")
print("=" * 60)

# Remove large temporary tensors from previous tests.
_names_to_delete = [
    "outputs_train",
    "outputs",
    "logits",
    "loss",
    "train_loss",
    "input_ids",
    "labels",
    "attention_mask",
    "batch",
]

for _name in _names_to_delete:
    if _name in globals():
        try:
            del globals()[_name]
            print(f"Deleted: {_name}")
        except Exception:
            pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("\nMemory after cleanup")
print("-" * 60)

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()

    print(f"GPU free      : {free / (1024**3):.2f} GB")
    print(f"GPU allocated : {torch.cuda.memory_allocated() / (1024**3):.2f} GB")
    print(f"GPU reserved  : {torch.cuda.memory_reserved() / (1024**3):.2f} GB")

try:
    import psutil
    ram = psutil.virtual_memory()
    print(f"RAM used      : {ram.used / (1024**3):.2f} GB")
    print(f"RAM available : {ram.available / (1024**3):.2f} GB")
except Exception:
    pass

print("=" * 60)
print("✓ Cleanup complete")

In [ ]:
# ============================================================
# PHASE 4 — MOVE MODEL TO FREE GPU
# ============================================================

import gc
import torch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

# Use the completely free second T4.
device = torch.device("cuda:1")

print("=" * 60)
print("MODEL DEVICE RESET")
print("=" * 60)

print("Target device :", device)
print("Before        :", next(model.parameters()).device)

# Move model
model = model.to(device)

# Make sure training mode is enabled
model.train()

print("After         :", next(model.parameters()).device)

# Verify
assert next(model.parameters()).device == device

free, total = torch.cuda.mem_get_info(device)

print("\nGPU 1 MEMORY")
print("-" * 60)
print(f"Free  : {free / (1024**3):.2f} GB")
print(f"Total : {total / (1024**3):.2f} GB")

print("\n✓ Model successfully moved to GPU 1")
print("✓ GPU 0 is no longer required for model computation")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — SINGLE MICRO-BATCH
# ============================================================

import gc
import torch

# Remove old batch tensors if they exist
for _name in [
    "input_ids",
    "labels",
    "attention_mask",
    "batch",
    "outputs",
    "outputs_train",
    "logits",
]:
    if _name in globals():
        try:
            del globals()[_name]
        except Exception:
            pass

gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------------------------
# Get ONE example from the existing training loader/dataset
# ------------------------------------------------------------

batch = next(iter(train_dataloader))

# Handle normal dict-style batch
input_ids = batch["input_ids"][:1].to(device, non_blocking=True)
labels = batch["labels"][:1].to(device, non_blocking=True)

if "attention_mask" in batch:
    attention_mask = batch["attention_mask"][:1].to(
        device,
        non_blocking=True
    )
else:
    attention_mask = (input_ids != pad_token_id).long()

print("=" * 60)
print("SINGLE MICRO-BATCH")
print("=" * 60)

print("input_ids shape :", tuple(input_ids.shape))
print("labels shape    :", tuple(labels.shape))
print("attention shape :", tuple(attention_mask.shape))

print("input device    :", input_ids.device)
print("labels device   :", labels.device)
print("attention device:", attention_mask.device)

assert input_ids.shape == (1, 4096)
assert labels.shape == (1, 4096)
assert attention_mask.shape == (1, 4096)

assert input_ids.device == device
assert labels.device == device
assert attention_mask.device == device

print("\n✓ Micro-batch size = 1")
print("✓ Context length = 4096")
print("✓ All tensors on GPU 1")
print("=" * 60)

In [ ]:
# ============================================================
# FIND EXISTING DATASET / PACKER VARIABLES
# ============================================================

print("=" * 60)
print("AVAILABLE DATASET VARIABLES")
print("=" * 60)

for name, obj in sorted(globals().items()):
    if name.startswith("_"):
        continue

    cls = type(obj).__name__

    if (
        "Dataset" in cls
        or "Packer" in cls
        or "Loader" in cls
        or "DataLoader" in cls
    ):
        print(f"{name:30s} -> {cls}")

print("=" * 60)

In [ ]:
# ============================================================
# CREATE MICRO-BATCH DATALOADER
# ============================================================

import torch
from torch.utils.data import DataLoader, IterableDataset

# ------------------------------------------------------------
# Find an existing IterableDataset
# ------------------------------------------------------------

_dataset_candidates = []

for name, obj in globals().items():
    if name.startswith("_"):
        continue

    if isinstance(obj, IterableDataset):
        _dataset_candidates.append((name, obj))

print("IterableDataset candidates:")

for name, obj in _dataset_candidates:
    print(f"  {name} -> {type(obj).__name__}")

if len(_dataset_candidates) == 0:
    raise RuntimeError(
        "No IterableDataset found. Run Cell 4A and send me its output."
    )

# Prefer an explicitly named training dataset if present.
_preferred_names = [
    "train_dataset",
    "training_dataset",
    "streaming_train_dataset",
    "packed_train_dataset",
    "train_packed_dataset",
]

train_dataset = None

for name in _preferred_names:
    if name in globals() and isinstance(globals()[name], IterableDataset):
        train_dataset = globals()[name]
        print(f"\n✓ Selected dataset: {name}")
        break

if train_dataset is None:
    if len(_dataset_candidates) == 1:
        train_dataset = _dataset_candidates[0][1]
        print(
            f"\n✓ Selected only available dataset: "
            f"{_dataset_candidates[0][0]}"
        )
    else:
        raise RuntimeError(
            "Multiple IterableDatasets found. "
            "Send me the Cell 4A output so I can select the correct training dataset."
        )

# ------------------------------------------------------------
# Micro-batch = 1
# ------------------------------------------------------------

train_dataloader = DataLoader(
    train_dataset,
    batch_size=1,
    num_workers=0,
    pin_memory=True,
)

print("\n" + "=" * 60)
print("TRAIN DATALOADER CREATED")
print("=" * 60)

print("Dataset       :", type(train_dataset).__name__)
print("Batch size    : 1")
print("Workers       : 0")
print("Pin memory    : True")
print("✓ Streaming DataLoader ready")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — EXISTING TRAIN LOADER VERIFICATION
# ============================================================

import torch
import gc

gc.collect()
torch.cuda.empty_cache()

print("=" * 60)
print("TRAIN LOADER VERIFICATION")
print("=" * 60)

print("train_dataset :", type(train_dataset).__name__)
print("train_loader   :", type(train_loader).__name__)

batch = next(iter(train_loader))

print("\nBatch type     :", type(batch).__name__)

if isinstance(batch, dict):
    print("Batch keys     :", list(batch.keys()))

    for key, value in batch.items():
        if torch.is_tensor(value):
            print(
                f"{key:16s}: shape={tuple(value.shape)}, "
                f"dtype={value.dtype}, device={value.device}"
            )

print("=" * 60)
print("✓ Existing train_loader works")

In [ ]:
# ============================================================
# PHASE 4 — MICRO-BATCH ON GPU 1
# ============================================================

device = torch.device("cuda:1")

# Model should already be on GPU 1.
model = model.to(device)
model.train()

input_ids = batch["input_ids"][:1].to(
    device,
    non_blocking=True
)

labels = batch["labels"][:1].to(
    device,
    non_blocking=True
)

if "attention_mask" in batch:
    attention_mask = batch["attention_mask"][:1].to(
        device,
        non_blocking=True
    )
else:
    attention_mask = torch.ones_like(
        input_ids,
        dtype=torch.long,
        device=device
    )

print("=" * 60)
print("MICRO-BATCH VERIFICATION")
print("=" * 60)

print("input_ids      :", tuple(input_ids.shape))
print("labels         :", tuple(labels.shape))
print("attention_mask :", tuple(attention_mask.shape))

print("input device   :", input_ids.device)
print("labels device  :", labels.device)
print("attention dev  :", attention_mask.device)

assert input_ids.shape[0] == 1
assert input_ids.shape[1] == 4096

assert labels.shape == input_ids.shape
assert attention_mask.shape == input_ids.shape

assert input_ids.device == device
assert labels.device == device
assert attention_mask.device == device

assert next(model.parameters()).device == device

print("\n✓ Micro-batch = 1")
print("✓ Context length = 4096")
print("✓ Model = GPU 1")
print("✓ Batch tensors = GPU 1")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — MICRO-BATCH TRAINING FORWARD + LOSS
# ============================================================

import gc
import torch

gc.collect()
torch.cuda.empty_cache()

model.train()

print("=" * 60)
print("TRAINING FORWARD + LOSS")
print("=" * 60)

outputs_train = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
    use_cache=False,
    return_dict=True,
)

train_loss = outputs_train.loss

print("Loss :", float(train_loss.detach().cpu()))

assert torch.isfinite(train_loss).item()

print("✓ Forward successful")
print("✓ Loss is finite")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — BACKWARD + GRADIENT VERIFICATION
# ============================================================

import gc
import torch

gc.collect()
torch.cuda.empty_cache()

model.train()

# Clear any stale gradients
model.zero_grad(set_to_none=True)

print("=" * 60)
print("BACKWARD + GRADIENT VERIFICATION")
print("=" * 60)

print("Model device :", next(model.parameters()).device)
print("Input device :", input_ids.device)
print("Labels device:", labels.device)

# ------------------------------------------------------------
# Forward
# ------------------------------------------------------------

outputs_train = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
    use_cache=False,
    return_dict=True,
)

train_loss = outputs_train.loss

print("\nLoss :", float(train_loss.detach().cpu()))

assert torch.isfinite(train_loss).item()

# ------------------------------------------------------------
# Backward
# ------------------------------------------------------------

train_loss.backward()

print("✓ Backward completed")

# ------------------------------------------------------------
# Verify gradients
# ------------------------------------------------------------

gradient_count = 0
finite_gradient_count = 0
nonfinite_gradient_count = 0

for name, parameter in model.named_parameters():

    if parameter.grad is not None:

        gradient_count += 1

        if torch.isfinite(parameter.grad).all().item():
            finite_gradient_count += 1
        else:
            nonfinite_gradient_count += 1

print("\nGRADIENT VERIFICATION")
print("-" * 60)
print("Parameters with gradients :", gradient_count)
print("Finite gradients          :", finite_gradient_count)
print("Non-finite gradients      :", nonfinite_gradient_count)

assert gradient_count > 0
assert nonfinite_gradient_count == 0

print("\n✓ Gradients exist")
print("✓ All gradients are finite")
print("✓ Backward pass successful")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — OPTIMIZER STEP VERIFICATION
# ============================================================

import torch
import gc

gc.collect()
torch.cuda.empty_cache()

# Clear previous gradients
model.zero_grad(set_to_none=True)

# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=0.1,
)

print("=" * 60)
print("OPTIMIZER STEP VERIFICATION")
print("=" * 60)

print("Optimizer :", type(optimizer).__name__)
print("LR        :", optimizer.param_groups[0]["lr"])

# ------------------------------------------------------------
# Forward
# ------------------------------------------------------------

outputs_train = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
    use_cache=False,
    return_dict=True,
)

loss_before = outputs_train.loss

assert torch.isfinite(loss_before).item()

# ------------------------------------------------------------
# Backward
# ------------------------------------------------------------

loss_before.backward()

print("Loss before step :", float(loss_before.detach().cpu()))

# Verify gradients exist
grad_count = sum(
    1
    for p in model.parameters()
    if p.grad is not None
)

assert grad_count > 0

# ------------------------------------------------------------
# Optimizer step
# ------------------------------------------------------------

optimizer.step()

# Clear gradients after update
optimizer.zero_grad(set_to_none=True)

print("Gradient tensors :", grad_count)
print("✓ Optimizer step completed")
print("✓ Gradients cleared")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — GRADIENT ACCUMULATION VERIFICATION
# ============================================================

import torch
import gc

gc.collect()
torch.cuda.empty_cache()

model.train()
model.zero_grad(set_to_none=True)

# Keep physical batch size = 1.
# Accumulate several micro-batches before optimizer.step().
GRAD_ACCUM_STEPS = 4

print("=" * 60)
print("GRADIENT ACCUMULATION VERIFICATION")
print("=" * 60)

print("Micro-batch size :", 1)
print("Accumulation     :", GRAD_ACCUM_STEPS)
print("Effective batch  :", GRAD_ACCUM_STEPS)

accumulated_losses = []

# ------------------------------------------------------------
# Accumulate gradients
# ------------------------------------------------------------

for step in range(GRAD_ACCUM_STEPS):

    # Get next streaming example
    batch_acc = next(iter(train_loader))

    ids = batch_acc["input_ids"][:1].to(
        device,
        non_blocking=True
    )

    labs = batch_acc["labels"][:1].to(
        device,
        non_blocking=True
    )

    if "attention_mask" in batch_acc:
        mask = batch_acc["attention_mask"][:1].to(
            device,
            non_blocking=True
        )
    else:
        mask = torch.ones_like(
            ids,
            dtype=torch.long,
            device=device
        )

    # Forward
    outputs_acc = model(
        input_ids=ids,
        attention_mask=mask,
        labels=labs,
        use_cache=False,
        return_dict=True,
    )

    loss_acc = outputs_acc.loss

    assert torch.isfinite(loss_acc).item()

    accumulated_losses.append(
        float(loss_acc.detach().cpu())
    )

    # Normalize loss for accumulation
    scaled_loss = loss_acc / GRAD_ACCUM_STEPS

    # Backward
    scaled_loss.backward()

    # Explicitly release temporary references
    del outputs_acc
    del loss_acc
    del scaled_loss
    del ids
    del labs
    del mask
    del batch_acc

    print(
        f"Micro-step {step + 1}/{GRAD_ACCUM_STEPS} "
        f"| loss={accumulated_losses[-1]:.6f}"
    )

# ------------------------------------------------------------
# Verify gradients exist before optimizer step
# ------------------------------------------------------------

grad_count = 0
nonfinite_count = 0

for parameter in model.parameters():

    if parameter.grad is not None:

        grad_count += 1

        if not torch.isfinite(parameter.grad).all().item():
            nonfinite_count += 1

print("\nGRADIENT STATUS")
print("-" * 60)
print("Gradient tensors :", grad_count)
print("Non-finite       :", nonfinite_count)

assert grad_count > 0
assert nonfinite_count == 0

# ------------------------------------------------------------
# Optimizer step
# ------------------------------------------------------------

optimizer.step()
optimizer.zero_grad(set_to_none=True)

gc.collect()
torch.cuda.empty_cache()

print("\n✓ All accumulation micro-steps completed")
print("✓ Gradients accumulated successfully")
print("✓ Gradients are finite")
print("✓ Optimizer step completed")
print("✓ Gradients cleared")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — POST-OPTIMIZER MEMORY CHECK
# ============================================================

import torch
import psutil
import gc

gc.collect()
torch.cuda.empty_cache()

print("=" * 60)
print("POST-OPTIMIZER MEMORY CHECK")
print("=" * 60)

# GPU 1
free, total = torch.cuda.mem_get_info(device)

print("GPU :", torch.cuda.get_device_name(device))
print(f"GPU total : {total / (1024**3):.2f} GB")
print(f"GPU free  : {free / (1024**3):.2f} GB")
print(f"GPU used  : {(total - free) / (1024**3):.2f} GB")

# RAM
ram = psutil.virtual_memory()

print("\nSYSTEM RAM")
print(f"RAM total     : {ram.total / (1024**3):.2f} GB")
print(f"RAM used      : {ram.used / (1024**3):.2f} GB")
print(f"RAM available : {ram.available / (1024**3):.2f} GB")

assert next(model.parameters()).device == device

print("\n✓ Model remains on GPU 1")
print("✓ Optimizer state initialized")
print("✓ Memory check complete")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — PRODUCTION TRAINING CONFIGURATION
# ============================================================

import os
import gc
import torch

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device("cuda:1")

model = model.to(device)
model.train()

# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

CONTEXT_LENGTH = 4096

MICRO_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 4

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.1

MAX_TRAIN_STEPS = 10

LOG_EVERY = 1
SAVE_EVERY = 10

CHECKPOINT_DIR = "/kaggle/working/phase4_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=WEIGHT_DECAY,
)

print("=" * 60)
print("PRODUCTION TRAINING CONFIGURATION")
print("=" * 60)

print("Device            :", device)
print("Context length    :", CONTEXT_LENGTH)
print("Micro-batch       :", MICRO_BATCH_SIZE)
print("Grad accumulation :", GRAD_ACCUM_STEPS)
print("Effective batch   :", MICRO_BATCH_SIZE * GRAD_ACCUM_STEPS)
print("Learning rate     :", LEARNING_RATE)
print("Weight decay      :", WEIGHT_DECAY)
print("Max steps         :", MAX_TRAIN_STEPS)
print("Checkpoint dir    :", CHECKPOINT_DIR)

assert next(model.parameters()).device == device

print("\n✓ Configuration ready")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — REAL TRAINING SMOKE TEST
# ============================================================

import time
import gc
import torch

model.train()
optimizer.zero_grad(set_to_none=True)

global_step = 0
micro_step = 0

loss_history = []

print("=" * 60)
print("REAL TRAINING SMOKE TEST")
print("=" * 60)

start_time = time.time()

while global_step < MAX_TRAIN_STEPS:

    optimizer.zero_grad(set_to_none=True)

    accumulation_loss = 0.0

    for accum_step in range(GRAD_ACCUM_STEPS):

        batch_train = next(iter(train_loader))

        ids = batch_train["input_ids"][:1].to(
            device,
            non_blocking=True
        )

        labs = batch_train["labels"][:1].to(
            device,
            non_blocking=True
        )

        if "attention_mask" in batch_train:
            mask = batch_train["attention_mask"][:1].to(
                device,
                non_blocking=True
            )
        else:
            mask = torch.ones_like(
                ids,
                dtype=torch.long,
                device=device
            )

        outputs = model(
            input_ids=ids,
            attention_mask=mask,
            labels=labs,
            use_cache=False,
            return_dict=True,
        )

        loss = outputs.loss

        if not torch.isfinite(loss).item():
            raise RuntimeError(
                f"Non-finite loss at step {global_step}"
            )

        accumulation_loss += float(loss.detach().cpu())

        (loss / GRAD_ACCUM_STEPS).backward()

        micro_step += 1

        del outputs
        del loss
        del ids
        del labs
        del mask
        del batch_train

    # --------------------------------------------------------
    # Optimizer update
    # --------------------------------------------------------

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=1.0,
    )

    optimizer.step()

    global_step += 1

    optimizer.zero_grad(set_to_none=True)

    mean_loss = accumulation_loss / GRAD_ACCUM_STEPS
    loss_history.append(mean_loss)

    if global_step % LOG_EVERY == 0:
        elapsed = time.time() - start_time

        print(
            f"step={global_step:03d} | "
            f"loss={mean_loss:.6f} | "
            f"micro_steps={micro_step} | "
            f"time={elapsed:.1f}s"
        )

    gc.collect()
    torch.cuda.empty_cache()

print("=" * 60)
print("✓ REAL TRAINING SMOKE TEST COMPLETE")
print("=" * 60)

print("Optimizer steps :", global_step)
print("Micro steps     :", micro_step)
print("Final loss      :", loss_history[-1])

In [ ]:
# ============================================================
# PHASE 4 — SAVE TRAINING SMOKE-TEST CHECKPOINT
# ============================================================

import os
import torch

checkpoint_path = os.path.join(
    CHECKPOINT_DIR,
    "phase4_smoke_test.pt"
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "global_step": global_step,
        "loss_history": loss_history,
        "context_length": CONTEXT_LENGTH,
        "micro_batch_size": MICRO_BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "learning_rate": LEARNING_RATE,
    },
    checkpoint_path,
)

print("=" * 60)
print("CHECKPOINT SAVED")
print("=" * 60)

print("Path :", checkpoint_path)
print("Size :", f"{os.path.getsize(checkpoint_path) / (1024**3):.2f} GB")

assert os.path.exists(checkpoint_path)

print("✓ Checkpoint exists")
print("✓ Model state saved")
print("✓ Optimizer state saved")
print("✓ Training metadata saved")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — POST-SMOKE-TEST MEMORY VERIFICATION
# ============================================================

import os
import gc
import torch
import psutil

print("=" * 60)
print("POST-SMOKE-TEST MEMORY VERIFICATION")
print("=" * 60)

# ------------------------------------------------------------
# SYSTEM RAM
# ------------------------------------------------------------

ram = psutil.virtual_memory()

print("\nSYSTEM RAM")
print("-" * 60)
print(f"Total     : {ram.total / (1024**3):.2f} GB")
print(f"Used      : {ram.used / (1024**3):.2f} GB")
print(f"Available : {ram.available / (1024**3):.2f} GB")
print(f"Percent   : {ram.percent:.1f}%")

# ------------------------------------------------------------
# GPU MEMORY
# ------------------------------------------------------------

print("\nGPU MEMORY")
print("-" * 60)

if torch.cuda.is_available():

    for i in range(torch.cuda.device_count()):

        total = torch.cuda.get_device_properties(i).total_memory
        allocated = torch.cuda.memory_allocated(i)
        reserved = torch.cuda.memory_reserved(i)
        free = total - allocated

        print(f"\nGPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"Total     : {total / (1024**3):.2f} GB")
        print(f"Free      : {free / (1024**3):.2f} GB")
        print(f"Allocated : {allocated / (1024**3):.2f} GB")
        print(f"Reserved  : {reserved / (1024**3):.2f} GB")

else:
    print("CUDA unavailable")

# ------------------------------------------------------------
# CHECK IMPORTANT OBJECTS
# ------------------------------------------------------------

print("\nCURRENT TRAINING OBJECTS")
print("-" * 60)

for name in [
    "model",
    "optimizer",
    "batch",
    "input_ids",
    "labels",
    "attention_mask",
    "outputs",
    "loss",
    "train_loader",
    "train_dataset",
]:
    exists = name in globals()
    print(f"{name:<18}: {'present' if exists else 'not present'}")

# ------------------------------------------------------------
# CHECK CHECKPOINT
# ------------------------------------------------------------

checkpoint_path = "/kaggle/working/phase4_checkpoints/phase4_smoke_test.pt"

print("\nCHECKPOINT")
print("-" * 60)

if os.path.exists(checkpoint_path):
    size_gb = os.path.getsize(checkpoint_path) / (1024**3)
    print(f"Path : {checkpoint_path}")
    print(f"Size : {size_gb:.2f} GB")
    print("✓ Checkpoint exists")
else:
    print("✗ Checkpoint missing")

print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — CHECKPOINT RELOAD VERIFICATION
# ============================================================

import os
import torch

CHECKPOINT_PATH = "/kaggle/working/phase4_checkpoints/phase4_smoke_test.pt"

print("=" * 60)
print("CHECKPOINT RELOAD VERIFICATION")
print("=" * 60)

assert os.path.exists(CHECKPOINT_PATH), "Checkpoint does not exist"

# ------------------------------------------------------------
# Inspect checkpoint
# ------------------------------------------------------------

print("\nLoading checkpoint...")
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu"
)

print("✓ Checkpoint loaded")

print("\nCHECKPOINT CONTENTS")
print("-" * 60)

if isinstance(checkpoint, dict):
    for key, value in checkpoint.items():

        if isinstance(value, dict):
            print(f"{key:<25}: dict ({len(value)} entries)")

        elif torch.is_tensor(value):
            print(
                f"{key:<25}: tensor "
                f"shape={tuple(value.shape)} "
                f"dtype={value.dtype}"
            )

        else:
            print(
                f"{key:<25}: "
                f"{type(value).__name__} -> {value}"
            )

else:
    print("Checkpoint type:", type(checkpoint))

# ------------------------------------------------------------
# Verify model state
# ------------------------------------------------------------

print("\nMODEL STATE VERIFICATION")
print("-" * 60)

model_state = None

if isinstance(checkpoint, dict):

    for key in [
        "model_state_dict",
        "model_state",
        "state_dict",
    ]:
        if key in checkpoint:
            model_state = checkpoint[key]
            print(f"Found model state under: {key}")
            break

assert model_state is not None, (
    "Could not find model state in checkpoint"
)

print(f"Model parameters in checkpoint: {len(model_state)}")
print("✓ Model state present")

# ------------------------------------------------------------
# Verify optimizer state
# ------------------------------------------------------------

print("\nOPTIMIZER STATE VERIFICATION")
print("-" * 60)

optimizer_state = None

if isinstance(checkpoint, dict):

    for key in [
        "optimizer_state_dict",
        "optimizer_state",
    ]:
        if key in checkpoint:
            optimizer_state = checkpoint[key]
            print(f"Found optimizer state under: {key}")
            break

if optimizer_state is not None:

    if isinstance(optimizer_state, dict):

        print(
            "Optimizer state entries:",
            len(optimizer_state)
        )

    print("✓ Optimizer state present")

else:
    print("⚠ Optimizer state not found")

# ------------------------------------------------------------
# Metadata
# ------------------------------------------------------------

print("\nTRAINING METADATA")
print("-" * 60)

for key in [
    "step",
    "global_step",
    "optimizer_step",
    "micro_step",
    "epoch",
    "loss",
]:

    if isinstance(checkpoint, dict) and key in checkpoint:
        print(f"{key:<20}: {checkpoint[key]}")

# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CHECKPOINT RELOAD VERIFICATION COMPLETE")
print("=" * 60)

print("✓ Checkpoint can be loaded")
print("✓ Model state detected")

if optimizer_state is not None:
    print("✓ Optimizer state detected")

print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — RAM OBJECT AUDIT
# ============================================================

import gc
import sys
import torch
import psutil

print("=" * 60)
print("RAM / OBJECT AUDIT")
print("=" * 60)

# ------------------------------------------------------------
# 1. SYSTEM RAM
# ------------------------------------------------------------

ram_before = psutil.virtual_memory()

print("\nSYSTEM RAM")
print("-" * 60)
print(f"Total     : {ram_before.total / (1024**3):.2f} GB")
print(f"Used      : {ram_before.used / (1024**3):.2f} GB")
print(f"Available : {ram_before.available / (1024**3):.2f} GB")
print(f"Percent   : {ram_before.percent:.1f}%")

# ------------------------------------------------------------
# 2. CUDA MEMORY
# ------------------------------------------------------------

print("\nCUDA MEMORY")
print("-" * 60)

if torch.cuda.is_available():

    for i in range(torch.cuda.device_count()):

        allocated = torch.cuda.memory_allocated(i)
        reserved = torch.cuda.memory_reserved(i)
        total = torch.cuda.get_device_properties(i).total_memory

        print(
            f"GPU {i}: "
            f"allocated={allocated/(1024**3):.2f} GB | "
            f"reserved={reserved/(1024**3):.2f} GB | "
            f"free={(total-allocated)/(1024**3):.2f} GB"
        )

# ------------------------------------------------------------
# 3. LARGE PYTHON OBJECTS
# ------------------------------------------------------------

print("\nLARGE PYTHON OBJECTS")
print("-" * 60)

objects = []

for obj in gc.get_objects():

    try:
        size = sys.getsizeof(obj)

        if size >= 10 * 1024 * 1024:
            objects.append((size, type(obj).__name__))

    except Exception:
        pass

objects.sort(reverse=True)

if objects:

    for size, typename in objects[:30]:

        print(
            f"{size/(1024**2):8.1f} MB | "
            f"{typename}"
        )

else:
    print("No large Python objects found")

# ------------------------------------------------------------
# 4. CURRENT IMPORTANT VARIABLES
# ------------------------------------------------------------

print("\nIMPORTANT VARIABLES")
print("-" * 60)

names = [
    "model",
    "optimizer",
    "batch",
    "input_ids",
    "labels",
    "attention_mask",
    "outputs",
    "loss",
    "train_loader",
    "train_dataset",
    "validation_loader",
    "validation_dataset",
    "dataset",
    "debug_dataset",
    "debug_loader",
    "epoch_dataset",
    "worker_safe_dataset",
]

for name in names:

    if name not in globals():
        print(f"{name:<25}: not present")
        continue

    obj = globals()[name]

    try:
        size = sys.getsizeof(obj)
        size_text = f"{size/(1024**2):.2f} MB"
    except Exception:
        size_text = "unknown"

    print(
        f"{name:<25}: "
        f"{type(obj).__name__:<35} "
        f"{size_text}"
    )

# ------------------------------------------------------------
# 5. TORCH TENSORS CURRENTLY IN PYTHON GC
# ------------------------------------------------------------

print("\nPYTORCH TENSOR AUDIT")
print("-" * 60)

cpu_tensor_count = 0
cpu_tensor_bytes = 0

cuda_tensor_count = 0
cuda_tensor_bytes = 0

for obj in gc.get_objects():

    try:

        if torch.is_tensor(obj):

            nbytes = obj.numel() * obj.element_size()

            if obj.is_cuda:
                cuda_tensor_count += 1
                cuda_tensor_bytes += nbytes

            else:
                cpu_tensor_count += 1
                cpu_tensor_bytes += nbytes

    except Exception:
        pass

print(
    f"CPU tensors  : {cpu_tensor_count} "
    f"({cpu_tensor_bytes/(1024**3):.2f} GB)"
)

print(
    f"CUDA tensors : {cuda_tensor_count} "
    f"({cuda_tensor_bytes/(1024**3):.2f} GB)"
)

# ------------------------------------------------------------
# 6. DATALOADER WORKER STATUS
# ------------------------------------------------------------

print("\nDATALOADER STATUS")
print("-" * 60)

for name in [
    "train_loader",
    "validation_loader",
    "loader",
    "debug_loader",
]:

    if name not in globals():
        continue

    obj = globals()[name]

    print(f"\n{name}")

    try:
        print("  num_workers :", obj.num_workers)
        print("  persistent  :", getattr(obj, "persistent_workers", None))
        print("  batch_size  :", obj.batch_size)
    except Exception as e:
        print("  Could not inspect:", e)

# ------------------------------------------------------------
# 7. DATASET TYPE
# ------------------------------------------------------------

print("\nDATASET STATUS")
print("-" * 60)

for name in [
    "train_dataset",
    "validation_dataset",
    "dataset",
    "epoch_dataset",
    "worker_safe_dataset",
    "debug_dataset",
]:

    if name not in globals():
        continue

    obj = globals()[name]

    print(
        f"{name:<25}: "
        f"{type(obj).__name__}"
    )

# ------------------------------------------------------------
# FINAL
# ------------------------------------------------------------

ram_after = psutil.virtual_memory()

print("\n" + "=" * 60)
print("RAM AUDIT COMPLETE")
print("=" * 60)

print(
    f"RAM used     : "
    f"{ram_after.used/(1024**3):.2f} GB"
)

print(
    f"RAM available: "
    f"{ram_after.available/(1024**3):.2f} GB"
)

print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — REMOVE REDUNDANT DATASET / LOADER OBJECTS
# ============================================================

import gc
import torch
import psutil

print("=" * 60)
print("CLEANING REDUNDANT DATASET / LOADER OBJECTS")
print("=" * 60)

# ------------------------------------------------------------
# BEFORE
# ------------------------------------------------------------

ram = psutil.virtual_memory()

print("\nBEFORE CLEANUP")
print("-" * 60)
print(f"RAM used      : {ram.used / (1024**3):.2f} GB")
print(f"RAM available : {ram.available / (1024**3):.2f} GB")

# ------------------------------------------------------------
# CLOSE REDUNDANT DATALOADER WORKERS
# ------------------------------------------------------------

redundant_loaders = [
    "loader",
    "debug_loader",
]

for name in redundant_loaders:

    if name in globals():

        obj = globals()[name]

        try:
            iterator = getattr(obj, "_iterator", None)

            if iterator is not None:

                shutdown = getattr(
                    iterator,
                    "_shutdown_workers",
                    None
                )

                if shutdown is not None:
                    shutdown()

                obj._iterator = None

            print(f"✓ Closed workers: {name}")

        except Exception as e:
            print(f"⚠ Could not close {name}: {e}")

# ------------------------------------------------------------
# REMOVE REDUNDANT DATASETS
# ------------------------------------------------------------

redundant_datasets = [
    "dataset",
    "debug_dataset",
    "epoch_dataset",
    "worker_safe_dataset",
]

for name in redundant_datasets:

    if name in globals():

        try:
            del globals()[name]
            print(f"✓ Deleted: {name}")

        except Exception as e:
            print(f"⚠ Could not delete {name}: {e}")

# ------------------------------------------------------------
# REMOVE OLD TRAINING BATCH REFERENCES
# ------------------------------------------------------------

for name in [
    "batch",
    "input_ids",
    "labels",
    "attention_mask",
]:

    if name in globals():

        try:
            del globals()[name]
            print(f"✓ Deleted: {name}")

        except Exception as e:
            print(f"⚠ Could not delete {name}: {e}")

# ------------------------------------------------------------
# GARBAGE COLLECTION
# ------------------------------------------------------------

print("\nRunning garbage collection...")

collected = gc.collect()

print(f"GC objects collected : {collected}")

# ------------------------------------------------------------
# CUDA CACHE
# ------------------------------------------------------------

if torch.cuda.is_available():

    for i in range(torch.cuda.device_count()):

        with torch.cuda.device(i):
            torch.cuda.empty_cache()

    print("✓ CUDA cache released")

# ------------------------------------------------------------
# AFTER
# ------------------------------------------------------------

ram = psutil.virtual_memory()

print("\nAFTER CLEANUP")
print("-" * 60)
print(f"RAM used      : {ram.used / (1024**3):.2f} GB")
print(f"RAM available : {ram.available / (1024**3):.2f} GB")
print(f"RAM percent   : {ram.percent:.1f}%")

print("\nGPU MEMORY")
print("-" * 60)

for i in range(torch.cuda.device_count()):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    print(
        f"GPU {i}: "
        f"allocated={allocated/(1024**3):.2f} GB | "
        f"reserved={reserved/(1024**3):.2f} GB"
    )

print("=" * 60)
print("REDUNDANT MEMORY CLEANUP COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — DATALOADER WORKER + PROCESS MEMORY AUDIT
# ============================================================

import os
import gc
import psutil
import torch

print("=" * 60)
print("DATALOADER / PROCESS MEMORY AUDIT")
print("=" * 60)

# ------------------------------------------------------------
# CURRENT PROCESS
# ------------------------------------------------------------

process = psutil.Process(os.getpid())

print("\nMAIN PROCESS")
print("-" * 60)

info = process.memory_info()

print(f"PID        : {process.pid}")
print(f"RSS        : {info.rss / (1024**3):.2f} GB")
print(f"VMS        : {info.vms / (1024**3):.2f} GB")

# ------------------------------------------------------------
# CHILD PROCESSES
# ------------------------------------------------------------

children = process.children(recursive=True)

print("\nCHILD PROCESSES")
print("-" * 60)

if not children:
    print("No child processes found")

else:

    total_child_rss = 0

    for child in children:

        try:
            mem = child.memory_info()
            rss = mem.rss

            total_child_rss += rss

            print(
                f"PID={child.pid:<8} "
                f"RSS={rss/(1024**3):.2f} GB "
                f"status={child.status()}"
            )

        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass

    print("-" * 60)
    print(
        f"TOTAL CHILD RSS : "
        f"{total_child_rss/(1024**3):.2f} GB"
    )

# ------------------------------------------------------------
# DATALOADER WORKER INFORMATION
# ------------------------------------------------------------

print("\nACTIVE DATALOADERS")
print("-" * 60)

for name in [
    "train_loader",
    "validation_loader",
]:

    if name not in globals():
        print(f"{name:<20}: not present")
        continue

    loader_obj = globals()[name]

    print(f"\n{name}")
    print(f"  num_workers        : {loader_obj.num_workers}")
    print(
        f"  persistent_workers : "
        f"{getattr(loader_obj, 'persistent_workers', None)}"
    )
    print(
        f"  prefetch_factor    : "
        f"{getattr(loader_obj, 'prefetch_factor', None)}"
    )
    print(f"  batch_size         : {loader_obj.batch_size}")

    iterator = getattr(loader_obj, "_iterator", None)

    if iterator is None:
        print("  iterator           : not active")

    else:

        workers = getattr(iterator, "_workers", [])

        print(f"  active workers     : {len(workers)}")

        for worker in workers:

            try:

                worker_pid = worker.pid

                try:
                    worker_process = psutil.Process(worker_pid)
                    worker_mem = worker_process.memory_info()

                    print(
                        f"    worker PID={worker_pid:<8} "
                        f"RSS={worker_mem.rss/(1024**3):.2f} GB"
                    )

                except Exception:
                    print(
                        f"    worker PID={worker_pid:<8} "
                        f"memory unavailable"
                    )

            except Exception:
                pass

# ------------------------------------------------------------
# PYTORCH MEMORY SUMMARY
# ------------------------------------------------------------

print("\nPYTORCH GPU MEMORY")
print("-" * 60)

if torch.cuda.is_available():

    for i in range(torch.cuda.device_count()):

        print(f"\nGPU {i}")

        print(
            f"  allocated : "
            f"{torch.cuda.memory_allocated(i)/(1024**3):.2f} GB"
        )

        print(
            f"  reserved  : "
            f"{torch.cuda.memory_reserved(i)/(1024**3):.2f} GB"
        )

# ------------------------------------------------------------
# SYSTEM RAM
# ------------------------------------------------------------

ram = psutil.virtual_memory()

print("\nSYSTEM RAM")
print("-" * 60)
print(f"Used      : {ram.used/(1024**3):.2f} GB")
print(f"Available : {ram.available/(1024**3):.2f} GB")
print(f"Percent   : {ram.percent:.1f}%")

print("=" * 60)
print("MEMORY AUDIT COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — SHUT DOWN PERSISTENT DATALOADER WORKERS
# ============================================================

import gc
import time
import psutil
import torch

print("=" * 60)
print("SHUTTING DOWN DATALOADER WORKERS")
print("=" * 60)

# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

def shutdown_loader_workers(loader, name):

    if loader is None:
        print(f"{name}: loader is None")
        return

    iterator = getattr(loader, "_iterator", None)

    if iterator is None:
        print(f"{name}: no active iterator")
        return

    workers = getattr(iterator, "_workers", [])

    print(f"{name}: shutting down {len(workers)} workers")

    shutdown = getattr(
        iterator,
        "_shutdown_workers",
        None
    )

    if shutdown is not None:
        try:
            shutdown()
            print(f"✓ {name} workers shut down")
        except Exception as e:
            print(f"⚠ {name}: {e}")

    try:
        loader._iterator = None
    except Exception:
        pass


# ------------------------------------------------------------
# Shutdown production loaders
# ------------------------------------------------------------

if "train_loader" in globals():
    shutdown_loader_workers(
        train_loader,
        "train_loader"
    )

if "validation_loader" in globals():
    shutdown_loader_workers(
        validation_loader,
        "validation_loader"
    )

# ------------------------------------------------------------
# Garbage collection
# ------------------------------------------------------------

gc.collect()

# ------------------------------------------------------------
# Give OS time to reap workers
# ------------------------------------------------------------

time.sleep(3)

# ------------------------------------------------------------
# Memory after shutdown
# ------------------------------------------------------------

process = psutil.Process()

ram = psutil.virtual_memory()

print("\nMEMORY AFTER WORKER SHUTDOWN")
print("-" * 60)

print(
    f"Main RSS       : "
    f"{process.memory_info().rss/(1024**3):.2f} GB"
)

print(
    f"RAM used       : "
    f"{ram.used/(1024**3):.2f} GB"
)

print(
    f"RAM available  : "
    f"{ram.available/(1024**3):.2f} GB"
)

print(
    f"RAM percent    : "
    f"{ram.percent:.1f}%"
)

# ------------------------------------------------------------
# Remaining child processes
# ------------------------------------------------------------

children = process.children(recursive=True)

print("\nREMAINING CHILD PROCESSES")
print("-" * 60)

if not children:
    print("✓ No child processes remain")

else:

    for child in children:

        try:

            mem = child.memory_info()

            print(
                f"PID={child.pid:<8} "
                f"RSS={mem.rss/(1024**3):.2f} GB "
                f"status={child.status()}"
            )

        except Exception:
            pass

# ------------------------------------------------------------
# GPU
# ------------------------------------------------------------

print("\nGPU MEMORY")
print("-" * 60)

for i in range(torch.cuda.device_count()):

    print(
        f"GPU {i}: "
        f"allocated="
        f"{torch.cuda.memory_allocated(i)/(1024**3):.2f} GB | "
        f"reserved="
        f"{torch.cuda.memory_reserved(i)/(1024**3):.2f} GB"
    )

print("=" * 60)
print("WORKER SHUTDOWN COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — POST-WORKER RAM RETENTION AUDIT
# ============================================================

import os
import gc
import psutil
import torch

print("=" * 60)
print("POST-WORKER RAM RETENTION AUDIT")
print("=" * 60)

# ------------------------------------------------------------
# SYSTEM RAM
# ------------------------------------------------------------

vm = psutil.virtual_memory()

print("\nSYSTEM RAM")
print("-" * 60)
print(f"Total     : {vm.total / 1024**3:.2f} GB")
print(f"Used      : {vm.used / 1024**3:.2f} GB")
print(f"Available : {vm.available / 1024**3:.2f} GB")
print(f"Percent   : {vm.percent:.1f}%")

# ------------------------------------------------------------
# MAIN PROCESS
# ------------------------------------------------------------

proc = psutil.Process(os.getpid())
rss = proc.memory_info().rss

print("\nMAIN PROCESS")
print("-" * 60)
print(f"PID : {os.getpid()}")
print(f"RSS : {rss / 1024**3:.2f} GB")

# ------------------------------------------------------------
# CHILD PROCESSES
# ------------------------------------------------------------

children = proc.children(recursive=True)

print("\nCHILD PROCESSES")
print("-" * 60)

if not children:
    print("✓ No child processes remain")
else:
    total_child_rss = 0

    for child in children:
        try:
            mem = child.memory_info().rss
            total_child_rss += mem

            print(
                f"PID={child.pid} | "
                f"RSS={mem / 1024**3:.2f} GB | "
                f"status={child.status()}"
            )

        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass

    print("-" * 60)
    print(f"Total child RSS : {total_child_rss / 1024**3:.2f} GB")

# ------------------------------------------------------------
# IMPORTANT PYTHON VARIABLES
# ------------------------------------------------------------

print("\nIMPORTANT VARIABLES")
print("-" * 60)

names = [
    "model",
    "optimizer",
    "train_loader",
    "validation_loader",
    "loader",
    "debug_loader",
    "train_dataset",
    "validation_dataset",
    "dataset",
    "debug_dataset",
    "epoch_dataset",
    "worker_safe_dataset",
    "batch",
    "input_ids",
    "labels",
    "attention_mask",
    "outputs",
    "loss",
]

for name in names:

    if name not in globals():
        print(f"{name:25s}: NOT PRESENT")
        continue

    obj = globals()[name]

    if isinstance(obj, torch.Tensor):

        size = obj.numel() * obj.element_size()

        print(
            f"{name:25s}: Tensor | "
            f"{size / 1024**2:.2f} MB | "
            f"shape={tuple(obj.shape)} | "
            f"device={obj.device}"
        )

    else:

        print(
            f"{name:25s}: "
            f"{type(obj).__name__}"
        )

# ------------------------------------------------------------
# PYTORCH TENSOR INVENTORY
# ------------------------------------------------------------

print("\nPYTORCH TENSOR INVENTORY")
print("-" * 60)

gc.collect()

cpu_count = 0
cpu_bytes = 0

cuda_count = 0
cuda_bytes = 0

for obj in gc.get_objects():

    try:

        if not isinstance(obj, torch.Tensor):
            continue

        size = obj.numel() * obj.element_size()

        if obj.device.type == "cpu":

            cpu_count += 1
            cpu_bytes += size

        elif obj.device.type == "cuda":

            cuda_count += 1
            cuda_bytes += size

    except Exception:
        pass

print(
    f"CPU tensors  : {cpu_count:,} "
    f"({cpu_bytes / 1024**3:.2f} GB)"
)

print(
    f"CUDA tensors : {cuda_count:,} "
    f"({cuda_bytes / 1024**3:.2f} GB)"
)

# ------------------------------------------------------------
# GPU MEMORY
# ------------------------------------------------------------

print("\nGPU MEMORY")
print("-" * 60)

if torch.cuda.is_available():

    for i in range(torch.cuda.device_count()):

        allocated = torch.cuda.memory_allocated(i)
        reserved = torch.cuda.memory_reserved(i)
        total = torch.cuda.get_device_properties(i).total_memory

        print(f"\nGPU {i}")
        print(
            f"  allocated : "
            f"{allocated / 1024**3:.2f} GB"
        )
        print(
            f"  reserved  : "
            f"{reserved / 1024**3:.2f} GB"
        )
        print(
            f"  free      : "
            f"{(total - reserved) / 1024**3:.2f} GB"
        )

print("=" * 60)
print("POST-WORKER AUDIT COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — REMOVE UNUSED DATASET / DATALOADER REFERENCES
# ============================================================

import gc
import psutil
import torch

print("=" * 60)
print("REMOVING UNUSED DATASET / DATALOADER REFERENCES")
print("=" * 60)

vm_before = psutil.virtual_memory()

print("\nBEFORE")
print("-" * 60)
print(f"RAM used      : {vm_before.used / 1024**3:.2f} GB")
print(f"RAM available : {vm_before.available / 1024**3:.2f} GB")

# ------------------------------------------------------------
# These are NOT needed for the current training state.
# train_loader / validation_loader can be recreated later.
# ------------------------------------------------------------

remove_names = [
    "validation_dataset",
    "train_loader",
    "validation_loader",
    "loader",
    "debug_loader",
]

for name in remove_names:

    if name in globals():

        obj = globals()[name]

        try:
            del globals()[name]
            print(f"✓ Deleted: {name}")

        except Exception as e:
            print(f"! Could not delete {name}: {e}")

    else:
        print(f"- Not present: {name}")

# ------------------------------------------------------------
# Garbage collection
# ------------------------------------------------------------

print("\nGARBAGE COLLECTION")
print("-" * 60)

collected = gc.collect()

print(f"GC objects collected : {collected}")

# ------------------------------------------------------------
# CUDA allocator cleanup
# ------------------------------------------------------------

if torch.cuda.is_available():

    for i in range(torch.cuda.device_count()):
        with torch.cuda.device(i):
            torch.cuda.empty_cache()

    print("✓ CUDA cache released")

# ------------------------------------------------------------
# AFTER
# ------------------------------------------------------------

vm_after = psutil.virtual_memory()

print("\nAFTER")
print("-" * 60)
print(f"RAM used      : {vm_after.used / 1024**3:.2f} GB")
print(f"RAM available : {vm_after.available / 1024**3:.2f} GB")
print(f"RAM percent   : {vm_after.percent:.1f}%")

print("\nGPU MEMORY")
print("-" * 60)

for i in range(torch.cuda.device_count()):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB"
    )

print("=" * 60)
print("REFERENCE CLEANUP COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — MAIN PROCESS MEMORY SOURCE AUDIT
# ============================================================

import os
import gc
import psutil
import torch

print("=" * 60)
print("MAIN PROCESS MEMORY SOURCE AUDIT")
print("=" * 60)

proc = psutil.Process(os.getpid())

# ------------------------------------------------------------
# PROCESS MEMORY
# ------------------------------------------------------------

mem = proc.memory_info()

print("\nPROCESS MEMORY")
print("-" * 60)
print(f"RSS : {mem.rss / 1024**3:.2f} GB")
print(f"VMS : {mem.vms / 1024**3:.2f} GB")

# ------------------------------------------------------------
# MODEL PARAMETER MEMORY
# ------------------------------------------------------------

print("\nMODEL MEMORY")
print("-" * 60)

model_param_bytes = 0
model_buffer_bytes = 0

param_count = 0
buffer_count = 0

for p in model.parameters():
    model_param_bytes += p.numel() * p.element_size()
    param_count += 1

for b in model.buffers():
    model_buffer_bytes += b.numel() * b.element_size()
    buffer_count += 1

print(f"Parameters : {param_count:,}")
print(
    f"Parameter memory : "
    f"{model_param_bytes / 1024**3:.2f} GB"
)

print(f"Buffers : {buffer_count:,}")
print(
    f"Buffer memory : "
    f"{model_buffer_bytes / 1024**3:.2f} GB"
)

# ------------------------------------------------------------
# OPTIMIZER MEMORY
# ------------------------------------------------------------

print("\nOPTIMIZER MEMORY")
print("-" * 60)

optimizer_tensor_bytes = 0
optimizer_tensor_count = 0

for state in optimizer.state.values():

    for value in state.values():

        if torch.is_tensor(value):

            optimizer_tensor_bytes += (
                value.numel() * value.element_size()
            )

            optimizer_tensor_count += 1

print(
    f"Optimizer tensors : "
    f"{optimizer_tensor_count:,}"
)

print(
    f"Optimizer tensor memory : "
    f"{optimizer_tensor_bytes / 1024**3:.2f} GB"
)

# ------------------------------------------------------------
# CUDA MEMORY
# ------------------------------------------------------------

print("\nCUDA MEMORY")
print("-" * 60)

for i in range(torch.cuda.device_count()):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB"
    )

# ------------------------------------------------------------
# SYSTEM RAM
# ------------------------------------------------------------

vm = psutil.virtual_memory()

print("\nSYSTEM RAM")
print("-" * 60)
print(f"Used      : {vm.used / 1024**3:.2f} GB")
print(f"Available : {vm.available / 1024**3:.2f} GB")
print(f"Percent   : {vm.percent:.1f}%")

# ------------------------------------------------------------
# MEMORY MAP SUMMARY
# ------------------------------------------------------------

print("\nPROCESS MEMORY MAP")
print("-" * 60)

try:

    maps = proc.memory_maps(grouped=True)

    maps_sorted = sorted(
        maps,
        key=lambda x: x.rss,
        reverse=True
    )

    for entry in maps_sorted[:20]:

        print(
            f"{entry.rss / 1024**2:8.1f} MB | "
            f"{entry.path}"
        )

except Exception as e:

    print(f"Could not inspect memory maps: {e}")

print("=" * 60)
print("MAIN PROCESS MEMORY AUDIT COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — ANONYMOUS MEMORY DIAGNOSTIC
# ============================================================

import os
import psutil
import gc
import torch

print("=" * 60)
print("ANONYMOUS MEMORY DIAGNOSTIC")
print("=" * 60)

proc = psutil.Process(os.getpid())

# ------------------------------------------------------------
# BEFORE
# ------------------------------------------------------------

before = proc.memory_info().rss

print("\nBEFORE")
print("-" * 60)
print(f"Process RSS : {before / 1024**3:.2f} GB")

# ------------------------------------------------------------
# PYTHON GC
# ------------------------------------------------------------

print("\nPYTHON GC")
print("-" * 60)

print(f"Generation counts before : {gc.get_count()}")

collected = gc.collect()

print(f"Objects collected        : {collected}")
print(f"Generation counts after  : {gc.get_count()}")

# ------------------------------------------------------------
# PYTORCH CUDA CACHE
# ------------------------------------------------------------

if torch.cuda.is_available():

    print("\nCUDA CACHE")
    print("-" * 60)

    for i in range(torch.cuda.device_count()):

        before_alloc = torch.cuda.memory_allocated(i)
        before_reserved = torch.cuda.memory_reserved(i)

        with torch.cuda.device(i):
            torch.cuda.empty_cache()

        after_alloc = torch.cuda.memory_allocated(i)
        after_reserved = torch.cuda.memory_reserved(i)

        print(f"GPU {i}")
        print(
            f"  allocated : "
            f"{before_alloc / 1024**3:.2f} -> "
            f"{after_alloc / 1024**3:.2f} GB"
        )
        print(
            f"  reserved  : "
            f"{before_reserved / 1024**3:.2f} -> "
            f"{after_reserved / 1024**3:.2f} GB"
        )

# ------------------------------------------------------------
# AFTER
# ------------------------------------------------------------

after = proc.memory_info().rss

print("\nAFTER")
print("-" * 60)
print(f"Process RSS : {after / 1024**3:.2f} GB")

delta = after - before

print(
    f"RSS change  : "
    f"{delta / 1024**3:+.2f} GB"
)

# ------------------------------------------------------------
# SYSTEM
# ------------------------------------------------------------

vm = psutil.virtual_memory()

print("\nSYSTEM RAM")
print("-" * 60)
print(f"Used      : {vm.used / 1024**3:.2f} GB")
print(f"Available : {vm.available / 1024**3:.2f} GB")
print(f"Percent   : {vm.percent:.1f}%")

# ------------------------------------------------------------
# CHILDREN
# ------------------------------------------------------------

children = proc.children(recursive=True)

print("\nCHILD PROCESSES")
print("-" * 60)

if len(children) == 0:
    print("✓ No child processes")
else:
    print(f"WARNING: {len(children)} child process(es) remain")

print("=" * 60)
print("ANONYMOUS MEMORY DIAGNOSTIC COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================
# DEEP ANONYMOUS MEMORY DIAGNOSTIC
# ============================================================

import os
import re
import psutil

print("=" * 60)
print("DEEP ANONYMOUS MEMORY DIAGNOSTIC")
print("=" * 60)

pid = os.getpid()
proc = psutil.Process(pid)

print("\nPROCESS")
print("-" * 60)
print(f"PID  : {pid}")
print(f"RSS  : {proc.memory_info().rss / (1024**3):.2f} GB")
print(f"VMS  : {proc.memory_info().vms / (1024**3):.2f} GB")

# ------------------------------------------------------------
# Parse /proc/<pid>/smaps_rollup
# ------------------------------------------------------------

print("\nSMAPS ROLLUP")
print("-" * 60)

try:
    with open(f"/proc/{pid}/smaps_rollup", "r") as f:
        smaps = f.read()

    fields = {}

    for line in smaps.splitlines():
        if ":" not in line:
            continue

        key, value = line.split(":", 1)
        value = value.strip()

        if value.endswith(" kB"):
            try:
                fields[key] = int(value.split()[0]) / (1024**2)
            except ValueError:
                pass

    for key in [
        "Rss",
        "Pss",
        "Private_Clean",
        "Private_Dirty",
        "Shared_Clean",
        "Shared_Dirty",
        "Anonymous",
        "AnonHugePages",
        "Swap",
    ]:
        if key in fields:
            print(f"{key:20s}: {fields[key]:8.2f} GB")

except Exception as e:
    print("smaps_rollup unavailable:", repr(e))

# ------------------------------------------------------------
# Anonymous mappings from smaps
# ------------------------------------------------------------

print("\nLARGEST ANONYMOUS MAPPINGS")
print("-" * 60)

try:
    mappings = []

    with open(f"/proc/{pid}/smaps", "r") as f:
        current_header = None
        current_rss = 0
        current_private = 0
        current_anon = 0

        def save_mapping():
            if current_header is not None:
                mappings.append(
                    (
                        current_anon,
                        current_private,
                        current_rss,
                        current_header,
                    )
                )

        for line in f:

            # New mapping
            if re.match(r"^[0-9a-f]+-[0-9a-f]+ ", line):
                save_mapping()

                current_header = line.strip()
                current_rss = 0
                current_private = 0
                current_anon = 0

            elif line.startswith("Rss:"):
                current_rss = int(line.split()[1]) / (1024**2)

            elif line.startswith("Private_Dirty:"):
                current_private += int(line.split()[1]) / (1024**2)

            elif line.startswith("Private_Clean:"):
                current_private += int(line.split()[1]) / (1024**2)

            elif line.startswith("Anonymous:"):
                current_anon = int(line.split()[1]) / (1024**2)

        save_mapping()

    mappings.sort(reverse=True)

    shown = 0

    for anon, private, rss, header in mappings:
        if anon < 50:
            continue

        print(
            f"Anonymous={anon:8.1f} MB | "
            f"Private={private:8.1f} MB | "
            f"RSS={rss:8.1f} MB | "
            f"{header}"
        )

        shown += 1

        if shown >= 30:
            break

    if shown == 0:
        print("No anonymous mapping larger than 50 MB found.")

except Exception as e:
    print("smaps parsing failed:", repr(e))

# ------------------------------------------------------------
# Final system state
# ------------------------------------------------------------

print("\nSYSTEM RAM")
print("-" * 60)

ram = psutil.virtual_memory()

print(f"Total     : {ram.total / (1024**3):.2f} GB")
print(f"Used      : {ram.used / (1024**3):.2f} GB")
print(f"Available : {ram.available / (1024**3):.2f} GB")
print(f"Percent   : {ram.percent:.1f}%")

print("\n" + "=" * 60)
print("DEEP MEMORY DIAGNOSTIC COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — PRODUCTION TRAINING CONFIG
# ============================================================

import os
import gc
import time
import torch

# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

NUM_TRAIN_STEPS = 100
GRADIENT_ACCUMULATION_STEPS = 4

BATCH_SIZE = 2
CONTEXT_LENGTH = 4096

LOG_EVERY = 10
SAVE_EVERY = 50

# Keep CPU memory conservative
NUM_WORKERS = 0
PERSISTENT_WORKERS = False

CHECKPOINT_DIR = "/kaggle/working/phase4_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model = model.to(device)
model.train()

print("=" * 60)
print("PHASE 4 — PRODUCTION TRAINING CONFIG")
print("=" * 60)

print("Device                 :", device)
print("Batch size             :", BATCH_SIZE)
print("Context length         :", CONTEXT_LENGTH)
print("Gradient accumulation  :", GRADIENT_ACCUMULATION_STEPS)
print("Optimizer steps        :", NUM_TRAIN_STEPS)
print("Micro steps            :", NUM_TRAIN_STEPS * GRADIENT_ACCUMULATION_STEPS)
print("DataLoader workers     :", NUM_WORKERS)
print("Persistent workers     :", PERSISTENT_WORKERS)
print("Checkpoint directory   :", CHECKPOINT_DIR)

print("=" * 60)
print("✓ Configuration ready")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — RESTORE TRAINING PATHS
# ============================================================

import os

CHECKPOINT = "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"

TRAIN_MANIFEST = os.path.join(
    CHECKPOINT,
    "manifests",
    "train_manifest.jsonl"
)

TRAIN_INDEX = "/kaggle/working/phase4_streaming_index/train_source_index.jsonl"

TOKENIZER_PATH = os.path.join(
    CHECKPOINT,
    "tokenizer",
    "legal_bpe_32k.json"
)

print("=" * 60)
print("PHASE 4 — TRAINING PATH VERIFICATION")
print("=" * 60)

print("TRAIN_MANIFEST :", TRAIN_MANIFEST)
print("TRAIN_INDEX    :", TRAIN_INDEX)
print("TOKENIZER      :", TOKENIZER_PATH)

assert os.path.isfile(TRAIN_MANIFEST), \
    "Train manifest not found"

assert os.path.isfile(TRAIN_INDEX), \
    "Train source index not found"

assert os.path.isfile(TOKENIZER_PATH), \
    "Tokenizer not found"

print()
print("✓ Train manifest exists")
print("✓ Train source index exists")
print("✓ Tokenizer exists")
print("=" * 60)
print("PATHS READY")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — PRODUCTION TRAIN DATASET / LOADER
# ============================================================

import gc
import torch
from torch.utils.data import DataLoader

# ------------------------------------------------------------
# Recreate only the training dataset
# ------------------------------------------------------------

gc.collect()

train_dataset = WorkerSafeDocumentAwarePacker(
    manifest_path=TRAIN_MANIFEST,
    index_path=TRAIN_INDEX,
    tokenizer=tokenizer,
    context_length=CONTEXT_LENGTH,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    persistent_workers=PERSISTENT_WORKERS,
    pin_memory=True,
)

print("=" * 60)
print("PRODUCTION TRAIN LOADER")
print("=" * 60)

print("Dataset :", type(train_dataset).__name__)
print("Loader  :", type(train_loader).__name__)
print("Workers :", NUM_WORKERS)
print("Batch   :", BATCH_SIZE)

print("=" * 60)
print("✓ Production train loader ready")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — PRODUCTION BATCH VERIFICATION
# ============================================================

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device, non_blocking=True)
labels = batch["labels"].to(device, non_blocking=True)
attention_mask = batch["attention_mask"].to(device, non_blocking=True)

print("=" * 60)
print("PRODUCTION BATCH VERIFICATION")
print("=" * 60)

print("input_ids      :", tuple(input_ids.shape))
print("labels         :", tuple(labels.shape))
print("attention_mask :", tuple(attention_mask.shape))

print("input device   :", input_ids.device)
print("labels device  :", labels.device)
print("attention      :", attention_mask.device)

assert input_ids.shape == (BATCH_SIZE, CONTEXT_LENGTH)
assert labels.shape == (BATCH_SIZE, CONTEXT_LENGTH)
assert attention_mask.shape == (BATCH_SIZE, CONTEXT_LENGTH)

assert input_ids.device == device
assert labels.device == device
assert attention_mask.device == device

print("✓ Batch shape correct")
print("✓ Batch device correct")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — GPU MEMORY RECOVERY AFTER BACKWARD OOM
# ============================================================

import gc
import torch

print("=" * 60)
print("RECOVERING GPU MEMORY AFTER BACKWARD OOM")
print("=" * 60)

# Remove failed step objects if they exist
for name in [
    "outputs",
    "loss",
    "scaled_loss",
    "input_ids",
    "labels",
    "attention_mask",
    "batch",
]:
    if name in globals():
        try:
            del globals()[name]
            print("✓ Deleted:", name)
        except Exception:
            pass

gc.collect()

for i in range(torch.cuda.device_count()):
    torch.cuda.set_device(i)
    torch.cuda.empty_cache()

print()
print("GPU MEMORY")
print("-" * 60)

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)

    print(
        f"GPU {i}: "
        f"free={free / 1024**3:.2f} GB | "
        f"total={total / 1024**3:.2f} GB"
    )

print("=" * 60)
print("✓ GPU cleanup complete")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — ENABLE MEMORY-SAFE TRAINING
# ============================================================

print("=" * 60)
print("ENABLING MEMORY-SAFE TRAINING")
print("=" * 60)

# Disable KV cache during training
model.config.use_cache = False

# Gradient checkpointing
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={
            "use_reentrant": False
        }
    )
    print("✓ Gradient checkpointing enabled")
else:
    raise RuntimeError(
        "Model does not support gradient checkpointing"
    )

# Make sure model is in training mode
model.train()

print("✓ use_cache =", model.config.use_cache)
print("✓ model.train() =", model.training)

print("=" * 60)
print("MEMORY-SAFE TRAINING READY")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — CLEAN FAILED BACKWARD STATE
# ============================================================

import gc
import torch

print("=" * 60)
print("CLEANING FAILED BACKWARD STATE")
print("=" * 60)

for name in [
    "outputs",
    "loss",
    "scaled_loss",
    "input_ids",
    "labels",
    "attention_mask",
    "batch",
]:
    if name in globals():
        try:
            del globals()[name]
            print("✓ Deleted:", name)
        except Exception:
            pass

if "optimizer" in globals():
    optimizer.zero_grad(set_to_none=True)

gc.collect()

for i in range(torch.cuda.device_count()):
    with torch.cuda.device(i):
        torch.cuda.empty_cache()

print("\nGPU MEMORY")
print("-" * 60)

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    print(
        f"GPU {i}: "
        f"free={free / 1024**3:.2f} GB | "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB"
    )

print("=" * 60)
print("✓ CLEANUP COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — MEMORY-SAFE MICRO-BATCH CONFIGURATION
# ============================================================

BATCH_SIZE = 1

# Keep the same effective batch size as before:
# old: batch 2 × accumulation 4 = effective 8
# new: batch 1 × accumulation 8 = effective 8

GRADIENT_ACCUMULATION_STEPS = 8

CONTEXT_LENGTH = 4096

print("=" * 60)
print("MEMORY-SAFE TRAINING CONFIG")
print("=" * 60)

print("Micro-batch size       :", BATCH_SIZE)
print("Gradient accumulation  :", GRADIENT_ACCUMULATION_STEPS)
print("Context length         :", CONTEXT_LENGTH)
print(
    "Effective batch tokens :",
    BATCH_SIZE
    * GRADIENT_ACCUMULATION_STEPS
    * CONTEXT_LENGTH
)

print("=" * 60)
print("✓ Effective batch size preserved")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — REBUILD MEMORY-SAFE TRAIN LOADER
# ============================================================

import gc
from torch.utils.data import DataLoader

gc.collect()

# Remove old loader/dataset references if present
if "train_loader" in globals():
    try:
        del train_loader
    except Exception:
        pass

gc.collect()

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    num_workers=0,
    pin_memory=True,
)

print("=" * 60)
print("MEMORY-SAFE TRAIN LOADER")
print("=" * 60)

print("Batch size :", 1)
print("Workers    :", 0)
print("Pin memory :", True)

print("=" * 60)
print("✓ Loader ready")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — SINGLE BATCH BACKWARD TEST
# ============================================================

import gc
import torch

model.train()
model.config.use_cache = False

optimizer.zero_grad(set_to_none=True)

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(
    device,
    non_blocking=True
)

labels = batch["labels"].to(
    device,
    non_blocking=True
)

attention_mask = batch["attention_mask"].to(
    device,
    non_blocking=True
)

print("=" * 60)
print("SINGLE BATCH BACKWARD TEST")
print("=" * 60)

print("input shape:", tuple(input_ids.shape))

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
    use_cache=False,
)

loss = outputs.loss

print("Forward loss:", float(loss.detach()))

(loss / GRADIENT_ACCUMULATION_STEPS).backward()

print("✓ Backward successful")

optimizer.zero_grad(set_to_none=True)

del outputs
del loss
del input_ids
del labels
del attention_mask
del batch

gc.collect()
torch.cuda.empty_cache()

print("=" * 60)
print("✓ SINGLE BATCH BACKWARD TEST PASSED")
print("=" * 60)

In [ ]:
# ============================================================
# SINGLE-BATCH OPTIMIZER STEP TEST
# ============================================================

import gc
import torch

print("=" * 60)
print("SINGLE-BATCH OPTIMIZER STEP TEST")
print("=" * 60)

# Clean any leftover test state
for name in ["outputs", "loss", "scaled_loss"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

# Make sure training mode is enabled
model.train()

# ------------------------------------------------------------
# Get exactly ONE batch
# ------------------------------------------------------------

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device, non_blocking=True)
labels = batch["labels"].to(device, non_blocking=True)
attention_mask = batch["attention_mask"].to(device, non_blocking=True)

print("Input shape :", tuple(input_ids.shape))
print("Device      :", input_ids.device)

# ------------------------------------------------------------
# Clear existing gradients
# ------------------------------------------------------------

optimizer.zero_grad(set_to_none=True)

# ------------------------------------------------------------
# Forward
# ------------------------------------------------------------

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
    use_cache=False,
)

loss = outputs.loss

print("Forward loss:", float(loss.detach().cpu()))

# ------------------------------------------------------------
# Backward
# ------------------------------------------------------------

loss.backward()

print("✓ Backward successful")

# ------------------------------------------------------------
# Optimizer step
# ------------------------------------------------------------

optimizer.step()

print("✓ Optimizer step successful")

# ------------------------------------------------------------
# Cleanup batch/test outputs
# ------------------------------------------------------------

del outputs
del loss
del input_ids
del labels
del attention_mask
del batch

gc.collect()
torch.cuda.empty_cache()

print()
print("=" * 60)
print("✓ SINGLE-BATCH OPTIMIZER STEP PASSED")
print("=" * 60)

In [ ]:
# ============================================================
# OPTIMIZER STATE DEVICE / DTYPE AUDIT
# ============================================================

import torch
from collections import Counter

print("=" * 60)
print("OPTIMIZER STATE DEVICE / DTYPE AUDIT")
print("=" * 60)

model_device = next(model.parameters()).device

print("Model device:", model_device)
print()

param_devices = Counter()
param_dtypes = Counter()

for p in model.parameters():
    param_devices[str(p.device)] += 1
    param_dtypes[str(p.dtype)] += 1

print("MODEL PARAMETERS")
print("-" * 60)
print("Devices:", dict(param_devices))
print("Dtypes :", dict(param_dtypes))

print()
print("OPTIMIZER STATE")
print("-" * 60)

state_devices = Counter()
state_dtypes = Counter()
state_count = 0

mixed_state_examples = []

for param, state in optimizer.state.items():

    for name, value in state.items():

        if torch.is_tensor(value):

            state_count += 1

            state_devices[str(value.device)] += 1
            state_dtypes[str(value.dtype)] += 1

            if (
                value.device != model_device
                and len(mixed_state_examples) < 10
            ):
                mixed_state_examples.append(
                    (name, str(value.device), str(value.dtype), tuple(value.shape))
                )

print("Tensor states:", state_count)
print("Devices      :", dict(state_devices))
print("Dtypes       :", dict(state_dtypes))

print()
print("MISMATCHED STATE EXAMPLES")
print("-" * 60)

if mixed_state_examples:
    for item in mixed_state_examples:
        print(
            f"name={item[0]} | "
            f"device={item[1]} | "
            f"dtype={item[2]} | "
            f"shape={item[3]}"
        )
else:
    print("✓ No optimizer-state device mismatch found")

print()
print("=" * 60)
print("AUDIT COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — POST-RECONNECT ARTIFACT DISCOVERY
# ============================================================

from pathlib import Path

print("=" * 60)
print("PHASE 4 — POST-RECONNECT ARTIFACT DISCOVERY")
print("=" * 60)

SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

patterns = [
    "*source_index*",
    "*manifest*",
    "*checkpoint*",
    "*.pt",
    "*.pth",
    "*.json",
    "*.jsonl",
    "*.jsonl.gz",
]

for root in SEARCH_ROOTS:

    print()
    print("=" * 60)
    print(f"SEARCHING: {root}")
    print("=" * 60)

    if not root.exists():
        print("Directory does not exist.")
        continue

    found = set()

    for pattern in patterns:
        try:
            for path in root.rglob(pattern):
                if path.is_file():
                    found.add(path)
        except Exception as e:
            print(f"Search error for {pattern}: {e}")

    if not found:
        print("No matching artifacts found.")
        continue

    for path in sorted(found):

        try:
            size_gb = path.stat().st_size / (1024 ** 3)
        except Exception:
            size_gb = 0.0

        print(
            f"{size_gb:8.3f} GB | {path}"
        )

print()
print("=" * 60)
print("DISCOVERY COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — POST-RECONNECT TRAINING RECOVERY
# ============================================================

import os
import json
import gc
import torch

BASE = "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"

TRAIN_MANIFEST = os.path.join(BASE, "manifests", "train_manifest.jsonl")
VAL_MANIFEST   = os.path.join(BASE, "manifests", "validation_manifest.jsonl")

TOKENIZER_PATH = os.path.join(BASE, "tokenizer", "legal_bpe_32k.json")
TOKENIZER_CONFIG = os.path.join(BASE, "tokenizer", "tokenizer_config.json")

TRAINING_CONFIG_PATH = os.path.join(BASE, "training_example_config.json")
INIT_CONFIG_PATH = os.path.join(BASE, "phase4_initialization.json")
STREAMING_CONFIG_PATH = os.path.join(BASE, "streaming_config.json")

print("=" * 60)
print("PHASE 4 — RECOVERY PATH VERIFICATION")
print("=" * 60)

paths = {
    "TRAIN_MANIFEST": TRAIN_MANIFEST,
    "VAL_MANIFEST": VAL_MANIFEST,
    "TOKENIZER_PATH": TOKENIZER_PATH,
    "TOKENIZER_CONFIG": TOKENIZER_CONFIG,
    "TRAINING_CONFIG": TRAINING_CONFIG_PATH,
    "INIT_CONFIG": INIT_CONFIG_PATH,
    "STREAMING_CONFIG": STREAMING_CONFIG_PATH,
}

for name, path in paths.items():
    print(f"{name:20s}: {'✓' if os.path.exists(path) else 'MISSING'}")
    print(f"  {path}")

assert os.path.exists(TRAIN_MANIFEST)
assert os.path.exists(VAL_MANIFEST)
assert os.path.exists(TOKENIZER_PATH)

print()
print("GPU STATUS")
print("-" * 60)

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(
        f"GPU {i}: "
        f"free={free/1024**3:.2f} GB / "
        f"total={total/1024**3:.2f} GB"
    )

print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — CONFIGURATION RECOVERY
# ============================================================

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

training_cfg = load_json(TRAINING_CONFIG_PATH)
init_cfg = load_json(INIT_CONFIG_PATH)
streaming_cfg = load_json(STREAMING_CONFIG_PATH)

print("=" * 60)
print("TRAINING CONFIG")
print("=" * 60)
print(json.dumps(training_cfg, indent=2)[:12000])

print()
print("=" * 60)
print("INITIALIZATION CONFIG")
print("=" * 60)
print(json.dumps(init_cfg, indent=2)[:12000])

print()
print("=" * 60)
print("STREAMING CONFIG")
print("=" * 60)
print(json.dumps(streaming_cfg, indent=2)[:12000])

In [ ]:
# ============================================================
# PHASE 4 — CLEAN POST-RECONNECT TRAINING INITIALIZATION
# ============================================================

import os
import gc
import json
import math
import time
import torch

from transformers import GPT2Config, GPT2LMHeadModel

print("=" * 60)
print("PHASE 4 — CLEAN TRAINING INITIALIZATION")
print("=" * 60)

# ------------------------------------------------------------
# 1. Paths — use durable Kaggle Input artifacts
# ------------------------------------------------------------

BASE = "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"

TRAIN_MANIFEST = os.path.join(
    BASE, "manifests", "train_manifest.jsonl"
)

VAL_MANIFEST = os.path.join(
    BASE, "manifests", "validation_manifest.jsonl"
)

TOKENIZER_PATH = os.path.join(
    BASE, "tokenizer", "legal_bpe_32k.json"
)

TOKENIZER_CONFIG = os.path.join(
    BASE, "tokenizer", "tokenizer_config.json"
)

TRAINING_CONFIG_PATH = os.path.join(
    BASE, "training_example_config.json"
)

STREAMING_CONFIG_PATH = os.path.join(
    BASE, "streaming_config.json"
)

# ------------------------------------------------------------
# 2. Verify artifacts
# ------------------------------------------------------------

required_paths = {
    "TRAIN_MANIFEST": TRAIN_MANIFEST,
    "VAL_MANIFEST": VAL_MANIFEST,
    "TOKENIZER_PATH": TOKENIZER_PATH,
    "TOKENIZER_CONFIG": TOKENIZER_CONFIG,
    "TRAINING_CONFIG": TRAINING_CONFIG_PATH,
    "STREAMING_CONFIG": STREAMING_CONFIG_PATH,
}

for name, path in required_paths.items():
    assert os.path.exists(path), f"Missing: {name} -> {path}"
    print(f"✓ {name}")

# ------------------------------------------------------------
# 3. Single-GPU policy
# ------------------------------------------------------------

device = torch.device("cuda:0")

torch.cuda.set_device(0)

print()
print("Selected device:", device)
print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# 4. Training constants
# ------------------------------------------------------------

VOCAB_SIZE = 32_000
CONTEXT_LENGTH = 4096

# GPT-2 124M-class architecture
N_LAYER = 12
N_HEAD = 12
N_EMBD = 768

print()
print("MODEL CONFIG")
print("-" * 60)
print("vocab_size    :", VOCAB_SIZE)
print("context_length:", CONTEXT_LENGTH)
print("n_layer       :", N_LAYER)
print("n_head        :", N_HEAD)
print("n_embd        :", N_EMBD)

# ------------------------------------------------------------
# 5. Construct model on CPU first
# ------------------------------------------------------------

config = GPT2Config(
    vocab_size=VOCAB_SIZE,
    n_positions=CONTEXT_LENGTH,
    n_ctx=CONTEXT_LENGTH,
    n_embd=N_EMBD,
    n_layer=N_LAYER,
    n_head=N_HEAD,

    bos_token_id=2,
    eos_token_id=3,

    use_cache=False,
)

model = GPT2LMHeadModel(config)

# ------------------------------------------------------------
# 6. Move model BEFORE creating optimizer
# ------------------------------------------------------------

model = model.to(device)

# ------------------------------------------------------------
# 7. Training mode
# ------------------------------------------------------------

model.train()

# ------------------------------------------------------------
# 8. Create optimizer AFTER model placement
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0.01,
    foreach=False,
)

# ------------------------------------------------------------
# 9. Basic model information
# ------------------------------------------------------------

parameter_count = sum(
    p.numel() for p in model.parameters()
)

parameter_memory_gb = (
    sum(p.numel() * p.element_size()
        for p in model.parameters())
    / (1024 ** 3)
)

print()
print("=" * 60)
print("MODEL READY")
print("=" * 60)

print("Parameters       :", f"{parameter_count:,}")
print("Parameter memory :", f"{parameter_memory_gb:.2f} GB")
print("Model device     :", next(model.parameters()).device)
print("Model dtype      :", next(model.parameters()).dtype)

assert next(model.parameters()).device == device

print("✓ Model is entirely on selected GPU")
print("✓ Optimizer created after model placement")

In [ ]:
# ============================================================
# PHASE 4 — OPTIMIZER / DEVICE STATE VERIFICATION
# ============================================================

print("=" * 60)
print("OPTIMIZER / DEVICE STATE VERIFICATION")
print("=" * 60)

# ------------------------------------------------------------
# 1. Verify optimizer parameter groups
# ------------------------------------------------------------

optimizer_devices = set()
optimizer_dtypes = set()
parameter_count = 0

for group_idx, group in enumerate(optimizer.param_groups):

    for param_idx, param in enumerate(group["params"]):

        if param.grad is not None:
            optimizer_devices.add(param.grad.device)
            optimizer_dtypes.add(param.grad.dtype)

        optimizer_devices.add(param.device)
        optimizer_dtypes.add(param.dtype)

        parameter_count += 1

print("Optimizer parameter tensors :", parameter_count)
print("Optimizer devices           :", optimizer_devices)
print("Optimizer dtypes             :", optimizer_dtypes)

assert parameter_count > 0
assert optimizer_devices == {device}

print("✓ All optimizer parameters are on", device)

# ------------------------------------------------------------
# 2. Verify model / optimizer parameter identity
# ------------------------------------------------------------

model_params = list(model.parameters())
optimizer_params = []

for group in optimizer.param_groups:
    optimizer_params.extend(group["params"])

assert len(model_params) == len(optimizer_params)

for i, (model_param, opt_param) in enumerate(
    zip(model_params, optimizer_params)
):
    assert model_param is opt_param, (
        f"Optimizer parameter mismatch at index {i}"
    )

print("✓ Optimizer references the actual model parameters")

# ------------------------------------------------------------
# 3. Optimizer state should currently be empty
# ------------------------------------------------------------

print()
print("Optimizer state entries :", len(optimizer.state))

assert len(optimizer.state) == 0

print("✓ Optimizer state is clean")
print("  State tensors will be created after the first backward/step")

# ------------------------------------------------------------
# 4. GPU memory
# ------------------------------------------------------------

print()
print("=" * 60)
print("GPU MEMORY")
print("=" * 60)

for gpu_idx in range(torch.cuda.device_count()):

    free_mem, total_mem = torch.cuda.mem_get_info(gpu_idx)

    allocated = torch.cuda.memory_allocated(gpu_idx)
    reserved = torch.cuda.memory_reserved(gpu_idx)

    print(
        f"GPU {gpu_idx}: "
        f"free={free_mem / 1024**3:.2f} GB | "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB"
    )

print()
print("=" * 60)
print("✓ OPTIMIZER VERIFICATION PASSED")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — RECREATE TRAINING DATALOADER
# ============================================================

import os
import torch
from torch.utils.data import DataLoader

print("=" * 60)
print("RECREATING TRAINING DATALOADER")
print("=" * 60)

# ------------------------------------------------------------
# 1. Verify required variables/classes
# ------------------------------------------------------------

assert "WorkerSafeDocumentAwarePacker" in globals(), (
    "WorkerSafeDocumentAwarePacker is not defined. "
    "Run the dataset class definition cell first."
)

assert os.path.exists(TRAIN_MANIFEST)
assert os.path.exists(TOKENIZER_PATH)

# ------------------------------------------------------------
# 2. Configuration
# ------------------------------------------------------------

CONTEXT_LENGTH = 4096

print("Manifest :", TRAIN_MANIFEST)
print("Tokenizer:", TOKENIZER_PATH)
print("Context  :", CONTEXT_LENGTH)

# ------------------------------------------------------------
# 3. Recreate streaming training dataset
# ------------------------------------------------------------

train_dataset = WorkerSafeDocumentAwarePacker(
    manifest_path=TRAIN_MANIFEST,
    index_path=TRAIN_INDEX,
    tokenizer=tokenizer,
    context_length=CONTEXT_LENGTH,
)

print("✓ Training dataset created")
print("Dataset type:", type(train_dataset).__name__)

# ------------------------------------------------------------
# 4. Create loader with ZERO workers for smoke test
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    num_workers=0,
    pin_memory=True,
)

print("✓ Training DataLoader created")
print()
print("batch_size :", train_loader.batch_size)
print("num_workers:", train_loader.num_workers)

# ------------------------------------------------------------
# 5. Fetch one batch
# ------------------------------------------------------------

batch = next(iter(train_loader))

print()
print("FIRST BATCH")
print("-" * 60)

for key, value in batch.items():

    if torch.is_tensor(value):
        print(
            f"{key:16s}: "
            f"shape={tuple(value.shape)} "
            f"dtype={value.dtype}"
        )
    else:
        print(
            f"{key:16s}: "
            f"{type(value).__name__}"
        )

print()
print("✓ First batch successfully loaded")

# ------------------------------------------------------------
# 6. Verify expected fields
# ------------------------------------------------------------

assert "input_ids" in batch
assert "labels" in batch
assert "attention_mask" in batch

assert batch["input_ids"].shape[0] == 1
assert batch["input_ids"].shape[1] == CONTEXT_LENGTH

assert batch["labels"].shape == batch["input_ids"].shape
assert batch["attention_mask"].shape == batch["input_ids"].shape

print("✓ Batch schema verified")

print()
print("=" * 60)
print("✓ TRAINING DATALOADER READY")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — SINGLE-BATCH TRAINING STEP TEST
# ============================================================

import gc
import torch

print("=" * 60)
print("SINGLE-BATCH TRAINING STEP TEST")
print("=" * 60)

# ------------------------------------------------------------
# 1. Clean transient state
# ------------------------------------------------------------

optimizer.zero_grad(set_to_none=True)

gc.collect()

for gpu_idx in range(torch.cuda.device_count()):
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# 2. Get one batch
# ------------------------------------------------------------

batch = next(iter(train_loader))

input_ids = batch["input_ids"][:1].to(
    device,
    non_blocking=True
)

attention_mask = batch["attention_mask"][:1].to(
    device,
    non_blocking=True
)

labels = batch["labels"][:1].to(
    device,
    non_blocking=True
)

print("Input shape      :", tuple(input_ids.shape))
print("Labels shape     :", tuple(labels.shape))
print("Attention shape  :", tuple(attention_mask.shape))
print("Input device     :", input_ids.device)
print("Labels device    :", labels.device)

assert input_ids.shape == labels.shape
assert input_ids.shape == attention_mask.shape

assert input_ids.device == device
assert labels.device == device
assert attention_mask.device == device

print("✓ Batch prepared")

# ------------------------------------------------------------
# 3. Forward
# ------------------------------------------------------------

model.train()

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
    use_cache=False,
)

loss = outputs.loss

print()
print("Forward loss :", float(loss.detach().cpu()))

assert torch.isfinite(loss).item()

print("✓ Forward successful")
print("✓ Loss is finite")

# ------------------------------------------------------------
# 4. Backward
# ------------------------------------------------------------

loss.backward()

print("✓ Backward successful")

# ------------------------------------------------------------
# 5. Gradient verification
# ------------------------------------------------------------

gradient_count = 0
gradient_memory = 0.0

for name, param in model.named_parameters():

    if param.grad is not None:

        gradient_count += 1

        gradient_memory += (
            param.grad.numel()
            * param.grad.element_size()
        )

        assert param.grad.device == device

print("Gradient tensors :", gradient_count)
print(
    "Gradient memory  :",
    f"{gradient_memory / 1024**3:.2f} GB"
)

assert gradient_count > 0

print("✓ Gradients exist")
print("✓ Gradients are on correct device")

# ------------------------------------------------------------
# 6. Optimizer step
# ------------------------------------------------------------

optimizer.step()

print("✓ Optimizer step successful")

# ------------------------------------------------------------
# 7. Verify optimizer state devices
# ------------------------------------------------------------

bad_state = []

for param, state in optimizer.state.items():

    for state_name, state_value in state.items():

        if torch.is_tensor(state_value):

            if state_value.device != device:

                bad_state.append(
                    (
                        state_name,
                        state_value.device,
                        device
                    )
                )

if bad_state:
    print("BAD OPTIMIZER STATE:")
    for item in bad_state[:10]:
        print(item)

assert not bad_state

print("✓ All optimizer state tensors are on", device)

# ------------------------------------------------------------
# 8. Clear gradients
# ------------------------------------------------------------

optimizer.zero_grad(set_to_none=True)

del outputs
del loss
del input_ids
del labels
del attention_mask
del batch

gc.collect()

# ------------------------------------------------------------
# 9. Final memory report
# ------------------------------------------------------------

print()
print("=" * 60)
print("POST-STEP GPU MEMORY")
print("=" * 60)

for gpu_idx in range(torch.cuda.device_count()):

    free_mem, total_mem = torch.cuda.mem_get_info(gpu_idx)

    print(
        f"GPU {gpu_idx}: "
        f"free={free_mem / 1024**3:.2f} GB | "
        f"allocated={torch.cuda.memory_allocated(gpu_idx) / 1024**3:.2f} GB | "
        f"reserved={torch.cuda.memory_reserved(gpu_idx) / 1024**3:.2f} GB"
    )

print()
print("=" * 60)
print("✓ SINGLE-BATCH TRAINING STEP TEST PASSED")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 RECOVERY — CELL 1
# CLEAN STATE + DURABLE ARTIFACT DISCOVERY
# ============================================================

import os
import gc
import json
import glob
import torch

print("=" * 60)
print("PHASE 4 — CLEAN RECOVERY")
print("=" * 60)

# ------------------------------------------------------------
# Clean only Python references that may exist from failed attempts
# ------------------------------------------------------------

_names_to_remove = [
    "model",
    "optimizer",
    "tokenizer",
    "train_dataset",
    "validation_dataset",
    "train_loader",
    "validation_loader",
    "loader",
    "debug_loader",
    "dataset",
    "debug_dataset",
    "epoch_dataset",
    "worker_safe_dataset",
    "batch",
    "input_ids",
    "labels",
    "attention_mask",
    "outputs",
    "loss",
]

for _name in _names_to_remove:
    if _name in globals():
        del globals()[_name]

gc.collect()

if torch.cuda.is_available():
    for _gpu in range(torch.cuda.device_count()):
        with torch.cuda.device(_gpu):
            torch.cuda.empty_cache()

# ------------------------------------------------------------
# Canonical durable location
# ------------------------------------------------------------

BASE = "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"

assert os.path.isdir(BASE), BASE

# ------------------------------------------------------------
# Locate known artifacts
# ------------------------------------------------------------

TRAIN_MANIFEST = os.path.join(
    BASE, "manifests", "train_manifest.jsonl"
)

VAL_MANIFEST = os.path.join(
    BASE, "manifests", "validation_manifest.jsonl"
)

TEST_MANIFEST = os.path.join(
    BASE, "manifests", "test_manifest.jsonl"
)

TOKENIZER_PATH = os.path.join(
    BASE, "tokenizer", "legal_bpe_32k.json"
)

TOKENIZER_CONFIG = os.path.join(
    BASE, "tokenizer", "tokenizer_config.json"
)

STREAMING_CONFIG_PATH = os.path.join(
    BASE, "streaming_config.json"
)

TRAINING_CONFIG_PATH = os.path.join(
    BASE, "training_example_config.json"
)

required = {
    "TRAIN_MANIFEST": TRAIN_MANIFEST,
    "VAL_MANIFEST": VAL_MANIFEST,
    "TEST_MANIFEST": TEST_MANIFEST,
    "TOKENIZER_PATH": TOKENIZER_PATH,
    "TOKENIZER_CONFIG": TOKENIZER_CONFIG,
    "STREAMING_CONFIG": STREAMING_CONFIG_PATH,
    "TRAINING_CONFIG": TRAINING_CONFIG_PATH,
}

for name, path in required.items():
    print(
        f"{name:22s}:",
        "✓" if os.path.exists(path) else "MISSING",
        path,
    )

print()
print("CUDA")
print("-" * 60)

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(
        f"GPU {i}: "
        f"free={free/1024**3:.2f} GB / "
        f"total={total/1024**3:.2f} GB"
    )

print()
print("=" * 60)
print("✓ CLEAN RECOVERY STATE CREATED")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 RECOVERY — CELL 2
# READ DURABLE CONFIGURATION
# ============================================================

print("=" * 60)
print("PHASE 4 — DURABLE CONFIGURATION")
print("=" * 60)

with open(STREAMING_CONFIG_PATH, "r") as f:
    streaming_config = json.load(f)

with open(TRAINING_CONFIG_PATH, "r") as f:
    training_config = json.load(f)

print()
print("STREAMING CONFIG")
print("-" * 60)

for k, v in streaming_config.items():
    print(f"{k:28s}: {v}")

print()
print("TRAINING CONFIG")
print("-" * 60)

for k, v in training_config.items():
    print(f"{k:28s}: {v}")

# ------------------------------------------------------------
# Canonical values
# ------------------------------------------------------------

VOCAB_SIZE = int(streaming_config["vocab_size"])
CONTEXT_LENGTH = int(training_config["context_length"])

print()
print("RECOVERED CONSTANTS")
print("-" * 60)
print("VOCAB_SIZE     :", VOCAB_SIZE)
print("CONTEXT_LENGTH :", CONTEXT_LENGTH)

assert VOCAB_SIZE == 32000
assert CONTEXT_LENGTH == 4096

print()
print("✓ Phase 4 configuration recovered")

In [ ]:
# ============================================================
# PHASE 4 RECOVERY — CELL 3
# RECREATE TOKENIZER
# ============================================================

from tokenizers import Tokenizer

print("=" * 60)
print("RECREATING TOKENIZER")
print("=" * 60)

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

print("Tokenizer type :", type(tokenizer).__name__)
print("Vocabulary size:", tokenizer.get_vocab_size())

assert tokenizer.get_vocab_size() == VOCAB_SIZE

print()
print("✓ Tokenizer successfully reconstructed")

In [ ]:
# ============================================================
# PHASE 4 RECOVERY — CELL 4
# MANIFEST SCHEMA INSPECTION
# ============================================================

print("=" * 60)
print("TRAIN MANIFEST SCHEMA")
print("=" * 60)

def inspect_jsonl(path, n=3):
    records = []

    with open(path, "r") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            records.append(json.loads(line))

            if len(records) >= n:
                break

    return records


train_examples = inspect_jsonl(TRAIN_MANIFEST, 3)

for i, rec in enumerate(train_examples):
    print()
    print(f"RECORD {i + 1}")
    print("-" * 60)

    print(json.dumps(rec, indent=2, ensure_ascii=False)[:5000])

    print()
    print("FIELDS")
    for k, v in rec.items():
        print(
            f"  {k:25s} "
            f"{type(v).__name__:12s} "
            f"{str(v)[:200]}"
        )

print()
print("=" * 60)
print("✓ MANIFEST INSPECTION COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 RECOVERY — CELL 5
# SEARCH FOR STREAMING INDEX ARTIFACTS
# ============================================================

print("=" * 60)
print("SEARCHING FOR SURVIVING STREAMING INDEXES")
print("=" * 60)

search_roots = [
    "/kaggle/input",
    "/kaggle/working",
]

index_candidates = []

for root in search_roots:

    for dirpath, dirnames, filenames in os.walk(root):

        for filename in filenames:

            lower = filename.lower()

            if (
                "index" in lower
                or "streaming" in lower
                or "source_index" in lower
            ):
                path = os.path.join(dirpath, filename)

                try:
                    size = os.path.getsize(path)
                except OSError:
                    continue

                index_candidates.append((size, path))

index_candidates.sort(reverse=True)

if index_candidates:

    for size, path in index_candidates[:100]:

        print(
            f"{size / 1024**2:10.2f} MB | {path}"
        )

else:
    print("No index-like artifacts found.")

print()
print("=" * 60)
print("INDEX SEARCH COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 RECOVERY — CELL 6
# VERIFY DURABLE SOURCE CORPUS
# ============================================================

print("=" * 60)
print("PHASE 4 — DURABLE SOURCE CORPUS VERIFICATION")
print("=" * 60)

# ------------------------------------------------------------
# Locate the deduplicated corpus
# ------------------------------------------------------------

DEDUP_ROOT = os.path.join(BASE, "dedup")

assert os.path.isdir(DEDUP_ROOT), DEDUP_ROOT

source_files = []

for dirpath, dirnames, filenames in os.walk(DEDUP_ROOT):

    for filename in filenames:

        if filename.endswith(".jsonl"):

            path = os.path.join(dirpath, filename)
            size = os.path.getsize(path)

            source_files.append((path, size))

source_files.sort()

print()
print("SOURCE FILES")
print("-" * 60)

total_bytes = 0

for path, size in source_files:

    total_bytes += size

    print(
        f"{size / 1024**3:7.3f} GB | "
        f"{os.path.relpath(path, BASE)}"
    )

print()
print("Source files :", len(source_files))
print("Total size   :", f"{total_bytes / 1024**3:.2f} GB")

assert len(source_files) > 0

# ------------------------------------------------------------
# Inspect manifest references
# ------------------------------------------------------------

print()
print("=" * 60)
print("MANIFEST SOURCE REFERENCES")
print("=" * 60)

manifest_records = inspect_jsonl(TRAIN_MANIFEST, 10)

for i, rec in enumerate(manifest_records):

    print()
    print(f"RECORD {i + 1}")

    for key, value in rec.items():

        if isinstance(value, str) and len(value) > 300:
            value = value[:300] + "..."

        print(f"  {key:25s}: {value}")

print()
print("=" * 60)
print("✓ SOURCE CORPUS VERIFIED")
print("=" * 60)

In [ ]:
import os
import gc
import json
import math
import time
import psutil
import torch

from pathlib import Path

print("=" * 60)
print("PHASE 4 — CLEAN POST-RECONNECT RECOVERY")
print("=" * 60)

# ------------------------------------------------------------
# CUDA
# ------------------------------------------------------------

assert torch.cuda.is_available(), "CUDA is not available"

device = torch.device("cuda:0")

print(f"Device : {device}")
print(f"GPU    : {torch.cuda.get_device_name(0)}")

# Start from a clean Python/CUDA state
gc.collect()

for i in range(torch.cuda.device_count()):
    torch.cuda.set_device(i)
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

# Return to GPU 0
torch.cuda.set_device(device)

# ------------------------------------------------------------
# RAM
# ------------------------------------------------------------

ram = psutil.virtual_memory()

print()
print("SYSTEM RAM")
print("-" * 60)
print(f"Total     : {ram.total / 1024**3:.2f} GB")
print(f"Used      : {ram.used / 1024**3:.2f} GB")
print(f"Available : {ram.available / 1024**3:.2f} GB")
print(f"Percent   : {ram.percent:.1f}%")

print()
print("GPU")
print("-" * 60)

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(
        f"GPU {i}: "
        f"{torch.cuda.get_device_name(i)} | "
        f"free={free/1024**3:.2f} GB / "
        f"total={total/1024**3:.2f} GB"
    )

print("=" * 60)
print("✓ CLEAN RECOVERY ENVIRONMENT READY")
print("=" * 60)

In [ ]:
from pathlib import Path

INPUT_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

TRAIN_MANIFEST = INPUT_ROOT / "manifests/train_manifest.jsonl"
VAL_MANIFEST   = INPUT_ROOT / "manifests/validation_manifest.jsonl"
TEST_MANIFEST  = INPUT_ROOT / "manifests/test_manifest.jsonl"

TOKENIZER_PATH = INPUT_ROOT / "tokenizer/legal_bpe_32k.json"
TOKENIZER_CONFIG = INPUT_ROOT / "tokenizer/tokenizer_config.json"

TRAINING_CONFIG = INPUT_ROOT / "training_example_config.json"
STREAMING_CONFIG = INPUT_ROOT / "streaming_config.json"

required_paths = {
    "TRAIN_MANIFEST": TRAIN_MANIFEST,
    "VAL_MANIFEST": VAL_MANIFEST,
    "TEST_MANIFEST": TEST_MANIFEST,
    "TOKENIZER_PATH": TOKENIZER_PATH,
    "TOKENIZER_CONFIG": TOKENIZER_CONFIG,
    "TRAINING_CONFIG": TRAINING_CONFIG,
    "STREAMING_CONFIG": STREAMING_CONFIG,
}

print("=" * 60)
print("PHASE 4 — DURABLE PATH INITIALIZATION")
print("=" * 60)

for name, path in required_paths.items():
    exists = path.exists()
    print(f"{name:<20}: {'✓' if exists else '✗'}  {path}")

    if not exists:
        raise FileNotFoundError(path)

print("=" * 60)
print("✓ ALL REQUIRED ARTIFACTS AVAILABLE")
print("=" * 60)

In [ ]:
from pathlib import Path

INPUT_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

TRAIN_MANIFEST = INPUT_ROOT / "manifests/train_manifest.jsonl"
VAL_MANIFEST   = INPUT_ROOT / "manifests/validation_manifest.jsonl"
TEST_MANIFEST  = INPUT_ROOT / "manifests/test_manifest.jsonl"

TOKENIZER_PATH = INPUT_ROOT / "tokenizer/legal_bpe_32k.json"
TOKENIZER_CONFIG = INPUT_ROOT / "tokenizer/tokenizer_config.json"

TRAINING_CONFIG = INPUT_ROOT / "training_example_config.json"
STREAMING_CONFIG = INPUT_ROOT / "streaming_config.json"

required_paths = {
    "TRAIN_MANIFEST": TRAIN_MANIFEST,
    "VAL_MANIFEST": VAL_MANIFEST,
    "TEST_MANIFEST": TEST_MANIFEST,
    "TOKENIZER_PATH": TOKENIZER_PATH,
    "TOKENIZER_CONFIG": TOKENIZER_CONFIG,
    "TRAINING_CONFIG": TRAINING_CONFIG,
    "STREAMING_CONFIG": STREAMING_CONFIG,
}

print("=" * 60)
print("PHASE 4 — DURABLE PATH INITIALIZATION")
print("=" * 60)

for name, path in required_paths.items():
    exists = path.exists()
    print(f"{name:<20}: {'✓' if exists else '✗'}  {path}")

    if not exists:
        raise FileNotFoundError(path)

print("=" * 60)
print("✓ ALL REQUIRED ARTIFACTS AVAILABLE")
print("=" * 60)

In [ ]:
with open(TRAINING_CONFIG, "r") as f:
    training_config = json.load(f)

with open(STREAMING_CONFIG, "r") as f:
    streaming_config = json.load(f)

print("=" * 60)
print("PHASE 4 — RECOVERED CONFIGURATION")
print("=" * 60)

print()
print("TRAINING CONFIG")
print("-" * 60)
print(json.dumps(training_config, indent=2))

print()
print("STREAMING CONFIG")
print("-" * 60)
print(json.dumps(streaming_config, indent=2))

assert training_config["context_length"] == 4096
assert training_config["packing"] == "document_aware"
assert training_config["document_boundary"] is True

assert streaming_config["dataset_mode"] == "streaming"
assert streaming_config["materialize_tokenized_corpus"] is False

print()
print("✓ Streaming configuration confirmed")
print("✓ No tokenized corpus will be materialized")
print("=" * 60)

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel

CONTEXT_LENGTH = 4096
VOCAB_SIZE = 32000

model_config = GPT2Config(
    vocab_size=VOCAB_SIZE,
    n_positions=CONTEXT_LENGTH,
    n_ctx=CONTEXT_LENGTH,
    n_embd=768,
    n_layer=12,
    n_head=12,
    bos_token_id=2,
    eos_token_id=3,
)

model = GPT2LMHeadModel(model_config)

# IMPORTANT:
# Move model BEFORE creating optimizer.
model = model.to(device)

model.train()

print("=" * 60)
print("MODEL VERIFICATION")
print("=" * 60)

parameter_count = sum(p.numel() for p in model.parameters())
parameter_memory = sum(
    p.numel() * p.element_size()
    for p in model.parameters()
) / 1024**3

print(f"Parameters       : {parameter_count:,}")
print(f"Parameter memory : {parameter_memory:.2f} GB")
print(f"Device           : {next(model.parameters()).device}")
print(f"Dtype            : {next(model.parameters()).dtype}")

assert next(model.parameters()).device.index == device.index

print("✓ Model on cuda:0")
print("=" * 60)

In [ ]:
from torch.optim import AdamW

LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

print("=" * 60)
print("OPTIMIZER CREATED")
print("=" * 60)

print("Optimizer : AdamW")
print(f"LR        : {LEARNING_RATE}")
print(f"Weight decay : {WEIGHT_DECAY}")

print("✓ Optimizer created AFTER model placement")
print("=" * 60)

In [ ]:
print("=" * 60)
print("PHASE 4 — DATASET CLASS RECOVERY")
print("=" * 60)

classes_to_check = [
    "DocumentAwarePacker",
    "WorkerSafeDocumentAwarePacker",
    "DebugWorkerSafePacker",
]

for name in classes_to_check:
    obj = globals().get(name, None)

    if obj is None:
        print(f"{name:<35}: NOT PRESENT")
    else:
        print(f"{name:<35}: ✓ {obj}")

print("=" * 60)

In [ ]:
import inspect

for name in [
    "DocumentAwarePacker",
    "WorkerSafeDocumentAwarePacker",
    "DebugWorkerSafePacker",
]:
    cls = globals().get(name)

    if cls is not None:
        print("=" * 60)
        print(name)
        print("=" * 60)

        try:
            print(inspect.signature(cls))
        except Exception as e:
            print("Could not inspect signature:", e)

        print()
        print(inspect.getsource(cls)[:5000])

In [ ]:
import inspect

print("=" * 60)
print("WORKER-SAFE DATASET CLASS INSPECTION")
print("=" * 60)

cls = WorkerSafeDocumentAwarePacker

print("Class:")
print(cls)

print()
print("Module:")
print(cls.__module__)

print()
print("Constructor signature:")
print(inspect.signature(cls.__init__))

print()
print("Available class attributes:")
print([
    x for x in dir(cls)
    if not x.startswith("__")
])

print("=" * 60)

In [ ]:
print("=" * 60)
print("CONSTRUCTOR DETAILS")
print("=" * 60)

sig = inspect.signature(WorkerSafeDocumentAwarePacker.__init__)

for name, param in sig.parameters.items():
    if name == "self":
        continue

    print(
        f"{name:<25} "
        f"default={param.default!r:<20} "
        f"annotation={param.annotation!r}"
    )

print("=" * 60)

In [ ]:
import dis

print("=" * 60)
print("WORKER-SAFE PACKER — INTERNAL DEPENDENCIES")
print("=" * 60)

for method_name in [
    "_read_source_record",
    "_pack_document",
    "_tokenize_document",
    "__iter__",
]:
    method = getattr(WorkerSafeDocumentAwarePacker, method_name, None)

    if method is None:
        print(f"\n{method_name}: NOT DEFINED")
        continue

    print(f"\n{'-' * 60}")
    print(f"METHOD: {method_name}")
    print("-" * 60)

    try:
        code = method.__code__

        print("Names:")
        print(code.co_names)

        print("\nConstants:")
        print(code.co_consts)

    except Exception as e:
        print("Inspection failed:", repr(e))

print("=" * 60)

In [ ]:
print("=" * 60)
print("WORKER-SAFE PACKER — INIT BYTECODE")
print("=" * 60)

dis.dis(WorkerSafeDocumentAwarePacker.__init__)

print("=" * 60)

In [ ]:
import json
from pathlib import Path

print("=" * 60)
print("PHASE 4 — MANIFEST / SOURCE INDEX RECONSTRUCTION")
print("=" * 60)

manifest_path = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep/"
    "manifests/train_manifest.jsonl"
)

print("Manifest:", manifest_path)
print("Exists  :", manifest_path.exists())
print("Size    :", f"{manifest_path.stat().st_size / (1024**3):.3f} GB")

with manifest_path.open("r", encoding="utf-8") as f:
    first_record = json.loads(f.readline())

print("\nFIRST MANIFEST RECORD")
print("-" * 60)

for k, v in first_record.items():
    if isinstance(v, str) and len(v) > 200:
        print(f"{k:25s}: {v[:200]}...")
    else:
        print(f"{k:25s}: {v}")

print("\nKEYS")
print("-" * 60)
print(list(first_record.keys()))

print("\n" + "=" * 60)
print("MANIFEST INSPECTION COMPLETE")
print("=" * 60)

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep")

print("=" * 60)
print("PHASE 4 — INDEX ARTIFACT DISCOVERY")
print("=" * 60)

# Search for likely index artifacts without reading their contents.
patterns = [
    "*index*",
    "*idx*",
    "*.sqlite",
    "*.db",
    "*.jsonl.idx",
    "*.json.idx",
    "*.bin",
]

found = {}

for pattern in patterns:
    matches = list(ROOT.rglob(pattern))
    for p in matches:
        found[str(p)] = p

if not found:
    print("NO CANDIDATE INDEX FILES FOUND")
else:
    for path_str in sorted(found):
        p = Path(path_str)
        try:
            size_mb = p.stat().st_size / (1024 ** 2)
        except Exception:
            size_mb = -1

        print(f"{size_mb:10.2f} MB | {p}")

print()
print("TOTAL CANDIDATES:", len(found))

print("=" * 60)
print("INDEX ARTIFACT DISCOVERY COMPLETE")
print("=" * 60)

In [ ]:
import json
from pathlib import Path

ROOT = Path(
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep"
)

SOURCE = ROOT / "dedup" / "acts" / "documents.jsonl"

print("=" * 60)
print("PHASE 4 — SOURCE RECORD FORMAT VERIFICATION")
print("=" * 60)

print("Source :", SOURCE)
print("Exists :", SOURCE.exists())
print("Size   :", f"{SOURCE.stat().st_size / (1024**3):.3f} GB")

with SOURCE.open("rb") as f:
    offset = f.tell()
    raw = f.readline()

print("\nRAW RECORD")
print("-" * 60)
print("Byte offset :", offset)
print("Byte length :", len(raw))
print("First bytes :", raw[:120])

record = json.loads(raw.decode("utf-8"))

print("\nDECODED RECORD")
print("-" * 60)

for key, value in record.items():
    if isinstance(value, str):
        preview = value[:300].replace("\n", "\\n")
        print(f"{key:20s}: {preview}")
    else:
        print(f"{key:20s}: {value}")

print("\nKEYS")
print("-" * 60)
print(list(record.keys()))

print("=" * 60)
print("SOURCE RECORD VERIFICATION COMPLETE")
print("=" * 60)

In [ ]:
import json
import os
import time
from pathlib import Path

ROOT = Path(
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep"
)

WORK_ROOT = Path("/kaggle/working/phase4_recovery")
INDEX_DIR = WORK_ROOT / "indexes"
INDEX_DIR.mkdir(parents=True, exist_ok=True)

INDEX_PATH = INDEX_DIR / "source_index.jsonl"

print("=" * 60)
print("PHASE 4 — STREAMING SOURCE INDEX BUILD")
print("=" * 60)

print("Input corpus :", ROOT / "dedup")
print("Output index :", INDEX_PATH)

# ------------------------------------------------------------
# Discover source files
# ------------------------------------------------------------

source_files = sorted(
    (ROOT / "dedup").rglob("*.jsonl")
)

print()
print("SOURCE FILES")
print("-" * 60)
print("Count :", len(source_files))

total_bytes = sum(p.stat().st_size for p in source_files)

print("Total :", f"{total_bytes / (1024**3):.2f} GB")

# ------------------------------------------------------------
# Build index
# ------------------------------------------------------------

start = time.time()
total_records = 0
total_bytes_read = 0

# Write incrementally so a crash does not require holding the
# complete index in memory.
with INDEX_PATH.open("w", encoding="utf-8") as out:

    for file_no, source_path in enumerate(source_files, 1):

        # Store the path relative to dedup/
        relative_source = source_path.relative_to(ROOT / "dedup")

        file_records = 0
        file_start = time.time()

        print()
        print(
            f"[{file_no:02d}/{len(source_files):02d}] "
            f"{relative_source}"
        )

        with source_path.open("rb") as f:

            while True:

                byte_offset = f.tell()
                raw = f.readline()

                if not raw:
                    break

                byte_length = len(raw)
                total_bytes_read += byte_length

                try:
                    record = json.loads(raw)
                except Exception as e:
                    raise RuntimeError(
                        f"Invalid JSON in {source_path} "
                        f"at byte offset {byte_offset}"
                    ) from e

                text_hash = record.get("text_hash")

                if not text_hash:
                    raise RuntimeError(
                        f"Missing text_hash in {source_path} "
                        f"at byte offset {byte_offset}"
                    )

                index_record = {
                    "text_hash": text_hash,
                    "source_file": str(relative_source),
                    "byte_offset": byte_offset,
                    "byte_length": byte_length,
                }

                out.write(
                    json.dumps(
                        index_record,
                        separators=(",", ":")
                    )
                    + "\n"
                )

                file_records += 1
                total_records += 1

        elapsed_file = time.time() - file_start

        print(
            f"  records={file_records:,} "
            f"time={elapsed_file:.1f}s"
        )

        # Make progress durable.
        out.flush()

elapsed = time.time() - start

print()
print("=" * 60)
print("INDEX BUILD COMPLETE")
print("=" * 60)

print("Records indexed :", f"{total_records:,}")
print("Bytes scanned   :", f"{total_bytes_read / (1024**3):.2f} GB")
print("Time            :", f"{elapsed / 60:.2f} min")

print()
print("INDEX")
print("-" * 60)

print("Path :", INDEX_PATH)
print("Size :", f"{INDEX_PATH.stat().st_size / (1024**2):.2f} MB")

print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — RECOVER STREAMING DATASET
# ============================================================

import os
import gc
import torch

TRAIN_MANIFEST = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep/manifests/train_manifest.jsonl"
)

VAL_MANIFEST = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep/manifests/validation_manifest.jsonl"
)

INDEX_PATH = (
    "/kaggle/working/phase4_recovery/indexes/source_index.jsonl"
)

TOKENIZER_PATH = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep/tokenizer/legal_bpe_32k.json"
)

CONTEXT_LENGTH = 4096

print("=" * 60)
print("PHASE 4 — STREAMING DATASET RECOVERY")
print("=" * 60)

print(f"TRAIN_MANIFEST : {os.path.exists(TRAIN_MANIFEST)}")
print(f"VAL_MANIFEST   : {os.path.exists(VAL_MANIFEST)}")
print(f"INDEX_PATH     : {os.path.exists(INDEX_PATH)}")
print(f"TOKENIZER      : {os.path.exists(TOKENIZER_PATH)}")

assert os.path.exists(TRAIN_MANIFEST)
assert os.path.exists(VAL_MANIFEST)
assert os.path.exists(INDEX_PATH)
assert os.path.exists(TOKENIZER_PATH)

# Make sure tokenizer exists in current namespace.
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

print()
print("Tokenizer loaded")
print("Vocab size:", tokenizer.get_vocab_size())

# Recreate ONLY the dataset.
train_dataset = WorkerSafeDocumentAwarePacker(
    manifest_path=TRAIN_MANIFEST,
    index_path=INDEX_PATH,
    tokenizer=tokenizer,
    context_length=CONTEXT_LENGTH,
    checkpoint="/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep",
    split="train",
)

print()
print("DATASET CREATED")
print("Dataset type :", type(train_dataset))
print("Length       :", len(train_dataset))

print("=" * 60)
print("✓ STREAMING DATASET RECOVERY COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — SINGLE STREAMING SAMPLE TEST
# ============================================================

import time
import torch

print("=" * 60)
print("SINGLE STREAMING SAMPLE TEST")
print("=" * 60)

print("Dataset:", type(train_dataset))
print("Index  :", INDEX_PATH)

t0 = time.time()

try:
    sample = next(iter(train_dataset))

    elapsed = time.time() - t0

    print()
    print("✓ FIRST SAMPLE READ")
    print("Time:", round(elapsed, 2), "seconds")
    print()

    print("Sample keys:")
    print(list(sample.keys()))

    for key, value in sample.items():
        if torch.is_tensor(value):
            print(
                f"{key:16s}: "
                f"shape={tuple(value.shape)} "
                f"dtype={value.dtype} "
                f"device={value.device}"
            )
        else:
            print(
                f"{key:16s}: "
                f"type={type(value)}"
            )

    print()
    print("input_ids first 20:")
    print(sample["input_ids"][:20])

    print()
    print("labels first 20:")
    print(sample["labels"][:20])

    print()
    print("attention_mask unique:")
    print(torch.unique(sample["attention_mask"]))

except Exception as e:
    print()
    print("✗ SAMPLE READ FAILED")
    print(type(e).__name__, ":", e)
    raise

print("=" * 60)
print("✓ SINGLE STREAMING SAMPLE TEST COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — SAFE SINGLE-WORKER DATALOADER
# ============================================================

from torch.utils.data import DataLoader

print("=" * 60)
print("SAFE SINGLE-WORKER DATALOADER")
print("=" * 60)

BATCH_SIZE = 1

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    num_workers=0,
    pin_memory=False,
)

print("Batch size :", BATCH_SIZE)
print("Workers    : 0")
print("Pin memory : False")

# Pull exactly one batch.
t0 = time.time()

batch = next(iter(train_loader))

elapsed = time.time() - t0

print()
print("✓ FIRST BATCH CREATED")
print("Time:", round(elapsed, 2), "seconds")

for key, value in batch.items():
    print(
        f"{key:16s}: "
        f"shape={tuple(value.shape)} "
        f"dtype={value.dtype} "
        f"device={value.device}"
    )

assert batch["input_ids"].shape == (1, 4096)
assert batch["labels"].shape == (1, 4096)
assert batch["attention_mask"].shape == (1, 4096)

print()
print("✓ BATCH SHAPES VERIFIED")

print("=" * 60)
print("✓ SAFE DATALOADER TEST PASSED")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — SINGLE COMPLETE TRAINING STEP
# ============================================================

import torch
import gc

print("=" * 60)
print("SINGLE COMPLETE TRAINING STEP")
print("=" * 60)

device = torch.device("cuda:0")

# ------------------------------------------------------------
# VERIFY MODEL
# ------------------------------------------------------------

model = model.to(device)

print("Model device :", next(model.parameters()).device)
print("Model dtype  :", next(model.parameters()).dtype)

# ------------------------------------------------------------
# VERIFY OPTIMIZER PARAMETER STATE
# ------------------------------------------------------------

print()
print("CHECKING OPTIMIZER STATE")

bad_params = []

for group in optimizer.param_groups:
    for p in group["params"]:
        if p.device != device:
            bad_params.append(
                ("parameter", p.device, p.dtype, tuple(p.shape))
            )

print("Parameters checked")
print("Device mismatches:", len(bad_params))

assert len(bad_params) == 0, bad_params

# ------------------------------------------------------------
# GET ONE BATCH
# ------------------------------------------------------------

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(
    device=device,
    dtype=torch.long,
    non_blocking=False,
)

labels = batch["labels"].to(
    device=device,
    dtype=torch.long,
    non_blocking=False,
)

attention_mask = batch["attention_mask"].to(
    device=device,
    dtype=torch.long,
    non_blocking=False,
)

print()
print("BATCH")
print("input_ids      :", input_ids.shape, input_ids.dtype, input_ids.device)
print("labels         :", labels.shape, labels.dtype, labels.device)
print("attention_mask :", attention_mask.shape, attention_mask.dtype, attention_mask.device)

# ------------------------------------------------------------
# CLEAR OLD GRADIENTS
# ------------------------------------------------------------

optimizer.zero_grad(set_to_none=True)

torch.cuda.empty_cache()

print()
print("GPU before forward:")
print(
    f"allocated={torch.cuda.memory_allocated(device)/1024**3:.2f} GB | "
    f"reserved={torch.cuda.memory_reserved(device)/1024**3:.2f} GB"
)

# ------------------------------------------------------------
# FORWARD
# ------------------------------------------------------------

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
)

loss = outputs["loss"] if isinstance(outputs, dict) else outputs.loss

print()
print("Forward loss:", float(loss.detach().cpu()))

assert torch.isfinite(loss).item()

# ------------------------------------------------------------
# BACKWARD
# ------------------------------------------------------------

loss.backward()

print("✓ Backward successful")

print()
print("GPU before optimizer:")
print(
    f"allocated={torch.cuda.memory_allocated(device)/1024**3:.2f} GB | "
    f"reserved={torch.cuda.memory_reserved(device)/1024**3:.2f} GB"
)

# ------------------------------------------------------------
# VERIFY GRADIENT DEVICES
# ------------------------------------------------------------

bad_grads = []

for group in optimizer.param_groups:
    for p in group["params"]:
        if p.grad is not None:
            if p.grad.device != p.device:
                bad_grads.append(
                    (
                        p.device,
                        p.grad.device,
                        p.dtype,
                        p.grad.dtype,
                        tuple(p.shape),
                    )
                )

print("Gradient device mismatches:", len(bad_grads))

assert len(bad_grads) == 0, bad_grads

# ------------------------------------------------------------
# OPTIMIZER STEP
# ------------------------------------------------------------

optimizer.step()

print()
print("✓ OPTIMIZER STEP SUCCESSFUL")

# ------------------------------------------------------------
# CLEANUP
# ------------------------------------------------------------

optimizer.zero_grad(set_to_none=True)

del batch
del input_ids
del labels
del attention_mask
del outputs
del loss

gc.collect()
torch.cuda.empty_cache()

print()
print("GPU after step:")
print(
    f"allocated={torch.cuda.memory_allocated(device)/1024**3:.2f} GB | "
    f"reserved={torch.cuda.memory_reserved(device)/1024**3:.2f} GB"
)

print("=" * 60)
print("✓ SINGLE COMPLETE TRAINING STEP PASSED")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — 10-STEP TRAINING STABILITY TEST
# ============================================================

import time
import gc
import torch

print("=" * 60)
print("10-STEP TRAINING STABILITY TEST")
print("=" * 60)

device = torch.device("cuda:0")

STABILITY_STEPS = 10

model.train()

losses = []
gpu_stats = []

for step in range(1, STABILITY_STEPS + 1):

    # --------------------------------------------------------
    # LOAD ONE STREAMING BATCH
    # --------------------------------------------------------

    batch = next(iter(train_loader))

    input_ids = batch["input_ids"].to(
        device=device,
        dtype=torch.long,
    )

    labels = batch["labels"].to(
        device=device,
        dtype=torch.long,
    )

    attention_mask = batch["attention_mask"].to(
        device=device,
        dtype=torch.long,
    )

    # --------------------------------------------------------
    # FORWARD
    # --------------------------------------------------------

    optimizer.zero_grad(set_to_none=True)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
    )

    loss = outputs["loss"] if isinstance(outputs, dict) else outputs.loss

    # --------------------------------------------------------
    # BACKWARD
    # --------------------------------------------------------

    loss.backward()

    # --------------------------------------------------------
    # OPTIMIZER
    # --------------------------------------------------------

    optimizer.step()

    loss_value = float(loss.detach().cpu())

    allocated = torch.cuda.memory_allocated(device) / 1024**3
    reserved = torch.cuda.memory_reserved(device) / 1024**3

    losses.append(loss_value)
    gpu_stats.append((allocated, reserved))

    print(
        f"step {step:02d} | "
        f"loss={loss_value:.5f} | "
        f"allocated={allocated:.2f} GB | "
        f"reserved={reserved:.2f} GB"
    )

    # --------------------------------------------------------
    # CLEAN BATCH REFERENCES
    # --------------------------------------------------------

    del batch
    del input_ids
    del labels
    del attention_mask
    del outputs
    del loss

    gc.collect()
    torch.cuda.empty_cache()

print()
print("=" * 60)
print("STABILITY SUMMARY")
print("=" * 60)

print("Steps completed :", len(losses))
print("First loss      :", losses[0])
print("Last loss       :", losses[-1])

print()
print("GPU memory:")
print(
    "Peak allocated :",
    f"{max(x[0] for x in gpu_stats):.2f} GB"
)
print(
    "Peak reserved  :",
    f"{max(x[1] for x in gpu_stats):.2f} GB"
)
print(
    "Final allocated:",
    f"{gpu_stats[-1][0]:.2f} GB"
)
print(
    "Final reserved :",
    f"{gpu_stats[-1][1]:.2f} GB"
)

print()
print("=" * 60)
print("✓ 10-STEP STABILITY TEST COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — FINISH RECOVERY SNAPSHOT
# Fix PosixPath JSON serialization
# ============================================================

import os
import json
import glob

PERSIST_DIR = "/kaggle/working/phase4_persistent_recovery"

# Force all paths to plain strings
train_manifest_str = str(TRAIN_MANIFEST)
val_manifest_str = str(VAL_MANIFEST)
test_manifest_str = str(TEST_MANIFEST)
index_path_str = str(INDEX_PATH)
tokenizer_path_str = str(TOKENIZER_PATH)

recovery_config = {
    "phase": 4,

    "model_checkpoint": "phase4_model_optimizer_checkpoint.pt",
    "source_index": "source_index.jsonl",

    "train_manifest": train_manifest_str,
    "validation_manifest": val_manifest_str,
    "test_manifest": test_manifest_str,

    "tokenizer_path": tokenizer_path_str,

    "model": {
        "vocab_size": 32000,
        "context_length": 4096,
        "n_layer": 12,
        "n_head": 12,
        "n_embd": 768,
        "parameters": 112777728,
        "dtype": "torch.float32",
    },

    "dataset": {
        "dataset_mode": "streaming",
        "packing": "document_aware",
        "document_boundary": True,
        "context_length": 4096,
        "seed": 42,
    },

    "training": {
        "batch_size": 1,
        "gradient_accumulation_steps": 8,
        "effective_batch_size": 8,
        "optimizer": "AdamW",
        "learning_rate": 5e-5,
        "weight_decay": 0.01,
        "max_training_steps": 1000,
    },

    "checkpointing": {
        "original_checkpoint_every": 250,
        "persistent_snapshot": True,
    },

    "architecture_status": {
        "source_corpus": True,
        "train_manifest": True,
        "validation_manifest": True,
        "test_manifest": True,
        "tokenizer": True,
        "tokenizer_config": True,
        "streaming_config": True,
        "source_index": True,
        "model_state": True,
        "optimizer_state": True,
        "model_hyperparameters": True,
        "dataset_hyperparameters": True,
        "training_hyperparameters": True,
        "dataset_class_in_notebook": True,
    },
}

config_path = os.path.join(
    PERSIST_DIR,
    "phase4_recovery_config.json"
)

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(recovery_config, f, indent=2)

print("=" * 60)
print("RECOVERY CONFIG SAVED")
print("=" * 60)

# Verify all expected recovery artifacts
expected = [
    "phase4_model_optimizer_checkpoint.pt",
    "source_index.jsonl",
    "phase4_recovery_config.json",
]

all_ok = True

for name in expected:
    path = os.path.join(PERSIST_DIR, name)

    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 ** 2)
        print(f"✓ {name:<45} {size_mb:,.2f} MB")
    else:
        print(f"✗ MISSING: {name}")
        all_ok = False

print()
print("Current training step    :", globals().get("global_step", "unknown"))
print("Current optimizer step   :", globals().get("optimizer_steps", "unknown"))

print()

if all_ok:
    print("✓ ALL RECOVERY ARTIFACTS PRESENT")
else:
    raise RuntimeError("Recovery package is incomplete.")

print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4 — FINAL DURABLE RECOVERY BUNDLE
# RUN THIS BEFORE RESTARTING TRAINING
# ============================================================

from pathlib import Path
import json
import shutil
import hashlib
import os
import inspect
import torch

CHECKPOINT_ROOT = Path("/kaggle/working/phase4_recovery")
BUNDLE_ROOT = CHECKPOINT_ROOT / "durable_recovery_bundle"

BUNDLE_ROOT.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("PHASE 4 — BUILDING DURABLE RECOVERY BUNDLE")
print("=" * 60)

# ------------------------------------------------------------
# 1. KNOWN DURABLE DATASET ARTIFACTS
# ------------------------------------------------------------

DATASET_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

ARTIFACTS = {
    "train_manifest": DATASET_ROOT / "manifests/train_manifest.jsonl",
    "validation_manifest": DATASET_ROOT / "manifests/validation_manifest.jsonl",
    "test_manifest": DATASET_ROOT / "manifests/test_manifest.jsonl",

    "tokenizer": DATASET_ROOT / "tokenizer/legal_bpe_32k.json",
    "tokenizer_config": DATASET_ROOT / "tokenizer/tokenizer_config.json",
    "training_config": DATASET_ROOT / "training_example_config.json",
    "streaming_config": DATASET_ROOT / "streaming_config.json",

    "initialization_config":
        DATASET_ROOT / "phase4_initialization.json",

    "dedup_audit":
        DATASET_ROOT / "phase4_dedup_audit.json",

    "duplicate_audit":
        DATASET_ROOT / "phase4_duplicate_audit.json",

    "split_policy":
        DATASET_ROOT / "phase4_dedup_split_policy.json",

    "text_quality_audit":
        DATASET_ROOT / "phase4_text_quality_audit.json",

    "pre_tokenization_checkpoint":
        DATASET_ROOT / "phase4_pre_tokenization_checkpoint.json",
}

# ------------------------------------------------------------
# 2. CURRENT WORKING ARTIFACTS
# ------------------------------------------------------------

WORKING_ARTIFACTS = {
    "source_index":
        CHECKPOINT_ROOT / "indexes/source_index.jsonl",

    "model_optimizer_checkpoint":
        CHECKPOINT_ROOT / "phase4_model_optimizer_checkpoint.pt",

    "recovery_config":
        CHECKPOINT_ROOT / "phase4_recovery_config.json",
}

# ------------------------------------------------------------
# 3. VERIFY EVERYTHING FIRST
# ------------------------------------------------------------

print("\nDURABLE DATASET ARTIFACTS")
print("-" * 60)

missing = []

for name, path in ARTIFACTS.items():
    exists = path.exists()
    print(f"{name:30s}: {'✓' if exists else '✗'} {path}")

    if not exists:
        missing.append(str(path))

print("\nWORKING RECOVERY ARTIFACTS")
print("-" * 60)

for name, path in WORKING_ARTIFACTS.items():
    exists = path.exists()
    size = path.stat().st_size / (1024**2) if exists else 0

    print(
        f"{name:30s}: "
        f"{'✓' if exists else '✗'} "
        f"{size:,.2f} MB | {path}"
    )

    if not exists:
        missing.append(str(path))

if missing:
    print("\n❌ MISSING ARTIFACTS")
    for x in missing:
        print(" ", x)

    raise RuntimeError(
        "Do not restart training yet. Recovery artifacts are missing."
    )

print("\n✓ ALL REQUIRED ARTIFACTS FOUND")

# ------------------------------------------------------------
# 4. COPY DURABLE ARTIFACTS INTO RECOVERY BUNDLE
# ------------------------------------------------------------

print("\nCOPYING ARTIFACTS")
print("-" * 60)

for name, src in ARTIFACTS.items():
    dst = BUNDLE_ROOT / "dataset_artifacts" / src.name
    dst.parent.mkdir(parents=True, exist_ok=True)

    shutil.copy2(src, dst)

    print(f"✓ {name}")

for name, src in WORKING_ARTIFACTS.items():
    dst = BUNDLE_ROOT / "training_artifacts" / src.name
    dst.parent.mkdir(parents=True, exist_ok=True)

    shutil.copy2(src, dst)

    print(f"✓ {name}")

print("\n✓ ARTIFACT COPY COMPLETE")

# ------------------------------------------------------------
# 5. SAVE MODEL ARCHITECTURE SOURCE
# ------------------------------------------------------------

model_source_path = BUNDLE_ROOT / "code" / "model_definition.py"
model_source_path.parent.mkdir(parents=True, exist_ok=True)

model_source = globals().get("model", None)

if model_source is not None:
    model_class = type(model_source)

    try:
        source = inspect.getsource(model_class)

        model_source_path.write_text(
            "# Recovered Phase-4 model definition\n\n"
            + source
            + "\n"
        )

        print("✓ Model class source saved")

    except Exception as e:
        model_source_path.write_text(
            "# Model source could not be recovered via inspect.getsource()\n"
            f"# Reason: {repr(e)}\n"
            f"# Model class: {model_class}\n"
        )

        print(
            "⚠ Model class source unavailable through inspect; "
            "recorded class identity"
        )
else:
    model_source_path.write_text(
        "# Model object was not present in globals() during recovery save.\n"
    )

# ------------------------------------------------------------
# 6. SAVE DATASET CLASS SOURCE
# ------------------------------------------------------------

dataset_source_path = BUNDLE_ROOT / "code" / "dataset_definition.py"

packer_cls = globals().get(
    "WorkerSafeDocumentAwarePacker",
    None
)

if packer_cls is not None:

    try:
        source = inspect.getsource(packer_cls)

        dataset_source_path.write_text(
            "# Recovered Phase-4 streaming dataset\n\n"
            + source
            + "\n"
        )

        print("✓ Dataset class source saved")

    except Exception as e:

        dataset_source_path.write_text(
            "# inspect.getsource() unavailable after reconnect.\n"
            f"# Reason: {repr(e)}\n"
            f"# Class: {packer_cls}\n"
            f"# Module: {packer_cls.__module__}\n"
            "\n"
            "# IMPORTANT INTERNAL METHODS CONFIRMED:\n"
            "# _read_source_record\n"
            "# _extract_text\n"
            "# _tokenize_document\n"
            "# _pack_document\n"
            "# __iter__\n"
        )

        print(
            "⚠ Dataset source unavailable through inspect; "
            "recorded verified class structure"
        )
else:
    dataset_source_path.write_text(
        "# WorkerSafeDocumentAwarePacker was not present.\n"
    )

# ------------------------------------------------------------
# 7. SAVE CURRENT TRAINING STATE
# ------------------------------------------------------------

training_state = {
    "phase": 4,

    "status": "recovery_bundle_created_before_training_restart",

    "model": {
        "parameters": 112_777_728,
        "vocab_size": 32_000,
        "context_length": 4096,
        "n_layer": 12,
        "n_head": 12,
        "n_embd": 768,
        "dtype": "float32",
        "device": "cuda:0",
    },

    "training": {
        "batch_size": 1,
        "gradient_accumulation_steps": 8,
        "effective_batch_size": 8,
        "learning_rate": 5e-5,
        "weight_decay": 0.01,
        "optimizer": "AdamW",
    },

    "streaming": {
        "dataset_mode": "streaming",
        "document_aware_packing": True,
        "document_boundary": True,
        "materialize_tokenized_corpus": False,
        "train_records": 3_020_655,
        "validation_records": 168_665,
        "test_records": 167_274,
        "source_records_indexed": 3_356_594,
    },

    "recovery": {
        "checkpoint_step": 30,
        "optimizer_step": 30,
        "source_index": "training_artifacts/source_index.jsonl",
        "model_optimizer_checkpoint":
            "training_artifacts/phase4_model_optimizer_checkpoint.pt",
        "recovery_config":
            "training_artifacts/phase4_recovery_config.json",
    },

    "verified_tests": {
        "single_streaming_sample": True,
        "single_backward": True,
        "single_optimizer_step": True,
        "ten_step_training_stability": True,
        "gradient_device_mismatches": 0,
    }
}

state_path = BUNDLE_ROOT / "training_state.json"

with open(state_path, "w") as f:
    json.dump(training_state, f, indent=2)

print("✓ Training state saved")

# ------------------------------------------------------------
# 8. CHECKSUM EVERYTHING
# ------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


print("\nGENERATING CHECKSUM MANIFEST")
print("-" * 60)

checksums = {}

for path in sorted(BUNDLE_ROOT.rglob("*")):

    if not path.is_file():
        continue

    rel = str(path.relative_to(BUNDLE_ROOT))

    print("Hashing:", rel)

    checksums[rel] = {
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }

checksum_path = BUNDLE_ROOT / "SHA256SUMS.json"

with open(checksum_path, "w") as f:
    json.dump(checksums, f, indent=2)

print("✓ SHA256 manifest saved")

# ------------------------------------------------------------
# 9. FINAL INVENTORY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("RECOVERY BUNDLE COMPLETE")
print("=" * 60)

total = 0

for path in BUNDLE_ROOT.rglob("*"):
    if path.is_file():
        total += path.stat().st_size

print(f"Bundle path : {BUNDLE_ROOT}")
print(f"Files       : {len(checksums)}")
print(f"Total size  : {total / (1024**3):.2f} GB")

print("\nKEY FILES")
print("-" * 60)

for path in [
    BUNDLE_ROOT / "training_artifacts/source_index.jsonl",
    BUNDLE_ROOT / "training_artifacts/phase4_model_optimizer_checkpoint.pt",
    BUNDLE_ROOT / "training_artifacts/phase4_recovery_config.json",
    BUNDLE_ROOT / "training_state.json",
    BUNDLE_ROOT / "SHA256SUMS.json",
]:
    print(
        f"{path.name:45s} "
        f"{path.stat().st_size / (1024**2):,.2f} MB"
    )

print("\n✓ RECOVERY BUNDLE READY")

In [ ]:
# ============================================================
# PHASE 4 — EXACT RECOVERY ARTIFACT LOCATOR
# DIAGNOSTIC ONLY — NO FILES MODIFIED
# ============================================================

from pathlib import Path

roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"),
]

targets = [
    "phase4_model_optimizer_checkpoint.pt",
    "phase4_recovery_config.json",
    "source_index.jsonl",
    "train_manifest.jsonl",
    "validation_manifest.jsonl",
    "test_manifest.jsonl",
    "legal_bpe_32k.json",
    "tokenizer_config.json",
    "training_example_config.json",
    "streaming_config.json",
    "phase4_initialization.json",
    "phase4_dedup_audit.json",
    "phase4_duplicate_audit.json",
    "phase4_dedup_split_policy.json",
    "phase4_text_quality_audit.json",
    "phase4_pre_tokenization_checkpoint.json",
]

print("=" * 70)
print("PHASE 4 — EXACT RECOVERY ARTIFACT LOCATOR")
print("=" * 70)

found = {}

for root in roots:
    print(f"\nSEARCH ROOT: {root}")
    print("-" * 70)

    if not root.exists():
        print("ROOT DOES NOT EXIST")
        continue

    for target in targets:

        matches = list(root.rglob(target))

        if matches:
            for p in matches:
                size_mb = p.stat().st_size / (1024**2)
                found.setdefault(target, []).append(p)

                print(
                    f"✓ {target:45s} "
                    f"{size_mb:10.2f} MB | {p}"
                )

print("\n" + "=" * 70)
print("RECOVERY ARTIFACT SUMMARY")
print("=" * 70)

for target in targets:
    matches = found.get(target, [])

    if matches:
        print(f"✓ {target}")
        for p in matches:
            print(f"    {p}")
    else:
        print(f"✗ {target}")

print("\n" + "=" * 70)
print("IMPORTANT")
print("=" * 70)

checkpoint_matches = found.get(
    "phase4_model_optimizer_checkpoint.pt", []
)

index_matches = found.get(
    "source_index.jsonl", []
)

config_matches = found.get(
    "phase4_recovery_config.json", []
)

print(
    "Model/optimizer checkpoint :",
    "FOUND" if checkpoint_matches else "MISSING"
)

print(
    "Source index               :",
    "FOUND" if index_matches else "MISSING"
)

print(
    "Recovery config            :",
    "FOUND" if config_matches else "MISSING"
)

print("=" * 70)

In [ ]:
# ============================================================
# PHASE 4 — FINAL PERSISTENT RECOVERY PACKAGE
# ============================================================

from pathlib import Path
import json
import shutil
import hashlib
import os
import inspect
import torch

DATASET_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

WORKING_RECOVERY = Path(
    "/kaggle/working/phase4_persistent_recovery"
)

BUNDLE = WORKING_RECOVERY / "phase4_final_recovery"

BUNDLE.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("PHASE 4 — FINAL PERSISTENT RECOVERY PACKAGE")
print("=" * 70)

# ------------------------------------------------------------
# SOURCE OF TRUTH
# ------------------------------------------------------------

persistent_artifacts = {
    "manifests/train_manifest.jsonl":
        DATASET_ROOT / "manifests/train_manifest.jsonl",

    "manifests/validation_manifest.jsonl":
        DATASET_ROOT / "manifests/validation_manifest.jsonl",

    "manifests/test_manifest.jsonl":
        DATASET_ROOT / "manifests/test_manifest.jsonl",

    "tokenizer/legal_bpe_32k.json":
        DATASET_ROOT / "tokenizer/legal_bpe_32k.json",

    "tokenizer/tokenizer_config.json":
        DATASET_ROOT / "tokenizer/tokenizer_config.json",

    "training_example_config.json":
        DATASET_ROOT / "training_example_config.json",

    "streaming_config.json":
        DATASET_ROOT / "streaming_config.json",

    "phase4_initialization.json":
        DATASET_ROOT / "phase4_initialization.json",

    "phase4_dedup_audit.json":
        DATASET_ROOT / "phase4_dedup_audit.json",

    "phase4_duplicate_audit.json":
        DATASET_ROOT / "phase4_duplicate_audit.json",

    "phase4_dedup_split_policy.json":
        DATASET_ROOT / "phase4_dedup_split_policy.json",

    "phase4_text_quality_audit.json":
        DATASET_ROOT / "phase4_text_quality_audit.json",

    "phase4_pre_tokenization_checkpoint.json":
        DATASET_ROOT / "phase4_pre_tokenization_checkpoint.json",
}

recovery_artifacts = {
    "training/phase4_model_optimizer_checkpoint.pt":
        WORKING_RECOVERY / "phase4_model_optimizer_checkpoint.pt",

    "training/phase4_recovery_config.json":
        WORKING_RECOVERY / "phase4_recovery_config.json",

    "indexes/source_index.jsonl":
        WORKING_RECOVERY / "source_index.jsonl",
}

# ------------------------------------------------------------
# VERIFY
# ------------------------------------------------------------

print("\nVERIFYING INPUTS")
print("-" * 70)

missing = []

for rel, src in {**persistent_artifacts, **recovery_artifacts}.items():

    if not src.exists():
        missing.append((rel, src))
        print(f"✗ {rel}")
    else:
        size = src.stat().st_size / (1024 ** 2)
        print(f"✓ {rel:50s} {size:10.2f} MB")

if missing:
    print("\n❌ MISSING:")
    for rel, src in missing:
        print(" ", rel, "->", src)

    raise RuntimeError(
        "Recovery package was NOT created because artifacts are missing."
    )

print("\n✓ ALL REQUIRED ARTIFACTS FOUND")

# ------------------------------------------------------------
# COPY
# ------------------------------------------------------------

print("\nBUILDING PACKAGE")
print("-" * 70)

for rel, src in {**persistent_artifacts, **recovery_artifacts}.items():

    dst = BUNDLE / rel
    dst.parent.mkdir(parents=True, exist_ok=True)

    shutil.copy2(src, dst)

    print(f"✓ {rel}")

# ------------------------------------------------------------
# CAPTURE CURRENT PYTHON ENVIRONMENT
# ------------------------------------------------------------

env_info = {
    "python": os.sys.version,
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
}

if torch.cuda.is_available():
    env_info["gpu"] = torch.cuda.get_device_name(0)

with open(BUNDLE / "environment.json", "w") as f:
    json.dump(env_info, f, indent=2)

print("✓ environment.json")

# ------------------------------------------------------------
# SAVE VERIFIED ARCHITECTURE CONFIG
# ------------------------------------------------------------

architecture = {
    "phase": 4,
    "vocab_size": 32000,
    "context_length": 4096,
    "n_layer": 12,
    "n_head": 12,
    "n_embd": 768,
    "parameters": 112777728,
    "dtype": "float32",
    "device_used": "cuda:0",
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
}

with open(BUNDLE / "model_architecture.json", "w") as f:
    json.dump(architecture, f, indent=2)

print("✓ model_architecture.json")

# ------------------------------------------------------------
# SAVE VERIFIED TRAINING CONFIG
# ------------------------------------------------------------

training = {
    "batch_size": 1,
    "gradient_accumulation_steps": 8,
    "effective_batch_size": 8,
    "learning_rate": 5e-5,
    "weight_decay": 0.01,
    "optimizer": "AdamW",
    "max_training_steps": 1000,

    "context_length": 4096,

    "dataset_mode": "streaming",
    "document_aware_packing": True,
    "document_boundary": True,
    "materialize_tokenized_corpus": False,

    "seed": 42,

    "train_records": 3020655,
    "validation_records": 168665,
    "test_records": 167274,

    "checkpoint_step": 30,
    "optimizer_step": 30,
}

with open(BUNDLE / "training_state.json", "w") as f:
    json.dump(training, f, indent=2)

print("✓ training_state.json")

# ------------------------------------------------------------
# RECOVERY TEST RESULTS
# ------------------------------------------------------------

tests = {
    "source_corpus_verified": True,
    "manifest_verified": True,
    "tokenizer_verified": True,
    "source_index_built": True,
    "streaming_sample_verified": True,
    "single_backward_verified": True,
    "optimizer_step_verified": True,
    "ten_step_stability_verified": True,
    "gradient_device_mismatches": 0,

    "streaming_records_indexed": 3356594,

    "checkpoint_step": 30,
    "optimizer_step": 30,
}

with open(BUNDLE / "verification_results.json", "w") as f:
    json.dump(tests, f, indent=2)

print("✓ verification_results.json")

# ------------------------------------------------------------
# CHECKSUMS
# ------------------------------------------------------------

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


print("\nCREATING CHECKSUM MANIFEST")
print("-" * 70)

checksums = {}

for p in sorted(BUNDLE.rglob("*")):

    if not p.is_file():
        continue

    rel = str(p.relative_to(BUNDLE))

    checksums[rel] = {
        "size_bytes": p.stat().st_size,
        "sha256": sha256(p),
    }

    print("✓", rel)

with open(BUNDLE / "SHA256SUMS.json", "w") as f:
    json.dump(checksums, f, indent=2)

print("\n✓ SHA256SUMS.json")

# ------------------------------------------------------------
# FINAL MANIFEST
# ------------------------------------------------------------

manifest = {
    "name": "CS781 Phase 4 Final Recovery Package",
    "phase": 4,
    "created_from": str(WORKING_RECOVERY),
    "source_corpus": {
        "persistent": True,
        "files": 72,
        "size_gb": 15.59,
    },
    "training": {
        "checkpoint_step": 30,
        "optimizer_step": 30,
        "max_steps": 1000,
    },
    "model": architecture,
    "files": sorted(checksums.keys()),
}

with open(BUNDLE / "RECOVERY_MANIFEST.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("✓ RECOVERY_MANIFEST.json")

# ------------------------------------------------------------
# SIZE
# ------------------------------------------------------------

total_bytes = sum(
    p.stat().st_size
    for p in BUNDLE.rglob("*")
    if p.is_file()
)

print("\n" + "=" * 70)
print("FINAL RECOVERY PACKAGE READY")
print("=" * 70)

print("Path :", BUNDLE)
print("Files:", len(list(BUNDLE.rglob("*"))))
print("Size :", f"{total_bytes / (1024**3):.2f} GB")

print("\nCritical artifacts:")
print(
    "  checkpoint :",
    (BUNDLE / "training/phase4_model_optimizer_checkpoint.pt").exists()
)
print(
    "  index      :",
    (BUNDLE / "indexes/source_index.jsonl").exists()
)
print(
    "  manifests  :",
    (BUNDLE / "manifests/train_manifest.jsonl").exists()
)
print(
    "  tokenizer  :",
    (BUNDLE / "tokenizer/legal_bpe_32k.json").exists()
)
print(
    "  verification:",
    (BUNDLE / "verification_results.json").exists()
)

print("\n✓ PACKAGE BUILD COMPLETE")

In [ ]:
from pathlib import Path
import os

PACKAGE = Path("/kaggle/working/phase4_persistent_recovery/phase4_final_recovery")

print("=" * 70)
print("FINAL PACKAGE CHECK BEFORE PERSISTING")
print("=" * 70)

required = [
    "RECOVERY_MANIFEST.json",
    "SHA256SUMS.json",
    "environment.json",
    "model_architecture.json",
    "training_state.json",
    "verification_results.json",

    "manifests/train_manifest.jsonl",
    "manifests/validation_manifest.jsonl",
    "manifests/test_manifest.jsonl",

    "tokenizer/legal_bpe_32k.json",
    "tokenizer/tokenizer_config.json",

    "indexes/source_index.jsonl",

    "training/phase4_model_optimizer_checkpoint.pt",
    "training/phase4_recovery_config.json",
]

missing = []

for rel in required:
    p = PACKAGE / rel
    if p.exists():
        size_mb = p.stat().st_size / (1024**2)
        print(f"✓ {rel:<55} {size_mb:,.2f} MB")
    else:
        print(f"✗ MISSING: {rel}")
        missing.append(rel)

print("=" * 70)

if missing:
    raise RuntimeError(
        "DO NOT PUSH YET. Missing artifacts:\n" +
        "\n".join(missing)
    )

print("✓ ALL CRITICAL ARTIFACTS PRESENT")
print(f"Package: {PACKAGE}")
print("=" * 70)

In [ ]:
import os
import subprocess
from pathlib import Path

print("Kaggle environment:")
for k in [
    "KAGGLE_USERNAME",
    "KAGGLE_KERNEL_RUN_TYPE",
    "KAGGLE_URL_BASE",
]:
    print(f"{k}: {os.environ.get(k)}")

print("\nKaggle config locations:")
for p in [
    Path("/root/.kaggle/kaggle.json"),
    Path("/home/sandbox/.kaggle/kaggle.json"),
    Path("/kaggle/.kaggle/kaggle.json"),
]:
    print(f"{p}: {p.exists()}")

print("\nKaggle CLI:")
r = subprocess.run(
    ["which", "kaggle"],
    capture_output=True,
    text=True
)
print(r.stdout.strip() or "NOT FOUND")

In [ ]:
# ============================================================
# PHASE 4 — PUSH FINAL RECOVERY PACKAGE TO KAGGLE DATASET
# ============================================================

from pathlib import Path
import json
import os
import subprocess
import sys

PACKAGE = Path(
    "/kaggle/working/phase4_persistent_recovery/phase4_final_recovery"
)

DATASET_SLUG = "cs781-phase4-final-recovery"

print("=" * 70)
print("PHASE 4 — PERSISTENT DATASET UPLOAD")
print("=" * 70)

# ------------------------------------------------------------
# 1. Verify package
# ------------------------------------------------------------

if not PACKAGE.exists():
    raise RuntimeError(f"Recovery package does not exist: {PACKAGE}")

checkpoint = PACKAGE / "training/phase4_model_optimizer_checkpoint.pt"
index_file = PACKAGE / "indexes/source_index.jsonl"
manifest = PACKAGE / "RECOVERY_MANIFEST.json"

for p in [checkpoint, index_file, manifest]:
    if not p.exists():
        raise RuntimeError(f"Critical artifact missing: {p}")

print("✓ Recovery package exists")
print(f"  {PACKAGE}")

# ------------------------------------------------------------
# 2. Find Kaggle username
# ------------------------------------------------------------

username = os.environ.get("KAGGLE_USERNAME")

if not username:
    # Try Kaggle config
    config_path = Path.home() / ".kaggle" / "kaggle.json"

    if config_path.exists():
        with open(config_path) as f:
            cfg = json.load(f)
        username = cfg.get("username")

if not username:
    raise RuntimeError(
        "Kaggle username not found. "
        "Set KAGGLE_USERNAME or make sure ~/.kaggle/kaggle.json exists."
    )

print(f"✓ Kaggle username: {username}")

# ------------------------------------------------------------
# 3. Create/update dataset metadata
# ------------------------------------------------------------

metadata = {
    "title": "CS781 Phase 4 Final Recovery",
    "id": f"{username}/{DATASET_SLUG}",
    "licenses": [
        {
            "name": "CC0-1.0"
        }
    ],
    "description": (
        "CS781 Phase 4 persistent recovery package containing "
        "model/optimizer checkpoint, source index, manifests, tokenizer, "
        "training state, architecture metadata, verification results, "
        "checksums, and recovery configuration."
    )
}

metadata_path = PACKAGE / "dataset-metadata.json"

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("✓ dataset-metadata.json created")
print(f"  Dataset ID: {username}/{DATASET_SLUG}")

# ------------------------------------------------------------
# 4. Check Kaggle CLI
# ------------------------------------------------------------

try:
    result = subprocess.run(
        ["kaggle", "--version"],
        capture_output=True,
        text=True,
        check=True
    )
    print(f"✓ {result.stdout.strip()}")
except Exception as e:
    raise RuntimeError(
        "Kaggle CLI is unavailable in this notebook."
    ) from e

# ------------------------------------------------------------
# 5. Push dataset
# ------------------------------------------------------------

print()
print("=" * 70)
print("UPLOADING RECOVERY PACKAGE")
print("=" * 70)

cmd = [
    "kaggle",
    "datasets",
    "create",
    "-p",
    str(PACKAGE),
    "-r",
    "zip",
]

print("Running Kaggle Dataset upload...")
print(f"Dataset: {username}/{DATASET_SLUG}")
print()

result = subprocess.run(
    cmd,
    text=True
)

if result.returncode != 0:
    raise RuntimeError(
        f"Kaggle Dataset upload failed with exit code "
        f"{result.returncode}"
    )

print()
print("=" * 70)
print("✓ RECOVERY DATASET PUSH COMPLETE")
print("=" * 70)
print(f"Dataset: {username}/{DATASET_SLUG}")
print()
print("IMPORTANT:")
print("Attach this Dataset to the notebook before resuming training.")
print("Do NOT delete the current /kaggle/working recovery package yet.")
print("=" * 70)

In [ ]:
import os
os.environ["KAGGLE_USERNAME"] = "oipranesh"

In [ ]:
from pathlib import Path
import json
import hashlib

# Kaggle normally mounts it here.
RECOVERY = Path("/kaggle/input/datasets/oipranesh/cs781-phase4-final-recovery")

print("=" * 70)
print("PHASE 4 — PERSISTENT RECOVERY DATASET VERIFICATION")
print("=" * 70)
print("Path:", RECOVERY)

required = [
    "RECOVERY_MANIFEST.json",
    "SHA256SUMS.json",
    "environment.json",
    "model_architecture.json",
    "training_state.json",
    "verification_results.json",

    "manifests/train_manifest.jsonl",
    "manifests/validation_manifest.jsonl",
    "manifests/test_manifest.jsonl",

    "tokenizer/legal_bpe_32k.json",
    "tokenizer/tokenizer_config.json",

    "indexes/source_index.jsonl",

    "training/phase4_model_optimizer_checkpoint.pt",
    "training/phase4_recovery_config.json",
]

missing = []

for rel in required:
    p = RECOVERY / rel

    if p.exists():
        size_mb = p.stat().st_size / (1024 ** 2)
        print(f"✓ {rel:<55} {size_mb:,.2f} MB")
    else:
        print(f"✗ MISSING: {rel}")
        missing.append(rel)

print("=" * 70)

if missing:
    raise RuntimeError(
        "PERSISTENT RECOVERY PACKAGE IS INCOMPLETE:\n"
        + "\n".join(missing)
    )

# ------------------------------------------------------------
# Verify recovery manifest
# ------------------------------------------------------------

manifest_path = RECOVERY / "RECOVERY_MANIFEST.json"

try:
    with open(manifest_path) as f:
        recovery_manifest = json.load(f)

    print("✓ RECOVERY_MANIFEST.json readable")
except Exception as e:
    raise RuntimeError(
        f"Recovery manifest could not be read: {e}"
    )

# ------------------------------------------------------------
# Verify checkpoint can at least be read
# ------------------------------------------------------------

checkpoint = RECOVERY / "training/phase4_model_optimizer_checkpoint.pt"

with open(checkpoint, "rb") as f:
    header = f.read(16)

if len(header) == 0:
    raise RuntimeError("Checkpoint is empty.")

print("✓ Model/optimizer checkpoint readable")
print(f"  Size: {checkpoint.stat().st_size / (1024**2):,.2f} MB")

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("=" * 70)
print("✓ PERSISTENT RECOVERY PACKAGE VERIFIED")
print("=" * 70)
print()
print("Checkpoint       : PRESENT")
print("Source index     : PRESENT")
print("Manifests        : PRESENT")
print("Tokenizer        : PRESENT")
print("Training state   : PRESENT")
print("Recovery metadata: PRESENT")
print("Checksums        : PRESENT")
print()
print("✓ SAFE TO PROCEED TO RESUME TRAINING")
print("=" * 70)

In [ ]:
# ============================================================
# PHASE 4 — STABILIZED PRODUCTION RESUME
# SAFE RESUME + DETERMINISTIC STREAM RECOVERY
# GRADIENT CLIPPING + FINITE-STATE CHECKS
# STEP-SPECIFIC + ROLLING CHECKPOINTS
# ============================================================

import os
import gc
import json
import time
import hashlib
import math
import shutil
import torch


# ============================================================
# CONFIGURATION
# ============================================================

DEVICE = torch.device("cuda:0")

CONTEXT_LENGTH = 4096
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8

MAX_TRAINING_STEPS = 1000

LOG_EVERY = 10
CHECKPOINT_EVERY = 50

# ============================================================
# STABILITY SETTINGS
# ============================================================

# Original training LR was 5e-5.
# Use a lower LR for the stabilized continuation.
RESUME_LEARNING_RATE = 1e-5

# Maximum global gradient norm.
MAX_GRAD_NORM = 1.0


# ============================================================
# CHECKPOINT DIRECTORY
# ============================================================

CHECKPOINT_DIR = (
    "/kaggle/working/phase4_checkpoints"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)


# ============================================================
# CHECKPOINT PATHS
# ============================================================

VERIFIED_STEP_500 = os.path.join(
    CHECKPOINT_DIR,
    "checkpoint_step_000500.pt"
)

LATEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "checkpoint_latest.pt"
)

LATEST_CHECKPOINT_TMP = (
    LATEST_CHECKPOINT + ".tmp"
)

RECOVERY_STATE_PATH = os.path.join(
    CHECKPOINT_DIR,
    "training_recovery_state.json"
)

TRAINING_LOG_PATH = os.path.join(
    CHECKPOINT_DIR,
    "training_log.jsonl"
)


# ============================================================
# BASIC ENVIRONMENT CHECKS
# ============================================================

print("=" * 70)
print("PHASE 4 — STABILIZED SAFE RESUME")
print("=" * 70)

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA is not available. "
        "Training has NOT started."
    )

if DEVICE.index != 0:

    raise RuntimeError(
        f"Expected cuda:0, got {DEVICE}"
    )

if model is None:

    raise RuntimeError(
        "Model is not defined."
    )

if optimizer is None:

    raise RuntimeError(
        "Optimizer is not defined."
    )

if train_loader is None:

    raise RuntimeError(
        "train_loader is not defined."
    )

print(
    "GPU:",
    torch.cuda.get_device_name(DEVICE)
)

print(
    "GPU memory:",
    f"{torch.cuda.get_device_properties(DEVICE).total_memory / 1024**3:.2f} GB"
)


# ============================================================
# CHECKPOINT DISCOVERY
# ============================================================

resume_candidates = []


# ------------------------------------------------------------
# Rolling checkpoint
# ------------------------------------------------------------

if os.path.isfile(
    LATEST_CHECKPOINT
):

    resume_candidates.append(
        LATEST_CHECKPOINT
    )


# ------------------------------------------------------------
# Step-specific checkpoints
# ------------------------------------------------------------

for filename in os.listdir(
    CHECKPOINT_DIR
):

    if not filename.startswith(
        "checkpoint_step_"
    ):
        continue

    if not filename.endswith(
        ".pt"
    ):
        continue

    path = os.path.join(
        CHECKPOINT_DIR,
        filename
    )

    if os.path.isfile(path):

        resume_candidates.append(
            path
        )


# ------------------------------------------------------------
# Verified step-500 fallback
# ------------------------------------------------------------

if os.path.isfile(
    VERIFIED_STEP_500
):

    resume_candidates.append(
        VERIFIED_STEP_500
    )


if not resume_candidates:

    raise FileNotFoundError(
        "\n"
        "NO USABLE CHECKPOINT FOUND.\n\n"
        "Training has NOT started."
    )


# ============================================================
# VALIDATE CHECKPOINT CANDIDATES
# ============================================================

print()
print("=" * 70)
print("CHECKPOINT CANDIDATES")
print("=" * 70)

required_keys = {
    "phase",
    "step",
    "optimizer_steps",
    "model_state_dict",
    "optimizer_state_dict",
    "context_length",
    "batch_size",
    "gradient_accumulation_steps",
}

valid_candidates = []

for path in resume_candidates:

    try:

        print()
        print(
            "Checking:",
            os.path.basename(path)
        )

        checkpoint = torch.load(
            path,
            map_location="cpu"
        )

        missing_keys = (
            required_keys
            - set(checkpoint.keys())
        )

        if missing_keys:

            print(
                "  ✗ Missing keys:",
                sorted(missing_keys)
            )

            del checkpoint
            continue


        step = int(
            checkpoint["step"]
        )

        optimizer_step = int(
            checkpoint[
                "optimizer_steps"
            ]
        )


        if step != optimizer_step:

            print(
                "  ✗ Step / optimizer-step mismatch"
            )

            del checkpoint
            continue


        if (
            int(
                checkpoint[
                    "context_length"
                ]
            )
            != CONTEXT_LENGTH
        ):

            print(
                "  ✗ Context length mismatch"
            )

            del checkpoint
            continue


        if (
            int(
                checkpoint[
                    "batch_size"
                ]
            )
            != BATCH_SIZE
        ):

            print(
                "  ✗ Batch size mismatch"
            )

            del checkpoint
            continue


        if (
            int(
                checkpoint[
                    "gradient_accumulation_steps"
                ]
            )
            != GRADIENT_ACCUMULATION_STEPS
        ):

            print(
                "  ✗ Gradient accumulation mismatch"
            )

            del checkpoint
            continue


        if (
            checkpoint.get("phase")
            != 4
        ):

            print(
                "  ✗ Phase mismatch"
            )

            del checkpoint
            continue


        model_state = checkpoint[
            "model_state_dict"
        ]

        optimizer_state = checkpoint[
            "optimizer_state_dict"
        ]


        if not isinstance(
            model_state,
            dict
        ):

            print(
                "  ✗ Invalid model state"
            )

            del checkpoint
            continue


        if not isinstance(
            optimizer_state,
            dict
        ):

            print(
                "  ✗ Invalid optimizer state"
            )

            del checkpoint
            continue


        print(
            f"  ✓ Valid checkpoint at step {step}"
        )

        print(
            f"  Size: "
            f"{os.path.getsize(path) / 1024**3:.3f} GB"
        )

        valid_candidates.append(
            (
                step,
                path
            )
        )

        del checkpoint


    except Exception as exc:

        print(
            "  ✗ Could not load:",
            repr(exc)
        )


gc.collect()


if not valid_candidates:

    raise RuntimeError(
        "\n"
        "NO VALID CHECKPOINT COULD BE LOADED.\n\n"
        "Training has NOT started."
    )


# ============================================================
# SELECT HIGHEST VALID CHECKPOINT
# ============================================================

valid_candidates.sort(
    key=lambda x: x[0],
    reverse=True
)

RESUME_STEP, RESUME_CHECKPOINT = (
    valid_candidates[0]
)


print()
print("=" * 70)
print("SELECTED RESUME CHECKPOINT")
print("=" * 70)

print(
    "Path:",
    RESUME_CHECKPOINT
)

print(
    "Step:",
    RESUME_STEP
)

print(
    "Size:",
    f"{os.path.getsize(RESUME_CHECKPOINT) / 1024**3:.3f} GB"
)


# ============================================================
# SAFETY CONDITION
# ============================================================

# Your verified recovery point is step 950.
# Do not silently fall back to step 500.

if RESUME_STEP < 950:

    raise RuntimeError(
        "\n"
        f"Highest available checkpoint is step "
        f"{RESUME_STEP}.\n\n"
        "The verified recovery point is step 950.\n"
        "Do NOT resume from an older checkpoint "
        "without explicitly auditing it.\n\n"
        "Training has NOT started."
    )


if RESUME_STEP >= MAX_TRAINING_STEPS:

    print()
    print(
        "Training target already reached."
    )

    raise SystemExit


# ============================================================
# SANITY CHECKS
# ============================================================

assert DEVICE.type == "cuda"

model = model.to(
    DEVICE
)

model.train()


print()
print("=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print(
    "Context length:",
    CONTEXT_LENGTH
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Gradient accumulation:",
    GRADIENT_ACCUMULATION_STEPS
)

print(
    "Effective batch:",
    BATCH_SIZE
    * GRADIENT_ACCUMULATION_STEPS
)

print(
    "Resume step:",
    RESUME_STEP
)

print(
    "Target step:",
    MAX_TRAINING_STEPS
)

print(
    "Remaining steps:",
    MAX_TRAINING_STEPS
    - RESUME_STEP
)

print(
    "Resume learning rate:",
    RESUME_LEARNING_RATE
)

print(
    "Maximum gradient norm:",
    MAX_GRAD_NORM
)


# ============================================================
# MODEL DEVICE CHECK
# ============================================================

for name, parameter in (
    model.named_parameters()
):

    if parameter.device != DEVICE:

        raise RuntimeError(
            f"Model parameter on wrong device: "
            f"{name} -> {parameter.device}"
        )


print(
    "✓ Model parameters on cuda:0"
)


# ============================================================
# LOAD RESUME CHECKPOINT
# ============================================================

print()
print("=" * 70)
print("LOADING RESUME CHECKPOINT")
print("=" * 70)

checkpoint = torch.load(
    RESUME_CHECKPOINT,
    map_location=DEVICE
)


# ------------------------------------------------------------
# Re-validate state
# ------------------------------------------------------------

if int(
    checkpoint["step"]
) != RESUME_STEP:

    raise RuntimeError(
        "Checkpoint step changed unexpectedly."
    )


if int(
    checkpoint["optimizer_steps"]
) != RESUME_STEP:

    raise RuntimeError(
        "Checkpoint optimizer-step mismatch."
    )


# ============================================================
# RESTORE MODEL
# ============================================================

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

print(
    "✓ Model state restored"
)


# ============================================================
# RESTORE OPTIMIZER
# ============================================================

optimizer.load_state_dict(
    checkpoint[
        "optimizer_state_dict"
    ]
)

print(
    "✓ Optimizer state restored"
)


global_step = int(
    checkpoint["step"]
)

optimizer_steps = int(
    checkpoint["optimizer_steps"]
)


print(
    "✓ Training step:",
    global_step
)

print(
    "✓ Optimizer step:",
    optimizer_steps
)


# ============================================================
# LOWER LEARNING RATE
# ============================================================

print()
print("=" * 70)
print("APPLYING STABILIZED LEARNING RATE")
print("=" * 70)

for group in optimizer.param_groups:

    old_lr = float(
        group["lr"]
    )

    group["lr"] = (
        RESUME_LEARNING_RATE
    )

    print(
        f"LR: {old_lr:.8g} -> "
        f"{group['lr']:.8g}"
    )


# ============================================================
# RECHECK MODEL + OPTIMIZER DEVICES
# ============================================================

for name, parameter in (
    model.named_parameters()
):

    if parameter.device != DEVICE:

        raise RuntimeError(
            f"Restored parameter on wrong device: "
            f"{name} -> {parameter.device}"
        )


for group in optimizer.param_groups:

    for parameter in group["params"]:

        if parameter.device != DEVICE:

            raise RuntimeError(
                "Restored optimizer parameter "
                "is not on cuda:0."
            )


print(
    "✓ Restored model/optimizer device state verified"
)


# ============================================================
# VERIFY CURRENT PARAMETERS ARE FINITE
# ============================================================

print()
print(
    "Checking restored model parameters..."
)

for name, parameter in (
    model.named_parameters()
):

    if not torch.isfinite(
        parameter.detach()
    ).all():

        raise RuntimeError(
            "\n"
            "RESTORED MODEL CONTAINS "
            "NON-FINITE PARAMETERS.\n"
            f"Parameter: {name}\n\n"
            "Training has NOT started."
        )


print(
    "✓ All restored model parameters are finite"
)


# ============================================================
# VERIFY OPTIMIZER STATE IS FINITE
# ============================================================

print(
    "Checking restored optimizer state..."
)

for parameter, state in (
    optimizer.state.items()
):

    for state_name, value in (
        state.items()
    ):

        if torch.is_tensor(value):

            if not torch.isfinite(
                value
            ).all():

                raise RuntimeError(
                    "\n"
                    "RESTORED OPTIMIZER STATE "
                    "CONTAINS NON-FINITE VALUES.\n"
                    f"State: {state_name}\n\n"
                    "Training has NOT started."
                )


print(
    "✓ Restored optimizer state is finite"
)


del checkpoint

gc.collect()
torch.cuda.empty_cache()


# ============================================================
# STREAMING ITERATOR
# ============================================================

train_iter = iter(
    train_loader
)

print(
    "✓ Fresh streaming iterator created"
)


# ============================================================
# DETERMINISTIC STREAM RESTORE
# ============================================================

MICROBATCHES_ALREADY_CONSUMED = (
    global_step
    * GRADIENT_ACCUMULATION_STEPS
)


print()
print("=" * 70)
print("RESTORING STREAM POSITION")
print("=" * 70)

print(
    "Completed optimizer steps:",
    global_step
)

print(
    "Microbatches to skip:",
    MICROBATCHES_ALREADY_CONSUMED
)


stream_resume_start = time.time()


for skip_idx in range(
    MICROBATCHES_ALREADY_CONSUMED
):

    try:

        batch = next(
            train_iter
        )

    except StopIteration:

        print(
            "Streaming iterator reached end "
            "during recovery; restarting iterator."
        )

        train_iter = iter(
            train_loader
        )

        batch = next(
            train_iter
        )

    del batch


    if (
        (skip_idx + 1) % 500 == 0
        or
        (skip_idx + 1)
        == MICROBATCHES_ALREADY_CONSUMED
    ):

        print(
            f"  stream position: "
            f"{skip_idx + 1}/"
            f"{MICROBATCHES_ALREADY_CONSUMED}"
        )


stream_resume_time = (
    time.time()
    - stream_resume_start
)


print()
print(
    "✓ Stream position restored"
)

print(
    "Stream restore time:",
    f"{stream_resume_time:.2f} seconds"
)


# ============================================================
# SHA256 HELPER
# ============================================================

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


# ============================================================
# RECOVERY STATE HELPER
# ============================================================

def save_recovery_state():

    state = {

        "phase":
            4,

        "training_step":
            int(global_step),

        "optimizer_steps":
            int(optimizer_steps),

        "context_length":
            CONTEXT_LENGTH,

        "batch_size":
            BATCH_SIZE,

        "gradient_accumulation_steps":
            GRADIENT_ACCUMULATION_STEPS,

        "stream_microbatches_consumed":
            int(
                global_step
                * GRADIENT_ACCUMULATION_STEPS
            ),

        "resume_source":
            RESUME_CHECKPOINT,

        "resume_learning_rate":
            RESUME_LEARNING_RATE,

        "max_grad_norm":
            MAX_GRAD_NORM,

        "latest_checkpoint":
            LATEST_CHECKPOINT,

        "train_manifest":
            str(
                TRAIN_MANIFEST
            ),

        "validation_manifest":
            str(
                globals().get(
                    "VAL_MANIFEST",
                    ""
                )
            ),

        "test_manifest":
            str(
                globals().get(
                    "TEST_MANIFEST",
                    ""
                )
            ),

        "index_path":
            str(
                INDEX_PATH
            ),

        "tokenizer_path":
            str(
                TOKENIZER_PATH
            ),

        "saved_at_unix":
            time.time(),
    }


    tmp_state = (
        RECOVERY_STATE_PATH
        + ".tmp"
    )


    with open(
        tmp_state,
        "w"
    ) as f:

        json.dump(
            state,
            f,
            indent=2
        )


    os.replace(
        tmp_state,
        RECOVERY_STATE_PATH
    )


# ============================================================
# TRAINING LOG HELPER
# ============================================================

def append_training_log(
    step,
    loss,
    elapsed_seconds,
    grad_norm=None,
    clipped_grad_norm=None
):

    record = {

        "step":
            int(step),

        "loss":
            float(loss),

        "elapsed_seconds":
            float(elapsed_seconds),

        "timestamp":
            time.time(),

        "learning_rate":
            float(
                optimizer.param_groups[0]["lr"]
            ),

        "grad_norm":
            (
                float(grad_norm)
                if grad_norm is not None
                else None
            ),

        "clipped_grad_norm":
            (
                float(
                    clipped_grad_norm
                )
                if clipped_grad_norm
                is not None
                else None
            ),
    }


    with open(
        TRAINING_LOG_PATH,
        "a"
    ) as f:

        f.write(
            json.dumps(record)
            + "\n"
        )


# ============================================================
# CHECKPOINT SAVE FUNCTION
# ============================================================

def save_checkpoint(
    step
):

    print()
    print("=" * 70)
    print(
        f"SAVING CHECKPOINT AT STEP {step}"
    )
    print("=" * 70)


    step_checkpoint_path = os.path.join(
        CHECKPOINT_DIR,
        f"checkpoint_step_{step:06d}.pt"
    )


    checkpoint = {

        "phase":
            4,

        "step":
            int(step),

        "optimizer_steps":
            int(optimizer_steps),

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "context_length":
            CONTEXT_LENGTH,

        "batch_size":
            BATCH_SIZE,

        "gradient_accumulation_steps":
            GRADIENT_ACCUMULATION_STEPS,

        "train_manifest":
            str(
                TRAIN_MANIFEST
            ),

        "validation_manifest":
            str(
                globals().get(
                    "VAL_MANIFEST",
                    ""
                )
            ),

        "test_manifest":
            str(
                globals().get(
                    "TEST_MANIFEST",
                    ""
                )
            ),

        "index_path":
            str(
                INDEX_PATH
            ),

        "tokenizer_path":
            str(
                TOKENIZER_PATH
            ),

        "checkpoint_type":
            "production_training_stabilized",

        "learning_rate":
            RESUME_LEARNING_RATE,

        "max_grad_norm":
            MAX_GRAD_NORM,

        "created_at_unix":
            time.time(),
    }


    # ========================================================
    # STEP-SPECIFIC CHECKPOINT
    # ========================================================

    step_tmp = (
        step_checkpoint_path
        + ".tmp"
    )


    if os.path.exists(
        step_tmp
    ):

        os.remove(
            step_tmp
        )


    torch.save(
        checkpoint,
        step_tmp
    )


    if not os.path.isfile(
        step_tmp
    ):

        raise RuntimeError(
            "Step checkpoint temporary file "
            "was not created."
        )


    step_size = os.path.getsize(
        step_tmp
    )


    print(
        "Step temporary size:",
        f"{step_size / 1024**3:.3f} GB"
    )


    if step_size < 1_000_000_000:

        os.remove(
            step_tmp
        )

        raise RuntimeError(
            "Step checkpoint is unexpectedly small."
        )


    os.replace(
        step_tmp,
        step_checkpoint_path
    )


    print(
        "✓ Step checkpoint saved:",
        step_checkpoint_path
    )


    # ========================================================
    # ROLLING CHECKPOINT
    # ========================================================

    if os.path.exists(
        LATEST_CHECKPOINT_TMP
    ):

        os.remove(
            LATEST_CHECKPOINT_TMP
        )


    torch.save(
        checkpoint,
        LATEST_CHECKPOINT_TMP
    )


    latest_tmp_size = os.path.getsize(
        LATEST_CHECKPOINT_TMP
    )


    print(
        "Rolling temporary size:",
        f"{latest_tmp_size / 1024**3:.3f} GB"
    )


    if latest_tmp_size < 1_000_000_000:

        os.remove(
            LATEST_CHECKPOINT_TMP
        )

        raise RuntimeError(
            "Rolling checkpoint is unexpectedly small.\n"
            "Existing rolling checkpoint was NOT replaced."
        )


    os.replace(
        LATEST_CHECKPOINT_TMP,
        LATEST_CHECKPOINT
    )


    print(
        "✓ Rolling checkpoint updated"
    )


    print(
        "Rolling checkpoint SHA256:",
        sha256_file(
            LATEST_CHECKPOINT
        )
    )


    save_recovery_state()


    print(
        "✓ Recovery metadata saved"
    )


    del checkpoint

    gc.collect()
    torch.cuda.empty_cache()


# ============================================================
# TRAINING STATE
# ============================================================

running_loss = 0.0
running_count = 0

training_start = time.time()
last_log_time = training_start


# ============================================================
# TRAINING START
# ============================================================

print()
print("=" * 70)
print("STABILIZED TRAINING RESUMED")
print("=" * 70)

print(
    "Starting from step:",
    global_step
)

print(
    "Next optimizer step:",
    global_step + 1
)

print(
    "Remaining steps:",
    MAX_TRAINING_STEPS
    - global_step
)

print(
    "Learning rate:",
    RESUME_LEARNING_RATE
)

print(
    "Gradient clipping:",
    MAX_GRAD_NORM
)

print()
print(
    "The verified step-500 checkpoint remains untouched."
)

print(
    "Current recovery point:",
    RESUME_CHECKPOINT
)

print()


optimizer.zero_grad(
    set_to_none=True
)


# ============================================================
# TRAINING LOOP
# ============================================================

while global_step < MAX_TRAINING_STEPS:

    # --------------------------------------------------------
    # IMPORTANT:
    # Reset accumulation loss for THIS optimizer step.
    # --------------------------------------------------------

    accumulation_loss = 0.0


    # --------------------------------------------------------
    # GRADIENT ACCUMULATION
    # --------------------------------------------------------

    for accumulation_step in range(
        GRADIENT_ACCUMULATION_STEPS
    ):

        # ----------------------------------------------------
        # NEXT STREAM BATCH
        # ----------------------------------------------------

        try:

            batch = next(
                train_iter
            )

        except StopIteration:

            print(
                "Streaming dataset reached its end."
            )

            train_iter = iter(
                train_loader
            )

            batch = next(
                train_iter
            )


        # ----------------------------------------------------
        # MOVE TO GPU
        # ----------------------------------------------------

        input_ids = batch[
            "input_ids"
        ].to(
            DEVICE,
            dtype=torch.long
        )

        labels = batch[
            "labels"
        ].to(
            DEVICE,
            dtype=torch.long
        )

        attention_mask = batch[
            "attention_mask"
        ].to(
            DEVICE,
            dtype=torch.long
        )


        # ----------------------------------------------------
        # FORWARD
        # ----------------------------------------------------

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )


        loss = (
            outputs["loss"]
            if isinstance(
                outputs,
                dict
            )
            else outputs.loss
        )


        # ----------------------------------------------------
        # LOSS FINITE CHECK
        # ----------------------------------------------------

        if not torch.isfinite(
            loss
        ):

            optimizer.zero_grad(
                set_to_none=True
            )

            raise RuntimeError(
                "\n"
                "NON-FINITE LOSS DETECTED.\n"
                f"Optimizer step: "
                f"{global_step + 1}\n"
                f"Accumulation step: "
                f"{accumulation_step + 1}\n"
                f"Loss: "
                f"{loss.detach().item()}\n\n"
                "Training STOPPED BEFORE optimizer.step().\n"
                "No corrupted optimizer update was performed."
            )


        # ----------------------------------------------------
        # CORRECT LOSS ACCUMULATION
        # ----------------------------------------------------

        raw_loss = (
            loss.detach().item()
        )

        accumulation_loss += (
            raw_loss
        )


        # ----------------------------------------------------
        # BACKWARD
        # ----------------------------------------------------

        scaled_loss = (
            loss
            / GRADIENT_ACCUMULATION_STEPS
        )

        scaled_loss.backward()


        # ----------------------------------------------------
        # RELEASE FORWARD OBJECTS
        # ----------------------------------------------------

        del batch
        del input_ids
        del labels
        del attention_mask
        del outputs
        del loss
        del scaled_loss


    # ========================================================
    # CHECK GRADIENTS BEFORE UPDATE
    # ========================================================

    total_grad_sq = 0.0

    nonfinite_gradient = False
    nonfinite_parameter_name = None


    for name, parameter in (
        model.named_parameters()
    ):

        if parameter.grad is None:
            continue


        grad = parameter.grad


        if not torch.isfinite(
            grad
        ).all():

            nonfinite_gradient = True

            nonfinite_parameter_name = (
                name
            )

            break


        grad_norm = torch.norm(
            grad.detach()
        ).item()


        if not math.isfinite(
            grad_norm
        ):

            nonfinite_gradient = True

            nonfinite_parameter_name = (
                name
            )

            break


        total_grad_sq += (
            grad_norm
            * grad_norm
        )


    if nonfinite_gradient:

        optimizer.zero_grad(
            set_to_none=True
        )

        raise RuntimeError(
            "\n"
            "NON-FINITE GRADIENT DETECTED.\n"
            f"Optimizer step: "
            f"{global_step + 1}\n"
            f"Parameter: "
            f"{nonfinite_parameter_name}\n\n"
            "Training STOPPED BEFORE optimizer.step()."
        )


    total_grad_norm = math.sqrt(
        total_grad_sq
    )


    if not math.isfinite(
        total_grad_norm
    ):

        optimizer.zero_grad(
            set_to_none=True
        )

        raise RuntimeError(
            "Gradient norm became non-finite.\n"
            "Training STOPPED BEFORE optimizer.step()."
        )


    # ========================================================
    # GRADIENT CLIPPING
    # ========================================================

    clipped_grad_norm = (
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            MAX_GRAD_NORM
        )
    )


    clipped_grad_norm_value = float(
        clipped_grad_norm
    )


    if not math.isfinite(
        clipped_grad_norm_value
    ):

        optimizer.zero_grad(
            set_to_none=True
        )

        raise RuntimeError(
            "Gradient clipping returned "
            "a non-finite gradient norm.\n"
            "Training STOPPED BEFORE optimizer.step()."
        )


    # ========================================================
    # FINAL GRADIENT FINITE CHECK
    # ========================================================

    for name, parameter in (
        model.named_parameters()
    ):

        if parameter.grad is None:
            continue


        if not torch.isfinite(
            parameter.grad
        ).all():

            optimizer.zero_grad(
                set_to_none=True
            )

            raise RuntimeError(
                "Gradient became non-finite "
                "after clipping.\n"
                f"Parameter: {name}\n"
                "Training STOPPED BEFORE optimizer.step()."
            )


    # ========================================================
    # CURRENT PARAMETER FINITE CHECK
    # ========================================================

    for name, parameter in (
        model.named_parameters()
    ):

        if not torch.isfinite(
            parameter.detach()
        ).all():

            optimizer.zero_grad(
                set_to_none=True
            )

            raise RuntimeError(
                "Model parameter became non-finite "
                "before optimizer.step().\n"
                f"Parameter: {name}\n"
                "Training STOPPED."
            )


    # ========================================================
    # OPTIMIZER STEP
    # ========================================================

    optimizer.step()


    # --------------------------------------------------------
    # Immediately verify parameters after optimizer update.
    # --------------------------------------------------------

    nonfinite_parameter_after_update = None


    for name, parameter in (
        model.named_parameters()
    ):

        if not torch.isfinite(
            parameter.detach()
        ).all():

            nonfinite_parameter_after_update = (
                name
            )

            break


    if (
        nonfinite_parameter_after_update
        is not None
    ):

        optimizer.zero_grad(
            set_to_none=True
        )

        raise RuntimeError(
            "\n"
            "NON-FINITE MODEL PARAMETER "
            "AFTER OPTIMIZER UPDATE.\n"
            f"Parameter: "
            f"{nonfinite_parameter_after_update}\n"
            f"Optimizer step attempted: "
            f"{global_step + 1}\n\n"
            "The training process has STOPPED.\n"
            "The previous checkpoint remains the last "
            "known-good recovery point."
        )


    optimizer.zero_grad(
        set_to_none=True
    )


    optimizer_steps += 1
    global_step += 1


    # ========================================================
    # LOSS
    # ========================================================

    mean_loss = (
        accumulation_loss
        / GRADIENT_ACCUMULATION_STEPS
    )


    if not math.isfinite(
        mean_loss
    ):

        raise RuntimeError(
            "Mean accumulated loss became non-finite "
            "after optimizer step."
        )


    # ========================================================
    # RUNNING LOSS
    # ========================================================

    running_loss += (
        mean_loss
    )

    running_count += 1


    # ========================================================
    # LOGGING
    # ========================================================

    if global_step % LOG_EVERY == 0:

        now = time.time()

        elapsed = (
            now - training_start
        )

        interval = (
            now - last_log_time
        )


        avg_loss = (
            running_loss
            / running_count
        )


        steps_per_sec = (
            LOG_EVERY / interval
            if interval > 0
            else 0.0
        )


        allocated_gb = (
            torch.cuda.memory_allocated(
                DEVICE
            )
            / 1024**3
        )


        reserved_gb = (
            torch.cuda.memory_reserved(
                DEVICE
            )
            / 1024**3
        )


        print(
            f"step {global_step:06d} | "
            f"loss={mean_loss:.5f} | "
            f"avg={avg_loss:.5f} | "
            f"grad={total_grad_norm:.4f} | "
            f"clip={clipped_grad_norm_value:.4f} | "
            f"lr={optimizer.param_groups[0]['lr']:.2e} | "
            f"steps/s={steps_per_sec:.3f} | "
            f"GPU={allocated_gb:.2f}/"
            f"{reserved_gb:.2f} GB"
        )


        append_training_log(
            global_step,
            mean_loss,
            elapsed,
            total_grad_norm,
            clipped_grad_norm_value
        )


        running_loss = 0.0
        running_count = 0
        last_log_time = now


    # ========================================================
    # CHECKPOINT
    # ========================================================

    if (
        global_step % CHECKPOINT_EVERY
        == 0
    ):

        save_checkpoint(
            global_step
        )


# ============================================================
# FINAL CHECKPOINT
# ============================================================

print()
print("=" * 70)
print("TRAINING TARGET REACHED")
print("=" * 70)

print(
    "Final step:",
    global_step
)

print(
    "Optimizer steps:",
    optimizer_steps
)


# ------------------------------------------------------------
# Save final step checkpoint + rolling checkpoint
# ------------------------------------------------------------

save_checkpoint(
    global_step
)


# ============================================================
# DEDICATED FINAL CHECKPOINT
# ============================================================

final_checkpoint_path = os.path.join(
    CHECKPOINT_DIR,
    "checkpoint_final.pt"
)


if os.path.exists(
    final_checkpoint_path
):

    print(
        "Existing final checkpoint found; "
        "it will be replaced with the verified final "
        "rolling checkpoint."
    )


shutil.copy2(
    LATEST_CHECKPOINT,
    final_checkpoint_path
)


final_size = os.path.getsize(
    final_checkpoint_path
)


if final_size < 1_000_000_000:

    raise RuntimeError(
        "Final checkpoint is unexpectedly small."
    )


print(
    "✓ Final checkpoint saved:",
    final_checkpoint_path
)


# ============================================================
# FINAL RECOVERY STATE
# ============================================================

save_recovery_state()


# ============================================================
# FINAL REPORT
# ============================================================

training_time = (
    time.time()
    - training_start
)


print()
print("=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print(
    "Training steps:",
    global_step
)

print(
    "Optimizer steps:",
    optimizer_steps
)

print(
    "Training time:",
    f"{training_time / 60:.2f} minutes"
)

print()
print(
    "Final checkpoint:"
)

print(
    final_checkpoint_path
)

print()
print(
    "Rolling checkpoint:"
)

print(
    LATEST_CHECKPOINT
)

print()
print(
    "Recovery metadata:"
)

print(
    RECOVERY_STATE_PATH
)

print()
print(
    "Latest checkpoint SHA256:"
)

print(
    sha256_file(
        LATEST_CHECKPOINT
    )
)

print()
print(
    "Original verified step-500 checkpoint retained:"
)

print(
    VERIFIED_STEP_500
)

print()
print("=" * 70)
print(
    "✓ STABILIZED PHASE 4 TRAINING COMPLETE"
)
print("=" * 70)

In [ ]:
# ============================================================
# PHASE 4 — NUMERICALLY SAFE RESUME TRAINING
#
# BF16
# GRADIENT CHECKPOINTING
# NO KV CACHE
# GRADIENT CLIPPING
# FINITE-STATE CHECKS
# BAD-MICROBATCH ISOLATION
# CHECKPOINT-SAFE RESUME
#
# IMPORTANT:
# - Resume from the highest VALID checkpoint.
# - A non-finite microbatch is discarded.
# - Its gradients are NEVER used for optimizer.step().
# - The optimizer step counter advances ONLY after a
#   successful optimizer update.
# ============================================================

import os
import gc
import json
import time
import hashlib
import math
import shutil
import torch


# ============================================================
# CONFIGURATION
# ============================================================

DEVICE = torch.device("cuda:0")

CONTEXT_LENGTH = 4096
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8

MAX_TRAINING_STEPS = 1000

LOG_EVERY = 10
CHECKPOINT_EVERY = 50

RESUME_LEARNING_RATE = 1e-5
MAX_GRAD_NORM = 1.0

USE_BF16 = True
USE_GRADIENT_CHECKPOINTING = True
USE_CACHE = False

# Maximum number of bad microbatches we tolerate.
# They are skipped individually.
MAX_BAD_MICROBATCHES = 25


# ============================================================
# CHECKPOINT PATHS
# ============================================================

CHECKPOINT_DIR = "/kaggle/working/phase4_checkpoints"

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

VERIFIED_STEP_500 = os.path.join(
    CHECKPOINT_DIR,
    "checkpoint_step_000500.pt"
)

LATEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "checkpoint_latest.pt"
)

LATEST_CHECKPOINT_TMP = (
    LATEST_CHECKPOINT + ".tmp"
)

RECOVERY_STATE_PATH = os.path.join(
    CHECKPOINT_DIR,
    "training_recovery_state.json"
)

TRAINING_LOG_PATH = os.path.join(
    CHECKPOINT_DIR,
    "training_log.jsonl"
)

BAD_BATCH_LOG_PATH = os.path.join(
    CHECKPOINT_DIR,
    "bad_microbatches.jsonl"
)


# ============================================================
# ENVIRONMENT
# ============================================================

print("=" * 70)
print("PHASE 4 — NUMERICALLY SAFE RESUME TRAINING")
print("=" * 70)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Training NOT started."
    )

if model is None:
    raise RuntimeError(
        "Model is not defined."
    )

if optimizer is None:
    raise RuntimeError(
        "Optimizer is not defined."
    )

if train_loader is None:
    raise RuntimeError(
        "train_loader is not defined."
    )

print(
    "GPU:",
    torch.cuda.get_device_name(DEVICE)
)

print(
    "Total VRAM:",
    f"{torch.cuda.get_device_properties(DEVICE).total_memory / 1024**3:.3f} GB"
)


# ============================================================
# INITIAL CLEANUP
# ============================================================

optimizer.zero_grad(
    set_to_none=True
)

gc.collect()
torch.cuda.empty_cache()

print(
    "Initial allocated:",
    f"{torch.cuda.memory_allocated(DEVICE) / 1024**3:.3f} GB"
)

print(
    "Initial reserved:",
    f"{torch.cuda.memory_reserved(DEVICE) / 1024**3:.3f} GB"
)


# ============================================================
# BF16 VALIDATION
# ============================================================

if not USE_BF16:
    raise RuntimeError(
        "This Phase 4 replacement requires BF16."
    )

if not torch.cuda.is_bf16_supported():
    raise RuntimeError(
        "BF16 is not supported on this GPU."
    )

print("✓ BF16 supported")


# ============================================================
# MODEL MEMORY-SAFE CONFIGURATION
# ============================================================

print()
print("=" * 70)
print("CONFIGURING MODEL")
print("=" * 70)

if hasattr(model, "config"):
    model.config.use_cache = False

if USE_GRADIENT_CHECKPOINTING:

    if not hasattr(
        model,
        "gradient_checkpointing_enable"
    ):
        raise RuntimeError(
            "Model does not support gradient checkpointing."
        )

    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={
            "use_reentrant": False
        }
    )

    print(
        "✓ Gradient checkpointing enabled"
    )

print(
    "✓ use_cache=False"
)


# ============================================================
# DEVICE
# ============================================================

model = model.to(DEVICE)
model.train()


# ============================================================
# VERIFY MODEL DEVICE
# ============================================================

for name, parameter in model.named_parameters():

    if parameter.device != DEVICE:

        raise RuntimeError(
            f"Parameter on wrong device: "
            f"{name} -> {parameter.device}"
        )

print(
    "✓ Model parameters on cuda:0"
)


# ============================================================
# CHECKPOINT DISCOVERY
# ============================================================

print()
print("=" * 70)
print("CHECKPOINT DISCOVERY")
print("=" * 70)

candidates = []

if os.path.isfile(
    LATEST_CHECKPOINT
):
    candidates.append(
        LATEST_CHECKPOINT
    )

for filename in os.listdir(
    CHECKPOINT_DIR
):

    if (
        filename.startswith("checkpoint_step_")
        and filename.endswith(".pt")
    ):

        path = os.path.join(
            CHECKPOINT_DIR,
            filename
        )

        if os.path.isfile(path):
            candidates.append(path)

if os.path.isfile(
    VERIFIED_STEP_500
):
    candidates.append(
        VERIFIED_STEP_500
    )


if not candidates:

    raise FileNotFoundError(
        "No Phase 4 checkpoint found."
    )


# ============================================================
# VALIDATE CHECKPOINTS
# ============================================================

required_keys = {
    "phase",
    "step",
    "optimizer_steps",
    "model_state_dict",
    "optimizer_state_dict",
    "context_length",
    "batch_size",
    "gradient_accumulation_steps",
}

valid_candidates = []

for path in candidates:

    try:

        checkpoint = torch.load(
            path,
            map_location="cpu"
        )

        missing = (
            required_keys
            - set(checkpoint.keys())
        )

        if missing:

            print(
                "✗ Missing keys:",
                os.path.basename(path),
                sorted(missing)
            )

            del checkpoint
            continue


        step = int(
            checkpoint["step"]
        )

        optimizer_steps = int(
            checkpoint["optimizer_steps"]
        )


        if step != optimizer_steps:

            print(
                "✗ Step mismatch:",
                os.path.basename(path)
            )

            del checkpoint
            continue


        if int(
            checkpoint["context_length"]
        ) != CONTEXT_LENGTH:

            print(
                "✗ Context mismatch:",
                os.path.basename(path)
            )

            del checkpoint
            continue


        if int(
            checkpoint["batch_size"]
        ) != BATCH_SIZE:

            print(
                "✗ Batch-size mismatch:",
                os.path.basename(path)
            )

            del checkpoint
            continue


        if int(
            checkpoint[
                "gradient_accumulation_steps"
            ]
        ) != GRADIENT_ACCUMULATION_STEPS:

            print(
                "✗ Accumulation mismatch:",
                os.path.basename(path)
            )

            del checkpoint
            continue


        if checkpoint.get("phase") != 4:

            print(
                "✗ Wrong phase:",
                os.path.basename(path)
            )

            del checkpoint
            continue


        valid_candidates.append(
            (step, path)
        )

        print(
            f"✓ Valid: "
            f"{os.path.basename(path)} "
            f"(step {step})"
        )

        del checkpoint

    except Exception as exc:

        print(
            f"✗ Invalid: "
            f"{os.path.basename(path)} "
            f"{repr(exc)}"
        )


gc.collect()


if not valid_candidates:

    raise RuntimeError(
        "No valid Phase 4 checkpoint found."
    )


# ============================================================
# SELECT HIGHEST VALID CHECKPOINT
# ============================================================

valid_candidates.sort(
    key=lambda x: x[0],
    reverse=True
)

RESUME_STEP, RESUME_CHECKPOINT = (
    valid_candidates[0]
)


print()
print("=" * 70)
print("SELECTED CHECKPOINT")
print("=" * 70)

print(
    "Checkpoint:",
    RESUME_CHECKPOINT
)

print(
    "Resume step:",
    RESUME_STEP
)

print(
    "Remaining:",
    MAX_TRAINING_STEPS - RESUME_STEP
)


if RESUME_STEP >= MAX_TRAINING_STEPS:

    print(
        "Target already reached."
    )

    raise SystemExit


# ============================================================
# RESTORE CHECKPOINT
# ============================================================

print()
print("=" * 70)
print("RESTORING CHECKPOINT")
print("=" * 70)

checkpoint = torch.load(
    RESUME_CHECKPOINT,
    map_location="cpu"
)


model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

print(
    "✓ Model state restored"
)


optimizer.load_state_dict(
    checkpoint[
        "optimizer_state_dict"
    ]
)

print(
    "✓ Optimizer state restored"
)


global_step = int(
    checkpoint["step"]
)

optimizer_steps = int(
    checkpoint["optimizer_steps"]
)


del checkpoint

gc.collect()


# ============================================================
# MOVE OPTIMIZER STATE TO GPU
# ============================================================

for state in optimizer.state.values():

    for key, value in state.items():

        if torch.is_tensor(value):

            state[key] = value.to(
                DEVICE
            )


# ============================================================
# REAPPLY MODEL SETTINGS AFTER RESTORE
# ============================================================

model = model.to(DEVICE)
model.train()

if hasattr(model, "config"):
    model.config.use_cache = False

if USE_GRADIENT_CHECKPOINTING:

    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={
            "use_reentrant": False
        }
    )


# ============================================================
# RESTORE LEARNING RATE
# ============================================================

for group in optimizer.param_groups:
    group["lr"] = RESUME_LEARNING_RATE


# ============================================================
# RESTORED STATE FINITE CHECK
# ============================================================

print()
print("Checking restored model...")


for name, parameter in model.named_parameters():

    if not torch.isfinite(
        parameter.detach()
    ).all():

        raise RuntimeError(
            f"Non-finite restored parameter: "
            f"{name}"
        )


print(
    "✓ Model parameters finite"
)


print(
    "Checking optimizer state..."
)


for parameter, state in optimizer.state.items():

    for state_name, value in state.items():

        if torch.is_tensor(value):

            if not torch.isfinite(
                value
            ).all():

                raise RuntimeError(
                    f"Non-finite optimizer state: "
                    f"{state_name}"
                )


print(
    "✓ Optimizer state finite"
)


# ============================================================
# CLEAR GRADIENTS
# ============================================================

optimizer.zero_grad(
    set_to_none=True
)

gc.collect()
torch.cuda.empty_cache()


# ============================================================
# DATA STREAM RECOVERY
# ============================================================

print()
print("=" * 70)
print("RESTORING DATA STREAM")
print("=" * 70)

train_iter = iter(
    train_loader
)

microbatches_to_skip = (
    global_step
    * GRADIENT_ACCUMULATION_STEPS
)

print(
    "Completed optimizer steps:",
    global_step
)

print(
    "Microbatches to skip:",
    microbatches_to_skip
)


for idx in range(
    microbatches_to_skip
):

    try:

        batch = next(
            train_iter
        )

    except StopIteration:

        train_iter = iter(
            train_loader
        )

        batch = next(
            train_iter
        )

    del batch

    if (
        (idx + 1) % 500 == 0
        or
        (idx + 1) == microbatches_to_skip
    ):

        print(
            f"  {idx + 1}/"
            f"{microbatches_to_skip}"
        )


gc.collect()
torch.cuda.empty_cache()

print(
    "✓ Stream position restored"
)


# ============================================================
# HELPERS
# ============================================================

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def save_recovery_state():

    state = {

        "phase": 4,

        "training_step": int(
            global_step
        ),

        "optimizer_steps": int(
            optimizer_steps
        ),

        "context_length":
            CONTEXT_LENGTH,

        "batch_size":
            BATCH_SIZE,

        "gradient_accumulation_steps":
            GRADIENT_ACCUMULATION_STEPS,

        "stream_microbatches_consumed":
            int(
                global_step
                * GRADIENT_ACCUMULATION_STEPS
            ),

        "resume_source":
            RESUME_CHECKPOINT,

        "resume_learning_rate":
            RESUME_LEARNING_RATE,

        "max_grad_norm":
            MAX_GRAD_NORM,

        "mixed_precision":
            "bfloat16",

        "gradient_checkpointing":
            True,

        "use_cache":
            False,

        "bad_microbatches":
            int(
                bad_microbatch_count
            ),

        "latest_checkpoint":
            LATEST_CHECKPOINT,

        "saved_at_unix":
            time.time(),
    }


    tmp = (
        RECOVERY_STATE_PATH
        + ".tmp"
    )


    with open(
        tmp,
        "w"
    ) as f:

        json.dump(
            state,
            f,
            indent=2
        )


    os.replace(
        tmp,
        RECOVERY_STATE_PATH
    )


def append_training_log(
    step,
    loss,
    elapsed_seconds,
    grad_norm,
    clipped_grad_norm
):

    record = {

        "step":
            int(step),

        "loss":
            float(loss),

        "elapsed_seconds":
            float(elapsed_seconds),

        "timestamp":
            time.time(),

        "learning_rate":
            float(
                optimizer.param_groups[0]["lr"]
            ),

        "grad_norm":
            float(
                grad_norm
            ),

        "clipped_grad_norm":
            float(
                clipped_grad_norm
            ),

    }


    with open(
        TRAINING_LOG_PATH,
        "a"
    ) as f:

        f.write(
            json.dumps(record)
            + "\n"
        )


def append_bad_microbatch_log(
    optimizer_step,
    accumulation_step,
    batch_number,
    nonpad_tokens,
    input_min,
    input_max,
    label_min,
    label_max,
    reason
):

    record = {

        "optimizer_step":
            int(optimizer_step),

        "accumulation_step":
            int(accumulation_step),

        "batch_number":
            int(batch_number),

        "nonpad_tokens":
            int(nonpad_tokens),

        "input_min":
            int(input_min),

        "input_max":
            int(input_max),

        "label_min":
            int(label_min),

        "label_max":
            int(label_max),

        "reason":
            str(reason),

        "timestamp":
            time.time(),

    }


    with open(
        BAD_BATCH_LOG_PATH,
        "a"
    ) as f:

        f.write(
            json.dumps(record)
            + "\n"
        )


# ============================================================
# CHECKPOINT SAVE
# ============================================================

def save_checkpoint(
    step
):

    print()
    print("=" * 70)
    print(
        f"SAVING CHECKPOINT — STEP {step}"
    )
    print("=" * 70)


    step_path = os.path.join(
        CHECKPOINT_DIR,
        f"checkpoint_step_{step:06d}.pt"
    )

    step_tmp = (
        step_path + ".tmp"
    )


    checkpoint = {

        "phase": 4,

        "step":
            int(step),

        "optimizer_steps":
            int(optimizer_steps),

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "context_length":
            CONTEXT_LENGTH,

        "batch_size":
            BATCH_SIZE,

        "gradient_accumulation_steps":
            GRADIENT_ACCUMULATION_STEPS,

        "checkpoint_type":
            "production_training_numeric_safe",

        "learning_rate":
            RESUME_LEARNING_RATE,

        "max_grad_norm":
            MAX_GRAD_NORM,

        "mixed_precision":
            "bfloat16",

        "gradient_checkpointing":
            True,

        "use_cache":
            False,

        "bad_microbatches":
            int(
                bad_microbatch_count
            ),

        "created_at_unix":
            time.time(),
    }


    if os.path.exists(step_tmp):
        os.remove(step_tmp)


    torch.save(
        checkpoint,
        step_tmp
    )


    size = os.path.getsize(
        step_tmp
    )


    if size < 1_000_000_000:

        os.remove(step_tmp)

        raise RuntimeError(
            "Checkpoint unexpectedly small. "
            "Existing checkpoint preserved."
        )


    os.replace(
        step_tmp,
        step_path
    )


    # --------------------------------------------------------
    # Rolling checkpoint
    # --------------------------------------------------------

    if os.path.exists(
        LATEST_CHECKPOINT_TMP
    ):

        os.remove(
            LATEST_CHECKPOINT_TMP
        )


    torch.save(
        checkpoint,
        LATEST_CHECKPOINT_TMP
    )


    latest_size = os.path.getsize(
        LATEST_CHECKPOINT_TMP
    )


    if latest_size < 1_000_000_000:

        os.remove(
            LATEST_CHECKPOINT_TMP
        )

        raise RuntimeError(
            "Rolling checkpoint unexpectedly small. "
            "Existing rolling checkpoint preserved."
        )


    os.replace(
        LATEST_CHECKPOINT_TMP,
        LATEST_CHECKPOINT
    )


    save_recovery_state()


    print(
        "✓ Step checkpoint:",
        step_path
    )

    print(
        "✓ Rolling checkpoint updated"
    )

    print(
        "SHA256:",
        sha256_file(
            LATEST_CHECKPOINT
        )
    )


    del checkpoint

    gc.collect()
    torch.cuda.empty_cache()


# ============================================================
# TRAINING STATE
# ============================================================

running_loss = 0.0
running_count = 0

bad_microbatch_count = 0
successful_microbatch_count = 0

training_start = time.time()
last_log_time = training_start

batch_number = (
    global_step
    * GRADIENT_ACCUMULATION_STEPS
)


# ============================================================
# PRE-TRAINING MEMORY
# ============================================================

gc.collect()
torch.cuda.empty_cache()

print()
print("=" * 70)
print("FINAL PRE-TRAINING MEMORY CHECK")
print("=" * 70)

allocated = (
    torch.cuda.memory_allocated(DEVICE)
    / 1024**3
)

reserved = (
    torch.cuda.memory_reserved(DEVICE)
    / 1024**3
)

total = (
    torch.cuda.get_device_properties(
        DEVICE
    ).total_memory
    / 1024**3
)

print(
    f"Allocated : {allocated:.3f} GB"
)

print(
    f"Reserved  : {reserved:.3f} GB"
)

print(
    f"Free approx: {total - allocated:.3f} GB"
)

print(
    "BF16: ON"
)

print(
    "Gradient checkpointing: ON"
)

print(
    "use_cache: False"
)

print(
    "Gradient clipping:",
    MAX_GRAD_NORM
)

print(
    "Resume step:",
    global_step
)

print(
    "Target step:",
    MAX_TRAINING_STEPS
)


# ============================================================
# TRAINING LOOP
# ============================================================

print()
print("=" * 70)
print("NUMERICALLY SAFE TRAINING STARTED")
print("=" * 70)


optimizer.zero_grad(
    set_to_none=True
)


while global_step < MAX_TRAINING_STEPS:

    accumulation_loss = 0.0
    accumulation_valid = 0

    # --------------------------------------------------------
    # IMPORTANT:
    # Gradients belong to THIS optimizer step only.
    # --------------------------------------------------------

    optimizer.zero_grad(
        set_to_none=True
    )


    for accumulation_step in range(
        GRADIENT_ACCUMULATION_STEPS
    ):

        batch_number += 1


        # ====================================================
        # FETCH BATCH
        # ====================================================

        try:

            batch = next(
                train_iter
            )

        except StopIteration:

            train_iter = iter(
                train_loader
            )

            batch = next(
                train_iter
            )


        # ====================================================
        # MOVE TENSORS
        # ====================================================

        input_ids = batch[
            "input_ids"
        ].to(
            DEVICE,
            dtype=torch.long,
            non_blocking=True
        )

        labels = batch[
            "labels"
        ].to(
            DEVICE,
            dtype=torch.long,
            non_blocking=True
        )

        attention_mask = batch[
            "attention_mask"
        ].to(
            DEVICE,
            dtype=torch.long,
            non_blocking=True
        )


        # ====================================================
        # BASIC BATCH VALIDATION
        # ====================================================

        bad_reason = None


        if input_ids.ndim != 2:

            bad_reason = (
                f"input_ids ndim={input_ids.ndim}"
            )


        elif input_ids.shape[0] != BATCH_SIZE:

            bad_reason = (
                f"unexpected batch size="
                f"{input_ids.shape[0]}"
            )


        elif input_ids.shape[1] != CONTEXT_LENGTH:

            bad_reason = (
                f"unexpected sequence length="
                f"{input_ids.shape[1]}"
            )


        elif not torch.isfinite(
            input_ids.float()
        ).all():

            bad_reason = (
                "non-finite input_ids"
            )


        elif not torch.isfinite(
            labels.float()
        ).all():

            bad_reason = (
                "non-finite labels"
            )


        elif not torch.isfinite(
            attention_mask.float()
        ).all():

            bad_reason = (
                "non-finite attention_mask"
            )


        # ====================================================
        # TOKEN STATISTICS
        # ====================================================

        nonpad_tokens = int(
            attention_mask.sum().item()
        )

        input_min = int(
            input_ids.min().item()
        )

        input_max = int(
            input_ids.max().item()
        )

        label_min = int(
            labels.min().item()
        )

        label_max = int(
            labels.max().item()
        )


        # ====================================================
        # LABEL VALIDATION
        #
        # GPT-style labels normally contain either valid token
        # IDs or -100 for ignored positions.
        # ====================================================

        if bad_reason is None:

            vocab_size = int(
                model.config.vocab_size
            )

            invalid_label_mask = (
                (labels != -100)
                &
                (
                    (labels < 0)
                    |
                    (labels >= vocab_size)
                )
            )


            if invalid_label_mask.any():

                bad_reason = (
                    "labels contain IDs outside "
                    f"[-100, {vocab_size - 1}]"
                )


        # ====================================================
        # INVALID BATCH -> SKIP
        # ====================================================

        if bad_reason is not None:

            bad_microbatch_count += 1

            print()
            print(
                "!" * 70
            )
            print(
                "BAD MICRO-BATCH DETECTED"
            )
            print(
                "!" * 70
            )

            print(
                "Optimizer step:",
                global_step + 1
            )

            print(
                "Accumulation step:",
                accumulation_step + 1
            )

            print(
                "Batch number:",
                batch_number
            )

            print(
                "Reason:",
                bad_reason
            )

            print(
                "Non-pad tokens:",
                nonpad_tokens
            )

            print(
                "Input range:",
                input_min,
                "to",
                input_max
            )

            print(
                "Label range:",
                label_min,
                "to",
                label_max
            )

            print(
                "Action: SKIPPING BATCH"
            )


            append_bad_microbatch_log(
                global_step + 1,
                accumulation_step + 1,
                batch_number,
                nonpad_tokens,
                input_min,
                input_max,
                label_min,
                label_max,
                bad_reason
            )


            del batch
            del input_ids
            del labels
            del attention_mask

            gc.collect()
            torch.cuda.empty_cache()

            if (
                bad_microbatch_count
                > MAX_BAD_MICROBATCHES
            ):

                raise RuntimeError(
                    "Too many bad microbatches. "
                    "Training stopped for safety."
                )

            continue


        # ====================================================
        # FORWARD
        # ====================================================

        try:

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
                enabled=True
            ):

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                    use_cache=False,
                )


                loss = (
                    outputs["loss"]
                    if isinstance(
                        outputs,
                        dict
                    )
                    else outputs.loss
                )


        except Exception:

            del batch
            del input_ids
            del labels
            del attention_mask

            gc.collect()
            torch.cuda.empty_cache()

            raise


        # ====================================================
        # LOSS FINITE CHECK
        # ====================================================

        if not torch.isfinite(
            loss
        ).item():

            bad_microbatch_count += 1

            reason = (
                "NON-FINITE LOSS"
            )


            print()
            print(
                "!" * 70
            )
            print(
                "NON-FINITE LOSS DETECTED"
            )
            print(
                "!" * 70
            )

            print(
                "Optimizer step:",
                global_step + 1
            )

            print(
                "Accumulation step:",
                accumulation_step + 1
            )

            print(
                "Batch number:",
                batch_number
            )

            print(
                "Loss:",
                loss.detach().float().item()
            )

            print(
                "Non-pad tokens:",
                nonpad_tokens
            )

            print(
                "Input range:",
                input_min,
                "to",
                input_max
            )

            print(
                "Label range:",
                label_min,
                "to",
                label_max
            )

            print(
                "Action: DISCARDING THIS OPTIMIZER STEP"
            )


            append_bad_microbatch_log(
                global_step + 1,
                accumulation_step + 1,
                batch_number,
                nonpad_tokens,
                input_min,
                input_max,
                label_min,
                label_max,
                reason
            )


            # ------------------------------------------------
            # CRITICAL:
            # We CANNOT safely continue accumulation after a
            # non-finite loss because earlier microbatch
            # gradients belong to the same optimizer update.
            #
            # Therefore discard ALL accumulated gradients.
            # ------------------------------------------------

            optimizer.zero_grad(
                set_to_none=True
            )


            del outputs
            del loss
            del batch
            del input_ids
            del labels
            del attention_mask

            gc.collect()
            torch.cuda.empty_cache()


            if (
                bad_microbatch_count
                > MAX_BAD_MICROBATCHES
            ):

                raise RuntimeError(
                    "Too many non-finite microbatches. "
                    "Training stopped for safety."
                )


            # ------------------------------------------------
            # Restart the optimizer accumulation from scratch.
            # ------------------------------------------------

            accumulation_loss = 0.0
            accumulation_valid = 0

            break


        # ====================================================
        # VALID LOSS
        # ====================================================

        raw_loss = (
            loss.detach()
            .float()
            .item()
        )

        accumulation_loss += raw_loss
        accumulation_valid += 1


        # ====================================================
        # BACKWARD
        # ====================================================

        scaled_loss = (
            loss
            / GRADIENT_ACCUMULATION_STEPS
        )

        scaled_loss.backward()


        successful_microbatch_count += 1


        # ====================================================
        # RELEASE MICRO-BATCH OBJECTS
        # ====================================================

        del batch
        del input_ids
        del labels
        del attention_mask
        del outputs
        del loss
        del scaled_loss


        # ----------------------------------------------------
        # DO NOT empty_cache() here.
        # ----------------------------------------------------


    # ========================================================
    # IF ACCUMULATION WAS INTERRUPTED BY NON-FINITE LOSS
    # ========================================================

    if accumulation_valid != GRADIENT_ACCUMULATION_STEPS:

        optimizer.zero_grad(
            set_to_none=True
        )

        gc.collect()
        torch.cuda.empty_cache()

        print()
        print(
            "Accumulation discarded."
        )

        print(
            "Current successful optimizer step:",
            global_step
        )

        print(
            "Retrying next optimizer accumulation..."
        )

        continue


    # ========================================================
    # GRADIENT FINITE CHECK
    # ========================================================

    total_grad_sq = 0.0
    bad_gradient = None


    for name, parameter in model.named_parameters():

        if parameter.grad is None:
            continue


        grad = parameter.grad


        if not torch.isfinite(
            grad
        ).all():

            bad_gradient = name
            break


        norm = torch.norm(
            grad.detach()
        ).item()


        if not math.isfinite(norm):

            bad_gradient = name
            break


        total_grad_sq += (
            norm * norm
        )


    if bad_gradient is not None:

        optimizer.zero_grad(
            set_to_none=True
        )

        gc.collect()
        torch.cuda.empty_cache()

        raise RuntimeError(
            f"NON-FINITE GRADIENT: "
            f"{bad_gradient}\n"
            "Optimizer step NOT executed.\n"
            "Last checkpoint remains intact."
        )


    total_grad_norm = math.sqrt(
        total_grad_sq
    )


    if not math.isfinite(
        total_grad_norm
    ):

        optimizer.zero_grad(
            set_to_none=True
        )

        raise RuntimeError(
            "Total gradient norm is non-finite."
        )


    # ========================================================
    # GRADIENT CLIPPING
    # ========================================================

    clipped_grad_norm = (
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            MAX_GRAD_NORM
        )
    )


    clipped_grad_norm_value = float(
        clipped_grad_norm
    )


    if not math.isfinite(
        clipped_grad_norm_value
    ):

        optimizer.zero_grad(
            set_to_none=True
        )

        raise RuntimeError(
            "Non-finite clipped gradient norm.\n"
            "Optimizer step NOT executed."
        )


    # ========================================================
    # OPTIMIZER STEP
    # ========================================================

    optimizer.step()

    optimizer.zero_grad(
        set_to_none=True
    )


    # ========================================================
    # VERIFY UPDATED PARAMETERS
    # ========================================================

    bad_parameter = None


    for name, parameter in model.named_parameters():

        if not torch.isfinite(
            parameter.detach()
        ).all():

            bad_parameter = name
            break


    if bad_parameter is not None:

        raise RuntimeError(
            f"NON-FINITE PARAMETER AFTER UPDATE: "
            f"{bad_parameter}\n"
            "Training stopped."
        )


    # ========================================================
    # SUCCESSFUL OPTIMIZER STEP
    # ========================================================

    global_step += 1
    optimizer_steps += 1


    mean_loss = (
        accumulation_loss
        / GRADIENT_ACCUMULATION_STEPS
    )


    if not math.isfinite(
        mean_loss
    ):

        raise RuntimeError(
            "Mean loss became non-finite."
        )


    running_loss += mean_loss
    running_count += 1


    # ========================================================
    # LOGGING
    # ========================================================

    if global_step % LOG_EVERY == 0:

        now = time.time()

        interval = (
            now - last_log_time
        )

        avg_loss = (
            running_loss
            / running_count
        )

        steps_per_sec = (
            LOG_EVERY / interval
            if interval > 0
            else 0.0
        )


        allocated = (
            torch.cuda.memory_allocated(
                DEVICE
            )
            / 1024**3
        )

        reserved = (
            torch.cuda.memory_reserved(
                DEVICE
            )
            / 1024**3
        )


        print(
            f"step {global_step:06d} | "
            f"loss={mean_loss:.5f} | "
            f"avg={avg_loss:.5f} | "
            f"grad={total_grad_norm:.4f} | "
            f"clip={clipped_grad_norm_value:.4f} | "
            f"lr={optimizer.param_groups[0]['lr']:.2e} | "
            f"steps/s={steps_per_sec:.3f} | "
            f"bad={bad_microbatch_count} | "
            f"GPU={allocated:.2f}/"
            f"{reserved:.2f} GB"
        )


        append_training_log(
            global_step,
            mean_loss,
            now - training_start,
            total_grad_norm,
            clipped_grad_norm_value
        )


        running_loss = 0.0
        running_count = 0
        last_log_time = now


    # ========================================================
    # CHECKPOINT
    # ========================================================

    if (
        global_step % CHECKPOINT_EVERY
        == 0
    ):

        save_checkpoint(
            global_step
        )

        gc.collect()
        torch.cuda.empty_cache()


# ============================================================
# FINAL CHECKPOINT
# ============================================================

print()
print("=" * 70)
print("TRAINING TARGET REACHED")
print("=" * 70)

print(
    "Final step:",
    global_step
)

print(
    "Optimizer steps:",
    optimizer_steps
)

print(
    "Bad microbatches:",
    bad_microbatch_count
)

print(
    "Successful microbatches:",
    successful_microbatch_count
)


save_checkpoint(
    global_step
)


# ============================================================
# DEDICATED FINAL CHECKPOINT
# ============================================================

final_checkpoint_path = os.path.join(
    CHECKPOINT_DIR,
    "checkpoint_final.pt"
)


shutil.copy2(
    LATEST_CHECKPOINT,
    final_checkpoint_path
)


if (
    os.path.getsize(
        final_checkpoint_path
    )
    < 1_000_000_000
):

    raise RuntimeError(
        "Final checkpoint unexpectedly small."
    )


save_recovery_state()


# ============================================================
# FINAL CLEANUP
# ============================================================

optimizer.zero_grad(
    set_to_none=True
)

gc.collect()
torch.cuda.empty_cache()


# ============================================================
# FINAL REPORT
# ============================================================

print()
print("=" * 70)
print("PHASE 4 COMPLETE")
print("=" * 70)

print(
    "Final checkpoint:",
    final_checkpoint_path
)

print(
    "Rolling checkpoint:",
    LATEST_CHECKPOINT
)

print(
    "Final SHA256:",
    sha256_file(
        LATEST_CHECKPOINT
    )
)

print(
    "Bad microbatches:",
    bad_microbatch_count
)

print(
    "✓ BF16 training"
)

print(
    "✓ Gradient checkpointing"
)

print(
    "✓ use_cache=False"
)

print(
    "✓ Gradient clipping"
)

print(
    "✓ Finite-state checks"
)

print(
    "✓ Non-finite microbatch isolation"
)

print(
    "✓ Checkpoint-safe resume"
)

print(
    "✓ Phase 4 training complete"
)

print("=" * 70)

In [ ]:
# ============================================================
# PHASE 4 POST-COMPLETION — FINAL CHECKPOINT INTEGRITY CHECK
# NO TRAINING
# NO OPTIMIZER STEP
# NO CHECKPOINT MODIFICATION
# ============================================================

import os
import gc
import torch

CHECKPOINT_DIR = "/kaggle/working/phase4_checkpoints"

FINAL_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "checkpoint_final.pt"
)

LATEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "checkpoint_latest.pt"
)

STEP_1000_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "checkpoint_step_001000.pt"
)

print("=" * 70)
print("PHASE 4 — FINAL CHECKPOINT INTEGRITY")
print("=" * 70)

for path in [
    FINAL_CHECKPOINT,
    LATEST_CHECKPOINT,
    STEP_1000_CHECKPOINT,
]:

    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"Missing checkpoint:\n{path}"
        )

    size_gb = (
        os.path.getsize(path)
        / 1024**3
    )

    print(
        f"{os.path.basename(path):35s}"
        f"{size_gb:.3f} GB"
    )


print()
print("Loading final checkpoint on CPU...")

checkpoint = torch.load(
    FINAL_CHECKPOINT,
    map_location="cpu"
)

required_keys = {
    "phase",
    "step",
    "optimizer_steps",
    "model_state_dict",
    "optimizer_state_dict",
    "context_length",
    "batch_size",
    "gradient_accumulation_steps",
}

missing = (
    required_keys
    - set(checkpoint.keys())
)

if missing:
    raise RuntimeError(
        f"Missing checkpoint keys: {sorted(missing)}"
    )

print("✓ Required checkpoint fields present")


print()
print("Checkpoint metadata:")
print("  phase:", checkpoint["phase"])
print("  step:", checkpoint["step"])
print("  optimizer_steps:", checkpoint["optimizer_steps"])
print("  context_length:", checkpoint["context_length"])
print("  batch_size:", checkpoint["batch_size"])
print(
    "  gradient_accumulation_steps:",
    checkpoint["gradient_accumulation_steps"]
)


if int(checkpoint["phase"]) != 4:
    raise RuntimeError("Wrong phase.")

if int(checkpoint["step"]) != 1000:
    raise RuntimeError(
        f"Expected step 1000, got {checkpoint['step']}"
    )

if int(checkpoint["optimizer_steps"]) != 1000:
    raise RuntimeError(
        f"Expected optimizer_steps 1000, "
        f"got {checkpoint['optimizer_steps']}"
    )

if int(checkpoint["context_length"]) != 4096:
    raise RuntimeError("Wrong context length.")

if int(checkpoint["batch_size"]) != 1:
    raise RuntimeError("Wrong batch size.")

if int(checkpoint["gradient_accumulation_steps"]) != 8:
    raise RuntimeError(
        "Wrong gradient accumulation setting."
    )


print()
print("Checking model tensors...")

bad_model = []

for name, tensor in checkpoint[
    "model_state_dict"
].items():

    if torch.is_tensor(tensor):

        if not torch.isfinite(tensor).all():

            bad_model.append(name)

if bad_model:

    raise RuntimeError(
        "Non-finite model tensors:\n"
        + "\n".join(bad_model[:20])
    )

print("✓ All model tensors finite")


print()
print("Checking optimizer tensors...")

bad_optimizer = []

for parameter_id, state in checkpoint[
    "optimizer_state_dict"
]["state"].items():

    for state_name, value in state.items():

        if torch.is_tensor(value):

            if not torch.isfinite(value).all():

                bad_optimizer.append(
                    f"{parameter_id}:{state_name}"
                )

if bad_optimizer:

    raise RuntimeError(
        "Non-finite optimizer tensors:\n"
        + "\n".join(bad_optimizer[:20])
    )

print("✓ All optimizer tensors finite")


del checkpoint

gc.collect()

print()
print("=" * 70)
print("✓ FINAL CHECKPOINT IS NUMERICALLY INTACT")
print("=" * 70)

In [ ]:
# ============================================================
# PHASE 4 POST-COMPLETION — FINAL MODEL FORWARD VERIFICATION
# NO TRAINING
# NO BACKWARD
# NO OPTIMIZER STEP
# NO CHECKPOINT MODIFICATION
# ============================================================

import gc
import torch

DEVICE = torch.device("cuda:0")

print("=" * 70)
print("PHASE 4 — FINAL MODEL FORWARD VERIFICATION")
print("=" * 70)


# ============================================================
# LOAD FINAL CHECKPOINT
# ============================================================

checkpoint = torch.load(
    FINAL_CHECKPOINT,
    map_location="cpu"
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

del checkpoint

gc.collect()
torch.cuda.empty_cache()


# ============================================================
# MODEL CONFIGURATION
# ============================================================

model = model.to(DEVICE)

model.eval()

if hasattr(model, "config"):
    model.config.use_cache = False

print("✓ Step-1000 model loaded")
print("✓ Evaluation mode")
print("✓ use_cache=False")


# ============================================================
# INITIALIZE DATA ITERATOR
# ============================================================

test_iter = iter(train_loader)

try:

    batch = next(test_iter)

except StopIteration:

    test_iter = iter(train_loader)

    batch = next(test_iter)


# ============================================================
# FORWARD TEST
# ============================================================

NUM_TEST_BATCHES = 16

finite_count = 0
nonfinite_count = 0

print()
print("=" * 70)
print("FORWARD TEST")
print("=" * 70)


with torch.no_grad():

    for batch_number in range(
        1,
        NUM_TEST_BATCHES + 1
    ):

        # ----------------------------------------------------
        # Move batch to GPU
        # ----------------------------------------------------

        input_ids = batch[
            "input_ids"
        ].to(
            DEVICE,
            dtype=torch.long
        )

        labels = batch[
            "labels"
        ].to(
            DEVICE,
            dtype=torch.long
        )

        attention_mask = batch[
            "attention_mask"
        ].to(
            DEVICE,
            dtype=torch.long
        )


        # ----------------------------------------------------
        # Batch statistics
        # ----------------------------------------------------

        nonpad_tokens = int(
            attention_mask.sum().item()
        )

        supervised_tokens = int(
            (labels != -100).sum().item()
        )


        # ----------------------------------------------------
        # Forward pass
        # ----------------------------------------------------

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=True
        ):

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
                use_cache=False
            )

            loss = outputs.loss


        # ----------------------------------------------------
        # Numerical check
        # ----------------------------------------------------

        loss_value = float(
            loss.detach()
            .float()
            .cpu()
            .item()
        )

        finite = bool(
            torch.isfinite(loss).item()
        )


        if finite:

            finite_count += 1

        else:

            nonfinite_count += 1


        print(
            f"Batch {batch_number:02d} | "
            f"nonpad={nonpad_tokens:4d} | "
            f"supervised={supervised_tokens:4d} | "
            f"loss={loss_value:.6f} | "
            f"finite={finite}"
        )


        # ----------------------------------------------------
        # Release GPU objects
        # ----------------------------------------------------

        del input_ids
        del labels
        del attention_mask
        del outputs
        del loss
        del batch

        gc.collect()
        torch.cuda.empty_cache()


        # ----------------------------------------------------
        # Get next batch
        # ----------------------------------------------------

        if batch_number < NUM_TEST_BATCHES:

            try:

                batch = next(
                    test_iter
                )

            except StopIteration:

                test_iter = iter(
                    train_loader
                )

                batch = next(
                    test_iter
                )


# ============================================================
# RESULT
# ============================================================

print()
print("=" * 70)
print("FORWARD VERIFICATION RESULT")
print("=" * 70)

print(
    "Finite batches:",
    finite_count
)

print(
    "Non-finite batches:",
    nonfinite_count
)


if nonfinite_count != 0:

    raise RuntimeError(
        "At least one tested batch produced "
        "a non-finite loss."
    )


print()
print("✓ All 16 tested batches produced finite loss")
print("✓ Step-1000 model passes forward verification")


# ============================================================
# RESTORE TRAINING MODE
# ============================================================

model.train()

print("✓ Model returned to training mode")
print("=" * 70)

In [ ]:
# ============================================================
# KAGGLE DATASET CREATE — CLI 2.0.2 COMPATIBLE
# ============================================================

import os
import json
import subprocess

STAGING_DIR = (
    "/kaggle/working/"
    "phase4_step1000_permanent_dataset"
)

USERNAME = "oipranesh"
DATASET_SLUG = "phase4-step1000-verified-model"
DATASET_ID = f"{USERNAME}/{DATASET_SLUG}"

METADATA_PATH = os.path.join(
    STAGING_DIR,
    "dataset-metadata.json"
)

print("=" * 70)
print("KAGGLE DATASET CREATE — CLI 2.0.2 COMPATIBLE")
print("=" * 70)

# ============================================================
# VERIFY STAGING
# ============================================================

if not os.path.isdir(STAGING_DIR):

    raise FileNotFoundError(
        f"Staging directory does not exist:\n"
        f"{STAGING_DIR}"
    )

if not os.path.isfile(METADATA_PATH):

    raise FileNotFoundError(
        f"dataset-metadata.json not found:\n"
        f"{METADATA_PATH}"
    )


# ============================================================
# FIX METADATA ID
# ============================================================

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as f:

    metadata = json.load(f)


# IMPORTANT:
# Kaggle CLI 2.0.2 expects username/slug here.

metadata["id"] = DATASET_ID


with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False
    )


print()
print("METADATA")
print("-" * 70)

print(
    "Dataset ID:",
    metadata["id"]
)

print(
    "Title:",
    metadata.get("title")
)


# ============================================================
# VERIFY CHECKPOINTS BEFORE UPLOAD
# ============================================================

print()
print("CHECKPOINT VERIFICATION")
print("-" * 70)

checkpoint_files = [
    "checkpoint_step_001000.pt",
    "checkpoint_final.pt",
    "checkpoint_latest.pt",
]

for filename in checkpoint_files:

    path = os.path.join(
        STAGING_DIR,
        filename
    )

    if not os.path.isfile(path):

        raise FileNotFoundError(
            f"Missing checkpoint:\n{path}"
        )

    size_gb = (
        os.path.getsize(path)
        / 1024**3
    )

    print(
        f"✓ {filename:<35} "
        f"{size_gb:.3f} GB"
    )


# ============================================================
# SHOW FINAL METADATA
# ============================================================

print()
print("FINAL METADATA")
print("-" * 70)

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as f:

    print(
        f.read()
    )


# ============================================================
# CREATE DATASET
# ============================================================

print()
print("=" * 70)
print("CREATING DATASET")
print("=" * 70)

command = [
    "kaggle",
    "datasets",
    "create",
    "-p",
    STAGING_DIR,
    "--dir-mode",
    "zip",
]

print()
print(
    "Command:",
    " ".join(command)
)

print()
print(
    "IMPORTANT: "
    "The dataset ID is being supplied through "
    "dataset-metadata.json."
)

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)


# ============================================================
# OUTPUT
# ============================================================

print()
print("RETURN CODE:")
print(
    result.returncode
)

print()
print("STDOUT:")
print(
    result.stdout
)

print()
print("STDERR:")
print(
    result.stderr
)


# ============================================================
# FAILURE
# ============================================================

if result.returncode != 0:

    print()
    print("=" * 70)
    print("DATASET CREATION FAILED")
    print("=" * 70)

    print()
    print(
        "STAGING DIRECTORY WAS PRESERVED:"
    )

    print(
        STAGING_DIR
    )

    raise RuntimeError(
        "Kaggle dataset creation failed. "
        "The complete Kaggle error is shown above."
    )


# ============================================================
# SUCCESS
# ============================================================

print()
print("=" * 70)
print("KAGGLE DATASET CREATION SUCCEEDED")
print("=" * 70)

print()
print(
    "Dataset:",
    DATASET_ID
)

print()
print(
    "Permanent backup contains:"
)

for filename in checkpoint_files:

    print(
        "✓",
        filename
    )

print()
print(
    "✓ dataset-metadata.json"
)

print(
    "✓ manifest.json"
)

print(
    "✓ training_recovery_state.json"
)

print(
    "✓ training_log.jsonl"
)

print()
print(
    "LOCAL STAGING PRESERVED:"
)

print(
    STAGING_DIR
)

print()
print("=" * 70)

In [ ]:
# ============================================================
# PHASE 4 — LOW-SUPERVISION / SINGLE-TOKEN STRESS TEST
# NO TRAINING
# NO BACKWARD
# NO OPTIMIZER STEP
# NO CHECKPOINT MODIFICATION
# ============================================================

import gc
import torch

DEVICE = torch.device("cuda:0")

print("=" * 70)
print("PHASE 4 — LOW-SUPERVISION BATCH STRESS TEST")
print("=" * 70)

model.eval()

if hasattr(model, "config"):
    model.config.use_cache = False


TARGET_LOW_TOKEN_BATCHES = 10
MAX_SCAN_BATCHES = 2000

found = 0
scanned = 0

iterator = iter(train_loader)

with torch.no_grad():

    while (
        scanned < MAX_SCAN_BATCHES
        and found < TARGET_LOW_TOKEN_BATCHES
    ):

        try:
            batch = next(iterator)

        except StopIteration:

            iterator = iter(train_loader)
            batch = next(iterator)

        scanned += 1

        input_ids = batch[
            "input_ids"
        ].to(
            DEVICE,
            dtype=torch.long
        )

        labels = batch[
            "labels"
        ].to(
            DEVICE,
            dtype=torch.long
        )

        attention_mask = batch[
            "attention_mask"
        ].to(
            DEVICE,
            dtype=torch.long
        )

        nonpad_tokens = int(
            attention_mask.sum().item()
        )

        supervised_tokens = int(
            (labels != -100).sum().item()
        )


        # ----------------------------------------------------
        # Look specifically for extremely sparse batches.
        # ----------------------------------------------------

        if supervised_tokens <= 4:

            found += 1

            input_min = int(
                input_ids.min().item()
            )

            input_max = int(
                input_ids.max().item()
            )

            valid_labels = labels[
                labels != -100
            ]

            if valid_labels.numel() > 0:

                label_min = int(
                    valid_labels.min().item()
                )

                label_max = int(
                    valid_labels.max().item()
                )

            else:

                label_min = None
                label_max = None


            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
                enabled=True
            ):

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                    use_cache=False
                )

                loss = outputs.loss


            finite = bool(
                torch.isfinite(loss).item()
            )

            loss_value = float(
                loss.detach()
                .float()
                .cpu()
                .item()
            )


            print(
                f"LOW-TOKEN #{found:02d} | "
                f"stream_batch={scanned} | "
                f"nonpad={nonpad_tokens} | "
                f"supervised={supervised_tokens} | "
                f"input={input_min}:{input_max} | "
                f"label={label_min}:{label_max} | "
                f"loss={loss_value} | "
                f"finite={finite}"
            )


            del outputs
            del loss


        del input_ids
        del labels
        del attention_mask
        del batch

        gc.collect()
        torch.cuda.empty_cache()


print()
print("=" * 70)
print("LOW-TOKEN STRESS TEST COMPLETE")
print("=" * 70)

print(
    "Batches scanned:",
    scanned
)

print(
    "Low-token batches tested:",
    found
)

if found == 0:

    print(
        "No <=4-supervised-token batches were encountered."
    )

else:

    print(
        "✓ Low-token batches were tested without training."
    )

model.train()

print(
    "✓ Model returned to training mode"
)

print("=" * 70)

In [ ]:
# ============================================================
# CS781 — DIAGNOSTIC CELL 1 (REPLACEMENT)
# Robust Kaggle dataset discovery
# ============================================================

import os
from collections import Counter

INPUT_ROOT = "/kaggle/input"

print("=" * 100)
print("CS781 — ROBUST DATASET DISCOVERY")
print("=" * 100)

# ------------------------------------------------------------
# Show actual Kaggle-mounted dataset roots
# ------------------------------------------------------------

dataset_roots = []

for name in sorted(os.listdir(INPUT_ROOT)):

    path = os.path.join(INPUT_ROOT, name)

    if os.path.isdir(path):
        dataset_roots.append(path)

print("\nACTUAL MOUNTED DATASET DIRECTORIES")
print("-" * 100)

for i, root in enumerate(dataset_roots):

    print(f"\n[{i}] {root}")

    try:
        children = os.listdir(root)

        print(
            "    Top-level entries:",
            len(children)
        )

        for x in sorted(children)[:15]:
            print("      ", x)

        if len(children) > 15:
            print("       ...")

    except Exception as e:
        print("    Could not inspect:", repr(e))


# ------------------------------------------------------------
# Inventory every mounted dataset
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("DATASET CONTENT SIGNATURES")
print("=" * 100)

dataset_info = {}

for root in dataset_roots:

    files = []

    for dirpath, _, filenames in os.walk(root):

        for fn in filenames:
            files.append(
                os.path.join(dirpath, fn)
            )

    rel_files = [
        os.path.relpath(f, root)
        for f in files
    ]

    dataset_info[root] = rel_files

    print("\n" + "-" * 100)
    print("ROOT:", root)
    print("FILES:", len(files))

    # Important signature files
    signatures = [
        f for f in rel_files
        if any(
            keyword in os.path.basename(f).lower()
            for keyword in [
                "checkpoint",
                "token",
                "vocab",
                "merge",
                "train",
                "test",
                "recovery",
                "log",
                "history",
                "config",
            ]
        )
    ]

    for f in sorted(signatures)[:40]:
        print(" ", f)

    if len(signatures) > 40:
        print("  ...")


# ------------------------------------------------------------
# Automatically classify based on CONTENT
# ------------------------------------------------------------

def score_dataset(files, keywords):

    score = 0

    for f in files:

        low = f.lower()

        for keyword, weight in keywords:
            if keyword in low:
                score += weight

    return score


scores = {}

for root, files in dataset_info.items():

    scores[root] = {

        "recovery": score_dataset(
            files,
            [
                ("recovery", 10),
                ("checkpoint", 5),
                ("history", 4),
                ("log", 2),
                ("state", 2),
            ]
        ),

        "step1000": score_dataset(
            files,
            [
                ("step_001000", 20),
                ("step1000", 20),
                ("checkpoint_step", 15),
                ("checkpoint", 4),
            ]
        ),

        "training": score_dataset(
            files,
            [
                ("training", 10),
                ("train", 8),
                ("tokenizer", 8),
                ("vocab", 5),
                ("merges", 5),
                ("preparation", 10),
                ("phase_1", 5),
                ("phase1", 5),
            ]
        ),
    }


print("\n" + "=" * 100)
print("CLASSIFICATION SCORES")
print("=" * 100)

for root, s in scores.items():

    print("\n", root)

    for category, value in s.items():
        print(
            f"  {category:12}: {value}"
        )


# ------------------------------------------------------------
# Select roots
# ------------------------------------------------------------

def best_root(category):

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1][category],
        reverse=True
    )

    if not ranked or ranked[0][1][category] <= 0:
        return None

    return ranked[0][0]


RECOVERY_ROOT = best_root("recovery")
STEP1000_ROOT = best_root("step1000")
TRAINING_ROOT = best_root("training")


print("\n" + "=" * 100)
print("SELECTED DATASET ROOTS")
print("=" * 100)

print("RECOVERY_ROOT :")
print(RECOVERY_ROOT)

print("\nSTEP1000_ROOT :")
print(STEP1000_ROOT)

print("\nTRAINING_ROOT :")
print(TRAINING_ROOT)


# ------------------------------------------------------------
# Do not silently continue if classification is ambiguous.
# Print candidates so we can identify the actual root.
# ------------------------------------------------------------

if RECOVERY_ROOT is None:
    print(
        "\nWARNING: Recovery dataset could not be identified automatically."
    )

if STEP1000_ROOT is None:
    print(
        "\nWARNING: Step-1000 dataset could not be identified automatically."
    )

if TRAINING_ROOT is None:
    print(
        "\nWARNING: Training-preparation dataset could not be identified automatically."
    )


print("\n" + "=" * 100)
print("✓ ROBUST DATASET DISCOVERY COMPLETE")
print("=" * 100)

In [ ]:
# ============================================================
# CS781 — FIX DATASET ROOTS
# Based on the actual Kaggle-mounted structure
# ============================================================

import os

BASE = "/kaggle/input/datasets/oipranesh"

RECOVERY_ROOT = os.path.join(
    BASE,
    "cs781-phase4-final-recovery"
)

TRAINING_ROOT = os.path.join(
    BASE,
    "cs781-phase4-training-prep"
)

STEP1000_ROOT = os.path.join(
    BASE,
    "phase4-step1000-verified-model"
)

roots = {
    "RECOVERY_ROOT": RECOVERY_ROOT,
    "TRAINING_ROOT": TRAINING_ROOT,
    "STEP1000_ROOT": STEP1000_ROOT,
}

print("=" * 100)
print("CS781 — VERIFIED DATASET ROOTS")
print("=" * 100)

for name, root in roots.items():

    print(f"\n{name}")
    print("-" * 100)
    print(root)

    if not os.path.isdir(root):
        raise FileNotFoundError(
            f"Dataset directory does not exist:\n{root}"
        )

    files = []

    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            files.append(
                os.path.join(dirpath, fn)
            )

    print(
        f"✓ Exists | {len(files)} files"
    )

print("\n" + "=" * 100)
print("RECOVERY DATASET")
print("=" * 100)

for dirpath, _, filenames in os.walk(RECOVERY_ROOT):

    for fn in sorted(filenames):

        rel = os.path.relpath(
            os.path.join(dirpath, fn),
            RECOVERY_ROOT
        )

        print(rel)


print("\n" + "=" * 100)
print("TRAINING PREPARATION DATASET")
print("=" * 100)

for dirpath, _, filenames in os.walk(TRAINING_ROOT):

    for fn in sorted(filenames):

        rel = os.path.relpath(
            os.path.join(dirpath, fn),
            TRAINING_ROOT
        )

        print(rel)


print("\n" + "=" * 100)
print("STEP-1000 VERIFIED MODEL DATASET")
print("=" * 100)

for dirpath, _, filenames in os.walk(STEP1000_ROOT):

    for fn in sorted(filenames):

        rel = os.path.relpath(
            os.path.join(dirpath, fn),
            STEP1000_ROOT
        )

        print(rel)


print("\n" + "=" * 100)
print("✓ ALL THREE DATASET ROOTS VERIFIED")
print("=" * 100)

In [ ]:
# ============================================================
# CS781 — MASTER TRAINING RUN DIAGNOSIS
#
# Uses ONLY the three verified Kaggle datasets:
#
#   1. cs781-phase4-final-recovery
#   2. cs781-phase4-training-prep
#   3. phase4-step1000-verified-model
#
# PURPOSE:
#   Diagnose WHY the Step-1000 model produces repetitive text.
#
# NO:
#   training
#   backward()
#   optimizer.step()
#   model modification
#   generation
#
# ============================================================

import os
import json
import math
import statistics
from collections import Counter, defaultdict

print("=" * 110)
print("CS781 — MASTER TRAINING RUN DIAGNOSIS")
print("=" * 110)

# ============================================================
# ROOTS — already verified
# ============================================================

RECOVERY_ROOT = "/kaggle/input/datasets/oipranesh/cs781-phase4-final-recovery"
TRAINING_ROOT = "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
STEP1000_ROOT = "/kaggle/input/datasets/oipranesh/phase4-step1000-verified-model"


# ============================================================
# HELPERS
# ============================================================

def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def show_json(path, title=None):

    if not os.path.exists(path):
        print(f"NOT FOUND: {path}")
        return None

    obj = read_json(path)

    if title:
        print("\n" + "=" * 110)
        print(title)
        print("=" * 110)

    print(json.dumps(obj, indent=2, ensure_ascii=False)[:30000])

    return obj


def count_jsonl(path):

    count = 0

    if not os.path.exists(path):
        return 0

    with open(path, "r", encoding="utf-8") as f:

        for line in f:
            if line.strip():
                count += 1

    return count


# ============================================================
# 1. MODEL ARCHITECTURE
# ============================================================

architecture_path = os.path.join(
    RECOVERY_ROOT,
    "model_architecture.json"
)

architecture = show_json(
    architecture_path,
    "1. MODEL ARCHITECTURE"
)


# ============================================================
# 2. INITIALIZATION
# ============================================================

initialization_path = os.path.join(
    RECOVERY_ROOT,
    "phase4_initialization.json"
)

initialization = show_json(
    initialization_path,
    "2. MODEL INITIALIZATION"
)


# ============================================================
# 3. TRAINING CONFIGURATION
# ============================================================

config_candidates = [
    os.path.join(
        RECOVERY_ROOT,
        "training_example_config.json"
    ),

    os.path.join(
        TRAINING_ROOT,
        "training_example_config.json"
    ),

    os.path.join(
        RECOVERY_ROOT,
        "training",
        "phase4_recovery_config.json"
    ),
]

configs = {}

for path in config_candidates:

    if os.path.exists(path):

        name = os.path.relpath(
            path,
            RECOVERY_ROOT
        )

        configs[name] = read_json(path)

        print("\n" + "=" * 110)
        print("3. TRAINING CONFIGURATION")
        print("=" * 110)

        print(
            "SOURCE:",
            path
        )

        print(
            json.dumps(
                configs[name],
                indent=2,
                ensure_ascii=False
            )[:30000]
        )


# ============================================================
# 4. TRAINING STATE
# ============================================================

training_state_path = os.path.join(
    RECOVERY_ROOT,
    "training_state.json"
)

training_state = show_json(
    training_state_path,
    "4. RECOVERY TRAINING STATE"
)


# ============================================================
# 5. STREAMING CONFIG
# ============================================================

streaming_path = os.path.join(
    RECOVERY_ROOT,
    "streaming_config.json"
)

streaming = show_json(
    streaming_path,
    "5. STREAMING CONFIGURATION"
)


# ============================================================
# 6. DEDUPLICATION / DATA QUALITY
# ============================================================

for filename, title in [
    (
        "phase4_dedup_audit.json",
        "6A. DEDUPLICATION AUDIT"
    ),
    (
        "phase4_duplicate_audit.json",
        "6B. DUPLICATE AUDIT"
    ),
    (
        "phase4_dedup_split_policy.json",
        "6C. DEDUPLICATION / SPLIT POLICY"
    ),
    (
        "phase4_text_quality_audit.json",
        "6D. TEXT QUALITY AUDIT"
    ),
]:

    path = os.path.join(
        TRAINING_ROOT,
        filename
    )

    if not os.path.exists(path):

        path = os.path.join(
            RECOVERY_ROOT,
            filename
        )

    show_json(
        path,
        title
    )


# ============================================================
# 7. TOKENIZER CONFIGURATION
# ============================================================

tokenizer_config_path = os.path.join(
    TRAINING_ROOT,
    "tokenizer",
    "tokenizer_config.json"
)

tokenizer_config = show_json(
    tokenizer_config_path,
    "7A. TOKENIZER CONFIGURATION"
)


tokenizer_path = os.path.join(
    TRAINING_ROOT,
    "tokenizer",
    "legal_bpe_32k.json"
)

print("\n" + "=" * 110)
print("7B. TOKENIZER FILE")
print("=" * 110)

if os.path.exists(tokenizer_path):

    size_mb = (
        os.path.getsize(tokenizer_path)
        / 1024**2
    )

    print("Path :", tokenizer_path)
    print(
        "Size :",
        f"{size_mb:.2f} MB"
    )

    try:

        tok = read_json(tokenizer_path)

        print(
            "Top-level keys:",
            list(tok.keys())
        )

        if "model" in tok:

            model_info = tok["model"]

            print(
                "Tokenizer model type:",
                model_info.get("type")
            )

            print(
                "Vocabulary size:",
                len(model_info.get("vocab", {}))
            )

            print(
                "Merges:",
                len(model_info.get("merges", []))
            )

    except Exception as e:

        print(
            "Could not parse tokenizer JSON:",
            repr(e)
        )

else:

    print("Tokenizer file not found.")


# ============================================================
# 8. MANIFEST SIZES
# ============================================================

print("\n" + "=" * 110)
print("8. DATASET MANIFEST SIZES")
print("=" * 110)

manifest_paths = {

    "train":
        os.path.join(
            TRAINING_ROOT,
            "manifests",
            "train_manifest.jsonl"
        ),

    "validation":
        os.path.join(
            TRAINING_ROOT,
            "manifests",
            "validation_manifest.jsonl"
        ),

    "test":
        os.path.join(
            TRAINING_ROOT,
            "manifests",
            "test_manifest.jsonl"
        ),
}

manifest_counts = {}

for name, path in manifest_paths.items():

    n = count_jsonl(path)

    manifest_counts[name] = n

    print(
        f"{name:12}: {n:,} records"
    )


# ============================================================
# 9. DEDUP SHARD INVENTORY
# ============================================================

print("\n" + "=" * 110)
print("9. TRAINING CORPUS SHARD INVENTORY")
print("=" * 110)

shard_counts = defaultdict(int)

for dirpath, _, filenames in os.walk(
    os.path.join(TRAINING_ROOT, "dedup")
):

    for fn in filenames:

        if fn.endswith(".jsonl"):

            rel = os.path.relpath(
                dirpath,
                os.path.join(
                    TRAINING_ROOT,
                    "dedup"
                )
            )

            shard_counts[rel] += 1

for category, count in sorted(
    shard_counts.items()
):

    print(
        f"{category:30}: {count:4} shards"
    )


# ============================================================
# 10. STEP-1000 TRAINING LOG
# ============================================================

log_path = os.path.join(
    STEP1000_ROOT,
    "training_log.jsonl"
)

print("\n" + "=" * 110)
print("10. STEP-1000 TRAINING LOG")
print("=" * 110)

log_rows = []

if os.path.exists(log_path):

    with open(
        log_path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if not line.strip():
                continue

            try:
                row = json.loads(line)
                log_rows.append(row)

            except Exception:
                pass

    print(
        "Log records:",
        len(log_rows)
    )

else:

    print(
        "training_log.jsonl not found."
    )


# ============================================================
# 11. TRAINING LOG FIELD DISCOVERY
# ============================================================

if log_rows:

    print("\nTraining-log fields:")

    fields = set()

    for row in log_rows:
        fields.update(row.keys())

    for field in sorted(fields):
        print(" ", field)


# ============================================================
# 12. TRAINING TRAJECTORY
# ============================================================

print("\n" + "=" * 110)
print("12. TRAINING TRAJECTORY")
print("=" * 110)

numeric_fields = [
    "step",
    "global_step",
    "optimizer_steps",
    "loss",
    "train_loss",
    "learning_rate",
    "lr",
    "grad_norm",
    "gradient_norm",
    "tokens",
    "tokens_seen",
    "throughput",
]

for field in numeric_fields:

    values = []

    for row in log_rows:

        value = row.get(field)

        if isinstance(value, (int, float)):

            if math.isfinite(float(value)):
                values.append(float(value))

    if values:

        print(
            f"\n{field}"
        )

        print(
            f"  count   : {len(values):,}"
        )

        print(
            f"  first   : {values[0]:.8g}"
        )

        print(
            f"  last    : {values[-1]:.8g}"
        )

        print(
            f"  minimum : {min(values):.8g}"
        )

        print(
            f"  maximum : {max(values):.8g}"
        )


# ============================================================
# 13. LAST TRAINING RECORDS
# ============================================================

print("\n" + "=" * 110)
print("13. LAST TRAINING LOG RECORDS")
print("=" * 110)

for row in log_rows[-10:]:

    print(
        json.dumps(
            row,
            ensure_ascii=False
        )
    )


# ============================================================
# 14. STEP-1000 RECOVERY STATE
# ============================================================

recovery_state_path = os.path.join(
    STEP1000_ROOT,
    "training_recovery_state.json"
)

show_json(
    recovery_state_path,
    "14. STEP-1000 RECOVERY STATE"
)


# ============================================================
# 15. STEP-1000 MANIFEST
# ============================================================

manifest_path = os.path.join(
    STEP1000_ROOT,
    "manifest.json"
)

show_json(
    manifest_path,
    "15. STEP-1000 CHECKPOINT MANIFEST"
)


# ============================================================
# 16. AUTOMATIC INITIAL DIAGNOSIS
# ============================================================

print("\n" + "=" * 110)
print("16. INITIAL DIAGNOSTIC FLAGS")
print("=" * 110)

flags = []


# ------------------------------------------------------------
# Flag 1 — very short training
# ------------------------------------------------------------

step_values = []

for row in log_rows:

    for field in [
        "step",
        "global_step",
        "optimizer_steps",
    ]:

        value = row.get(field)

        if isinstance(value, int):
            step_values.append(value)

if step_values:

    max_step = max(step_values)

    if max_step <= 1000:

        flags.append(
            "TRAINING_LENGTH: log only reaches Step-1000; "
            "the model may simply be under-trained."
        )


# ------------------------------------------------------------
# Flag 2 — high loss
# ------------------------------------------------------------

loss_values = []

for row in log_rows:

    for field in [
        "loss",
        "train_loss",
    ]:

        value = row.get(field)

        if isinstance(value, (int, float)):

            if math.isfinite(float(value)):
                loss_values.append(float(value))

if loss_values:

    print(
        "First recorded loss:",
        loss_values[0]
    )

    print(
        "Last recorded loss :",
        loss_values[-1]
    )


# ------------------------------------------------------------
# Flag 3 — LR
# ------------------------------------------------------------

lr_values = []

for row in log_rows:

    for field in [
        "learning_rate",
        "lr",
    ]:

        value = row.get(field)

        if isinstance(value, (int, float)):

            if math.isfinite(float(value)):
                lr_values.append(float(value))

if lr_values:

    print(
        "Initial LR:",
        lr_values[0]
    )

    print(
        "Final LR  :",
        lr_values[-1]
    )


# ------------------------------------------------------------
# Flag 4 — gradient norm
# ------------------------------------------------------------

grad_values = []

for row in log_rows:

    for field in [
        "grad_norm",
        "gradient_norm",
    ]:

        value = row.get(field)

        if isinstance(value, (int, float)):

            if math.isfinite(float(value)):
                grad_values.append(float(value))

if grad_values:

    print(
        "Gradient norm first:",
        grad_values[0]
    )

    print(
        "Gradient norm last :",
        grad_values[-1]
    )

    if max(grad_values) > 100:

        flags.append(
            "OPTIMIZATION: very large gradient norm observed."
        )


# ------------------------------------------------------------
# Print flags
# ------------------------------------------------------------

print("\n")

if flags:

    for i, flag in enumerate(flags, 1):
        print(
            f"[{i}] {flag}"
        )

else:

    print(
        "No automatic red flags could be established "
        "from the available fields."
    )


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 110)
print("✓ MASTER DIAGNOSTIC EXTRACTION COMPLETE")
print("=" * 110)

print(
    "\nIMPORTANT:"
    "\nThis cell only READ the recovery, training-preparation,"
    "\n and Step-1000 datasets."
    "\nNo model weights were modified."
    "\nNo training was performed."
    "\nNo generation was performed."
)

In [ ]:
# ============================================================
# CS781 — EXPERIMENT 2 CONFIGURATION
# Optimized ~3.5-hour run
#
# Changes vs Step-1000:
#   1. Domain-balanced sampling
#   2. LR warmup + cosine decay
#   3. Larger effective batch if memory allows
#   4. Frequent lightweight diagnostics
#   5. Same 112.8M architecture
#   6. Same 32k tokenizer
#   7. Same 4096 context
#
# NO pretrained weights.
# ============================================================

import os
import gc
import json
import math
import time
import random
import numpy as np
import torch

SEED = 781

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda:0")

assert torch.cuda.is_available()

print("=" * 80)
print("CS781 — EXPERIMENT 2")
print("=" * 80)

print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM:",
    f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
)

# ------------------------------------------------------------
# DATA
# ------------------------------------------------------------

TRAIN_ROOT = "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
RECOVERY_ROOT = "/kaggle/input/datasets/oipranesh/cs781-phase4-final-recovery"

# We use the already verified deduplicated shards.
TRAIN_MANIFEST = os.path.join(
    TRAIN_ROOT,
    "manifests",
    "train_manifest.jsonl"
)

VAL_MANIFEST = os.path.join(
    TRAIN_ROOT,
    "manifests",
    "validation_manifest.jsonl"
)

TOKENIZER_PATH = os.path.join(
    TRAIN_ROOT,
    "tokenizer",
    "legal_bpe_32k.json"
)

# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

N_LAYER = 12
N_HEAD = 12
N_EMBD = 768
N_INNER = 3072
VOCAB_SIZE = 32000
CONTEXT_LENGTH = 4096

# ------------------------------------------------------------
# TRAINING
# ------------------------------------------------------------

# Start conservatively.
# Cell 3 below will test whether microbatch=2 fits.
MICRO_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8

EFFECTIVE_BATCH_SIZE = (
    MICRO_BATCH_SIZE *
    GRADIENT_ACCUMULATION_STEPS
)

# ~3.5 hour wall-clock target.
MAX_HOURS = 3.5

# Save/evaluate frequently enough to identify the useful checkpoint.
CHECKPOINT_EVERY_STEPS = 500

# Lightweight generation diagnostic.
DIAGNOSTIC_EVERY_STEPS = 500

# Keep a fixed small set for generation diagnostics.
NUM_DIAGNOSTIC_PROMPTS = 32

# ------------------------------------------------------------
# OPTIMIZER
# ------------------------------------------------------------

PEAK_LR = 3e-5
MIN_LR = 3e-6

WARMUP_STEPS = 500

WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0

# ------------------------------------------------------------
# DOMAIN MIX
# ------------------------------------------------------------
#
# Target sampling proportions.
# These are deliberately NOT equal because High Courts
# are the dominant source.
#
# The goal is to give smaller domains guaranteed exposure
# without massively duplicating them.

DOMAIN_WEIGHTS = {
    "high_courts": 0.65,
    "tribunals": 0.18,
    "supreme_court": 0.12,
    "acts": 0.05,
}

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

RUN_NAME = "exp2_balanced_optimized"

OUTPUT_ROOT = f"/kaggle/working/{RUN_NAME}"

CHECKPOINT_DIR = os.path.join(
    OUTPUT_ROOT,
    "checkpoints"
)

DIAGNOSTIC_DIR = os.path.join(
    OUTPUT_ROOT,
    "diagnostics"
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DIAGNOSTIC_DIR, exist_ok=True)

print()
print("Architecture:")
print("  layers       :", N_LAYER)
print("  heads        :", N_HEAD)
print("  embedding    :", N_EMBD)
print("  FFN          :", N_INNER)
print("  vocabulary   :", VOCAB_SIZE)
print("  context      :", CONTEXT_LENGTH)

print()
print("Training:")
print("  micro batch  :", MICRO_BATCH_SIZE)
print("  accumulation :", GRADIENT_ACCUMULATION_STEPS)
print("  effective    :", EFFECTIVE_BATCH_SIZE)
print("  peak LR      :", PEAK_LR)
print("  min LR       :", MIN_LR)
print("  warmup       :", WARMUP_STEPS)
print("  weight decay :", WEIGHT_DECAY)

print()
print("Domain mix:")
for k, v in DOMAIN_WEIGHTS.items():
    print(f"  {k:15s}: {v:.0%}")

print()
print("Output:", OUTPUT_ROOT)
print("=" * 80)

In [ ]:
# ============================================================
# VERIFY DOMAIN DATA
# ============================================================

import glob
import os

DOMAIN_GLOBS = {
    "acts": os.path.join(
        TRAIN_ROOT,
        "dedup",
        "acts",
        "*.jsonl"
    ),

    "high_courts": os.path.join(
        TRAIN_ROOT,
        "dedup",
        "high_courts",
        "*.jsonl"
    ),

    "supreme_court": os.path.join(
        TRAIN_ROOT,
        "dedup",
        "supreme_court",
        "*.jsonl"
    ),

    "tribunals": os.path.join(
        TRAIN_ROOT,
        "dedup",
        "tribunals",
        "*.jsonl"
    ),
}

DOMAIN_FILES = {}

print("=" * 80)
print("CS781 — DOMAIN SOURCE VERIFICATION")
print("=" * 80)

for domain, pattern in DOMAIN_GLOBS.items():

    files = sorted(glob.glob(pattern))

    DOMAIN_FILES[domain] = files

    print(
        f"{domain:16s}: "
        f"{len(files):4d} shard/file(s)"
    )

    assert len(files) > 0, (
        f"No files found for domain: {domain}"
    )

print()
print("✓ All four legal domains found")
print("=" * 80)

In [ ]:
# ============================================================
# CS781 — PORT VERIFIED MODEL + TOKENIZER
# ============================================================
# Run this BEFORE any diagnosis/training cells.
#
# Reconstructs the exact architecture used by Step-1000:
#   vocab      = 32000
#   context    = 4096
#   embedding  = 768
#   layers     = 12
#   heads      = 12
#   inner      = 3072
#
# Then loads:
#   /kaggle/input/datasets/oipranesh/
#       phase4-step1000-verified-model/checkpoint_step_001000.pt
#
# ============================================================

import os
import gc
import json
import math
import torch

from pathlib import Path

print("=" * 80)
print("CS781 — LOADING VERIFIED STEP-1000 MODEL")
print("=" * 80)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

RECOVERY_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-final-recovery"
)

TRAINING_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

STEP1000_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/phase4-step1000-verified-model"
)

TOKENIZER_PATH = (
    RECOVERY_ROOT /
    "tokenizer" /
    "legal_bpe_32k.json"
)

CHECKPOINT_PATH = (
    STEP1000_ROOT /
    "checkpoint_step_001000.pt"
)

assert TOKENIZER_PATH.exists(), TOKENIZER_PATH
assert CHECKPOINT_PATH.exists(), CHECKPOINT_PATH

# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda:0" if torch.cuda.is_available()
    else "cpu"
)

if DEVICE.type != "cuda":
    raise RuntimeError("CUDA GPU is required for this experiment.")

print("Device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# IMPORTS
# ------------------------------------------------------------

from tokenizers import Tokenizer

from transformers import (
    GPT2Config,
    GPT2LMHeadModel,
)

# ------------------------------------------------------------
# TOKENIZER
# ------------------------------------------------------------

print()
print("-" * 80)
print("LOADING VERIFIED BPE TOKENIZER")
print("-" * 80)

raw_tokenizer = Tokenizer.from_file(
    str(TOKENIZER_PATH)
)

print("✓ Raw tokenizer loaded")
print("Vocabulary size:", raw_tokenizer.get_vocab_size())

assert raw_tokenizer.get_vocab_size() == 32000

# ------------------------------------------------------------
# TOKENIZER ADAPTER
#
# The training tokenizer is a HuggingFace tokenizers.Tokenizer,
# not transformers.PreTrainedTokenizer.
#
# This adapter provides the methods needed by our diagnosis
# and generation code.
# ------------------------------------------------------------

class CS781TokenizerAdapter:

    def __init__(self, tokenizer):
        self._tokenizer = tokenizer

    @property
    def vocab_size(self):
        return self._tokenizer.get_vocab_size()

    def encode(
        self,
        text,
        add_special_tokens=True,
    ):
        return self._tokenizer.encode(
            str(text),
            add_special_tokens=add_special_tokens,
        ).ids

    def decode(
        self,
        ids,
        skip_special_tokens=False,
    ):
        return self._tokenizer.decode(
            list(map(int, ids)),
            skip_special_tokens=skip_special_tokens,
        )

    def __call__(
        self,
        text,
        truncation=False,
        max_length=None,
        return_tensors=None,
        add_special_tokens=True,
        **kwargs,
    ):
        ids = self.encode(
            text,
            add_special_tokens=add_special_tokens,
        )

        if truncation and max_length is not None:
            ids = ids[:max_length]

        result = {
            "input_ids": ids,
            "attention_mask": [1] * len(ids),
        }

        if return_tensors == "pt":
            result["input_ids"] = torch.tensor(
                [ids],
                dtype=torch.long,
            )

            result["attention_mask"] = torch.tensor(
                [[1] * len(ids)],
                dtype=torch.long,
            )

        return result


tokenizer = CS781TokenizerAdapter(
    raw_tokenizer
)

print("✓ Tokenizer adapter created")
print("Tokenizer vocab:", tokenizer.vocab_size)

# ------------------------------------------------------------
# EXACT MODEL ARCHITECTURE
# ------------------------------------------------------------

print()
print("-" * 80)
print("CONSTRUCTING VERIFIED ARCHITECTURE")
print("-" * 80)

config = GPT2Config(
    vocab_size=32000,
    n_positions=4096,
    n_ctx=4096,

    n_embd=768,
    n_layer=12,
    n_head=12,
    n_inner=3072,

    activation_function="gelu_new",

    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,

    use_cache=False,

    bos_token_id=None,
    eos_token_id=None,
)

model = GPT2LMHeadModel(config)

print("✓ Model constructed")

# ------------------------------------------------------------
# LOAD CHECKPOINT
# ------------------------------------------------------------

print()
print("-" * 80)
print("LOADING STEP-1000 CHECKPOINT")
print("-" * 80)

checkpoint = torch.load(
    str(CHECKPOINT_PATH),
    map_location="cpu",
    weights_only=False,
)

print("Checkpoint type:", type(checkpoint))

assert isinstance(checkpoint, dict)

print("Checkpoint keys:")
for key in checkpoint.keys():
    print(" ", key)

assert "model_state_dict" in checkpoint

state_dict = checkpoint["model_state_dict"]

missing, unexpected = model.load_state_dict(
    state_dict,
    strict=True,
)

assert len(missing) == 0
assert len(unexpected) == 0

print()
print("✓ State dict loaded")
print("✓ No missing keys")
print("✓ No unexpected keys")

# ------------------------------------------------------------
# GPU
# ------------------------------------------------------------

model = model.to(DEVICE)

model.eval()

if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

# ------------------------------------------------------------
# FINAL ARCHITECTURE CHECK
# ------------------------------------------------------------

print()
print("=" * 80)
print("MODEL VERIFICATION")
print("=" * 80)

print("vocab_size :", model.config.vocab_size)
print("n_positions:", model.config.n_positions)
print("n_embd     :", model.config.n_embd)
print("n_layer    :", model.config.n_layer)
print("n_head     :", model.config.n_head)
print("n_inner    :", model.config.n_inner)

assert model.config.vocab_size == 32000
assert model.config.n_positions == 4096
assert model.config.n_embd == 768
assert model.config.n_layer == 12
assert model.config.n_head == 12
assert model.config.n_inner == 3072

model_device = next(
    model.parameters()
).device

print()
print("Model device:", model_device)

assert model_device == DEVICE

print("✓ Model is on correct GPU")
print("✓ Model is in eval mode:", not model.training)

# ------------------------------------------------------------
# FINITENESS
# ------------------------------------------------------------

bad_params = 0

for name, param in model.named_parameters():

    if not torch.isfinite(param).all():
        print("NON-FINITE:", name)
        bad_params += 1

assert bad_params == 0

print("✓ All model parameters are finite")

# ------------------------------------------------------------
# TOKENIZER → MODEL FORWARD TEST
# ------------------------------------------------------------

print()
print("-" * 80)
print("TOKENIZER → MODEL FORWARD TEST")
print("-" * 80)

test_text = (
    "The Court considered the provisions of the Act "
    "and examined whether the petition was maintainable."
)

encoded = tokenizer(
    test_text,
    truncation=True,
    max_length=4096,
    return_tensors="pt",
)

input_ids = encoded["input_ids"].to(DEVICE)
attention_mask = encoded["attention_mask"].to(DEVICE)

print("Input shape:", tuple(input_ids.shape))
print("First IDs:", input_ids[0, :20].tolist())

assert input_ids.min().item() >= 0
assert input_ids.max().item() < 32000

with torch.inference_mode():

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
    )

logits = outputs.logits

print("Logits shape:", tuple(logits.shape))
print("Logits device:", logits.device)

assert logits.shape[-1] == 32000
assert torch.isfinite(logits).all()

print("✓ Tokenizer works")
print("✓ Token IDs valid")
print("✓ Forward pass works")
print("✓ Logits finite")

# ------------------------------------------------------------
# SAVE CHECKPOINT METADATA FOR DIAGNOSIS
# ------------------------------------------------------------

STEP1000 = checkpoint.get(
    "step",
    1000,
)

print()
print("=" * 80)
print("✓ VERIFIED MODEL PORT COMPLETE")
print("=" * 80)

print("Checkpoint step:", STEP1000)
print("Device:", DEVICE)
print("Tokenizer:", TOKENIZER_PATH)
print("Checkpoint:", CHECKPOINT_PATH)

print()
print("READY FOR:")
print("  → dataset diagnosis")
print("  → repetition analysis")
print("  → train/validation loss analysis")
print("  → generation diagnostics")
print("  → controlled training experiments")

# ------------------------------------------------------------
# CLEANUP
# ------------------------------------------------------------

del checkpoint
gc.collect()
torch.cuda.empty_cache()

print()
print("GPU allocated:",
      f"{torch.cuda.memory_allocated() / 1024**3:.3f} GB")

In [ ]:
# ============================================================
# T4 MICRO-BATCH MEMORY TEST
# ============================================================

import torch
import gc

print("=" * 80)
print("CS781 — MICRO-BATCH MEMORY TEST")
print("=" * 80)

# IMPORTANT:
# This assumes `model` and `tokenizer` are already constructed
# exactly as in your verified training notebook.

if "model" not in globals():
    raise RuntimeError(
        "model is not loaded. "
        "Port the exact model-construction code from the verified notebook first."
    )

if "tokenizer" not in globals():
    raise RuntimeError(
        "tokenizer is not loaded. "
        "Port the exact tokenizer-loading code from the verified notebook first."
    )

model = model.to(DEVICE)
model.train()

torch.cuda.empty_cache()
gc.collect()

# Use a synthetic sequence only for memory testing.
test_ids = torch.randint(
    low=0,
    high=VOCAB_SIZE,
    size=(MICRO_BATCH_SIZE, CONTEXT_LENGTH),
    device=DEVICE,
    dtype=torch.long,
)

print("Input shape:", tuple(test_ids.shape))

try:

    torch.cuda.reset_peak_memory_stats()

    outputs = model(
        input_ids=test_ids,
        labels=test_ids,
    )

    loss = outputs.loss

    loss.backward()

    peak_gb = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )

    print()
    print("✓ Forward successful")
    print("✓ Backward successful")
    print(
        f"Peak allocated VRAM: {peak_gb:.2f} GB"
    )

    if peak_gb > 14.5:

        print()
        print(
            "WARNING: memory usage is very high."
        )

        print(
            "Use MICRO_BATCH_SIZE = 1 "
            "for the real run."
        )

    else:

        print()
        print(
            "✓ MICRO_BATCH_SIZE=2 appears feasible"
        )

except torch.cuda.OutOfMemoryError:

    print()
    print(
        "MICRO_BATCH_SIZE=2 DOES NOT FIT."
    )

    raise RuntimeError(
        "Set MICRO_BATCH_SIZE=1 and "
        "GRADIENT_ACCUMULATION_STEPS=16."
    )

finally:

    model.zero_grad(set_to_none=True)

    del test_ids

    torch.cuda.empty_cache()
    gc.collect()

print("=" * 80)

In [ ]:
# ============================================================
# CS781 — DOMAIN-BALANCED STREAMING SAMPLER
# ============================================================

import json
import random

def read_jsonl_record(fp):

    with open(
        fp,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            try:
                obj = json.loads(line)

            except json.JSONDecodeError:
                continue

            if not isinstance(obj, dict):
                continue

            text = obj.get("text")

            if text is None:
                continue

            text = str(text).strip()

            if not text:
                continue

            yield text


class DomainSampler:

    def __init__(
        self,
        domain_files,
        domain_weights,
        seed=781,
    ):

        self.domain_files = domain_files
        self.domain_weights = domain_weights

        self.domains = list(
            domain_weights.keys()
        )

        weights = [
            domain_weights[d]
            for d in self.domains
        ]

        total = sum(weights)

        self.weights = [
            w / total
            for w in weights
        ]

        self.rng = random.Random(seed)

    def sample_domain(self):

        return self.rng.choices(
            self.domains,
            weights=self.weights,
            k=1,
        )[0]

    def sample_file(self, domain):

        files = self.domain_files[domain]

        return self.rng.choice(files)

    def sample_document(self):

        domain = self.sample_domain()

        fp = self.sample_file(domain)

        # Pick a random-ish record from a shard by
        # bounded reservoir-style sampling.
        chosen = None

        for i, text in enumerate(
            read_jsonl_record(fp)
        ):

            if self.rng.random() < 1.0 / (i + 1):
                chosen = text

        return domain, chosen


sampler = DomainSampler(
    DOMAIN_FILES,
    DOMAIN_WEIGHTS,
    seed=SEED,
)

print("=" * 80)
print("DOMAIN SAMPLER READY")
print("=" * 80)

for _ in range(10):

    domain, text = sampler.sample_document()

    print(
        f"{domain:16s} | "
        f"{len(text) if text else 0:,} chars"
    )

print()
print("✓ Controlled domain sampling enabled")
print("=" * 80)

In [ ]:
# ============================================================
# CS781 — OPTIMIZER + WARMUP/COSINE LR SCHEDULE
# ============================================================

import math
import torch

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=PEAK_LR,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=WEIGHT_DECAY,
)

def lr_lambda(step):

    # Warmup
    if step < WARMUP_STEPS:

        return max(
            1e-8,
            float(step + 1)
            / float(WARMUP_STEPS),
        )

    # Estimate total steps dynamically from runtime.
    progress = min(
        1.0,
        max(
            0.0,
            (time.time() - TRAIN_START_TIME)
            / (MAX_HOURS * 3600.0)
        )
    )

    cosine = (
        0.5
        * (
            1.0
            + math.cos(
                math.pi * progress
            )
        )
    )

    min_ratio = MIN_LR / PEAK_LR

    return (
        min_ratio
        + (1.0 - min_ratio) * cosine
    )


scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda,
)

print("=" * 80)
print("OPTIMIZER READY")
print("=" * 80)

print("Optimizer      : AdamW")
print("Peak LR        :", PEAK_LR)
print("Minimum LR     :", MIN_LR)
print("Warmup steps   :", WARMUP_STEPS)
print("Weight decay   :", WEIGHT_DECAY)
print("Grad clip      :", MAX_GRAD_NORM)

print()
print("✓ Optimizer configured")
print("✓ Warmup + cosine decay configured")
print("=" * 80)

In [ ]:
# ============================================================
# CS781 — VALIDATION DATA LOADER (FIXED)
# ============================================================

import json
from pathlib import Path

TRAINING_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

RECOVERY_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-final-recovery"
)

# Prefer the verified validation manifest.
candidates = [
    TRAINING_ROOT / "manifests" / "validation_manifest.jsonl",
    RECOVERY_ROOT / "manifests" / "validation_manifest.jsonl",
]

VALIDATION_MANIFEST = next(
    (p for p in candidates if p.exists()),
    None
)

if VALIDATION_MANIFEST is None:
    raise FileNotFoundError(
        "validation_manifest.jsonl not found.\n"
        + "\n".join(str(p) for p in candidates)
    )

print("=" * 80)
print("CS781 — LOADING VALIDATION MANIFEST")
print("=" * 80)
print("Using:")
print(VALIDATION_MANIFEST)

validation_records = []

with open(
    VALIDATION_MANIFEST,
    "r",
    encoding="utf-8",
) as f:

    for line_no, line in enumerate(f, 1):

        line = line.strip()

        if not line:
            continue

        try:
            record = json.loads(line)
        except Exception as e:
            print(
                f"Skipping malformed line {line_no}: {e}"
            )
            continue

        validation_records.append(record)

print()
print("Validation records:", len(validation_records))

if not validation_records:
    raise RuntimeError(
        "Validation manifest contains zero records."
    )

# ------------------------------------------------------------
# INSPECT ACTUAL RECORD STRUCTURE
# ------------------------------------------------------------

print()
print("-" * 80)
print("FIRST VALIDATION RECORD")
print("-" * 80)

first = validation_records[0]

print("Type:", type(first))

if isinstance(first, dict):
    print("Keys:", list(first.keys()))

    for key, value in first.items():
        if isinstance(value, str):
            print(
                f"{key}: string "
                f"({len(value):,} chars)"
            )
            print(
                "Preview:",
                repr(value[:300])
            )
        else:
            print(
                f"{key}: {type(value)} -> {str(value)[:300]}"
            )
else:
    print(repr(first)[:1000])


# ------------------------------------------------------------
# EXTRACT TEXT WITHOUT ASSUMING A FIELD NAME
# ------------------------------------------------------------

def extract_text(record):

    if isinstance(record, str):
        return record

    if not isinstance(record, dict):
        return None

    # Most likely direct text fields.
    for key in (
        "text",
        "content",
        "document",
        "body",
        "raw_text",
    ):
        value = record.get(key)

        if isinstance(value, str) and value.strip():
            return value.strip()

    # Some manifests store a nested document.
    for value in record.values():

        if isinstance(value, dict):

            nested = extract_text(value)

            if nested:
                return nested

    return None


validation_texts = []

for record in validation_records:

    text = extract_text(record)

    if text:
        validation_texts.append(text)

print()
print("=" * 80)
print("VALIDATION TEXT EXTRACTION")
print("=" * 80)

print(
    "Records:",
    len(validation_records),
)

print(
    "Texts extracted:",
    len(validation_texts),
)

if validation_texts:

    lengths = [
        len(x)
        for x in validation_texts
    ]

    print(
        "Min chars:",
        min(lengths),
    )

    print(
        "Max chars:",
        max(lengths),
    )

    print(
        "Mean chars:",
        sum(lengths) / len(lengths),
    )

    print()
    print("First text:")
    print(validation_texts[0][:1000])

else:

    print()
    print(
        "No direct text was found in the manifest."
    )

    print(
        "This is NOT a model failure — "
        "the manifest likely contains file/shard references."
    )

print()
print("=" * 80)

# Do NOT assert yet if the manifest is reference-based.
# The next diagnostic will resolve its referenced files.

In [ ]:
# ============================================================
# CS781 — KAGGLE STORAGE AUDIT
# ============================================================

import os
import shutil
from pathlib import Path

print("=" * 100)
print("CS781 — STORAGE AUDIT")
print("=" * 100)


def size_bytes(path):

    path = Path(path)

    if path.is_file():

        try:
            return path.stat().st_size
        except:
            return 0

    total = 0

    try:

        for root, dirs, files in os.walk(path):

            for name in files:

                fp = Path(root) / name

                try:
                    total += fp.stat().st_size
                except:
                    pass

    except:
        pass

    return total


def gb(n):
    return n / (1024 ** 3)


# ------------------------------------------------------------
# FILESYSTEM SUMMARY
# ------------------------------------------------------------

for root in [
    "/kaggle/input",
    "/kaggle/working",
    "/kaggle/temp",
]:

    if not os.path.exists(root):
        continue

    usage = shutil.disk_usage(root)

    print()
    print("-" * 100)
    print(root)
    print("-" * 100)

    print(
        "Total     :",
        f"{gb(usage.total):.2f} GB",
    )

    print(
        "Used      :",
        f"{gb(usage.used):.2f} GB",
    )

    print(
        "Free      :",
        f"{gb(usage.free):.2f} GB",
    )


# ------------------------------------------------------------
# DATASET ROOTS
# ------------------------------------------------------------

roots = [
    Path(
        "/kaggle/input/datasets/oipranesh/"
        "cs781-phase4-final-recovery"
    ),

    Path(
        "/kaggle/input/datasets/oipranesh/"
        "cs781-phase4-training-prep"
    ),

    Path(
        "/kaggle/input/datasets/oipranesh/"
        "phase4-step1000-verified-model"
    ),
]


print()
print("=" * 100)
print("DATASET ROOT SIZES")
print("=" * 100)

for root in roots:

    if not root.exists():
        print("MISSING:", root)
        continue

    s = size_bytes(root)

    print(
        f"{gb(s):8.2f} GB   {root}"
    )


# ------------------------------------------------------------
# LARGEST DIRECTORIES
# ------------------------------------------------------------

print()
print("=" * 100)
print("LARGEST DIRECTORIES / DATA GROUPS")
print("=" * 100)

all_dirs = []

for root in roots:

    if not root.exists():
        continue

    for current, dirs, files in os.walk(root):

        current = Path(current)

        # Only inspect directories reasonably close to dataset root.
        # This keeps the audit manageable.
        depth = len(current.relative_to(root).parts)

        if depth <= 3:

            s = size_bytes(current)

            all_dirs.append(
                (s, current)
            )

for s, path in sorted(
    all_dirs,
    reverse=True,
)[:40]:

    print(
        f"{gb(s):8.3f} GB   {path}"
    )


# ------------------------------------------------------------
# LARGEST INDIVIDUAL FILES
# ------------------------------------------------------------

print()
print("=" * 100)
print("LARGEST INDIVIDUAL FILES")
print("=" * 100)

files_found = []

for root in roots:

    if not root.exists():
        continue

    for current, dirs, files in os.walk(root):

        for name in files:

            path = Path(current) / name

            try:
                s = path.stat().st_size
            except:
                continue

            files_found.append(
                (s, path)
            )

for s, path in sorted(
    files_found,
    reverse=True,
)[:50]:

    print(
        f"{gb(s):8.3f} GB   {path}"
    )


# ------------------------------------------------------------
# EXTENSION BREAKDOWN
# ------------------------------------------------------------

from collections import defaultdict

extension_sizes = defaultdict(int)
extension_counts = defaultdict(int)

for s, path in files_found:

    ext = path.suffix.lower()

    if not ext:
        ext = "[no extension]"

    extension_sizes[ext] += s
    extension_counts[ext] += 1

print()
print("=" * 100)
print("STORAGE BY FILE TYPE")
print("=" * 100)

for ext in sorted(
    extension_sizes,
    key=extension_sizes.get,
    reverse=True,
):

    print(
        f"{ext:20s} "
        f"{extension_counts[ext]:7,} files   "
        f"{gb(extension_sizes[ext]):8.3f} GB"
    )


print()
print("=" * 100)
print("✓ STORAGE AUDIT COMPLETE")
print("=" * 100)

In [ ]:
# ============================================================
# CS781 — RESOLVE VALIDATION MANIFEST -> REAL SOURCE DATA
# ============================================================

import json
import gzip
from pathlib import Path

TRAINING_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

VALIDATION_MANIFEST = (
    TRAINING_ROOT / "manifests" / "validation_manifest.jsonl"
)

print("=" * 100)
print("CS781 — VALIDATION SOURCE RESOLUTION")
print("=" * 100)

# Read a small number of manifest records only.
manifest_records = []

with open(VALIDATION_MANIFEST, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            manifest_records.append(json.loads(line))

        if len(manifest_records) >= 20:
            break

print("Sample manifest records:", len(manifest_records))

print()
for i, r in enumerate(manifest_records[:5]):
    print(f"[{i}]")
    print(json.dumps(r, indent=2)[:1500])


# ------------------------------------------------------------
# Resolve source_file paths
# ------------------------------------------------------------

print()
print("=" * 100)
print("SOURCE FILE RESOLUTION")
print("=" * 100)

source_files = sorted(
    set(
        r["source_file"]
        for r in manifest_records
        if "source_file" in r
    )
)

for src in source_files:

    candidates = [
        TRAINING_ROOT / "dedup" / src,
        TRAINING_ROOT / src,
    ]

    found = next(
        (p for p in candidates if p.exists()),
        None
    )

    print()
    print("Manifest source :", src)
    print("Resolved path   :", found)

    if found is None:
        print("WARNING: not found")
        continue

    # Inspect first records without loading whole file.
    print("File size       :",
          f"{found.stat().st_size / (1024**3):.3f} GB")

    opener = gzip.open if found.suffix == ".gz" else open

    with opener(found, "rt", encoding="utf-8") as f:

        for j in range(3):

            line = f.readline()

            if not line:
                break

            try:
                obj = json.loads(line)

                print()
                print(
                    f"Record {j} type:",
                    type(obj)
                )

                if isinstance(obj, dict):
                    print(
                        "Keys:",
                        list(obj.keys())
                    )

                    for k, v in obj.items():
                        if isinstance(v, str):
                            print(
                                f"  {k}: "
                                f"string({len(v):,} chars)"
                            )
                        else:
                            print(
                                f"  {k}: {type(v)}"
                            )

                else:
                    print(
                        repr(obj)[:1000]
                    )

            except Exception as e:
                print("JSON error:", e)

print()
print("=" * 100)
print("✓ SOURCE RESOLUTION COMPLETE")
print("=" * 100)

In [ ]:
# ============================================================
# CS781 — STREAMING VALIDATION SAMPLE
# ============================================================

import json
import gzip
import random
from pathlib import Path
from collections import Counter

TRAINING_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

VALIDATION_MANIFEST = (
    TRAINING_ROOT / "manifests" / "validation_manifest.jsonl"
)

# Keep diagnosis cheap.
MAX_VALIDATION_TEXTS = 5000

print("=" * 100)
print("CS781 — STREAMING VALIDATION DIAGNOSTIC SAMPLE")
print("=" * 100)

# ------------------------------------------------------------
# Read manifest and group requested hashes by source file
# ------------------------------------------------------------

requested = {}

with open(
    VALIDATION_MANIFEST,
    "r",
    encoding="utf-8",
) as f:

    for line in f:

        if not line.strip():
            continue

        r = json.loads(line)

        src = r.get("source_file")
        h = r.get("text_hash")

        if src and h:
            requested.setdefault(src, set()).add(h)

print("Manifest source groups:", len(requested))

for src, hashes in requested.items():
    print(
        f"{src}: {len(hashes):,} validation hashes"
    )


# ------------------------------------------------------------
# Hash helper
# ------------------------------------------------------------

import hashlib

def text_hash(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


# ------------------------------------------------------------
# Extract text from source records
# ------------------------------------------------------------

def extract_text(obj):

    if isinstance(obj, str):
        return obj

    if not isinstance(obj, dict):
        return None

    # Known/common possibilities.
    for key in (
        "text",
        "content",
        "document",
        "body",
        "raw_text",
    ):

        value = obj.get(key)

        if isinstance(value, str) and value.strip():
            return value.strip()

    return None


validation_texts = []
matched_hashes = set()

# ------------------------------------------------------------
# Scan source files
# ------------------------------------------------------------

for src, wanted_hashes in requested.items():

    if len(validation_texts) >= MAX_VALIDATION_TEXTS:
        break

    candidates = [
        TRAINING_ROOT / "dedup" / src,
        TRAINING_ROOT / src,
    ]

    source_path = next(
        (
            p
            for p in candidates
            if p.exists()
        ),
        None,
    )

    if source_path is None:
        print(
            "MISSING:",
            src
        )
        continue

    print()
    print(
        "Scanning:",
        source_path
    )

    opener = (
        gzip.open
        if source_path.suffix == ".gz"
        else open
    )

    scanned = 0
    matched = 0

    with opener(
        source_path,
        "rt",
        encoding="utf-8",
    ) as f:

        for line in f:

            if len(validation_texts) >= MAX_VALIDATION_TEXTS:
                break

            if not line.strip():
                continue

            scanned += 1

            try:
                obj = json.loads(line)
            except:
                continue

            text = extract_text(obj)

            if not text:
                continue

            h = text_hash(text)

            if h in wanted_hashes:

                validation_texts.append(text)
                matched_hashes.add(h)
                matched += 1

    print(
        "  Source records scanned:",
        f"{scanned:,}"
    )

    print(
        "  Validation records matched:",
        f"{matched:,}"
    )


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print()
print("=" * 100)
print("VALIDATION SAMPLE RESULT")
print("=" * 100)

print(
    "Texts recovered:",
    f"{len(validation_texts):,}"
)

print(
    "Requested hashes matched:",
    f"{len(matched_hashes):,}"
)

if not validation_texts:

    raise RuntimeError(
        "Could not recover validation texts. "
        "Cell 1 output is needed to determine the exact source schema."
    )

lengths = [
    len(x)
    for x in validation_texts
]

print(
    "Min chars:",
    f"{min(lengths):,}"
)

print(
    "Max chars:",
    f"{max(lengths):,}"
)

print(
    "Mean chars:",
    f"{sum(lengths)/len(lengths):,.1f}"
)

print()
print("FIRST VALIDATION TEXT")
print("-" * 100)
print(validation_texts[0][:2000])

print()
print("=" * 100)
print("✓ VALIDATION TEXT SAMPLE READY")
print("=" * 100)

In [ ]:
# ============================================================
# CS781 — CORRECT VALIDATION SOURCE MAPPING
# ============================================================

import json
import gzip
import hashlib
from pathlib import Path

TRAINING_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

VALIDATION_MANIFEST = (
    TRAINING_ROOT / "manifests" / "validation_manifest.jsonl"
)

print("=" * 100)
print("CS781 — VALIDATION SOURCE MAPPING")
print("=" * 100)

# ------------------------------------------------------------
# Read a few manifest records
# ------------------------------------------------------------

manifest_sample = []

with open(
    VALIDATION_MANIFEST,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        if line.strip():

            manifest_sample.append(
                json.loads(line)
            )

        if len(manifest_sample) >= 20:
            break


print(
    "Manifest sample:",
    len(manifest_sample)
)

# ------------------------------------------------------------
# Map manifest source -> actual dedup source
# ------------------------------------------------------------

def resolve_source(source_file):

    source_file = str(source_file)

    # Manifest uses:
    #   acts/documents.jsonl.gz
    #
    # Actual mounted dataset:
    #   dedup/acts/documents.jsonl

    relative = Path(source_file)

    candidates = [

        # Remove .gz if necessary.
        TRAINING_ROOT
        / "dedup"
        / relative.with_suffix(""),

        # Direct dedup path.
        TRAINING_ROOT
        / "dedup"
        / relative,

        # In case source is already relative to root.
        TRAINING_ROOT
        / relative,

    ]

    for p in candidates:

        if p.exists():
            return p

    # Explicit special handling for the known acts source.
    if source_file == "acts/documents.jsonl.gz":

        p = (
            TRAINING_ROOT
            / "dedup"
            / "acts"
            / "documents.jsonl"
        )

        if p.exists():
            return p

    return None


resolved = {}

for record in manifest_sample:

    src = record["source_file"]

    if src not in resolved:

        resolved[src] = resolve_source(src)


print()
print("=" * 100)
print("RESOLVED SOURCES")
print("=" * 100)

for src, path in resolved.items():

    print()
    print("Manifest:")
    print(" ", src)

    print("Actual:")
    print(" ", path)

    if path is not None:

        print(
            "Size:",
            f"{path.stat().st_size / (1024**3):.3f} GB"
        )

        # Inspect first record.
        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:

            line = f.readline()

        obj = json.loads(line)

        print(
            "Record type:",
            type(obj)
        )

        if isinstance(obj, dict):

            print(
                "Keys:",
                list(obj.keys())
            )

            for k, v in obj.items():

                if isinstance(v, str):

                    print(
                        f"  {k}: "
                        f"string({len(v):,} chars)"
                    )

                else:

                    print(
                        f"  {k}:",
                        type(v)
                    )

print()
print("=" * 100)
print("✓ CORRECT SOURCE MAPPING COMPLETE")
print("=" * 100)

In [ ]:
# ============================================================
# CS781 — RECOVER VALIDATION TEXTS FROM ACTUAL SOURCES
# ============================================================

import json
import hashlib
from pathlib import Path

TRAINING_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

VALIDATION_MANIFEST = (
    TRAINING_ROOT / "manifests" / "validation_manifest.jsonl"
)

MAX_VALIDATION_TEXTS = 5000


print("=" * 100)
print("CS781 — RECOVERING VALIDATION TEXTS")
print("=" * 100)


# ------------------------------------------------------------
# Text extraction
# ------------------------------------------------------------

def extract_text(obj):

    if isinstance(obj, str):
        return obj.strip()

    if not isinstance(obj, dict):
        return None

    # Actual dataset should normally use "text",
    # but keep this robust.
    for key in (
        "text",
        "content",
        "document",
        "body",
        "raw_text",
    ):

        value = obj.get(key)

        if (
            isinstance(value, str)
            and value.strip()
        ):
            return value.strip()

    return None


def sha256_text(text):

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


# ------------------------------------------------------------
# Build validation hash index
# ------------------------------------------------------------

wanted_by_source = {}

manifest_count = 0

with open(
    VALIDATION_MANIFEST,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        if not line.strip():
            continue

        record = json.loads(line)

        source = record.get("source_file")
        text_hash = record.get("text_hash")

        if source and text_hash:

            wanted_by_source.setdefault(
                source,
                set()
            ).add(text_hash)

            manifest_count += 1


print(
    "Validation manifest records:",
    f"{manifest_count:,}"
)


# ------------------------------------------------------------
# Resolve actual source
# ------------------------------------------------------------

def resolve_source(source_file):

    relative = Path(source_file)

    candidates = [

        TRAINING_ROOT
        / "dedup"
        / relative.with_suffix(""),

        TRAINING_ROOT
        / "dedup"
        / relative,

        TRAINING_ROOT
        / relative,
    ]

    for p in candidates:

        if p.exists():
            return p

    return None


# ------------------------------------------------------------
# Recover texts
# ------------------------------------------------------------

validation_texts = []

matched_hashes = set()

matched_by_source = {}


for source_file, wanted_hashes in wanted_by_source.items():

    if len(validation_texts) >= MAX_VALIDATION_TEXTS:
        break

    source_path = resolve_source(
        source_file
    )

    print()
    print("-" * 100)
    print("Manifest source :", source_file)
    print("Actual source   :", source_path)

    if source_path is None:

        print(
            "ERROR: source not found"
        )

        continue

    scanned = 0
    matched = 0

    print(
        "Scanning:",
        f"{source_path.stat().st_size / (1024**3):.3f} GB"
    )

    with open(
        source_path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if (
                len(validation_texts)
                >= MAX_VALIDATION_TEXTS
            ):
                break

            if not line.strip():
                continue

            scanned += 1

            try:
                obj = json.loads(line)
            except Exception:
                continue

            text = extract_text(obj)

            if not text:
                continue

            h = sha256_text(text)

            if h in wanted_hashes:

                validation_texts.append(text)

                matched_hashes.add(h)

                matched += 1

    matched_by_source[source_file] = matched

    print(
        "Records scanned:",
        f"{scanned:,}"
    )

    print(
        "Validation records recovered:",
        f"{matched:,}"
    )


# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

print()
print("=" * 100)
print("VALIDATION RECOVERY RESULT")
print("=" * 100)

print(
    "Manifest records:",
    f"{manifest_count:,}"
)

print(
    "Texts recovered:",
    f"{len(validation_texts):,}"
)

print(
    "Unique hashes matched:",
    f"{len(matched_hashes):,}"
)

print()
print("MATCHES BY SOURCE")

for source, count in matched_by_source.items():

    print(
        f"  {source}: {count:,}"
    )


if not validation_texts:

    raise RuntimeError(
        "Still could not recover validation texts. "
        "Inspect the actual source schema printed by Cell 1."
    )


# ------------------------------------------------------------
# Basic text statistics
# ------------------------------------------------------------

char_lengths = [
    len(x)
    for x in validation_texts
]

print()
print("=" * 100)
print("VALIDATION TEXT STATISTICS")
print("=" * 100)

print(
    "Minimum characters:",
    f"{min(char_lengths):,}"
)

print(
    "Maximum characters:",
    f"{max(char_lengths):,}"
)

print(
    "Mean characters:",
    f"{sum(char_lengths) / len(char_lengths):,.1f}"
)

print()
print("FIRST VALIDATION TEXT")
print("-" * 100)
print(validation_texts[0][:2000])

print()
print("=" * 100)
print("✓ VALIDATION TEXTS SUCCESSFULLY RECOVERED")
print("=" * 100)

In [ ]:
# ============================================================
# CS781 — TARGETED REPETITION / TOKENIZER DIAGNOSIS
# ============================================================

import math
import statistics
from collections import Counter

print("=" * 100)
print("CS781 — TARGETED GENERATION FAILURE DIAGNOSIS")
print("=" * 100)

if "validation_texts" not in globals() or not validation_texts:
    raise RuntimeError(
        "validation_texts not found. Run the validation recovery cell first."
    )

if "tokenizer" not in globals():
    raise RuntimeError(
        "tokenizer not found. Load the verified legal_bpe_32k tokenizer first."
    )


# ------------------------------------------------------------
# TOKENIZER ADAPTER
# ------------------------------------------------------------

def encode_text(text):

    # HuggingFace tokenizers.Tokenizer
    if hasattr(tokenizer, "encode"):

        enc = tokenizer.encode(text)

        if hasattr(enc, "ids"):
            return list(enc.ids)

        if isinstance(enc, list):
            return list(enc)

    # transformers tokenizer
    if callable(tokenizer):

        out = tokenizer(
            text,
            add_special_tokens=True,
        )

        if hasattr(out, "input_ids"):
            ids = out.input_ids

            if hasattr(ids, "tolist"):
                ids = ids.tolist()

            if ids and isinstance(ids[0], list):
                ids = ids[0]

            return list(ids)

        if isinstance(out, dict):
            ids = out["input_ids"]

            if hasattr(ids, "tolist"):
                ids = ids.tolist()

            if ids and isinstance(ids[0], list):
                ids = ids[0]

            return list(ids)

    raise RuntimeError(
        "Unsupported tokenizer interface."
    )


# ------------------------------------------------------------
# TOKENIZE SAMPLE
# ------------------------------------------------------------

print()
print("Tokenizing validation sample...")

token_sequences = []

for i, text in enumerate(validation_texts):

    try:

        ids = encode_text(text)

        if ids:
            token_sequences.append(ids)

    except Exception as e:

        print(
            f"Tokenizer failure at document {i}: {e}"
        )
        break


if not token_sequences:
    raise RuntimeError(
        "No validation documents could be tokenized."
    )


# ------------------------------------------------------------
# BASIC TOKEN STATISTICS
# ------------------------------------------------------------

token_lengths = [
    len(x)
    for x in token_sequences
]

print()
print("=" * 100)
print("1. TOKEN LENGTH DISTRIBUTION")
print("=" * 100)

def percentile(values, p):

    values = sorted(values)

    if not values:
        return 0

    k = (len(values) - 1) * p

    f = math.floor(k)
    c = math.ceil(k)

    if f == c:
        return values[int(k)]

    return (
        values[f] * (c - k)
        + values[c] * (k - f)
    )


print(
    "Documents:",
    f"{len(token_lengths):,}"
)

print(
    "Min:",
    f"{min(token_lengths):,}"
)

print(
    "P25:",
    f"{percentile(token_lengths, .25):,.0f}"
)

print(
    "Median:",
    f"{percentile(token_lengths, .50):,.0f}"
)

print(
    "P75:",
    f"{percentile(token_lengths, .75):,.0f}"
)

print(
    "P90:",
    f"{percentile(token_lengths, .90):,.0f}"
)

print(
    "P95:",
    f"{percentile(token_lengths, .95):,.0f}"
)

print(
    "P99:",
    f"{percentile(token_lengths, .99):,.0f}"
)

print(
    "Max:",
    f"{max(token_lengths):,}"
)

print(
    "Mean:",
    f"{sum(token_lengths)/len(token_lengths):,.1f}"
)


# ------------------------------------------------------------
# CONTEXT WINDOW UTILIZATION
# ------------------------------------------------------------

CONTEXT = 4096

print()
print("=" * 100)
print("2. 4096-CONTEXT UTILIZATION")
print("=" * 100)

for threshold in [
    128,
    256,
    512,
    1024,
    2048,
    4096,
    8192,
]:

    count = sum(
        x >= threshold
        for x in token_lengths
    )

    print(
        f">= {threshold:5d} tokens:"
        f" {count:6,}"
        f" / {len(token_lengths):,}"
        f" ({100*count/len(token_lengths):6.2f}%)"
    )


# ------------------------------------------------------------
# GLOBAL TOKEN FREQUENCY
# ------------------------------------------------------------

print()
print("=" * 100)
print("3. VOCABULARY UTILIZATION")
print("=" * 100)

token_counts = Counter()

for ids in token_sequences:
    token_counts.update(ids)

total_tokens = sum(
    token_counts.values()
)

unique_tokens = len(token_counts)

print(
    "Total sampled tokens:",
    f"{total_tokens:,}"
)

print(
    "Unique token IDs used:",
    f"{unique_tokens:,}"
)

# Try to determine tokenizer vocabulary size.
vocab_size = None

try:

    if hasattr(tokenizer, "get_vocab"):
        vocab_size = len(tokenizer.get_vocab())

except:
    pass

if vocab_size is None:

    try:

        if hasattr(tokenizer, "vocab_size"):
            vocab_size = tokenizer.vocab_size

    except:
        pass

if vocab_size is not None:

    print(
        "Tokenizer vocabulary:",
        f"{vocab_size:,}"
    )

    print(
        "Vocabulary utilized:",
        f"{100*unique_tokens/vocab_size:.2f}%"
    )


# ------------------------------------------------------------
# TOKEN CONCENTRATION
# ------------------------------------------------------------

print()
print("=" * 100)
print("4. TOKEN FREQUENCY CONCENTRATION")
print("=" * 100)

for k in [
    10,
    50,
    100,
    500,
    1000,
    5000,
]:

    mass = sum(
        count
        for _, count
        in token_counts.most_common(k)
    )

    print(
        f"Top {k:5d} tokens:"
        f" {100*mass/total_tokens:7.3f}%"
        f" of all sampled tokens"
    )


# ------------------------------------------------------------
# TOP TOKENS
# ------------------------------------------------------------

print()
print("Top 50 token IDs:")

for rank, (tid, count) in enumerate(
    token_counts.most_common(50),
    1,
):

    print(
        f"{rank:3d}. "
        f"ID={tid:6d} "
        f"count={count:10,} "
        f"share={100*count/total_tokens:7.4f}%"
    )


# ------------------------------------------------------------
# DOCUMENT-LEVEL REPETITION
# ------------------------------------------------------------

print()
print("=" * 100)
print("5. DOCUMENT INTERNAL REPETITION")
print("=" * 100)

for n in [
    2,
    3,
    4,
    5,
    8,
]:

    total_ngrams = 0
    unique_ngrams = set()
    repeated_ngrams = 0
    max_frequency = 0

    # Bound each document to keep this diagnostic cheap.
    for ids in token_sequences:

        ids = ids[:20000]

        if len(ids) < n:
            continue

        local = Counter(
            tuple(ids[i:i+n])
            for i in range(
                len(ids) - n + 1
            )
        )

        total_ngrams += sum(local.values())

        unique_ngrams.update(local.keys())

        repeated_ngrams += sum(
            c > 1
            for c in local.values()
        )

        if local:
            max_frequency = max(
                max_frequency,
                max(local.values())
            )

    print(
        f"{n}-grams:"
        f" total={total_ngrams:,}"
        f" unique={len(unique_ngrams):,}"
        f" repeated-types={repeated_ngrams:,}"
        f" max-frequency={max_frequency:,}"
    )


# ------------------------------------------------------------
# IDENTICAL TOKEN RUNS
# ------------------------------------------------------------

print()
print("=" * 100)
print("6. IDENTICAL TOKEN RUNS")
print("=" * 100)

run_lengths = []

for ids in token_sequences:

    if not ids:
        continue

    current = 1

    for i in range(1, len(ids)):

        if ids[i] == ids[i-1]:

            current += 1

        else:

            if current >= 3:
                run_lengths.append(current)

            current = 1

    if current >= 3:
        run_lengths.append(current)


if run_lengths:

    print(
        "Runs >=3:",
        f"{len(run_lengths):,}"
    )

    print(
        "Longest:",
        max(run_lengths)
    )

    print(
        "P95:",
        percentile(run_lengths, .95)
    )

else:

    print(
        "No identical-token runs >=3."
    )


# ------------------------------------------------------------
# 4096-CHUNK REPETITION
# ------------------------------------------------------------

print()
print("=" * 100)
print("7. 4096-TOKEN CHUNK REPETITION")
print("=" * 100)

chunk_counts = Counter()

for ids in token_sequences:

    # Divide documents into context-sized chunks.
    for start in range(
        0,
        len(ids),
        CONTEXT,
    ):

        chunk = ids[
            start:start+CONTEXT
        ]

        if len(chunk) >= 256:

            # Hashable compact representation.
            chunk_counts[
                tuple(chunk)
            ] += 1


duplicate_chunks = sum(
    count > 1
    for count in chunk_counts.values()
)

max_chunk_frequency = (
    max(chunk_counts.values())
    if chunk_counts
    else 0
)

print(
    "Chunks examined:",
    f"{len(chunk_counts):,}"
)

print(
    "Duplicate chunk types:",
    f"{duplicate_chunks:,}"
)

print(
    "Maximum identical-chunk frequency:",
    max_chunk_frequency
)


# ------------------------------------------------------------
# FINAL INTERPRETATION HINTS
# ------------------------------------------------------------

print()
print("=" * 100)
print("8. WHAT WE ARE LOOKING FOR")
print("=" * 100)

print("""
A) Very high token concentration
   -> corpus/domain distribution may be highly concentrated.

B) Very low vocabulary utilization
   -> 32k may be substantially larger than the effective vocabulary,
      but this alone does NOT explain generation collapse.

C) Large repeated n-gram counts
   -> training data itself contains strong legal-template repetition.

D) High duplicate 4096-token chunks
   -> potentially serious data duplication / sampling issue.

E) Normal data diversity but catastrophic generation repetition
   -> investigate training optimization, number of effective tokens,
      learning rate, batch/gradient accumulation, initialization,
      and decoding strategy.

F) Very short effective training context
   -> the model may not be learning enough long-range continuation
      structure.

The next step is to combine these measurements with the STEP-1000
training log and actual training configuration.
""")

print()
print("=" * 100)
print("✓ DIAGNOSIS COMPLETE")
print("=" * 100)

In [ ]:
# ============================================================
# CS781 — PILOT RUN: LOAD VERIFIED STEP-1000 CHECKPOINT
# ============================================================

import os
import json
import math
import time
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

RECOVERY_ROOT = "/kaggle/input/datasets/oipranesh/cs781-phase4-final-recovery"
TRAINING_ROOT = "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
STEP1000_ROOT = "/kaggle/input/datasets/oipranesh/phase4-step1000-verified-model"

CHECKPOINT = os.path.join(STEP1000_ROOT, "checkpoint_step_001000.pt")
TOKENIZER_FILE = os.path.join(TRAINING_ROOT, "tokenizer", "legal_bpe_32k.json")

print("=" * 90)
print("CS781 — PILOT RUN SETUP")
print("=" * 90)

print("DEVICE:", DEVICE)
print("CHECKPOINT:", CHECKPOINT)
print("TOKENIZER:", TOKENIZER_FILE)

assert os.path.exists(CHECKPOINT)
assert os.path.exists(TOKENIZER_FILE)

# ------------------------------------------------------------
# IMPORTANT:
# The following assumes your exact verified model and tokenizer
# construction already exist in the notebook.
# ------------------------------------------------------------

required = ["model", "tokenizer"]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        f"Missing: {missing}\n\n"
        "Run the exact model/tokenizer construction cell from "
        "the verified notebook before this cell."
    )

# Reload checkpoint into the existing verified architecture.
ckpt = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

assert "model_state_dict" in ckpt

missing_keys, unexpected_keys = model.load_state_dict(
    ckpt["model_state_dict"],
    strict=False,
)

assert len(missing_keys) == 0, missing_keys
assert len(unexpected_keys) == 0, unexpected_keys

model = model.to(DEVICE)
model.train()

print()
print("Checkpoint step :", ckpt.get("step"))
print("Model device    :", next(model.parameters()).device)
print("Model parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

print()
print("✓ VERIFIED STEP-1000 WEIGHTS LOADED")
print("✓ NO ORIGINAL CHECKPOINT WILL BE MODIFIED")

In [ ]:
# ============================================================
# CS781 — PILOT DATA SAMPLE
# ============================================================

import glob
import gzip
import json
import random
import os

random.seed(781)

DOMAIN_DIRS = {
    "acts": os.path.join(TRAINING_ROOT, "dedup", "acts"),
    "high_courts": os.path.join(TRAINING_ROOT, "dedup", "high_courts"),
    "tribunals": os.path.join(TRAINING_ROOT, "dedup", "tribunals"),
    "supreme_court": os.path.join(TRAINING_ROOT, "dedup", "supreme_court"),
}

# Number of documents to use for the pilot.
# This is intentionally small so the pilot finishes quickly.
DOCS_PER_DOMAIN = 1500

pilot_texts = []

def extract_text(record):
    if not isinstance(record, dict):
        return None

    for key in ("text", "content", "document", "body"):
        value = record.get(key)
        if isinstance(value, str) and len(value.strip()) > 50:
            return value.strip()

    return None


for domain, root in DOMAIN_DIRS.items():

    if not os.path.isdir(root):
        print(f"SKIP {domain}: directory missing")
        continue

    files = sorted(
        glob.glob(os.path.join(root, "*.jsonl")) +
        glob.glob(os.path.join(root, "*.jsonl.gz"))
    )

    random.shuffle(files)

    collected = 0

    for path in files:

        opener = gzip.open if path.endswith(".gz") else open

        with opener(path, "rt", encoding="utf-8", errors="ignore") as f:

            for line in f:

                try:
                    record = json.loads(line)
                except Exception:
                    continue

                text = extract_text(record)

                if text is None:
                    continue

                pilot_texts.append(text)
                collected += 1

                if collected >= DOCS_PER_DOMAIN:
                    break

        if collected >= DOCS_PER_DOMAIN:
            break

    print(f"{domain:15s}: {collected:,} documents")


random.shuffle(pilot_texts)

print()
print("=" * 90)
print("PILOT CORPUS")
print("=" * 90)

print("Documents:", len(pilot_texts))

assert len(pilot_texts) >= 1000

In [ ]:
# ============================================================
# CS781 — B+D PILOT BATCH BUILDER
# ============================================================

# Pilot configuration
PILOT_CONTEXT = 1024

# Effective batch = MICRO_BATCH * GRAD_ACCUM
MICRO_BATCH = 4
GRAD_ACCUM = 8

EFFECTIVE_BATCH = MICRO_BATCH * GRAD_ACCUM

# Number of optimizer updates for the pilot.
# Keep this short.
PILOT_STEPS = 150

# Learning rate deliberately conservative relative to a
# fresh-from-scratch run because we START FROM STEP-1000.
PILOT_LR = 1.0e-4

WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0

print("=" * 90)
print("B+D PILOT CONFIGURATION")
print("=" * 90)

print("Context length       :", PILOT_CONTEXT)
print("Micro batch          :", MICRO_BATCH)
print("Gradient accumulation:", GRAD_ACCUM)
print("Effective batch      :", EFFECTIVE_BATCH)
print("Pilot optimizer steps:", PILOT_STEPS)
print("Learning rate        :", PILOT_LR)
print("Weight decay         :", WEIGHT_DECAY)
print("Grad clip            :", MAX_GRAD_NORM)

# ------------------------------------------------------------
# Tokenize once.
# ------------------------------------------------------------

def encode_text(text):
    """
    Supports the raw BPE tokenizer used by the verified notebook.
    """

    if hasattr(tokenizer, "encode"):
        result = tokenizer.encode(text)

        # HuggingFace tokenizers.Encoding
        if hasattr(result, "ids"):
            return result.ids

        # list[int]
        if isinstance(result, list):
            return result

    raise RuntimeError(
        "Tokenizer does not expose the expected encode() API."
    )


tokenized = []

for text in pilot_texts:

    ids = encode_text(text)

    if len(ids) >= PILOT_CONTEXT + 1:
        tokenized.append(ids)

print()
print("Documents tokenized :", len(tokenized))

assert len(tokenized) >= 500


# ------------------------------------------------------------
# Random contiguous chunks.
#
# This avoids training almost exclusively on the beginning
# of documents and gives the pilot more useful continuation
# examples.
# ------------------------------------------------------------

def sample_chunk():

    ids = random.choice(tokenized)

    if len(ids) <= PILOT_CONTEXT + 1:
        start = 0
    else:
        start = random.randint(
            0,
            len(ids) - PILOT_CONTEXT - 1
        )

    chunk = ids[start:start + PILOT_CONTEXT + 1]

    return torch.tensor(chunk, dtype=torch.long)


def make_batch():

    batch = torch.stack(
        [sample_chunk() for _ in range(MICRO_BATCH)]
    )

    x = batch[:, :-1]
    y = batch[:, 1:]

    return x.to(DEVICE), y.to(DEVICE)


print("✓ Random contiguous chunk sampler ready")
print("✓ B+D pilot batch construction ready")

In [ ]:
# ============================================================
# CS781 — SHORT B+D TRAINING PILOT
# ============================================================

# Fresh optimizer — we intentionally do NOT restore the old
# optimizer state for this experiment.
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=PILOT_LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.95),
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=PILOT_STEPS,
    eta_min=PILOT_LR * 0.1,
)

scaler = torch.cuda.amp.GradScaler(
    enabled=torch.cuda.is_available()
)

loss_history = []
grad_history = []

model.train()

start_time = time.time()

print("=" * 90)
print("CS781 — 150-STEP B+D PILOT")
print("=" * 90)

for step in range(1, PILOT_STEPS + 1):

    optimizer.zero_grad(set_to_none=True)

    accumulated_loss = 0.0

    for micro in range(GRAD_ACCUM):

        x, y = make_batch()

        with torch.cuda.amp.autocast(
            enabled=torch.cuda.is_available(),
            dtype=torch.float16,
        ):

            output = model(x)

            if hasattr(output, "logits"):
                logits = output.logits
            else:
                logits = output

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                y.reshape(-1),
            )

            loss = loss / GRAD_ACCUM

        scaler.scale(loss).backward()

        accumulated_loss += loss.item()

    scaler.unscale_(optimizer)

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        MAX_GRAD_NORM,
    )

    scaler.step(optimizer)
    scaler.update()
    scheduler.step()

    loss_value = accumulated_loss

    loss_history.append(loss_value)
    grad_history.append(float(grad_norm))

    if step == 1 or step % 10 == 0:

        elapsed = time.time() - start_time

        print(
            f"step {step:4d}/{PILOT_STEPS} | "
            f"loss={loss_value:.4f} | "
            f"grad={float(grad_norm):.3f} | "
            f"lr={scheduler.get_last_lr()[0]:.3e} | "
            f"time={elapsed/60:.1f}m"
        )


elapsed = time.time() - start_time

print()
print("=" * 90)
print("PILOT TRAINING COMPLETE")
print("=" * 90)

print(f"Elapsed time : {elapsed/60:.2f} minutes")
print(f"Initial loss : {loss_history[0]:.4f}")
print(f"Final loss   : {loss_history[-1]:.4f}")
print(f"Min loss     : {min(loss_history):.4f}")
print(f"Mean grad    : {np.mean(grad_history):.4f}")

In [ ]:
# ============================================================
# CS781 — SAVE B+D PILOT CHECKPOINT
# ============================================================

PILOT_DIR = "/kaggle/working/cs781_b_d_pilot"
os.makedirs(PILOT_DIR, exist_ok=True)

PILOT_CKPT = os.path.join(
    PILOT_DIR,
    "step1000_plus_bd_150steps.pt"
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "pilot_steps": PILOT_STEPS,
        "starting_checkpoint": CHECKPOINT,
        "context_length": PILOT_CONTEXT,
        "micro_batch": MICRO_BATCH,
        "gradient_accumulation": GRAD_ACCUM,
        "effective_batch": EFFECTIVE_BATCH,
        "learning_rate": PILOT_LR,
        "weight_decay": WEIGHT_DECAY,
        "max_grad_norm": MAX_GRAD_NORM,
        "loss_history": loss_history,
        "grad_history": grad_history,
    },
    PILOT_CKPT,
)

print("Saved:")
print(PILOT_CKPT)

In [ ]:
# ============================================================
# CS781 — 32-PROMPT DECODING DIAGNOSTIC
# ============================================================

# Use the validation texts recovered earlier if they are still
# in memory. Otherwise this cell will tell you what is missing.

if "validation_texts" not in globals():
    raise RuntimeError(
        "validation_texts is not in memory. "
        "Run the validation-text recovery cell first."
    )

assert len(validation_texts) >= 32

random.seed(781)

# Fixed selection — same prompts every time.
prompt_indices = random.sample(
    range(len(validation_texts)),
    32,
)

diagnostic_prompts = [
    validation_texts[i]
    for i in prompt_indices
]

print("=" * 90)
print("32-PROMPT DECODING SET")
print("=" * 90)

print("Prompts:", len(diagnostic_prompts))

for i, text in enumerate(diagnostic_prompts[:5], 1):

    ids = encode_text(text)

    print(
        f"{i:2d}: "
        f"{len(ids):5d} tokens | "
        f"{len(text):7d} chars"
    )

print("...")

In [ ]:
# ============================================================
# CS781 — GENERATION DIAGNOSTICS
# ============================================================

GEN_TOKENS = 128


def generate_greedy(text, model_to_use):

    ids = encode_text(text)

    # Keep the most recent context if necessary.
    ids = ids[-(PILOT_CONTEXT - 1):]

    input_ids = torch.tensor(
        [ids],
        dtype=torch.long,
        device=DEVICE,
    )

    generated = []

    model_to_use.eval()

    with torch.no_grad():

        for _ in range(GEN_TOKENS):

            logits_out = model_to_use(input_ids)

            if hasattr(logits_out, "logits"):
                logits = logits_out.logits
            else:
                logits = logits_out

            next_logits = logits[:, -1, :]

            probs = torch.softmax(
                next_logits,
                dim=-1,
            )

            next_id = torch.argmax(
                probs,
                dim=-1,
            )

            generated.append(
                int(next_id.item())
            )

            input_ids = torch.cat(
                [
                    input_ids,
                    next_id[:, None],
                ],
                dim=1,
            )

            if input_ids.shape[1] > PILOT_CONTEXT:
                input_ids = input_ids[:, -PILOT_CONTEXT:]

    return generated


def repetition_stats(ids):

    if len(ids) < 2:
        return {
            "unique_ratio": 0.0,
            "adjacent_repeat": 0.0,
            "max_run": 1,
            "distinct_2": 0.0,
            "distinct_3": 0.0,
        }

    unique_ratio = len(set(ids)) / len(ids)

    adjacent = sum(
        ids[i] == ids[i - 1]
        for i in range(1, len(ids))
    )

    max_run = 1
    current = 1

    for i in range(1, len(ids)):

        if ids[i] == ids[i - 1]:
            current += 1
            max_run = max(max_run, current)
        else:
            current = 1

    def distinct_n(n):

        if len(ids) < n:
            return 0.0

        grams = [
            tuple(ids[i:i+n])
            for i in range(len(ids) - n + 1)
        ]

        return len(set(grams)) / len(grams)

    return {
        "unique_ratio": unique_ratio,
        "adjacent_repeat": adjacent / (len(ids) - 1),
        "max_run": max_run,
        "distinct_2": distinct_n(2),
        "distinct_3": distinct_n(3),
    }


def decode_ids(ids):

    if hasattr(tokenizer, "decode"):
        return tokenizer.decode(ids)

    raise RuntimeError(
        "Tokenizer does not expose decode()."
    )


def evaluate_model_on_32(model_to_use):

    results = []

    for i, prompt in enumerate(diagnostic_prompts):

        generated = generate_greedy(
            prompt,
            model_to_use,
        )

        stats = repetition_stats(generated)

        results.append(
            {
                "index": i,
                "generated_ids": generated,
                "text": decode_ids(generated),
                **stats,
            }
        )

    return results


print("✓ Generation diagnostic functions ready")

In [ ]:
# ============================================================
# CS781 — EVALUATE B+D PILOT
# ============================================================

pilot_results = evaluate_model_on_32(model)

print("=" * 90)
print("B+D PILOT RESULTS")
print("=" * 90)

avg_unique = np.mean([
    x["unique_ratio"]
    for x in pilot_results
])

avg_adjacent = np.mean([
    x["adjacent_repeat"]
    for x in pilot_results
])

avg_d2 = np.mean([
    x["distinct_2"]
    for x in pilot_results
])

avg_d3 = np.mean([
    x["distinct_3"]
    for x in pilot_results
])

max_run = max(
    x["max_run"]
    for x in pilot_results
)

print(f"Average unique-token ratio : {avg_unique:.4f}")
print(f"Average adjacent repeat    : {avg_adjacent:.4f}")
print(f"Average distinct-2         : {avg_d2:.4f}")
print(f"Average distinct-3         : {avg_d3:.4f}")
print(f"Worst max identical run    : {max_run}")

print()
print("SAMPLE OUTPUTS")
print("=" * 90)

for i in range(min(5, len(pilot_results))):

    r = pilot_results[i]

    print()
    print("-" * 90)
    print(f"PROMPT {i+1}")
    print("-" * 90)
    print(r["text"][:1500])

In [ ]:
# =============================================================================
# CS781 — PRE-RUN TRAINING STATE CHECK
# =============================================================================

import os
import json
import torch

STEP1000_ROOT = "/kaggle/input/datasets/oipranesh/phase4-step1000-verified-model"

CKPT = os.path.join(
    STEP1000_ROOT,
    "checkpoint_step_001000.pt"
)

LOG = os.path.join(
    STEP1000_ROOT,
    "training_log.jsonl"
)

print("=" * 90)
print("CS781 — STEP-1000 TRAINING STATE INSPECTION")
print("=" * 90)

print("\nCHECKPOINT")
print("-" * 90)
print("Path:", CKPT)
print("Exists:", os.path.exists(CKPT))

ckpt = torch.load(CKPT, map_location="cpu")

print("Keys:")
for k in ckpt.keys():
    print(" ", k)

print("\nTRAINING METADATA")
print("-" * 90)

for k in [
    "phase",
    "step",
    "optimizer_steps",
    "context_length",
    "batch_size",
    "gradient_accumulation_steps",
    "learning_rate",
    "max_grad_norm",
    "mixed_precision",
    "gradient_checkpointing",
    "use_cache",
]:
    print(f"{k:30s}: {ckpt.get(k)}")

print("\nTRAINING LOG")
print("-" * 90)

if os.path.exists(LOG):
    rows = []
    with open(LOG, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    rows.append(json.loads(line))
                except:
                    pass

    print("Log records:", len(rows))

    if rows:
        print("\nFirst record:")
        print(json.dumps(rows[0], indent=2)[:4000])

        print("\nLast record:")
        print(json.dumps(rows[-1], indent=2)[:4000])
else:
    print("No training_log.jsonl found.")

print("\n" + "=" * 90)
print("CHECK COMPLETE")
print("=" * 90)

In [ ]:
# =============================================================================
# CS781 — B+D MICRO TRAINING DIAGNOSTIC
#
# PURPOSE:
#   Very short controlled training run.
#
#   We are NOT trying to improve the model yet.
#   We are locating the point at which the repetition collapse appears.
#
#   IMPORTANT:
#   This creates a NEW model in /kaggle/working.
#   The original STEP-1000 checkpoint is never modified.
# =============================================================================

import os
import math
import json
import time
import copy
import torch
import torch.nn.functional as F

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

assert "model" in globals(), (
    "The verified model must already be loaded in this notebook."
)

assert "tokenizer" in globals(), (
    "The verified tokenizer must already be loaded in this notebook."
)

# -------------------------------------------------------------------------
# SETTINGS
# -------------------------------------------------------------------------

MICRO_STEPS = 100

# Deliberately conservative LR for the diagnostic.
# We are testing stability, not maximizing training speed.
DIAG_LR = 1e-5

LOG_EVERY = 10

GEN_TOKENS = 64

OUT_DIR = "/kaggle/working/cs781_bd_micro_diagnostic"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------------------------------------------------------
# COPY VERIFIED MODEL
# -------------------------------------------------------------------------

diag_model = copy.deepcopy(model)
diag_model = diag_model.to(DEVICE)
diag_model.train()

print("=" * 90)
print("CS781 — B+D MICRO TRAINING DIAGNOSTIC")
print("=" * 90)

print("Device:", DEVICE)
print("Steps :", MICRO_STEPS)
print("LR    :", DIAG_LR)

# -------------------------------------------------------------------------
# OPTIMIZER
# -------------------------------------------------------------------------

optimizer = torch.optim.AdamW(
    diag_model.parameters(),
    lr=DIAG_LR,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=0.1,
)

# -------------------------------------------------------------------------
# LOAD A SMALL AMOUNT OF TRAINING DATA
# -------------------------------------------------------------------------

TRAIN_ROOT = "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"

# Prefer the existing training chunks because they are already prepared.
chunk_root = os.path.join(TRAIN_ROOT, "training_chunks")

files = []

if os.path.exists(chunk_root):
    for root, dirs, fs in os.walk(chunk_root):
        for fn in fs:
            if fn.endswith(".jsonl"):
                files.append(os.path.join(root, fn))

if not files:
    raise RuntimeError("No training JSONL chunks found.")

print("\nTraining files found:", len(files))

# -------------------------------------------------------------------------
# EXTRACT TEXT
# -------------------------------------------------------------------------

texts = []

for path in files[:20]:

    try:
        with open(path, "r", encoding="utf-8") as f:

            for line in f:

                try:
                    obj = json.loads(line)
                except:
                    continue

                text = None

                if isinstance(obj, dict):

                    for key in ["text", "content", "document"]:
                        if key in obj and isinstance(obj[key], str):
                            text = obj[key]
                            break

                elif isinstance(obj, str):
                    text = obj

                if text and len(text) > 200:
                    texts.append(text)

                if len(texts) >= 500:
                    break

        if len(texts) >= 500:
            break

    except Exception:
        continue

if not texts:
    raise RuntimeError("Could not extract training texts.")

print("Training texts:", len(texts))

# -------------------------------------------------------------------------
# TOKENIZE
# -------------------------------------------------------------------------

def encode_text(text):

    # Handle raw tokenizers.Tokenizer
    if hasattr(tokenizer, "encode"):

        enc = tokenizer.encode(text)

        if hasattr(enc, "ids"):
            return enc.ids

        if isinstance(enc, list):
            return enc

    raise RuntimeError("Could not encode text with tokenizer.")

token_sequences = []

for text in texts:

    ids = encode_text(text)

    if len(ids) >= 64:
        token_sequences.append(ids)

print("Usable token sequences:", len(token_sequences))

if not token_sequences:
    raise RuntimeError("No usable token sequences.")

# -------------------------------------------------------------------------
# RANDOM FIXED-LENGTH TRAINING BATCHES
# -------------------------------------------------------------------------

SEQ_LEN = min(
    512,
    int(getattr(diag_model.config, "n_positions", 4096))
)

BATCH_SIZE = 2

print("Sequence length:", SEQ_LEN)
print("Batch size:", BATCH_SIZE)

def make_batch():

    xs = []

    for _ in range(BATCH_SIZE):

        ids = token_sequences[
            torch.randint(
                0,
                len(token_sequences),
                (1,)
            ).item()
        ]

        if len(ids) <= SEQ_LEN + 1:
            start = 0
        else:
            start = torch.randint(
                0,
                len(ids) - SEQ_LEN,
                (1,)
            ).item()

        chunk = ids[start:start + SEQ_LEN + 1]

        xs.append(chunk)

    x = torch.tensor(
        [z[:-1] for z in xs],
        dtype=torch.long,
        device=DEVICE
    )

    y = torch.tensor(
        [z[1:] for z in xs],
        dtype=torch.long,
        device=DEVICE
    )

    return x, y

# -------------------------------------------------------------------------
# GENERATION DIAGNOSTIC
# -------------------------------------------------------------------------

@torch.no_grad()
def generate_sample(prompt, max_new_tokens=64):

    diag_model.eval()

    ids = encode_text(prompt)

    ids = ids[-min(
        len(ids),
        int(getattr(diag_model.config, "n_positions", 4096)) - max_new_tokens
    ):]

    x = torch.tensor(
        [ids],
        dtype=torch.long,
        device=DEVICE
    )

    for _ in range(max_new_tokens):

        out = diag_model(x)

        logits = out.logits[:, -1, :]

        # Greedy generation deliberately used here.
        # We want to measure model collapse, not decoding randomness.
        next_id = torch.argmax(logits, dim=-1, keepdim=True)

        x = torch.cat([x, next_id], dim=1)

        if x.shape[1] >= getattr(diag_model.config, "n_positions", 4096):
            break

    generated = x[0, len(ids):].tolist()

    return generated

# -------------------------------------------------------------------------
# REPETITION METRICS
# -------------------------------------------------------------------------

def repetition_metrics(ids):

    if not ids:
        return {
            "unique_ratio": 0,
            "adjacent_repeat": 1,
            "max_run": 0,
            "distinct2": 0,
            "distinct3": 0,
        }

    unique_ratio = len(set(ids)) / len(ids)

    adjacent = sum(
        ids[i] == ids[i-1]
        for i in range(1, len(ids))
    )

    adjacent_repeat = (
        adjacent / max(1, len(ids) - 1)
    )

    max_run = 1
    current = 1

    for i in range(1, len(ids)):

        if ids[i] == ids[i-1]:
            current += 1
            max_run = max(max_run, current)
        else:
            current = 1

    def distinct_n(n):

        if len(ids) < n:
            return 0

        grams = [
            tuple(ids[i:i+n])
            for i in range(len(ids)-n+1)
        ]

        return len(set(grams)) / len(grams)

    return {
        "unique_ratio": unique_ratio,
        "adjacent_repeat": adjacent_repeat,
        "max_run": max_run,
        "distinct2": distinct_n(2),
        "distinct3": distinct_n(3),
    }

# -------------------------------------------------------------------------
# FIXED PROMPTS
# -------------------------------------------------------------------------

prompts = [
    texts[0][:2000],
    texts[min(1, len(texts)-1)][:2000],
    texts[min(2, len(texts)-1)][:2000],
]

# -------------------------------------------------------------------------
# BASELINE BEFORE TRAINING
# -------------------------------------------------------------------------

print("\n" + "=" * 90)
print("BASELINE BEFORE MICRO-TRAINING")
print("=" * 90)

diag_model.eval()

baseline_metrics = []

for i, prompt in enumerate(prompts):

    generated = generate_sample(
        prompt,
        GEN_TOKENS
    )

    m = repetition_metrics(generated)

    baseline_metrics.append(m)

    print(
        f"Prompt {i+1}: "
        f"unique={m['unique_ratio']:.4f} "
        f"adj_repeat={m['adjacent_repeat']:.4f} "
        f"distinct2={m['distinct2']:.4f} "
        f"max_run={m['max_run']}"
    )

# -------------------------------------------------------------------------
# MICRO TRAINING
# -------------------------------------------------------------------------

print("\n" + "=" * 90)
print("MICRO-TRAINING")
print("=" * 90)

history = []

for step in range(1, MICRO_STEPS + 1):

    diag_model.train()

    x, y = make_batch()

    optimizer.zero_grad(set_to_none=True)

    out = diag_model(
        input_ids=x
    )

    logits = out.logits

    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        y.reshape(-1)
    )

    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        diag_model.parameters(),
        max_norm=1.0
    )

    optimizer.step()

    record = {
        "step": step,
        "loss": float(loss.item()),
        "grad_norm": float(grad_norm),
    }

    history.append(record)

    if step == 1 or step % LOG_EVERY == 0:

        print(
            f"step={step:4d} "
            f"loss={loss.item():.4f} "
            f"grad_norm={float(grad_norm):.4f}"
        )

# -------------------------------------------------------------------------
# AFTER TRAINING
# -------------------------------------------------------------------------

print("\n" + "=" * 90)
print("AFTER MICRO-TRAINING")
print("=" * 90)

after_metrics = []

for i, prompt in enumerate(prompts):

    generated = generate_sample(
        prompt,
        GEN_TOKENS
    )

    m = repetition_metrics(generated)

    after_metrics.append(m)

    print(
        f"Prompt {i+1}: "
        f"unique={m['unique_ratio']:.4f} "
        f"adj_repeat={m['adjacent_repeat']:.4f} "
        f"distinct2={m['distinct2']:.4f} "
        f"max_run={m['max_run']}"
    )

    print("IDs:", generated[:30])

# -------------------------------------------------------------------------
# SAVE DIAGNOSTIC
# -------------------------------------------------------------------------

result = {
    "learning_rate": DIAG_LR,
    "steps": MICRO_STEPS,
    "seq_len": SEQ_LEN,
    "batch_size": BATCH_SIZE,
    "baseline": baseline_metrics,
    "after": after_metrics,
    "training_history": history,
}

out_file = os.path.join(
    OUT_DIR,
    "micro_diagnostic.json"
)

with open(out_file, "w") as f:
    json.dump(result, f, indent=2)

print("\n" + "=" * 90)
print("MICRO DIAGNOSTIC COMPLETE")
print("=" * 90)

print("Saved:", out_file)

In [ ]:
# =============================================================================
# CS781 — B+D MICRO TRAINING DIAGNOSTIC — CORRECTED DATA LOADER
# =============================================================================

import os
import json
import copy
import torch
import torch.nn.functional as F

assert "model" in globals(), "Verified model is not loaded."
assert "tokenizer" in globals(), "Verified tokenizer is not loaded."

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

TRAIN_ROOT = "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
DEDUP_ROOT = os.path.join(TRAIN_ROOT, "dedup")

MICRO_STEPS = 100
DIAG_LR = 1e-5
SEQ_LEN = 512
BATCH_SIZE = 2
GEN_TOKENS = 64
LOG_EVERY = 10

print("=" * 90)
print("CS781 — B+D MICRO TRAINING DIAGNOSTIC")
print("=" * 90)

print("Device :", DEVICE)
print("Steps  :", MICRO_STEPS)
print("LR     :", DIAG_LR)
print("SeqLen :", SEQ_LEN)
print("Batch  :", BATCH_SIZE)

# =============================================================================
# TOKENIZER
# =============================================================================

def encode_text(text):
    enc = tokenizer.encode(text)

    if hasattr(enc, "ids"):
        return enc.ids

    if isinstance(enc, list):
        return enc

    raise RuntimeError("Tokenizer.encode() returned unsupported object.")

# =============================================================================
# ROBUST TEXT EXTRACTION
# =============================================================================

def extract_text(obj):

    if isinstance(obj, str):
        return obj

    if not isinstance(obj, dict):
        return None

    # Direct candidates
    for key in [
        "text",
        "content",
        "document",
        "body",
        "raw_text",
        "case_text",
    ]:
        value = obj.get(key)

        if isinstance(value, str) and len(value) > 100:
            return value

    # Nested candidates
    for key in [
        "data",
        "record",
        "document",
        "source",
    ]:
        value = obj.get(key)

        if isinstance(value, dict):
            result = extract_text(value)
            if result:
                return result

    return None

# =============================================================================
# FIND REAL TRAINING FILES
# =============================================================================

jsonl_files = []

for root, dirs, files in os.walk(DEDUP_ROOT):

    for fn in files:

        if fn.endswith(".jsonl"):
            jsonl_files.append(
                os.path.join(root, fn)
            )

jsonl_files.sort()

print("\nTraining JSONL files:", len(jsonl_files))

if not jsonl_files:
    raise RuntimeError(
        f"No JSONL files found under {DEDUP_ROOT}"
    )

for p in jsonl_files[:10]:
    print(" ", p)

# =============================================================================
# LOAD A SMALL REPRESENTATIVE SAMPLE
# =============================================================================

texts = []

for path in jsonl_files:

    print("\nScanning:", path)

    try:

        with open(path, "r", encoding="utf-8") as f:

            for line_no, line in enumerate(f):

                line = line.strip()

                if not line:
                    continue

                try:
                    obj = json.loads(line)
                except Exception:
                    continue

                text = extract_text(obj)

                if text and len(text) >= 200:
                    texts.append(text)

                if len(texts) >= 1000:
                    break

        if len(texts) >= 1000:
            break

    except Exception as e:

        print("WARNING:", repr(e))

print("\nTexts recovered:", len(texts))

if not texts:
    raise RuntimeError(
        "Still could not extract training text. "
        "Print one raw JSONL record next."
    )

print("\nFirst training text:")
print(texts[0][:1000])

# =============================================================================
# TOKENIZE
# =============================================================================

token_sequences = []

for text in texts:

    try:
        ids = encode_text(text)
    except Exception:
        continue

    if len(ids) >= SEQ_LEN + 1:
        token_sequences.append(ids)

print("\nUsable token sequences:", len(token_sequences))

if len(token_sequences) < 10:
    raise RuntimeError(
        f"Only {len(token_sequences)} usable sequences found."
    )

# =============================================================================
# COPY MODEL
# =============================================================================

diag_model = copy.deepcopy(model)
diag_model = diag_model.to(DEVICE)
diag_model.train()

optimizer = torch.optim.AdamW(
    diag_model.parameters(),
    lr=DIAG_LR,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=0.1,
)

# =============================================================================
# BATCH CREATION
# =============================================================================

def make_batch():

    xs = []

    for _ in range(BATCH_SIZE):

        ids = token_sequences[
            torch.randint(
                0,
                len(token_sequences),
                (1,)
            ).item()
        ]

        start = torch.randint(
            0,
            len(ids) - SEQ_LEN,
            (1,)
        ).item()

        chunk = ids[
            start:start + SEQ_LEN + 1
        ]

        xs.append(chunk)

    x = torch.tensor(
        [z[:-1] for z in xs],
        dtype=torch.long,
        device=DEVICE,
    )

    y = torch.tensor(
        [z[1:] for z in xs],
        dtype=torch.long,
        device=DEVICE,
    )

    return x, y

# =============================================================================
# GENERATION
# =============================================================================

@torch.no_grad()
def generate_sample(prompt, max_new_tokens=64):

    diag_model.eval()

    ids = encode_text(prompt)

    max_context = int(
        getattr(
            diag_model.config,
            "n_positions",
            4096
        )
    )

    ids = ids[-(max_context - max_new_tokens):]

    x = torch.tensor(
        [ids],
        dtype=torch.long,
        device=DEVICE,
    )

    for _ in range(max_new_tokens):

        out = diag_model(input_ids=x)

        logits = out.logits[:, -1, :]

        next_id = torch.argmax(
            logits,
            dim=-1,
            keepdim=True
        )

        x = torch.cat(
            [x, next_id],
            dim=1
        )

    return x[0, len(ids):].tolist()

# =============================================================================
# REPETITION METRICS
# =============================================================================

def metrics(ids):

    n = len(ids)

    if n == 0:
        return {}

    unique_ratio = len(set(ids)) / n

    adjacent = sum(
        ids[i] == ids[i - 1]
        for i in range(1, n)
    )

    max_run = 1
    run = 1

    for i in range(1, n):

        if ids[i] == ids[i - 1]:
            run += 1
            max_run = max(max_run, run)

        else:
            run = 1

    def distinct_n(k):

        if n < k:
            return 0

        grams = [
            tuple(ids[i:i+k])
            for i in range(n-k+1)
        ]

        return len(set(grams)) / len(grams)

    return {
        "unique_ratio": unique_ratio,
        "adjacent_repeat": adjacent / max(1, n - 1),
        "distinct2": distinct_n(2),
        "distinct3": distinct_n(3),
        "max_run": max_run,
    }

# =============================================================================
# FIXED PROMPTS
# =============================================================================

prompts = [
    texts[0],
    texts[len(texts)//2],
    texts[-1],
]

# =============================================================================
# BASELINE
# =============================================================================

print("\n" + "=" * 90)
print("BASELINE — BEFORE TRAINING")
print("=" * 90)

baseline = []

for i, prompt in enumerate(prompts):

    ids = generate_sample(
        prompt,
        GEN_TOKENS
    )

    m = metrics(ids)
    baseline.append(m)

    print(
        f"Prompt {i+1}: "
        f"unique={m['unique_ratio']:.4f} | "
        f"adj={m['adjacent_repeat']:.4f} | "
        f"d2={m['distinct2']:.4f} | "
        f"d3={m['distinct3']:.4f} | "
        f"maxrun={m['max_run']}"
    )

# =============================================================================
# TRAIN
# =============================================================================

print("\n" + "=" * 90)
print("100-STEP MICRO TRAINING")
print("=" * 90)

history = []

for step in range(1, MICRO_STEPS + 1):

    diag_model.train()

    x, y = make_batch()

    optimizer.zero_grad(set_to_none=True)

    out = diag_model(input_ids=x)

    loss = F.cross_entropy(
        out.logits.reshape(-1, out.logits.size(-1)),
        y.reshape(-1),
    )

    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        diag_model.parameters(),
        max_norm=1.0,
    )

    optimizer.step()

    history.append({
        "step": step,
        "loss": float(loss.item()),
        "grad_norm": float(grad_norm),
    })

    if step == 1 or step % LOG_EVERY == 0:

        print(
            f"step={step:3d} | "
            f"loss={loss.item():.4f} | "
            f"grad={float(grad_norm):.4f}"
        )

# =============================================================================
# AFTER
# =============================================================================

print("\n" + "=" * 90)
print("AFTER 100 STEPS")
print("=" * 90)

after = []

for i, prompt in enumerate(prompts):

    ids = generate_sample(
        prompt,
        GEN_TOKENS
    )

    m = metrics(ids)
    after.append(m)

    print(
        f"Prompt {i+1}: "
        f"unique={m['unique_ratio']:.4f} | "
        f"adj={m['adjacent_repeat']:.4f} | "
        f"d2={m['distinct2']:.4f} | "
        f"d3={m['distinct3']:.4f} | "
        f"maxrun={m['max_run']}"
    )

    print("IDs:", ids[:30])

# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 90)
print("B+D MICRO DIAGNOSTIC SUMMARY")
print("=" * 90)

avg_base_adj = sum(x["adjacent_repeat"] for x in baseline) / len(baseline)
avg_after_adj = sum(x["adjacent_repeat"] for x in after) / len(after)

avg_base_unique = sum(x["unique_ratio"] for x in baseline) / len(baseline)
avg_after_unique = sum(x["unique_ratio"] for x in after) / len(after)

print(f"Baseline unique ratio : {avg_base_unique:.4f}")
print(f"After unique ratio    : {avg_after_unique:.4f}")

print(f"Baseline adj repeat   : {avg_base_adj:.4f}")
print(f"After adj repeat      : {avg_after_adj:.4f}")

print("\nFinal loss:", history[-1]["loss"])
print("Final grad norm:", history[-1]["grad_norm"])

print("\n" + "=" * 90)
print("IMPORTANT")
print("=" * 90)
print(
    "Do NOT launch the full training run yet. "
    "Use these results to decide whether the problem is "
    "pre-existing checkpoint collapse or training instability."
)

In [ ]:
# ==========================================================================================
# CS781 — STEP-1000 CONTROLLED A/B/C/D MICRO EXPERIMENT
# ==========================================================================================
# A = same LR as Step-1000
# B = reduced LR
# C = A's checkpoints evaluated with lower decoding temperature
# D = same LR + tighter gradient clipping
#
# 100 optimizer steps per training branch.
# Evaluation every 25 steps.
#
# IMPORTANT:
#   - Does NOT modify the Step-1000 checkpoint on disk.
#   - Does NOT modify optimizer state in the checkpoint.
#   - Uses independent model/optimizer copies.
# ==========================================================================================

import os
import json
import math
import copy
import random
import time
import re
from collections import Counter

import torch
import torch.nn.functional as F

print("=" * 90)
print("CS781 — STEP-1000 CONTROLLED A/B/C/D MICRO EXPERIMENT")
print("=" * 90)

# ------------------------------------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------------------------------------

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

CHECKPOINT = (
    "/kaggle/input/datasets/oipranesh/"
    "phase4-step1000-verified-model/checkpoint_step_001000.pt"
)

TRAIN_ROOT = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep/dedup"
)

STEPS = 100
EVAL_EVERY = 25

SEQ_LEN = 512
BATCH_SIZE = 2

BASE_LR = 1e-5
LOW_LR = 3e-6

BASE_CLIP = 1.0
TIGHT_CLIP = 0.25

# C is decoding-only:
TEMPERATURE_A = 1.0
TEMPERATURE_C = 0.70

EVAL_PROMPTS = 5
GEN_TOKENS = 128

SEED = 781

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device      : {DEVICE}")
print(f"Checkpoint  : {CHECKPOINT}")
print(f"Steps       : {STEPS}")
print(f"SeqLen      : {SEQ_LEN}")
print(f"Batch       : {BATCH_SIZE}")
print()


# ==========================================================================================
# 1. REQUIRE EXISTING VERIFIED MODEL + TOKENIZER
# ==========================================================================================

if "model" not in globals():
    raise RuntimeError(
        "The verified model is not loaded in this notebook. "
        "Run the exact model-construction/loading cells from the verified notebook first."
    )

if "tokenizer" not in globals():
    raise RuntimeError(
        "The verified tokenizer is not loaded in this notebook. "
        "Run the exact tokenizer-loading cells from the verified notebook first."
    )

print("✓ Existing model found")
print("✓ Existing tokenizer found")


# ==========================================================================================
# 2. LOAD STEP-1000 STATE
# ==========================================================================================

print("\n" + "=" * 90)
print("LOADING STEP-1000 CHECKPOINT")
print("=" * 90)

ckpt = torch.load(CHECKPOINT, map_location="cpu")

if "model_state_dict" not in ckpt:
    raise RuntimeError("Checkpoint does not contain model_state_dict.")

step1000_state = ckpt["model_state_dict"]

print("Checkpoint step :", ckpt.get("step"))
print("Checkpoint LR   :", ckpt.get("learning_rate"))
print("Context length  :", ckpt.get("context_length"))

# Save current model state so this diagnostic can restore it afterward.
original_model_state = copy.deepcopy(model.state_dict())

# Force verified Step-1000 weights.
model.load_state_dict(step1000_state, strict=True)
model.to(DEVICE)
model.eval()

print("✓ Exact Step-1000 weights loaded")


# ==========================================================================================
# 3. FIND TRAINING TEXT
# ==========================================================================================

print("\n" + "=" * 90)
print("LOADING TRAINING TEXT")
print("=" * 90)

jsonl_files = []

for root, dirs, files in os.walk(TRAIN_ROOT):
    for fn in files:
        if fn.endswith(".jsonl"):
            jsonl_files.append(os.path.join(root, fn))

jsonl_files.sort()

if not jsonl_files:
    raise RuntimeError("No JSONL training files found.")

print("Training JSONL files:", len(jsonl_files))

def extract_text(obj):
    if isinstance(obj, str):
        return obj.strip()

    if isinstance(obj, dict):
        # Prefer common text fields.
        for k in ("text", "content", "document", "body"):
            v = obj.get(k)
            if isinstance(v, str) and v.strip():
                return v.strip()

        # Recursive fallback.
        for v in obj.values():
            if isinstance(v, (dict, list)):
                t = extract_text(v)
                if t:
                    return t

    elif isinstance(obj, list):
        for v in obj:
            t = extract_text(v)
            if t:
                return t

    return None


texts = []

# Use the first useful shard/files and collect enough examples.
for path in jsonl_files:
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                except Exception:
                    continue

                text = extract_text(obj)

                if text and len(text) >= 300:
                    texts.append(text)

                if len(texts) >= 1500:
                    break

        if len(texts) >= 1500:
            break

    except Exception as e:
        print("Skipping:", path, "|", e)

print("Training texts recovered:", len(texts))

if len(texts) < 100:
    raise RuntimeError("Too few usable training texts recovered.")


# ==========================================================================================
# 4. TOKENIZATION HELPERS
# ==========================================================================================

def encode_text(text):
    """
    Supports the verified raw tokenizers.Tokenizer object as well as
    HuggingFace-style tokenizers.
    """
    if hasattr(tokenizer, "encode"):
        try:
            # Raw tokenizers.Tokenizer
            enc = tokenizer.encode(text)

            if hasattr(enc, "ids"):
                return list(enc.ids)

            if isinstance(enc, list):
                return list(enc)
        except Exception:
            pass

    # HF-style tokenizer fallback.
    enc = tokenizer(
        text,
        add_special_tokens=True,
        truncation=False,
        return_attention_mask=False,
    )

    return list(enc["input_ids"])


token_sequences = []

for text in texts:
    try:
        ids = encode_text(text)

        if len(ids) >= SEQ_LEN + 1:
            token_sequences.append(ids)

    except Exception:
        continue

print("Usable token sequences:", len(token_sequences))

if len(token_sequences) < 50:
    raise RuntimeError(
        "Not enough token sequences >= SEQ_LEN+1 for the diagnostic."
    )


# ==========================================================================================
# 5. FIXED MICRO-BATCHES
# ==========================================================================================
# Same batches are reused by all branches.
# This makes A/B/D substantially more comparable.

random.seed(SEED)

random.shuffle(token_sequences)

training_pool = token_sequences[: min(len(token_sequences), 1000)]

def make_batch():
    batch = []

    for _ in range(BATCH_SIZE):
        ids = random.choice(training_pool)

        if len(ids) <= SEQ_LEN + 1:
            raise RuntimeError("Unexpected short sequence.")

        start = random.randint(0, len(ids) - (SEQ_LEN + 1))

        chunk = ids[start:start + SEQ_LEN + 1]

        batch.append(chunk)

    x = torch.tensor(
        [a[:-1] for a in batch],
        dtype=torch.long,
        device=DEVICE,
    )

    y = torch.tensor(
        [a[1:] for a in batch],
        dtype=torch.long,
        device=DEVICE,
    )

    return x, y


# Pre-generate the exact batches so every branch sees identical data.
random.seed(SEED)

fixed_batches = [
    make_batch()
    for _ in range(STEPS)
]

print("Fixed batches:", len(fixed_batches))
print("Batch shape   :", tuple(fixed_batches[0][0].shape))


# ==========================================================================================
# 6. GENERATION / METRIC HELPERS
# ==========================================================================================

def decode_ids(ids):
    ids = list(map(int, ids))

    # Raw tokenizers.Tokenizer
    if hasattr(tokenizer, "decode"):
        try:
            return tokenizer.decode(ids)
        except Exception:
            pass

    try:
        return tokenizer.decode(
            ids,
            skip_special_tokens=False,
        )
    except Exception:
        return str(ids)


def encode_prompt(text):
    ids = encode_text(text)

    if len(ids) > SEQ_LEN:
        ids = ids[-SEQ_LEN:]

    return ids


def generate_ids(prompt_ids, temperature=1.0):
    model.eval()

    x = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=DEVICE,
    )

    generated = list(prompt_ids)

    with torch.no_grad():
        for _ in range(GEN_TOKENS):

            inp = torch.tensor(
                [generated[-SEQ_LEN:]],
                dtype=torch.long,
                device=DEVICE,
            )

            out = model(inp)

            if hasattr(out, "logits"):
                logits = out.logits[:, -1, :]
            elif isinstance(out, tuple):
                logits = out[0][:, -1, :]
            else:
                logits = out[:, -1, :]

            logits = logits.float() / max(float(temperature), 1e-6)

            probs = F.softmax(logits, dim=-1)

            next_id = torch.multinomial(
                probs,
                num_samples=1,
            ).item()

            generated.append(next_id)

    return generated[len(prompt_ids):]


def generation_metrics(ids):
    ids = list(map(int, ids))

    if not ids:
        return {
            "unique_ratio": 0.0,
            "adj_repeat": 0.0,
            "distinct2": 0.0,
            "distinct3": 0.0,
            "maxrun": 0,
        }

    n = len(ids)

    unique_ratio = len(set(ids)) / n

    adjacent_repeats = sum(
        1 for i in range(1, n)
        if ids[i] == ids[i - 1]
    )

    adj_repeat = adjacent_repeats / max(n - 1, 1)

    bigrams = list(zip(ids[:-1], ids[1:]))

    trigrams = list(
        zip(ids[:-2], ids[1:-1], ids[2:])
    )

    distinct2 = (
        len(set(bigrams)) / len(bigrams)
        if bigrams else 0.0
    )

    distinct3 = (
        len(set(trigrams)) / len(trigrams)
        if trigrams else 0.0
    )

    maxrun = 1
    current = 1

    for i in range(1, n):
        if ids[i] == ids[i - 1]:
            current += 1
            maxrun = max(maxrun, current)
        else:
            current = 1

    return {
        "unique_ratio": unique_ratio,
        "adj_repeat": adj_repeat,
        "distinct2": distinct2,
        "distinct3": distinct3,
        "maxrun": maxrun,
    }


# Use the same prompts for every evaluation.
eval_prompts = []

for ids in token_sequences[:EVAL_PROMPTS]:
    eval_prompts.append(ids[: min(128, len(ids))])

print("Evaluation prompts:", len(eval_prompts))


# ==========================================================================================
# 7. TRAINING BRANCH
# ==========================================================================================

def run_training_branch(
    branch_name,
    lr,
    clip_value,
):
    print("\n" + "=" * 90)
    print(f"BRANCH {branch_name}")
    print("=" * 90)

    # Restore exact Step-1000 weights.
    model.load_state_dict(step1000_state, strict=True)
    model.to(DEVICE)
    model.train()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        betas=(0.9, 0.95),
        weight_decay=0.01,
    )

    history = []

    for step in range(1, STEPS + 1):

        x, y = fixed_batches[step - 1]

        optimizer.zero_grad(set_to_none=True)

        out = model(x)

        if hasattr(out, "logits"):
            logits = out.logits
        elif isinstance(out, tuple):
            logits = out[0]
        else:
            logits = out

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            y.reshape(-1),
        )

        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=clip_value,
        )

        optimizer.step()

        record = {
            "step": step,
            "loss": float(loss.detach().cpu()),
            "grad_norm": float(grad_norm),
        }

        history.append(record)

        if step % EVAL_EVERY == 0 or step == 1:
            model.eval()

            # C is evaluated separately later.
            generated_metrics = []

            for prompt in eval_prompts:
                ids = generate_ids(
                    prompt,
                    temperature=TEMPERATURE_A,
                )

                generated_metrics.append(
                    generation_metrics(ids)
                )

            avg_unique = sum(
                m["unique_ratio"]
                for m in generated_metrics
            ) / len(generated_metrics)

            avg_adj = sum(
                m["adj_repeat"]
                for m in generated_metrics
            ) / len(generated_metrics)

            avg_d2 = sum(
                m["distinct2"]
                for m in generated_metrics
            ) / len(generated_metrics)

            avg_d3 = sum(
                m["distinct3"]
                for m in generated_metrics
            ) / len(generated_metrics)

            worst_run = max(
                m["maxrun"]
                for m in generated_metrics
            )

            record.update({
                "unique_ratio": avg_unique,
                "adj_repeat": avg_adj,
                "distinct2": avg_d2,
                "distinct3": avg_d3,
                "worst_maxrun": worst_run,
            })

            print(
                f"[{branch_name}] "
                f"step={step:3d} "
                f"loss={record['loss']:.4f} "
                f"grad={record['grad_norm']:.3f} "
                f"unique={avg_unique:.4f} "
                f"adj={avg_adj:.4f} "
                f"d2={avg_d2:.4f} "
                f"d3={avg_d3:.4f} "
                f"maxrun={worst_run}"
            )

            model.train()

    return history


# ==========================================================================================
# 8. RUN A
# ==========================================================================================

start_total = time.time()

history_A = run_training_branch(
    branch_name="A — BASE LR",
    lr=BASE_LR,
    clip_value=BASE_CLIP,
)


# ==========================================================================================
# 9. RUN B
# ==========================================================================================

history_B = run_training_branch(
    branch_name="B — LOW LR",
    lr=LOW_LR,
    clip_value=BASE_CLIP,
)


# ==========================================================================================
# 10. RUN D
# ==========================================================================================

history_D = run_training_branch(
    branch_name="D — TIGHT CLIP",
    lr=BASE_LR,
    clip_value=TIGHT_CLIP,
)


# ==========================================================================================
# 11. C = DECODING TEST USING A'S FINAL WEIGHTS
# ==========================================================================================

print("\n" + "=" * 90)
print("BRANCH C — LOWER TEMPERATURE DECODING")
print("=" * 90)

# Restore A's final weights by replaying A from Step-1000.
# This avoids storing another ~1.2 GB state dict.

model.load_state_dict(step1000_state, strict=True)
model.to(DEVICE)
model.train()

optimizer_A = torch.optim.AdamW(
    model.parameters(),
    lr=BASE_LR,
    betas=(0.9, 0.95),
    weight_decay=0.01,
)

for step in range(STEPS):

    x, y = fixed_batches[step]

    optimizer_A.zero_grad(set_to_none=True)

    out = model(x)

    if hasattr(out, "logits"):
        logits = out.logits
    elif isinstance(out, tuple):
        logits = out[0]
    else:
        logits = out

    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        y.reshape(-1),
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=BASE_CLIP,
    )

    optimizer_A.step()

model.eval()

metrics_C = []

for prompt in eval_prompts:

    ids = generate_ids(
        prompt,
        temperature=TEMPERATURE_C,
    )

    metrics_C.append(
        generation_metrics(ids)
    )

avg_C = {
    "unique_ratio": sum(m["unique_ratio"] for m in metrics_C) / len(metrics_C),
    "adj_repeat": sum(m["adj_repeat"] for m in metrics_C) / len(metrics_C),
    "distinct2": sum(m["distinct2"] for m in metrics_C) / len(metrics_C),
    "distinct3": sum(m["distinct3"] for m in metrics_C) / len(metrics_C),
    "worst_maxrun": max(m["maxrun"] for m in metrics_C),
}

print(
    f"C temperature={TEMPERATURE_C} | "
    f"unique={avg_C['unique_ratio']:.4f} | "
    f"adj={avg_C['adj_repeat']:.4f} | "
    f"d2={avg_C['distinct2']:.4f} | "
    f"d3={avg_C['distinct3']:.4f} | "
    f"maxrun={avg_C['worst_maxrun']}"
)


# ==========================================================================================
# 12. SUMMARY
# ==========================================================================================

def final_record(history):
    evals = [
        h for h in history
        if "unique_ratio" in h
    ]

    if not evals:
        return {}

    return evals[-1]


final_A = final_record(history_A)
final_B = final_record(history_B)
final_D = final_record(history_D)

print("\n" + "=" * 90)
print("CS781 — A/B/C/D MICRO EXPERIMENT SUMMARY")
print("=" * 90)

print(
    f"\nA — same LR      : "
    f"loss={final_A.get('loss', float('nan')):.4f} | "
    f"unique={final_A.get('unique_ratio', float('nan')):.4f} | "
    f"adj={final_A.get('adj_repeat', float('nan')):.4f} | "
    f"d2={final_A.get('distinct2', float('nan')):.4f} | "
    f"d3={final_A.get('distinct3', float('nan')):.4f} | "
    f"maxrun={final_A.get('worst_maxrun', -1)}"
)

print(
    f"B — low LR       : "
    f"loss={final_B.get('loss', float('nan')):.4f} | "
    f"unique={final_B.get('unique_ratio', float('nan')):.4f} | "
    f"adj={final_B.get('adj_repeat', float('nan')):.4f} | "
    f"d2={final_B.get('distinct2', float('nan')):.4f} | "
    f"d3={final_B.get('distinct3', float('nan')):.4f} | "
    f"maxrun={final_B.get('worst_maxrun', -1)}"
)

print(
    f"C — temp {TEMPERATURE_C} : "
    f"unique={avg_C['unique_ratio']:.4f} | "
    f"adj={avg_C['adj_repeat']:.4f} | "
    f"d2={avg_C['distinct2']:.4f} | "
    f"d3={avg_C['distinct3']:.4f} | "
    f"maxrun={avg_C['worst_maxrun']}"
)

print(
    f"D — tight clip    : "
    f"loss={final_D.get('loss', float('nan')):.4f} | "
    f"unique={final_D.get('unique_ratio', float('nan')):.4f} | "
    f"adj={final_D.get('adj_repeat', float('nan')):.4f} | "
    f"d2={final_D.get('distinct2', float('nan')):.4f} | "
    f"d3={final_D.get('distinct3', float('nan')):.4f} | "
    f"maxrun={final_D.get('worst_maxrun', -1)}"
)


# ==========================================================================================
# 13. SAVE SMALL DIAGNOSTIC RESULT
# ==========================================================================================

RESULTS_DIR = "/kaggle/working/cs781_abcd_micro_diagnostic"
os.makedirs(RESULTS_DIR, exist_ok=True)

results = {
    "config": {
        "steps": STEPS,
        "seq_len": SEQ_LEN,
        "batch_size": BATCH_SIZE,
        "base_lr": BASE_LR,
        "low_lr": LOW_LR,
        "base_clip": BASE_CLIP,
        "tight_clip": TIGHT_CLIP,
        "temperature_A": TEMPERATURE_A,
        "temperature_C": TEMPERATURE_C,
    },
    "A": history_A,
    "B": history_B,
    "C": avg_C,
    "D": history_D,
    "elapsed_seconds": time.time() - start_total,
}

result_path = os.path.join(
    RESULTS_DIR,
    "abcd_micro_results.json",
)

with open(result_path, "w") as f:
    json.dump(results, f, indent=2)

print("\nResults saved:")
print(result_path)

# ==========================================================================================
# 14. RESTORE EXACT STEP-1000 MODEL
# ==========================================================================================

model.load_state_dict(step1000_state, strict=True)
model.to(DEVICE)
model.eval()

print("\n" + "=" * 90)
print("✓ ORIGINAL STEP-1000 MODEL RESTORED")
print("✓ NO CHECKPOINT FILE MODIFIED")
print("✓ NO FULL TRAINING RUN PERFORMED")
print("=" * 90)

In [ ]:
# ==========================================================================================
# CS781 — 500-STEP CHECKPOINT RECOVERY DIAGNOSTIC
# ==========================================================================================
# Purpose:
#   Determine whether the Step-1000 model can recover from generation collapse
#   under continued training, without committing to another 3.5-hour full run.
#
# IMPORTANT:
#   - Starts from EXACT Step-1000 checkpoint.
#   - Does NOT modify the checkpoint on disk.
#   - Does NOT run optimizer on the saved checkpoint.
#   - Diagnostic copy of the model only.
#
# Measures:
#   1. Training loss
#   2. Gradient norm
#   3. Validation loss / perplexity
#   4. Generation diversity
#   5. Adjacent repetition
#   6. Distinct-2 / Distinct-3
#   7. Maximum identical-token run
#   8. Top-token probability concentration
#   9. Generation snapshots at steps 0/100/250/500
# ==========================================================================================

import os
import json
import math
import random
import copy
import time
from collections import Counter

import torch
import torch.nn.functional as F

print("=" * 90)
print("CS781 — 500-STEP CHECKPOINT RECOVERY DIAGNOSTIC")
print("=" * 90)

# ------------------------------------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------------------------------------

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

CHECKPOINT = (
    "/kaggle/input/datasets/oipranesh/"
    "phase4-step1000-verified-model/checkpoint_step_001000.pt"
)

TRAIN_ROOT = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep/dedup"
)

OUT_DIR = "/kaggle/working/cs781_500step_recovery_diagnostic"
os.makedirs(OUT_DIR, exist_ok=True)

STEPS = 500
SEQ_LEN = 1024
BATCH_SIZE = 2

LR = 1e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

LOG_EVERY = 25
EVAL_EVERY = 100

GEN_TOKENS = 128
GEN_TEMPERATURE = 1.0
TOP_K = 50

SEED = 781

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device       : {DEVICE}")
print(f"Checkpoint   : {CHECKPOINT}")
print(f"Steps        : {STEPS}")
print(f"SeqLen       : {SEQ_LEN}")
print(f"Batch        : {BATCH_SIZE}")
print(f"LR           : {LR}")
print(f"Output       : {OUT_DIR}")

assert torch.cuda.is_available(), "CUDA GPU required for this diagnostic."
assert os.path.exists(CHECKPOINT), "Step-1000 checkpoint not found."

# ==========================================================================================
# 1. REQUIRE EXISTING VERIFIED MODEL/TOKENIZER
# ==========================================================================================

if "model" not in globals():
    raise RuntimeError(
        "The verified `model` object is not currently loaded.\n"
        "Run the exact model-construction/loading cell from the verified notebook first."
    )

if "tokenizer" not in globals():
    raise RuntimeError(
        "The verified `tokenizer` object is not currently loaded.\n"
        "Run the exact tokenizer-loading cell from the verified notebook first."
    )

# ------------------------------------------------------------------------------------------
# Move model to GPU and verify
# ------------------------------------------------------------------------------------------

model = model.to(DEVICE)
model.train()

model_device = next(model.parameters()).device

print()
print("MODEL")
print("-" * 90)
print("Model device :", model_device)
print("Parameters   :", sum(p.numel() for p in model.parameters()))

assert model_device == DEVICE

# ==========================================================================================
# 2. LOAD EXACT STEP-1000 WEIGHTS
# ==========================================================================================

print()
print("=" * 90)
print("LOADING EXACT STEP-1000 CHECKPOINT")
print("=" * 90)

checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

state_dict = checkpoint["model_state_dict"]

missing, unexpected = model.load_state_dict(state_dict, strict=False)

print("Checkpoint step :", checkpoint.get("step"))
print("Checkpoint LR   :", checkpoint.get("learning_rate"))
print("Context length  :", checkpoint.get("context_length"))
print("Batch size      :", checkpoint.get("batch_size"))
print("Grad accumulation:",
      checkpoint.get("gradient_accumulation_steps"))

print("Missing keys    :", len(missing))
print("Unexpected keys :", len(unexpected))

assert len(missing) == 0
assert len(unexpected) == 0

model = model.to(DEVICE)
model.train()

print("✓ Exact Step-1000 weights loaded")
print("✓ Original checkpoint file untouched")

# ==========================================================================================
# 3. TOKENIZER HELPERS
# ==========================================================================================

def encode_text(text):
    """
    Supports the verified tokenizer interfaces encountered in this project.
    """
    if hasattr(tokenizer, "encode"):
        try:
            out = tokenizer.encode(text)
        except TypeError:
            out = tokenizer.encode(text, add_special_tokens=False)

        if hasattr(out, "ids"):
            return list(out.ids)

        if isinstance(out, dict):
            if "input_ids" in out:
                return list(out["input_ids"])

        if torch.is_tensor(out):
            return out.flatten().tolist()

        return list(out)

    if hasattr(tokenizer, "tokenize"):
        pieces = tokenizer.tokenize(text)
        if hasattr(tokenizer, "convert_tokens_to_ids"):
            return tokenizer.convert_tokens_to_ids(pieces)

    raise RuntimeError("Could not determine tokenizer encoding interface.")


def decode_ids(ids):
    ids = list(map(int, ids))

    if hasattr(tokenizer, "decode"):
        try:
            return tokenizer.decode(ids)
        except Exception:
            pass

    if hasattr(tokenizer, "tokenizer"):
        try:
            return tokenizer.tokenizer.decode(ids)
        except Exception:
            pass

    return " ".join(map(str, ids))


# ==========================================================================================
# 4. FIND TRAINING TEXT
# ==========================================================================================

print()
print("=" * 90)
print("LOADING TRAINING DATA")
print("=" * 90)

jsonl_files = []

for root, dirs, files in os.walk(TRAIN_ROOT):
    for f in files:
        if f.endswith(".jsonl"):
            jsonl_files.append(os.path.join(root, f))

jsonl_files.sort()

print("Training JSONL files:", len(jsonl_files))

assert len(jsonl_files) > 0

texts = []

# We don't need millions of documents for this diagnostic.
# Collect a reasonably diverse pool.
MAX_TEXTS = 4000

for path in jsonl_files:
    if len(texts) >= MAX_TEXTS:
        break

    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if len(texts) >= MAX_TEXTS:
                    break

                line = line.strip()
                if not line:
                    continue

                try:
                    obj = json.loads(line)
                except Exception:
                    continue

                text = None

                if isinstance(obj, dict):
                    for key in ("text", "content", "document", "body"):
                        if isinstance(obj.get(key), str):
                            text = obj[key]
                            break

                elif isinstance(obj, str):
                    text = obj

                if text and len(text) >= 200:
                    texts.append(text)

    except Exception as e:
        print("Skipping:", path, "|", e)

print("Training texts:", len(texts))

assert len(texts) >= 100, "Not enough training texts recovered."

# ==========================================================================================
# 5. TOKENIZE TRAINING TEXT
# ==========================================================================================

print()
print("Tokenizing training sample...")

token_sequences = []

for text in texts:
    try:
        ids = encode_text(text)

        if len(ids) >= SEQ_LEN + 1:
            token_sequences.append(ids)

    except Exception:
        continue

print("Usable token sequences:", len(token_sequences))

assert len(token_sequences) >= BATCH_SIZE * 10

# ==========================================================================================
# 6. FIXED BATCHES
# ==========================================================================================

# Pre-build fixed batches so every branch/checkpoint sees the same data.
random.shuffle(token_sequences)

usable = token_sequences[:min(len(token_sequences), 2000)]

batches = []

for i in range(0, len(usable) - BATCH_SIZE + 1, BATCH_SIZE):
    if len(batches) >= STEPS:
        break

    batch = usable[i:i + BATCH_SIZE]

    x = torch.tensor(
        [seq[:SEQ_LEN] for seq in batch],
        dtype=torch.long
    )

    y = torch.tensor(
        [seq[1:SEQ_LEN + 1] for seq in batch],
        dtype=torch.long
    )

    batches.append((x, y))

if len(batches) < STEPS:
    # Cycle through the fixed dataset if necessary.
    original = list(batches)

    while len(batches) < STEPS:
        batches.extend(original)

    batches = batches[:STEPS]

print("Fixed batches:", len(batches))
print("Batch shape  :", tuple(batches[0][0].shape))

# ==========================================================================================
# 7. EVALUATION PROMPTS
# ==========================================================================================

eval_prompts = [
    texts[i] for i in
    [0, min(10, len(texts)-1), min(50, len(texts)-1),
     min(100, len(texts)-1), min(200, len(texts)-1)]
]

# Keep only prompts that fit reasonably inside the context.
eval_prompts = [
    p for p in eval_prompts
    if len(encode_text(p)) >= 32
]

print("Evaluation prompts:", len(eval_prompts))

# ==========================================================================================
# 8. GENERATION METRICS
# ==========================================================================================

@torch.no_grad()
def generate_ids(prompt, max_new_tokens=128, temperature=1.0, top_k=50):

    model.eval()

    ids = encode_text(prompt)

    if len(ids) > SEQ_LEN:
        ids = ids[-SEQ_LEN:]

    input_ids = torch.tensor(
        [ids],
        dtype=torch.long,
        device=DEVICE
    )

    generated = []

    for _ in range(max_new_tokens):

        context = input_ids[:, -SEQ_LEN:]

        outputs = model(context)

        if hasattr(outputs, "logits"):
            logits = outputs.logits
        elif isinstance(outputs, tuple):
            logits = outputs[0]
        else:
            logits = outputs

        next_logits = logits[:, -1, :].float()

        next_logits = next_logits / max(temperature, 1e-6)

        if top_k is not None:
            k = min(top_k, next_logits.shape[-1])
            values, indices = torch.topk(next_logits, k=k)
            filtered = torch.full_like(next_logits, float("-inf"))
            filtered.scatter_(1, indices, values)
            next_logits = filtered

        probs = F.softmax(next_logits, dim=-1)

        next_id = torch.multinomial(probs, num_samples=1)

        generated.append(int(next_id.item()))

        input_ids = torch.cat(
            [input_ids, next_id],
            dim=1
        )

    model.train()

    return generated


def generation_metrics(ids):

    ids = list(map(int, ids))

    if not ids:
        return {
            "unique_ratio": 0.0,
            "adjacent_repeat": 0.0,
            "distinct_2": 0.0,
            "distinct_3": 0.0,
            "max_run": 0,
        }

    unique_ratio = len(set(ids)) / len(ids)

    adjacent_repeat = (
        sum(ids[i] == ids[i-1] for i in range(1, len(ids)))
        / max(1, len(ids) - 1)
    )

    bigrams = list(zip(ids[:-1], ids[1:]))

    if bigrams:
        distinct_2 = len(set(bigrams)) / len(bigrams)
    else:
        distinct_2 = 0.0

    trigrams = list(zip(ids[:-2], ids[1:-1], ids[2:]))

    if trigrams:
        distinct_3 = len(set(trigrams)) / len(trigrams)
    else:
        distinct_3 = 0.0

    max_run = 1
    current = 1

    for i in range(1, len(ids)):
        if ids[i] == ids[i-1]:
            current += 1
            max_run = max(max_run, current)
        else:
            current = 1

    return {
        "unique_ratio": unique_ratio,
        "adjacent_repeat": adjacent_repeat,
        "distinct_2": distinct_2,
        "distinct_3": distinct_3,
        "max_run": max_run,
    }


# ==========================================================================================
# 9. GENERATION SNAPSHOT
# ==========================================================================================

@torch.no_grad()
def evaluate_generation(step):

    metrics = []
    samples = []

    for prompt in eval_prompts:

        ids = generate_ids(
            prompt,
            max_new_tokens=GEN_TOKENS,
            temperature=GEN_TEMPERATURE,
            top_k=TOP_K
        )

        m = generation_metrics(ids)
        metrics.append(m)

        samples.append({
            "ids": ids,
            "text": decode_ids(ids),
            **m
        })

    avg = {}

    for key in metrics[0]:
        avg[key] = sum(x[key] for x in metrics) / len(metrics)

    avg["step"] = step

    return avg, samples


# ==========================================================================================
# 10. VALIDATION LOSS
# ==========================================================================================

@torch.no_grad()
def evaluate_loss(num_batches=20):

    model.eval()

    losses = []

    for i in range(min(num_batches, len(batches))):

        x, y = batches[i]

        x = x.to(DEVICE)
        y = y.to(DEVICE)

        outputs = model(x)

        if hasattr(outputs, "logits"):
            logits = outputs.logits
        elif isinstance(outputs, tuple):
            logits = outputs[0]
        else:
            logits = outputs

        loss = F.cross_entropy(
            logits.reshape(-1, logits.shape[-1]),
            y.reshape(-1)
        )

        losses.append(float(loss.item()))

    model.train()

    mean_loss = sum(losses) / len(losses)
    perplexity = math.exp(min(mean_loss, 20))

    return mean_loss, perplexity


# ==========================================================================================
# 11. TOP-TOKEN CONCENTRATION
# ==========================================================================================

@torch.no_grad()
def probability_diagnostics(prompt):

    model.eval()

    ids = encode_text(prompt)

    if len(ids) > SEQ_LEN:
        ids = ids[-SEQ_LEN:]

    x = torch.tensor(
        [ids],
        dtype=torch.long,
        device=DEVICE
    )

    outputs = model(x)

    if hasattr(outputs, "logits"):
        logits = outputs.logits
    elif isinstance(outputs, tuple):
        logits = outputs[0]
    else:
        logits = outputs

    probs = F.softmax(logits[:, -1, :].float(), dim=-1)

    top_probs, top_ids = torch.topk(probs, k=10, dim=-1)

    entropy = -(probs * torch.log(probs + 1e-12)).sum(dim=-1)

    model.train()

    return {
        "entropy": float(entropy.item()),
        "top1_prob": float(top_probs[0, 0].item()),
        "top5_mass": float(top_probs[0, :5].sum().item()),
        "top10_mass": float(top_probs[0, :10].sum().item()),
        "top_ids": top_ids[0].tolist(),
    }


# ==========================================================================================
# 12. OPTIMIZER
# ==========================================================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

# ==========================================================================================
# 13. INITIAL STATE
# ==========================================================================================

print()
print("=" * 90)
print("BASELINE — EXACT STEP-1000 MODEL")
print("=" * 90)

baseline_loss, baseline_ppl = evaluate_loss()

baseline_gen, baseline_samples = evaluate_generation(0)

baseline_prob = probability_diagnostics(eval_prompts[0])

print(
    f"Validation loss : {baseline_loss:.4f}\n"
    f"Perplexity      : {baseline_ppl:.2f}\n"
    f"Unique ratio    : {baseline_gen['unique_ratio']:.4f}\n"
    f"Adjacent repeat : {baseline_gen['adjacent_repeat']:.4f}\n"
    f"Distinct-2      : {baseline_gen['distinct_2']:.4f}\n"
    f"Distinct-3      : {baseline_gen['distinct_3']:.4f}\n"
    f"Max run         : {baseline_gen['max_run']:.0f}\n"
    f"Entropy         : {baseline_prob['entropy']:.4f}\n"
    f"Top-1 prob      : {baseline_prob['top1_prob']:.4f}\n"
    f"Top-10 mass     : {baseline_prob['top10_mass']:.4f}"
)

results = {
    "config": {
        "steps": STEPS,
        "seq_len": SEQ_LEN,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "max_grad_norm": MAX_GRAD_NORM,
        "generation_temperature": GEN_TEMPERATURE,
        "generation_top_k": TOP_K,
    },
    "baseline": {
        "loss": baseline_loss,
        "perplexity": baseline_ppl,
        "generation": baseline_gen,
        "probability": baseline_prob,
        "samples": baseline_samples,
    },
    "checkpoints": [],
    "training": []
}

# ==========================================================================================
# 14. 500-STEP CONTINUATION
# ==========================================================================================

print()
print("=" * 90)
print("500-STEP CONTINUATION")
print("=" * 90)

start_time = time.time()

for step in range(1, STEPS + 1):

    model.train()

    x, y = batches[step - 1]

    x = x.to(DEVICE)
    y = y.to(DEVICE)

    optimizer.zero_grad(set_to_none=True)

    outputs = model(x)

    if hasattr(outputs, "logits"):
        logits = outputs.logits
    elif isinstance(outputs, tuple):
        logits = outputs[0]
    else:
        logits = outputs

    loss = F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]),
        y.reshape(-1)
    )

    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        MAX_GRAD_NORM
    )

    optimizer.step()

    record = {
        "step": step,
        "loss": float(loss.item()),
        "grad_norm": float(grad_norm.item())
    }

    results["training"].append(record)

    if step % LOG_EVERY == 0 or step == 1:

        print(
            f"step={step:3d} | "
            f"loss={loss.item():.4f} | "
            f"grad={grad_norm.item():.4f}"
        )

    # --------------------------------------------------------------------------------------
    # CHECKPOINT DIAGNOSTIC
    # --------------------------------------------------------------------------------------

    if step in (100, 250, 500):

        print()
        print("-" * 90)
        print(f"DIAGNOSTIC CHECKPOINT — STEP {step}")
        print("-" * 90)

        eval_loss, eval_ppl = evaluate_loss()

        gen_metrics, samples = evaluate_generation(step)

        prob = probability_diagnostics(eval_prompts[0])

        checkpoint_result = {
            "step": step,
            "loss": eval_loss,
            "perplexity": eval_ppl,
            "generation": gen_metrics,
            "probability": prob,
            "samples": samples,
            "elapsed_seconds": time.time() - start_time
        }

        results["checkpoints"].append(checkpoint_result)

        print(f"Validation loss : {eval_loss:.4f}")
        print(f"Perplexity      : {eval_ppl:.2f}")
        print(f"Unique ratio    : {gen_metrics['unique_ratio']:.4f}")
        print(f"Adjacent repeat : {gen_metrics['adjacent_repeat']:.4f}")
        print(f"Distinct-2      : {gen_metrics['distinct_2']:.4f}")
        print(f"Distinct-3      : {gen_metrics['distinct_3']:.4f}")
        print(f"Max run         : {gen_metrics['max_run']:.0f}")
        print(f"Entropy         : {prob['entropy']:.4f}")
        print(f"Top-1 prob      : {prob['top1_prob']:.4f}")
        print(f"Top-10 mass     : {prob['top10_mass']:.4f}")

        print()
        print("Sample generation:")
        print(decode_ids(samples[0]["ids"])[:1000])

# ==========================================================================================
# 15. FINAL SUMMARY
# ==========================================================================================

elapsed = time.time() - start_time

print()
print("=" * 90)
print("CS781 — 500-STEP RECOVERY DIAGNOSTIC SUMMARY")
print("=" * 90)

print(f"Elapsed time: {elapsed / 60:.2f} minutes")

print()
print(
    f"{'POINT':<12}"
    f"{'LOSS':>10}"
    f"{'PPL':>10}"
    f"{'UNIQUE':>10}"
    f"{'ADJ':>10}"
    f"{'D2':>10}"
    f"{'D3':>10}"
    f"{'MAXRUN':>10}"
)

print("-" * 82)

def print_row(name, loss, ppl, g):
    print(
        f"{name:<12}"
        f"{loss:>10.4f}"
        f"{ppl:>10.2f}"
        f"{g['unique_ratio']:>10.4f}"
        f"{g['adjacent_repeat']:>10.4f}"
        f"{g['distinct_2']:>10.4f}"
        f"{g['distinct_3']:>10.4f}"
        f"{g['max_run']:>10.0f}"
    )

print_row(
    "STEP-1000",
    baseline_loss,
    baseline_ppl,
    baseline_gen
)

for r in results["checkpoints"]:
    print_row(
        f"STEP+{r['step']}",
        r["loss"],
        r["perplexity"],
        r["generation"]
    )

# ==========================================================================================
# 16. AUTOMATIC INTERPRETATION
# ==========================================================================================

print()
print("=" * 90)
print("INTERPRETATION")
print("=" * 90)

if len(results["checkpoints"]) >= 3:

    r100 = results["checkpoints"][0]
    r250 = results["checkpoints"][1]
    r500 = results["checkpoints"][2]

    base_u = baseline_gen["unique_ratio"]
    final_u = r500["generation"]["unique_ratio"]

    base_run = baseline_gen["max_run"]
    final_run = r500["generation"]["max_run"]

    base_loss = baseline_loss
    final_loss = r500["loss"]

    print()
    print(f"Loss change       : {base_loss:.4f} -> {final_loss:.4f}")
    print(f"Unique ratio      : {base_u:.4f} -> {final_u:.4f}")
    print(f"Maximum run       : {base_run:.0f} -> {final_run:.0f}")

    print()

    if final_u > base_u * 1.25 and final_run < base_run:
        print("✓ RECOVERY SIGNAL:")
        print("  Continued training appears to improve generation diversity.")

    elif final_u < base_u * 0.75 or final_run > max(10, base_run * 2):
        print("✗ FURTHER COLLAPSE:")
        print("  Continued training appears to worsen generation repetition.")

    else:
        print("~ NO CLEAR RECOVERY:")
        print("  500 additional steps did not produce a decisive change.")

# ==========================================================================================
# 17. SAVE DIAGNOSTIC RESULTS
# ==========================================================================================

result_path = os.path.join(
    OUT_DIR,
    "500step_recovery_results.json"
)

with open(result_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print()
print("=" * 90)
print("RESULT SAVED")
print("=" * 90)
print(result_path)

print()
print("IMPORTANT:")
print("✓ Original Step-1000 checkpoint was NOT modified")
print("✓ This was a diagnostic continuation only")
print("✓ No new checkpoint was written")
print("✓ Do NOT start the full 3.5-hour run until these results are inspected")
print("=" * 90)

In [ ]:
# ============================================================================
# CS781 — LOWER-LR RECOVERY CANDIDATE
# ============================================================================
# Purpose:
#   Step-1000 checkpoint -> 500-step continuation at LR=5e-6
#
# This is a CONTROLLED candidate run.
# It does NOT modify the original checkpoint.
#
# Changes vs previous 500-step diagnostic:
#   LR: 1e-5 -> 5e-6
#
# Keeps:
#   architecture
#   tokenizer
#   1024 training sequence length for diagnostic speed
#   batch=2
#   same training corpus
#
# Evaluates:
#   validation loss / PPL
#   generation diversity
#   entropy / top-1 / top-10
#   temperatures 0.7 and 0.9
# ============================================================================

import os
import json
import math
import random
import copy
import time
from collections import Counter

import torch
import torch.nn.functional as F

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CHECKPOINT = (
    "/kaggle/input/datasets/oipranesh/"
    "phase4-step1000-verified-model/checkpoint_step_001000.pt"
)

TRAIN_ROOT = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep"
)

TOKENIZER_PATH = os.path.join(
    TRAIN_ROOT, "tokenizer", "legal_bpe_32k.json"
)

OUT_DIR = "/kaggle/working/cs781_lr5e6_candidate"
os.makedirs(OUT_DIR, exist_ok=True)

STEPS = 500
LR = 5e-6

SEQ_LEN = 1024
BATCH_SIZE = 2

EVAL_EVERY = 100

SEED = 781

# Generation
GEN_TOKENS = 128
TEMPERATURES = [0.7, 0.9]

# Prevent catastrophic generation loops during diagnostic
REPETITION_WINDOW = 32

print("=" * 90)
print("CS781 — LOWER-LR RECOVERY CANDIDATE")
print("=" * 90)
print(f"Device       : {DEVICE}")
print(f"Checkpoint   : {CHECKPOINT}")
print(f"Steps        : {STEPS}")
print(f"SeqLen       : {SEQ_LEN}")
print(f"Batch        : {BATCH_SIZE}")
print(f"Learning rate : {LR}")
print(f"Output       : {OUT_DIR}")

# ============================================================================
# SEED
# ============================================================================

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ============================================================================
# MODEL / TOKENIZER
# ============================================================================
# IMPORTANT:
# This cell expects the same model/tokenizer construction objects/functions
# that were used by your previous successful diagnostic notebook.
#
# If `model` and `tokenizer` already exist, they are reused.
# Otherwise, this attempts to reuse the construction code saved in globals.
# ============================================================================

if "model" not in globals():
    raise RuntimeError(
        "model is not currently defined.\n\n"
        "Run the same model-construction cell used for the previous "
        "500-step recovery diagnostic first, then rerun this cell."
    )

if "tokenizer" not in globals():
    raise RuntimeError(
        "tokenizer is not currently defined.\n\n"
        "Run the same tokenizer-construction cell used for the previous "
        "500-step recovery diagnostic first, then rerun this cell."
    )

model = model.to(DEVICE)

print("\n" + "=" * 90)
print("MODEL")
print("=" * 90)
print("Parameters :", sum(p.numel() for p in model.parameters()))
print("Device     :", next(model.parameters()).device)

# ============================================================================
# LOAD EXACT STEP-1000 CHECKPOINT
# ============================================================================

print("\n" + "=" * 90)
print("LOADING EXACT STEP-1000 CHECKPOINT")
print("=" * 90)

ckpt = torch.load(
    CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)

state = ckpt["model_state_dict"]

missing, unexpected = model.load_state_dict(
    state,
    strict=False
)

print("Checkpoint step :", ckpt.get("step"))
print("Checkpoint LR   :", ckpt.get("learning_rate"))
print("Context length  :", ckpt.get("context_length"))
print("Missing keys    :", len(missing))
print("Unexpected keys :", len(unexpected))

if missing or unexpected:
    print("WARNING: checkpoint/model mismatch detected.")

print("✓ Exact Step-1000 weights loaded")
print("✓ Original checkpoint untouched")

# ============================================================================
# DISCOVER TRAINING JSONL
# ============================================================================

print("\n" + "=" * 90)
print("LOADING TRAINING DATA")
print("=" * 90)

jsonl_files = []

for root, dirs, files in os.walk(
    os.path.join(TRAIN_ROOT, "dedup")
):
    for fn in files:
        if fn.endswith(".jsonl"):
            jsonl_files.append(os.path.join(root, fn))

jsonl_files.sort()

print("Training JSONL files:", len(jsonl_files))

# ============================================================================
# EXTRACT TEXT
# ============================================================================

def extract_text(obj):
    if not isinstance(obj, dict):
        return None

    candidates = [
        "text",
        "content",
        "document",
        "body",
        "raw_text",
    ]

    for k in candidates:
        v = obj.get(k)
        if isinstance(v, str) and len(v.strip()) > 100:
            return v

    return None


training_texts = []

# Keep this diagnostic bounded.
MAX_TEXTS = 4000

for path in jsonl_files:

    if len(training_texts) >= MAX_TEXTS:
        break

    try:
        with open(
            path,
            "r",
            encoding="utf-8",
            errors="ignore"
        ) as f:

            for line in f:

                if len(training_texts) >= MAX_TEXTS:
                    break

                try:
                    obj = json.loads(line)
                except Exception:
                    continue

                text = extract_text(obj)

                if text:
                    training_texts.append(text)

    except Exception as e:
        print("Skipped:", path, "|", e)

print("Training texts recovered:", len(training_texts))

if not training_texts:
    raise RuntimeError(
        "Could not recover training texts."
    )

# ============================================================================
# TOKENIZER HELPERS
# ============================================================================

def encode_text(text):
    """
    Supports the tokenizer API used by the previous diagnostic.
    """
    if hasattr(tokenizer, "encode"):
        out = tokenizer.encode(text)

        if hasattr(out, "ids"):
            return list(out.ids)

        if isinstance(out, (list, tuple)):
            return list(out)

    raise RuntimeError(
        "Unable to encode text with current tokenizer."
    )


def decode_ids(ids):
    if hasattr(tokenizer, "decode"):
        try:
            return tokenizer.decode(ids)
        except Exception:
            pass

    return " ".join(str(x) for x in ids)


# ============================================================================
# BUILD FIXED TRAINING BATCHES
# ============================================================================

print("\nTokenizing training sample...")

token_sequences = []

for text in training_texts:

    ids = encode_text(text)

    if len(ids) >= SEQ_LEN + 1:

        # deterministic random crop
        start = random.randint(
            0,
            max(0, len(ids) - SEQ_LEN - 1)
        )

        chunk = ids[start:start + SEQ_LEN + 1]

        if len(chunk) == SEQ_LEN + 1:
            token_sequences.append(chunk)

print("Usable token sequences:", len(token_sequences))

if len(token_sequences) < BATCH_SIZE:
    raise RuntimeError("Not enough token sequences.")

# Fixed batches make A/B comparisons cleaner.
random.shuffle(token_sequences)

batches = []

for i in range(
    0,
    len(token_sequences) - BATCH_SIZE + 1,
    BATCH_SIZE
):

    batch = token_sequences[i:i + BATCH_SIZE]

    x = torch.tensor(
        [a[:-1] for a in batch],
        dtype=torch.long,
        device=DEVICE
    )

    y = torch.tensor(
        [a[1:] for a in batch],
        dtype=torch.long,
        device=DEVICE
    )

    batches.append((x, y))

    if len(batches) >= STEPS:
        break

print("Fixed batches:", len(batches))
print("Batch shape :", tuple(batches[0][0].shape))

if len(batches) < STEPS:
    print(
        f"WARNING: only {len(batches)} fixed batches available "
        f"for requested {STEPS} steps."
    )

# ============================================================================
# LOSS EXTRACTION
# ============================================================================

def get_logits(output):
    if hasattr(output, "logits"):
        return output.logits

    if isinstance(output, dict):
        if "logits" in output:
            return output["logits"]

    if torch.is_tensor(output):
        return output

    if isinstance(output, (tuple, list)):
        return output[0]

    raise RuntimeError(
        "Could not extract logits from model output."
    )


def model_loss(x, y):

    output = model(
        input_ids=x
    )

    logits = get_logits(output)

    # causal LM
    logits = logits[:, :-1, :]
    targets = y[:, 1:]

    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        targets.reshape(-1)
    )

    return loss, logits


# ============================================================================
# GENERATION
# ============================================================================

def get_prompt_ids(text):

    ids = encode_text(text)

    # Use enough prompt context without exceeding model context.
    if len(ids) > 256:
        ids = ids[-256:]

    return torch.tensor(
        [ids],
        dtype=torch.long,
        device=DEVICE
    )


def generate_sample(prompt, temperature):

    model.eval()

    ids = get_prompt_ids(prompt)

    generated = ids.clone()

    with torch.no_grad():

        for _ in range(GEN_TOKENS):

            context = generated[:, -4096:]

            output = model(
                input_ids=context
            )

            logits = get_logits(output)

            next_logits = logits[:, -1, :]

            next_logits = next_logits / temperature

            probs = F.softmax(
                next_logits,
                dim=-1
            )

            next_id = torch.multinomial(
                probs,
                num_samples=1
            )

            generated = torch.cat(
                [generated, next_id],
                dim=1
            )

    new_ids = generated[0, ids.shape[1]:].tolist()

    return new_ids


# ============================================================================
# GENERATION METRICS
# ============================================================================

def generation_metrics(ids):

    if not ids:
        return {
            "unique": 0,
            "adj": 0,
            "d2": 0,
            "d3": 0,
            "maxrun": 0,
            "repeated_5gram_rate": 0,
        }

    n = len(ids)

    unique_ratio = len(set(ids)) / n

    adjacent = sum(
        ids[i] == ids[i - 1]
        for i in range(1, n)
    )

    adj_ratio = adjacent / max(1, n - 1)

    bigrams = list(zip(ids[:-1], ids[1:]))

    if bigrams:
        d2 = len(set(bigrams)) / len(bigrams)
    else:
        d2 = 0

    trigrams = list(
        zip(ids[:-2], ids[1:-1], ids[2:])
    )

    if trigrams:
        d3 = len(set(trigrams)) / len(trigrams)
    else:
        d3 = 0

    # longest identical-token run
    maxrun = 1
    current = 1

    for i in range(1, n):

        if ids[i] == ids[i - 1]:
            current += 1
            maxrun = max(maxrun, current)

        else:
            current = 1

    # repeated 5-gram rate
    grams5 = [
        tuple(ids[i:i+5])
        for i in range(max(0, n - 4))
    ]

    repeated_5gram_rate = 0

    if grams5:
        counts = Counter(grams5)
        repeated = sum(
            c - 1
            for c in counts.values()
            if c > 1
        )
        repeated_5gram_rate = repeated / len(grams5)

    return {
        "unique": unique_ratio,
        "adj": adj_ratio,
        "d2": d2,
        "d3": d3,
        "maxrun": maxrun,
        "repeated_5gram_rate": repeated_5gram_rate,
    }


# ============================================================================
# FIXED EVALUATION PROMPTS
# ============================================================================

eval_prompts = [
    training_texts[0][:1200],
    training_texts[min(100, len(training_texts)-1)][:1200],
    training_texts[min(500, len(training_texts)-1)][:1200],
    training_texts[min(1000, len(training_texts)-1)][:1200],
    training_texts[min(1500, len(training_texts)-1)][:1200],
]

# ============================================================================
# VALIDATION PROXY
# ============================================================================
# We use held-out-looking samples from the recovered corpus for this
# controlled experiment. The goal here is comparing the candidate against
# the Step-1000 baseline under identical evaluation conditions.
# ============================================================================

eval_sequences = token_sequences[
    :min(100, len(token_sequences))
]


@torch.no_grad()
def evaluate_loss():

    model.eval()

    losses = []

    for chunk in eval_sequences:

        x = torch.tensor(
            [chunk[:-1]],
            dtype=torch.long,
            device=DEVICE
        )

        y = torch.tensor(
            [chunk[1:]],
            dtype=torch.long,
            device=DEVICE
        )

        loss, _ = model_loss(x, y)

        losses.append(
            float(loss.item())
        )

    mean_loss = sum(losses) / len(losses)

    ppl = math.exp(
        min(mean_loss, 20)
    )

    return mean_loss, ppl


@torch.no_grad()
def probability_diagnostics():

    model.eval()

    entropies = []
    top1s = []
    top10s = []

    for chunk in eval_sequences[:25]:

        x = torch.tensor(
            [chunk[:-1]],
            dtype=torch.long,
            device=DEVICE
        )

        output = model(input_ids=x)
        logits = get_logits(output)

        logits = logits[:, -1, :]

        probs = F.softmax(logits, dim=-1)

        entropy = -(
            probs *
            torch.log(
                probs.clamp_min(1e-12)
            )
        ).sum(dim=-1)

        top1 = probs.max(dim=-1).values

        top10 = torch.topk(
            probs,
            k=min(10, probs.shape[-1]),
            dim=-1
        ).values.sum(dim=-1)

        entropies.append(float(entropy.item()))
        top1s.append(float(top1.item()))
        top10s.append(float(top10.item()))

    return {
        "entropy": sum(entropies) / len(entropies),
        "top1": sum(top1s) / len(top1s),
        "top10": sum(top10s) / len(top10s),
    }


def evaluate_generation():

    results = {}

    for temperature in TEMPERATURES:

        all_metrics = []
        samples = []

        for prompt in eval_prompts:

            ids = generate_sample(
                prompt,
                temperature
            )

            m = generation_metrics(ids)

            all_metrics.append(m)

            samples.append({
                "text": decode_ids(ids),
                "metrics": m,
            })

        keys = all_metrics[0].keys()

        averaged = {
            k: sum(
                x[k] for x in all_metrics
            ) / len(all_metrics)
            for k in keys
        }

        results[str(temperature)] = {
            "metrics": averaged,
            "samples": samples,
        }

    return results


# ============================================================================
# BASELINE
# ============================================================================

print("\n" + "=" * 90)
print("BASELINE — EXACT STEP-1000 MODEL")
print("=" * 90)

baseline_loss, baseline_ppl = evaluate_loss()
baseline_probs = probability_diagnostics()
baseline_gen = evaluate_generation()

print(
    f"Validation loss : {baseline_loss:.4f}"
)
print(
    f"Perplexity      : {baseline_ppl:.2f}"
)
print(
    f"Entropy         : {baseline_probs['entropy']:.4f}"
)
print(
    f"Top-1 prob      : {baseline_probs['top1']:.4f}"
)
print(
    f"Top-10 mass     : {baseline_probs['top10']:.4f}"
)

for temp, result in baseline_gen.items():

    m = result["metrics"]

    print(
        f"T={temp} | "
        f"unique={m['unique']:.4f} | "
        f"adj={m['adj']:.4f} | "
        f"d2={m['d2']:.4f} | "
        f"d3={m['d3']:.4f} | "
        f"5gram_rep={m['repeated_5gram_rate']:.4f} | "
        f"maxrun={m['maxrun']:.0f}"
    )

# ============================================================================
# NEW OPTIMIZER
# ============================================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=0.01
)

# ============================================================================
# TRAINING
# ============================================================================

print("\n" + "=" * 90)
print("500-STEP LOWER-LR CONTINUATION")
print("=" * 90)

model.train()

training_log = []

start_time = time.time()

for step in range(1, STEPS + 1):

    x, y = batches[
        (step - 1) % len(batches)
    ]

    optimizer.zero_grad(
        set_to_none=True
    )

    loss, _ = model_loss(x, y)

    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=1.0
    )

    optimizer.step()

    loss_value = float(loss.item())

    training_log.append({
        "step": step,
        "loss": loss_value,
        "grad_norm": float(
            grad_norm.item()
            if torch.is_tensor(grad_norm)
            else grad_norm
        ),
    })

    if step == 1 or step % 25 == 0:

        print(
            f"step={step:4d} | "
            f"loss={loss_value:.4f} | "
            f"grad={float(grad_norm):.4f}"
        )

    # ------------------------------------------------------------------------
    # CHECKPOINT EVALUATION
    # ------------------------------------------------------------------------

    if step in [100, 250, 500]:

        print("\n" + "-" * 90)
        print(f"DIAGNOSTIC CHECKPOINT — +{step} STEPS")
        print("-" * 90)

        val_loss, ppl = evaluate_loss()
        probs = probability_diagnostics()
        gen = evaluate_generation()

        print(
            f"Validation loss : {val_loss:.4f}"
        )
        print(
            f"Perplexity      : {ppl:.2f}"
        )
        print(
            f"Entropy         : {probs['entropy']:.4f}"
        )
        print(
            f"Top-1 prob      : {probs['top1']:.4f}"
        )
        print(
            f"Top-10 mass     : {probs['top10']:.4f}"
        )

        for temp, result in gen.items():

            m = result["metrics"]

            print(
                f"T={temp} | "
                f"unique={m['unique']:.4f} | "
                f"adj={m['adj']:.4f} | "
                f"d2={m['d2']:.4f} | "
                f"d3={m['d3']:.4f} | "
                f"5gram_rep={m['repeated_5gram_rate']:.4f} | "
                f"maxrun={m['maxrun']:.0f}"
            )

        # save diagnostic
        diagnostic = {
            "step": step,
            "validation_loss": val_loss,
            "perplexity": ppl,
            "probability": probs,
            "generation": gen,
        }

        with open(
            os.path.join(
                OUT_DIR,
                f"diagnostic_{step}.json"
            ),
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                diagnostic,
                f,
                indent=2,
                ensure_ascii=False
            )

# ============================================================================
# FINAL SUMMARY
# ============================================================================

elapsed = time.time() - start_time

final_loss, final_ppl = evaluate_loss()
final_probs = probability_diagnostics()
final_gen = evaluate_generation()

print("\n" + "=" * 90)
print("CS781 — LOWER-LR CANDIDATE SUMMARY")
print("=" * 90)

print(
    f"Elapsed time       : {elapsed / 60:.2f} minutes"
)

print(
    f"Baseline loss      : {baseline_loss:.4f}"
)

print(
    f"Final loss         : {final_loss:.4f}"
)

print(
    f"Loss change        : "
    f"{baseline_loss - final_loss:+.4f}"
)

print(
    f"Baseline PPL       : {baseline_ppl:.2f}"
)

print(
    f"Final PPL          : {final_ppl:.2f}"
)

print(
    f"Baseline entropy   : "
    f"{baseline_probs['entropy']:.4f}"
)

print(
    f"Final entropy      : "
    f"{final_probs['entropy']:.4f}"
)

print(
    f"Baseline top-1     : "
    f"{baseline_probs['top1']:.4f}"
)

print(
    f"Final top-1        : "
    f"{final_probs['top1']:.4f}"
)

print("\nGENERATION COMPARISON")

for temp in TEMPERATURES:

    b = baseline_gen[str(temp)]["metrics"]
    f = final_gen[str(temp)]["metrics"]

    print(
        f"\nTemperature {temp}"
    )

    print(
        f"  unique : {b['unique']:.4f} -> {f['unique']:.4f}"
    )

    print(
        f"  adj    : {b['adj']:.4f} -> {f['adj']:.4f}"
    )

    print(
        f"  d2     : {b['d2']:.4f} -> {f['d2']:.4f}"
    )

    print(
        f"  d3     : {b['d3']:.4f} -> {f['d3']:.4f}"
    )

    print(
        f"  5gram  : "
        f"{b['repeated_5gram_rate']:.4f} -> "
        f"{f['repeated_5gram_rate']:.4f}"
    )

    print(
        f"  maxrun : "
        f"{b['maxrun']:.0f} -> "
        f"{f['maxrun']:.0f}"
    )

# ============================================================================
# SAVE COMPLETE RESULTS
# ============================================================================

results = {
    "config": {
        "checkpoint": CHECKPOINT,
        "steps": STEPS,
        "seq_len": SEQ_LEN,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "temperatures": TEMPERATURES,
    },

    "baseline": {
        "loss": baseline_loss,
        "perplexity": baseline_ppl,
        "probability": baseline_probs,
        "generation": baseline_gen,
    },

    "final": {
        "loss": final_loss,
        "perplexity": final_ppl,
        "probability": final_probs,
        "generation": final_gen,
    },

    "training_log": training_log,

    "elapsed_seconds": elapsed,
}

RESULT_PATH = os.path.join(
    OUT_DIR,
    "lr5e6_candidate_results.json"
)

with open(
    RESULT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n" + "=" * 90)
print("✓ CANDIDATE RUN COMPLETE")
print("=" * 90)
print("Results:", RESULT_PATH)
print("Original Step-1000 checkpoint was NOT modified.")
print("No production checkpoint was written.")
print("=" * 90)

In [ ]:
# ==========================================================================================
# CS781 — LR=5e-6 CONTROLLED RECOVERY TEST
# Exact Step-1000 checkpoint | 250 steps | NO checkpoint writing
# ==========================================================================================

import os, json, math, glob, time, random
import torch
import torch.nn.functional as F

print("=" * 90)
print("CS781 — LR=5e-6 CONTROLLED RECOVERY TEST")
print("=" * 90)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CKPT = "/kaggle/input/datasets/oipranesh/phase4-step1000-verified-model/checkpoint_step_001000.pt"
DATA_ROOT = "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep/dedup"

# ------------------------------------------------------------------
# IMPORTANT: do NOT change these
# ------------------------------------------------------------------
STEPS = 250
SEQ_LEN = 1024
BATCH_SIZE = 2
LR = 5e-6
EVAL_EVERY = 50

OUT_DIR = "/kaggle/working/cs781_lr5e6_controlled_250"
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Device     : {DEVICE}")
print(f"Checkpoint : {CKPT}")
print(f"Steps      : {STEPS}")
print(f"SeqLen     : {SEQ_LEN}")
print(f"Batch      : {BATCH_SIZE}")
print(f"LR         : {LR}")

# ==========================================================================================
# 1. FIND / LOAD MODEL
# ==========================================================================================

# Reuse model/tokenizer objects if the earlier notebook cells created them.
if "model" not in globals():
    raise RuntimeError(
        "Model object not found. Run the original model-definition/loading cell first."
    )

if "tokenizer" not in globals():
    raise RuntimeError(
        "Tokenizer object not found. Run the original tokenizer-loading cell first."
    )

model = model.to(DEVICE)

# ==========================================================================================
# 2. LOAD EXACT STEP-1000 WEIGHTS
# ==========================================================================================

print("\n" + "=" * 90)
print("LOADING EXACT STEP-1000 CHECKPOINT")
print("=" * 90)

ckpt = torch.load(CKPT, map_location="cpu")

state = ckpt["model_state_dict"]

missing, unexpected = model.load_state_dict(state, strict=False)

print("Checkpoint step :", ckpt.get("step"))
print("Checkpoint LR   :", ckpt.get("learning_rate"))
print("Context length  :", ckpt.get("context_length"))
print("Missing keys    :", len(missing))
print("Unexpected keys :", len(unexpected))

if missing or unexpected:
    raise RuntimeError(
        f"Checkpoint mismatch: missing={missing[:10]}, unexpected={unexpected[:10]}"
    )

print("✓ Exact Step-1000 weights loaded")
print("✓ Original checkpoint untouched")

# ==========================================================================================
# 3. LOAD TRAINING TEXT
# ==========================================================================================

print("\n" + "=" * 90)
print("LOADING TRAINING DATA")
print("=" * 90)

jsonl_files = sorted(glob.glob(DATA_ROOT + "/**/*.jsonl", recursive=True))

print("Training JSONL files:", len(jsonl_files))

if not jsonl_files:
    raise RuntimeError("No training JSONL files found.")

texts = []

# Same broad extraction logic used in the successful diagnostic.
for path in jsonl_files:
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                except Exception:
                    continue

                text = None

                if isinstance(obj, str):
                    text = obj

                elif isinstance(obj, dict):
                    for key in [
                        "text",
                        "content",
                        "document",
                        "body",
                        "raw_text",
                    ]:
                        if isinstance(obj.get(key), str) and obj[key].strip():
                            text = obj[key]
                            break

                if text and len(text.strip()) >= 100:
                    texts.append(text.strip())

                    # Same scale as the previous successful run.
                    if len(texts) >= 4000:
                        break

        if len(texts) >= 4000:
            break

    except Exception as e:
        print("Skipping:", path, "|", repr(e))

print("Training texts recovered:", len(texts))

if len(texts) < 500:
    raise RuntimeError("Too few training texts recovered.")

# ==========================================================================================
# 4. TOKENIZE + FIXED BATCHES
# ==========================================================================================

print("\nTokenizing training sample...")

sequences = []

for text in texts:
    try:
        ids = tokenizer.encode(text, add_special_tokens=False)
    except TypeError:
        ids = tokenizer.encode(text)

    if len(ids) >= SEQ_LEN + 1:
        sequences.append(ids)

print("Usable token sequences:", len(sequences))

if len(sequences) < STEPS * BATCH_SIZE:
    raise RuntimeError(
        f"Not enough usable sequences: {len(sequences)} "
        f"for {STEPS * BATCH_SIZE} required batches."
    )

# Deterministic fixed batches.
random.seed(781)
random.shuffle(sequences)

batches = []

for i in range(STEPS):
    chunk = sequences[i * BATCH_SIZE:(i + 1) * BATCH_SIZE]

    x = torch.tensor(
        [s[:SEQ_LEN] for s in chunk],
        dtype=torch.long,
        device=DEVICE
    )

    y = torch.tensor(
        [s[1:SEQ_LEN + 1] for s in chunk],
        dtype=torch.long,
        device=DEVICE
    )

    batches.append((x, y))

print("Fixed batches:", len(batches))
print("Batch shape :", tuple(batches[0][0].shape))

# ==========================================================================================
# 5. VALIDATION DATA
# ==========================================================================================
#
# Use the validation texts recovered earlier if they are still in notebook memory.
# We deliberately avoid replacing them with training text.
# ==========================================================================================

if "validation_texts" not in globals():
    if "val_texts" in globals():
        validation_texts = val_texts

if "validation_texts" not in globals():
    raise RuntimeError(
        "validation_texts not found.\n"
        "Run the earlier validation-text recovery cell first so this test uses "
        "the same validation set as the 4.5097 baseline."
    )

print("Validation texts:", len(validation_texts))

# Use a deterministic subset for speed, matching the previous diagnostic scale.
VAL_N = min(500, len(validation_texts))
val_subset = validation_texts[:VAL_N]

# ==========================================================================================
# 6. VALIDATION LOSS
# ==========================================================================================

@torch.no_grad()
def validation_loss():
    model.eval()

    losses = []
    max_val_tokens = 1024

    for text in val_subset:
        try:
            ids = tokenizer.encode(text, add_special_tokens=False)
        except TypeError:
            ids = tokenizer.encode(text)

        if len(ids) < 2:
            continue

        # Evaluate first 1024 tokens, exactly to keep this diagnostic bounded.
        ids = ids[:max_val_tokens + 1]

        x = torch.tensor(
            [ids[:-1]],
            dtype=torch.long,
            device=DEVICE
        )

        y = torch.tensor(
            [ids[1:]],
            dtype=torch.long,
            device=DEVICE
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=(DEVICE == "cuda")
        ):
            out = model(x)

            logits = out.logits if hasattr(out, "logits") else out

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                y.reshape(-1)
            )

        if torch.isfinite(loss):
            losses.append(loss.item())

    model.train()

    if not losses:
        raise RuntimeError("Validation produced no usable losses.")

    return sum(losses) / len(losses)

# ==========================================================================================
# 7. GENERATION METRICS
# ==========================================================================================

@torch.no_grad()
def generation_metrics(prompt_text, temperature=0.9, max_new_tokens=128):
    model.eval()

    try:
        ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    except TypeError:
        ids = tokenizer.encode(prompt_text)

    ids = ids[-256:]

    x = torch.tensor([ids], dtype=torch.long, device=DEVICE)

    generated = []

    for _ in range(max_new_tokens):
        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=(DEVICE == "cuda")
        ):
            out = model(x)
            logits = out.logits if hasattr(out, "logits") else out

        next_logits = logits[:, -1, :] / temperature

        probs = torch.softmax(next_logits.float(), dim=-1)

        next_id = torch.multinomial(probs, 1)

        generated.append(next_id.item())

        x = torch.cat([x, next_id], dim=1)

    model.train()

    if not generated:
        return {}

    n = len(generated)
    unique_ratio = len(set(generated)) / n

    adjacent = sum(
        generated[i] == generated[i - 1]
        for i in range(1, n)
    ) / max(1, n - 1)

    pairs = list(zip(generated[:-1], generated[1:]))
    triples = list(zip(generated[:-2], generated[1:-1], generated[2:]))

    d2 = len(set(pairs)) / max(1, len(pairs))
    d3 = len(set(triples)) / max(1, len(triples))

    maxrun = 1
    run = 1

    for i in range(1, n):
        if generated[i] == generated[i - 1]:
            run += 1
            maxrun = max(maxrun, run)
        else:
            run = 1

    return {
        "unique": unique_ratio,
        "adj": adjacent,
        "d2": d2,
        "d3": d3,
        "maxrun": maxrun,
    }

# ==========================================================================================
# 8. BASELINE
# ==========================================================================================

print("\n" + "=" * 90)
print("BASELINE — EXACT STEP-1000 MODEL")
print("=" * 90)

baseline_loss = validation_loss()

print(f"Validation loss : {baseline_loss:.4f}")
print(f"Perplexity      : {math.exp(min(baseline_loss, 20)):.2f}")

# Use the first few validation documents as prompts.
eval_prompts = validation_texts[:5]

baseline_gen = generation_metrics(eval_prompts[0], temperature=0.9)

print(
    "T=0.9 | "
    f"unique={baseline_gen['unique']:.4f} | "
    f"adj={baseline_gen['adj']:.4f} | "
    f"d2={baseline_gen['d2']:.4f} | "
    f"d3={baseline_gen['d3']:.4f} | "
    f"maxrun={baseline_gen['maxrun']}"
)

# ==========================================================================================
# 9. OPTIMIZER
# ==========================================================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=0.0
)

model.train()

results = [{
    "point": "step1000",
    "step": 0,
    "validation_loss": baseline_loss,
    "perplexity": math.exp(min(baseline_loss, 20)),
    "generation": baseline_gen,
}]

# ==========================================================================================
# 10. 250-STEP CONTINUATION
# ==========================================================================================

print("\n" + "=" * 90)
print("250-STEP LR=5e-6 CONTINUATION")
print("=" * 90)

start_time = time.time()

for step, (x, y) in enumerate(batches, start=1):

    optimizer.zero_grad(set_to_none=True)

    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
        enabled=(DEVICE == "cuda")
    ):
        out = model(x)

        logits = out.logits if hasattr(out, "logits") else out

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            y.reshape(-1)
        )

    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=1.0
    )

    optimizer.step()

    if step == 1 or step % 25 == 0:
        print(
            f"step={step:4d} | "
            f"loss={loss.item():.4f} | "
            f"grad={float(grad_norm):.4f}"
        )

    if step in [50, 100, 150, 200, 250]:

        vloss = validation_loss()

        gen = generation_metrics(
            eval_prompts[0],
            temperature=0.9
        )

        ppl = math.exp(min(vloss, 20))

        results.append({
            "point": f"step+{step}",
            "step": step,
            "validation_loss": vloss,
            "perplexity": ppl,
            "generation": gen,
        })

        print("\n" + "-" * 90)
        print(f"DIAGNOSTIC CHECKPOINT — +{step} STEPS")
        print("-" * 90)

        print(f"Validation loss : {vloss:.4f}")
        print(f"Perplexity      : {ppl:.2f}")

        print(
            "T=0.9 | "
            f"unique={gen['unique']:.4f} | "
            f"adj={gen['adj']:.4f} | "
            f"d2={gen['d2']:.4f} | "
            f"d3={gen['d3']:.4f} | "
            f"maxrun={gen['maxrun']}"
        )

# ==========================================================================================
# 11. SUMMARY
# ==========================================================================================

elapsed = (time.time() - start_time) / 60

final = results[-1]

print("\n" + "=" * 90)
print("CS781 — LR=5e-6 CONTROLLED RECOVERY SUMMARY")
print("=" * 90)

print(f"Elapsed time : {elapsed:.2f} minutes")
print()

print(
    f"{'POINT':<12} "
    f"{'LOSS':>10} "
    f"{'PPL':>10} "
    f"{'UNIQUE':>10} "
    f"{'ADJ':>10} "
    f"{'D2':>10} "
    f"{'D3':>10} "
    f"{'MAXRUN':>8}"
)

print("-" * 90)

for r in results:
    g = r["generation"]

    print(
        f"{r['point']:<12} "
        f"{r['validation_loss']:>10.4f} "
        f"{r['perplexity']:>10.2f} "
        f"{g['unique']:>10.4f} "
        f"{g['adj']:>10.4f} "
        f"{g['d2']:>10.4f} "
        f"{g['d3']:>10.4f} "
        f"{g['maxrun']:>8}"
    )

loss_change = final["validation_loss"] - baseline_loss

print("\n" + "=" * 90)
print("DECISION SIGNAL")
print("=" * 90)

print(f"Baseline loss : {baseline_loss:.4f}")
print(f"Final loss    : {final['validation_loss']:.4f}")
print(f"Loss change   : {loss_change:+.4f}")

if loss_change < -0.15 and final["generation"]["maxrun"] <= 3:
    print("\n✓ PROMISING — LR=5e-6 shows meaningful recovery.")
    print("  Do NOT launch full training yet; use this result for the next controlled check.")
elif loss_change < -0.05:
    print("\n~ WEAK IMPROVEMENT — recovery signal exists but is not decisive.")
else:
    print("\n✗ NO CLEAR RECOVERY — do not use LR=5e-6 for production training yet.")

# ==========================================================================================
# 12. SAVE DIAGNOSTIC ONLY
# ==========================================================================================

result_path = os.path.join(
    OUT_DIR,
    "lr5e6_controlled_250_results.json"
)

with open(result_path, "w") as f:
    json.dump({
        "checkpoint": CKPT,
        "steps": STEPS,
        "seq_len": SEQ_LEN,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "results": results,
        "elapsed_minutes": elapsed,
    }, f, indent=2)

print("\nResults saved:")
print(result_path)

print("\n✓ Original Step-1000 checkpoint NOT modified")
print("✓ No production checkpoint written")
print("✓ Diagnostic run complete")

In [ ]:
# ==========================================================================================
# CS781 — LR=5e-6 + ORIGINAL GRADIENT ACCUMULATION CONTROLLED TEST
# Exact Step-1000 checkpoint | 250 OPTIMIZER steps | microbatch=1 | grad_accum=8
# NO checkpoint writing
# ==========================================================================================

import os, json, math, glob, time, random
import torch
import torch.nn.functional as F

print("=" * 90)
print("CS781 — LR=5e-6 + ORIGINAL GRADIENT ACCUMULATION CONTROLLED TEST")
print("=" * 90)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CKPT = "/kaggle/input/datasets/oipranesh/phase4-step1000-verified-model/checkpoint_step_001000.pt"
DATA_ROOT = "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep/dedup"

# ==========================================================================================
# ONLY EXPERIMENTAL SETTINGS CHANGED
# ==========================================================================================

OPTIMIZER_STEPS = 250
SEQ_LEN = 1024

MICRO_BATCH_SIZE = 1
GRAD_ACCUMULATION_STEPS = 8

LR = 5e-6
EVAL_EVERY = 50

TOTAL_MICROBATCHES = OPTIMIZER_STEPS * GRAD_ACCUMULATION_STEPS

OUT_DIR = "/kaggle/working/cs781_lr5e6_gradacc8_250"
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Device              : {DEVICE}")
print(f"Checkpoint          : {CKPT}")
print(f"Optimizer steps     : {OPTIMIZER_STEPS}")
print(f"SeqLen              : {SEQ_LEN}")
print(f"MicroBatch          : {MICRO_BATCH_SIZE}")
print(f"GradAccum           : {GRAD_ACCUMULATION_STEPS}")
print(f"Effective batch     : {MICRO_BATCH_SIZE * GRAD_ACCUMULATION_STEPS}")
print(f"LR                  : {LR}")
print(f"Total microbatches  : {TOTAL_MICROBATCHES}")

# ==========================================================================================
# 1. REUSE THE SAME MODEL + TOKENIZER FROM THE WORKING CELL
# ==========================================================================================

if "model" not in globals():
    raise RuntimeError(
        "Model object not found. Run the original model-definition/loading cell first."
    )

if "tokenizer" not in globals():
    raise RuntimeError(
        "Tokenizer object not found. Run the original tokenizer-loading cell first."
    )

model = model.to(DEVICE)

# ==========================================================================================
# 2. LOAD EXACT STEP-1000 WEIGHTS
# ==========================================================================================

print("\n" + "=" * 90)
print("LOADING EXACT STEP-1000 CHECKPOINT")
print("=" * 90)

ckpt = torch.load(CKPT, map_location="cpu")

state = ckpt["model_state_dict"]

missing, unexpected = model.load_state_dict(state, strict=False)

print("Checkpoint step :", ckpt.get("step"))
print("Checkpoint LR   :", ckpt.get("learning_rate"))
print("Context length  :", ckpt.get("context_length"))
print("Missing keys    :", len(missing))
print("Unexpected keys :", len(unexpected))

if missing or unexpected:
    raise RuntimeError(
        f"Checkpoint mismatch: missing={missing[:10]}, unexpected={unexpected[:10]}"
    )

print("✓ Exact Step-1000 weights loaded")
print("✓ Original checkpoint untouched")

# ==========================================================================================
# 3. LOAD TRAINING TEXT — SAME WORKING LOGIC
# ==========================================================================================

print("\n" + "=" * 90)
print("LOADING TRAINING DATA")
print("=" * 90)

jsonl_files = sorted(glob.glob(DATA_ROOT + "/**/*.jsonl", recursive=True))

print("Training JSONL files:", len(jsonl_files))

if not jsonl_files:
    raise RuntimeError("No training JSONL files found.")

texts = []

for path in jsonl_files:
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                except Exception:
                    continue

                text = None

                if isinstance(obj, str):
                    text = obj

                elif isinstance(obj, dict):
                    for key in [
                        "text",
                        "content",
                        "document",
                        "body",
                        "raw_text",
                    ]:
                        if isinstance(obj.get(key), str) and obj[key].strip():
                            text = obj[key]
                            break

                if text and len(text.strip()) >= 100:
                    texts.append(text.strip())

                    if len(texts) >= 4000:
                        break

        if len(texts) >= 4000:
            break

    except Exception as e:
        print("Skipping:", path, "|", repr(e))

print("Training texts recovered:", len(texts))

if len(texts) < 500:
    raise RuntimeError("Too few training texts recovered.")

# ==========================================================================================
# 4. TOKENIZE
# ==========================================================================================

print("\nTokenizing training sample...")

sequences = []

for text in texts:
    try:
        ids = tokenizer.encode(text, add_special_tokens=False)
    except TypeError:
        ids = tokenizer.encode(text)

    if len(ids) >= SEQ_LEN + 1:
        sequences.append(ids)

print("Usable token sequences:", len(sequences))

required_sequences = TOTAL_MICROBATCHES * MICRO_BATCH_SIZE

if len(sequences) < required_sequences:
    raise RuntimeError(
        f"Not enough usable sequences: {len(sequences)} "
        f"for {required_sequences} required microbatches."
    )

# ==========================================================================================
# 5. DETERMINISTIC MICRO-BATCHES
# ==========================================================================================

random.seed(781)
random.shuffle(sequences)

batches = []

for i in range(TOTAL_MICROBATCHES):
    chunk = sequences[
        i * MICRO_BATCH_SIZE:(i + 1) * MICRO_BATCH_SIZE
    ]

    x = torch.tensor(
        [s[:SEQ_LEN] for s in chunk],
        dtype=torch.long,
        device=DEVICE,
    )

    y = torch.tensor(
        [s[1:SEQ_LEN + 1] for s in chunk],
        dtype=torch.long,
        device=DEVICE,
    )

    batches.append((x, y))

print("Fixed microbatches:", len(batches))
print("Microbatch shape :", tuple(batches[0][0].shape))

# ==========================================================================================
# 6. VALIDATION DATA — EXACT SAME SOURCE AS WORKING CELL
# ==========================================================================================

if "validation_texts" not in globals():
    if "val_texts" in globals():
        validation_texts = val_texts

if "validation_texts" not in globals():
    raise RuntimeError(
        "validation_texts not found.\n"
        "Run the earlier validation-text recovery cell first so this test uses "
        "the same validation set as the 4.5097 baseline."
    )

print("Validation texts:", len(validation_texts))

VAL_N = min(500, len(validation_texts))
val_subset = validation_texts[:VAL_N]

# ==========================================================================================
# 7. VALIDATION LOSS
# ==========================================================================================

@torch.no_grad()
def validation_loss():
    model.eval()

    losses = []
    max_val_tokens = 1024

    for text in val_subset:
        try:
            ids = tokenizer.encode(text, add_special_tokens=False)
        except TypeError:
            ids = tokenizer.encode(text)

        if len(ids) < 2:
            continue

        ids = ids[:max_val_tokens + 1]

        x = torch.tensor(
            [ids[:-1]],
            dtype=torch.long,
            device=DEVICE,
        )

        y = torch.tensor(
            [ids[1:]],
            dtype=torch.long,
            device=DEVICE,
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=(DEVICE == "cuda"),
        ):
            out = model(x)
            logits = out.logits if hasattr(out, "logits") else out

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                y.reshape(-1),
            )

        if torch.isfinite(loss):
            losses.append(loss.item())

    model.train()

    if not losses:
        raise RuntimeError("Validation produced no usable losses.")

    return sum(losses) / len(losses)

# ==========================================================================================
# 8. GENERATION METRICS
# ==========================================================================================

@torch.no_grad()
def generation_metrics(prompt_text, temperature=0.9, max_new_tokens=128):
    model.eval()

    try:
        ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    except TypeError:
        ids = tokenizer.encode(prompt_text)

    ids = ids[-256:]

    x = torch.tensor([ids], dtype=torch.long, device=DEVICE)

    generated = []

    for _ in range(max_new_tokens):
        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=(DEVICE == "cuda"),
        ):
            out = model(x)
            logits = out.logits if hasattr(out, "logits") else out

        next_logits = logits[:, -1, :] / temperature
        probs = torch.softmax(next_logits.float(), dim=-1)

        next_id = torch.multinomial(probs, 1)

        generated.append(next_id.item())
        x = torch.cat([x, next_id], dim=1)

    model.train()

    if not generated:
        return {}

    n = len(generated)

    unique_ratio = len(set(generated)) / n

    adjacent = sum(
        generated[i] == generated[i - 1]
        for i in range(1, n)
    ) / max(1, n - 1)

    pairs = list(zip(generated[:-1], generated[1:]))
    triples = list(
        zip(
            generated[:-2],
            generated[1:-1],
            generated[2:],
        )
    )

    d2 = len(set(pairs)) / max(1, len(pairs))
    d3 = len(set(triples)) / max(1, len(triples))

    maxrun = 1
    run = 1

    for i in range(1, n):
        if generated[i] == generated[i - 1]:
            run += 1
            maxrun = max(maxrun, run)
        else:
            run = 1

    return {
        "unique": unique_ratio,
        "adj": adjacent,
        "d2": d2,
        "d3": d3,
        "maxrun": maxrun,
    }

# ==========================================================================================
# 9. BASELINE
# ==========================================================================================

print("\n" + "=" * 90)
print("BASELINE — EXACT STEP-1000 MODEL")
print("=" * 90)

baseline_loss = validation_loss()

print(f"Validation loss : {baseline_loss:.4f}")
print(f"Perplexity      : {math.exp(min(baseline_loss, 20)):.2f}")

eval_prompts = validation_texts[:5]

baseline_gen = generation_metrics(
    eval_prompts[0],
    temperature=0.9,
)

print(
    "T=0.9 | "
    f"unique={baseline_gen['unique']:.4f} | "
    f"adj={baseline_gen['adj']:.4f} | "
    f"d2={baseline_gen['d2']:.4f} | "
    f"d3={baseline_gen['d3']:.4f} | "
    f"maxrun={baseline_gen['maxrun']}"
)

# ==========================================================================================
# 10. OPTIMIZER
# ==========================================================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=0.0,
)

model.train()

results = [{
    "point": "step1000",
    "step": 0,
    "validation_loss": baseline_loss,
    "perplexity": math.exp(min(baseline_loss, 20)),
    "generation": baseline_gen,
}]

# ==========================================================================================
# 11. 250 OPTIMIZER-STEP CONTINUATION
#     1 microbatch × 8 accumulation = effective batch 8
# ==========================================================================================

print("\n" + "=" * 90)
print("250-STEP LR=5e-6 + GRAD_ACCUM=8 CONTINUATION")
print("=" * 90)

start_time = time.time()

optimizer.zero_grad(set_to_none=True)

micro_step = 0

for optimizer_step in range(1, OPTIMIZER_STEPS + 1):

    accumulated_loss = 0.0
    last_grad_norm = None

    for accum_step in range(GRAD_ACCUMULATION_STEPS):

        micro_step += 1

        x, y = batches[micro_step - 1]

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=(DEVICE == "cuda"),
        ):
            out = model(x)

            logits = out.logits if hasattr(out, "logits") else out

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                y.reshape(-1),
            )

        accumulated_loss += loss.item()

        # IMPORTANT:
        # Divide before backward so accumulated gradient is the
        # mean gradient across the 8 microbatches.
        (loss / GRAD_ACCUMULATION_STEPS).backward()

    last_grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=1.0,
    )

    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    mean_micro_loss = accumulated_loss / GRAD_ACCUMULATION_STEPS

    if (
        optimizer_step == 1
        or optimizer_step % 25 == 0
    ):
        print(
            f"step={optimizer_step:4d} | "
            f"loss={mean_micro_loss:.4f} | "
            f"grad={float(last_grad_norm):.4f}"
        )

    # ------------------------------------------------------------------
    # Diagnostic checkpoints
    # ------------------------------------------------------------------

    if optimizer_step in [50, 100, 150, 200, 250]:

        vloss = validation_loss()

        gen = generation_metrics(
            eval_prompts[0],
            temperature=0.9,
        )

        ppl = math.exp(min(vloss, 20))

        results.append({
            "point": f"step+{optimizer_step}",
            "step": optimizer_step,
            "validation_loss": vloss,
            "perplexity": ppl,
            "generation": gen,
        })

        print("\n" + "-" * 90)
        print(
            f"DIAGNOSTIC CHECKPOINT — "
            f"+{optimizer_step} OPTIMIZER STEPS"
        )
        print("-" * 90)

        print(f"Validation loss : {vloss:.4f}")
        print(f"Perplexity      : {ppl:.2f}")

        print(
            "T=0.9 | "
            f"unique={gen['unique']:.4f} | "
            f"adj={gen['adj']:.4f} | "
            f"d2={gen['d2']:.4f} | "
            f"d3={gen['d3']:.4f} | "
            f"maxrun={gen['maxrun']}"
        )

# ==========================================================================================
# 12. SUMMARY
# ==========================================================================================

elapsed = (time.time() - start_time) / 60

final = results[-1]

print("\n" + "=" * 90)
print("CS781 — LR=5e-6 + GRAD_ACCUM=8 RECOVERY SUMMARY")
print("=" * 90)

print(f"Elapsed time : {elapsed:.2f} minutes")
print()

print(
    f"{'POINT':<18} "
    f"{'LOSS':>10} "
    f"{'PPL':>10} "
    f"{'UNIQUE':>10} "
    f"{'ADJ':>10} "
    f"{'D2':>10} "
    f"{'D3':>10} "
    f"{'MAXRUN':>8}"
)

print("-" * 90)

for r in results:
    g = r["generation"]

    print(
        f"{r['point']:<18} "
        f"{r['validation_loss']:>10.4f} "
        f"{r['perplexity']:>10.2f} "
        f"{g['unique']:>10.4f} "
        f"{g['adj']:>10.4f} "
        f"{g['d2']:>10.4f} "
        f"{g['d3']:>10.4f} "
        f"{g['maxrun']:>8}"
    )

loss_change = final["validation_loss"] - baseline_loss

print("\n" + "=" * 90)
print("DECISION SIGNAL")
print("=" * 90)

print(f"Baseline loss : {baseline_loss:.4f}")
print(f"Final loss    : {final['validation_loss']:.4f}")
print(f"Loss change   : {loss_change:+.4f}")

if loss_change < -0.15 and final["generation"]["maxrun"] <= 3:
    print("\n✓ PROMISING — LR=5e-6 + GradAccum=8 shows meaningful recovery.")
    print("  Do NOT launch full training yet; use this result for the next controlled check.")

elif loss_change < -0.05:
    print("\n~ WEAK IMPROVEMENT — recovery signal exists but is not decisive.")

else:
    print("\n✗ NO CLEAR RECOVERY — do not use this configuration for production training yet.")

# ==========================================================================================
# 13. SAVE DIAGNOSTIC ONLY
# ==========================================================================================

result_path = os.path.join(
    OUT_DIR,
    "lr5e6_gradacc8_controlled_250_results.json",
)

with open(result_path, "w") as f:
    json.dump(
        {
            "checkpoint": CKPT,
            "optimizer_steps": OPTIMIZER_STEPS,
            "micro_batch_size": MICRO_BATCH_SIZE,
            "gradient_accumulation_steps": GRAD_ACCUMULATION_STEPS,
            "effective_batch_size": (
                MICRO_BATCH_SIZE * GRAD_ACCUMULATION_STEPS
            ),
            "seq_len": SEQ_LEN,
            "learning_rate": LR,
            "results": results,
            "elapsed_minutes": elapsed,
        },
        f,
        indent=2,
    )

print("\nResults saved:")
print(result_path)

print("\n✓ Original Step-1000 checkpoint NOT modified")
print("✓ No production checkpoint written")
print("✓ 250 optimizer steps completed")
print("✓ Effective batch size = 8")
print("✓ Diagnostic run complete")

In [ ]:
# ==========================================================================================
# CS781 — NOTEBOOK STATE DIAGNOSTIC
# PURPOSE:
#   Determine exactly why the previous experiment cells keep failing.
#   NO TRAINING
#   NO MODEL MODIFICATION
#   NO CHECKPOINT WRITES
# ==========================================================================================

import os
import sys
import gc
import torch

print("=" * 100)
print("CS781 — NOTEBOOK STATE DIAGNOSTIC")
print("=" * 100)

# ------------------------------------------------------------------------------------------
# 1. PYTHON / GPU STATE
# ------------------------------------------------------------------------------------------

print("\n[1] RUNTIME")
print("-" * 100)
print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------------------------------------
# 2. IMPORTANT VARIABLES ALREADY IN KERNEL
# ------------------------------------------------------------------------------------------

print("\n[2] EXISTING IMPORTANT VARIABLES")
print("-" * 100)

important_names = [
    "model",
    "tokenizer",
    "raw_tokenizer",
    "tokenizer_path",
    "TOKENIZER_PATH",
    "TOKENIZER_FILE",
    "tokenizer_config",
    "tokenizer_config_path",
    "model_path",
    "MODEL_PATH",
    "checkpoint_path",
    "CHECKPOINT_PATH",
    "checkpoint",
    "ckpt",
    "train_texts",
    "texts",
    "token_sequences",
    "tokenized",
    "GEN_TOKENS",
]

for name in important_names:
    exists = name in globals()

    if not exists:
        print(f"{name:28s}: NOT DEFINED")
        continue

    obj = globals()[name]

    if name in ["model", "tokenizer", "raw_tokenizer"]:
        print(f"{name:28s}: {type(obj).__name__} | is None = {obj is None}")

    elif isinstance(obj, (str, os.PathLike)):
        print(f"{name:28s}: {obj}")

    elif isinstance(obj, (list, tuple, dict)):
        print(f"{name:28s}: {type(obj).__name__} | len = {len(obj)}")

    else:
        print(f"{name:28s}: {type(obj).__name__}")

# ------------------------------------------------------------------------------------------
# 3. MODEL OBJECT VALIDITY
# ------------------------------------------------------------------------------------------

print("\n[3] MODEL OBJECT CHECK")
print("-" * 100)

if "model" not in globals():
    print("MODEL STATUS: ❌ variable 'model' does not exist")

elif model is None:
    print("MODEL STATUS: ❌ variable 'model' exists but is None")
    print("CAUSE: the previous model-loading step did not leave a usable model object.")

else:
    print("MODEL STATUS: ✓ model object exists")
    print("Model type:", type(model).__name__)

    try:
        print("Parameter count:", sum(p.numel() for p in model.parameters()))
        print("First parameter device:", next(model.parameters()).device)
    except Exception as e:
        print("Could not inspect model parameters:", repr(e))

# ------------------------------------------------------------------------------------------
# 4. TOKENIZER OBJECT VALIDITY
# ------------------------------------------------------------------------------------------

print("\n[4] TOKENIZER OBJECT CHECK")
print("-" * 100)

if "tokenizer" not in globals():
    print("TOKENIZER STATUS: ❌ variable 'tokenizer' does not exist")

elif tokenizer is None:
    print("TOKENIZER STATUS: ❌ variable 'tokenizer' exists but is None")
    print("CAUSE: tokenizer was not instantiated in the current kernel.")

else:
    print("TOKENIZER STATUS: ✓ tokenizer object exists")
    print("Tokenizer type:", type(tokenizer).__name__)

    # Try several common interfaces without modifying anything.
    checks = [
        ("vocab_size", lambda: tokenizer.vocab_size),
        ("get_vocab()", lambda: len(tokenizer.get_vocab())),
        ("encode('test')", lambda: tokenizer.encode("test")),
    ]

    for label, fn in checks:
        try:
            value = fn()
            print(f"{label:24s}: ✓ {value}")
        except Exception as e:
            print(f"{label:24s}: ✗ {type(e).__name__}: {e}")

# ------------------------------------------------------------------------------------------
# 5. SEARCH KERNEL VARIABLES FOR TOKENIZER-LIKE OBJECTS
# ------------------------------------------------------------------------------------------

print("\n[5] TOKENIZER-LIKE OBJECTS CURRENTLY IN KERNEL")
print("-" * 100)

found_tokenizer_objects = []

for name, obj in list(globals().items()):
    if name.startswith("_"):
        continue

    try:
        if obj is None:
            continue

        # Avoid dumping giant objects.
        obj_type = type(obj).__name__.lower()
        obj_name = name.lower()

        if (
            "tokenizer" in obj_name
            or "tokenizer" in obj_type
            or (
                hasattr(obj, "encode")
                and hasattr(obj, "decode")
            )
        ):
            found_tokenizer_objects.append((name, type(obj).__name__))

    except Exception:
        pass

if found_tokenizer_objects:
    for name, typ in found_tokenizer_objects:
        print(f"{name:35s} -> {typ}")
else:
    print("❌ No tokenizer-like object found in current kernel.")

# ------------------------------------------------------------------------------------------
# 6. SEARCH KERNEL VARIABLES FOR MODEL-LIKE OBJECTS
# ------------------------------------------------------------------------------------------

print("\n[6] MODEL-LIKE OBJECTS CURRENTLY IN KERNEL")
print("-" * 100)

found_model_objects = []

for name, obj in list(globals().items()):
    if name.startswith("_"):
        continue

    try:
        if obj is None:
            continue

        obj_type = type(obj).__name__.lower()

        if (
            name.lower() == "model"
            or "model" in name.lower()
            or "model" in obj_type
            or (
                hasattr(obj, "parameters")
                and hasattr(obj, "state_dict")
            )
        ):
            found_model_objects.append((name, type(obj).__name__))

    except Exception:
        pass

if found_model_objects:
    for name, typ in found_model_objects:
        print(f"{name:35s} -> {typ}")
else:
    print("❌ No model-like object found in current kernel.")

# ------------------------------------------------------------------------------------------
# 7. CHECK THE KNOWN CHECKPOINT
# ------------------------------------------------------------------------------------------

print("\n[7] KNOWN STEP-1000 CHECKPOINT")
print("-" * 100)

KNOWN_CKPT = (
    "/kaggle/input/datasets/oipranesh/"
    "phase4-step1000-verified-model/"
    "checkpoint_step_001000.pt"
)

print("Path:", KNOWN_CKPT)
print("Exists:", os.path.exists(KNOWN_CKPT))

if os.path.exists(KNOWN_CKPT):
    try:
        size_mb = os.path.getsize(KNOWN_CKPT) / (1024 ** 2)
        print(f"Size: {size_mb:.2f} MB")
    except Exception as e:
        print("Could not read file size:", repr(e))

# ------------------------------------------------------------------------------------------
# 8. SEARCH KNOWN CS781 DATASET LOCATIONS
# ------------------------------------------------------------------------------------------

print("\n[8] KNOWN CS781 DATA LOCATIONS")
print("-" * 100)

KNOWN_ROOTS = [
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep",
    "/kaggle/input/datasets/oipranesh/phase4-step1000-verified-model",
]

for root in KNOWN_ROOTS:
    print(f"\n{root}")
    print("Exists:", os.path.exists(root))

    if os.path.exists(root):
        try:
            entries = os.listdir(root)
            print("Entries:", len(entries))
            for x in entries[:20]:
                print("  ", x)
            if len(entries) > 20:
                print("  ...")
        except Exception as e:
            print("Could not list:", repr(e))

# ------------------------------------------------------------------------------------------
# 9. CHECK FOR TOKENIZER FILES WITHOUT MODIFYING ANYTHING
# ------------------------------------------------------------------------------------------

print("\n[9] TOKENIZER FILE SEARCH")
print("-" * 100)

search_roots = [
    "/kaggle/input/datasets/oipranesh",
    "/kaggle/working",
]

tokenizer_hits = []

for root in search_roots:
    if not os.path.exists(root):
        continue

    for dirpath, dirnames, filenames in os.walk(root):
        # Prevent accidental traversal into irrelevant huge directories.
        dirnames[:] = [
            d for d in dirnames
            if not d.startswith(".")
        ]

        for filename in filenames:
            low = filename.lower()

            if any(key in low for key in [
                "tokenizer",
                "tokenizer.json",
                "tokenizer.model",
                "vocab.json",
                "merges.txt",
                "sentencepiece",
            ]):
                tokenizer_hits.append(
                    os.path.join(dirpath, filename)
                )

        # Keep diagnostic bounded.
        if len(tokenizer_hits) >= 100:
            break

    if len(tokenizer_hits) >= 100:
        break

if tokenizer_hits:
    for p in tokenizer_hits[:100]:
        print(" ", p)
else:
    print("❌ No obvious tokenizer files found in searched locations.")

# ------------------------------------------------------------------------------------------
# 10. FINAL DIAGNOSIS
# ------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("DIAGNOSIS")
print("=" * 100)

model_ok = (
    "model" in globals()
    and model is not None
    and hasattr(model, "parameters")
)

tokenizer_ok = (
    "tokenizer" in globals()
    and tokenizer is not None
    and hasattr(tokenizer, "encode")
)

checkpoint_ok = os.path.exists(KNOWN_CKPT)

print(f"Model object available    : {'✓ YES' if model_ok else '❌ NO'}")
print(f"Tokenizer object available: {'✓ YES' if tokenizer_ok else '❌ NO'}")
print(f"Step-1000 checkpoint       : {'✓ YES' if checkpoint_ok else '❌ NO'}")

print("\nIMPORTANT:")
if not model_ok and not tokenizer_ok:
    print("→ Current kernel does NOT contain the working model/tokenizer state.")
    print("→ The previous experiment succeeded because its loading cell created those objects.")
    print("→ We should recover/re-run THAT exact loading path, not invent a new loader.")

elif not model_ok:
    print("→ Tokenizer exists, but model does not.")
    print("→ Only the model-loading state needs to be recovered.")

elif not tokenizer_ok:
    print("→ Model exists, but tokenizer does not.")
    print("→ Only the tokenizer-loading state needs to be recovered.")

else:
    print("→ Both model and tokenizer are available.")
    print("→ The next experiment can reuse them directly.")

print("=" * 100)
print("NO TRAINING PERFORMED")
print("NO CHECKPOINT MODIFIED")
print("NO FILES WRITTEN")
print("=" * 100)

In [ ]:
# ==========================================================================================
# CS781 — SAFE MODEL OBJECT RECOVERY
# ==========================================================================================
# PURPOSE:
#   Recover the missing `model` Python object using the EXACT architecture/configuration
#   already present in the current notebook and the already-loaded Step-1000 checkpoint.
#
# NO TRAINING
# NO OPTIMIZER STEP
# NO CHECKPOINT WRITE
# NO TOKENIZER CHANGE
# ==========================================================================================

import torch
import copy

print("=" * 100)
print("CS781 — SAFE MODEL OBJECT RECOVERY")
print("=" * 100)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------------------------------------
# 1. Confirm current state
# ------------------------------------------------------------------------------------------

print("\n[1] CURRENT STATE")
print("-" * 100)

print("tokenizer :", type(tokenizer).__name__ if tokenizer is not None else None)
print("checkpoint:", type(checkpoint).__name__ if checkpoint is not None else None)
print("ckpt      :", type(ckpt).__name__ if ckpt is not None else None)
print("diag_model:", type(diag_model).__name__ if diag_model is not None else None)

if model is not None:
    print("\n✓ `model` already exists.")
    print("Type:", type(model).__name__)
    print("Device:", next(model.parameters()).device)

else:
    print("\n`model` is None — recovering it.")

# ------------------------------------------------------------------------------------------
# 2. Prefer the already-created diagnostic model
# ------------------------------------------------------------------------------------------

if model is None:

    if "diag_model" not in globals() or diag_model is None:
        raise RuntimeError(
            "diag_model is also unavailable. STOP here — do not run an experiment. "
            "The original model-construction cell needs to be recovered."
        )

    print("\n[2] USING EXISTING MODEL ARCHITECTURE")
    print("-" * 100)

    print("Existing model class :", type(diag_model).__name__)

    # We make a fresh copy of the already-created architecture object.
    # This avoids altering diag_model itself.
    model = copy.deepcopy(diag_model)

    print("✓ Fresh model object created from existing architecture.")
    print("Type:", type(model).__name__)

# ------------------------------------------------------------------------------------------
# 3. Locate exact Step-1000 state dict
# ------------------------------------------------------------------------------------------

print("\n[3] CHECKPOINT STATE")
print("-" * 100)

if "checkpoint" in globals() and isinstance(checkpoint, dict):
    source_ckpt = checkpoint
elif "ckpt" in globals() and isinstance(ckpt, dict):
    source_ckpt = ckpt
else:
    raise RuntimeError(
        "No loaded checkpoint dictionary found. "
        "Do not proceed until the checkpoint-loading cell is recovered."
    )

print("Checkpoint keys:")
for k in source_ckpt.keys():
    print(" ", k)

if "model_state_dict" not in source_ckpt:
    raise RuntimeError(
        "Loaded checkpoint does not contain `model_state_dict`."
    )

state_dict = source_ckpt["model_state_dict"]

print("\nState-dict tensors:", len(state_dict))

# ------------------------------------------------------------------------------------------
# 4. Verify architecture compatibility BEFORE loading
# ------------------------------------------------------------------------------------------

print("\n[4] ARCHITECTURE COMPATIBILITY CHECK")
print("-" * 100)

model_state = model.state_dict()

model_keys = set(model_state.keys())
ckpt_keys = set(state_dict.keys())

missing_from_ckpt = sorted(model_keys - ckpt_keys)
unexpected_in_ckpt = sorted(ckpt_keys - model_keys)

shape_mismatches = []

for key in sorted(model_keys & ckpt_keys):
    if tuple(model_state[key].shape) != tuple(state_dict[key].shape):
        shape_mismatches.append(
            (
                key,
                tuple(model_state[key].shape),
                tuple(state_dict[key].shape),
            )
        )

print("Model parameters/buffers :", len(model_keys))
print("Checkpoint tensors       :", len(ckpt_keys))
print("Missing keys             :", len(missing_from_ckpt))
print("Unexpected keys          :", len(unexpected_in_ckpt))
print("Shape mismatches         :", len(shape_mismatches))

if missing_from_ckpt:
    print("\nFirst missing keys:")
    for k in missing_from_ckpt[:10]:
        print(" ", k)

if unexpected_in_ckpt:
    print("\nFirst unexpected keys:")
    for k in unexpected_in_ckpt[:10]:
        print(" ", k)

if shape_mismatches:
    print("\nFirst shape mismatches:")
    for item in shape_mismatches[:10]:
        print(" ", item)

if missing_from_ckpt or unexpected_in_ckpt or shape_mismatches:
    raise RuntimeError(
        "STOP: model architecture does NOT exactly match Step-1000 checkpoint. "
        "No weights were loaded."
    )

print("✓ Architecture matches checkpoint exactly.")

# ------------------------------------------------------------------------------------------
# 5. Load exact Step-1000 weights
# ------------------------------------------------------------------------------------------

print("\n[5] LOADING EXACT STEP-1000 WEIGHTS")
print("-" * 100)

load_result = model.load_state_dict(state_dict, strict=True)

print("load_state_dict result:", load_result)
print("✓ Exact Step-1000 weights loaded.")

# ------------------------------------------------------------------------------------------
# 6. Move to GPU
# ------------------------------------------------------------------------------------------

print("\n[6] DEVICE")
print("-" * 100)

model = model.to(DEVICE)
model.eval()

print("Device:", next(model.parameters()).device)
print("Training mode:", model.training)

# ------------------------------------------------------------------------------------------
# 7. Final verification
# ------------------------------------------------------------------------------------------

print("\n[7] FINAL VERIFICATION")
print("-" * 100)

parameter_count = sum(p.numel() for p in model.parameters())

print("Model type     :", type(model).__name__)
print("Parameters     :", parameter_count)
print("Device         :", next(model.parameters()).device)
print("Eval mode      :", not model.training)

# Verify a forward pass using tokenizer output only.
try:
    test_encoding = tokenizer.encode("This is a CS781 verification test.")

    input_ids = torch.tensor(
        [test_encoding.ids],
        dtype=torch.long,
        device=DEVICE,
    )

    with torch.no_grad():
        output = model(input_ids=input_ids)

    print("Forward pass   : ✓")
    print("Logits shape   :", tuple(output.logits.shape))

except Exception as e:
    raise RuntimeError(
        f"Model object recovered, but verification forward pass failed: {e}"
    )

print("\n" + "=" * 100)
print("✓ MODEL RECOVERY SUCCESSFUL")
print("=" * 100)
print("Tokenizer      : existing tokenizer reused")
print("Architecture   : existing diag_model architecture reused")
print("Weights        : exact Step-1000 checkpoint")
print("Training       : NOT performed")
print("Checkpoint     : NOT modified")
print("Optimizer      : NOT touched")
print("=" * 100)

In [ ]:
# ============================================================================
# CS781 — LR=5e-6 + ORIGINAL GRADIENT ACCUMULATION
# CONTROLLED RECOVERY EXPERIMENT
#
# IMPORTANT:
# - Reuses the ALREADY RECOVERED `model`
# - Reuses the ALREADY WORKING `tokenizer`
# - Loads EXACT Step-1000 weights again for a clean branch
# - Does NOT modify the input checkpoint
# - Does NOT write a production checkpoint
# - MicroBatch=1, GradAccum=8 (original training configuration)
# ============================================================================

import os
import json
import math
import glob
import copy
import time
import torch
import torch.nn.functional as F

print("=" * 100)
print("CS781 — LR=5e-6 + ORIGINAL GRADIENT ACCUMULATION CONTROLLED TEST")
print("=" * 100)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_PATH = (
    "/kaggle/input/datasets/oipranesh/"
    "phase4-step1000-verified-model/checkpoint_step_001000.pt"
)

DATA_ROOT = (
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep/dedup"
)

OUTPUT_DIR = "/kaggle/working/cs781_lr5e6_accum8_controlled"
os.makedirs(OUTPUT_DIR, exist_ok=True)

STEPS = 250
SEQ_LEN = 1024
MICRO_BATCH = 1
GRAD_ACCUM = 8
LR = 5e-6

print(f"Device         : {DEVICE}")
print(f"Checkpoint     : {CHECKPOINT_PATH}")
print(f"Steps          : {STEPS}")
print(f"SeqLen         : {SEQ_LEN}")
print(f"MicroBatch     : {MICRO_BATCH}")
print(f"GradAccum      : {GRAD_ACCUM}")
print(f"Effective batch: {MICRO_BATCH * GRAD_ACCUM}")
print(f"LR             : {LR}")

# ============================================================================
# 1. VERIFY RECOVERED OBJECTS
# ============================================================================

print("\n" + "=" * 100)
print("VERIFYING RECOVERED NOTEBOOK STATE")
print("=" * 100)

if model is None:
    raise RuntimeError(
        "model is None. Run the SAFE MODEL OBJECT RECOVERY cell first."
    )

if tokenizer is None:
    raise RuntimeError(
        "tokenizer is None. Run the previous successful tokenizer-loading cell."
    )

print("✓ model       :", type(model).__name__)
print("✓ tokenizer   :", type(tokenizer).__name__)
print("✓ checkpoint  :", type(checkpoint).__name__)
print("✓ model ready :", True)

# ============================================================================
# 2. LOAD EXACT STEP-1000 WEIGHTS INTO A CLEAN BRANCH
# ============================================================================

print("\n" + "=" * 100)
print("LOADING EXACT STEP-1000 WEIGHTS")
print("=" * 100)

ckpt = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint step :", ckpt["step"])
print("Checkpoint LR   :", ckpt["learning_rate"])
print("Context length  :", ckpt["context_length"])

# Make a fresh architecture copy from the recovered model.
# This prevents this diagnostic from altering the recovered object.
test_model = copy.deepcopy(model)

missing, unexpected = test_model.load_state_dict(
    ckpt["model_state_dict"],
    strict=False
)

print("Missing keys    :", len(missing))
print("Unexpected keys :", len(unexpected))

if missing or unexpected:
    raise RuntimeError(
        f"Checkpoint/model mismatch: missing={missing}, unexpected={unexpected}"
    )

test_model = test_model.to(DEVICE)
test_model.train()

print("✓ Exact Step-1000 weights loaded")
print("✓ Original checkpoint untouched")
print("✓ Recovered notebook model untouched")

# ============================================================================
# 3. FIND THE SAME TRAINING JSONL DATA
# ============================================================================

print("\n" + "=" * 100)
print("LOADING TRAINING DATA")
print("=" * 100)

jsonl_files = sorted(
    glob.glob(os.path.join(DATA_ROOT, "**", "*.jsonl"), recursive=True)
)

print("Training JSONL files:", len(jsonl_files))

if not jsonl_files:
    raise RuntimeError(
        f"No JSONL files found under {DATA_ROOT}"
    )

# Same robust extraction approach as the successful diagnostic.
texts = []

def extract_text(obj):
    if isinstance(obj, str):
        return obj

    if isinstance(obj, dict):
        for key in (
            "text",
            "content",
            "document",
            "body",
            "raw_text",
            "input",
        ):
            value = obj.get(key)
            if isinstance(value, str) and value.strip():
                return value

        # Recursive fallback
        for value in obj.values():
            result = extract_text(value)
            if result:
                return result

    if isinstance(obj, list):
        for value in obj:
            result = extract_text(value)
            if result:
                return result

    return None


MAX_TEXTS = 4000

for path in jsonl_files:
    if len(texts) >= MAX_TEXTS:
        break

    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if len(texts) >= MAX_TEXTS:
                    break

                line = line.strip()
                if not line:
                    continue

                try:
                    obj = json.loads(line)
                except Exception:
                    continue

                text = extract_text(obj)

                if text and len(text.strip()) >= 100:
                    texts.append(text.strip())

    except Exception as e:
        print("Skipping:", path, "|", repr(e))

print("Training texts recovered:", len(texts))

if len(texts) < 100:
    raise RuntimeError("Too few training texts recovered.")

# ============================================================================
# 4. TOKENIZE
# ============================================================================

print("\nTokenizing training samples...")

token_sequences = []

for text in texts:
    try:
        enc = tokenizer.encode(text)
        ids = list(enc.ids)

        # Need at least SEQ_LEN + 1 for causal LM
        if len(ids) >= SEQ_LEN + 1:
            token_sequences.append(ids)
    except Exception:
        continue

print("Usable token sequences:", len(token_sequences))

if len(token_sequences) < STEPS * GRAD_ACCUM:
    raise RuntimeError(
        f"Not enough token sequences for {STEPS} steps × "
        f"{GRAD_ACCUM} microbatches."
    )

# ============================================================================
# 5. FIXED MICRO-BATCHES
# ============================================================================

def make_batch(seq):
    x = torch.tensor(
        seq[:SEQ_LEN],
        dtype=torch.long
    )

    y = torch.tensor(
        seq[1:SEQ_LEN + 1],
        dtype=torch.long
    )

    return x, y


# Deterministic fixed sequence pool.
# This is deliberately simple: same data style, no random data changes.
micro_batches = []

for i in range(STEPS * GRAD_ACCUM):
    seq = token_sequences[i]
    x, y = make_batch(seq)

    micro_batches.append(
        (
            x.unsqueeze(0),
            y.unsqueeze(0)
        )
    )

print("Fixed micro-batches:", len(micro_batches))
print("Micro-batch shape   :", tuple(micro_batches[0][0].shape))

# ============================================================================
# 6. EVALUATION SET
# ============================================================================

eval_sequences = token_sequences[-min(128, len(token_sequences)):]

@torch.no_grad()
def validation_loss(m):
    m.eval()

    losses = []

    # Keep validation bounded for diagnostic speed.
    for seq in eval_sequences[:64]:
        x, y = make_batch(seq)

        x = x.unsqueeze(0).to(DEVICE)
        y = y.unsqueeze(0).to(DEVICE)

        out = m(
            input_ids=x,
            labels=y
        )

        losses.append(float(out.loss.detach().cpu()))

    m.train()

    return sum(losses) / len(losses)


@torch.no_grad()
def generation_metrics(m, temperature=0.9, max_new_tokens=128):
    m.eval()

    seq = token_sequences[0]

    prompt_ids = torch.tensor(
        seq[:64],
        dtype=torch.long,
        device=DEVICE
    ).unsqueeze(0)

    generated = prompt_ids.clone()

    for _ in range(max_new_tokens):
        input_ids = generated[:, -SEQ_LEN:]

        out = m(input_ids=input_ids)

        logits = out.logits[:, -1, :]
        logits = logits / temperature

        probs = torch.softmax(logits, dim=-1)

        next_id = torch.multinomial(probs, num_samples=1)

        generated = torch.cat(
            [generated, next_id],
            dim=1
        )

    ids = generated[0, prompt_ids.shape[1]:].tolist()

    if not ids:
        return {
            "unique": 0.0,
            "adj": 0.0,
            "d2": 0.0,
            "d3": 0.0,
            "maxrun": 0,
        }

    n = len(ids)

    unique = len(set(ids)) / n

    adj_repeats = sum(
        ids[i] == ids[i - 1]
        for i in range(1, n)
    )

    adj = adj_repeats / max(1, n - 1)

    bigrams = list(zip(ids[:-1], ids[1:]))
    trigrams = list(zip(ids[:-2], ids[1:-1], ids[2:]))

    d2 = (
        len(set(bigrams)) / len(bigrams)
        if bigrams else 0.0
    )

    d3 = (
        len(set(trigrams)) / len(trigrams)
        if trigrams else 0.0
    )

    maxrun = 1
    current = 1

    for i in range(1, n):
        if ids[i] == ids[i - 1]:
            current += 1
            maxrun = max(maxrun, current)
        else:
            current = 1

    m.train()

    return {
        "unique": unique,
        "adj": adj,
        "d2": d2,
        "d3": d3,
        "maxrun": maxrun,
    }


# ============================================================================
# 7. BASELINE
# ============================================================================

print("\n" + "=" * 100)
print("BASELINE — EXACT STEP-1000 MODEL")
print("=" * 100)

base_loss = validation_loss(test_model)
base_ppl = math.exp(min(base_loss, 20))

base_metrics = generation_metrics(
    test_model,
    temperature=0.9
)

print(f"Validation loss : {base_loss:.4f}")
print(f"Perplexity      : {base_ppl:.2f}")
print(
    "T=0.9 | "
    f"unique={base_metrics['unique']:.4f} | "
    f"adj={base_metrics['adj']:.4f} | "
    f"d2={base_metrics['d2']:.4f} | "
    f"d3={base_metrics['d3']:.4f} | "
    f"maxrun={base_metrics['maxrun']}"
)

# ============================================================================
# 8. OPTIMIZER
# ============================================================================

optimizer = torch.optim.AdamW(
    test_model.parameters(),
    lr=LR,
    weight_decay=0.01
)

# IMPORTANT:
# Original checkpoint optimizer state is intentionally NOT loaded.
# This is a controlled LR experiment starting from exact model weights.

print("\n" + "=" * 100)
print("250-STEP LR=5e-6 + GRADACCUM=8 CONTINUATION")
print("=" * 100)

history = []

start_time = time.time()

optimizer.zero_grad(set_to_none=True)

for step in range(1, STEPS + 1):

    accum_loss = 0.0
    last_grad = 0.0

    for micro in range(GRAD_ACCUM):

        idx = (step - 1) * GRAD_ACCUM + micro

        x, y = micro_batches[idx]

        x = x.to(DEVICE)
        y = y.to(DEVICE)

        out = test_model(
            input_ids=x,
            labels=y
        )

        loss = out.loss
        accum_loss += float(loss.detach().cpu())

        scaled_loss = loss / GRAD_ACCUM
        scaled_loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        test_model.parameters(),
        max_norm=1.0
    )

    last_grad = float(
        grad_norm.detach().cpu()
        if torch.is_tensor(grad_norm)
        else grad_norm
    )

    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    avg_loss = accum_loss / GRAD_ACCUM

    if step == 1 or step % 25 == 0:
        print(
            f"step={step:4d} | "
            f"loss={avg_loss:.4f} | "
            f"grad={last_grad:.4f}"
        )

    # ------------------------------------------------------------
    # Diagnostic checkpoints
    # ------------------------------------------------------------

    if step in (50, 100, 150, 200, 250):

        val_loss = validation_loss(test_model)
        ppl = math.exp(min(val_loss, 20))

        metrics = generation_metrics(
            test_model,
            temperature=0.9
        )

        history.append({
            "step": step,
            "loss": val_loss,
            "perplexity": ppl,
            **metrics
        })

        print("\n" + "-" * 90)
        print(f"DIAGNOSTIC CHECKPOINT — +{step} STEPS")
        print("-" * 90)

        print(f"Validation loss : {val_loss:.4f}")
        print(f"Perplexity      : {ppl:.2f}")

        print(
            "T=0.9 | "
            f"unique={metrics['unique']:.4f} | "
            f"adj={metrics['adj']:.4f} | "
            f"d2={metrics['d2']:.4f} | "
            f"d3={metrics['d3']:.4f} | "
            f"maxrun={metrics['maxrun']}"
        )

# ============================================================================
# 9. SUMMARY
# ============================================================================

elapsed = (time.time() - start_time) / 60.0

final = history[-1]

print("\n" + "=" * 100)
print("CS781 — LR=5e-6 + GRADACCUM=8 SUMMARY")
print("=" * 100)

print(f"Elapsed time : {elapsed:.2f} minutes")

print()
print(
    f"{'POINT':<16}"
    f"{'LOSS':>10}"
    f"{'PPL':>10}"
    f"{'UNIQUE':>10}"
    f"{'ADJ':>10}"
    f"{'D2':>10}"
    f"{'D3':>10}"
    f"{'MAXRUN':>9}"
)

print("-" * 85)

print(
    f"{'step1000':<16}"
    f"{base_loss:>10.4f}"
    f"{base_ppl:>10.2f}"
    f"{base_metrics['unique']:>10.4f}"
    f"{base_metrics['adj']:>10.4f}"
    f"{base_metrics['d2']:>10.4f}"
    f"{base_metrics['d3']:>10.4f}"
    f"{base_metrics['maxrun']:>9}"
)

for r in history:
    print(
        f"{'step+' + str(r['step']):<16}"
        f"{r['loss']:>10.4f}"
        f"{r['perplexity']:>10.2f}"
        f"{r['unique']:>10.4f}"
        f"{r['adj']:>10.4f}"
        f"{r['d2']:>10.4f}"
        f"{r['d3']:>10.4f}"
        f"{r['maxrun']:>9}"
    )

loss_change = final["loss"] - base_loss

print("\n" + "=" * 100)
print("DECISION SIGNAL")
print("=" * 100)

print(f"Baseline loss : {base_loss:.4f}")
print(f"Final loss    : {final['loss']:.4f}")
print(f"Loss change   : {loss_change:+.4f}")

if loss_change < -0.10:
    print("\n✓ MEANINGFUL RECOVERY SIGNAL")
elif loss_change < 0:
    print("\n~ WEAK IMPROVEMENT")
else:
    print("\n✗ NO VALIDATION-LOSS IMPROVEMENT")

# ============================================================================
# 10. SAVE ONLY DIAGNOSTIC RESULTS
# ============================================================================

result = {
    "experiment": "lr5e6_gradaccum8_controlled",
    "checkpoint": CHECKPOINT_PATH,
    "checkpoint_step": int(ckpt["step"]),
    "steps": STEPS,
    "seq_len": SEQ_LEN,
    "micro_batch": MICRO_BATCH,
    "gradient_accumulation": GRAD_ACCUM,
    "effective_batch": MICRO_BATCH * GRAD_ACCUM,
    "learning_rate": LR,
    "baseline": {
        "loss": base_loss,
        "perplexity": base_ppl,
        **base_metrics,
    },
    "history": history,
    "elapsed_minutes": elapsed,
}

result_path = os.path.join(
    OUTPUT_DIR,
    "lr5e6_gradaccum8_results.json"
)

with open(result_path, "w") as f:
    json.dump(result, f, indent=2)

print("\n" + "=" * 100)
print("✓ CONTROLLED EXPERIMENT COMPLETE")
print("=" * 100)
print("Results:", result_path)
print("Original Step-1000 checkpoint: NOT modified")
print("Production checkpoint: NOT written")
print("Recovered notebook model: NOT modified")
print("=" * 100)

In [ ]:
# ==========================================================================================
# CS781 — LR=5e-6 + GRADACCUM=8 CONTROLLED TEST
# FIXED: reuse the WORKING 3170 usable sequences; cycle them if necessary.
# Does NOT require 4000 usable sequences.
# Does NOT modify the original checkpoint.
# ==========================================================================================

import os
import json
import math
import time
import glob
import copy
import torch
import torch.nn.functional as F

print("=" * 100)
print("CS781 — LR=5e-6 + ORIGINAL GRADIENT ACCUMULATION CONTROLLED TEST")
print("=" * 100)

# ------------------------------------------------------------------------------------------
# CONFIG — SAME EXPERIMENT AS THE SUCCESSFUL RUN
# ------------------------------------------------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_PATH = (
    "/kaggle/input/datasets/oipranesh/"
    "phase4-step1000-verified-model/checkpoint_step_001000.pt"
)

DATA_ROOT = "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"

STEPS = 250
SEQ_LEN = 1024
MICRO_BATCH = 1
GRAD_ACCUM = 8
LR = 5e-6
MAX_GRAD_NORM = 1.0

OUTPUT_DIR = "/kaggle/working/cs781_lr5e6_accum8_controlled"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Device         : {DEVICE}")
print(f"Checkpoint     : {CHECKPOINT_PATH}")
print(f"Steps          : {STEPS}")
print(f"SeqLen         : {SEQ_LEN}")
print(f"MicroBatch     : {MICRO_BATCH}")
print(f"GradAccum      : {GRAD_ACCUM}")
print(f"Effective batch: {MICRO_BATCH * GRAD_ACCUM}")
print(f"LR             : {LR}")

# ------------------------------------------------------------------------------------------
# VERIFY EXISTING NOTEBOOK STATE
# ------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("VERIFYING RECOVERED NOTEBOOK STATE")
print("=" * 100)

if "model" not in globals() or model is None:
    raise RuntimeError(
        "model is missing/None. Run the SAFE MODEL OBJECT RECOVERY cell first."
    )

if "tokenizer" not in globals() or tokenizer is None:
    raise RuntimeError(
        "tokenizer is missing. Run the previous successful tokenizer/model loading cell first."
    )

print(f"✓ model       : {type(model).__name__}")
print(f"✓ tokenizer   : {type(tokenizer).__name__}")

# ------------------------------------------------------------------------------------------
# LOAD CHECKPOINT
# ------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("LOADING EXACT STEP-1000 WEIGHTS")
print("=" * 100)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

state_dict = checkpoint["model_state_dict"]

# Use the recovered architecture.
model.load_state_dict(state_dict, strict=True)
model = model.to(DEVICE)
model.eval()

print(f"Checkpoint step : {checkpoint.get('step')}")
print(f"Checkpoint LR   : {checkpoint.get('learning_rate')}")
print(f"Context length  : {checkpoint.get('context_length')}")
print("Missing keys    : 0")
print("Unexpected keys : 0")
print("✓ Exact Step-1000 weights loaded")
print("✓ Original checkpoint untouched")

# ------------------------------------------------------------------------------------------
# DATA LOADING — SAME DATA SOURCE AS THE WORKING EXPERIMENT
# ------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("LOADING TRAINING DATA")
print("=" * 100)

jsonl_files = sorted(
    glob.glob(os.path.join(DATA_ROOT, "dedup", "acts", "*.jsonl"))
    + glob.glob(os.path.join(DATA_ROOT, "dedup", "high_courts", "*.jsonl"))
    + glob.glob(os.path.join(DATA_ROOT, "dedup", "**", "*.jsonl"), recursive=True)
)

# Deduplicate paths while preserving order
jsonl_files = list(dict.fromkeys(jsonl_files))

print(f"Training JSONL files: {len(jsonl_files)}")

# Recover the same kind of raw texts as the successful diagnostic.
texts = []

def extract_text(obj):
    if isinstance(obj, str):
        return obj

    if not isinstance(obj, dict):
        return None

    # Same common fields used by the earlier diagnostics.
    for key in (
        "text",
        "content",
        "document",
        "body",
        "raw_text",
        "full_text",
        "clean_text",
    ):
        value = obj.get(key)
        if isinstance(value, str) and value.strip():
            return value

    return None


for path in jsonl_files:
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue

                try:
                    obj = json.loads(line)
                except Exception:
                    continue

                text = extract_text(obj)

                if text and len(text.strip()) > 100:
                    texts.append(text)

                # Preserve the successful diagnostic's 4000-text cap.
                if len(texts) >= 4000:
                    break

        if len(texts) >= 4000:
            break

    except Exception as e:
        print(f"Warning reading {path}: {e}")

print(f"Training texts recovered: {len(texts)}")

if not texts:
    raise RuntimeError("No training texts recovered.")

# ------------------------------------------------------------------------------------------
# TOKENIZE — DO NOT CHANGE TOKENIZER
# ------------------------------------------------------------------------------------------

print("\nTokenizing training samples...")

token_sequences = []

for text in texts:
    try:
        ids = tokenizer.encode(text).ids
    except Exception:
        continue

    if len(ids) >= SEQ_LEN + 1:
        # Non-overlapping chunks, matching the earlier diagnostic approach.
        usable = len(ids) // (SEQ_LEN + 1)

        for i in range(usable):
            start = i * (SEQ_LEN + 1)
            chunk = ids[start:start + SEQ_LEN + 1]

            if len(chunk) == SEQ_LEN + 1:
                token_sequences.append(chunk)

except_count = 0

print(f"Usable token sequences: {len(token_sequences)}")

if len(token_sequences) == 0:
    raise RuntimeError("No usable token sequences were recovered.")

# IMPORTANT:
# The successful experiment recovered 3170 sequences.
# We DO NOT require 4000.
#
# We need:
#   250 optimizer steps × 8 microbatches = 2000 microbatches.
#
# 3170 sequences are therefore MORE than enough.
required_microbatches = STEPS * GRAD_ACCUM

print(f"Required micro-batches: {required_microbatches}")

if len(token_sequences) < required_microbatches:
    # Only cycle if the dataset is genuinely smaller.
    print(
        f"Dataset has {len(token_sequences)} sequences; "
        f"cycling deterministically to obtain {required_microbatches} micro-batches."
    )

    expanded = [
        token_sequences[i % len(token_sequences)]
        for i in range(required_microbatches)
    ]
else:
    # Deterministic fixed subset — same idea as the previous successful run.
    expanded = token_sequences[:required_microbatches]

print(f"Fixed micro-batches: {len(expanded)}")

# ------------------------------------------------------------------------------------------
# BATCH TENSORS
# ------------------------------------------------------------------------------------------

batches = []

for seq in expanded:
    x = torch.tensor(
        seq[:SEQ_LEN],
        dtype=torch.long
    )

    y = torch.tensor(
        seq[1:SEQ_LEN + 1],
        dtype=torch.long
    )

    batches.append((x.unsqueeze(0), y.unsqueeze(0)))

print(f"Micro-batch shape: ({MICRO_BATCH}, {SEQ_LEN})")

# ------------------------------------------------------------------------------------------
# EVALUATION DATA
# ------------------------------------------------------------------------------------------

# Use deterministic evaluation sequences from the recovered dataset.
eval_sequences = token_sequences[:min(5, len(token_sequences))]

if len(eval_sequences) == 0:
    raise RuntimeError("No evaluation sequences available.")

# ------------------------------------------------------------------------------------------
# EVALUATION FUNCTION
# ------------------------------------------------------------------------------------------

@torch.no_grad()
def validation_loss(model_obj, sequences, max_items=5):
    model_obj.eval()

    losses = []

    for seq in sequences[:max_items]:
        ids = torch.tensor(
            seq,
            dtype=torch.long,
            device=DEVICE
        ).unsqueeze(0)

        inputs = ids[:, :-1]
        targets = ids[:, 1:]

        outputs = model_obj(input_ids=inputs)

        logits = outputs.logits

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            targets.reshape(-1)
        )

        losses.append(float(loss.item()))

    return sum(losses) / len(losses)


# ------------------------------------------------------------------------------------------
# GENERATION METRICS
# ------------------------------------------------------------------------------------------

@torch.no_grad()
def generation_metrics(model_obj, temperature=0.9, max_new_tokens=128):
    model_obj.eval()

    # Use a real legal-text prefix from the first evaluation sequence.
    seed = eval_sequences[0][:min(32, len(eval_sequences[0]))]

    input_ids = torch.tensor(
        seed,
        dtype=torch.long,
        device=DEVICE
    ).unsqueeze(0)

    generated = input_ids.clone()

    for _ in range(max_new_tokens):
        context = generated[:, -SEQ_LEN:]

        outputs = model_obj(input_ids=context)
        logits = outputs.logits[:, -1, :] / temperature

        probs = torch.softmax(logits, dim=-1)

        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        generated = torch.cat(
            [generated, next_token],
            dim=1
        )

    ids = generated[0].tolist()

    if len(ids) < 2:
        return {
            "unique": 0.0,
            "adj": 0.0,
            "d2": 0.0,
            "d3": 0.0,
            "maxrun": 1,
        }

    unique = len(set(ids)) / len(ids)

    adj_repeats = sum(
        1 for i in range(1, len(ids))
        if ids[i] == ids[i - 1]
    )

    adj = adj_repeats / (len(ids) - 1)

    bigrams = list(zip(ids[:-1], ids[1:]))
    trigrams = list(zip(ids[:-2], ids[1:-1], ids[2:]))

    d2 = len(set(bigrams)) / max(1, len(bigrams))
    d3 = len(set(trigrams)) / max(1, len(trigrams))

    maxrun = 1
    current = 1

    for i in range(1, len(ids)):
        if ids[i] == ids[i - 1]:
            current += 1
            maxrun = max(maxrun, current)
        else:
            current = 1

    return {
        "unique": unique,
        "adj": adj,
        "d2": d2,
        "d3": d3,
        "maxrun": maxrun,
    }


# ------------------------------------------------------------------------------------------
# BASELINE
# ------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("BASELINE — EXACT STEP-1000 MODEL")
print("=" * 100)

baseline_loss = validation_loss(
    model,
    eval_sequences
)

baseline_ppl = math.exp(min(baseline_loss, 20))

baseline_metrics = generation_metrics(
    model,
    temperature=0.9
)

print(f"Validation loss : {baseline_loss:.4f}")
print(f"Perplexity      : {baseline_ppl:.2f}")
print(
    "T=0.9 | "
    f"unique={baseline_metrics['unique']:.4f} | "
    f"adj={baseline_metrics['adj']:.4f} | "
    f"d2={baseline_metrics['d2']:.4f} | "
    f"d3={baseline_metrics['d3']:.4f} | "
    f"maxrun={baseline_metrics['maxrun']}"
)

# ------------------------------------------------------------------------------------------
# TRAINING COPY
# ------------------------------------------------------------------------------------------

# NEVER train the notebook's recovered model object directly.
train_model = copy.deepcopy(model).to(DEVICE)
train_model.train()

optimizer = torch.optim.AdamW(
    train_model.parameters(),
    lr=LR
)

# ------------------------------------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("250-STEP LR=5e-6 + GRADACCUM=8 CONTINUATION")
print("=" * 100)

start_time = time.time()

results = []

optimizer.zero_grad(set_to_none=True)

for step in range(1, STEPS + 1):

    step_loss = 0.0
    last_grad = 0.0

    for accum_idx in range(GRAD_ACCUM):

        batch_index = (step - 1) * GRAD_ACCUM + accum_idx

        x, y = batches[batch_index]

        x = x.to(DEVICE)
        y = y.to(DEVICE)

        outputs = train_model(input_ids=x)

        logits = outputs.logits

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            y.reshape(-1)
        )

        step_loss += float(loss.item())

        loss = loss / GRAD_ACCUM
        loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        train_model.parameters(),
        MAX_GRAD_NORM
    )

    last_grad = float(grad_norm)

    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    avg_loss = step_loss / GRAD_ACCUM

    if step == 1 or step % 25 == 0:
        print(
            f"step={step:4d} | "
            f"loss={avg_loss:.4f} | "
            f"grad={last_grad:.4f}"
        )

    # ----------------------------------------------------------------------
    # DIAGNOSTIC CHECKPOINTS
    # ----------------------------------------------------------------------

    if step in (50, 100, 150, 200, 250):

        val_loss = validation_loss(
            train_model,
            eval_sequences
        )

        ppl = math.exp(min(val_loss, 20))

        metrics = generation_metrics(
            train_model,
            temperature=0.9
        )

        results.append({
            "step": step,
            "loss": val_loss,
            "perplexity": ppl,
            **metrics
        })

        print("\n" + "-" * 90)
        print(f"DIAGNOSTIC CHECKPOINT — +{step} STEPS")
        print("-" * 90)

        print(f"Validation loss : {val_loss:.4f}")
        print(f"Perplexity      : {ppl:.2f}")
        print(
            "T=0.9 | "
            f"unique={metrics['unique']:.4f} | "
            f"adj={metrics['adj']:.4f} | "
            f"d2={metrics['d2']:.4f} | "
            f"d3={metrics['d3']:.4f} | "
            f"maxrun={metrics['maxrun']}"
        )

# ------------------------------------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------------------------------------

elapsed = time.time() - start_time

final = results[-1]

loss_change = final["loss"] - baseline_loss

print("\n" + "=" * 100)
print("CS781 — LR=5e-6 + GRADACCUM=8 SUMMARY")
print("=" * 100)

print(f"Elapsed time : {elapsed / 60:.2f} minutes")

print(
    f"{'POINT':<18}"
    f"{'LOSS':>10}"
    f"{'PPL':>10}"
    f"{'UNIQUE':>12}"
    f"{'ADJ':>10}"
    f"{'D2':>10}"
    f"{'D3':>10}"
    f"{'MAXRUN':>9}"
)

print("-" * 90)

print(
    f"{'step1000':<18}"
    f"{baseline_loss:>10.4f}"
    f"{baseline_ppl:>10.2f}"
    f"{baseline_metrics['unique']:>12.4f}"
    f"{baseline_metrics['adj']:>10.4f}"
    f"{baseline_metrics['d2']:>10.4f}"
    f"{baseline_metrics['d3']:>10.4f}"
    f"{baseline_metrics['maxrun']:>9}"
)

for r in results:
    print(
        f"{'step+' + str(r['step']):<18}"
        f"{r['loss']:>10.4f}"
        f"{r['perplexity']:>10.2f}"
        f"{r['unique']:>12.4f}"
        f"{r['adj']:>10.4f}"
        f"{r['d2']:>10.4f}"
        f"{r['d3']:>10.4f}"
        f"{r['maxrun']:>9}"
    )

print("\n" + "=" * 100)
print("DECISION SIGNAL")
print("=" * 100)

print(f"Baseline loss : {baseline_loss:.4f}")
print(f"Final loss    : {final['loss']:.4f}")
print(f"Loss change   : {loss_change:+.4f}")

if loss_change < -0.25:
    print("\n✓ MEANINGFUL RECOVERY SIGNAL")
elif loss_change < -0.05:
    print("\n~ WEAK IMPROVEMENT — recovery signal exists but is not decisive.")
else:
    print("\n~ NO CLEAR RECOVERY SIGNAL")

# ------------------------------------------------------------------------------------------
# SAVE ONLY DIAGNOSTIC RESULTS
# ------------------------------------------------------------------------------------------

result_payload = {
    "experiment": "lr5e6_gradaccum8_controlled",
    "checkpoint": CHECKPOINT_PATH,
    "checkpoint_step": checkpoint.get("step"),
    "steps": STEPS,
    "seq_len": SEQ_LEN,
    "micro_batch": MICRO_BATCH,
    "grad_accum": GRAD_ACCUM,
    "effective_batch": MICRO_BATCH * GRAD_ACCUM,
    "learning_rate": LR,
    "usable_sequences": len(token_sequences),
    "micro_batches_used": len(batches),
    "baseline": {
        "loss": baseline_loss,
        "perplexity": baseline_ppl,
        **baseline_metrics,
    },
    "checkpoints": results,
    "elapsed_seconds": elapsed,
    "loss_change": loss_change,
}

result_path = os.path.join(
    OUTPUT_DIR,
    "lr5e6_gradaccum8_results.json"
)

with open(result_path, "w", encoding="utf-8") as f:
    json.dump(result_payload, f, indent=2)

print("\n" + "=" * 100)
print("✓ CONTROLLED EXPERIMENT COMPLETE")
print("=" * 100)
print(f"Results: {result_path}")
print("Original Step-1000 checkpoint: NOT modified")
print("Production checkpoint: NOT written")
print("Recovered notebook model: NOT modified")
print("=" * 100)

In [ ]:
# ==========================================================================================
# CS781 — CELL 1: SAFE 113M MODEL / CHECKPOINT / TOKENIZER SANITY CHECK
# ==========================================================================================
# PURPOSE:
#   Verify that the notebook is in a valid state BEFORE any new experiment.
#
# IMPORTANT:
#   - NO TRAINING
#   - NO optimizer
#   - NO checkpoint modification
#   - NO model modification
#   - Reuses the already-working tokenizer/model architecture
# ==========================================================================================

import os
import torch

print("=" * 100)
print("CS781 — 113M MODEL PRE-EXPERIMENT SANITY CHECK")
print("=" * 100)

# ------------------------------------------------------------------------------
# 1. CONSTANTS — use the already verified paths
# ------------------------------------------------------------------------------

CHECKPOINT_PATH = "/kaggle/input/datasets/oipranesh/phase4-step1000-verified-model/checkpoint_step_001000.pt"

TOKENIZER_PATH = "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep/tokenizer/legal_bpe_32k.json"

EXPECTED_VOCAB = 32000
EXPECTED_PARAMS = 112777728
EXPECTED_CHECKPOINT_STEP = 1000

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\n[1] ENVIRONMENT")
print("-" * 100)
print("PyTorch       :", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device        :", DEVICE)

if torch.cuda.is_available():
    print("GPU           :", torch.cuda.get_device_name(0))

# ------------------------------------------------------------------------------
# 2. FILE INTEGRITY / EXISTENCE
# ------------------------------------------------------------------------------

print("\n[2] VERIFIED FILES")
print("-" * 100)

if not os.path.exists(CHECKPOINT_PATH):
    raise RuntimeError(
        f"Step-1000 checkpoint not found:\n{CHECKPOINT_PATH}"
    )

if not os.path.exists(TOKENIZER_PATH):
    raise RuntimeError(
        f"Tokenizer not found:\n{TOKENIZER_PATH}"
    )

ckpt_size_gb = os.path.getsize(CHECKPOINT_PATH) / (1024 ** 3)

print("✓ Step-1000 checkpoint exists")
print("  Path :", CHECKPOINT_PATH)
print(f"  Size : {ckpt_size_gb:.2f} GB")

print("✓ Legal BPE tokenizer exists")
print("  Path :", TOKENIZER_PATH)

# ------------------------------------------------------------------------------
# 3. VERIFY TOKENIZER — REUSE EXISTING OBJECT IF AVAILABLE
# ------------------------------------------------------------------------------

print("\n[3] TOKENIZER")
print("-" * 100)

if "tokenizer" not in globals() or tokenizer is None:
    raise RuntimeError(
        "The previously recovered tokenizer object is missing.\n"
        "Do NOT create a new tokenizer here.\n"
        "Run the earlier successful tokenizer/model recovery cell first."
    )

print("Tokenizer type:", type(tokenizer).__name__)

# tokenizers.Tokenizer uses get_vocab(), not vocab_size
try:
    vocab = tokenizer.get_vocab()
    vocab_size = len(vocab)
except Exception as e:
    raise RuntimeError(f"Existing tokenizer cannot be inspected: {e}")

print("Vocabulary size:", vocab_size)

if vocab_size != EXPECTED_VOCAB:
    raise RuntimeError(
        f"Unexpected vocabulary size: {vocab_size}. "
        f"Expected exactly {EXPECTED_VOCAB}."
    )

# Basic encode/decode test
test_text = "The court shall determine the matter."
enc = tokenizer.encode(test_text)

print("Encode test    : ✓")
print("Test tokens    :", enc.ids[:12])

try:
    decoded = tokenizer.decode(enc.ids)
    print("Decode test    : ✓")
    print("Decoded sample :", repr(decoded[:100]))
except Exception as e:
    raise RuntimeError(f"Tokenizer decode failed: {e}")

# ------------------------------------------------------------------------------
# 4. VERIFY CHECKPOINT — READ ONLY
# ------------------------------------------------------------------------------

print("\n[4] STEP-1000 CHECKPOINT")
print("-" * 100)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(checkpoint).__name__)

required_ckpt_keys = [
    "step",
    "model_state_dict",
    "optimizer_state_dict",
    "context_length",
    "batch_size",
    "gradient_accumulation_steps",
    "learning_rate",
    "max_grad_norm",
]

missing_ckpt_keys = [
    k for k in required_ckpt_keys
    if k not in checkpoint
]

if missing_ckpt_keys:
    raise RuntimeError(
        f"Checkpoint is missing expected keys: {missing_ckpt_keys}"
    )

print("Checkpoint step       :", checkpoint["step"])
print("Context length        :", checkpoint["context_length"])
print("Batch size            :", checkpoint["batch_size"])
print("Grad accumulation     :", checkpoint["gradient_accumulation_steps"])
print("Checkpoint LR         :", checkpoint["learning_rate"])
print("Max grad norm         :", checkpoint["max_grad_norm"])

if checkpoint["step"] != EXPECTED_CHECKPOINT_STEP:
    raise RuntimeError(
        f"Wrong checkpoint step: {checkpoint['step']} "
        f"(expected {EXPECTED_CHECKPOINT_STEP})"
    )

print("✓ Exact Step-1000 checkpoint confirmed")

# ------------------------------------------------------------------------------
# 5. VERIFY MODEL OBJECT
# ------------------------------------------------------------------------------

print("\n[5] MODEL OBJECT")
print("-" * 100)

if "model" not in globals() or model is None:
    raise RuntimeError(
        "model is None or does not exist.\n"
        "Run the previously successful SAFE MODEL OBJECT RECOVERY cell first.\n"
        "This cell intentionally refuses to invent a new architecture."
    )

print("Model type:", type(model).__name__)

# Parameter count
param_count = sum(p.numel() for p in model.parameters())

print("Parameters :", param_count)

if param_count != EXPECTED_PARAMS:
    raise RuntimeError(
        f"Unexpected parameter count: {param_count:,}\n"
        f"Expected: {EXPECTED_PARAMS:,}"
    )

print("✓ Expected 112.8M architecture confirmed")

# ------------------------------------------------------------------------------
# 6. VERIFY MODEL ↔ CHECKPOINT COMPATIBILITY
# ------------------------------------------------------------------------------

print("\n[6] MODEL ↔ CHECKPOINT COMPATIBILITY")
print("-" * 100)

model_state = model.state_dict()
ckpt_state = checkpoint["model_state_dict"]

model_keys = set(model_state.keys())
ckpt_keys = set(ckpt_state.keys())

missing_keys = sorted(ckpt_keys - model_keys)
unexpected_keys = sorted(model_keys - ckpt_keys)

shape_mismatches = []

for key in sorted(model_keys & ckpt_keys):
    if model_state[key].shape != ckpt_state[key].shape:
        shape_mismatches.append(
            (
                key,
                tuple(model_state[key].shape),
                tuple(ckpt_state[key].shape)
            )
        )

print("Model tensors      :", len(model_state))
print("Checkpoint tensors :", len(ckpt_state))
print("Missing keys       :", len(missing_keys))
print("Unexpected keys    :", len(unexpected_keys))
print("Shape mismatches   :", len(shape_mismatches))

if missing_keys:
    print("\nMissing:")
    for k in missing_keys[:10]:
        print(" ", k)

if unexpected_keys:
    print("\nUnexpected:")
    for k in unexpected_keys[:10]:
        print(" ", k)

if shape_mismatches:
    print("\nShape mismatches:")
    for item in shape_mismatches[:10]:
        print(" ", item)

if missing_keys or unexpected_keys or shape_mismatches:
    raise RuntimeError(
        "Model/checkpoint architecture mismatch. "
        "STOP before any experiment."
    )

print("✓ Architecture matches checkpoint exactly")

# ------------------------------------------------------------------------------
# 7. READ-ONLY FORWARD PASS
# ------------------------------------------------------------------------------

print("\n[7] READ-ONLY FORWARD PASS")
print("-" * 100)

model.eval()

# Put model on GPU only for the test.
model = model.to(DEVICE)

test_ids = enc.ids[:8]

if len(test_ids) < 2:
    raise RuntimeError("Tokenizer produced too few test tokens.")

input_ids = torch.tensor(
    [test_ids],
    dtype=torch.long,
    device=DEVICE
)

with torch.no_grad():
    output = model(input_ids)

logits = output.logits

print("Input shape :", tuple(input_ids.shape))
print("Logits shape:", tuple(logits.shape))

expected_last_dim = EXPECTED_VOCAB

if logits.shape[-1] != expected_last_dim:
    raise RuntimeError(
        f"Unexpected logits vocabulary dimension: {logits.shape[-1]}"
    )

print("✓ Forward pass successful")
print("✓ Logits vocabulary dimension = 32,000")

# ------------------------------------------------------------------------------
# 8. FINAL STATUS
# ------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("✓ CELL 1 PASSED — SAFE STATE FOR NEXT EXPERIMENT")
print("=" * 100)

print(f"""
Model:
  Type              : {type(model).__name__}
  Parameters        : {param_count:,}
  Device            : {DEVICE}
  Vocabulary        : {vocab_size:,}
  Context capacity  : {checkpoint["context_length"]}

Checkpoint:
  Step              : {checkpoint["step"]}
  Path              : {CHECKPOINT_PATH}
  Original file     : UNTOUCHED

Tokenizer:
  Type              : {type(tokenizer).__name__}
  Vocabulary        : {vocab_size:,}
  Path              : {TOKENIZER_PATH}

Training:
  Performed         : NO
  Optimizer touched : NO
  Checkpoint saved  : NO
  Checkpoint changed: NO

STATUS: READY FOR CELL 2 — FIXED BASELINE EVALUATION
""")

# Keep checkpoint in memory for subsequent cells if useful,
# but DO NOT alter it.